<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Notebook 10 — Taxonomy and Knowledge Engineering</h2>

<p>
هدف این Notebook ساخت و اعتبارسنجی لایه مهندسی دانش برای Dataset پذیرفته‌شده فاز ۲ است.
این مرحله باید محتوای موجود را به ساختار موضوعی و مفهومی قابل استفاده برای مراحل بعدی RAG متصل کند، بدون اینکه Source Attribution، Knowledge Authority، Conversation Context یا Lineage از بین برود.
</p>

<p>
ورودی اصلی این مرحله، Dataset پذیرفته‌شده Notebook 09 و شواهد Acceptance و Checkpoint همان مرحله است.
</p>

<p>
در این Notebook هنوز Chunking، Embedding، Indexing یا Retrieval انجام نمی‌شود.
Chunking متعلق به Notebook 11 است و هیچ Rechunking در این مرحله مجاز نیست.
</p>

<h3>دامنه این Notebook</h3>

<ul>
<li>کشف ساختار یا Taxonomy موجود در Repository.</li>
<li>تعریف یا تکمیل Taxonomy سواد مالی بر اساس Evidence واقعی Corpus.</li>
<li>تعریف Topic، Competency، Prerequisite و نوع محتوای آموزشی در صورت پشتیبانی داده.</li>
<li>حفظ تفکیک میان Knowledge Authority، Source Attribution و Conversation Context.</li>
<li>طراحی نگاشت قابل توضیح Documentها به Taxonomy.</li>
<li>بازبینی Coverage، Conflict و موارد نیازمند Human Review.</li>
<li>تولید Artifactهای نسخه‌دار فقط پس از Discovery و Validation.</li>
</ul>

<h3>خارج از دامنه</h3>

<ul>
<li>Chunking یا Rechunking.</li>
<li>Embedding.</li>
<li>Dense/Sparse Index.</li>
<li>Hybrid Retrieval و Reranking.</li>
<li>Generation و Citation.</li>
</ul>

<h3>شرط پایان Notebook</h3>

<p>
Notebook فقط زمانی Done محسوب می‌شود که Taxonomy و نگاشت‌های پذیرفته‌شده دارای Evidence، Version، Lineage و Validation مشخص باشند، تمام Gateها PASS شوند و Restart Kernel → Run All بدون Error انجام شود.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Repository / Artifact Preflight</h3>

<p>
قبل از طراحی Taxonomy باید وضعیت واقعی Repository و خروجی‌های Notebook 09 از روی Disk بررسی شوند.
این مرحله کاملاً Read-Only است و هیچ فایل یا Artifactی ایجاد، حذف یا اصلاح نمی‌کند.
</p>

<p>
در این بررسی موارد زیر کنترل می‌شوند:
</p>

<ul>
<li>Repository Root، Branch، HEAD و Git Status.</li>
<li>وجود Notebook 10 در صورت ساخته‌شدن قبلی.</li>
<li>Dataset پذیرفته‌شده Notebook 09، تعداد رکورد و SHA-256 آن.</li>
<li>Acceptance Report و Checkpoint نهایی Notebook 09.</li>
<li>فایل‌ها و Artifactهای Taxonomy یا Knowledge Engineering که ممکن است از قبل وجود داشته باشند.</li>
<li>Schemaها و Manifestهای مرتبط موجود.</li>
<li>اسناد Roadmap یا Plan موجود در خود Repository که به Notebook 10 اشاره می‌کنند.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>Repository و Branch مورد انتظار شناسایی شوند.</li>
<li>Dataset پذیرفته‌شده Notebook 09 موجود و قابل خواندن باشد.</li>
<li>Count و Hash آن با Artifact پذیرفته‌شده NB09 تطبیق داشته باشد.</li>
<li>Acceptance Report معتبر و بدون خرابی باشد.</li>
<li>Checkpoint معتبر باشد و وضعیت PASS مرحله قبل قابل اثبات باشد.</li>
</ul>

<p>
وجود نداشتن Taxonomy قبلی FAIL محسوب نمی‌شود؛ هدف این Preflight دقیقاً کشف این موضوع است.
</p>

</div>

In [44]:
# Code Cell 01

from pathlib import Path
import hashlib
import json
import os
import subprocess


EXPECTED_BRANCH = "rag-phase2"
EXPECTED_DATASET_RECORDS = 19456
EXPECTED_DATASET_SHA256 = "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
EXPECTED_ACCEPTANCE_SHA256 = "14dfa748988db17a19c6d9d5e1eac2eb7d23c0a1699dcc7add18a2bf461fe4f3"


def git(*args):
    return subprocess.run(
        ["git", *args],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def count_jsonl(path):
    count = 0
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            json.loads(line)
            count += 1
    return count


def relevant_json_scalars(value, prefix=""):
    rows = []

    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            rows.extend(relevant_json_scalars(child, path))

    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(relevant_json_scalars(child, f"{prefix}[{index}]"))

    elif any(
        term in prefix.lower()
        for term in ("status", "pass", "allow", "notebook", "next", "gate", "checkpoint")
    ):
        rows.append((prefix, value))

    return rows


repo_root = Path(git("rev-parse", "--show-toplevel")).resolve()
branch = git("branch", "--show-current")
head = git("rev-parse", "HEAD")
git_status = git("status", "--short")

target_notebook = repo_root / "notebooks/phase2/10_taxonomy_and_knowledge_engineering.ipynb"

accepted_dataset = (
    repo_root
    / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)
acceptance_report = (
    repo_root
    / "reports/phase2/deduplication_quality_acceptance_v1.json"
)
checkpoint = (
    repo_root
    / "artifacts/phase2/checkpoints/09_deduplication_and_quality.json"
)

print("Repository root:", repo_root)
print("Branch:", branch)
print("HEAD:", head)
print("Git working tree:", "CLEAN" if not git_status else "HAS LOCAL CHANGES")

if git_status:
    print("\nGit status:")
    print(git_status)

print("\nNotebook 10:")
print(
    "- Existing file:"
    if target_notebook.exists()
    else "- No existing Notebook 10 file found"
)
if target_notebook.exists():
    print("- Path:", target_notebook.relative_to(repo_root))
    print("- Size:", target_notebook.stat().st_size, "bytes")

required_paths = {
    "NB09 accepted dataset": accepted_dataset,
    "NB09 acceptance report": acceptance_report,
    "NB09 checkpoint": checkpoint,
}

print("\nRequired NB09 artifacts:")
for label, path in required_paths.items():
    print(f"- {label}: {'FOUND' if path.is_file() else 'MISSING'}")
    print("  Path:", path.relative_to(repo_root))

missing_required = [
    label for label, path in required_paths.items() if not path.is_file()
]

if missing_required:
    print("\nMissing required artifacts:")
    for label in missing_required:
        print("-", label)

assert not missing_required, (
    "Preflight cannot continue because required NB09 artifacts are missing."
)

dataset_records = count_jsonl(accepted_dataset)
dataset_sha256 = sha256_file(accepted_dataset)
acceptance_sha256 = sha256_file(acceptance_report)

with acceptance_report.open("r", encoding="utf-8") as handle:
    acceptance_data = json.load(handle)

with checkpoint.open("r", encoding="utf-8") as handle:
    checkpoint_data = json.load(handle)

print("\nNB09 accepted dataset validation:")
print("- Records:", dataset_records)
print("- Expected records:", EXPECTED_DATASET_RECORDS)
print("- SHA-256:", dataset_sha256)
print("- Expected SHA-256:", EXPECTED_DATASET_SHA256)
print(
    "- Record count match:",
    "PASS" if dataset_records == EXPECTED_DATASET_RECORDS else "FAIL",
)
print(
    "- Dataset hash match:",
    "PASS" if dataset_sha256 == EXPECTED_DATASET_SHA256 else "FAIL",
)

print("\nNB09 acceptance report:")
print("- JSON parse: PASS")
print("- SHA-256:", acceptance_sha256)
print("- Expected SHA-256:", EXPECTED_ACCEPTANCE_SHA256)
print(
    "- Acceptance hash match:",
    "PASS" if acceptance_sha256 == EXPECTED_ACCEPTANCE_SHA256 else "FAIL",
)

print("\nRelevant acceptance values:")
acceptance_scalars = relevant_json_scalars(acceptance_data)
for key, value in acceptance_scalars[:40]:
    print(f"- {key}: {value}")
if len(acceptance_scalars) > 40:
    print("- Additional relevant values:", len(acceptance_scalars) - 40)

print("\nRelevant checkpoint values:")
checkpoint_scalars = relevant_json_scalars(checkpoint_data)
for key, value in checkpoint_scalars[:40]:
    print(f"- {key}: {value}")
if len(checkpoint_scalars) > 40:
    print("- Additional relevant values:", len(checkpoint_scalars) - 40)

skip_dirs = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    "node_modules",
    "raw",
    "embeddings",
    "index",
}

taxonomy_terms = (
    "taxonomy",
    "ontology",
    "competency",
    "knowledge_engineering",
    "knowledge-engineering",
)

taxonomy_files = []

for root, dirs, files in os.walk(repo_root):
    dirs[:] = [name for name in dirs if name not in skip_dirs]

    for filename in files:
        relative = (Path(root) / filename).relative_to(repo_root)
        normalized_name = str(relative).lower()

        if any(term in normalized_name for term in taxonomy_terms):
            taxonomy_files.append(relative)

taxonomy_files = sorted(set(taxonomy_files))

print("\nExisting taxonomy / knowledge-engineering files:")
if taxonomy_files:
    for path in taxonomy_files[:100]:
        print("-", path)
    if len(taxonomy_files) > 100:
        print("- Additional matches:", len(taxonomy_files) - 100)
else:
    print("- None discovered by filename")

candidate_schema_roots = [
    repo_root / "schemas",
    repo_root / "schemas/phase2",
]

schema_files = sorted(
    {
        path.relative_to(repo_root)
        for root in candidate_schema_roots
        if root.exists()
        for path in root.rglob("*")
        if path.is_file()
    }
)

manifest_roots = [
    repo_root / "data/phase2/manifests",
]

manifest_files = sorted(
    {
        path.relative_to(repo_root)
        for root in manifest_roots
        if root.exists()
        for path in root.rglob("*")
        if path.is_file()
    }
)

print("\nExisting schemas:")
if schema_files:
    for path in schema_files[:80]:
        print("-", path)
    if len(schema_files) > 80:
        print("- Additional schema files:", len(schema_files) - 80)
else:
    print("- None found")

print("\nExisting manifests:")
if manifest_files:
    for path in manifest_files[:80]:
        print("-", path)
    if len(manifest_files) > 80:
        print("- Additional manifest files:", len(manifest_files) - 80)
else:
    print("- None found")

roadmap_name_terms = ("roadmap", "execution_plan", "execution-plan", "handoff")
roadmap_files = []

for root, dirs, files in os.walk(repo_root):
    dirs[:] = [name for name in dirs if name not in skip_dirs]

    for filename in files:
        lower_name = filename.lower()
        if any(term in lower_name for term in roadmap_name_terms):
            path = Path(root) / filename
            if path.suffix.lower() in {".md", ".txt", ".json"}:
                roadmap_files.append(path)

roadmap_files = sorted(set(roadmap_files))

roadmap_mentions = []

for path in roadmap_files:
    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue

    if (
        "10_taxonomy_and_knowledge_engineering" in text
        or "Notebook 10" in text
        or "notebook 10" in text
    ):
        roadmap_mentions.append(path.relative_to(repo_root))

print("\nRepository roadmap / handoff files mentioning Notebook 10:")
if roadmap_mentions:
    for path in roadmap_mentions:
        print("-", path)
else:
    print("- No repository-local roadmap mention discovered")

checkpoint_has_pass = any(
    isinstance(value, str) and value.upper() == "PASS"
    and "status" in key.lower()
    for key, value in checkpoint_scalars
)

assert branch == EXPECTED_BRANCH, (
    f"Expected branch {EXPECTED_BRANCH!r}, found {branch!r}."
)
assert dataset_records == EXPECTED_DATASET_RECORDS, (
    "NB09 accepted dataset record count does not match the accepted handoff."
)
assert dataset_sha256 == EXPECTED_DATASET_SHA256, (
    "NB09 accepted dataset hash does not match the accepted handoff."
)
assert acceptance_sha256 == EXPECTED_ACCEPTANCE_SHA256, (
    "NB09 acceptance report hash does not match the accepted handoff."
)
assert checkpoint_has_pass, (
    "A PASS status was not found in the parsed NB09 checkpoint."
)

print("\nCODE CELL 01 GATE: PASS")
print("Notebook 10 read-only preflight completed successfully.")

Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Branch: rag-phase2
HEAD: 08f8d00a0c3c14ce87dc7ca4be9773842c138120
Git working tree: HAS LOCAL CHANGES

Git status:
M 10_taxonomy_and_knowledge_engineering.ipynb
?? ../../artifacts/

Notebook 10:
- Existing file:
- Path: notebooks/phase2/10_taxonomy_and_knowledge_engineering.ipynb
- Size: 2051723 bytes

Required NB09 artifacts:
- NB09 accepted dataset: FOUND
  Path: data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl
- NB09 acceptance report: FOUND
  Path: reports/phase2/deduplication_quality_acceptance_v1.json
- NB09 checkpoint: FOUND
  Path: artifacts/phase2/checkpoints/09_deduplication_and_quality.json

NB09 accepted dataset validation:
- Records: 19456
- Expected records: 19456
- SHA-256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
- Expected SHA-256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
- Record count match: PASS
- Dataset hash matc

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery ساختار داده و ظرفیت واقعی برای Taxonomy</h3>

<p>
پیش از تعریف Topic، Competency یا هر نوع برچسب، باید ساختار واقعی Dataset پذیرفته‌شده بررسی شود.
Taxonomy نباید بر اساس حدس یا یک Schema فرضی طراحی شود.
</p>

<p>
این مرحله فقط خواندنی است و بررسی می‌کند:
</p>

<ul>
<li>Schema رسمی Document چه فیلدهایی دارد.</li>
<li>رکوردهای واقعی Dataset چه ساختاری دارند.</li>
<li>Source Typeها و توزیع آن‌ها چیست.</li>
<li>Metadata مرتبط با Attribution، Authority، Speaker، Author، Locator و Lineage چه پوششی دارد.</li>
<li>آیا Topic، Tag، Category یا Metadata موضوعی از مراحل قبلی وجود دارد که بتوان از آن به‌عنوان Evidence اولیه استفاده کرد.</li>
<li>Roadmap موجود در Repository دقیقاً چه Scope و خروجی‌ای برای Notebook 10 تعیین کرده است.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>Schema و Dataset بدون خطا قابل خواندن باشند.</li>
<li>Field inventory واقعی استخراج شود.</li>
<li>Metadata موضوعی موجود یا نبود آن مشخص شود.</li>
<li>هیچ Taxonomy جدیدی در این مرحله ساخته نشود.</li>
</ul>

</div>

In [45]:
# Code Cell 02

from collections import Counter, defaultdict
from pathlib import Path
import json


schema_path = repo_root / "schemas/phase2/document.schema.json"
dataset_path = (
    repo_root
    / "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)
roadmap_path = (
    repo_root
    / "data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md"
)

assert schema_path.is_file(), f"Missing schema: {schema_path}"
assert dataset_path.is_file(), f"Missing dataset: {dataset_path}"
assert roadmap_path.is_file(), f"Missing roadmap: {roadmap_path}"


with schema_path.open("r", encoding="utf-8") as handle:
    document_schema = json.load(handle)

schema_properties = document_schema.get("properties", {})
schema_required = document_schema.get("required", [])

print("Document schema:")
print("- Properties:", len(schema_properties))
print("- Required fields:", len(schema_required))

print("\nSchema fields:")
for field_name in sorted(schema_properties):
    required_marker = "REQUIRED" if field_name in schema_required else "optional"
    print(f"- {field_name}: {required_marker}")


records = []

with dataset_path.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        record = json.loads(line)
        records.append(record)

print("\nDataset:")
print("- Records loaded:", len(records))


top_level_fields = Counter()
nested_fields = defaultdict(Counter)

for record in records:
    for key, value in record.items():
        top_level_fields[key] += 1

        if isinstance(value, dict):
            for nested_key in value:
                nested_fields[key][nested_key] += 1


print("\nTop-level field coverage:")
for field_name, count in sorted(
    top_level_fields.items(),
    key=lambda item: (-item[1], item[0]),
):
    print(
        f"- {field_name}: "
        f"{count}/{len(records)} "
        f"({count / len(records):.1%})"
    )


print("\nNested object fields:")
for parent_name in sorted(nested_fields):
    print(f"\n[{parent_name}]")
    for child_name, count in sorted(
        nested_fields[parent_name].items(),
        key=lambda item: (-item[1], item[0]),
    ):
        print(
            f"- {child_name}: "
            f"{count}/{len(records)} "
            f"({count / len(records):.1%})"
        )


def get_nested(record, dotted_path):
    current = record

    for part in dotted_path.split("."):
        if not isinstance(current, dict):
            return None
        current = current.get(part)

    return current


candidate_fields = [
    "source_type",
    "content_origin",
    "knowledge_owner",
    "author",
    "speaker",
    "title",
    "tags",
    "topics",
    "category",
    "categories",
    "program",
    "attribution.speaker",
    "attribution.author",
    "attribution.knowledge_owner",
    "attribution.content_origin",
    "metadata.tags",
    "metadata.topic",
    "metadata.topics",
    "metadata.category",
    "metadata.categories",
    "metadata.program",
]

print("\nCandidate taxonomy / authority metadata coverage:")

for field_path in candidate_fields:
    values = [
        get_nested(record, field_path)
        for record in records
    ]

    present_values = [
        value
        for value in values
        if value not in (None, "", [], {})
    ]

    if not present_values:
        continue

    print(
        f"- {field_path}: "
        f"{len(present_values)}/{len(records)} "
        f"({len(present_values) / len(records):.1%})"
    )


source_type_counter = Counter(
    record.get("source_type")
    for record in records
)

print("\nSource type distribution:")
for source_type, count in sorted(
    source_type_counter.items(),
    key=lambda item: (-item[1], str(item[0])),
):
    print(f"- {source_type}: {count}")


metadata_value_fields = [
    "content_origin",
    "knowledge_owner",
    "author",
    "speaker",
    "attribution.speaker",
    "attribution.author",
    "attribution.knowledge_owner",
    "attribution.content_origin",
]

print("\nSelected metadata value distributions:")

for field_path in metadata_value_fields:
    counter = Counter()

    for record in records:
        value = get_nested(record, field_path)

        if value in (None, "", [], {}):
            continue

        if isinstance(value, list):
            for item in value:
                counter[str(item)] += 1
        else:
            counter[str(value)] += 1

    if not counter:
        continue

    print(f"\n[{field_path}]")
    for value, count in counter.most_common(20):
        print(f"- {value}: {count}")

    if len(counter) > 20:
        print("- Additional distinct values:", len(counter) - 20)


topic_like_paths = []

search_terms = (
    "topic",
    "tag",
    "category",
    "subject",
    "competency",
    "skill",
    "theme",
)

def discover_paths(value, prefix=""):
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key

            if any(term in key.lower() for term in search_terms):
                topic_like_paths.append(path)

            discover_paths(child, path)

    elif isinstance(value, list):
        for child in value[:3]:
            discover_paths(child, prefix)


for record in records[:1000]:
    discover_paths(record)

topic_like_paths = sorted(set(topic_like_paths))

print("\nTopic-like fields discovered in first 1,000 records:")
if topic_like_paths:
    for path in topic_like_paths:
        print("-", path)
else:
    print("- None discovered")


print("\nRepresentative record keys by source type:")

seen_source_types = set()

for record in records:
    source_type = record.get("source_type")

    if source_type in seen_source_types:
        continue

    seen_source_types.add(source_type)

    print(f"\n[{source_type}]")
    print("- Top-level keys:", sorted(record.keys()))

    for key, value in record.items():
        if isinstance(value, dict):
            print(f"- {key} keys:", sorted(value.keys()))

    if len(seen_source_types) == len(source_type_counter):
        break


roadmap_text = roadmap_path.read_text(encoding="utf-8")
roadmap_lines = roadmap_text.splitlines()

start_index = None
end_index = None

for index, line in enumerate(roadmap_lines):
    if "Notebook 10" in line and "10_taxonomy_and_knowledge_engineering" in line:
        start_index = index
        break

if start_index is not None:
    for index in range(start_index + 1, len(roadmap_lines)):
        line = roadmap_lines[index]

        if line.startswith("## Notebook 11"):
            end_index = index
            break

if start_index is None:
    print("\nNotebook 10 roadmap section:")
    print("- NOT FOUND")
else:
    if end_index is None:
        end_index = min(start_index + 80, len(roadmap_lines))

    section = "\n".join(
        roadmap_lines[start_index:end_index]
    ).strip()

    print("\nNotebook 10 roadmap section:")
    print(section)


assert len(records) == 19456, (
    "Unexpected accepted dataset record count."
)

assert len(source_type_counter) > 0, (
    "No source_type values were discovered."
)

assert start_index is not None, (
    "Notebook 10 scope was not found in repository roadmap."
)

print("\nCODE CELL 02 GATE: PASS")
print("Dataset and taxonomy-input discovery completed successfully.")

Document schema:
- Properties: 15
- Required fields: 15

Schema fields:
- attribution: REQUIRED
- content_origin: REQUIRED
- document_id: REQUIRED
- language: REQUIRED
- lineage: REQUIRED
- locator: REQUIRED
- raw_content_hash: REQUIRED
- rights: REQUIRED
- schema_version: REQUIRED
- source_id: REQUIRED
- source_metadata: REQUIRED
- source_type: REQUIRED
- text: REQUIRED
- text_sha256: REQUIRED
- title: REQUIRED

Dataset:
- Records loaded: 19456

Top-level field coverage:
- attribution: 19456/19456 (100.0%)
- content_origin: 19456/19456 (100.0%)
- document_id: 19456/19456 (100.0%)
- language: 19456/19456 (100.0%)
- lineage: 19456/19456 (100.0%)
- locator: 19456/19456 (100.0%)
- raw_content_hash: 19456/19456 (100.0%)
- rights: 19456/19456 (100.0%)
- schema_version: 19456/19456 (100.0%)
- source_id: 19456/19456 (100.0%)
- source_metadata: 19456/19456 (100.0%)
- source_type: 19456/19456 (100.0%)
- text: 19456/19456 (100.0%)
- text_sha256: 19456/19456 (100.0%)
- title: 19456/19456 (100.0%)

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery شواهد اولیه برای طراحی Taxonomy</h3>

<p>
Dataset دارای Topic آماده و یکپارچه برای هر سه Source نیست.
Website دارای Tag است، اما Book و Transcript بیشتر از ساختارهایی مانند Heading، Work، Program و Title استفاده می‌کنند.
</p>

<p>
قبل از تعریف Taxonomy باید این شواهد واقعی استخراج و مقایسه شوند تا مشخص شود Taxonomy نسخه اول بر چه پایه‌ای می‌تواند ساخته شود.
</p>

<p>
این مرحله هیچ Topic، Competency یا Label جدیدی ایجاد نمی‌کند و فقط Evidence موجود را Inventory می‌کند.
</p>

<h4>موارد بررسی</h4>

<ul>
<li>ساختار و توزیع Tagهای Website.</li>
<li>Workها و Headingهای Book.</li>
<li>Programها و Titleهای Transcript.</li>
<li>توزیع <code>content_authority_type</code> و Attribution برای جلوگیری از مخلوط‌شدن Authority با Topic.</li>
<li>میزان پوشش ساختاری که می‌تواند Seed اولیه Taxonomy باشد.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>شواهد موضوعی هر سه Source بدون تغییر Dataset استخراج شوند.</li>
<li>Website Tagها، Book Headingها و Transcript Programها قابل مشاهده باشند.</li>
<li>هیچ Label جدید یا Topic حدسی ساخته نشود.</li>
</ul>

</div>

In [46]:
# Code Cell 03

from collections import Counter, defaultdict
import json


def nonempty(value):
    return value not in (None, "", [], {})


def normalize_scalar(value):
    if value is None:
        return None
    return str(value).strip()


def flatten_tag(tag):
    if isinstance(tag, str):
        value = tag.strip()
        return value if value else None

    if isinstance(tag, dict):
        preferred_keys = (
            "name",
            "title",
            "label",
            "tag_name",
            "tag_title",
            "slug",
            "tag_id",
            "id",
        )

        parts = []

        for key in preferred_keys:
            value = tag.get(key)

            if nonempty(value):
                parts.append(f"{key}={value}")

        if parts:
            return " | ".join(parts)

        return json.dumps(
            tag,
            ensure_ascii=False,
            sort_keys=True,
        )

    return str(tag)


def normalize_heading_path(value):
    if isinstance(value, list):
        parts = [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]
        return tuple(parts)

    if isinstance(value, str):
        value = value.strip()

        if not value:
            return tuple()

        return (value,)

    return tuple()


website_records = [
    record for record in records
    if record.get("source_type") == "website"
]

book_records = [
    record for record in records
    if record.get("source_type") == "book"
]

transcript_records = [
    record for record in records
    if record.get("source_type") == "transcript"
]

print("Source populations:")
print("- Website:", len(website_records))
print("- Book:", len(book_records))
print("- Transcript:", len(transcript_records))


website_tag_counter = Counter()
website_tag_shape_counter = Counter()
website_records_with_tags = 0

for record in website_records:
    tags = record.get("source_metadata", {}).get("tags")

    if not nonempty(tags):
        continue

    website_records_with_tags += 1
    website_tag_shape_counter[type(tags).__name__] += 1

    if isinstance(tags, list):
        tag_items = tags
    else:
        tag_items = [tags]

    for tag in tag_items:
        flattened = flatten_tag(tag)

        if flattened:
            website_tag_counter[flattened] += 1


print("\nWebsite tag evidence:")
print(
    "- Records with tags:",
    f"{website_records_with_tags}/{len(website_records)}"
)
print("- Distinct flattened tags:", len(website_tag_counter))
print("- Raw tag container shapes:", dict(website_tag_shape_counter))

print("\nMost common Website tags:")
for tag, count in website_tag_counter.most_common(50):
    print(f"- {tag}: {count}")

if len(website_tag_counter) > 50:
    print("- Additional distinct tags:", len(website_tag_counter) - 50)


website_title_counter = Counter(
    normalize_scalar(record.get("title"))
    for record in website_records
    if nonempty(record.get("title"))
)

print("\nRepresentative Website titles:")
for title, count in website_title_counter.most_common(30):
    print(f"- {title}: {count}")


book_work_counter = Counter()
book_heading_path_counter = Counter()
book_heading_component_counter = Counter()
book_heading_records = 0
book_heading_text_counter = Counter()

for record in book_records:
    locator = record.get("locator", {})
    metadata = record.get("source_metadata", {})

    work_id = (
        locator.get("work_id")
        or metadata.get("work_id")
    )

    if nonempty(work_id):
        book_work_counter[str(work_id)] += 1

    heading_path = (
        locator.get("heading_path")
        or metadata.get("heading_path")
    )

    normalized_path = normalize_heading_path(heading_path)

    if normalized_path:
        book_heading_records += 1
        book_heading_path_counter[normalized_path] += 1

        for component in normalized_path:
            book_heading_component_counter[component] += 1

    if metadata.get("is_heading") is True:
        heading_text = normalize_scalar(record.get("text"))

        if heading_text:
            book_heading_text_counter[heading_text] += 1


print("\nBook structural evidence:")
print("- Distinct Work IDs:", len(book_work_counter))

for work_id, count in book_work_counter.most_common():
    print(f"- Work {work_id}: {count} records")

print(
    "- Records with heading path:",
    f"{book_heading_records}/{len(book_records)}"
)
print("- Distinct heading paths:", len(book_heading_path_counter))
print("- Distinct heading components:", len(book_heading_component_counter))
print("- Explicit heading records:", sum(book_heading_text_counter.values()))

print("\nMost common Book heading components:")
for heading, count in book_heading_component_counter.most_common(60):
    print(f"- {heading}: {count}")

if len(book_heading_component_counter) > 60:
    print(
        "- Additional heading components:",
        len(book_heading_component_counter) - 60,
    )

print("\nExplicit Book heading texts:")
for heading, count in book_heading_text_counter.most_common(60):
    print(f"- {heading}: {count}")

if len(book_heading_text_counter) > 60:
    print(
        "- Additional explicit headings:",
        len(book_heading_text_counter) - 60,
    )


transcript_program_counter = Counter()
transcript_title_counter = Counter()
transcript_program_title_pairs = Counter()

for record in transcript_records:
    metadata = record.get("source_metadata", {})

    program = (
        metadata.get("program")
        or metadata.get("program_id")
        or record.get("locator", {}).get("program_id")
    )

    title = normalize_scalar(record.get("title"))

    if nonempty(program):
        transcript_program_counter[str(program)] += 1

    if title:
        transcript_title_counter[title] += 1

    if nonempty(program) and title:
        transcript_program_title_pairs[
            (str(program), title)
        ] += 1


print("\nTranscript structural evidence:")
print("- Distinct programs:", len(transcript_program_counter))
print("- Distinct titles:", len(transcript_title_counter))

print("\nTranscript programs:")
for program, count in transcript_program_counter.most_common():
    print(f"- {program}: {count}")

print("\nMost common Transcript titles:")
for title, count in transcript_title_counter.most_common(60):
    print(f"- {title}: {count}")

if len(transcript_title_counter) > 60:
    print(
        "- Additional transcript titles:",
        len(transcript_title_counter) - 60,
    )


authority_counter = Counter()
authority_by_source = defaultdict(Counter)

for record in records:
    source_type = record.get("source_type")
    authority_type = record.get("attribution", {}).get(
        "content_authority_type"
    )

    normalized_authority = (
        str(authority_type)
        if nonempty(authority_type)
        else "<null>"
    )

    authority_counter[normalized_authority] += 1
    authority_by_source[source_type][normalized_authority] += 1


print("\nContent authority type distribution:")
for authority_type, count in authority_counter.most_common():
    print(f"- {authority_type}: {count}")

print("\nContent authority type by source:")
for source_type in sorted(authority_by_source):
    print(f"\n[{source_type}]")

    for authority_type, count in authority_by_source[
        source_type
    ].most_common():
        print(f"- {authority_type}: {count}")


speaker_label_counter = Counter()

for record in transcript_records:
    speaker_label = record.get(
        "source_metadata",
        {},
    ).get("speaker")

    if nonempty(speaker_label):
        speaker_label_counter[str(speaker_label)] += 1


print("\nTranscript source speaker labels:")
print("- Distinct labels:", len(speaker_label_counter))

for speaker, count in speaker_label_counter.most_common(40):
    print(f"- {speaker}: {count}")

if len(speaker_label_counter) > 40:
    print(
        "- Additional speaker labels:",
        len(speaker_label_counter) - 40,
    )


seed_evidence_summary = {
    "website": {
        "records": len(website_records),
        "records_with_tags": website_records_with_tags,
        "distinct_tags": len(website_tag_counter),
    },
    "book": {
        "records": len(book_records),
        "distinct_work_ids": len(book_work_counter),
        "records_with_heading_path": book_heading_records,
        "distinct_heading_paths": len(book_heading_path_counter),
        "distinct_heading_components": len(
            book_heading_component_counter
        ),
        "explicit_heading_texts": len(book_heading_text_counter),
    },
    "transcript": {
        "records": len(transcript_records),
        "distinct_programs": len(transcript_program_counter),
        "distinct_titles": len(transcript_title_counter),
        "distinct_source_speaker_labels": len(speaker_label_counter),
    },
}

print("\nTaxonomy seed evidence summary:")
print(
    json.dumps(
        seed_evidence_summary,
        ensure_ascii=False,
        indent=2,
    )
)

assert website_records_with_tags > 0, (
    "No Website tag evidence was discovered."
)

assert len(book_work_counter) > 0, (
    "No Book work identity was discovered."
)

assert (
    len(book_heading_path_counter) > 0
    or len(book_heading_text_counter) > 0
), "No Book structural heading evidence was discovered."

assert len(transcript_program_counter) > 0, (
    "No Transcript program evidence was discovered."
)

assert len(transcript_title_counter) > 0, (
    "No Transcript title evidence was discovered."
)

print("\nCODE CELL 03 GATE: PASS")
print("Corpus-derived taxonomy seed evidence discovery completed successfully.")

Source populations:
- Website: 529
- Book: 2701
- Transcript: 16226

Website tag evidence:
- Records with tags: 525/529
- Distinct flattened tags: 86
- Raw tag container shapes: {'list': 525}

Most common Website tags:
- name=سواد مالی در جهان | tag_id=55: 104
- name=محصول آکادمی | tag_id=54: 78
- name=والدین بخوانند | tag_id=1031: 77
- name=معرفی کتاب‌های آکادمی | tag_id=2059: 74
- name=اهمیت سواد مالی | tag_id=5: 54
- name=تصمیم‌گیری مالی | tag_id=48: 53
- name=سیاست‌گذاران بخوانند | tag_id=1029: 51
- name=سواد مالی خانم‌ها | tag_id=1035: 47
- name=مربیان بخوانند | tag_id=1028: 39
- name=برنامه رادیویی چهلستون | tag_id=2045: 39
- name=معرفی کتاب | tag_id=52: 36
- name=برنامه‌ریزی مالی | tag_id=20: 34
- name=گفتگوهای اختصاصی | tag_id=2049: 33
- name=پس‌انداز | tag_id=23: 33
- name=آموزش سواد مالی | tag_id=53: 31
- name=بودجه‌بندی | tag_id=29: 30
- name=برنامه تلویزیونی عصر شیرین | tag_id=2047: 26
- name=تحلیل فیلم | tag_id=36: 23
- name=آموزه‌های اسلامی | tag_id=1030: 23
- name=اقتصاد

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی Cross-Source واژگان اولیه Taxonomy</h3>

<p>
Website Tagها و Headingهای کتاب می‌توانند Seed اولیه برای مهندسی دانش باشند، اما قبل از تعریف Taxonomy باید بررسی شود کدام مفاهیم واقعاً در بخش‌های مختلف Corpus حضور دارند.
</p>

<p>
در Transcriptها Metadata موضوعی مستقیم وجود ندارد؛ بنابراین نام Program نباید به‌عنوان Topic استفاده شود.
در این مرحله فقط بررسی می‌کنیم مفاهیم مشاهده‌شده در Website و Book چه میزان Evidence متنی در Website، Book و Transcript دارند.
</p>

<p>
این مرحله هنوز Taxonomy نهایی ایجاد نمی‌کند و هیچ Documentی را برچسب‌گذاری نمی‌کند.
هدف فقط ساخت یک Vocabulary Evidence Profile قابل اندازه‌گیری است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>نام واقعی Website Tagها استخراج شود.</li>
<li>Headingهای واقعی Book استخراج شوند.</li>
<li>واژگان مشترک و مستقل مشخص شوند.</li>
<li>برای Candidateهای قابل بررسی، پوشش متنی در هر سه Source اندازه‌گیری شود.</li>
<li>هیچ Program name، Speaker یا Authority field به‌عنوان Topic خودکار ثبت نشود.</li>
</ul>

</div>

In [47]:
# Code Cell 04

from collections import Counter
import re
import unicodedata


def normalize_fa_for_match(value):
    if value is None:
        return ""

    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\u200c", " ")
    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def website_tag_name(tag):
    if isinstance(tag, str):
        return tag.strip() or None

    if isinstance(tag, dict):
        for key in ("name", "title", "label", "tag_name", "tag_title"):
            value = tag.get(key)
            if nonempty(value):
                return str(value).strip()

    return None


website_tag_names = Counter()

for record in website_records:
    tags = record.get("source_metadata", {}).get("tags")

    if not nonempty(tags):
        continue

    if not isinstance(tags, list):
        tags = [tags]

    for tag in tags:
        name = website_tag_name(tag)

        if name:
            website_tag_names[name] += 1


explicit_book_headings = Counter()

for record in book_records:
    metadata = record.get("source_metadata", {})

    if metadata.get("is_heading") is True:
        heading = str(record.get("text", "")).strip()

        if heading:
            explicit_book_headings[heading] += 1


def strip_book_structure_prefix(text):
    value = normalize_fa_for_match(text)

    patterns = [
        r"^فصل\s*[۰-۹0-9]+\s*[:\-–—]?\s*",
        r"^درس\s*[۰-۹0-9]+\s*[:\-–—]?\s*",
        r"^کاربرگ\s*[۰-۹0-9]+\s*[:\-–—]?\s*",
        r"^خود\s*ارزیابی\s*[۰-۹0-9]+\s*[:\-–—]?\s*",
        r"^خودارزیابی\s*[۰-۹0-9]+\s*[:\-–—]?\s*",
    ]

    for pattern in patterns:
        value = re.sub(pattern, "", value).strip()

    return value


website_seed_terms = {
    normalize_fa_for_match(name): {
        "display": name,
        "website_tag_count": count,
    }
    for name, count in website_tag_names.items()
    if normalize_fa_for_match(name)
}


book_seed_terms = {}

for heading, count in explicit_book_headings.items():
    normalized = strip_book_structure_prefix(heading)

    if not normalized:
        continue

    if normalized not in book_seed_terms:
        book_seed_terms[normalized] = {
            "display": heading,
            "book_heading_count": 0,
        }

    book_seed_terms[normalized]["book_heading_count"] += count


shared_exact_terms = sorted(
    set(website_seed_terms)
    & set(book_seed_terms)
)


print("Seed vocabulary inventory:")
print("- Website tag names:", len(website_seed_terms))
print("- Explicit Book heading terms:", len(book_seed_terms))
print("- Exact normalized overlap:", len(shared_exact_terms))


print("\nExact Website ↔ Book overlaps:")
if shared_exact_terms:
    for term in shared_exact_terms:
        print(
            "-",
            term,
            "| website tag count:",
            website_seed_terms[term]["website_tag_count"],
            "| book heading count:",
            book_seed_terms[term]["book_heading_count"],
        )
else:
    print("- None")


candidate_terms = {}

for term, info in website_seed_terms.items():
    candidate_terms.setdefault(
        term,
        {
            "display_terms": set(),
            "website_tag_count": 0,
            "book_heading_count": 0,
        },
    )

    candidate_terms[term]["display_terms"].add(info["display"])
    candidate_terms[term]["website_tag_count"] += info[
        "website_tag_count"
    ]


for term, info in book_seed_terms.items():
    candidate_terms.setdefault(
        term,
        {
            "display_terms": set(),
            "website_tag_count": 0,
            "book_heading_count": 0,
        },
    )

    candidate_terms[term]["display_terms"].add(info["display"])
    candidate_terms[term]["book_heading_count"] += info[
        "book_heading_count"
    ]


source_texts = {
    "website": [
        normalize_fa_for_match(record.get("text"))
        for record in website_records
    ],
    "book": [
        normalize_fa_for_match(record.get("text"))
        for record in book_records
    ],
    "transcript": [
        normalize_fa_for_match(record.get("text"))
        for record in transcript_records
    ],
}


def document_match_count(term, texts):
    if len(term) < 3:
        return 0

    return sum(
        1
        for text in texts
        if term in text
    )


evidence_rows = []

for term, info in candidate_terms.items():
    if len(term) < 3:
        continue

    website_docs = document_match_count(
        term,
        source_texts["website"],
    )
    book_docs = document_match_count(
        term,
        source_texts["book"],
    )
    transcript_docs = document_match_count(
        term,
        source_texts["transcript"],
    )

    source_presence = sum(
        count > 0
        for count in (
            website_docs,
            book_docs,
            transcript_docs,
        )
    )

    evidence_rows.append(
        {
            "term": term,
            "display": " | ".join(
                sorted(info["display_terms"])
            ),
            "website_tag_count": info[
                "website_tag_count"
            ],
            "book_heading_count": info[
                "book_heading_count"
            ],
            "website_docs": website_docs,
            "book_docs": book_docs,
            "transcript_docs": transcript_docs,
            "source_presence": source_presence,
            "total_docs": (
                website_docs
                + book_docs
                + transcript_docs
            ),
        }
    )


evidence_rows.sort(
    key=lambda row: (
        -row["source_presence"],
        -row["total_docs"],
        -row["website_tag_count"],
        -row["book_heading_count"],
        row["term"],
    )
)


presence_distribution = Counter(
    row["source_presence"]
    for row in evidence_rows
)

print("\nCross-source presence distribution:")
for source_count in sorted(presence_distribution):
    print(
        f"- Present in {source_count} source type(s):",
        presence_distribution[source_count],
    )


three_source_rows = [
    row
    for row in evidence_rows
    if row["source_presence"] == 3
]

two_source_rows = [
    row
    for row in evidence_rows
    if row["source_presence"] == 2
]


print("\nCandidate terms evidenced in all three source types:")
print("- Count:", len(three_source_rows))

for row in three_source_rows[:80]:
    print(
        f"- {row['display']}"
        f" | W-tag={row['website_tag_count']}"
        f" | B-heading={row['book_heading_count']}"
        f" | W-docs={row['website_docs']}"
        f" | B-docs={row['book_docs']}"
        f" | T-docs={row['transcript_docs']}"
    )

if len(three_source_rows) > 80:
    print(
        "- Additional three-source candidates:",
        len(three_source_rows) - 80,
    )


print("\nHighest-evidence two-source candidates:")
print("- Count:", len(two_source_rows))

for row in two_source_rows[:60]:
    print(
        f"- {row['display']}"
        f" | W-tag={row['website_tag_count']}"
        f" | B-heading={row['book_heading_count']}"
        f" | W-docs={row['website_docs']}"
        f" | B-docs={row['book_docs']}"
        f" | T-docs={row['transcript_docs']}"
    )

if len(two_source_rows) > 60:
    print(
        "- Additional two-source candidates:",
        len(two_source_rows) - 60,
    )


print("\nTop Website-only seed evidence:")
website_ranked = sorted(
    website_seed_terms.items(),
    key=lambda item: (
        -item[1]["website_tag_count"],
        item[0],
    ),
)

for term, info in website_ranked[:60]:
    print(
        f"- {info['display']}: "
        f"{info['website_tag_count']}"
    )


print("\nTop Book heading seed evidence:")
book_ranked = sorted(
    book_seed_terms.items(),
    key=lambda item: (
        -item[1]["book_heading_count"],
        item[0],
    ),
)

for term, info in book_ranked[:60]:
    print(
        f"- {info['display']}: "
        f"{info['book_heading_count']}"
    )


assert len(website_seed_terms) > 0, (
    "No Website taxonomy seed terms were extracted."
)

assert len(book_seed_terms) > 0, (
    "No Book taxonomy seed terms were extracted."
)

assert len(evidence_rows) > 0, (
    "No cross-source evidence rows were produced."
)

assert len(three_source_rows) > 0, (
    "No candidate concept had textual evidence across all three source types."
)

print("\nCODE CELL 04 GATE: PASS")
print("Cross-source taxonomy vocabulary evidence validation completed successfully.")

Seed vocabulary inventory:
- Website tag names: 86
- Explicit Book heading terms: 154
- Exact normalized overlap: 11

Exact Website ↔ Book overlaps:
- اعتبارسنجی | website tag count: 2 | book heading count: 1
- بودجه بندی | website tag count: 30 | book heading count: 2
- تصمیم گیری مالی | website tag count: 53 | book heading count: 1
- سواد مالی در جهان | website tag count: 104 | book heading count: 1
- صندوق اضطراری | website tag count: 10 | book heading count: 1
- صندوق های سرمایه گذاری | website tag count: 9 | book heading count: 1
- مدیریت ریسک | website tag count: 18 | book heading count: 1
- مراقبت مالی | website tag count: 18 | book heading count: 1
- نیاز و خواسته | website tag count: 12 | book heading count: 1
- پس انداز | website tag count: 33 | book heading count: 1
- پول توجیبی | website tag count: 17 | book heading count: 1

Cross-source presence distribution:
- Present in 0 source type(s): 28
- Present in 1 source type(s): 102
- Present in 2 source type(s): 32
- Present i

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>طراحی اسکلت اولیه Topic Taxonomy بر پایه Evidence</h3>

<p>
شواهد مرحله قبل نشان داد که بخشی از Website Tagها واقعاً موضوع مالی هستند، اما بخشی دیگر نقش مخاطب، رسانه، مجموعه یا قالب محتوا را دارند.
بنابراین همه Tagها نباید مستقیماً به Topic تبدیل شوند.
</p>

<p>
برای ساخت نسخه اولیه، فصل‌ها و Headingهای ساختاری کتاب به‌عنوان قوی‌ترین Seed سلسله‌مراتبی استفاده می‌شوند و سپس با Tagهای Website و Evidence متنی Transcriptها اعتبارسنجی می‌شوند.
</p>

<p>
در این مرحله فقط یک Candidate Topic Hierarchy در حافظه ساخته می‌شود.
هیچ فایل، Artifact یا Label نهایی روی Documentها نوشته نمی‌شود.
</p>

<h4>هفت حوزه اصلی Candidate</h4>

<ul>
<li>مبانی و توانمندی سواد مالی</li>
<li>خرج، پس‌انداز و مدیریت پول</li>
<li>اعتبار، قرض، بدهی و بانک</li>
<li>کار، شغل و درآمد</li>
<li>سرمایه‌گذاری</li>
<li>ریسک و بیمه</li>
<li>تصمیم‌گیری مالی</li>
</ul>

<p>
این ساختار Candidate است و فقط در صورت تأیید Coverage و Evidence مراحل بعد می‌تواند وارد نسخه Freeze‌شده Taxonomy شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر Topic اصلی دارای Evidence واقعی در Corpus باشد.</li>
<li>هر Topic در Book دارای Evidence باشد.</li>
<li>Topicهای اصلی در بیش از یک Source قابل مشاهده باشند.</li>
<li>Alias تکراری بین Topicهای مختلف وجود نداشته باشد.</li>
<li>Tagهای مربوط به رسانه، مخاطب یا قالب محتوا وارد Topic hierarchy اصلی نشوند.</li>
</ul>

</div>

In [48]:
# Code Cell 05

topic_seed_v0 = {
    "financial_literacy_foundations": {
        "label_fa": "مبانی و توانمندی سواد مالی",
        "evidence_terms": [
            "مفهوم سواد مالی",
            "اهمیت سواد مالی",
            "استانداردهای سواد مالی",
            "پلکان توانمندی مالی",
            "سواد مالی در جهان",
            "آموزش سواد مالی",
        ],
    },
    "spending_saving_money_management": {
        "label_fa": "خرج، پس‌انداز و مدیریت پول",
        "evidence_terms": [
            "خرج و پس‌انداز",
            "پس‌انداز",
            "بودجه‌بندی",
            "مدیریت هزینه",
            "ثبت هزینه‌ها",
            "کاهش هزینه‌ها",
            "نیاز و خواسته",
            "خرید",
            "الگوهای خرید",
            "خرید هوشمندانه",
            "صندوق اضطراری",
            "پول توجیبی",
            "هدف پس‌اندازی",
        ],
    },
    "credit_debt_banking": {
        "label_fa": "اعتبار، قرض، بدهی و بانک",
        "evidence_terms": [
            "اعتبار و بدهی",
            "قرض",
            "بدهی",
            "قواعد قرض",
            "قرض گرفتن",
            "کارت اعتباری",
            "اعتبارسنجی",
            "بانک و اعتبار",
            "بانک و سپرده‌گذاری",
            "مدیریت بحران بدهی",
            "مزایا و معایب بدهی",
        ],
    },
    "work_income": {
        "label_fa": "کار، شغل و درآمد",
        "evidence_terms": [
            "کاریابی و درآمد",
            "یافتن شغل",
            "انتخاب شغل",
            "شغل خوب",
            "رضایت شغلی",
            "توسعه شغلی",
            "سبک‌های شغلی",
            "رویکردهای شغلی",
            "کسب درآمد",
            "کارآفرینی",
            "درآمد غیرفعال",
            "کار داوطلبانه",
        ],
    },
    "investment": {
        "label_fa": "سرمایه‌گذاری",
        "evidence_terms": [
            "سرمایه‌گذاری",
            "چرا سرمایه‌گذاری",
            "چگونگی سرمایه‌گذاری",
            "انواع سرمایه‌گذاری",
            "روش‌های سرمایه‌گذاری",
            "گزینه‌های سرمایه‌گذاری",
            "سرمایه‌گذاری مستقیم",
            "بازار سرمایه و بورس",
            "صندوق‌های سرمایه‌گذاری",
            "انواع صندوق‌های سرمایه‌گذاری",
            "ریسک و بازده سرمایه‌گذاری",
        ],
    },
    "risk_insurance": {
        "label_fa": "ریسک و بیمه",
        "evidence_terms": [
            "ریسک و بیمه",
            "مدیریت ریسک",
            "بیمه",
            "انواع بیمه",
        ],
    },
    "financial_decision_making": {
        "label_fa": "تصمیم‌گیری مالی",
        "evidence_terms": [
            "تصمیم‌گیری مالی",
            "هزینه فرصت",
            "انتخاب و هزینه‌فرصت",
            "هدف‌گذاری هوشمندانه",
            "تعادل میان حال و آینده",
            "مراقبت مالی",
            "همکاری مالی",
        ],
    },
}


def normalized_unique_terms(topic_registry):
    owners = {}

    for topic_id, topic in topic_registry.items():
        for raw_term in topic["evidence_terms"]:
            normalized = normalize_fa_for_match(raw_term)

            owners.setdefault(normalized, []).append(topic_id)

    return owners


term_owners = normalized_unique_terms(topic_seed_v0)

duplicate_aliases = {
    term: owners
    for term, owners in term_owners.items()
    if len(owners) > 1
}


print("Candidate Topic Taxonomy v0:")
print("- Top-level topics:", len(topic_seed_v0))
print("- Unique evidence terms:", len(term_owners))
print("- Cross-topic duplicate aliases:", len(duplicate_aliases))

if duplicate_aliases:
    for term, owners in sorted(duplicate_aliases.items()):
        print("-", term, "=>", owners)


topic_evidence_profile = {}

for topic_id, topic in topic_seed_v0.items():
    normalized_terms = sorted({
        normalize_fa_for_match(term)
        for term in topic["evidence_terms"]
        if normalize_fa_for_match(term)
    })

    source_counts = {}

    for source_type, texts in source_texts.items():
        matched_documents = 0

        for text in texts:
            if any(term in text for term in normalized_terms):
                matched_documents += 1

        source_counts[source_type] = matched_documents

    source_presence = sum(
        count > 0
        for count in source_counts.values()
    )

    website_tag_evidence = []

    for tag_name, count in website_tag_names.items():
        normalized_tag = normalize_fa_for_match(tag_name)

        if normalized_tag in normalized_terms:
            website_tag_evidence.append(
                (tag_name, count)
            )

    book_heading_evidence = []

    for heading, count in explicit_book_headings.items():
        normalized_heading = strip_book_structure_prefix(
            heading
        )

        if (
            normalized_heading in normalized_terms
            or any(
                term in normalized_heading
                for term in normalized_terms
            )
        ):
            book_heading_evidence.append(
                (heading, count)
            )

    topic_evidence_profile[topic_id] = {
        "label_fa": topic["label_fa"],
        "evidence_terms": normalized_terms,
        "source_counts": source_counts,
        "source_presence": source_presence,
        "website_tag_evidence": sorted(
            website_tag_evidence,
            key=lambda item: (-item[1], item[0]),
        ),
        "book_heading_evidence": sorted(
            book_heading_evidence,
            key=lambda item: (-item[1], item[0]),
        ),
    }


print("\nTopic evidence profile:")

for topic_id, profile in topic_evidence_profile.items():
    counts = profile["source_counts"]

    print(f"\n[{topic_id}]")
    print("- Label:", profile["label_fa"])
    print("- Evidence terms:", len(profile["evidence_terms"]))
    print("- Website documents:", counts["website"])
    print("- Book documents:", counts["book"])
    print("- Transcript documents:", counts["transcript"])
    print(
        "- Source-type coverage:",
        f"{profile['source_presence']}/3",
    )

    print("- Website tag evidence:")

    if profile["website_tag_evidence"]:
        for name, count in profile[
            "website_tag_evidence"
        ]:
            print(f"  - {name}: {count}")
    else:
        print("  - None")

    print("- Book heading evidence:")

    if profile["book_heading_evidence"]:
        for heading, count in profile[
            "book_heading_evidence"
        ][:20]:
            print(f"  - {heading}: {count}")

        if len(profile["book_heading_evidence"]) > 20:
            print(
                "  - Additional heading evidence:",
                len(
                    profile["book_heading_evidence"]
                ) - 20,
            )
    else:
        print("  - None")


media_audience_examples = [
    "والدین بخوانند",
    "سیاست‌گذاران بخوانند",
    "مربیان بخوانند",
    "برنامه رادیویی چهلستون",
    "برنامه تلویزیونی عصر شیرین",
    "برنامه تلویزیونی خانه تو",
    "پادکست فیه ما فی",
    "گفتگوهای اختصاصی",
    "معرفی کتاب",
    "معرفی کتاب‌های آکادمی",
    "محصول آکادمی",
]

topic_alias_set = set(term_owners)

incorrect_dimension_leakage = [
    value
    for value in media_audience_examples
    if normalize_fa_for_match(value) in topic_alias_set
]


print("\nDimension separation check:")
print(
    "- Media/audience/form examples incorrectly "
    "present as Topic aliases:",
    len(incorrect_dimension_leakage),
)

for value in incorrect_dimension_leakage:
    print("-", value)


topics_without_book_evidence = []
topics_without_cross_source_evidence = []

for topic_id, profile in topic_evidence_profile.items():
    if profile["source_counts"]["book"] == 0:
        topics_without_book_evidence.append(topic_id)

    if profile["source_presence"] < 2:
        topics_without_cross_source_evidence.append(
            topic_id
        )


print("\nValidation:")
print(
    "- Topics without Book evidence:",
    len(topics_without_book_evidence),
)
print(
    "- Topics without evidence in at least 2 sources:",
    len(topics_without_cross_source_evidence),
)
print(
    "- Duplicate aliases:",
    len(duplicate_aliases),
)
print(
    "- Dimension leakage:",
    len(incorrect_dimension_leakage),
)


assert len(topic_seed_v0) == 7, (
    "Unexpected number of candidate top-level topics."
)

assert not duplicate_aliases, (
    "The same evidence alias appears under multiple topics."
)

assert not topics_without_book_evidence, (
    "At least one candidate topic has no Book evidence."
)

assert not topics_without_cross_source_evidence, (
    "At least one candidate topic lacks cross-source evidence."
)

assert not incorrect_dimension_leakage, (
    "Audience/media/content-form metadata leaked into Topic aliases."
)

print("\nCODE CELL 05 GATE: PASS")
print(
    "Evidence-backed candidate top-level Topic Taxonomy "
    "validated successfully."
)

Candidate Topic Taxonomy v0:
- Top-level topics: 7
- Unique evidence terms: 64
- Cross-topic duplicate aliases: 0

Topic evidence profile:

[financial_literacy_foundations]
- Label: مبانی و توانمندی سواد مالی
- Evidence terms: 6
- Website documents: 81
- Book documents: 24
- Transcript documents: 94
- Source-type coverage: 3/3
- Website tag evidence:
  - سواد مالی در جهان: 104
  - اهمیت سواد مالی: 54
  - آموزش سواد مالی: 31
  - مفهوم سواد مالی: 19
- Book heading evidence:
  - استانداردهای سواد مالی: 1
  - خودارزیابی 1: موقعیت من در پلکان توانمندی مالی: 1
  - درس ۲: پلکان توانمندی مالی: 1
  - درس ۲۹: سواد مالی در جهان: 1

[spending_saving_money_management]
- Label: خرج، پس‌انداز و مدیریت پول
- Evidence terms: 13
- Website documents: 286
- Book documents: 274
- Transcript documents: 1330
- Source-type coverage: 3/3
- Website tag evidence:
  - پس‌انداز: 33
  - بودجه‌بندی: 30
  - خرید: 18
  - پول توجیبی: 17
  - نیاز و خواسته: 12
  - ثبت هزینه‌ها: 10
  - صندوق اضطراری: 10
  - کاهش هزینه‌ها:

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery ترتیب آموزشی برای Competency و Prerequisite</h3>

<p>
Roadmap از این Notebook می‌خواهد علاوه بر Topic، Competency و Prerequisite نیز تعریف شوند.
اما رابطه پیش‌نیازی نباید صرفاً از تصور عمومی درباره سواد مالی ساخته شود.
</p>

<p>
کتاب‌های Corpus دارای Work، Heading، Lesson و ترتیب Paragraph هستند و می‌توانند Evidence اولیه برای ترتیب آموزشی باشند.
در این مرحله ساختار واقعی هر Work استخراج می‌شود تا مشخص شود موضوعات اصلی در چه ترتیبی آموزش داده شده‌اند و کدام Topicها در چند اثر ترتیب مشابه دارند.
</p>

<p>
این مرحله هنوز Competency یا Prerequisite نهایی تولید نمی‌کند.
فقط Evidence لازم برای تصمیم مرحله بعد را استخراج می‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>سه Work موجود مستقل بررسی شوند.</li>
<li>Headingهای دارای Lesson/Chapter به ترتیب واقعی ظاهرشدن استخراج شوند.</li>
<li>Topicهای Candidate به ساختار کتاب نگاشت شوند.</li>
<li>ترتیب Topicها برای هر Work قابل مشاهده باشد.</li>
<li>هیچ Prerequisite بدون Evidence ثبت نشود.</li>
</ul>

</div>

In [49]:
# Code Cell 06

from collections import defaultdict
import re


def first_int_from_text(text):
    if not text:
        return None

    digit_map = str.maketrans(
        "۰۱۲۳۴۵۶۷۸۹",
        "0123456789",
    )

    normalized = str(text).translate(digit_map)

    match = re.search(r"\d+", normalized)

    if not match:
        return None

    return int(match.group())


def ordered_book_records_by_work(book_records):
    grouped = defaultdict(list)

    for record in book_records:
        locator = record.get("locator", {})
        metadata = record.get("source_metadata", {})

        work_id = (
            locator.get("work_id")
            or metadata.get("work_id")
        )

        paragraph_number = (
            locator.get("paragraph_number")
            or metadata.get("paragraph_number")
        )

        grouped[str(work_id)].append(
            (
                paragraph_number,
                record,
            )
        )

    ordered = {}

    for work_id, rows in grouped.items():
        ordered[work_id] = [
            record
            for _, record in sorted(
                rows,
                key=lambda item: (
                    item[0] is None,
                    item[0] if item[0] is not None else 10**12,
                ),
            )
        ]

    return ordered


books_by_work = ordered_book_records_by_work(
    book_records
)


print("Book works discovered:")
print("- Count:", len(books_by_work))

for work_id, work_records in books_by_work.items():
    print(
        f"- {work_id}: {len(work_records)} records"
    )


topic_term_map = {
    topic_id: {
        normalize_fa_for_match(term)
        for term in topic["evidence_terms"]
        if normalize_fa_for_match(term)
    }
    for topic_id, topic in topic_seed_v0.items()
}


def match_topic_from_heading(text):
    normalized = normalize_fa_for_match(text)

    matches = []

    for topic_id, terms in topic_term_map.items():
        matched_terms = [
            term
            for term in terms
            if term in normalized
            or normalized in term
        ]

        if matched_terms:
            matches.append(
                (
                    topic_id,
                    sorted(
                        matched_terms,
                        key=len,
                        reverse=True,
                    ),
                )
            )

    return matches


work_heading_sequences = {}
work_topic_sequences = {}


for work_id, work_records in books_by_work.items():
    heading_rows = []

    for record in work_records:
        metadata = record.get(
            "source_metadata",
            {},
        )

        if metadata.get("is_heading") is not True:
            continue

        text = str(
            record.get("text", "")
        ).strip()

        if not text:
            continue

        locator = record.get("locator", {})

        paragraph_number = (
            locator.get("paragraph_number")
            or metadata.get("paragraph_number")
        )

        heading_level = metadata.get(
            "heading_level"
        )

        heading_rows.append(
            {
                "paragraph_number": paragraph_number,
                "heading_level": heading_level,
                "text": text,
                "number": first_int_from_text(text),
                "topic_matches": match_topic_from_heading(
                    text
                ),
            }
        )

    work_heading_sequences[work_id] = heading_rows


    topic_sequence = []

    for row in heading_rows:
        for topic_id, matched_terms in row[
            "topic_matches"
        ]:
            if (
                not topic_sequence
                or topic_sequence[-1]["topic_id"]
                != topic_id
            ):
                topic_sequence.append(
                    {
                        "topic_id": topic_id,
                        "label_fa": topic_seed_v0[
                            topic_id
                        ]["label_fa"],
                        "heading": row["text"],
                        "paragraph_number": row[
                            "paragraph_number"
                        ],
                        "matched_terms": matched_terms,
                    }
                )

    work_topic_sequences[
        work_id
    ] = topic_sequence


print("\nOrdered structural headings by Work:")

for work_id, rows in work_heading_sequences.items():
    print(f"\n[{work_id}]")
    print("- Explicit headings:", len(rows))

    for row in rows[:120]:
        matches = [
            topic_seed_v0[topic_id]["label_fa"]
            for topic_id, _ in row[
                "topic_matches"
            ]
        ]

        match_text = (
            " | ".join(matches)
            if matches
            else "-"
        )

        print(
            f"- P{row['paragraph_number']} "
            f"| level={row['heading_level']} "
            f"| {row['text']} "
            f"| topic={match_text}"
        )

    if len(rows) > 120:
        print(
            "- Additional headings:",
            len(rows) - 120,
        )


print("\nCandidate Topic order by Work:")

for work_id, sequence in work_topic_sequences.items():
    print(f"\n[{work_id}]")

    if not sequence:
        print("- No candidate Topic mapping")
        continue

    for position, row in enumerate(
        sequence,
        start=1,
    ):
        print(
            f"{position}. "
            f"{row['label_fa']} "
            f"| heading={row['heading']} "
            f"| paragraph={row['paragraph_number']}"
        )


pair_order_counter = defaultdict(
    lambda: {
        "forward": 0,
        "reverse": 0,
        "works": [],
    }
)


for work_id, sequence in work_topic_sequences.items():
    first_positions = {}

    for index, row in enumerate(sequence):
        topic_id = row["topic_id"]

        if topic_id not in first_positions:
            first_positions[topic_id] = index

    topic_ids = sorted(first_positions)

    for i, topic_a in enumerate(topic_ids):
        for topic_b in topic_ids[i + 1:]:
            pos_a = first_positions[topic_a]
            pos_b = first_positions[topic_b]

            pair_key = tuple(
                sorted((topic_a, topic_b))
            )

            if pos_a < pos_b:
                earlier = topic_a
                later = topic_b
            else:
                earlier = topic_b
                later = topic_a

            canonical_a, canonical_b = pair_key

            if (
                earlier == canonical_a
                and later == canonical_b
            ):
                pair_order_counter[
                    pair_key
                ]["forward"] += 1
            else:
                pair_order_counter[
                    pair_key
                ]["reverse"] += 1

            pair_order_counter[
                pair_key
            ]["works"].append(
                {
                    "work_id": work_id,
                    "earlier": earlier,
                    "later": later,
                }
            )


print("\nCross-work ordering evidence:")

consistent_pairs = []
conflicting_pairs = []

for pair_key, evidence in sorted(
    pair_order_counter.items()
):
    total = (
        evidence["forward"]
        + evidence["reverse"]
    )

    if total < 2:
        continue

    if (
        evidence["forward"] == 0
        or evidence["reverse"] == 0
    ):
        consistent_pairs.append(
            (
                pair_key,
                evidence,
            )
        )
    else:
        conflicting_pairs.append(
            (
                pair_key,
                evidence,
            )
        )


print(
    "- Consistent topic-order pairs across "
    "at least 2 Works:",
    len(consistent_pairs),
)
print(
    "- Conflicting topic-order pairs:",
    len(conflicting_pairs),
)


print("\nConsistent order evidence:")

for pair_key, evidence in consistent_pairs:
    observations = evidence["works"]

    first_observation = observations[0]

    earlier = first_observation["earlier"]
    later = first_observation["later"]

    print(
        "-",
        topic_seed_v0[earlier]["label_fa"],
        "→",
        topic_seed_v0[later]["label_fa"],
        "| Works:",
        len(observations),
    )


print("\nConflicting order evidence:")

if conflicting_pairs:
    for pair_key, evidence in conflicting_pairs:
        print(
            "- Pair:",
            topic_seed_v0[
                pair_key[0]
            ]["label_fa"],
            "<->",
            topic_seed_v0[
                pair_key[1]
            ]["label_fa"],
        )

        for observation in evidence["works"]:
            print(
                "  -",
                observation["work_id"],
                ":",
                topic_seed_v0[
                    observation["earlier"]
                ]["label_fa"],
                "→",
                topic_seed_v0[
                    observation["later"]
                ]["label_fa"],
            )
else:
    print("- None")


works_with_topic_sequence = sum(
    bool(sequence)
    for sequence in work_topic_sequences.values()
)


print("\nValidation:")
print(
    "- Works with mapped Topic sequence:",
    f"{works_with_topic_sequence}/{len(books_by_work)}",
)
print(
    "- Consistent cross-work ordering pairs:",
    len(consistent_pairs),
)
print(
    "- Conflicting ordering pairs:",
    len(conflicting_pairs),
)


assert len(books_by_work) == 3, (
    "Expected three accepted Book works."
)

assert works_with_topic_sequence >= 2, (
    "Too few Works contain usable Topic-order evidence."
)

assert len(consistent_pairs) > 0, (
    "No repeatable Topic ordering evidence was found."
)

print("\nCODE CELL 06 GATE: PASS")
print(
    "Book-derived instructional ordering "
    "evidence discovered successfully."
)

Book works discovered:
- Count: 3
- book:adult-financial-literacy-course: 1333 records
- book:financial-literacy-for-all: 873 records
- book:trainer-training-financial-literacy: 495 records

Ordered structural headings by Work:

[book:adult-financial-literacy-course]
- Explicit headings: 98
- P102 | level=1 | سخن آکادمی هوش مالی | topic=-
- P120 | level=1 | فصل ا: اهمیت و تعریف سواد مالی | topic=-
- P121 | level=1 | درس ۱: جایگاه سواد مالی در توسعهٔ فردی | topic=-
- P139 | level=1 | درس ۲: پلکان توانمندی مالی | topic=مبانی و توانمندی سواد مالی
- P162 | level=2 | خودارزیابی 1: موقعیت من در پلکان توانمندی مالی | topic=مبانی و توانمندی سواد مالی
- P165 | level=2 | کاربرگ 1: نامه‌ای به خودتان | topic=-
- P168 | level=2 | کاربرگ 2: خودآزمایی مالی | topic=-
- P208 | level=1 | فصل 2: خرج و پس‌انداز | topic=خرج، پس‌انداز و مدیریت پول
- P209 | level=1 | درس ۳: نگرانی یا مسأله مالی | topic=-
- P211 | level=2 | نگرانی مالی | topic=-
- P214 | level=2 | مسأله مالی | topic=-
- P221 | level=2 | خود ا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery شواهد Competency از خودارزیابی، کاربرگ و تمرین</h3>

<p>
ترتیب Topicها در سه Work یکسان نیست و بنابراین ترتیب ارائه محتوا به‌تنهایی برای تعریف Prerequisite قطعی کافی نیست.
</p>

<p>
برای تعریف Competency باید بررسی شود منابع آموزشی از یادگیرنده چه کاری می‌خواهند انجام دهد:
خودارزیابی کند، وضعیت مالی خود را تحلیل کند، بودجه بسازد، بدهی را بررسی کند، گزینه سرمایه‌گذاری را مقایسه کند یا تصمیمی را ارزیابی کند.
</p>

<p>
در این مرحله Headingهای مربوط به <code>Self-assessment</code>، <code>Worksheet</code> و <code>Exercise</code> استخراج می‌شوند و به نزدیک‌ترین Context آموزشی و Topic مرتبط متصل می‌شوند.
</p>

<p>
این مرحله هنوز Competency نهایی یا Prerequisite ایجاد نمی‌کند و هیچ Artifactی روی Disk نوشته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>شواهد خودارزیابی، کاربرگ و تمرین از Workهای واقعی استخراج شوند.</li>
<li>Context آموزشی هر مورد تا حد امکان از ساختار Heading بازیابی شود.</li>
<li>موارد قابل اتصال به Topic مشخص شوند.</li>
<li>موارد بدون Topic یا مبهم نیز صریحاً باقی بمانند و حدس زده نشوند.</li>
<li>هیچ رابطه Prerequisite صرفاً از ترتیب فصل‌ها ساخته نشود.</li>
</ul>

</div>

In [50]:
# Code Cell 07

from collections import Counter, defaultdict
import re


def heading_level_as_int(value):
    if value is None:
        return None

    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def evidence_activity_type(text):
    normalized = normalize_fa_for_match(text)

    if re.search(r"خود\s*ارزیابی|خودارزیابی|خود\s*آزمایی|خودآزمایی", normalized):
        return "self_assessment"

    if "کاربرگ" in normalized:
        return "worksheet"

    if "تمرین" in normalized:
        return "exercise"

    return None


competency_evidence_rows = []


for work_id, work_records in books_by_work.items():

    heading_stack = {}

    for record in work_records:
        metadata = record.get("source_metadata", {})

        if metadata.get("is_heading") is not True:
            continue

        heading_text = str(record.get("text", "")).strip()

        if not heading_text:
            continue

        level = heading_level_as_int(
            metadata.get("heading_level")
        )

        locator = record.get("locator", {})

        paragraph_number = (
            locator.get("paragraph_number")
            or metadata.get("paragraph_number")
        )

        if level is not None:
            heading_stack[level] = heading_text

            deeper_levels = [
                stack_level
                for stack_level in heading_stack
                if stack_level > level
            ]

            for stack_level in deeper_levels:
                del heading_stack[stack_level]

        activity_type = evidence_activity_type(
            heading_text
        )

        if activity_type is None:
            continue

        parent_candidates = []

        if level is not None:
            parent_candidates = [
                (
                    stack_level,
                    stack_text,
                )
                for stack_level, stack_text
                in heading_stack.items()
                if stack_level < level
            ]

        parent_candidates.sort(
            key=lambda item: item[0],
            reverse=True,
        )

        nearest_parent = (
            parent_candidates[0][1]
            if parent_candidates
            else None
        )

        topic_matches = match_topic_from_heading(
            heading_text
        )

        if not topic_matches and nearest_parent:
            topic_matches = match_topic_from_heading(
                nearest_parent
            )

        matched_topic_ids = sorted({
            topic_id
            for topic_id, _ in topic_matches
        })

        competency_evidence_rows.append(
            {
                "work_id": work_id,
                "paragraph_number": paragraph_number,
                "heading_level": level,
                "activity_type": activity_type,
                "heading": heading_text,
                "nearest_parent": nearest_parent,
                "topic_ids": matched_topic_ids,
            }
        )


print("Competency-oriented evidence inventory:")
print(
    "- Total activity headings:",
    len(competency_evidence_rows),
)


activity_type_counter = Counter(
    row["activity_type"]
    for row in competency_evidence_rows
)

print("\nActivity types:")

for activity_type, count in activity_type_counter.most_common():
    print(f"- {activity_type}: {count}")


work_activity_counter = defaultdict(Counter)

for row in competency_evidence_rows:
    work_activity_counter[
        row["work_id"]
    ][
        row["activity_type"]
    ] += 1


print("\nActivity evidence by Work:")

for work_id in sorted(work_activity_counter):
    print(f"\n[{work_id}]")

    for activity_type, count in work_activity_counter[
        work_id
    ].most_common():
        print(f"- {activity_type}: {count}")


topic_activity_counter = defaultdict(Counter)
unmapped_rows = []

for row in competency_evidence_rows:
    if not row["topic_ids"]:
        unmapped_rows.append(row)
        continue

    for topic_id in row["topic_ids"]:
        topic_activity_counter[
            topic_id
        ][
            row["activity_type"]
        ] += 1


print("\nCompetency evidence by candidate Topic:")

for topic_id in topic_seed_v0:
    counter = topic_activity_counter.get(
        topic_id,
        Counter(),
    )

    print(
        f"\n[{topic_id}] "
        f"{topic_seed_v0[topic_id]['label_fa']}"
    )

    if not counter:
        print("- No directly mapped activity evidence")
        continue

    for activity_type, count in counter.most_common():
        print(f"- {activity_type}: {count}")


print("\nDetailed competency-oriented evidence:")

for work_id in sorted(books_by_work):
    print(f"\n[{work_id}]")

    work_rows = [
        row
        for row in competency_evidence_rows
        if row["work_id"] == work_id
    ]

    for row in work_rows:
        topic_labels = [
            topic_seed_v0[topic_id]["label_fa"]
            for topic_id in row["topic_ids"]
        ]

        topic_text = (
            " | ".join(topic_labels)
            if topic_labels
            else "UNMAPPED"
        )

        parent_text = (
            row["nearest_parent"]
            if row["nearest_parent"]
            else "-"
        )

        print(
            f"- P{row['paragraph_number']} "
            f"| {row['activity_type']} "
            f"| {row['heading']} "
            f"| parent={parent_text} "
            f"| topic={topic_text}"
        )


print("\nUnmapped competency-oriented evidence:")
print("- Count:", len(unmapped_rows))

for row in unmapped_rows[:80]:
    print(
        f"- {row['work_id']} "
        f"| P{row['paragraph_number']} "
        f"| {row['activity_type']} "
        f"| {row['heading']} "
        f"| parent={row['nearest_parent'] or '-'}"
    )

if len(unmapped_rows) > 80:
    print(
        "- Additional unmapped rows:",
        len(unmapped_rows) - 80,
    )


mapped_rows = [
    row
    for row in competency_evidence_rows
    if row["topic_ids"]
]

mapping_rate = (
    len(mapped_rows)
    / len(competency_evidence_rows)
    if competency_evidence_rows
    else 0.0
)


print("\nValidation:")
print(
    "- Total activity evidence:",
    len(competency_evidence_rows),
)
print(
    "- Mapped to candidate Topic:",
    len(mapped_rows),
)
print(
    "- Unmapped:",
    len(unmapped_rows),
)
print(
    "- Mapping rate:",
    f"{mapping_rate:.1%}",
)
print(
    "- Topics with direct activity evidence:",
    len(topic_activity_counter),
    "/",
    len(topic_seed_v0),
)


assert len(competency_evidence_rows) > 0, (
    "No competency-oriented Book evidence was found."
)

assert activity_type_counter["self_assessment"] > 0, (
    "No self-assessment evidence was found."
)

assert activity_type_counter["worksheet"] > 0, (
    "No worksheet evidence was found."
)

assert activity_type_counter["exercise"] > 0, (
    "No exercise evidence was found."
)

assert len(mapped_rows) > 0, (
    "No competency evidence could be connected to candidate Topics."
)

print("\nCODE CELL 07 GATE: PASS")
print(
    "Competency-oriented instructional evidence "
    "discovered successfully."
)

Competency-oriented evidence inventory:
- Total activity headings: 48

Activity types:
- self_assessment: 28
- exercise: 11
- worksheet: 9

Activity evidence by Work:

[book:adult-financial-literacy-course]
- self_assessment: 23
- worksheet: 9

[book:trainer-training-financial-literacy]
- exercise: 11
- self_assessment: 5

Competency evidence by candidate Topic:

[financial_literacy_foundations] مبانی و توانمندی سواد مالی
- self_assessment: 2
- worksheet: 1

[spending_saving_money_management] خرج، پس‌انداز و مدیریت پول
- exercise: 6
- self_assessment: 4
- worksheet: 1

[credit_debt_banking] اعتبار، قرض، بدهی و بانک
- self_assessment: 3
- worksheet: 2

[work_income] کار، شغل و درآمد
- worksheet: 2
- self_assessment: 1

[investment] سرمایه‌گذاری
- self_assessment: 4
- worksheet: 1

[risk_insurance] ریسک و بیمه
- self_assessment: 3

[financial_decision_making] تصمیم‌گیری مالی
- self_assessment: 4
- worksheet: 2
- exercise: 1

Detailed competency-oriented evidence:

[book:adult-financial-l

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی متن واقعی Activityها برای استخراج Competency</h3>

<p>
عنوان یک خودارزیابی، کاربرگ یا تمرین به‌تنهایی برای تعریف Competency کافی نیست.
برای مثال، عنوان «نمره اعتباری من» نشان می‌دهد موضوع چیست، اما دقیقاً مشخص نمی‌کند یادگیرنده باید چه توانایی‌ای را نشان دهد.
</p>

<p>
در این مرحله متن بعد از Activity Heading از ساختار واقعی کتاب استخراج می‌شود تا بتوان در مرحله بعد Competency را بر اساس رفتار یا توانایی مشاهده‌شده تعریف کرد.
</p>

<p>
برای جلوگیری از خروجی بسیار بزرگ، از هر Topic چند نمونه نماینده و تمام موارد Unmapped نمایش داده می‌شوند.
هیچ Competency جدیدی هنوز ساخته یا ذخیره نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>متن Activityها از ساختار واقعی Work استخراج شود.</li>
<li>برای هر Topic حداقل یک Activity دارای متن قابل بررسی باشد.</li>
<li>تمام Activityهای Unmapped برای بازبینی نمایش داده شوند.</li>
<li>هیچ Topic یا Competency از روی حدس اضافه نشود.</li>
</ul>

</div>

In [51]:
# Code Cell 08

from collections import defaultdict


def paragraph_number_of(record):
    locator = record.get("locator", {})
    metadata = record.get("source_metadata", {})

    return (
        locator.get("paragraph_number")
        or metadata.get("paragraph_number")
    )


def heading_level_of(record):
    metadata = record.get("source_metadata", {})

    try:
        value = metadata.get("heading_level")
        return int(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def extract_activity_body(
    work_records,
    activity_paragraph_number,
    activity_heading_level,
):
    start_index = None

    for index, record in enumerate(work_records):
        if (
            paragraph_number_of(record)
            == activity_paragraph_number
        ):
            start_index = index
            break

    if start_index is None:
        return {
            "status": "ACTIVITY_HEADING_NOT_FOUND",
            "paragraph_count": 0,
            "text": "",
        }

    collected = []

    for record in work_records[start_index + 1:]:
        metadata = record.get("source_metadata", {})

        is_heading = (
            metadata.get("is_heading") is True
        )

        current_level = heading_level_of(record)

        if is_heading:
            if (
                activity_heading_level is None
                or current_level is None
                or current_level <= activity_heading_level
            ):
                break

        text = str(record.get("text", "")).strip()

        if text:
            collected.append(text)

    body_text = "\n".join(collected).strip()

    return {
        "status": (
            "FOUND"
            if body_text
            else "EMPTY_BODY"
        ),
        "paragraph_count": len(collected),
        "text": body_text,
    }


activity_body_records = []

for row in competency_evidence_rows:
    work_id = row["work_id"]

    body = extract_activity_body(
        books_by_work[work_id],
        row["paragraph_number"],
        row["heading_level"],
    )

    activity_body_records.append(
        {
            **row,
            "body_status": body["status"],
            "body_paragraph_count": body[
                "paragraph_count"
            ],
            "body_text": body["text"],
        }
    )


body_status_counter = Counter(
    row["body_status"]
    for row in activity_body_records
)


print("Activity body extraction:")
print(
    "- Activity records:",
    len(activity_body_records),
)

for status, count in body_status_counter.most_common():
    print(f"- {status}: {count}")


mapped_body_by_topic = defaultdict(list)

for row in activity_body_records:
    if row["body_status"] != "FOUND":
        continue

    for topic_id in row["topic_ids"]:
        mapped_body_by_topic[
            topic_id
        ].append(row)


print("\nUsable body evidence by Topic:")

for topic_id in topic_seed_v0:
    rows = mapped_body_by_topic.get(
        topic_id,
        [],
    )

    activity_mix = Counter(
        row["activity_type"]
        for row in rows
    )

    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"{len(rows)} activities "
        f"| {dict(activity_mix)}"
    )


def choose_representative_rows(rows, limit=2):
    selected = []
    seen_types = set()

    for row in rows:
        activity_type = row["activity_type"]

        if activity_type not in seen_types:
            selected.append(row)
            seen_types.add(activity_type)

        if len(selected) >= limit:
            return selected

    for row in rows:
        if row in selected:
            continue

        selected.append(row)

        if len(selected) >= limit:
            break

    return selected


representative_rows = []

for topic_id in topic_seed_v0:
    topic_rows = mapped_body_by_topic.get(
        topic_id,
        [],
    )

    selected = choose_representative_rows(
        topic_rows,
        limit=2,
    )

    for row in selected:
        representative_rows.append(
            (
                topic_id,
                row,
            )
        )


print("\nRepresentative mapped Activity evidence:")

for topic_id, row in representative_rows:
    excerpt = row["body_text"][:1200].strip()

    print(
        "\n----------------------------------------"
    )
    print(
        "Topic:",
        topic_seed_v0[topic_id]["label_fa"],
    )
    print("Work:", row["work_id"])
    print(
        "Paragraph:",
        row["paragraph_number"],
    )
    print(
        "Activity type:",
        row["activity_type"],
    )
    print("Heading:", row["heading"])
    print(
        "Parent:",
        row["nearest_parent"] or "-",
    )
    print(
        "Body paragraphs:",
        row["body_paragraph_count"],
    )
    print("Excerpt:")
    print(excerpt if excerpt else "<EMPTY>")


unmapped_activity_bodies = [
    row
    for row in activity_body_records
    if not row["topic_ids"]
]


print("\n\nUnmapped Activity evidence:")
print(
    "- Count:",
    len(unmapped_activity_bodies),
)


for row in unmapped_activity_bodies:
    excerpt = row["body_text"][:1200].strip()

    print(
        "\n----------------------------------------"
    )
    print("Work:", row["work_id"])
    print(
        "Paragraph:",
        row["paragraph_number"],
    )
    print(
        "Activity type:",
        row["activity_type"],
    )
    print("Heading:", row["heading"])
    print(
        "Parent:",
        row["nearest_parent"] or "-",
    )
    print(
        "Body status:",
        row["body_status"],
    )
    print(
        "Body paragraphs:",
        row["body_paragraph_count"],
    )
    print("Excerpt:")
    print(excerpt if excerpt else "<EMPTY>")


topics_without_usable_activity_body = [
    topic_id
    for topic_id in topic_seed_v0
    if not mapped_body_by_topic.get(topic_id)
]


unmapped_without_body = [
    row
    for row in unmapped_activity_bodies
    if row["body_status"] != "FOUND"
]


print("\nValidation:")
print(
    "- Topics without usable Activity body:",
    len(topics_without_usable_activity_body),
)
print(
    "- Unmapped activities without body:",
    len(unmapped_without_body),
)


if topics_without_usable_activity_body:
    for topic_id in topics_without_usable_activity_body:
        print(
            "- Missing Topic body evidence:",
            topic_seed_v0[topic_id]["label_fa"],
        )


if unmapped_without_body:
    for row in unmapped_without_body:
        print(
            "- Unmapped activity without body:",
            row["work_id"],
            "|",
            row["heading"],
        )


assert len(activity_body_records) == len(
    competency_evidence_rows
), "Activity extraction population mismatch."

assert not topics_without_usable_activity_body, (
    "At least one candidate Topic has no usable "
    "Activity body evidence."
)

print("\nCODE CELL 08 GATE: PASS")
print(
    "Activity-body evidence for Competency "
    "modeling discovered successfully."
)

Activity body extraction:
- Activity records: 48
- FOUND: 48

Usable body evidence by Topic:
- مبانی و توانمندی سواد مالی: 3 activities | {'self_assessment': 2, 'worksheet': 1}
- خرج، پس‌انداز و مدیریت پول: 11 activities | {'self_assessment': 4, 'worksheet': 1, 'exercise': 6}
- اعتبار، قرض، بدهی و بانک: 5 activities | {'self_assessment': 3, 'worksheet': 2}
- کار، شغل و درآمد: 3 activities | {'self_assessment': 1, 'worksheet': 2}
- سرمایه‌گذاری: 5 activities | {'self_assessment': 4, 'worksheet': 1}
- ریسک و بیمه: 3 activities | {'self_assessment': 3}
- تصمیم‌گیری مالی: 7 activities | {'worksheet': 2, 'self_assessment': 4, 'exercise': 1}

Representative mapped Activity evidence:

----------------------------------------
Topic: مبانی و توانمندی سواد مالی
Work: book:adult-financial-literacy-course
Paragraph: 162
Activity type: self_assessment
Heading: خودارزیابی 1: موقعیت من در پلکان توانمندی مالی
Parent: درس ۲: پلکان توانمندی مالی
Body paragraphs: 2
Excerpt:
حتماً احساس می‌کنید همه این پل

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازبینی موارد Unmapped و اصلاح Candidate Taxonomy</h3>

<p>
بررسی متن واقعی Activityها نشان داد که بیشتر موارد Unmapped ناشی از محدود بودن Evidence Termهای نسخه اولیه Taxonomy هستند، نه لزوماً نبود Topic اصلی.
</p>

<p>
برای مثال:
</p>

<ul>
<li>«توسعه شغلی» و «سبک شغلی» به‌روشنی در حوزه کار، شغل و درآمد قرار می‌گیرند.</li>
<li>«صندوق‌های سه‌گانه» به مدیریت پس‌انداز و منابع مالی مرتبط است.</li>
<li>«تصویرسازی آینده» با هدف‌گذاری و تصمیم‌گیری مالی ارتباط دارد.</li>
<li>تورم، قیمت، بخش واقعی و عددی و عرضه و تقاضا نشان‌دهنده نیاز به یک Subtopic اقتصادی در زیر مبانی سواد مالی هستند.</li>
<li>مواردی که Evidence کافی برای نگاشت قطعی ندارند باید به‌صورت Review Required باقی بمانند.</li>
</ul>

<p>
این مرحله فقط Candidate Refinement را در حافظه ثبت می‌کند.
هیچ Document یا Activity به‌صورت نهایی برچسب‌گذاری نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام ۱۱ Activity قبلاً Unmapped دارای یک Disposition صریح باشند.</li>
<li>هر نگاشت Candidate دارای Reason و Confidence باشد.</li>
<li>موارد نامطمئن به‌جای حدس زدن Review Required باقی بمانند.</li>
<li>هیچ Topic ID خارج از Candidate Taxonomy استفاده نشود.</li>
<li>Subtopicهای جدید فقط بر اساس Evidence مشاهده‌شده پیشنهاد شوند.</li>
</ul>

</div>

In [52]:
# Code Cell 09

taxonomy_refinement_review_v0 = {
    (
        "book:adult-financial-literacy-course",
        221,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "work_income",
        "subtopic_candidate": "income_growth",
        "reason": (
            "Activity explicitly asks the learner to identify "
            "ways to increase income and evaluate implementation."
        ),
        "confidence": 0.98,
    },

    (
        "book:adult-financial-literacy-course",
        272,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "spending_saving_money_management",
        "subtopic_candidate": "three_fund_money_allocation",
        "reason": (
            "Activity evaluates savings, emergency funds, "
            "and allocation across three financial funds."
        ),
        "confidence": 0.99,
    },

    (
        "book:adult-financial-literacy-course",
        722,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "work_income",
        "subtopic_candidate": "career_development",
        "reason": (
            "Activity explicitly asks how to develop the learner's "
            "job and create higher and more sustainable income."
        ),
        "confidence": 0.99,
    },

    (
        "book:adult-financial-literacy-course",
        764,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "work_income",
        "subtopic_candidate": "career_style_and_orientation",
        "reason": (
            "Activity evaluates current work style, career identity, "
            "and desired occupational orientation."
        ),
        "confidence": 0.99,
    },

    (
        "book:trainer-training-financial-literacy",
        103,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "financial_literacy_foundations",
        "subtopic_candidate": "real_vs_nominal_financial_value",
        "reason": (
            "Activity asks the learner to distinguish numerical "
            "and real dimensions of financial condition and decisions."
        ),
        "confidence": 0.90,
    },

    (
        "book:trainer-training-financial-literacy",
        107,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "financial_literacy_foundations",
        "subtopic_candidate": "inflation_and_real_value",
        "reason": (
            "Activity explicitly analyzes inflation, changes in "
            "income, prices, and real versus numerical value."
        ),
        "confidence": 0.96,
    },

    (
        "book:trainer-training-financial-literacy",
        114,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "financial_literacy_foundations",
        "subtopic_candidate": "price_and_supply_demand",
        "reason": (
            "Activity explicitly asks for analysis of price through "
            "supply-and-demand and government price intervention."
        ),
        "confidence": 0.92,
    },

    (
        "book:trainer-training-financial-literacy",
        219,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "financial_decision_making",
        "subtopic_candidate": "future_visualization_and_goals",
        "reason": (
            "Activity asks the learner to compare past future images "
            "with desired future outcomes and articulate future goals."
        ),
        "confidence": 0.94,
    },

    (
        "book:trainer-training-financial-literacy",
        269,
    ): {
        "disposition": "REVIEW_REQUIRED",
        "topic_id": None,
        "subtopic_candidate": "giving_and_social_financial_relations",
        "reason": (
            "The activity concerns giving, emotional proximity, "
            "and social relationships, but the excerpt alone does not "
            "establish a sufficiently clear primary financial Topic."
        ),
        "confidence": None,
    },

    (
        "book:trainer-training-financial-literacy",
        409,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "work_income",
        "subtopic_candidate": "income_activities_for_children",
        "reason": (
            "Activity explicitly requires designing income-generating "
            "activities for children."
        ),
        "confidence": 0.98,
    },

    (
        "book:trainer-training-financial-literacy",
        467,
    ): {
        "disposition": "MAPPED_CANDIDATE",
        "topic_id": "work_income",
        "subtopic_candidate": "entrepreneurship_and_business_basics",
        "reason": (
            "Activity requires designing a business, identifying "
            "production resources, calculating cost, pricing, "
            "promotion, and profit."
        ),
        "confidence": 0.97,
    },
}


unmapped_lookup = {
    (
        row["work_id"],
        row["paragraph_number"],
    ): row
    for row in unmapped_activity_bodies
}


print("Unmapped Activity refinement review:")
print(
    "- Previously unmapped:",
    len(unmapped_lookup),
)
print(
    "- Review decisions:",
    len(taxonomy_refinement_review_v0),
)


missing_review_decisions = sorted(
    set(unmapped_lookup)
    - set(taxonomy_refinement_review_v0)
)

unexpected_review_decisions = sorted(
    set(taxonomy_refinement_review_v0)
    - set(unmapped_lookup)
)


valid_topic_ids = set(topic_seed_v0)

invalid_topic_assignments = []

for key, review in taxonomy_refinement_review_v0.items():
    topic_id = review["topic_id"]

    if (
        topic_id is not None
        and topic_id not in valid_topic_ids
    ):
        invalid_topic_assignments.append(
            (key, topic_id)
        )


print("\nDetailed review:")

for key in sorted(taxonomy_refinement_review_v0):
    row = unmapped_lookup[key]
    review = taxonomy_refinement_review_v0[key]

    topic_id = review["topic_id"]

    topic_label = (
        topic_seed_v0[topic_id]["label_fa"]
        if topic_id
        else "REVIEW_REQUIRED"
    )

    print("\n----------------------------------------")
    print("Work:", key[0])
    print("Paragraph:", key[1])
    print("Heading:", row["heading"])
    print("Parent:", row["nearest_parent"] or "-")
    print("Disposition:", review["disposition"])
    print("Topic:", topic_label)
    print(
        "Subtopic candidate:",
        review["subtopic_candidate"],
    )
    print(
        "Confidence:",
        review["confidence"],
    )
    print("Reason:", review["reason"])


disposition_counter = Counter(
    review["disposition"]
    for review in taxonomy_refinement_review_v0.values()
)


print("\nDisposition summary:")

for disposition, count in disposition_counter.items():
    print(f"- {disposition}: {count}")


subtopic_candidates_by_topic = defaultdict(list)
review_required_subtopics = []

for review in taxonomy_refinement_review_v0.values():
    if review["topic_id"] is None:
        review_required_subtopics.append(
            review["subtopic_candidate"]
        )
        continue

    subtopic_candidates_by_topic[
        review["topic_id"]
    ].append(
        {
            "subtopic": review[
                "subtopic_candidate"
            ],
            "confidence": review["confidence"],
            "reason": review["reason"],
        }
    )


print("\nCandidate Taxonomy refinements:")

for topic_id in topic_seed_v0:
    refinements = subtopic_candidates_by_topic.get(
        topic_id,
        [],
    )

    if not refinements:
        continue

    print(
        f"\n[{topic_id}] "
        f"{topic_seed_v0[topic_id]['label_fa']}"
    )

    for item in refinements:
        print(
            f"- {item['subtopic']} "
            f"| confidence={item['confidence']:.2f}"
        )


print("\nReview-required candidate concepts:")

if review_required_subtopics:
    for value in review_required_subtopics:
        print("-", value)
else:
    print("- None")


resolved_candidate_count = sum(
    review["disposition"]
    == "MAPPED_CANDIDATE"
    for review in taxonomy_refinement_review_v0.values()
)

review_required_count = sum(
    review["disposition"]
    == "REVIEW_REQUIRED"
    for review in taxonomy_refinement_review_v0.values()
)


effective_activity_topic_count = (
    len(mapped_rows)
    + resolved_candidate_count
)


print("\nUpdated Activity mapping status:")
print("- Total activities:", len(activity_body_records))
print(
    "- Previously mapped:",
    len(mapped_rows),
)
print(
    "- Newly mapped candidates:",
    resolved_candidate_count,
)
print(
    "- Explicit Review Required:",
    review_required_count,
)
print(
    "- Effective candidate mapping coverage:",
    f"{effective_activity_topic_count}/"
    f"{len(activity_body_records)}",
)
print(
    "- Candidate mapping rate:",
    f"{effective_activity_topic_count / len(activity_body_records):.1%}",
)


print("\nValidation:")
print(
    "- Missing review decisions:",
    len(missing_review_decisions),
)
print(
    "- Unexpected review decisions:",
    len(unexpected_review_decisions),
)
print(
    "- Invalid Topic assignments:",
    len(invalid_topic_assignments),
)
print(
    "- Review Required retained:",
    review_required_count,
)


assert not missing_review_decisions, (
    "At least one previously unmapped Activity "
    "has no explicit review decision."
)

assert not unexpected_review_decisions, (
    "Review registry contains an Activity that "
    "was not in the unmapped population."
)

assert not invalid_topic_assignments, (
    "At least one refinement uses an invalid Topic ID."
)

assert (
    resolved_candidate_count
    + review_required_count
    == len(unmapped_lookup)
), (
    "Every previously unmapped Activity must be "
    "either mapped as candidate or retained for review."
)

print("\nCODE CELL 09 GATE: PASS")
print(
    "Unmapped Activity review and candidate "
    "Taxonomy refinement completed successfully."
)

Unmapped Activity refinement review:
- Previously unmapped: 11
- Review decisions: 11

Detailed review:

----------------------------------------
Work: book:adult-financial-literacy-course
Paragraph: 221
Heading: خود ارزیابی 2: می‌خواهم درآمدم زیاد شود
Parent: درس ۳: نگرانی یا مسأله مالی
Disposition: MAPPED_CANDIDATE
Topic: کار، شغل و درآمد
Subtopic candidate: income_growth
Confidence: 0.98
Reason: Activity explicitly asks the learner to identify ways to increase income and evaluate implementation.

----------------------------------------
Work: book:adult-financial-literacy-course
Paragraph: 272
Heading: خودارزیابی ۵: صندوق‌های سه‌گانه من
Parent: درس ۶: صندوق‌های سه‌گانه
Disposition: MAPPED_CANDIDATE
Topic: خرج، پس‌انداز و مدیریت پول
Subtopic candidate: three_fund_money_allocation
Confidence: 0.99
Reason: Activity evaluates savings, emergency funds, and allocation across three financial funds.

----------------------------------------
Work: book:adult-financial-literacy-course
Paragra

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعریف جهت آموزشی محصول و Candidate Competency Model</h3>

<p>
این پروژه یک سامانه عمومی سرمایه‌گذاری یا موتور پیشنهاد خرید و فروش دارایی نیست.
دامنه اصلی آن آموزش، فهم، خودارزیابی، برنامه‌ریزی و توسعه توانمندی‌های سواد مالی بر پایه دانش دکتر کمیل رودی است.
</p>

<p>
بنابراین هدف Taxonomy صرفاً دسته‌بندی محتوا نیست؛ Taxonomy باید مشخص کند یادگیرنده در هر حوزه چه چیزی را باید بفهمد، چه چیزی را درباره وضعیت خود ارزیابی کند و در نهایت چه رفتار یا برنامه مالی‌ای بتواند اجرا و بازبینی کند.
</p>

<h4>مسیر کلی یادگیری</h4>

<ol>
<li>فهم مفهوم و اصول مالی</li>
<li>شناخت و ارزیابی وضعیت شخصی</li>
<li>تحلیل گزینه‌ها و پیامدها</li>
<li>برنامه‌ریزی مالی</li>
<li>اجرای رفتار و اقدام مناسب</li>
<li>بازبینی، اصلاح و پیشرفت</li>
</ol>

<p>
آزادی مالی در این پروژه یک افق آموزشی و توسعه‌ای است، نه یک نتیجه تضمین‌شده.
</p>

<h4>مرز مهم محصول</h4>

<p>
سیستم می‌تواند درباره مفاهیم سرمایه‌گذاری، ریسک، بازده، نقدشوندگی، هدف‌گذاری و مقایسه آموزشی گزینه‌ها توضیح دهد؛
اما نباید به موتور خرید و فروش، انتخاب شخصی دارایی یا اجرای معامله تبدیل شود.
</p>

<p>
در این مرحله یک Competency Model اولیه بر اساس شواهد واقعی Activityهای کتاب‌ها ساخته می‌شود.
این مدل هنوز Freeze نشده و روی Disk نیز نوشته نمی‌شود.
</p>

</div>

In [53]:
# Code Cell 10

knowledge_product_guardrails_v0 = {
    "domain": "financial_literacy",
    "primary_knowledge_authority": "Dr. Komeil Roudi",

    "product_role": [
        "education",
        "financial_literacy_guidance",
        "self_assessment",
        "financial_planning_support",
        "behavioral_financial_capability",
        "learning_progression",
    ],

    "learning_horizon": [
        "understand",
        "self_assess",
        "analyze",
        "plan",
        "act",
        "review_and_improve",
        "financial_capability",
        "financial_freedom_horizon",
    ],

    "financial_freedom_policy": {
        "role": "directional_learning_horizon",
        "guaranteed_outcome": False,
    },

    "investment_boundary": {
        "allowed": [
            "explain investment concepts",
            "teach risk and return",
            "teach liquidity and capital requirements",
            "help clarify investment goals",
            "educational comparison of option characteristics",
            "teach risk-management principles",
        ],
        "not_product_objective": [
            "personalized buy recommendation",
            "personalized sell recommendation",
            "security selection",
            "trade execution",
            "portfolio execution",
        ],
    },
}


competency_model_v0 = {
    "financial_literacy_foundations": {
        "label_fa": "مبانی و توانمندی سواد مالی",
        "competencies": [
            {
                "id": "FLF-C01",
                "stage": "understand",
                "statement_fa": (
                    "مفاهیم پایه سواد مالی و تفاوت میان "
                    "ارزش عددی و ارزش واقعی پول را درک کند."
                ),
                "evidence": [
                    "پلکان توانمندی مالی",
                    "بخش عددی و بخش واقعی",
                    "تورم",
                    "قیمت و عرضه و تقاضا",
                ],
            },
            {
                "id": "FLF-C02",
                "stage": "self_assess",
                "statement_fa": (
                    "وضعیت فعلی توانمندی مالی خود را "
                    "ارزیابی و توصیف کند."
                ),
                "evidence": [
                    "خودارزیابی موقعیت در پلکان توانمندی مالی",
                    "خودآزمایی مالی",
                ],
            },
            {
                "id": "FLF-C03",
                "stage": "plan",
                "statement_fa": (
                    "تصویری از وضعیت مالی مطلوب آینده "
                    "و مسیر حرکت به سمت آن تعریف کند."
                ),
                "evidence": [
                    "نامه‌ای به خودتان",
                    "نمای کلی وضعیت مالی مطلوب آینده",
                ],
            },
        ],
    },

    "spending_saving_money_management": {
        "label_fa": "خرج، پس‌انداز و مدیریت پول",
        "competencies": [
            {
                "id": "SSM-C01",
                "stage": "self_assess",
                "statement_fa": (
                    "الگوی هزینه و مصرف خود را شناسایی "
                    "و نقاط قابل اصلاح را پیدا کند."
                ),
                "evidence": [
                    "آگاهی از هزینه‌ها",
                    "الگوهای خرید",
                    "ثبت هزینه‌ها",
                ],
            },
            {
                "id": "SSM-C02",
                "stage": "plan",
                "statement_fa": (
                    "بودجه‌ای متناسب با درآمد، هزینه‌ها "
                    "و اهداف مالی تنظیم کند."
                ),
                "evidence": [
                    "کاربرگ ایده‌آل‌های مالی",
                    "بودجه‌بندی",
                ],
            },
            {
                "id": "SSM-C03",
                "stage": "plan",
                "statement_fa": (
                    "برای پس‌انداز، صندوق اضطراری و سایر "
                    "کارکردهای پول تخصیص مناسب طراحی کند."
                ),
                "evidence": [
                    "صندوق‌های سه‌گانه",
                    "هدف پس‌اندازی",
                    "صندوق اضطراری",
                ],
            },
            {
                "id": "SSM-C04",
                "stage": "act",
                "statement_fa": (
                    "رفتارهای عملی مدیریت هزینه، خرید "
                    "و پس‌انداز را اجرا و بازبینی کند."
                ),
                "evidence": [
                    "تمرین خرید هوشمندانه",
                    "تمرین بودجه‌بندی",
                    "تمرین پس‌انداز",
                ],
            },
        ],
    },

    "credit_debt_banking": {
        "label_fa": "اعتبار، قرض، بدهی و بانک",
        "competencies": [
            {
                "id": "CDB-C01",
                "stage": "understand",
                "statement_fa": (
                    "مفاهیم قرض، بدهی، اعتبار و خدمات "
                    "بانکی را از یکدیگر تشخیص دهد."
                ),
                "evidence": [
                    "قواعد قرض",
                    "کارت اعتباری",
                    "اعتبارسنجی",
                    "بانک و اعتبار",
                ],
            },
            {
                "id": "CDB-C02",
                "stage": "self_assess",
                "statement_fa": (
                    "وضعیت بدهی‌های خود را ثبت، طبقه‌بندی "
                    "و ارزیابی کند."
                ),
                "evidence": [
                    "نقشه بدهکاری من",
                    "قرض در نگاه من",
                ],
            },
            {
                "id": "CDB-C03",
                "stage": "plan",
                "statement_fa": (
                    "برای اصلاح یا مدیریت وضعیت بدهی "
                    "راهکارهای مناسب تدوین کند."
                ),
                "evidence": [
                    "نوع بدهی",
                    "دلیل بدهی",
                    "راه‌حل بهتر",
                ],
            },
        ],
    },

    "work_income": {
        "label_fa": "کار، شغل و درآمد",
        "competencies": [
            {
                "id": "WIN-C01",
                "stage": "self_assess",
                "statement_fa": (
                    "دانش، مهارت، علاقه و وضعیت شغلی "
                    "خود را ارزیابی کند."
                ),
                "evidence": [
                    "فهرست مشاغل مناسب من",
                    "سبک و رویکرد کاری من",
                ],
            },
            {
                "id": "WIN-C02",
                "stage": "plan",
                "statement_fa": (
                    "مسیر توسعه شغلی و افزایش درآمد "
                    "پایدار خود را طراحی کند."
                ),
                "evidence": [
                    "توسعه شغلی من",
                    "می‌خواهم درآمدم زیاد شود",
                ],
            },
            {
                "id": "WIN-C03",
                "stage": "act",
                "statement_fa": (
                    "فرصت‌های مناسب کار، فعالیت درآمدزا "
                    "یا کسب‌وکار را طراحی و ارزیابی کند."
                ),
                "evidence": [
                    "فعالیت درآمدزا برای کودکان",
                    "طراحی کسب‌وکار",
                    "منابع تولید",
                    "هزینه",
                    "قیمت",
                    "سود",
                ],
            },
        ],
    },

    "investment": {
        "label_fa": "سرمایه‌گذاری",
        "competencies": [
            {
                "id": "INV-C01",
                "stage": "understand",
                "statement_fa": (
                    "مفهوم سرمایه‌گذاری و ویژگی‌های اصلی "
                    "گزینه‌های سرمایه‌گذاری را درک کند."
                ),
                "evidence": [
                    "چرا سرمایه‌گذاری",
                    "گزینه‌های سرمایه‌گذاری",
                    "صندوق‌های سرمایه‌گذاری",
                ],
            },
            {
                "id": "INV-C02",
                "stage": "self_assess",
                "statement_fa": (
                    "هدف‌ها و تجربه‌های سرمایه‌گذاری "
                    "خود را ارزیابی کند."
                ),
                "evidence": [
                    "اولین سرمایه‌گذاری من",
                    "گزینه‌های سرمایه‌گذاری من",
                ],
            },
            {
                "id": "INV-C03",
                "stage": "analyze",
                "statement_fa": (
                    "گزینه‌های سرمایه‌گذاری را از نظر "
                    "ریسک، بازده، نقدشوندگی و سرمایه موردنیاز "
                    "به‌صورت آموزشی مقایسه کند."
                ),
                "evidence": [
                    "چهارگام سرمایه‌گذاری",
                ],
            },
            {
                "id": "INV-C04",
                "stage": "plan",
                "statement_fa": (
                    "بر اساس اهداف مالی خود یک برنامه "
                    "یادگیری و بررسی سرمایه‌گذاری تدوین کند، "
                    "بدون تبدیل سیستم به موتور توصیه خرید یا فروش."
                ),
                "evidence": [
                    "هدف سرمایه‌گذاری",
                    "گزینه‌های پیش‌رو",
                    "اقدام آینده",
                ],
            },
        ],
    },

    "risk_insurance": {
        "label_fa": "ریسک و بیمه",
        "competencies": [
            {
                "id": "RIN-C01",
                "stage": "understand",
                "statement_fa": (
                    "انواع مواجهه با ریسک و نقش بیمه "
                    "در مدیریت ریسک را درک کند."
                ),
                "evidence": [
                    "مدیریت ریسک",
                    "انواع بیمه",
                ],
            },
            {
                "id": "RIN-C02",
                "stage": "self_assess",
                "statement_fa": (
                    "نگرش و تحمل خود در برابر ریسک "
                    "و زیان مالی را ارزیابی کند."
                ),
                "evidence": [
                    "مواجهه من با ریسک",
                    "حد ضرر من",
                ],
            },
            {
                "id": "RIN-C03",
                "stage": "plan",
                "statement_fa": (
                    "برای ریسک‌های مالی مهم زندگی "
                    "روش‌های مدیریت و پوشش مناسب را بررسی کند."
                ),
                "evidence": [
                    "روش‌های چهارگانه مواجهه با ریسک",
                    "بیمه من",
                ],
            },
        ],
    },

    "financial_decision_making": {
        "label_fa": "تصمیم‌گیری مالی",
        "competencies": [
            {
                "id": "FDM-C01",
                "stage": "understand",
                "statement_fa": (
                    "عوامل مالی و غیرمالی اثرگذار بر "
                    "تصمیم‌های مالی را تشخیص دهد."
                ),
                "evidence": [
                    "مؤلفه‌های مالی در تصمیم‌گیری",
                    "مؤلفه‌های غیرمالی در تصمیم‌گیری",
                ],
            },
            {
                "id": "FDM-C02",
                "stage": "self_assess",
                "statement_fa": (
                    "دارایی‌ها، تمایلات، نیازها، خواسته‌ها "
                    "و الگوهای تصمیم‌گیری خود را ارزیابی کند."
                ),
                "evidence": [
                    "فهرست دارایی‌ها",
                    "ذهن ثروتمند یا فقیر",
                    "سبک پاسخگویی به تمایلات",
                ],
            },
            {
                "id": "FDM-C03",
                "stage": "analyze",
                "statement_fa": (
                    "پیامدهای انتخاب مالی و هزینه فرصت "
                    "گزینه‌ها را تحلیل کند."
                ),
                "evidence": [
                    "هزینه فرصت",
                    "انتخاب و هزینه‌فرصت",
                ],
            },
            {
                "id": "FDM-C04",
                "stage": "plan",
                "statement_fa": (
                    "اهداف مالی و تصویر آینده مطلوب خود "
                    "را به تصمیم‌ها و برنامه‌های عملی متصل کند."
                ),
                "evidence": [
                    "هدف‌گذاری هوشمندانه",
                    "تصویرسازی خواسته‌ها",
                ],
            },
        ],
    },
}


valid_stages = {
    "understand",
    "self_assess",
    "analyze",
    "plan",
    "act",
    "review_and_improve",
}


print("Knowledge Product Guardrails:")
print("- Domain:", knowledge_product_guardrails_v0["domain"])
print(
    "- Primary Knowledge Authority:",
    knowledge_product_guardrails_v0[
        "primary_knowledge_authority"
    ],
)
print(
    "- Financial freedom is guaranteed:",
    knowledge_product_guardrails_v0[
        "financial_freedom_policy"
    ]["guaranteed_outcome"],
)


print("\nInvestment boundary:")

for item in knowledge_product_guardrails_v0[
    "investment_boundary"
]["allowed"]:
    print("- ALLOWED:", item)

for item in knowledge_product_guardrails_v0[
    "investment_boundary"
]["not_product_objective"]:
    print("- NOT PRODUCT OBJECTIVE:", item)


print("\nCandidate Competency Model:")

competency_ids = []
invalid_stage_rows = []
topics_without_competency = []

for topic_id, topic in competency_model_v0.items():
    competencies = topic["competencies"]

    print(
        f"\n[{topic_id}] {topic['label_fa']}"
    )
    print("- Competencies:", len(competencies))

    if not competencies:
        topics_without_competency.append(
            topic_id
        )

    for competency in competencies:
        competency_ids.append(
            competency["id"]
        )

        if competency["stage"] not in valid_stages:
            invalid_stage_rows.append(
                (
                    topic_id,
                    competency["id"],
                    competency["stage"],
                )
            )

        print(
            f"- {competency['id']} "
            f"| stage={competency['stage']} "
            f"| {competency['statement_fa']}"
        )


duplicate_competency_ids = [
    competency_id
    for competency_id, count
    in Counter(competency_ids).items()
    if count > 1
]


missing_topic_models = (
    set(topic_seed_v0)
    - set(competency_model_v0)
)

unexpected_topic_models = (
    set(competency_model_v0)
    - set(topic_seed_v0)
)


total_competencies = sum(
    len(topic["competencies"])
    for topic in competency_model_v0.values()
)


print("\nValidation:")
print(
    "- Top-level Topics:",
    len(competency_model_v0),
)
print(
    "- Total candidate Competencies:",
    total_competencies,
)
print(
    "- Topics without Competency:",
    len(topics_without_competency),
)
print(
    "- Missing Topic models:",
    len(missing_topic_models),
)
print(
    "- Unexpected Topic models:",
    len(unexpected_topic_models),
)
print(
    "- Duplicate Competency IDs:",
    len(duplicate_competency_ids),
)
print(
    "- Invalid learning stages:",
    len(invalid_stage_rows),
)


assert knowledge_product_guardrails_v0[
    "domain"
] == "financial_literacy"

assert (
    knowledge_product_guardrails_v0[
        "financial_freedom_policy"
    ]["guaranteed_outcome"]
    is False
)

assert not missing_topic_models
assert not unexpected_topic_models
assert not topics_without_competency
assert not duplicate_competency_ids
assert not invalid_stage_rows

assert set(
    knowledge_product_guardrails_v0[
        "investment_boundary"
    ]["not_product_objective"]
) >= {
    "personalized buy recommendation",
    "personalized sell recommendation",
    "trade execution",
}


print("\nCODE CELL 10 GATE: PASS")
print(
    "Financial-literacy product direction and "
    "candidate Competency Model validated successfully."
)

Knowledge Product Guardrails:
- Domain: financial_literacy
- Primary Knowledge Authority: Dr. Komeil Roudi
- Financial freedom is guaranteed: False

Investment boundary:
- ALLOWED: explain investment concepts
- ALLOWED: teach risk and return
- ALLOWED: teach liquidity and capital requirements
- ALLOWED: help clarify investment goals
- ALLOWED: educational comparison of option characteristics
- ALLOWED: teach risk-management principles
- NOT PRODUCT OBJECTIVE: personalized buy recommendation
- NOT PRODUCT OBJECTIVE: personalized sell recommendation
- NOT PRODUCT OBJECTIVE: security selection
- NOT PRODUCT OBJECTIVE: trade execution
- NOT PRODUCT OBJECTIVE: portfolio execution

Candidate Competency Model:

[financial_literacy_foundations] مبانی و توانمندی سواد مالی
- Competencies: 3
- FLF-C01 | stage=understand | مفاهیم پایه سواد مالی و تفاوت میان ارزش عددی و ارزش واقعی پول را درک کند.
- FLF-C02 | stage=self_assess | وضعیت فعلی توانمندی مالی خود را ارزیابی و توصیف کند.
- FLF-C03 | stage=

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعریف سیاست Prerequisite و Learning Progression</h3>

<p>
ترتیب ارائه Topicها در کتاب‌های مختلف همیشه یکسان نیست.
بنابراین تقدم یک Topic در یک یا چند کتاب به‌تنهایی اثبات نمی‌کند که آن Topic پیش‌نیاز قطعی Topic بعدی است.
</p>

<p>
برای جلوگیری از ایجاد وابستگی آموزشی مصنوعی، در مدل دانش میان سه مفهوم تفاوت گذاشته می‌شود:
</p>

<ul>
<li><strong>Hard Prerequisite:</strong> پیش‌نیازی که بدون آن Competency بعدی عملاً قابل انجام نیست و باید Evidence روشن داشته باشد.</li>
<li><strong>Recommended Learning Progression:</strong> ترتیب پیشنهادی برای یادگیری، بدون ادعای الزام.</li>
<li><strong>Curricular Order Evidence:</strong> ترتیبی که واقعاً در یک یا چند Work مشاهده شده است.</li>
</ul>

<p>
در نسخه Candidate فعلی هیچ رابطه Cross-Topic به‌عنوان Hard Prerequisite ثبت نمی‌شود.
روابط بین Topicها فعلاً فقط به‌عنوان Evidence ترتیب آموزشی حفظ می‌شوند.
</p>

<p>
در سطح Competency نیز تنها زمانی رابطه پیشنهادی ساخته می‌شود که یک توانایی پیشرفته‌تر به‌طور معقول بر توانایی قبلی همان Topic تکیه داشته باشد.
این روابط هنوز Candidate هستند و پیش از Freeze باید Expert Review شوند.
</p>

<h4>اصل آموزشی</h4>

<p>
مسیر عمومی محصول از فهم به خودارزیابی، تحلیل، برنامه‌ریزی، اقدام و بازبینی حرکت می‌کند؛
اما همه Topicها الزاماً تمام این مراحل را ندارند و نباید مرحله‌ای بدون Evidence به آن‌ها تحمیل شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هیچ Cross-Topic Hard Prerequisite بدون Evidence ایجاد نشود.</li>
<li>ترتیب Curriculum از Prerequisite جدا نگه داشته شود.</li>
<li>تمام Competency IDهای مورد استفاده معتبر باشند.</li>
<li>هیچ Dependency cycle در Candidate graph وجود نداشته باشد.</li>
<li>روابط Candidate دارای نوع و Evidence Basis مشخص باشند.</li>
</ul>

</div>

In [54]:
# Code Cell 11

from collections import defaultdict, Counter


learning_stage_rank = {
    "understand": 10,
    "self_assess": 20,
    "analyze": 30,
    "plan": 40,
    "act": 50,
    "review_and_improve": 60,
}


competency_lookup = {}

for topic_id, topic in competency_model_v0.items():
    for competency in topic["competencies"]:
        competency_lookup[
            competency["id"]
        ] = {
            **competency,
            "topic_id": topic_id,
            "topic_label_fa": topic["label_fa"],
        }


prerequisite_policy_v0 = {
    "cross_topic_hard_prerequisites_allowed": False,
    "curriculum_order_is_prerequisite": False,
    "expert_review_required_before_freeze": True,
    "relationship_types": [
        "HARD_PREREQUISITE",
        "RECOMMENDED_LEARNING_PROGRESSION",
        "CURRICULAR_ORDER_EVIDENCE",
    ],
}


candidate_competency_relations_v0 = [
    {
        "from": "FLF-C01",
        "to": "FLF-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Understanding basic financial-literacy concepts "
            "supports meaningful self-assessment."
        ),
        "evidence_basis": "competency_stage_progression",
    },
    {
        "from": "FLF-C02",
        "to": "FLF-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Assessment of current capability supports defining "
            "a desired future financial state."
        ),
        "evidence_basis": "activity_and_competency_progression",
    },

    {
        "from": "SSM-C01",
        "to": "SSM-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Awareness of current spending patterns supports "
            "realistic budgeting."
        ),
        "evidence_basis": "book_activity_progression",
    },
    {
        "from": "SSM-C02",
        "to": "SSM-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Budget planning supports intentional allocation "
            "to savings and emergency purposes."
        ),
        "evidence_basis": "competency_progression",
    },
    {
        "from": "SSM-C03",
        "to": "SSM-C04",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "A defined allocation plan supports implementation "
            "and review of money-management behavior."
        ),
        "evidence_basis": "competency_stage_progression",
    },

    {
        "from": "CDB-C01",
        "to": "CDB-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Understanding debt and credit concepts supports "
            "classification of personal debt."
        ),
        "evidence_basis": "book_activity_progression",
    },
    {
        "from": "CDB-C02",
        "to": "CDB-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "A debt inventory and classification supports "
            "planning corrective actions."
        ),
        "evidence_basis": "debt_map_activity",
    },

    {
        "from": "WIN-C01",
        "to": "WIN-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Assessment of skills, interests, and work status "
            "supports career-development planning."
        ),
        "evidence_basis": "career_worksheet_progression",
    },
    {
        "from": "WIN-C02",
        "to": "WIN-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Career and income planning can support practical "
            "design of income-generating opportunities."
        ),
        "evidence_basis": "competency_stage_progression",
    },

    {
        "from": "INV-C01",
        "to": "INV-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Conceptual investment literacy supports more "
            "meaningful assessment of investment goals "
            "and prior experience."
        ),
        "evidence_basis": "investment_activity_progression",
    },
    {
        "from": "INV-C02",
        "to": "INV-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Clarified goals support educational comparison "
            "of investment-option characteristics."
        ),
        "evidence_basis": "four_step_investment_activity",
    },
    {
        "from": "INV-C03",
        "to": "INV-C04",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Educational comparison of risk, return, liquidity, "
            "and capital requirements supports planning."
        ),
        "evidence_basis": "four_step_investment_activity",
    },

    {
        "from": "RIN-C01",
        "to": "RIN-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Understanding risk-management approaches supports "
            "self-assessment of personal risk response."
        ),
        "evidence_basis": "risk_self_assessment_activity",
    },
    {
        "from": "RIN-C02",
        "to": "RIN-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Risk self-assessment supports evaluation of "
            "appropriate risk-management approaches."
        ),
        "evidence_basis": "competency_progression",
    },

    {
        "from": "FDM-C01",
        "to": "FDM-C02",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Understanding financial and nonfinancial decision "
            "factors supports self-assessment."
        ),
        "evidence_basis": "decision_making_activity_progression",
    },
    {
        "from": "FDM-C02",
        "to": "FDM-C03",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Awareness of personal assets, needs, wants, and "
            "decision patterns supports consequence analysis."
        ),
        "evidence_basis": "decision_making_activity_progression",
    },
    {
        "from": "FDM-C03",
        "to": "FDM-C04",
        "type": "RECOMMENDED_LEARNING_PROGRESSION",
        "reason": (
            "Analysis of alternatives and opportunity cost "
            "supports goal-linked financial planning."
        ),
        "evidence_basis": "competency_stage_progression",
    },
]


hard_prerequisite_relations_v0 = []


curricular_order_evidence_v0 = []

for pair_key, evidence in consistent_pairs:
    observations = evidence["works"]

    if not observations:
        continue

    earlier = observations[0]["earlier"]
    later = observations[0]["later"]

    curricular_order_evidence_v0.append(
        {
            "earlier_topic_id": earlier,
            "later_topic_id": later,
            "relationship_type": "CURRICULAR_ORDER_EVIDENCE",
            "supporting_work_count": len(observations),
            "supporting_works": sorted({
                item["work_id"]
                for item in observations
            }),
            "is_prerequisite": False,
        }
    )


curricular_order_conflicts_v0 = []

for pair_key, evidence in conflicting_pairs:
    curricular_order_conflicts_v0.append(
        {
            "topic_a": pair_key[0],
            "topic_b": pair_key[1],
            "relationship_type": "CURRICULAR_ORDER_CONFLICT",
            "observations": evidence["works"],
            "is_prerequisite": False,
        }
    )


print("Prerequisite policy:")
for key, value in prerequisite_policy_v0.items():
    print(f"- {key}: {value}")


print("\nCandidate competency learning progression:")
print(
    "- Relations:",
    len(candidate_competency_relations_v0),
)

for relation in candidate_competency_relations_v0:
    source = competency_lookup[
        relation["from"]
    ]
    target = competency_lookup[
        relation["to"]
    ]

    print(
        f"- {relation['from']} -> {relation['to']} "
        f"| {source['topic_label_fa']} "
        f"| {source['stage']} -> {target['stage']} "
        f"| {relation['type']}"
    )


invalid_relation_ids = []
cross_topic_candidate_relations = []
backward_stage_relations = []

for relation in candidate_competency_relations_v0:
    source_id = relation["from"]
    target_id = relation["to"]

    if source_id not in competency_lookup:
        invalid_relation_ids.append(source_id)

    if target_id not in competency_lookup:
        invalid_relation_ids.append(target_id)

    if (
        source_id not in competency_lookup
        or target_id not in competency_lookup
    ):
        continue

    source = competency_lookup[source_id]
    target = competency_lookup[target_id]

    if source["topic_id"] != target["topic_id"]:
        cross_topic_candidate_relations.append(
            relation
        )

    if (
        learning_stage_rank[source["stage"]]
        >
        learning_stage_rank[target["stage"]]
    ):
        backward_stage_relations.append(
            relation
        )


def graph_has_cycle(relations):
    adjacency = defaultdict(list)
    nodes = set()

    for relation in relations:
        source = relation["from"]
        target = relation["to"]

        adjacency[source].append(target)
        nodes.add(source)
        nodes.add(target)

    visiting = set()
    visited = set()

    def visit(node):
        if node in visiting:
            return True

        if node in visited:
            return False

        visiting.add(node)

        for neighbor in adjacency[node]:
            if visit(neighbor):
                return True

        visiting.remove(node)
        visited.add(node)
        return False

    return any(
        visit(node)
        for node in nodes
        if node not in visited
    )


candidate_graph_has_cycle = graph_has_cycle(
    candidate_competency_relations_v0
)


print("\nCurricular order evidence:")
print(
    "- Consistent cross-work order relations:",
    len(curricular_order_evidence_v0),
)

for relation in curricular_order_evidence_v0:
    print(
        "-",
        topic_seed_v0[
            relation["earlier_topic_id"]
        ]["label_fa"],
        "->",
        topic_seed_v0[
            relation["later_topic_id"]
        ]["label_fa"],
        "| works=",
        relation["supporting_work_count"],
        "| prerequisite=",
        relation["is_prerequisite"],
    )


print("\nCurricular order conflicts:")
print(
    "- Conflicting pairs:",
    len(curricular_order_conflicts_v0),
)

for conflict in curricular_order_conflicts_v0:
    print(
        "-",
        topic_seed_v0[
            conflict["topic_a"]
        ]["label_fa"],
        "<->",
        topic_seed_v0[
            conflict["topic_b"]
        ]["label_fa"],
    )


relation_type_counter = Counter(
    relation["type"]
    for relation in candidate_competency_relations_v0
)


print("\nValidation:")
print(
    "- Hard prerequisite relations:",
    len(hard_prerequisite_relations_v0),
)
print(
    "- Candidate learning-progression relations:",
    len(candidate_competency_relations_v0),
)
print(
    "- Invalid Competency IDs:",
    len(invalid_relation_ids),
)
print(
    "- Cross-topic candidate competency relations:",
    len(cross_topic_candidate_relations),
)
print(
    "- Backward-stage relations:",
    len(backward_stage_relations),
)
print(
    "- Candidate graph cycle:",
    candidate_graph_has_cycle,
)
print(
    "- Consistent curriculum-order evidence:",
    len(curricular_order_evidence_v0),
)
print(
    "- Curriculum-order conflicts preserved:",
    len(curricular_order_conflicts_v0),
)


assert not prerequisite_policy_v0[
    "cross_topic_hard_prerequisites_allowed"
]

assert prerequisite_policy_v0[
    "curriculum_order_is_prerequisite"
] is False

assert len(hard_prerequisite_relations_v0) == 0

assert not invalid_relation_ids

assert not cross_topic_candidate_relations

assert not backward_stage_relations

assert candidate_graph_has_cycle is False

assert len(curricular_order_evidence_v0) == 9

assert len(curricular_order_conflicts_v0) == 4


print("\nCODE CELL 11 GATE: PASS")
print(
    "Prerequisite policy and evidence-backed "
    "learning progression validated successfully."
)

Prerequisite policy:
- cross_topic_hard_prerequisites_allowed: False
- curriculum_order_is_prerequisite: False
- expert_review_required_before_freeze: True
- relationship_types: ['HARD_PREREQUISITE', 'RECOMMENDED_LEARNING_PROGRESSION', 'CURRICULAR_ORDER_EVIDENCE']

Candidate competency learning progression:
- Relations: 17
- FLF-C01 -> FLF-C02 | مبانی و توانمندی سواد مالی | understand -> self_assess | RECOMMENDED_LEARNING_PROGRESSION
- FLF-C02 -> FLF-C03 | مبانی و توانمندی سواد مالی | self_assess -> plan | RECOMMENDED_LEARNING_PROGRESSION
- SSM-C01 -> SSM-C02 | خرج، پس‌انداز و مدیریت پول | self_assess -> plan | RECOMMENDED_LEARNING_PROGRESSION
- SSM-C02 -> SSM-C03 | خرج، پس‌انداز و مدیریت پول | plan -> plan | RECOMMENDED_LEARNING_PROGRESSION
- SSM-C03 -> SSM-C04 | خرج، پس‌انداز و مدیریت پول | plan -> act | RECOMMENDED_LEARNING_PROGRESSION
- CDB-C01 -> CDB-C02 | اعتبار، قرض، بدهی و بانک | understand -> self_assess | RECOMMENDED_LEARNING_PROGRESSION
- CDB-C02 -> CDB-C03 | اعتبار، قرض، بد

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery شواهد Content Type در Corpus</h3>

<p>
در Taxonomy علاوه بر Topic و Competency، لازم است نقش آموزشی محتوا نیز مشخص شود.
Roadmap پنج نوع اصلی را پیشنهاد می‌کند:
مفهوم، مثال، تمرین، هشدار و راهکار.
</p>

<p>
اما این برچسب‌ها نباید فقط بر اساس وجود چند کلمه در متن به همه Documentها نسبت داده شوند.
ابتدا باید شواهد صریح و ساختاری موجود در Website، Book و Transcript بررسی شوند.
</p>

<h4>هدف این مرحله</h4>

<ul>
<li>بررسی فیلدهای ساختاری مرتبط با نوع محتوا.</li>
<li>شناسایی Headingهای صریح مانند تمرین، خودارزیابی، کاربرگ، مثال و منابع آموزشی.</li>
<li>بررسی عبارت‌های صریح مرتبط با هشدار، توصیه، راهکار و مثال.</li>
<li>تفکیک Content Type آموزشی از Source Type و Program Type.</li>
<li>تعیین اینکه کدام Content Typeها Evidence قوی و کدام‌ها فقط Evidence ضعیف دارند.</li>
</ul>

<p>
در این مرحله هنوز هیچ Documentی برچسب نهایی نمی‌گیرد و هیچ Artifactی نوشته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>شواهد ساختاری هر سه Source بررسی شوند.</li>
<li>Activityهای صریح کتاب به‌درستی از محتوای مفهومی جدا شوند.</li>
<li>عبارت‌های Candidate برای Example، Warning و Solution فقط به‌عنوان Evidence شمارش شوند.</li>
<li>Program، Speaker یا Source Type به‌عنوان Content Type استفاده نشوند.</li>
</ul>

</div>

In [55]:
# Code Cell 12

from collections import Counter, defaultdict
import re


content_type_seed_v0 = {
    "concept": {
        "label_fa": "مفهوم / آموزش",
        "explicit_patterns": [
            r"\bتعریف\b",
            r"\bمفهوم\b",
            r"\bچیستی\b",
            r"\bماهیت\b",
            r"\bانواع\b",
            r"\bاصول\b",
            r"\bقواعد\b",
        ],
    },
    "example": {
        "label_fa": "مثال / نمونه",
        "explicit_patterns": [
            r"\bمثال\b",
            r"\bنمونه\b",
            r"\bبرای مثال\b",
            r"\bمثلاً\b",
            r"\bمثلا\b",
        ],
    },
    "exercise": {
        "label_fa": "تمرین / خودارزیابی / کاربرگ",
        "explicit_patterns": [
            r"\bتمرین\b",
            r"خود\s*ارزیابی",
            r"\bخودارزیابی\b",
            r"خود\s*آزمایی",
            r"\bخودآزمایی\b",
            r"\bکاربرگ\b",
        ],
    },
    "warning": {
        "label_fa": "هشدار / خطر / اجتناب",
        "explicit_patterns": [
            r"\bهشدار\b",
            r"\bخطر\b",
            r"\bریسک\b",
            r"\bمراقب\b",
            r"\bاجتناب\b",
            r"\bناپسندی\b",
            r"\bکلاهبرداری\b",
        ],
    },
    "solution": {
        "label_fa": "راهکار / اقدام",
        "explicit_patterns": [
            r"\bراهکار\b",
            r"\bراه‌کار\b",
            r"\bراه حل\b",
            r"\bراه‌حل\b",
            r"\bگام اول\b",
            r"\bگام دوم\b",
            r"\bگام سوم\b",
            r"\bگام چهارم\b",
            r"\bاقدام\b",
        ],
    },
}


def normalized_text_for_pattern(value):
    text = normalize_fa_for_match(value)
    return text


def pattern_hit_count(text, patterns):
    return sum(
        bool(re.search(pattern, text))
        for pattern in patterns
    )


print("Content Type candidate registry:")
for content_type_id, spec in content_type_seed_v0.items():
    print(
        f"- {content_type_id}: {spec['label_fa']} "
        f"| patterns={len(spec['explicit_patterns'])}"
    )


book_structural_type_counter = Counter()
book_heading_role_counter = Counter()

for record in book_records:
    metadata = record.get("source_metadata", {})

    unit_type = metadata.get("book_unit_type")

    if nonempty(unit_type):
        book_structural_type_counter[
            str(unit_type)
        ] += 1

    if metadata.get("is_heading") is True:
        heading = normalize_fa_for_match(
            record.get("text")
        )

        if not heading:
            continue

        if re.search(
            r"تمرین|خود\s*ارزیابی|خودارزیابی|"
            r"خود\s*آزمایی|خودآزمایی|کاربرگ",
            heading,
        ):
            book_heading_role_counter[
                "explicit_exercise"
            ] += 1

        elif re.search(
            r"منابع آموزشی|منابع بیشتر|مطالعه بیشتر",
            heading,
        ):
            book_heading_role_counter[
                "resource_reference"
            ] += 1

        elif re.search(
            r"مثال|نمونه",
            heading,
        ):
            book_heading_role_counter[
                "explicit_example"
            ] += 1

        else:
            book_heading_role_counter[
                "other_heading"
            ] += 1


print("\nBook structural content evidence:")

print("- book_unit_type values:")
if book_structural_type_counter:
    for value, count in book_structural_type_counter.most_common():
        print(f"  - {value}: {count}")
else:
    print("  - None")

print("- heading role evidence:")
for value, count in book_heading_role_counter.most_common():
    print(f"  - {value}: {count}")


website_content_type_counter = Counter()

for record in website_records:
    value = record.get(
        "source_metadata",
        {},
    ).get("content_type")

    if nonempty(value):
        website_content_type_counter[
            str(value)
        ] += 1


print("\nWebsite explicit source_metadata.content_type:")

if website_content_type_counter:
    for value, count in website_content_type_counter.most_common():
        print(f"- {value}: {count}")
else:
    print("- None")


transcript_record_type_counter = Counter()

for record in transcript_records:
    value = record.get(
        "source_metadata",
        {},
    ).get("record_type")

    if nonempty(value):
        transcript_record_type_counter[
            str(value)
        ] += 1


print("\nTranscript record_type values:")
for value, count in transcript_record_type_counter.most_common():
    print(f"- {value}: {count}")


source_records = {
    "website": website_records,
    "book": book_records,
    "transcript": transcript_records,
}


content_type_evidence_profile = {
    content_type_id: {
        "label_fa": spec["label_fa"],
        "source_document_hits": {},
        "total_document_hits": 0,
    }
    for content_type_id, spec
    in content_type_seed_v0.items()
}


for content_type_id, spec in content_type_seed_v0.items():

    for source_type, source_population in source_records.items():
        hit_count = 0

        for record in source_population:
            text = normalized_text_for_pattern(
                record.get("text")
            )

            title = normalized_text_for_pattern(
                record.get("title")
            )

            combined = f"{title}\n{text}"

            if pattern_hit_count(
                combined,
                spec["explicit_patterns"],
            ) > 0:
                hit_count += 1

        content_type_evidence_profile[
            content_type_id
        ]["source_document_hits"][
            source_type
        ] = hit_count

        content_type_evidence_profile[
            content_type_id
        ]["total_document_hits"] += hit_count


print("\nLexical evidence profile:")

for content_type_id, profile in (
    content_type_evidence_profile.items()
):
    counts = profile["source_document_hits"]

    print(
        f"\n[{content_type_id}] "
        f"{profile['label_fa']}"
    )
    print(
        "- Website documents:",
        counts["website"],
    )
    print(
        "- Book documents:",
        counts["book"],
    )
    print(
        "- Transcript documents:",
        counts["transcript"],
    )
    print(
        "- Total lexical hits:",
        profile["total_document_hits"],
    )


explicit_activity_population = {
    (
        row["work_id"],
        row["paragraph_number"],
    )
    for row in competency_evidence_rows
}


book_explicit_activity_matches = 0

for row in competency_evidence_rows:
    if row["activity_type"] in {
        "self_assessment",
        "worksheet",
        "exercise",
    }:
        book_explicit_activity_matches += 1


print("\nExplicit Book Activity validation:")
print(
    "- Activity evidence rows:",
    len(competency_evidence_rows),
)
print(
    "- Explicit exercise-role rows:",
    book_explicit_activity_matches,
)


weak_only_types = []
strong_structural_types = []


content_type_evidence_strength_v0 = {
    "concept": "MIXED_STRUCTURAL_AND_LEXICAL",
    "example": "LEXICAL_ONLY_PENDING_VALIDATION",
    "exercise": "STRONG_STRUCTURAL_EVIDENCE",
    "warning": "LEXICAL_ONLY_PENDING_VALIDATION",
    "solution": "LEXICAL_ONLY_PENDING_VALIDATION",
}


for content_type_id, strength in (
    content_type_evidence_strength_v0.items()
):
    if strength == "STRONG_STRUCTURAL_EVIDENCE":
        strong_structural_types.append(
            content_type_id
        )

    if strength == "LEXICAL_ONLY_PENDING_VALIDATION":
        weak_only_types.append(
            content_type_id
        )


print("\nEvidence-strength classification:")

for content_type_id, strength in (
    content_type_evidence_strength_v0.items()
):
    print(
        f"- {content_type_id}: {strength}"
    )


print("\nImportant interpretation:")
print(
    "- Source Type, Program, Speaker, and record_type "
    "are not treated as educational Content Type."
)
print(
    "- Lexical matches are evidence candidates only; "
    "they are not final labels."
)
print(
    "- Exercise has direct structural evidence from "
    "Book headings and Activity extraction."
)


print("\nValidation:")
print(
    "- Candidate content types:",
    len(content_type_seed_v0),
)
print(
    "- Strong structural types:",
    len(strong_structural_types),
)
print(
    "- Lexical-only pending types:",
    len(weak_only_types),
)
print(
    "- Explicit Book activities:",
    book_explicit_activity_matches,
)


assert set(content_type_seed_v0) == {
    "concept",
    "example",
    "exercise",
    "warning",
    "solution",
}

assert "exercise" in strong_structural_types

assert book_explicit_activity_matches == 48

assert set(weak_only_types) == {
    "example",
    "warning",
    "solution",
}

assert all(
    profile["total_document_hits"] > 0
    for profile in content_type_evidence_profile.values()
)


print("\nCODE CELL 12 GATE: PASS")
print(
    "Content-Type evidence discovery "
    "completed successfully."
)

Content Type candidate registry:
- concept: مفهوم / آموزش | patterns=7
- example: مثال / نمونه | patterns=5
- exercise: تمرین / خودارزیابی / کاربرگ | patterns=6
- warning: هشدار / خطر / اجتناب | patterns=7
- solution: راهکار / اقدام | patterns=9

Book structural content evidence:
- book_unit_type values:
  - paragraph: 2689
  - supplemental_text: 12
- heading role evidence:
  - other_heading: 118
  - explicit_exercise: 48
  - resource_reference: 14

Website explicit source_metadata.content_type:
- None

Transcript record_type values:
- transcript_segment: 16226

Lexical evidence profile:

[concept] مفهوم / آموزش
- Website documents: 139
- Book documents: 71
- Transcript documents: 377
- Total lexical hits: 587

[example] مثال / نمونه
- Website documents: 118
- Book documents: 93
- Transcript documents: 1866
- Total lexical hits: 2077

[exercise] تمرین / خودارزیابی / کاربرگ
- Website documents: 27
- Book documents: 114
- Transcript documents: 125
- Total lexical hits: 266

[warning] هشد

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی نمونه‌ای Content Typeهای مبتنی بر Lexical Evidence</h3>

<p>
وجود واژه‌هایی مانند «مثال»، «ریسک»، «راهکار» یا «اقدام» به‌تنهایی اثبات نمی‌کند که نقش آموزشی کل Document همان Example، Warning یا Solution است.
</p>

<p>
به‌ویژه در Transcript ممکن است یک واژه فقط در میان یک توضیح طولانی ظاهر شده باشد و Content Type اصلی Segment چیز دیگری باشد.
</p>

<p>
بنابراین پیش از تعریف Rule نهایی، یک نمونه‌گیری Deterministic از Website، Book و Transcript انجام می‌شود تا مشخص شود Matchهای واژه‌ای چه میزان با نقش واقعی محتوا هم‌خوانی دارند.
</p>

<h4>اهداف این مرحله</h4>

<ul>
<li>بررسی نمونه واقعی <code>example</code>، <code>warning</code> و <code>solution</code> در هر سه Source.</li>
<li>نمایش Pattern واقعی که باعث Match شده است.</li>
<li>بررسی هم‌پوشانی چند Content Type در یک Document.</li>
<li>تفکیک Evidence واژه‌ای از Label نهایی.</li>
<li>جلوگیری از ساخت Classifier پرخطا بر پایه Keyword ساده.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>برای هر Content Type ضعیف از هر Source نمونه قابل بررسی استخراج شود.</li>
<li>نمونه‌گیری Deterministic باشد.</li>
<li>Patternهای Match شده قابل مشاهده باشند.</li>
<li>هم‌پوشانی Labelهای Candidate اندازه‌گیری شود.</li>
<li>هیچ Label نهایی روی Dataset نوشته نشود.</li>
</ul>

</div>

In [56]:
# Code Cell 13

import hashlib
from collections import Counter, defaultdict


weak_content_types = [
    "example",
    "warning",
    "solution",
]


def matched_patterns(text, patterns):
    hits = []

    for pattern in patterns:
        if re.search(pattern, text):
            hits.append(pattern)

    return hits


def content_type_matches_for_record(record):
    title = normalized_text_for_pattern(
        record.get("title")
    )

    text = normalized_text_for_pattern(
        record.get("text")
    )

    combined = f"{title}\n{text}"

    matches = {}

    for content_type_id, spec in (
        content_type_seed_v0.items()
    ):
        hits = matched_patterns(
            combined,
            spec["explicit_patterns"],
        )

        if hits:
            matches[content_type_id] = hits

    return matches


def deterministic_rank(record, salt):
    document_id = str(
        record.get("document_id", "")
    )

    payload = (
        f"{salt}|{document_id}"
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


lexical_match_index = defaultdict(
    lambda: defaultdict(list)
)

record_content_type_matches = {}


for source_type, population in source_records.items():

    for record in population:
        document_id = record.get("document_id")

        matches = content_type_matches_for_record(
            record
        )

        record_content_type_matches[
            document_id
        ] = matches

        for content_type_id in matches:
            lexical_match_index[
                content_type_id
            ][
                source_type
            ].append(record)


print("Lexical candidate populations:")

for content_type_id in weak_content_types:
    print(
        f"\n[{content_type_id}] "
        f"{content_type_seed_v0[content_type_id]['label_fa']}"
    )

    for source_type in (
        "website",
        "book",
        "transcript",
    ):
        count = len(
            lexical_match_index[
                content_type_id
            ][source_type]
        )

        print(
            f"- {source_type}: {count}"
        )


sample_size_per_source = 2

review_samples_v0 = []


for content_type_id in weak_content_types:

    for source_type in (
        "website",
        "book",
        "transcript",
    ):

        candidates = lexical_match_index[
            content_type_id
        ][source_type]

        ranked = sorted(
            candidates,
            key=lambda record: deterministic_rank(
                record,
                f"{content_type_id}:{source_type}:v0",
            ),
        )

        selected = ranked[
            :sample_size_per_source
        ]

        for record in selected:
            document_id = record.get(
                "document_id"
            )

            matches = (
                record_content_type_matches[
                    document_id
                ]
            )

            review_samples_v0.append(
                {
                    "content_type_id": content_type_id,
                    "source_type": source_type,
                    "document_id": document_id,
                    "title": record.get("title"),
                    "text": record.get("text", ""),
                    "matched_patterns": matches.get(
                        content_type_id,
                        [],
                    ),
                    "all_candidate_types": sorted(
                        matches
                    ),
                }
            )


print("\nDeterministic review sample:")
print(
    "- Samples:",
    len(review_samples_v0),
)


for index, sample in enumerate(
    review_samples_v0,
    start=1,
):
    excerpt = str(
        sample["text"]
    ).strip()

    excerpt = re.sub(
        r"\s+",
        " ",
        excerpt,
    )

    excerpt = excerpt[:900]

    print(
        "\n========================================"
    )
    print(
        f"Sample {index}"
    )
    print(
        "Candidate Content Type:",
        sample["content_type_id"],
    )
    print(
        "Source:",
        sample["source_type"],
    )
    print(
        "Document ID:",
        sample["document_id"],
    )
    print(
        "Title:",
        sample["title"],
    )
    print(
        "Matched patterns:",
        sample["matched_patterns"],
    )
    print(
        "All candidate Content Types:",
        sample["all_candidate_types"],
    )
    print("Excerpt:")
    print(
        excerpt
        if excerpt
        else "<EMPTY>"
    )


overlap_counter = Counter()

for document_id, matches in (
    record_content_type_matches.items()
):
    if not matches:
        continue

    overlap_counter[
        len(matches)
    ] += 1


print("\nCandidate Content-Type overlap distribution:")

for match_count in sorted(
    overlap_counter
):
    print(
        f"- Documents matching "
        f"{match_count} candidate type(s): "
        f"{overlap_counter[match_count]}"
    )


pair_overlap_counter = Counter()

for matches in record_content_type_matches.values():
    matched_types = sorted(matches)

    for index, type_a in enumerate(
        matched_types
    ):
        for type_b in matched_types[
            index + 1:
        ]:
            pair_overlap_counter[
                (type_a, type_b)
            ] += 1


print("\nMost common Content-Type pair overlaps:")

for (
    type_pair,
    count,
) in pair_overlap_counter.most_common(
    20
):
    print(
        f"- {type_pair[0]} + "
        f"{type_pair[1]}: {count}"
    )


weak_type_overlap_profile = {}

for content_type_id in weak_content_types:

    population = []

    for source_type in source_records:
        population.extend(
            lexical_match_index[
                content_type_id
            ][source_type]
        )

    multi_match = 0
    single_match = 0

    for record in population:
        matches = (
            record_content_type_matches[
                record.get("document_id")
            ]
        )

        if len(matches) == 1:
            single_match += 1
        else:
            multi_match += 1

    weak_type_overlap_profile[
        content_type_id
    ] = {
        "population": len(population),
        "single_candidate_only": single_match,
        "multi_candidate": multi_match,
    }


print("\nWeak Content-Type overlap profile:")

for content_type_id, profile in (
    weak_type_overlap_profile.items()
):
    population = profile["population"]

    overlap_rate = (
        profile["multi_candidate"]
        / population
        if population
        else 0.0
    )

    print(
        f"- {content_type_id}: "
        f"population={population} "
        f"| single={profile['single_candidate_only']} "
        f"| multi={profile['multi_candidate']} "
        f"| overlap_rate={overlap_rate:.1%}"
    )


sample_counts = Counter(
    (
        sample["content_type_id"],
        sample["source_type"],
    )
    for sample in review_samples_v0
)


missing_sample_groups = []

for content_type_id in weak_content_types:
    for source_type in (
        "website",
        "book",
        "transcript",
    ):
        if sample_counts[
            (
                content_type_id,
                source_type,
            )
        ] != sample_size_per_source:
            missing_sample_groups.append(
                (
                    content_type_id,
                    source_type,
                )
            )


print("\nValidation:")
print(
    "- Expected samples:",
    len(weak_content_types)
    * 3
    * sample_size_per_source,
)
print(
    "- Actual samples:",
    len(review_samples_v0),
)
print(
    "- Missing sample groups:",
    len(missing_sample_groups),
)
print(
    "- Pair-overlap combinations:",
    len(pair_overlap_counter),
)


assert len(review_samples_v0) == (
    len(weak_content_types)
    * 3
    * sample_size_per_source
)

assert not missing_sample_groups

assert all(
    len(
        lexical_match_index[
            content_type_id
        ][source_type]
    ) > 0
    for content_type_id in weak_content_types
    for source_type in (
        "website",
        "book",
        "transcript",
    )
)

print("\nCODE CELL 13 GATE: PASS")
print(
    "Deterministic lexical Content-Type "
    "sample validation set generated successfully."
)

Lexical candidate populations:

[example] مثال / نمونه
- website: 118
- book: 93
- transcript: 1866

[warning] هشدار / خطر / اجتناب
- website: 125
- book: 76
- transcript: 410

[solution] راهکار / اقدام
- website: 84
- book: 48
- transcript: 148

Deterministic review sample:
- Samples: 18

Sample 1
Candidate Content Type: example
Source: website
Document ID: website:848D005A
Title: قَهر
Matched patterns: ['\\bمثال\\b', '\\bبرای مثال\\b']
All candidate Content Types: ['concept', 'example']
Excerpt:
هنوز مردم فکر می‌کنند، پولداران ظالم‌اند. مضاف بر اینکه سیطره‌ی جریان چپ، در طول صد سال گذشته بر شعبه شعبه کردن فرهنگ ما مؤثر بوده است. هنوز اثرات آن تفکر، به نویسندگان اجازه نمی‌دهد از فایده تولید ثروت، کارآفرینی، آموزش حقوق مالکیت و ... صحبت کنند. تفکرات و اصول جریان چپ، به‌شدت در ایران و برخی کشورها نهادینه ‌شده است؛ برای مثال در کشور ما تصویر خوبی از ثروتمندان وجود ندارد درحالی‌که در آمریکا بهترین افراد کشور، ثروتمندان هستند. نویسندگان ترجیح می‌دهند بیشتر در مورد فقر بنویسند چون مبارزه با

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعریف سیاست Multi-role برای Content Type و کاهش False Positive</h3>

<p>
Sample Review نشان داد که Content Typeهای آموزشی الزاماً متقابلاً انحصاری نیستند.
یک Document می‌تواند همزمان نقش مفهومی، مثالی، هشداردهنده یا راهکارمحور داشته باشد.
</p>

<p>
بنابراین Content Type در این پروژه به‌صورت مجموعه‌ای از نقش‌های آموزشی تعریف می‌شود و هر نقش باید همراه با Evidence Strength ذخیره شود.
</p>

<h4>سیاست Candidate</h4>

<ul>
<li><strong>Exercise:</strong> فقط با Evidence ساختاری قوی از Heading و محدوده Activity کتاب.</li>
<li><strong>Example:</strong> عبارت صریح مثال یا نمونه، به‌عنوان Supporting Educational Role.</li>
<li><strong>Warning:</strong> فقط نشانه‌های صریح هشدار، کلاهبرداری، پرهیز یا منع؛ واژه عمومی «ریسک» یا «خطر» به‌تنهایی کافی نیست.</li>
<li><strong>Solution:</strong> عبارت صریح راهکار یا راه‌حل؛ واژه عمومی «اقدام» به‌تنهایی کافی نیست.</li>
<li><strong>Concept:</strong> نشانه‌های صریح تعریف، چیستی، ماهیت یا توضیح مفهومی؛ نه Default برای هر Document دیگر.</li>
</ul>

<p>
این مرحله Candidate Role تولید می‌کند، نه Final Human-approved Label.
هیچ Artifactی هنوز روی Disk نوشته نمی‌شود.
</p>

<h4>اصل مهم</h4>

<p>
نبود Content Role صریح به معنی بی‌ارزش بودن Document نیست.
ممکن است نقش واقعی یک متن فقط پس از Contextual Classification یا Expert Review مشخص شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Activityهای ساختاری کتاب با نقش Exercise شناسایی شوند.</li>
<li>Ruleهای پرخطر «ریسک = هشدار» و «اقدام = راهکار» حذف شوند.</li>
<li>Multi-role بودن مدل حفظ شود.</li>
<li>هر Candidate Role دارای Evidence Tier باشد.</li>
<li>هیچ Document بدون Evidence مجبور به دریافت Content Type نشود.</li>
</ul>

</div>

In [57]:
# Code Cell 14

from collections import Counter, defaultdict
import re


content_role_policy_v0 = {
    "label_mode": "MULTI_ROLE",
    "unlabeled_allowed": True,
    "final_label_requires_validation": True,

    "roles": {
        "concept": {
            "evidence_tier": "LEXICAL_CANDIDATE",
            "patterns": [
                r"\bتعریف\b",
                r"\bمفهوم\b",
                r"\bچیستی\b",
                r"\bماهیت\b",
                r"\bعبارت است از\b",
                r"\bیعنی\b",
            ],
        },

        "example": {
            "evidence_tier": "LEXICAL_SUPPORTING",
            "patterns": [
                r"\bبرای مثال\b",
                r"\bبه عنوان مثال\b",
                r"\bمثلاً\b",
                r"\bمثلا\b",
                r"\bمثال\b",
                r"\bنمونه\b",
            ],
        },

        "warning": {
            "evidence_tier": "LEXICAL_CANDIDATE",
            "patterns": [
                r"\bهشدار\b",
                r"\bکلاهبرداری\b",
                r"\bمراقب باشید\b",
                r"\bپرهیز\b",
                r"\bاجتناب کنید\b",
                r"\bخودداری کنید\b",
                r"\bنباید\b",
            ],
        },

        "solution": {
            "evidence_tier": "LEXICAL_CANDIDATE",
            "patterns": [
                r"\bراهکار\b",
                r"\bراه‌کار\b",
                r"\bراه حل\b",
                r"\bراه‌حل\b",
                r"\bراه چاره\b",
            ],
        },

        "exercise": {
            "evidence_tier": "STRUCTURAL_STRONG",
            "patterns": [],
        },
    },

    "explicitly_rejected_shortcuts": {
        "warning": [
            "risk_word_alone",
            "danger_word_alone",
        ],
        "solution": [
            "action_word_alone",
            "step_word_alone",
        ],
    },
}


print("Content Role policy:")
print(
    "- Label mode:",
    content_role_policy_v0["label_mode"],
)
print(
    "- Unlabeled allowed:",
    content_role_policy_v0[
        "unlabeled_allowed"
    ],
)
print(
    "- Final label requires validation:",
    content_role_policy_v0[
        "final_label_requires_validation"
    ],
)


for role_id, spec in content_role_policy_v0[
    "roles"
].items():
    print(
        f"- {role_id}: "
        f"{spec['evidence_tier']} "
        f"| patterns={len(spec['patterns'])}"
    )


def find_record_index_by_paragraph(
    work_records,
    paragraph_number,
):
    for index, record in enumerate(work_records):
        if (
            paragraph_number_of(record)
            == paragraph_number
        ):
            return index

    return None


book_activity_document_ids = set()
book_activity_spans = []


for activity in competency_evidence_rows:

    work_id = activity["work_id"]
    work_records = books_by_work[work_id]

    start_index = find_record_index_by_paragraph(
        work_records,
        activity["paragraph_number"],
    )

    if start_index is None:
        continue

    activity_level = activity[
        "heading_level"
    ]

    span_document_ids = []

    for index in range(
        start_index,
        len(work_records),
    ):
        record = work_records[index]

        if index > start_index:
            metadata = record.get(
                "source_metadata",
                {},
            )

            if metadata.get("is_heading") is True:
                current_level = heading_level_of(
                    record
                )

                if (
                    activity_level is None
                    or current_level is None
                    or current_level <= activity_level
                ):
                    break

        document_id = record.get(
            "document_id"
        )

        if document_id:
            span_document_ids.append(
                document_id
            )
            book_activity_document_ids.add(
                document_id
            )

    book_activity_spans.append(
        {
            "work_id": work_id,
            "activity_heading": activity[
                "heading"
            ],
            "activity_paragraph_number": activity[
                "paragraph_number"
            ],
            "document_count": len(
                span_document_ids
            ),
            "document_ids": span_document_ids,
        }
    )


print("\nStructural Exercise evidence:")
print(
    "- Activity spans:",
    len(book_activity_spans),
)
print(
    "- Unique Book documents inside "
    "Activity spans:",
    len(book_activity_document_ids),
)


empty_activity_spans = [
    span
    for span in book_activity_spans
    if not span["document_ids"]
]


print(
    "- Empty Activity spans:",
    len(empty_activity_spans),
)


candidate_content_roles_v0 = {}


for source_type, population in source_records.items():

    for record in population:
        document_id = record.get(
            "document_id"
        )

        title = normalized_text_for_pattern(
            record.get("title")
        )

        text = normalized_text_for_pattern(
            record.get("text")
        )

        combined = (
            f"{title}\n{text}"
        )

        roles = []

        if document_id in book_activity_document_ids:
            roles.append(
                {
                    "role": "exercise",
                    "evidence_tier": (
                        "STRUCTURAL_STRONG"
                    ),
                    "evidence": (
                        "BOOK_ACTIVITY_SPAN"
                    ),
                }
            )

        for role_id in (
            "concept",
            "example",
            "warning",
            "solution",
        ):
            spec = content_role_policy_v0[
                "roles"
            ][role_id]

            hits = [
                pattern
                for pattern in spec["patterns"]
                if re.search(
                    pattern,
                    combined,
                )
            ]

            if not hits:
                continue

            roles.append(
                {
                    "role": role_id,
                    "evidence_tier": spec[
                        "evidence_tier"
                    ],
                    "matched_patterns": hits,
                }
            )

        candidate_content_roles_v0[
            document_id
        ] = {
            "source_type": source_type,
            "roles": roles,
        }


role_document_counter = Counter()
role_by_source_counter = defaultdict(
    Counter
)

unlabeled_by_source = Counter()
multi_role_by_source = Counter()


for document_id, profile in (
    candidate_content_roles_v0.items()
):

    source_type = profile[
        "source_type"
    ]

    roles = [
        row["role"]
        for row in profile["roles"]
    ]

    if not roles:
        unlabeled_by_source[
            source_type
        ] += 1
        continue

    for role_id in set(roles):
        role_document_counter[
            role_id
        ] += 1

        role_by_source_counter[
            role_id
        ][source_type] += 1

    if len(set(roles)) > 1:
        multi_role_by_source[
            source_type
        ] += 1


print("\nCandidate role populations:")

for role_id in (
    "concept",
    "example",
    "exercise",
    "warning",
    "solution",
):
    print(
        f"\n[{role_id}]"
    )

    print(
        "- Total documents:",
        role_document_counter[
            role_id
        ],
    )

    for source_type in (
        "website",
        "book",
        "transcript",
    ):
        print(
            f"- {source_type}:",
            role_by_source_counter[
                role_id
            ][source_type],
        )


print("\nUnlabeled documents are allowed:")

for source_type in (
    "website",
    "book",
    "transcript",
):
    population_size = len(
        source_records[source_type]
    )

    unlabeled = unlabeled_by_source[
        source_type
    ]

    print(
        f"- {source_type}: "
        f"{unlabeled}/{population_size} "
        f"({unlabeled / population_size:.1%})"
    )


print("\nMulti-role candidate documents:")

for source_type in (
    "website",
    "book",
    "transcript",
):
    print(
        f"- {source_type}:",
        multi_role_by_source[
            source_type
        ],
    )


role_set_counter = Counter()

for profile in candidate_content_roles_v0.values():
    role_set = tuple(
        sorted({
            role["role"]
            for role in profile["roles"]
        })
    )

    if role_set:
        role_set_counter[
            role_set
        ] += 1


print("\nMost common Candidate role combinations:")

for role_set, count in role_set_counter.most_common(
    20
):
    print(
        "-",
        " + ".join(role_set),
        ":",
        count,
    )


warning_patterns = content_role_policy_v0[
    "roles"
]["warning"]["patterns"]

solution_patterns = content_role_policy_v0[
    "roles"
]["solution"]["patterns"]


forbidden_warning_shortcuts = [
    r"\bریسک\b",
    r"\bخطر\b",
]

forbidden_solution_shortcuts = [
    r"\bاقدام\b",
    r"\bگام اول\b",
    r"\bگام دوم\b",
    r"\bگام سوم\b",
    r"\bگام چهارم\b",
]


warning_shortcut_leakage = [
    pattern
    for pattern in forbidden_warning_shortcuts
    if pattern in warning_patterns
]

solution_shortcut_leakage = [
    pattern
    for pattern in forbidden_solution_shortcuts
    if pattern in solution_patterns
]


invalid_evidence_tiers = []

allowed_evidence_tiers = {
    "STRUCTURAL_STRONG",
    "LEXICAL_CANDIDATE",
    "LEXICAL_SUPPORTING",
}


for profile in candidate_content_roles_v0.values():
    for role in profile["roles"]:
        if (
            role["evidence_tier"]
            not in allowed_evidence_tiers
        ):
            invalid_evidence_tiers.append(
                role
            )


activity_heading_document_ids = set()

for activity in competency_evidence_rows:
    work_records = books_by_work[
        activity["work_id"]
    ]

    index = find_record_index_by_paragraph(
        work_records,
        activity["paragraph_number"],
    )

    if index is not None:
        document_id = work_records[
            index
        ].get("document_id")

        if document_id:
            activity_heading_document_ids.add(
                document_id
            )


activity_headings_without_exercise_role = []

for document_id in (
    activity_heading_document_ids
):
    roles = {
        row["role"]
        for row in candidate_content_roles_v0[
            document_id
        ]["roles"]
    }

    if "exercise" not in roles:
        activity_headings_without_exercise_role.append(
            document_id
        )


print("\nValidation:")
print(
    "- Activity heading documents:",
    len(activity_heading_document_ids),
)
print(
    "- Activity headings missing Exercise role:",
    len(
        activity_headings_without_exercise_role
    ),
)
print(
    "- Warning shortcut leakage:",
    len(warning_shortcut_leakage),
)
print(
    "- Solution shortcut leakage:",
    len(solution_shortcut_leakage),
)
print(
    "- Invalid evidence tiers:",
    len(invalid_evidence_tiers),
)
print(
    "- Documents with at least one Candidate role:",
    sum(
        bool(profile["roles"])
        for profile
        in candidate_content_roles_v0.values()
    ),
)
print(
    "- Documents with no forced role:",
    sum(
        not profile["roles"]
        for profile
        in candidate_content_roles_v0.values()
    ),
)


assert (
    content_role_policy_v0[
        "label_mode"
    ]
    == "MULTI_ROLE"
)

assert content_role_policy_v0[
    "unlabeled_allowed"
] is True

assert len(book_activity_spans) == 48

assert not empty_activity_spans

assert not (
    activity_headings_without_exercise_role
)

assert not warning_shortcut_leakage

assert not solution_shortcut_leakage

assert not invalid_evidence_tiers

assert role_document_counter[
    "exercise"
] >= 48


print("\nCODE CELL 14 GATE: PASS")
print(
    "Evidence-tiered Multi-role Content Type "
    "policy validated successfully."
)

Content Role policy:
- Label mode: MULTI_ROLE
- Unlabeled allowed: True
- Final label requires validation: True
- concept: LEXICAL_CANDIDATE | patterns=6
- example: LEXICAL_SUPPORTING | patterns=6
- warning: LEXICAL_CANDIDATE | patterns=7
- solution: LEXICAL_CANDIDATE | patterns=5
- exercise: STRUCTURAL_STRONG | patterns=0

Structural Exercise evidence:
- Activity spans: 48
- Unique Book documents inside Activity spans: 762
- Empty Activity spans: 0

Candidate role populations:

[concept]
- Total documents: 2252
- website: 138
- book: 75
- transcript: 2039

[example]
- Total documents: 2077
- website: 118
- book: 93
- transcript: 1866

[exercise]
- Total documents: 762
- website: 0
- book: 762
- transcript: 0

[warning]
- Total documents: 345
- website: 79
- book: 7
- transcript: 259

[solution]
- Total documents: 137
- website: 45
- book: 9
- transcript: 83

Unlabeled documents are allowed:
- website: 251/529 (47.4%)
- book: 1801/2701 (66.7%)
- transcript: 12671/16226 (78.1%)

Multi-r

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Candidate Document → Topic Mapping با Evidence و Confidence</h3>

<p>
تا این مرحله Topic Taxonomy، Competency Model، Learning Progression و Content Role Policy به‌صورت Candidate طراحی شده‌اند.
اکنون باید Documentهای پذیرفته‌شده Notebook 09 به Topicهای Candidate متصل شوند.
</p>

<p>
هدف این Mapping تولید برچسب نهایی نیست.
هر نگاشت باید قابل توضیح باشد و مشخص کند بر اساس چه Evidenceای پیشنهاد شده است.
</p>

<h4>Evidenceهای مورد استفاده</h4>

<ul>
<li><strong>Website Tag:</strong> تطبیق Tag واقعی Website با Evidence Termهای Topic.</li>
<li><strong>Book Heading Path:</strong> حضور Topic در ساختار Heading کتاب.</li>
<li><strong>Book Activity Context:</strong> قرار گرفتن Document در محدوده Activityای که Topic آن با Evidence مشخص شده است.</li>
<li><strong>Heading Record:</strong> Heading صریحی که مستقیماً به Topic اشاره دارد.</li>
<li><strong>Title:</strong> وجود عبارت موضوعی در Title.</li>
<li><strong>Body Text:</strong> وجود یک یا چند Evidence Term در متن Document.</li>
</ul>

<h4>سطوح Candidate</h4>

<ul>
<li><strong>STRONG_CANDIDATE:</strong> دارای Evidence ساختاری یا Metadata قوی.</li>
<li><strong>LEXICAL_CANDIDATE:</strong> بر پایه Evidence متنی و نیازمند Validation بعدی.</li>
<li><strong>UNMAPPED:</strong> Evidence کافی برای هیچ Topic پیدا نشده است.</li>
</ul>

<p>
یک Document می‌تواند بیش از یک Topic Candidate داشته باشد.
این موضوع به معنی خطا نیست، زیرا یک بخش آموزشی ممکن است چند مفهوم مرتبط را همزمان پوشش دهد.
</p>

<h4>Authority</h4>

<p>
فیلد <code>content_authority_type</code> دقیقاً از Dataset موجود حمل می‌شود.
اگر مقدار آن <code>null</code> باشد، در این Notebook از روی Author، Speaker یا Knowledge Owner مقدار جدیدی حدس زده نمی‌شود.
Topic Mapping و Knowledge Authority دو Dimension مستقل باقی می‌مانند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 19,456 Document بررسی شوند.</li>
<li>هر Mapping دارای Topic ID، Confidence و Reason باشد.</li>
<li>Strong Candidate فقط با Evidence کافی ایجاد شود.</li>
<li>Text-only Mapping از Strong Mapping جدا بماند.</li>
<li>Authority موجود بدون تغییر حفظ شود.</li>
<li>Document بدون Evidence مجبور به دریافت Topic نشود.</li>
<li>هیچ Artifactی هنوز روی Disk نوشته نشود.</li>
</ul>

</div>

In [58]:
# Code Cell 15

from collections import Counter, defaultdict
import re


topic_terms_normalized = {
    topic_id: sorted(
        {
            normalize_fa_for_match(term)
            for term in topic["evidence_terms"]
            if normalize_fa_for_match(term)
        },
        key=len,
        reverse=True,
    )
    for topic_id, topic in topic_seed_v0.items()
}


def matched_topic_terms(text, topic_id):
    normalized = normalize_fa_for_match(text)

    if not normalized:
        return []

    return [
        term
        for term in topic_terms_normalized[topic_id]
        if len(term) >= 3 and term in normalized
    ]


def flatten_heading_path_for_mapping(value):
    if isinstance(value, list):
        return " | ".join(
            str(item)
            for item in value
            if nonempty(item)
        )

    if isinstance(value, str):
        return value

    return ""


activity_topic_by_key = {}

for row in competency_evidence_rows:
    key = (
        row["work_id"],
        row["paragraph_number"],
    )

    topic_ids = list(
        row["topic_ids"]
    )

    if not topic_ids:
        refinement = taxonomy_refinement_review_v0.get(
            key
        )

        if (
            refinement
            and refinement["disposition"]
            == "MAPPED_CANDIDATE"
            and refinement["topic_id"]
        ):
            topic_ids = [
                refinement["topic_id"]
            ]

    activity_topic_by_key[key] = sorted(
        set(topic_ids)
    )


activity_span_topic_lookup = defaultdict(set)

for span in book_activity_spans:
    key = (
        span["work_id"],
        span["activity_paragraph_number"],
    )

    topic_ids = activity_topic_by_key.get(
        key,
        [],
    )

    for document_id in span["document_ids"]:
        for topic_id in topic_ids:
            activity_span_topic_lookup[
                document_id
            ].add(topic_id)


print("Activity-context Topic evidence:")
print(
    "- Documents with mapped Activity Topic context:",
    len(activity_span_topic_lookup),
)


def website_tag_names_for_record(record):
    tags = record.get(
        "source_metadata",
        {},
    ).get("tags")

    if not nonempty(tags):
        return []

    if not isinstance(tags, list):
        tags = [tags]

    values = []

    for tag in tags:
        name = website_tag_name(tag)

        if name:
            values.append(name)

    return values


def calculate_topic_candidate(record, topic_id):
    source_type = record.get(
        "source_type"
    )

    document_id = record.get(
        "document_id"
    )

    evidence = []
    score = 0.0
    structural_evidence = False

    topic_terms = topic_terms_normalized[
        topic_id
    ]

    title = record.get("title", "")
    text = record.get("text", "")

    title_hits = matched_topic_terms(
        title,
        topic_id,
    )

    text_hits = matched_topic_terms(
        text,
        topic_id,
    )

    if source_type == "website":
        tag_names = website_tag_names_for_record(
            record
        )

        matched_tags = []

        for tag_name in tag_names:
            normalized_tag = normalize_fa_for_match(
                tag_name
            )

            if normalized_tag in topic_terms:
                matched_tags.append(
                    tag_name
                )

        if matched_tags:
            score += 0.70
            structural_evidence = True

            evidence.append(
                {
                    "type": "WEBSITE_TAG_EXACT",
                    "weight": 0.70,
                    "values": sorted(
                        set(matched_tags)
                    ),
                }
            )

    if source_type == "book":
        locator = record.get(
            "locator",
            {},
        )

        metadata = record.get(
            "source_metadata",
            {},
        )

        heading_path = (
            locator.get("heading_path")
            or metadata.get("heading_path")
        )

        heading_text = (
            flatten_heading_path_for_mapping(
                heading_path
            )
        )

        heading_hits = matched_topic_terms(
            heading_text,
            topic_id,
        )

        if heading_hits:
            score += 0.70
            structural_evidence = True

            evidence.append(
                {
                    "type": "BOOK_HEADING_PATH",
                    "weight": 0.70,
                    "values": heading_hits,
                }
            )

        if topic_id in activity_span_topic_lookup.get(
            document_id,
            set(),
        ):
            score += 0.75
            structural_evidence = True

            evidence.append(
                {
                    "type": "BOOK_ACTIVITY_TOPIC_CONTEXT",
                    "weight": 0.75,
                    "values": [topic_id],
                }
            )

        if metadata.get("is_heading") is True:
            heading_record_hits = matched_topic_terms(
                text,
                topic_id,
            )

            if heading_record_hits:
                score += 0.65
                structural_evidence = True

                evidence.append(
                    {
                        "type": "BOOK_HEADING_RECORD",
                        "weight": 0.65,
                        "values": heading_record_hits,
                    }
                )

    if title_hits:
        score += 0.40

        evidence.append(
            {
                "type": "TITLE_LEXICAL",
                "weight": 0.40,
                "values": title_hits,
            }
        )

    distinct_text_hits = sorted(
        set(text_hits)
    )

    if len(distinct_text_hits) >= 3:
        body_weight = 0.55
    elif len(distinct_text_hits) == 2:
        body_weight = 0.40
    elif len(distinct_text_hits) == 1:
        body_weight = 0.25
    else:
        body_weight = 0.0

    if body_weight:
        score += body_weight

        evidence.append(
            {
                "type": "BODY_LEXICAL",
                "weight": body_weight,
                "values": distinct_text_hits,
            }
        )

    if not evidence:
        return None

    confidence = min(
        round(score, 3),
        0.99,
    )

    if structural_evidence and confidence >= 0.70:
        decision = "STRONG_CANDIDATE"
    else:
        decision = "LEXICAL_CANDIDATE"

    return {
        "topic_id": topic_id,
        "topic_label_fa": topic_seed_v0[
            topic_id
        ]["label_fa"],
        "decision": decision,
        "confidence": confidence,
        "reason": [
            item["type"]
            for item in evidence
        ],
        "evidence": evidence,
    }


document_topic_candidates_v0 = {}


for record in records:
    document_id = record[
        "document_id"
    ]

    candidates = []

    for topic_id in topic_seed_v0:
        candidate = calculate_topic_candidate(
            record,
            topic_id,
        )

        if candidate is not None:
            candidates.append(
                candidate
            )

    candidates.sort(
        key=lambda item: (
            -item["confidence"],
            item["topic_id"],
        )
    )

    attribution = record.get(
        "attribution",
        {},
    )

    document_topic_candidates_v0[
        document_id
    ] = {
        "document_id": document_id,
        "source_type": record.get(
            "source_type"
        ),
        "content_authority_type": attribution.get(
            "content_authority_type"
        ),
        "author": attribution.get(
            "author"
        ),
        "speaker": attribution.get(
            "speaker"
        ),
        "knowledge_owner": attribution.get(
            "knowledge_owner"
        ),
        "candidates": candidates,
    }


print("\nDocument Topic Mapping population:")
print(
    "- Documents evaluated:",
    len(document_topic_candidates_v0),
)


mapping_status_counter = Counter()
candidate_count_distribution = Counter()

topic_decision_counter = defaultdict(
    Counter
)

source_mapping_counter = defaultdict(
    Counter
)

authority_presence_counter = Counter()


for profile in document_topic_candidates_v0.values():

    candidates = profile[
        "candidates"
    ]

    source_type = profile[
        "source_type"
    ]

    candidate_count_distribution[
        len(candidates)
    ] += 1

    if not candidates:
        mapping_status_counter[
            "UNMAPPED"
        ] += 1

        source_mapping_counter[
            source_type
        ]["UNMAPPED"] += 1

    else:
        strong_count = sum(
            candidate["decision"]
            == "STRONG_CANDIDATE"
            for candidate in candidates
        )

        lexical_count = sum(
            candidate["decision"]
            == "LEXICAL_CANDIDATE"
            for candidate in candidates
        )

        if strong_count:
            mapping_status_counter[
                "HAS_STRONG_CANDIDATE"
            ] += 1

            source_mapping_counter[
                source_type
            ]["HAS_STRONG_CANDIDATE"] += 1

        elif lexical_count:
            mapping_status_counter[
                "LEXICAL_ONLY"
            ] += 1

            source_mapping_counter[
                source_type
            ]["LEXICAL_ONLY"] += 1

        for candidate in candidates:
            topic_decision_counter[
                candidate["topic_id"]
            ][
                candidate["decision"]
            ] += 1

    authority_value = profile[
        "content_authority_type"
    ]

    authority_key = (
        str(authority_value)
        if nonempty(authority_value)
        else "<null>"
    )

    authority_presence_counter[
        authority_key
    ] += 1


print("\nMapping status:")

for status, count in mapping_status_counter.items():
    print(
        f"- {status}: "
        f"{count}/{len(records)} "
        f"({count / len(records):.1%})"
    )


print("\nMapping status by Source:")

for source_type in (
    "website",
    "book",
    "transcript",
):
    population = len(
        source_records[source_type]
    )

    print(f"\n[{source_type}]")

    for status in (
        "HAS_STRONG_CANDIDATE",
        "LEXICAL_ONLY",
        "UNMAPPED",
    ):
        count = source_mapping_counter[
            source_type
        ][status]

        print(
            f"- {status}: "
            f"{count}/{population} "
            f"({count / population:.1%})"
        )


print("\nCandidate count distribution:")

for candidate_count in sorted(
    candidate_count_distribution
):
    count = candidate_count_distribution[
        candidate_count
    ]

    print(
        f"- {candidate_count} Topic candidate(s): "
        f"{count}"
    )


print("\nTopic candidate populations:")

for topic_id in topic_seed_v0:
    counter = topic_decision_counter[
        topic_id
    ]

    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"strong={counter['STRONG_CANDIDATE']} "
        f"| lexical={counter['LEXICAL_CANDIDATE']}"
    )


print("\nExisting content_authority_type values:")

for value, count in authority_presence_counter.most_common():
    print(
        f"- {value}: {count}"
    )


strong_examples = []
lexical_examples = []
unmapped_examples = []


for record in records:
    profile = document_topic_candidates_v0[
        record["document_id"]
    ]

    candidates = profile[
        "candidates"
    ]

    if (
        candidates
        and candidates[0]["decision"]
        == "STRONG_CANDIDATE"
        and len(strong_examples) < 12
    ):
        strong_examples.append(
            (
                record,
                profile,
            )
        )

    elif (
        candidates
        and all(
            candidate["decision"]
            == "LEXICAL_CANDIDATE"
            for candidate in candidates
        )
        and len(lexical_examples) < 12
    ):
        lexical_examples.append(
            (
                record,
                profile,
            )
        )

    elif (
        not candidates
        and len(unmapped_examples) < 12
    ):
        unmapped_examples.append(
            (
                record,
                profile,
            )
        )


def print_mapping_examples(
    title,
    examples,
):
    print(f"\n{title}:")

    for record, profile in examples:
        print(
            "\n----------------------------------------"
        )

        print(
            "Document:",
            profile["document_id"],
        )

        print(
            "Source:",
            profile["source_type"],
        )

        print(
            "Title:",
            record.get("title"),
        )

        print(
            "content_authority_type:",
            profile[
                "content_authority_type"
            ],
        )

        if not profile["candidates"]:
            print(
                "Candidates: NONE"
            )
            continue

        for candidate in profile[
            "candidates"
        ][:3]:
            print(
                f"- {candidate['topic_label_fa']} "
                f"| {candidate['decision']} "
                f"| confidence={candidate['confidence']:.2f} "
                f"| reason={candidate['reason']}"
            )


print_mapping_examples(
    "Representative Strong mappings",
    strong_examples,
)

print_mapping_examples(
    "Representative Lexical-only mappings",
    lexical_examples,
)

print_mapping_examples(
    "Representative Unmapped documents",
    unmapped_examples,
)


invalid_topic_ids = []
missing_reason_rows = []
invalid_confidence_rows = []
strong_without_structural_evidence = []


for profile in document_topic_candidates_v0.values():

    for candidate in profile[
        "candidates"
    ]:

        if candidate["topic_id"] not in topic_seed_v0:
            invalid_topic_ids.append(
                candidate["topic_id"]
            )

        if not candidate["reason"]:
            missing_reason_rows.append(
                (
                    profile["document_id"],
                    candidate["topic_id"],
                )
            )

        if not (
            0.0
            < candidate["confidence"]
            <= 0.99
        ):
            invalid_confidence_rows.append(
                (
                    profile["document_id"],
                    candidate["confidence"],
                )
            )

        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        ):
            evidence_types = {
                item["type"]
                for item in candidate[
                    "evidence"
                ]
            }

            structural_types = {
                "WEBSITE_TAG_EXACT",
                "BOOK_HEADING_PATH",
                "BOOK_ACTIVITY_TOPIC_CONTEXT",
                "BOOK_HEADING_RECORD",
            }

            if not (
                evidence_types
                & structural_types
            ):
                strong_without_structural_evidence.append(
                    (
                        profile[
                            "document_id"
                        ],
                        candidate[
                            "topic_id"
                        ],
                    )
                )


authority_preservation_mismatches = []

record_lookup = {
    record["document_id"]: record
    for record in records
}


for document_id, profile in (
    document_topic_candidates_v0.items()
):
    original = record_lookup[
        document_id
    ].get(
        "attribution",
        {},
    ).get(
        "content_authority_type"
    )

    if (
        profile["content_authority_type"]
        != original
    ):
        authority_preservation_mismatches.append(
            document_id
        )


print("\nValidation:")
print(
    "- Total mappings evaluated:",
    len(document_topic_candidates_v0),
)
print(
    "- Invalid Topic IDs:",
    len(invalid_topic_ids),
)
print(
    "- Missing Reason rows:",
    len(missing_reason_rows),
)
print(
    "- Invalid Confidence rows:",
    len(invalid_confidence_rows),
)
print(
    "- Strong mappings without structural evidence:",
    len(
        strong_without_structural_evidence
    ),
)
print(
    "- content_authority_type preservation mismatches:",
    len(
        authority_preservation_mismatches
    ),
)
print(
    "- Unmapped documents retained:",
    mapping_status_counter[
        "UNMAPPED"
    ],
)


assert len(
    document_topic_candidates_v0
) == 19456

assert not invalid_topic_ids

assert not missing_reason_rows

assert not invalid_confidence_rows

assert not (
    strong_without_structural_evidence
)

assert not (
    authority_preservation_mismatches
)

assert (
    sum(
        mapping_status_counter.values()
    )
    == 19456
)


print("\nCODE CELL 15 GATE: PASS")
print(
    "Explainable Candidate Document-to-Topic "
    "mapping completed successfully."
)

Activity-context Topic evidence:
- Documents with mapped Activity Topic context: 760

Document Topic Mapping population:
- Documents evaluated: 19456

Mapping status:
- LEXICAL_ONLY: 3280/19456 (16.9%)
- HAS_STRONG_CANDIDATE: 1985/19456 (10.2%)
- UNMAPPED: 14191/19456 (72.9%)

Mapping status by Source:

[website]
- HAS_STRONG_CANDIDATE: 340/529 (64.3%)
- LEXICAL_ONLY: 124/529 (23.4%)
- UNMAPPED: 65/529 (12.3%)

[book]
- HAS_STRONG_CANDIDATE: 1645/2701 (60.9%)
- LEXICAL_ONLY: 330/2701 (12.2%)
- UNMAPPED: 726/2701 (26.9%)

[transcript]
- HAS_STRONG_CANDIDATE: 0/16226 (0.0%)
- LEXICAL_ONLY: 2826/16226 (17.4%)
- UNMAPPED: 13400/16226 (82.6%)

Candidate count distribution:
- 0 Topic candidate(s): 14191
- 1 Topic candidate(s): 3305
- 2 Topic candidate(s): 1597
- 3 Topic candidate(s): 251
- 4 Topic candidate(s): 60
- 5 Topic candidate(s): 21
- 6 Topic candidate(s): 25
- 7 Topic candidate(s): 6

Topic candidate populations:
- مبانی و توانمندی سواد مالی: strong=276 | lexical=1363
- خرج، پس‌اندا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>نمونه‌گیری لایه‌بندی‌شده برای Expert Review و کشف Gapهای Taxonomy</h3>

<p>
Candidate Mapping اولیه نشان داد که Website و Book پوشش ساختاری مناسبی دارند، اما بیشتر Transcriptها یا فقط Lexical Candidate هستند یا هنوز Topic ندارند.
این وضعیت نباید با اضافه‌کردن فوری Keyword یا Topic جدید اصلاح شود.
</p>

<p>
ابتدا باید یک نمونه‌گیری Deterministic و لایه‌بندی‌شده انجام شود تا کیفیت واقعی Mapping بررسی شود.
این مرحله بخشی از Expert Review موردنیاز Notebook 10 است.
</p>

<h4>نمونه‌های مورد بررسی</h4>

<ul>
<li>Strong Mappingهای Website برای Topicهای مختلف.</li>
<li>Strong Mappingهای Book برای Topicهای مختلف.</li>
<li>Lexical Mappingهای Transcript برای Topicهای مختلف.</li>
<li>Documentهای Unmapped از هر سه Source.</li>
</ul>

<h4>Gap Discovery</h4>

<p>
هم‌زمان Metadata ساختاری Documentهای Unmapped بررسی می‌شود:
</p>

<ul>
<li>Tagهای Website که هنوز وارد Taxonomy نشده‌اند.</li>
<li>Headingهای Book که در Mapping پوشش نگرفته‌اند.</li>
<li>Program distribution در Transcriptهای Unmapped.</li>
</ul>

<p>
هدف این مرحله تشخیص تفاوت میان سه حالت است:
</p>

<ol>
<li>Taxonomy واقعاً Topic یا Subtopic مهمی کم دارد.</li>
<li>Topic وجود دارد ولی Alias و Evidence Termهای آن ناقص‌اند.</li>
<li>Document برای Mapping در سطح Segment اطلاعات کافی ندارد و نیازمند Context بزرگ‌تر در مراحل بعدی است.</li>
</ol>

<p>
هیچ Label جدید، Topic جدید یا Artifact نهایی در این مرحله ساخته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>نمونه Review از Website، Book و Transcript تولید شود.</li>
<li>Strong و Lexical Evidence جدا باقی بمانند.</li>
<li>نمونه Unmapped هر سه Source بررسی‌پذیر باشد.</li>
<li>Gapهای Metadata به‌صورت Evidence گزارش شوند.</li>
<li>هیچ Unmapped Document با حدس برچسب نخورد.</li>
</ul>

</div>

In [59]:
# Code Cell 16

from collections import Counter, defaultdict
import hashlib
import re


def mapping_tier(profile):
    candidates = profile["candidates"]

    if not candidates:
        return "UNMAPPED"

    if any(
        candidate["decision"] == "STRONG_CANDIDATE"
        for candidate in candidates
    ):
        return "HAS_STRONG_CANDIDATE"

    return "LEXICAL_ONLY"


def deterministic_document_rank(
    document_id,
    salt,
):
    payload = (
        f"{salt}|{document_id}"
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def record_excerpt(
    record,
    limit=700,
):
    text = str(
        record.get("text", "")
    ).strip()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text[:limit]


record_lookup = {
    record["document_id"]: record
    for record in records
}


# --------------------------------------------------
# 1. Stratified mapped review samples
# --------------------------------------------------

mapped_review_strata = []

for topic_id in topic_seed_v0:

    mapped_review_strata.append(
        {
            "source_type": "website",
            "tier": "HAS_STRONG_CANDIDATE",
            "topic_id": topic_id,
            "sample_count": 1,
        }
    )

    mapped_review_strata.append(
        {
            "source_type": "book",
            "tier": "HAS_STRONG_CANDIDATE",
            "topic_id": topic_id,
            "sample_count": 1,
        }
    )

    mapped_review_strata.append(
        {
            "source_type": "transcript",
            "tier": "LEXICAL_ONLY",
            "topic_id": topic_id,
            "sample_count": 1,
        }
    )


mapped_review_samples_v0 = []
missing_mapped_strata = []


for stratum in mapped_review_strata:

    candidates = []

    for document_id, profile in (
        document_topic_candidates_v0.items()
    ):
        if (
            profile["source_type"]
            != stratum["source_type"]
        ):
            continue

        if (
            mapping_tier(profile)
            != stratum["tier"]
        ):
            continue

        topic_candidates = [
            candidate
            for candidate in profile["candidates"]
            if (
                candidate["topic_id"]
                == stratum["topic_id"]
            )
        ]

        if not topic_candidates:
            continue

        topic_candidate = topic_candidates[0]

        if (
            stratum["tier"]
            == "HAS_STRONG_CANDIDATE"
            and topic_candidate["decision"]
            != "STRONG_CANDIDATE"
        ):
            continue

        candidates.append(
            (
                document_id,
                topic_candidate,
            )
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            deterministic_document_rank(
                item[0],
                (
                    f"NB10-review:"
                    f"{stratum['source_type']}:"
                    f"{stratum['tier']}:"
                    f"{stratum['topic_id']}:v0"
                ),
            )
        ),
    )

    selected = candidates[
        :stratum["sample_count"]
    ]

    if len(selected) < stratum[
        "sample_count"
    ]:
        missing_mapped_strata.append(
            {
                **stratum,
                "available": len(candidates),
            }
        )

    for document_id, topic_candidate in selected:

        mapped_review_samples_v0.append(
            {
                "document_id": document_id,
                "source_type": stratum[
                    "source_type"
                ],
                "tier": stratum["tier"],
                "review_topic_id": stratum[
                    "topic_id"
                ],
                "topic_candidate": topic_candidate,
            }
        )


# --------------------------------------------------
# 2. Deterministic unmapped samples
# --------------------------------------------------

unmapped_sample_count_per_source = 3
unmapped_review_samples_v0 = []


for source_type in (
    "website",
    "book",
    "transcript",
):

    candidates = [
        document_id
        for document_id, profile
        in document_topic_candidates_v0.items()
        if (
            profile["source_type"]
            == source_type
            and mapping_tier(profile)
            == "UNMAPPED"
        )
    ]

    candidates = sorted(
        candidates,
        key=lambda document_id: (
            deterministic_document_rank(
                document_id,
                (
                    f"NB10-unmapped-review:"
                    f"{source_type}:v0"
                ),
            )
        ),
    )

    selected = candidates[
        :unmapped_sample_count_per_source
    ]

    for document_id in selected:
        unmapped_review_samples_v0.append(
            {
                "document_id": document_id,
                "source_type": source_type,
            }
        )


# --------------------------------------------------
# 3. Print mapped review samples
# --------------------------------------------------

print("Stratified mapped Expert Review set:")
print(
    "- Requested strata:",
    len(mapped_review_strata),
)
print(
    "- Samples produced:",
    len(mapped_review_samples_v0),
)
print(
    "- Missing strata:",
    len(missing_mapped_strata),
)


if missing_mapped_strata:
    print("\nMissing mapped review strata:")

    for row in missing_mapped_strata:
        print(
            "-",
            row["source_type"],
            "|",
            row["tier"],
            "|",
            topic_seed_v0[
                row["topic_id"]
            ]["label_fa"],
            "| available=",
            row["available"],
        )


print("\nMapped Expert Review samples:")

for index, sample in enumerate(
    mapped_review_samples_v0,
    start=1,
):
    document_id = sample[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    profile = document_topic_candidates_v0[
        document_id
    ]

    candidate = sample[
        "topic_candidate"
    ]

    print(
        "\n========================================"
    )
    print(
        f"Mapped Review Sample {index}"
    )
    print(
        "Source:",
        sample["source_type"],
    )
    print(
        "Tier:",
        sample["tier"],
    )
    print(
        "Document ID:",
        document_id,
    )
    print(
        "Title:",
        record.get("title"),
    )
    print(
        "Review Topic:",
        candidate[
            "topic_label_fa"
        ],
    )
    print(
        "Decision:",
        candidate["decision"],
    )
    print(
        "Confidence:",
        candidate["confidence"],
    )
    print(
        "Reason:",
        candidate["reason"],
    )
    print(
        "content_authority_type:",
        profile[
            "content_authority_type"
        ],
    )

    if sample["source_type"] == "website":
        print(
            "Website tags:",
            website_tag_names_for_record(
                record
            ),
        )

    elif sample["source_type"] == "book":
        heading_path = (
            record.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            or record.get(
                "source_metadata",
                {},
            ).get(
                "heading_path"
            )
        )

        print(
            "Book heading path:",
            heading_path,
        )

    elif sample["source_type"] == "transcript":
        metadata = record.get(
            "source_metadata",
            {},
        )

        print(
            "Program:",
            metadata.get("program"),
        )
        print(
            "Source speaker:",
            metadata.get("speaker"),
        )

    print("Excerpt:")
    print(
        record_excerpt(record)
        or "<EMPTY>"
    )


# --------------------------------------------------
# 4. Print unmapped review samples
# --------------------------------------------------

print("\n\nUnmapped Expert Review samples:")
print(
    "- Samples:",
    len(unmapped_review_samples_v0),
)


for index, sample in enumerate(
    unmapped_review_samples_v0,
    start=1,
):
    document_id = sample[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    profile = document_topic_candidates_v0[
        document_id
    ]

    print(
        "\n========================================"
    )
    print(
        f"Unmapped Review Sample {index}"
    )
    print(
        "Source:",
        sample["source_type"],
    )
    print(
        "Document ID:",
        document_id,
    )
    print(
        "Title:",
        record.get("title"),
    )
    print(
        "content_authority_type:",
        profile[
            "content_authority_type"
        ],
    )

    if sample["source_type"] == "website":
        print(
            "Website tags:",
            website_tag_names_for_record(
                record
            ),
        )

    elif sample["source_type"] == "book":
        heading_path = (
            record.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            or record.get(
                "source_metadata",
                {},
            ).get(
                "heading_path"
            )
        )

        print(
            "Book heading path:",
            heading_path,
        )

    elif sample["source_type"] == "transcript":
        metadata = record.get(
            "source_metadata",
            {},
        )

        print(
            "Program:",
            metadata.get("program"),
        )
        print(
            "Source speaker:",
            metadata.get("speaker"),
        )

    print("Excerpt:")
    print(
        record_excerpt(record)
        or "<EMPTY>"
    )


# --------------------------------------------------
# 5. Website unmapped tag-gap evidence
# --------------------------------------------------

unmapped_website_tag_counter = Counter()

for record in website_records:
    profile = document_topic_candidates_v0[
        record["document_id"]
    ]

    if mapping_tier(profile) != "UNMAPPED":
        continue

    for tag_name in website_tag_names_for_record(
        record
    ):
        unmapped_website_tag_counter[
            tag_name
        ] += 1


print("\n\nUnmapped Website tag evidence:")
print(
    "- Distinct tags:",
    len(unmapped_website_tag_counter),
)

for tag_name, count in (
    unmapped_website_tag_counter.most_common(
        40
    )
):
    print(
        f"- {tag_name}: {count}"
    )


# --------------------------------------------------
# 6. Book unmapped heading-gap evidence
# --------------------------------------------------

unmapped_book_heading_counter = Counter()

for record in book_records:
    profile = document_topic_candidates_v0[
        record["document_id"]
    ]

    if mapping_tier(profile) != "UNMAPPED":
        continue

    heading_path = (
        record.get(
            "locator",
            {},
        ).get(
            "heading_path"
        )
        or record.get(
            "source_metadata",
            {},
        ).get(
            "heading_path"
        )
    )

    normalized_path = normalize_heading_path(
        heading_path
    )

    for heading_component in normalized_path:
        unmapped_book_heading_counter[
            heading_component
        ] += 1


print("\nUnmapped Book heading evidence:")
print(
    "- Distinct heading components:",
    len(unmapped_book_heading_counter),
)

for heading, count in (
    unmapped_book_heading_counter.most_common(
        50
    )
):
    print(
        f"- {heading}: {count}"
    )


# --------------------------------------------------
# 7. Transcript unmapped program evidence
# --------------------------------------------------

unmapped_transcript_program_counter = Counter()

for record in transcript_records:
    profile = document_topic_candidates_v0[
        record["document_id"]
    ]

    if mapping_tier(profile) != "UNMAPPED":
        continue

    metadata = record.get(
        "source_metadata",
        {},
    )

    program = (
        metadata.get("program")
        or metadata.get("program_id")
        or record.get(
            "locator",
            {},
        ).get("program_id")
    )

    unmapped_transcript_program_counter[
        str(program)
    ] += 1


print("\nUnmapped Transcript program evidence:")

for program, count in (
    unmapped_transcript_program_counter.most_common()
):
    print(
        f"- {program}: {count}"
    )


# --------------------------------------------------
# 8. Review-set validation
# --------------------------------------------------

review_source_counter = Counter(
    sample["source_type"]
    for sample
    in mapped_review_samples_v0
)

unmapped_review_source_counter = Counter(
    sample["source_type"]
    for sample
    in unmapped_review_samples_v0
)


print("\nValidation:")
print(
    "- Mapped Website review samples:",
    review_source_counter["website"],
)
print(
    "- Mapped Book review samples:",
    review_source_counter["book"],
)
print(
    "- Mapped Transcript review samples:",
    review_source_counter[
        "transcript"
    ],
)
print(
    "- Unmapped Website samples:",
    unmapped_review_source_counter[
        "website"
    ],
)
print(
    "- Unmapped Book samples:",
    unmapped_review_source_counter[
        "book"
    ],
)
print(
    "- Unmapped Transcript samples:",
    unmapped_review_source_counter[
        "transcript"
    ],
)
print(
    "- Unmapped Website tags discovered:",
    len(
        unmapped_website_tag_counter
    ),
)
print(
    "- Unmapped Book heading components:",
    len(
        unmapped_book_heading_counter
    ),
)
print(
    "- Unmapped Transcript programs:",
    len(
        unmapped_transcript_program_counter
    ),
)


assert len(
    unmapped_review_samples_v0
) == 9

assert (
    unmapped_review_source_counter[
        "website"
    ]
    == 3
)

assert (
    unmapped_review_source_counter[
        "book"
    ]
    == 3
)

assert (
    unmapped_review_source_counter[
        "transcript"
    ]
    == 3
)

assert review_source_counter[
    "website"
] > 0

assert review_source_counter[
    "book"
] > 0

assert review_source_counter[
    "transcript"
] > 0


print("\nCODE CELL 16 GATE: PASS")
print(
    "Stratified Expert Review and Taxonomy-gap "
    "evidence set generated successfully."
)

Stratified mapped Expert Review set:
- Requested strata: 21
- Samples produced: 21
- Missing strata: 0

Mapped Expert Review samples:

Mapped Review Sample 1
Source: website
Tier: HAS_STRONG_CANDIDATE
Document ID: website:9F0C270B
Title: جام قهرمانی سواد مالی
Review Topic: مبانی و توانمندی سواد مالی
Decision: STRONG_CANDIDATE
Confidence: 0.7
Reason: ['WEBSITE_TAG_EXACT']
content_authority_type: editorial_supervision
Website tags: ['سواد مالی در جهان', 'سیاست\u200cگذاران بخوانند']
Excerpt:
مارس و آوریل 2018، اروپا مجمع بانکداری اروپا پس از برگزاری موفقیت‌آمیز سه دوره «هفته پول اروپا»، در فکر ابتکاری دیگر برای افزایش سواد مالی دانش‌آموزان اروپایی است. نتیجه، طراحی مسابقه‌ها و چالش‌هایی است که هم برای دانش‌آموزان جذاب باشد و هم معلمان؛ به این ترتیب «آزمون پول اروپا» شکل می‌گیرد. انجمن‌های بانکداری ملی در 25 کشور اروپایی از جمله فرانسه، آلمان، اسپانیا، ایتالیا، هلند و بلژیک، آمادگی خود برای شرکت در این مسابقه را اعلام می‌کنند. ماه‌های مارس و آوریل سال 2018 اولین آزمون پول اروپا و اولین دور

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>طبقه‌بندی Gapهای Taxonomy و تعریف Aliasهای Evidence-backed</h3>

<p>
Expert Review نشان داد که Unmapped بودن Documentها سه علت متفاوت دارد:
</p>

<ol>
<li><strong>Alias/Subtopic Gap:</strong> موضوع در Taxonomy وجود دارد ولی واژگان آن ناقص است؛ مانند «تصمیم‌گیری‌های مالی» یا «کار و درآمد».</li>
<li><strong>Context Dimension:</strong> Metadata درباره مخاطب، رسانه، قالب یا موقعیت زندگی است و نباید Topic محسوب شود؛ مانند «والدین بخوانند»، «سواد مالی خانم‌ها» یا «برنامه رادیویی».</li>
<li><strong>Review / Context Required:</strong> مفهوم واقعی است اما Evidence فعلی برای تعیین Parent Topic کافی نیست؛ مانند بخشش، مالیات یا بعضی Segmentهای کوتاه Transcript.</li>
</ol>

<p>
در این مرحله فقط Aliasهایی اضافه می‌شوند که شواهد مستقیم کتاب، Website یا Activity Review برای آن‌ها وجود دارد.
هیچ Topic اصلی جدیدی ساخته نمی‌شود.
</p>

<h4>سیاست Transcript</h4>

<p>
Segmentهای کوتاه Transcript نباید فقط به دلیل نبود Keyword به‌عنوان Gap در Taxonomy تفسیر شوند.
برای آن‌ها در مرحله بعد Context محلی همان Episode/Conversation بررسی خواهد شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Aliasهای جدید فقط به Topicهای موجود متصل شوند.</li>
<li>Dimensionهای مخاطب، رسانه و قالب از Topic جدا بمانند.</li>
<li>موارد مبهم Review Required باقی بمانند.</li>
<li>تأثیر Aliasهای جدید روی Unmappedها فقط محاسبه شود و هنوز Dataset تغییر نکند.</li>
</ul>

</div>

In [60]:
# Code Cell 17

from collections import Counter, defaultdict


taxonomy_gap_registry_v0 = {
    # ------------------------------------------------
    # Evidence-backed Topic aliases / subtopics
    # ------------------------------------------------

    "book_heading_aliases": {
        "6. تصمیم‌گیری‌های مالی": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "financial_decision_making",
            "confidence": 0.99,
            "reason": (
                "Explicit Book section heading for the "
                "financial decision-making domain."
            ),
        },

        "3. کار و درآمد": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "work_income",
            "confidence": 0.99,
            "reason": (
                "Explicit Book section heading for work and income."
            ),
        },

        "کسب و کار کودکان": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "work_income",
            "confidence": 0.98,
            "reason": (
                "Activity evidence requires business design, "
                "costing, pricing and profit analysis."
            ),
        },

        "کار کودکان": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "work_income",
            "confidence": 0.97,
            "reason": (
                "Activity evidence explicitly concerns "
                "income-generating activities for children."
            ),
        },

        "مبادلۀ کالابه‌کالا": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "financial_literacy_foundations",
            "confidence": 0.92,
            "reason": (
                "Observed instructional context includes "
                "real versus nominal value, inflation, price "
                "and exchange concepts."
            ),
        },

        "تورم": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "financial_literacy_foundations",
            "confidence": 0.96,
            "reason": (
                "Direct evidence from reviewed Trainer activity "
                "on inflation and real financial value."
            ),
        },

        "تصویرسازی خواسته‌ها": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "financial_decision_making",
            "confidence": 0.94,
            "reason": (
                "Reviewed activity connects future visualization "
                "with desired outcomes and financial goal formation."
            ),
        },
    },

    "website_tag_aliases": {
        "مفهوم شغل": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "work_income",
            "confidence": 0.99,
            "reason": "Explicit Website tag naming the job concept.",
        },

        "مفهوم سرمایه‌گذاری": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "investment",
            "confidence": 0.99,
            "reason": (
                "Explicit Website tag naming the "
                "investment concept."
            ),
        },

        "مصارف پول": {
            "disposition": "TOPIC_ALIAS_CANDIDATE",
            "topic_id": "spending_saving_money_management",
            "confidence": 0.95,
            "reason": (
                "Website tag directly concerns uses/allocation "
                "of money."
            ),
        },
    },

    # ------------------------------------------------
    # Explicit non-Topic dimensions
    # ------------------------------------------------

    "context_dimensions": {
        "سواد مالی خانم‌ها": "AUDIENCE_CONTEXT",
        "خانم‌های دارای درآمد": "AUDIENCE_CONTEXT",
        "خانم‌های خانه‌دار": "AUDIENCE_CONTEXT",
        "والدین بخوانند": "AUDIENCE_CONTEXT",
        "مربیان بخوانند": "AUDIENCE_CONTEXT",
        "سیاست‌گذاران بخوانند": "AUDIENCE_CONTEXT",
        "صاحب‌نظران": "AUDIENCE_CONTEXT",

        "برنامه رادیویی چهلستون": "MEDIA_PROGRAM_CONTEXT",
        "برنامه تلویزیونی عصر شیرین": "MEDIA_PROGRAM_CONTEXT",
        "پادکست فیه ما فی": "MEDIA_PROGRAM_CONTEXT",
        "مجموعه برنامه‌های رادیویی": "MEDIA_PROGRAM_CONTEXT",

        "تحلیل فیلم": "MEDIA_FORM_CONTEXT",
        "سواد مالی در سینما": "MEDIA_FORM_CONTEXT",
        "جشنواره فیلم فجر": "MEDIA_FORM_CONTEXT",
        "بررسی بازی": "MEDIA_FORM_CONTEXT",
        "سواد مالی در ادبیات": "MEDIA_FORM_CONTEXT",

        "محصول آکادمی": "PUBLISHING_CONTEXT",
        "معرفی کتاب‌های آکادمی": "PUBLISHING_CONTEXT",
        "معرفی کتاب": "PUBLISHING_CONTEXT",

        "بازنشستگی": "LIFE_STAGE_CONTEXT",
        "آموزه‌های اسلامی": "PERSPECTIVE_CONTEXT",
    },

    # ------------------------------------------------
    # Structural Book headings that are not Topics
    # ------------------------------------------------

    "book_structure_dimensions": {
        "زمانی برای اندیشه": "STRUCTURAL_SECTION",
        "کلید پاسخ‌": "ANSWER_KEY_SECTION",
        "منابع آموزشی": "RESOURCE_SECTION",
        "کنکاش": "ACTIVITY_OR_REFLECTION_SECTION",
        "سخن آکادمی هوش مالی": "INTRODUCTORY_SECTION",
    },

    # ------------------------------------------------
    # Explicitly unresolved
    # ------------------------------------------------

    "review_required": {
        "بخشش": (
            "Financial/social relevance exists, but Parent Topic "
            "is not yet sufficiently established."
        ),
        "بخشش و  هدیه": (
            "Reviewed Activity remained unresolved in Cell 09."
        ),
        "مالیات": (
            "Valid financial-literacy concept, but current evidence "
            "does not establish whether it belongs under an existing "
            "Topic or requires a new Subtopic branch."
        ),
        "اقتصاد رفتاری": (
            "Cross-cutting knowledge area; should not be forced "
            "into one Topic without additional review."
        ),
        "انتخاب": (
            "Too generic to use as a Topic alias safely."
        ),
        "قیمت": (
            "Can belong to foundations or business context; "
            "heading alone is insufficient."
        ),
        "سود": (
            "Can refer to investment return or business profit; "
            "heading alone is ambiguous."
        ),
        "منابع تولید": (
            "Strong business context in some lessons, but heading "
            "alone requires contextual confirmation."
        ),
        "تفاوت کالا و خدمت": (
            "Economic/business foundation concept requiring "
            "contextual Parent Topic review."
        ),
        "معرفی محصول و تبلیغ": (
            "Likely entrepreneurship/business context, but preserve "
            "for contextual validation."
        ),
    },
}


# --------------------------------------------------
# Validate registry Topic IDs
# --------------------------------------------------

topic_alias_entries = []

for source_registry_name in (
    "book_heading_aliases",
    "website_tag_aliases",
):
    for alias, spec in taxonomy_gap_registry_v0[
        source_registry_name
    ].items():
        topic_alias_entries.append(
            {
                "source_registry": source_registry_name,
                "alias": alias,
                **spec,
            }
        )


invalid_alias_topic_ids = [
    row
    for row in topic_alias_entries
    if row["topic_id"] not in topic_seed_v0
]


duplicate_aliases = []

alias_counter = Counter(
    normalize_fa_for_match(
        row["alias"]
    )
    for row in topic_alias_entries
)

for alias, count in alias_counter.items():
    if count > 1:
        duplicate_aliases.append(
            (
                alias,
                count,
            )
        )


print("Taxonomy Gap Registry:")
print(
    "- Topic Alias candidates:",
    len(topic_alias_entries),
)
print(
    "- Context dimensions:",
    len(
        taxonomy_gap_registry_v0[
            "context_dimensions"
        ]
    ),
)
print(
    "- Book structural dimensions:",
    len(
        taxonomy_gap_registry_v0[
            "book_structure_dimensions"
        ]
    ),
)
print(
    "- Review Required concepts:",
    len(
        taxonomy_gap_registry_v0[
            "review_required"
        ]
    ),
)


print("\nEvidence-backed Alias candidates:")

for row in topic_alias_entries:
    print(
        f"- {row['alias']} "
        f"-> {topic_seed_v0[row['topic_id']]['label_fa']} "
        f"| confidence={row['confidence']:.2f}"
    )


print("\nNon-Topic Context dimensions:")

context_type_counter = Counter(
    taxonomy_gap_registry_v0[
        "context_dimensions"
    ].values()
)

for context_type, count in (
    context_type_counter.most_common()
):
    print(
        f"- {context_type}: {count}"
    )


print("\nExplicit Review Required concepts:")

for concept, reason in taxonomy_gap_registry_v0[
    "review_required"
].items():
    print(
        f"- {concept}: {reason}"
    )


# --------------------------------------------------
# Build alias additions by Topic
# --------------------------------------------------

topic_alias_additions_v1 = defaultdict(set)

for row in topic_alias_entries:
    topic_alias_additions_v1[
        row["topic_id"]
    ].add(
        normalize_fa_for_match(
            row["alias"]
        )
    )


print("\nCandidate alias additions by Topic:")

for topic_id in topic_seed_v0:
    additions = sorted(
        topic_alias_additions_v1.get(
            topic_id,
            set(),
        )
    )

    if not additions:
        continue

    print(
        f"\n[{topic_id}] "
        f"{topic_seed_v0[topic_id]['label_fa']}"
    )

    for alias in additions:
        print(
            "-",
            alias,
        )


# --------------------------------------------------
# Estimate impact ONLY on currently Unmapped docs
# No mutation of the Cell 15 mapping.
# --------------------------------------------------

def alias_hits_for_unmapped_record(
    record,
):
    source_type = record.get(
        "source_type"
    )

    hits = defaultdict(list)

    if source_type == "website":
        tag_names = website_tag_names_for_record(
            record
        )

        normalized_tags = {
            normalize_fa_for_match(
                tag_name
            )
            for tag_name in tag_names
        }

        for topic_id, aliases in (
            topic_alias_additions_v1.items()
        ):
            matched = sorted(
                normalized_tags
                & aliases
            )

            if matched:
                hits[topic_id].extend(
                    [
                        (
                            "WEBSITE_TAG_ALIAS",
                            value,
                        )
                        for value in matched
                    ]
                )

    elif source_type == "book":
        heading_path = (
            record.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            or record.get(
                "source_metadata",
                {},
            ).get(
                "heading_path"
            )
        )

        normalized_components = {
            normalize_fa_for_match(
                component
            )
            for component in normalize_heading_path(
                heading_path
            )
        }

        for topic_id, aliases in (
            topic_alias_additions_v1.items()
        ):
            matched = sorted(
                normalized_components
                & aliases
            )

            if matched:
                hits[topic_id].extend(
                    [
                        (
                            "BOOK_HEADING_ALIAS",
                            value,
                        )
                        for value in matched
                    ]
                )

    return dict(hits)


alias_impact_rows = []

for document_id, profile in (
    document_topic_candidates_v0.items()
):

    if mapping_tier(profile) != "UNMAPPED":
        continue

    record = record_lookup[
        document_id
    ]

    hits = alias_hits_for_unmapped_record(
        record
    )

    if not hits:
        continue

    alias_impact_rows.append(
        {
            "document_id": document_id,
            "source_type": profile[
                "source_type"
            ],
            "topic_hits": hits,
        }
    )


alias_impact_by_source = Counter()
alias_impact_by_topic = Counter()
alias_multi_topic_impact = 0


for row in alias_impact_rows:
    alias_impact_by_source[
        row["source_type"]
    ] += 1

    if len(row["topic_hits"]) > 1:
        alias_multi_topic_impact += 1

    for topic_id in row[
        "topic_hits"
    ]:
        alias_impact_by_topic[
            topic_id
        ] += 1


print("\nEstimated impact on currently Unmapped documents:")
print(
    "- Newly coverable documents:",
    len(alias_impact_rows),
)

for source_type in (
    "website",
    "book",
    "transcript",
):
    print(
        f"- {source_type}:",
        alias_impact_by_source[
            source_type
        ],
    )


print("\nEstimated impact by Topic:")

for topic_id in topic_seed_v0:
    count = alias_impact_by_topic[
        topic_id
    ]

    if count:
        print(
            f"- {topic_seed_v0[topic_id]['label_fa']}: "
            f"{count}"
        )


print(
    "\n- Multi-topic impacts:",
    alias_multi_topic_impact,
)


remaining_unmapped_estimate = (
    mapping_status_counter[
        "UNMAPPED"
    ]
    - len(alias_impact_rows)
)


print(
    "- Current Unmapped:",
    mapping_status_counter[
        "UNMAPPED"
    ],
)
print(
    "- Estimated Unmapped after safe alias refinement:",
    remaining_unmapped_estimate,
)


# --------------------------------------------------
# Transcript policy
# --------------------------------------------------

transcript_mapping_policy_v0 = {
    "isolated_segment_unmapped_is_taxonomy_failure": False,
    "program_name_is_topic_evidence": False,
    "speaker_identity_is_topic_evidence": False,
    "next_required_evidence": (
        "LOCAL_CONVERSATION_CONTEXT"
    ),
    "allowed_context_scope": (
        "same program/episode neighboring accepted segments"
    ),
    "rechunking_performed": False,
}


print("\nTranscript Mapping Policy:")

for key, value in (
    transcript_mapping_policy_v0.items()
):
    print(
        f"- {key}: {value}"
    )


print("\nValidation:")
print(
    "- Invalid Alias Topic IDs:",
    len(invalid_alias_topic_ids),
)
print(
    "- Duplicate normalized aliases:",
    len(duplicate_aliases),
)
print(
    "- Alias-impact rows:",
    len(alias_impact_rows),
)
print(
    "- Transcript rechunking performed:",
    transcript_mapping_policy_v0[
        "rechunking_performed"
    ],
)


assert not invalid_alias_topic_ids
assert not duplicate_aliases

assert all(
    row["disposition"]
    == "TOPIC_ALIAS_CANDIDATE"
    for row in topic_alias_entries
)

assert (
    transcript_mapping_policy_v0[
        "program_name_is_topic_evidence"
    ]
    is False
)

assert (
    transcript_mapping_policy_v0[
        "speaker_identity_is_topic_evidence"
    ]
    is False
)

assert (
    transcript_mapping_policy_v0[
        "rechunking_performed"
    ]
    is False
)

assert len(alias_impact_rows) > 0


print("\nCODE CELL 17 GATE: PASS")
print(
    "Taxonomy-gap dispositions and conservative "
    "Alias refinements validated successfully."
)

Taxonomy Gap Registry:
- Topic Alias candidates: 10
- Context dimensions: 21
- Book structural dimensions: 5
- Review Required concepts: 10

Evidence-backed Alias candidates:
- 6. تصمیم‌گیری‌های مالی -> تصمیم‌گیری مالی | confidence=0.99
- 3. کار و درآمد -> کار، شغل و درآمد | confidence=0.99
- کسب و کار کودکان -> کار، شغل و درآمد | confidence=0.98
- کار کودکان -> کار، شغل و درآمد | confidence=0.97
- مبادلۀ کالابه‌کالا -> مبانی و توانمندی سواد مالی | confidence=0.92
- تورم -> مبانی و توانمندی سواد مالی | confidence=0.96
- تصویرسازی خواسته‌ها -> تصمیم‌گیری مالی | confidence=0.94
- مفهوم شغل -> کار، شغل و درآمد | confidence=0.99
- مفهوم سرمایه‌گذاری -> سرمایه‌گذاری | confidence=0.99
- مصارف پول -> خرج، پس‌انداز و مدیریت پول | confidence=0.95

Non-Topic Context dimensions:
- AUDIENCE_CONTEXT: 7
- MEDIA_FORM_CONTEXT: 5
- MEDIA_PROGRAM_CONTEXT: 4
- PUBLISHING_CONTEXT: 3
- LIFE_STAGE_CONTEXT: 1
- PERSPECTIVE_CONTEXT: 1

Explicit Review Required concepts:
- بخشش: Financial/social relevance exis

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery نگاشت Topic برای Transcript با Local Conversation Context</h3>

<p>
بخش بزرگی از Transcriptها در سطح Segment منفرد کوتاه هستند.
نمونه Expert Review نشان داد Segmentهایی مانند «ولی آیا تو واقعیت تجربه‌اش کردی؟» یا «حق بیمه چقدر بود؟» بدون جمله‌های اطراف برای Topic Classification کافی نیستند.
</p>

<p>
این وضعیت نباید به‌عنوان شکست Taxonomy یا دلیل Rechunk فوری تفسیر شود.
در این مرحله فقط Context محلی همان Conversation Unit بررسی می‌شود.
</p>

<h4>سیاست Context</h4>

<ul>
<li>فقط Segmentهای همان Episode یا Conversation Unit مجازند.</li>
<li>Program Name و Speaker Identity هیچ امتیاز Topic ایجاد نمی‌کنند.</li>
<li>برای هر Segment Unmapped حداکثر سه Neighbor قبل و سه Neighbor بعد بررسی می‌شود.</li>
<li>Neighbor فقط وقتی Topic Vote می‌دهد که یک Top Candidate یکتای قابل تشخیص داشته باشد.</li>
<li>حداقل دو Neighbor باید از یک Topic حمایت کنند.</li>
<li>Topic غالب باید دست‌کم دوسوم Voteهای قابل استفاده را داشته باشد.</li>
<li>این نتیجه فقط <code>LOCAL_CONTEXT_CANDIDATE</code> است، نه Strong Mapping یا Hard Label.</li>
</ul>

<p>
اگر Evidence محلی کافی نباشد Segment همچنان Unmapped باقی می‌ماند.
هیچ Segment جدیدی ساخته نمی‌شود و هیچ Rechunk انجام نمی‌شود.
</p>

<h4>هدف</h4>

<p>
این مرحله مشخص می‌کند چه مقدار از Gap بزرگ Transcript صرفاً با Context موجود و بدون تغییر Dataset قابل توضیح است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Conversation Unitهای Transcript به‌صورت Deterministic استخراج شوند.</li>
<li>هیچ Neighbor از Conversation Unit دیگر وارد Context نشود.</li>
<li>فقط Segmentهای قبلاً Unmapped بررسی شوند.</li>
<li>حداقل دو Neighbor برای Candidate Context لازم باشد.</li>
<li>Program و Speaker در Topic Voting استفاده نشوند.</li>
<li>هیچ Rechunk یا Disk Write انجام نشود.</li>
</ul>

</div>

In [61]:
# Code Cell 18

from collections import Counter, defaultdict
import hashlib
import re


# --------------------------------------------------
# 1. Discover deterministic Transcript
#    conversation units and sequence positions
# --------------------------------------------------

def transcript_conversation_key(record):
    document_id = str(
        record.get("document_id", "")
    )

    # Observed accepted transcript IDs contain
    # conversation-level prefixes such as:
    #
    # program:...:episode-...:utterance:...
    # program:...:unit-...:utterance:...
    # program:...:episode-...:merged:...
    #
    # Remove only the segment-level suffix.
    prefix = re.sub(
        r":(?:utterance|merged):.*$",
        "",
        document_id,
    )

    if prefix != document_id:
        return (
            "DOCUMENT_ID_PREFIX",
            prefix,
        )

    # Conservative fallback to accepted metadata.
    metadata = record.get(
        "source_metadata",
        {},
    )

    locator = record.get(
        "locator",
        {},
    )

    program_id = (
        metadata.get("program_id")
        or locator.get("program_id")
        or metadata.get("program")
        or record.get("title")
    )

    season_number = (
        locator.get("season_number")
        or metadata.get("season_number")
    )

    episode_number = (
        locator.get("episode_number")
        or metadata.get("episode_number")
    )

    return (
        "METADATA_FALLBACK",
        str(program_id),
        str(season_number),
        str(episode_number),
    )


def transcript_sequence_position(
    record,
    fallback_index,
):
    metadata = record.get(
        "source_metadata",
        {},
    )

    locator = record.get(
        "locator",
        {},
    )

    utterance_index = metadata.get(
        "utterance_index"
    )

    if utterance_index is not None:
        try:
            return (
                0,
                int(utterance_index),
            )
        except (TypeError, ValueError):
            pass

    document_id = str(
        record.get("document_id", "")
    )

    utterance_match = re.search(
        r":utterance:(\d+)",
        document_id,
    )

    if utterance_match:
        return (
            1,
            int(
                utterance_match.group(1)
            ),
        )

    merged_match = re.search(
        r":merged:(\d+)",
        document_id,
    )

    if merged_match:
        return (
            2,
            int(
                merged_match.group(1)
            ),
        )

    source_line_start = locator.get(
        "source_line_start"
    )

    if source_line_start is not None:
        try:
            return (
                3,
                int(source_line_start),
            )
        except (TypeError, ValueError):
            pass

    return (
        9,
        fallback_index,
    )


transcript_conversation_groups = defaultdict(
    list
)


for fallback_index, record in enumerate(
    transcript_records
):
    conversation_key = (
        transcript_conversation_key(
            record
        )
    )

    transcript_conversation_groups[
        conversation_key
    ].append(
        {
            "record": record,
            "position": transcript_sequence_position(
                record,
                fallback_index,
            ),
        }
    )


for conversation_key in (
    transcript_conversation_groups
):
    transcript_conversation_groups[
        conversation_key
    ].sort(
        key=lambda item: (
            item["position"],
            item["record"].get(
                "document_id",
                "",
            ),
        )
    )


conversation_size_counter = Counter(
    len(rows)
    for rows
    in transcript_conversation_groups.values()
)


print("Transcript Conversation Context:")
print(
    "- Transcript documents:",
    len(transcript_records),
)
print(
    "- Conversation units:",
    len(
        transcript_conversation_groups
    ),
)

print(
    "- Smallest unit:",
    min(
        len(rows)
        for rows
        in transcript_conversation_groups.values()
    ),
)

print(
    "- Largest unit:",
    max(
        len(rows)
        for rows
        in transcript_conversation_groups.values()
    ),
)


# --------------------------------------------------
# 2. Unique Top Topic anchor policy
# --------------------------------------------------

def unique_top_topic_candidate(
    document_id,
):
    profile = (
        document_topic_candidates_v0[
            document_id
        ]
    )

    candidates = profile[
        "candidates"
    ]

    if not candidates:
        return None

    max_confidence = max(
        candidate["confidence"]
        for candidate in candidates
    )

    top_candidates = [
        candidate
        for candidate in candidates
        if (
            candidate["confidence"]
            == max_confidence
        )
    ]

    if len(top_candidates) != 1:
        return None

    top = top_candidates[0]

    return {
        "topic_id": top["topic_id"],
        "topic_label_fa": top[
            "topic_label_fa"
        ],
        "confidence": top[
            "confidence"
        ],
        "decision": top[
            "decision"
        ],
        "reason": top[
            "reason"
        ],
    }


# --------------------------------------------------
# 3. Build local context evidence
# --------------------------------------------------

context_radius = 3

transcript_local_context_candidates_v0 = {}


for conversation_key, rows in (
    transcript_conversation_groups.items()
):

    for index, item in enumerate(rows):
        record = item["record"]

        document_id = record[
            "document_id"
        ]

        profile = (
            document_topic_candidates_v0[
                document_id
            ]
        )

        # Only previously unmapped Transcript segments.
        if mapping_tier(
            profile
        ) != "UNMAPPED":
            continue

        start_index = max(
            0,
            index - context_radius,
        )

        end_index = min(
            len(rows),
            index + context_radius + 1,
        )

        neighbor_votes = []

        for neighbor_index in range(
            start_index,
            end_index,
        ):
            if neighbor_index == index:
                continue

            neighbor_record = rows[
                neighbor_index
            ]["record"]

            neighbor_document_id = (
                neighbor_record[
                    "document_id"
                ]
            )

            anchor = unique_top_topic_candidate(
                neighbor_document_id
            )

            if anchor is None:
                continue

            side = (
                "before"
                if neighbor_index < index
                else "after"
            )

            neighbor_votes.append(
                {
                    "document_id": (
                        neighbor_document_id
                    ),
                    "side": side,
                    "distance": abs(
                        neighbor_index
                        - index
                    ),
                    **anchor,
                }
            )

        topic_vote_counter = Counter(
            vote["topic_id"]
            for vote in neighbor_votes
        )

        if not topic_vote_counter:
            continue

        ranked_votes = (
            topic_vote_counter.most_common()
        )

        winner_topic_id = (
            ranked_votes[0][0]
        )

        winner_vote_count = (
            ranked_votes[0][1]
        )

        runner_up_vote_count = (
            ranked_votes[1][1]
            if len(ranked_votes) > 1
            else 0
        )

        total_votes = len(
            neighbor_votes
        )

        support_ratio = (
            winner_vote_count
            / total_votes
            if total_votes
            else 0.0
        )

        # Conservative evidence policy.
        if winner_vote_count < 2:
            continue

        if (
            winner_vote_count
            == runner_up_vote_count
        ):
            continue

        if support_ratio < (
            2 / 3
        ):
            continue

        winner_votes = [
            vote
            for vote in neighbor_votes
            if (
                vote["topic_id"]
                == winner_topic_id
            )
        ]

        supporting_sides = sorted({
            vote["side"]
            for vote in winner_votes
        })

        if (
            "before" in supporting_sides
            and "after" in supporting_sides
        ):
            context_shape = (
                "TWO_SIDED_CONTEXT"
            )
        else:
            context_shape = (
                "ONE_SIDED_CONTEXT"
            )

        transcript_local_context_candidates_v0[
            document_id
        ] = {
            "document_id": document_id,
            "conversation_key": (
                conversation_key
            ),
            "candidate_topic_id": (
                winner_topic_id
            ),
            "candidate_topic_label_fa": (
                topic_seed_v0[
                    winner_topic_id
                ]["label_fa"]
            ),
            "decision": (
                "LOCAL_CONTEXT_CANDIDATE"
            ),
            "context_shape": (
                context_shape
            ),
            "neighbor_vote_count": (
                total_votes
            ),
            "supporting_vote_count": (
                winner_vote_count
            ),
            "support_ratio": round(
                support_ratio,
                3,
            ),
            "supporting_sides": (
                supporting_sides
            ),
            "supporting_neighbors": (
                winner_votes
            ),
            "reason": (
                "SAME_CONVERSATION_NEIGHBOR_TOPIC_CONSENSUS"
            ),
        }


# --------------------------------------------------
# 4. Population statistics
# --------------------------------------------------

current_unmapped_transcript_count = sum(
    1
    for record in transcript_records
    if (
        mapping_tier(
            document_topic_candidates_v0[
                record["document_id"]
            ]
        )
        == "UNMAPPED"
    )
)


context_candidate_count = len(
    transcript_local_context_candidates_v0
)


remaining_unmapped_after_context_estimate = (
    current_unmapped_transcript_count
    - context_candidate_count
)


print("\nLocal Context Candidate population:")
print(
    "- Currently Unmapped Transcript:",
    current_unmapped_transcript_count,
)
print(
    "- Local Context candidates:",
    context_candidate_count,
)
print(
    "- Candidate coverage of Unmapped Transcript:",
    (
        f"{context_candidate_count / current_unmapped_transcript_count:.1%}"
        if current_unmapped_transcript_count
        else "0.0%"
    ),
)
print(
    "- Estimated still Unmapped after context:",
    remaining_unmapped_after_context_estimate,
)


context_shape_counter = Counter(
    row["context_shape"]
    for row in (
        transcript_local_context_candidates_v0.values()
    )
)


print("\nContext shape:")

for shape, count in (
    context_shape_counter.most_common()
):
    print(
        f"- {shape}: {count}"
    )


context_topic_counter = Counter(
    row["candidate_topic_id"]
    for row in (
        transcript_local_context_candidates_v0.values()
    )
)


print("\nContext Candidate Topics:")

for topic_id in topic_seed_v0:
    count = context_topic_counter[
        topic_id
    ]

    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"{count}"
    )


context_program_counter = Counter()

for document_id in (
    transcript_local_context_candidates_v0
):
    record = record_lookup[
        document_id
    ]

    metadata = record.get(
        "source_metadata",
        {},
    )

    program = (
        metadata.get("program")
        or metadata.get("program_id")
        or record.get("title")
    )

    context_program_counter[
        str(program)
    ] += 1


print("\nContext Candidates by Program:")

for program, count in (
    context_program_counter.most_common()
):
    print(
        f"- {program}: {count}"
    )


support_ratio_counter = Counter(
    row["support_ratio"]
    for row in (
        transcript_local_context_candidates_v0.values()
    )
)


print("\nSupport-ratio distribution:")

for ratio in sorted(
    support_ratio_counter
):
    print(
        f"- {ratio:.3f}: "
        f"{support_ratio_counter[ratio]}"
    )


# --------------------------------------------------
# 5. Deterministic representative review
#    one per Topic when available
# --------------------------------------------------

def context_sample_rank(
    document_id,
    topic_id,
):
    payload = (
        f"NB10-context-review:"
        f"{topic_id}:"
        f"{document_id}:v0"
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


context_review_samples_v0 = []


for topic_id in topic_seed_v0:

    candidates = [
        row
        for row in (
            transcript_local_context_candidates_v0.values()
        )
        if (
            row[
                "candidate_topic_id"
            ]
            == topic_id
        )
    ]

    if not candidates:
        continue

    candidates = sorted(
        candidates,
        key=lambda row: (
            context_sample_rank(
                row["document_id"],
                topic_id,
            )
        ),
    )

    context_review_samples_v0.append(
        candidates[0]
    )


print("\nRepresentative Local Context Review:")

for index, sample in enumerate(
    context_review_samples_v0,
    start=1,
):
    document_id = sample[
        "document_id"
    ]

    target_record = record_lookup[
        document_id
    ]

    print(
        "\n========================================"
    )
    print(
        f"Context Review Sample {index}"
    )
    print(
        "Target Document:",
        document_id,
    )
    print(
        "Program:",
        target_record.get(
            "source_metadata",
            {},
        ).get("program"),
    )
    print(
        "Candidate Topic:",
        sample[
            "candidate_topic_label_fa"
        ],
    )
    print(
        "Decision:",
        sample["decision"],
    )
    print(
        "Context shape:",
        sample[
            "context_shape"
        ],
    )
    print(
        "Supporting votes:",
        (
            f"{sample['supporting_vote_count']}/"
            f"{sample['neighbor_vote_count']}"
        ),
    )
    print(
        "Support ratio:",
        sample["support_ratio"],
    )

    print("Target excerpt:")
    print(
        record_excerpt(
            target_record,
            limit=500,
        )
        or "<EMPTY>"
    )

    print("Supporting neighbors:")

    for neighbor in sample[
        "supporting_neighbors"
    ]:
        neighbor_record = record_lookup[
            neighbor["document_id"]
        ]

        print(
            f"- {neighbor['side']} "
            f"| distance={neighbor['distance']} "
            f"| {neighbor['topic_label_fa']} "
            f"| confidence={neighbor['confidence']:.2f}"
        )

        print(
            "  ",
            record_excerpt(
                neighbor_record,
                limit=350,
            )
            or "<EMPTY>",
        )


# --------------------------------------------------
# 6. Independent validation
# --------------------------------------------------

invalid_target_status = []
insufficient_neighbor_support = []
cross_conversation_evidence = []
invalid_topic_ids = []
speaker_or_program_evidence_leakage = []


document_to_conversation_key = {}

for conversation_key, rows in (
    transcript_conversation_groups.items()
):
    for item in rows:
        document_to_conversation_key[
            item["record"]["document_id"]
        ] = conversation_key


for document_id, candidate in (
    transcript_local_context_candidates_v0.items()
):

    original_profile = (
        document_topic_candidates_v0[
            document_id
        ]
    )

    if (
        mapping_tier(
            original_profile
        )
        != "UNMAPPED"
    ):
        invalid_target_status.append(
            document_id
        )

    if (
        candidate[
            "supporting_vote_count"
        ]
        < 2
        or candidate[
            "support_ratio"
        ]
        < (2 / 3)
    ):
        insufficient_neighbor_support.append(
            document_id
        )

    if (
        candidate[
            "candidate_topic_id"
        ]
        not in topic_seed_v0
    ):
        invalid_topic_ids.append(
            document_id
        )

    target_conversation_key = (
        document_to_conversation_key[
            document_id
        ]
    )

    for neighbor in candidate[
        "supporting_neighbors"
    ]:
        neighbor_conversation_key = (
            document_to_conversation_key[
                neighbor["document_id"]
            ]
        )

        if (
            neighbor_conversation_key
            != target_conversation_key
        ):
            cross_conversation_evidence.append(
                (
                    document_id,
                    neighbor[
                        "document_id"
                    ],
                )
            )

    reason_text = str(
        candidate["reason"]
    ).lower()

    if (
        "speaker" in reason_text
        or "program" in reason_text
    ):
        speaker_or_program_evidence_leakage.append(
            document_id
        )


print("\nValidation:")
print(
    "- Context Candidate targets:",
    len(
        transcript_local_context_candidates_v0
    ),
)
print(
    "- Invalid target status:",
    len(invalid_target_status),
)
print(
    "- Insufficient neighbor support:",
    len(
        insufficient_neighbor_support
    ),
)
print(
    "- Cross-conversation evidence:",
    len(
        cross_conversation_evidence
    ),
)
print(
    "- Invalid Topic IDs:",
    len(invalid_topic_ids),
)
print(
    "- Speaker/Program evidence leakage:",
    len(
        speaker_or_program_evidence_leakage
    ),
)
print(
    "- Rechunking performed:",
    False,
)
print(
    "- Disk write performed:",
    False,
)


assert len(
    transcript_conversation_groups
) > 0

assert (
    current_unmapped_transcript_count
    == 13400
)

assert context_candidate_count > 0

assert not invalid_target_status
assert not insufficient_neighbor_support
assert not cross_conversation_evidence
assert not invalid_topic_ids

assert not (
    speaker_or_program_evidence_leakage
)


print("\nCODE CELL 18 GATE: PASS")
print(
    "Local Conversation Context evidence for "
    "Unmapped Transcript segments discovered successfully."
)

Transcript Conversation Context:
- Transcript documents: 16226
- Conversation units: 183
- Smallest unit: 3
- Largest unit: 438

Local Context Candidate population:
- Currently Unmapped Transcript: 13400
- Local Context candidates: 1755
- Candidate coverage of Unmapped Transcript: 13.1%
- Estimated still Unmapped after context: 11645

Context shape:
- TWO_SIDED_CONTEXT: 1045
- ONE_SIDED_CONTEXT: 710

Context Candidate Topics:
- مبانی و توانمندی سواد مالی: 29
- خرج، پس‌انداز و مدیریت پول: 778
- اعتبار، قرض، بدهی و بانک: 331
- کار، شغل و درآمد: 126
- سرمایه‌گذاری: 300
- ریسک و بیمه: 147
- تصمیم‌گیری مالی: 44

Context Candidates by Program:
- در جستجوی خوشبختی: 595
- چهلستون: 315
- یونی تهران: 248
- تربیت مربی: 163
- خانه تو: 148
- عصر شیرین: 98
- بانک ملت: 98
- میزان: 37
- ام کیو: 19
- کتاب باز: 18
- فناپ: 14
- اقیانوس آرام: 2

Support-ratio distribution:
- 0.667: 257
- 0.750: 65
- 0.800: 17
- 0.833: 6
- 1.000: 1410

Representative Local Context Review:

Context Review Sample 1
Target Do

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تجمیع Candidate Mapping v1 و تفکیک Document Topic از Conversation Topic Scope</h3>

<p>
Local Context در Transcript توانست برای 1,755 Segment بدون Topic، Context موضوعی قابل دفاع پیدا کند.
اما Expert Review نشان داد که قرار گرفتن یک Segment در Context یک Topic لزوماً به این معنی نیست که خود Segment محتوای مستقل آن Topic را بیان می‌کند.
</p>

<p>
برای مثال، یک جمله خوشامدگویی ممکن است میان چند Segment درباره سرمایه‌گذاری قرار گرفته باشد.
در چنین حالتی Topic برای بازیابی Context مفید است، اما نباید به‌عنوان Topic مستقل خود Document ثبت شود.
</p>

<h4>دو Dimension مستقل</h4>

<ul>
<li><strong>Document Topic Candidate:</strong> Topic خود Document با Evidence مستقیم یا ساختاری.</li>
<li><strong>Conversation Topic Scope Candidate:</strong> Topic Context محلی Segment، بدون ادعای اینکه خود Segment مستقلاً Topic-bearing است.</li>
</ul>

<p>
در این مرحله Aliasهای Evidence-backed سلول 17 روی Documentهای واقعاً Unmapped اعمال می‌شوند.
این Aliasها از Website Tag یا Book Heading ساختاری آمده‌اند و بنابراین می‌توانند Strong Candidate باشند.
</p>

<p>
اما 1,755 نتیجه Local Context سلول 18 به Document Topic ارتقا داده نمی‌شوند و فقط به‌عنوان Conversation Scope نگهداری می‌شوند.
</p>

<p>
هیچ Rechunk، Disk Write یا تغییر Source Attribution در این مرحله انجام نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>551 Alias Candidate مستند به Mapping Candidate اضافه شوند.</li>
<li>Local Context هیچ Document Topic جدیدی ایجاد نکند.</li>
<li>Conversation Scope در Dimension جداگانه ذخیره شود.</li>
<li>content_authority_type بدون تغییر حفظ شود.</li>
<li>تمام Topic IDها معتبر باشند.</li>
<li>هیچ Document از بین نرود یا تکراری نشود.</li>
</ul>

</div>

In [62]:
# Code Cell 19

from collections import Counter, defaultdict
import copy


# --------------------------------------------------
# 1. Build normalized Alias evidence registry
# --------------------------------------------------

normalized_alias_evidence_registry = {
    "WEBSITE_TAG_ALIAS": {},
    "BOOK_HEADING_ALIAS": {},
}


for alias, spec in taxonomy_gap_registry_v0[
    "website_tag_aliases"
].items():
    normalized_alias = normalize_fa_for_match(
        alias
    )

    normalized_alias_evidence_registry[
        "WEBSITE_TAG_ALIAS"
    ][normalized_alias] = {
        **spec,
        "alias": alias,
    }


for alias, spec in taxonomy_gap_registry_v0[
    "book_heading_aliases"
].items():
    normalized_alias = normalize_fa_for_match(
        alias
    )

    normalized_alias_evidence_registry[
        "BOOK_HEADING_ALIAS"
    ][normalized_alias] = {
        **spec,
        "alias": alias,
    }


# --------------------------------------------------
# 2. Start from Cell 15 candidate mapping
# --------------------------------------------------

document_topic_mapping_v1_candidate = copy.deepcopy(
    document_topic_candidates_v0
)


for profile in (
    document_topic_mapping_v1_candidate.values()
):
    profile["mapping_revision"] = (
        "NB10_CANDIDATE_V1"
    )

    profile["conversation_topic_scope"] = None


# --------------------------------------------------
# 3. Apply ONLY safe Alias impacts discovered
#    on previously Unmapped documents
# --------------------------------------------------

alias_added_document_ids = set()
alias_added_candidate_rows = []


for impact in alias_impact_rows:
    document_id = impact[
        "document_id"
    ]

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    # Cell 17 impact population must have
    # been unmapped in Cell 15.
    if profile["candidates"]:
        continue

    new_candidates = []

    for topic_id, evidence_rows in impact[
        "topic_hits"
    ].items():

        candidate_evidence = []
        candidate_confidences = []

        for (
            evidence_type,
            normalized_alias,
        ) in evidence_rows:

            registry = (
                normalized_alias_evidence_registry[
                    evidence_type
                ]
            )

            alias_spec = registry.get(
                normalized_alias
            )

            if alias_spec is None:
                continue

            assert (
                alias_spec["topic_id"]
                == topic_id
            )

            candidate_confidences.append(
                alias_spec[
                    "confidence"
                ]
            )

            candidate_evidence.append(
                {
                    "type": evidence_type,
                    "value": normalized_alias,
                    "source_alias": alias_spec[
                        "alias"
                    ],
                    "confidence": alias_spec[
                        "confidence"
                    ],
                    "reason": alias_spec[
                        "reason"
                    ],
                }
            )

        if not candidate_evidence:
            continue

        confidence = round(
            max(
                candidate_confidences
            ),
            3,
        )

        new_candidate = {
            "topic_id": topic_id,
            "topic_label_fa": (
                topic_seed_v0[
                    topic_id
                ]["label_fa"]
            ),
            "decision": (
                "STRONG_CANDIDATE"
            ),
            "confidence": confidence,
            "reason": sorted({
                row["type"]
                for row
                in candidate_evidence
            }),
            "evidence": (
                candidate_evidence
            ),
            "mapping_origin": (
                "SAFE_ALIAS_REFINEMENT"
            ),
        }

        new_candidates.append(
            new_candidate
        )

        alias_added_candidate_rows.append(
            {
                "document_id": document_id,
                **new_candidate,
            }
        )

    if new_candidates:
        new_candidates.sort(
            key=lambda item: (
                -item["confidence"],
                item["topic_id"],
            )
        )

        profile["candidates"] = (
            new_candidates
        )

        alias_added_document_ids.add(
            document_id
        )


# --------------------------------------------------
# 4. Preserve Local Conversation Context
#    as a separate dimension
# --------------------------------------------------

for document_id, context_candidate in (
    transcript_local_context_candidates_v0.items()
):

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    profile[
        "conversation_topic_scope"
    ] = {
        "topic_id": context_candidate[
            "candidate_topic_id"
        ],
        "topic_label_fa": context_candidate[
            "candidate_topic_label_fa"
        ],
        "decision": (
            "CONVERSATION_TOPIC_SCOPE_CANDIDATE"
        ),
        "context_shape": context_candidate[
            "context_shape"
        ],
        "supporting_vote_count": (
            context_candidate[
                "supporting_vote_count"
            ]
        ),
        "neighbor_vote_count": (
            context_candidate[
                "neighbor_vote_count"
            ]
        ),
        "support_ratio": (
            context_candidate[
                "support_ratio"
            ]
        ),
        "reason": context_candidate[
            "reason"
        ],
    }


# --------------------------------------------------
# 5. Validate Context was NOT promoted
#    to Document Topic
# --------------------------------------------------

context_promoted_to_document_topic = []


for document_id, context_candidate in (
    transcript_local_context_candidates_v0.items()
):

    original_profile = (
        document_topic_candidates_v0[
            document_id
        ]
    )

    revised_profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    # These targets were all UNMAPPED in Cell 15.
    assert not original_profile[
        "candidates"
    ]

    # No Transcript alias refinement exists,
    # therefore they must still have no
    # Document Topic Candidate.
    if revised_profile[
        "candidates"
    ]:
        context_promoted_to_document_topic.append(
            document_id
        )


# --------------------------------------------------
# 6. Candidate v1 coverage statistics
# --------------------------------------------------

v1_mapping_status_counter = Counter()
v1_source_mapping_counter = defaultdict(
    Counter
)

conversation_scope_counter = Counter()


for profile in (
    document_topic_mapping_v1_candidate.values()
):

    source_type = profile[
        "source_type"
    ]

    candidates = profile[
        "candidates"
    ]

    if not candidates:
        status = "UNMAPPED"

    elif any(
        candidate["decision"]
        == "STRONG_CANDIDATE"
        for candidate in candidates
    ):
        status = (
            "HAS_STRONG_CANDIDATE"
        )

    else:
        status = "LEXICAL_ONLY"

    v1_mapping_status_counter[
        status
    ] += 1

    v1_source_mapping_counter[
        source_type
    ][status] += 1

    if profile[
        "conversation_topic_scope"
    ] is not None:
        conversation_scope_counter[
            source_type
        ] += 1


print("Candidate Mapping v1:")
print(
    "- Documents:",
    len(
        document_topic_mapping_v1_candidate
    ),
)
print(
    "- Alias-refined documents:",
    len(alias_added_document_ids),
)
print(
    "- Alias candidate rows:",
    len(alias_added_candidate_rows),
)
print(
    "- Conversation Topic Scope candidates:",
    sum(
        conversation_scope_counter.values()
    ),
)


print("\nDocument Topic Mapping status v1:")

for status in (
    "HAS_STRONG_CANDIDATE",
    "LEXICAL_ONLY",
    "UNMAPPED",
):
    count = v1_mapping_status_counter[
        status
    ]

    print(
        f"- {status}: "
        f"{count}/{len(records)} "
        f"({count / len(records):.1%})"
    )


print("\nDocument Topic Mapping status by Source:")

for source_type in (
    "website",
    "book",
    "transcript",
):
    population = len(
        source_records[
            source_type
        ]
    )

    print(f"\n[{source_type}]")

    for status in (
        "HAS_STRONG_CANDIDATE",
        "LEXICAL_ONLY",
        "UNMAPPED",
    ):
        count = (
            v1_source_mapping_counter[
                source_type
            ][status]
        )

        print(
            f"- {status}: "
            f"{count}/{population} "
            f"({count / population:.1%})"
        )

    print(
        "- Conversation Topic Scope:",
        conversation_scope_counter[
            source_type
        ],
    )


# --------------------------------------------------
# 7. Effective Context-aware observability
#    (NOT the same as Document Topic coverage)
# --------------------------------------------------

transcript_document_topic_unmapped = (
    v1_source_mapping_counter[
        "transcript"
    ]["UNMAPPED"]
)


transcript_unmapped_with_context_scope = sum(
    1
    for profile
    in document_topic_mapping_v1_candidate.values()
    if (
        profile["source_type"]
        == "transcript"
        and not profile["candidates"]
        and profile[
            "conversation_topic_scope"
        ]
        is not None
    )
)


transcript_unmapped_without_context_scope = (
    transcript_document_topic_unmapped
    - transcript_unmapped_with_context_scope
)


print(
    "\nTranscript Context-aware observability:"
)
print(
    "- Transcript without Document Topic:",
    transcript_document_topic_unmapped,
)
print(
    "- Of those, with Conversation Topic Scope:",
    transcript_unmapped_with_context_scope,
)
print(
    "- Still without Document Topic or Context Scope:",
    transcript_unmapped_without_context_scope,
)


# --------------------------------------------------
# 8. Topic distribution after safe Alias refinement
# --------------------------------------------------

v1_topic_decision_counter = defaultdict(
    Counter
)


for profile in (
    document_topic_mapping_v1_candidate.values()
):
    for candidate in profile[
        "candidates"
    ]:
        v1_topic_decision_counter[
            candidate["topic_id"]
        ][
            candidate["decision"]
        ] += 1


print("\nCandidate Topic populations v1:")

for topic_id in topic_seed_v0:
    counter = (
        v1_topic_decision_counter[
            topic_id
        ]
    )

    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"strong={counter['STRONG_CANDIDATE']} "
        f"| lexical={counter['LEXICAL_CANDIDATE']}"
    )


# --------------------------------------------------
# 9. Independent validation
# --------------------------------------------------

invalid_topic_ids_v1 = []
invalid_confidence_rows_v1 = []
missing_reason_rows_v1 = []
authority_preservation_mismatches_v1 = []
invalid_context_topic_ids = []


for document_id, profile in (
    document_topic_mapping_v1_candidate.items()
):

    original_authority = (
        record_lookup[
            document_id
        ].get(
            "attribution",
            {},
        ).get(
            "content_authority_type"
        )
    )

    if (
        profile[
            "content_authority_type"
        ]
        != original_authority
    ):
        authority_preservation_mismatches_v1.append(
            document_id
        )

    for candidate in profile[
        "candidates"
    ]:

        if (
            candidate["topic_id"]
            not in topic_seed_v0
        ):
            invalid_topic_ids_v1.append(
                (
                    document_id,
                    candidate[
                        "topic_id"
                    ],
                )
            )

        if not (
            0.0
            < candidate[
                "confidence"
            ]
            <= 0.99
        ):
            invalid_confidence_rows_v1.append(
                document_id
            )

        if not candidate[
            "reason"
        ]:
            missing_reason_rows_v1.append(
                document_id
            )

    context_scope = profile[
        "conversation_topic_scope"
    ]

    if (
        context_scope is not None
        and context_scope[
            "topic_id"
        ] not in topic_seed_v0
    ):
        invalid_context_topic_ids.append(
            document_id
        )


print("\nValidation:")
print(
    "- Expected safe Alias impacts:",
    len(alias_impact_rows),
)
print(
    "- Applied Alias impacts:",
    len(alias_added_document_ids),
)
print(
    "- Context candidates preserved separately:",
    sum(
        conversation_scope_counter.values()
    ),
)
print(
    "- Context promoted to Document Topic:",
    len(
        context_promoted_to_document_topic
    ),
)
print(
    "- Invalid Document Topic IDs:",
    len(invalid_topic_ids_v1),
)
print(
    "- Invalid Context Topic IDs:",
    len(invalid_context_topic_ids),
)
print(
    "- Invalid Confidence rows:",
    len(
        invalid_confidence_rows_v1
    ),
)
print(
    "- Missing Reason rows:",
    len(missing_reason_rows_v1),
)
print(
    "- Authority preservation mismatches:",
    len(
        authority_preservation_mismatches_v1
    ),
)


assert len(
    document_topic_mapping_v1_candidate
) == 19456

assert len(
    alias_added_document_ids
) == len(
    alias_impact_rows
)

assert (
    len(alias_added_document_ids)
    == 551
)

assert (
    sum(
        conversation_scope_counter.values()
    )
    == 1755
)

assert not (
    context_promoted_to_document_topic
)

assert not invalid_topic_ids_v1
assert not invalid_context_topic_ids
assert not invalid_confidence_rows_v1
assert not missing_reason_rows_v1

assert not (
    authority_preservation_mismatches_v1
)

assert (
    sum(
        v1_mapping_status_counter.values()
    )
    == 19456
)


print("\nCODE CELL 19 GATE: PASS")
print(
    "Candidate Mapping v1 consolidated with "
    "Document Topic and Conversation Topic Scope "
    "kept as separate dimensions."
)

Candidate Mapping v1:
- Documents: 19456
- Alias-refined documents: 551
- Alias candidate rows: 551
- Conversation Topic Scope candidates: 1755

Document Topic Mapping status v1:
- HAS_STRONG_CANDIDATE: 2536/19456 (13.0%)
- LEXICAL_ONLY: 3280/19456 (16.9%)
- UNMAPPED: 13640/19456 (70.1%)

Document Topic Mapping status by Source:

[website]
- HAS_STRONG_CANDIDATE: 343/529 (64.8%)
- LEXICAL_ONLY: 124/529 (23.4%)
- UNMAPPED: 62/529 (11.7%)
- Conversation Topic Scope: 0

[book]
- HAS_STRONG_CANDIDATE: 2193/2701 (81.2%)
- LEXICAL_ONLY: 330/2701 (12.2%)
- UNMAPPED: 178/2701 (6.6%)
- Conversation Topic Scope: 0

[transcript]
- HAS_STRONG_CANDIDATE: 0/16226 (0.0%)
- LEXICAL_ONLY: 2826/16226 (17.4%)
- UNMAPPED: 13400/16226 (82.6%)
- Conversation Topic Scope: 1755

Transcript Context-aware observability:
- Transcript without Document Topic: 13400
- Of those, with Conversation Topic Scope: 1755
- Still without Document Topic or Context Scope: 11645

Candidate Topic populations v1:
- مبانی و توانم

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Pre-Freeze Coverage & Conflict Audit برای Taxonomy v1</h3>

<p>
پس از تجمیع Candidate Mapping v1، پیش از Freeze کردن Taxonomy باید وضعیت پوشش، ابهام، Content Role و Gateهای باز به‌صورت مستقل بررسی شود.
</p>

<p>
این Audit بین سه نوع Evidence تفاوت نگه می‌دارد:
</p>

<ul>
<li><strong>Strong Document Topic:</strong> نگاشت دارای Evidence ساختاری یا Metadata معتبر.</li>
<li><strong>Lexical Document Topic:</strong> نگاشت متنی Candidate که هنوز نیازمند Validation است.</li>
<li><strong>Conversation Topic Scope:</strong> Context موضوعی Transcript که Topic مستقل خود Segment محسوب نمی‌شود.</li>
</ul>

<h4>اهداف</h4>

<ul>
<li>اندازه‌گیری Coverage هر Topic در Website، Book و Transcript.</li>
<li>اندازه‌گیری تعداد Strong و Lexical Candidateها به‌صورت جدا.</li>
<li>شناسایی Documentهای دارای چند Strong Topic و چند Candidate Topic.</li>
<li>بررسی Content Roleهای Concept، Example، Exercise، Warning و Solution در هر Topic.</li>
<li>بررسی Authority موجود بدون استنتاج مقدارهای null.</li>
<li>ثبت Gapهای باز Taxonomy و Review Required.</li>
<li>تعیین آمادگی یا عدم آمادگی برای Freeze v1.</li>
</ul>

<h4>اصل مهم</h4>

<p>
کم بودن تعداد Document به‌تنهایی Knowledge Gap محسوب نمی‌شود.
Coverage در این مرحله فقط Evidence Inventory است.
تشخیص Gap واقعی آموزشی بعداً باید نسبت به Competency، Depth، Redundancy و Answerability انجام شود.
</p>

<h4>Freeze Gate</h4>

<p>
این Notebook فقط زمانی می‌تواند Taxonomy v1 را Freeze کند که Validation فنی PASS باشد و Expert/Specialist Review موردنیاز نیز صریحاً ثبت شده باشد.
در نبود Expert Review، وضعیت باید <code>BLOCKED_PENDING_EXPERT_REVIEW</code> باقی بماند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 19,456 Document در Audit حضور داشته باشند.</li>
<li>Strong، Lexical و Conversation Scope با هم مخلوط نشوند.</li>
<li>Multi-topic ambiguity اندازه‌گیری شود.</li>
<li>Content Role coverage برای تمام Topicها محاسبه شود.</li>
<li>Authority موجود بدون تغییر گزارش شود.</li>
<li>Review Requiredهای باز حذف یا پنهان نشوند.</li>
<li>Freeze بدون Expert Review انجام نشود.</li>
</ul>

</div>

In [63]:
# Code Cell 20

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Basic mapping populations
# --------------------------------------------------

total_documents = len(
    document_topic_mapping_v1_candidate
)

assert total_documents == 19456


strong_document_ids_by_topic = defaultdict(set)
lexical_document_ids_by_topic = defaultdict(set)

strong_topic_ids_by_document = defaultdict(set)
all_topic_ids_by_document = defaultdict(set)

conversation_scope_ids_by_topic = defaultdict(set)


for document_id, profile in (
    document_topic_mapping_v1_candidate.items()
):

    for candidate in profile["candidates"]:

        topic_id = candidate["topic_id"]

        all_topic_ids_by_document[
            document_id
        ].add(topic_id)

        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        ):
            strong_document_ids_by_topic[
                topic_id
            ].add(document_id)

            strong_topic_ids_by_document[
                document_id
            ].add(topic_id)

        elif (
            candidate["decision"]
            == "LEXICAL_CANDIDATE"
        ):
            lexical_document_ids_by_topic[
                topic_id
            ].add(document_id)

    context_scope = profile[
        "conversation_topic_scope"
    ]

    if context_scope is not None:
        conversation_scope_ids_by_topic[
            context_scope["topic_id"]
        ].add(document_id)


# --------------------------------------------------
# 2. Coverage by Source and Topic
# --------------------------------------------------

topic_source_coverage_v1 = {}


for topic_id in topic_seed_v0:

    topic_source_coverage_v1[
        topic_id
    ] = {}

    for source_type in (
        "website",
        "book",
        "transcript",
    ):

        strong_count = sum(
            1
            for document_id in (
                strong_document_ids_by_topic[
                    topic_id
                ]
            )
            if (
                document_topic_mapping_v1_candidate[
                    document_id
                ]["source_type"]
                == source_type
            )
        )

        lexical_count = sum(
            1
            for document_id in (
                lexical_document_ids_by_topic[
                    topic_id
                ]
            )
            if (
                document_topic_mapping_v1_candidate[
                    document_id
                ]["source_type"]
                == source_type
            )
        )

        conversation_scope_count = sum(
            1
            for document_id in (
                conversation_scope_ids_by_topic[
                    topic_id
                ]
            )
            if (
                document_topic_mapping_v1_candidate[
                    document_id
                ]["source_type"]
                == source_type
            )
        )

        topic_source_coverage_v1[
            topic_id
        ][source_type] = {
            "strong": strong_count,
            "lexical": lexical_count,
            "conversation_scope": (
                conversation_scope_count
            ),
        }


print("Topic Coverage by Source:")


for topic_id in topic_seed_v0:

    print(
        f"\n[{topic_id}] "
        f"{topic_seed_v0[topic_id]['label_fa']}"
    )

    for source_type in (
        "website",
        "book",
        "transcript",
    ):
        coverage = (
            topic_source_coverage_v1[
                topic_id
            ][source_type]
        )

        print(
            f"- {source_type}: "
            f"strong={coverage['strong']} "
            f"| lexical={coverage['lexical']} "
            f"| conversation_scope="
            f"{coverage['conversation_scope']}"
        )


# --------------------------------------------------
# 3. Multi-topic ambiguity
# --------------------------------------------------

strong_topic_count_distribution = Counter(
    len(topic_ids)
    for topic_ids in (
        strong_topic_ids_by_document.values()
    )
)

all_topic_count_distribution = Counter(
    len(profile["candidates"])
    for profile in (
        document_topic_mapping_v1_candidate.values()
    )
)


documents_with_multiple_strong_topics = {
    document_id: sorted(topic_ids)
    for document_id, topic_ids in (
        strong_topic_ids_by_document.items()
    )
    if len(topic_ids) > 1
}


documents_with_multiple_candidates = {
    document_id: sorted({
        candidate["topic_id"]
        for candidate in profile[
            "candidates"
        ]
    })
    for document_id, profile in (
        document_topic_mapping_v1_candidate.items()
    )
    if len({
        candidate["topic_id"]
        for candidate in profile[
            "candidates"
        ]
    }) > 1
}


print("\nMulti-topic ambiguity:")

print(
    "- Documents with >1 Strong Topic:",
    len(
        documents_with_multiple_strong_topics
    ),
)

print(
    "- Documents with >1 total Topic Candidate:",
    len(
        documents_with_multiple_candidates
    ),
)


print("\nStrong Topic count distribution:")

for topic_count in sorted(
    strong_topic_count_distribution
):
    print(
        f"- {topic_count} Strong Topic(s): "
        f"{strong_topic_count_distribution[topic_count]}"
    )


# --------------------------------------------------
# 4. Strong-topic Content Role coverage
# --------------------------------------------------

strong_topic_role_counter = defaultdict(
    Counter
)

strong_topic_no_role_counter = Counter()


for topic_id, document_ids in (
    strong_document_ids_by_topic.items()
):

    for document_id in document_ids:

        role_profile = (
            candidate_content_roles_v0[
                document_id
            ]
        )

        roles = {
            role["role"]
            for role in role_profile[
                "roles"
            ]
        }

        if not roles:
            strong_topic_no_role_counter[
                topic_id
            ] += 1
            continue

        for role_id in roles:
            strong_topic_role_counter[
                topic_id
            ][role_id] += 1


print("\nContent Role coverage within Strong Topic mappings:")


for topic_id in topic_seed_v0:

    strong_total = len(
        strong_document_ids_by_topic[
            topic_id
        ]
    )

    counter = strong_topic_role_counter[
        topic_id
    ]

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    print(
        "- Strong documents:",
        strong_total,
    )

    for role_id in (
        "concept",
        "example",
        "exercise",
        "warning",
        "solution",
    ):
        count = counter[
            role_id
        ]

        print(
            f"- {role_id}: {count}"
        )

    print(
        "- no explicit Candidate role:",
        strong_topic_no_role_counter[
            topic_id
        ],
    )


# --------------------------------------------------
# 5. Authority inventory within Strong mappings
# --------------------------------------------------

strong_topic_authority_counter = defaultdict(
    Counter
)


for topic_id, document_ids in (
    strong_document_ids_by_topic.items()
):

    for document_id in document_ids:

        authority = (
            document_topic_mapping_v1_candidate[
                document_id
            ][
                "content_authority_type"
            ]
        )

        authority_key = (
            str(authority)
            if nonempty(authority)
            else "<null>"
        )

        strong_topic_authority_counter[
            topic_id
        ][authority_key] += 1


print("\nExisting content_authority_type within Strong mappings:")


for topic_id in topic_seed_v0:

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    counter = (
        strong_topic_authority_counter[
            topic_id
        ]
    )

    if not counter:
        print("- None")
        continue

    for authority, count in (
        counter.most_common()
    ):
        print(
            f"- {authority}: {count}"
        )


# --------------------------------------------------
# 6. Competency coverage inventory
#
# This is model-level evidence coverage,
# NOT yet a scored Knowledge Gap analysis.
# --------------------------------------------------

competency_inventory_v1 = []


for topic_id, topic in (
    competency_model_v0.items()
):

    strong_docs = len(
        strong_document_ids_by_topic[
            topic_id
        ]
    )

    lexical_docs = len(
        lexical_document_ids_by_topic[
            topic_id
        ]
    )

    context_docs = len(
        conversation_scope_ids_by_topic[
            topic_id
        ]
    )

    for competency in topic[
        "competencies"
    ]:

        competency_inventory_v1.append(
            {
                "topic_id": topic_id,
                "competency_id": competency[
                    "id"
                ],
                "stage": competency[
                    "stage"
                ],
                "evidence_items": len(
                    competency[
                        "evidence"
                    ]
                ),
                "topic_strong_documents": (
                    strong_docs
                ),
                "topic_lexical_documents": (
                    lexical_docs
                ),
                "topic_conversation_scope": (
                    context_docs
                ),
                "status": (
                    "MODEL_EVIDENCE_PRESENT"
                    if competency[
                        "evidence"
                    ]
                    else "NO_MODEL_EVIDENCE"
                ),
            }
        )


competencies_without_model_evidence = [
    row
    for row in competency_inventory_v1
    if (
        row["status"]
        == "NO_MODEL_EVIDENCE"
    )
]


print("\nCompetency inventory:")

print(
    "- Total Competencies:",
    len(
        competency_inventory_v1
    ),
)

print(
    "- Competencies without model evidence:",
    len(
        competencies_without_model_evidence
    ),
)


# --------------------------------------------------
# 7. Open taxonomy/review gates
# --------------------------------------------------

open_taxonomy_review_items = []


for concept, reason in (
    taxonomy_gap_registry_v0[
        "review_required"
    ].items()
):

    open_taxonomy_review_items.append(
        {
            "concept": concept,
            "reason": reason,
            "status": "REVIEW_REQUIRED",
        }
    )


expert_review_registry_v0 = {
    "expert_review_completed": False,
    "review_scope": (
        "taxonomy, competency model, "
        "candidate mappings, unresolved concepts"
    ),
    "required_before_freeze": True,
    "reviewer": None,
    "review_date": None,
}


print("\nOpen review gates:")

print(
    "- Taxonomy concepts requiring review:",
    len(
        open_taxonomy_review_items
    ),
)

print(
    "- Expert review completed:",
    expert_review_registry_v0[
        "expert_review_completed"
    ],
)


# --------------------------------------------------
# 8. Pre-Freeze readiness
# --------------------------------------------------

technical_validation_pass = True


freeze_blockers_v0 = []


if open_taxonomy_review_items:
    freeze_blockers_v0.append(
        "OPEN_TAXONOMY_REVIEW_ITEMS"
    )


if (
    expert_review_registry_v0[
        "required_before_freeze"
    ]
    and not (
        expert_review_registry_v0[
            "expert_review_completed"
        ]
    )
):
    freeze_blockers_v0.append(
        "PENDING_EXPERT_REVIEW"
    )


if technical_validation_pass:
    technical_status = "PASS"
else:
    technical_status = "FAIL"


if freeze_blockers_v0:
    freeze_readiness_v0 = (
        "BLOCKED_PENDING_REVIEW"
    )
else:
    freeze_readiness_v0 = (
        "READY_FOR_FREEZE"
    )


print("\nPre-Freeze readiness:")

print(
    "- Technical status:",
    technical_status,
)

print(
    "- Freeze readiness:",
    freeze_readiness_v0,
)

print(
    "- Freeze blockers:",
    freeze_blockers_v0,
)


# --------------------------------------------------
# 9. Independent validation
# --------------------------------------------------

invalid_coverage_topic_ids = [
    topic_id
    for topic_id in (
        topic_source_coverage_v1
    )
    if topic_id not in topic_seed_v0
]


topics_without_any_strong_mapping = [
    topic_id
    for topic_id in topic_seed_v0
    if not strong_document_ids_by_topic[
        topic_id
    ]
]


invalid_role_ids = []

valid_role_ids = {
    "concept",
    "example",
    "exercise",
    "warning",
    "solution",
}


for topic_id, counter in (
    strong_topic_role_counter.items()
):
    for role_id in counter:
        if role_id not in valid_role_ids:
            invalid_role_ids.append(
                (
                    topic_id,
                    role_id,
                )
            )


authority_preservation_check = []


for document_id, profile in (
    document_topic_mapping_v1_candidate.items()
):

    original = record_lookup[
        document_id
    ].get(
        "attribution",
        {},
    ).get(
        "content_authority_type"
    )

    if (
        profile[
            "content_authority_type"
        ]
        != original
    ):
        authority_preservation_check.append(
            document_id
        )


print("\nValidation:")

print(
    "- Total audited documents:",
    total_documents,
)

print(
    "- Invalid coverage Topic IDs:",
    len(
        invalid_coverage_topic_ids
    ),
)

print(
    "- Topics without Strong Mapping:",
    len(
        topics_without_any_strong_mapping
    ),
)

print(
    "- Invalid Content Role IDs:",
    len(
        invalid_role_ids
    ),
)

print(
    "- Competencies without model evidence:",
    len(
        competencies_without_model_evidence
    ),
)

print(
    "- Authority preservation mismatches:",
    len(
        authority_preservation_check
    ),
)

print(
    "- Open Taxonomy Review items:",
    len(
        open_taxonomy_review_items
    ),
)

print(
    "- Freeze executed:",
    False,
)


assert total_documents == 19456

assert not invalid_coverage_topic_ids

assert not topics_without_any_strong_mapping

assert not invalid_role_ids

assert not (
    competencies_without_model_evidence
)

assert not (
    authority_preservation_check
)

assert (
    expert_review_registry_v0[
        "expert_review_completed"
    ]
    is False
)

assert (
    freeze_readiness_v0
    == "BLOCKED_PENDING_REVIEW"
)


print("\nCODE CELL 20 GATE: PASS")
print(
    "Pre-Freeze Coverage & Conflict Audit "
    "completed successfully; Taxonomy freeze "
    "is correctly blocked pending review."
)

Topic Coverage by Source:

[financial_literacy_foundations] مبانی و توانمندی سواد مالی
- website: strong=162 | lexical=35 | conversation_scope=0
- book: strong=141 | lexical=1234 | conversation_scope=0
- transcript: strong=0 | lexical=94 | conversation_scope=29

[spending_saving_money_management] خرج، پس‌انداز و مدیریت پول
- website: strong=96 | lexical=202 | conversation_scope=0
- book: strong=359 | lexical=150 | conversation_scope=0
- transcript: strong=0 | lexical=1330 | conversation_scope=778

[credit_debt_banking] اعتبار، قرض، بدهی و بانک
- website: strong=19 | lexical=110 | conversation_scope=0
- book: strong=215 | lexical=79 | conversation_scope=0
- transcript: strong=0 | lexical=686 | conversation_scope=331

[work_income] کار، شغل و درآمد
- website: strong=28 | lexical=79 | conversation_scope=0
- book: strong=360 | lexical=33 | conversation_scope=0
- transcript: strong=0 | lexical=239 | conversation_scope=126

[investment] سرمایه‌گذاری
- website: strong=39 | lexical=100 | conve

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Expert Review Packet برای موارد باز Taxonomy و Multi-Topic Conflict</h3>

<p>
Pre-Freeze Audit نشان داد Validation فنی PASS است، اما Freeze به دلیل دو Gate متوقف است:
</p>

<ul>
<li>۱۰ مفهوم Taxonomy هنوز <code>REVIEW_REQUIRED</code> هستند.</li>
<li>Expert/Specialist Review هنوز به‌صورت رسمی ثبت نشده است.</li>
</ul>

<p>
در این مرحله برای تصمیم‌گیری متخصص، یک Review Packet مبتنی بر Evidence واقعی Corpus ساخته می‌شود.
هیچ تصمیم تخصصی به‌صورت خودکار گرفته نمی‌شود.
</p>

<h4>برای هر مفهوم Review Required بررسی می‌شود:</h4>

<ul>
<li>چند Document در Website، Book و Transcript شواهد آن را دارند.</li>
<li>آیا مفهوم در Website Tag یا Book Heading به‌صورت ساختاری دیده می‌شود.</li>
<li>در Documentهای دارای آن مفهوم، چه Topic Candidateهایی بیشتر هم‌رخداد دارند.</li>
<li>نمونه‌های واقعی و Deterministic از محتوا.</li>
</ul>

<h4>Multi-Topic Conflict</h4>

<p>
Documentهای دارای بیش از یک Strong Topic لزوماً خطا نیستند.
ممکن است واقعاً چندموضوعی باشند.
بنابراین Pairهای پرتکرار Strong Topic و نمونه‌های واقعی آن‌ها نیز برای Expert Review استخراج می‌شوند.
</p>

<h4>خروجی این مرحله</h4>

<p>
این Cell فقط <strong>Evidence Packet</strong> می‌سازد.
وضعیت Expert Review همچنان <code>NOT_COMPLETED</code> باقی می‌ماند تا تصمیم انسانی صریحاً ثبت شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۰ مفهوم Review Required دارای Evidence Profile باشد.</li>
<li>Source distribution برای هر مفهوم مشخص شود.</li>
<li>Topic co-occurrence بدون تبدیل خودکار به Parent Topic گزارش شود.</li>
<li>نمونه‌های واقعی Corpus برای Review نمایش داده شوند.</li>
<li>Multi-Strong Topic pairها و نمونه‌هایشان استخراج شوند.</li>
<li>هیچ Review Required به‌طور خودکار Resolve نشود.</li>
<li>Freeze همچنان انجام نشود.</li>
</ul>

</div>

In [64]:
# Code Cell 21

from collections import Counter, defaultdict
import hashlib
import re


# --------------------------------------------------
# 1. Review concepts
# --------------------------------------------------

review_concepts = list(
    taxonomy_gap_registry_v0[
        "review_required"
    ].keys()
)


print("Expert Review Packet:")
print(
    "- Review Required concepts:",
    len(review_concepts),
)
print(
    "- Multi-Strong Topic documents:",
    len(
        documents_with_multiple_strong_topics
    ),
)


# --------------------------------------------------
# 2. Helpers
# --------------------------------------------------

def normalized_review_text(record):
    parts = [
        record.get("title", ""),
        record.get("text", ""),
    ]

    source_type = record.get(
        "source_type"
    )

    if source_type == "website":
        parts.extend(
            website_tag_names_for_record(
                record
            )
        )

    if source_type == "book":
        heading_path = (
            record.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            or record.get(
                "source_metadata",
                {},
            ).get(
                "heading_path"
            )
        )

        parts.extend(
            normalize_heading_path(
                heading_path
            )
        )

    return normalize_fa_for_match(
        " | ".join(
            str(part)
            for part in parts
            if nonempty(part)
        )
    )


def deterministic_review_rank(
    document_id,
    salt,
):
    payload = (
        f"{salt}|{document_id}"
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def compact_excerpt(
    record,
    limit=650,
):
    text = str(
        record.get("text", "")
    ).strip()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text[:limit]


# --------------------------------------------------
# 3. Build concept evidence profiles
# --------------------------------------------------

review_concept_profiles_v0 = {}


for concept in review_concepts:

    normalized_concept = (
        normalize_fa_for_match(
            concept
        )
    )

    matching_document_ids = []

    source_counter = Counter()

    structural_counter = Counter()

    cooccurring_topic_counter = Counter()

    strong_cooccurring_topic_counter = Counter()

    for record in records:

        document_id = record[
            "document_id"
        ]

        review_text = (
            normalized_review_text(
                record
            )
        )

        if (
            normalized_concept
            not in review_text
        ):
            continue

        matching_document_ids.append(
            document_id
        )

        source_type = record[
            "source_type"
        ]

        source_counter[
            source_type
        ] += 1

        # --------------------------
        # Structural evidence
        # --------------------------

        if source_type == "website":

            normalized_tags = {
                normalize_fa_for_match(
                    tag_name
                )
                for tag_name in (
                    website_tag_names_for_record(
                        record
                    )
                )
            }

            if (
                normalized_concept
                in normalized_tags
            ):
                structural_counter[
                    "WEBSITE_TAG_EXACT"
                ] += 1

        elif source_type == "book":

            heading_path = (
                record.get(
                    "locator",
                    {},
                ).get(
                    "heading_path"
                )
                or record.get(
                    "source_metadata",
                    {},
                ).get(
                    "heading_path"
                )
            )

            normalized_heading_components = {
                normalize_fa_for_match(
                    component
                )
                for component in (
                    normalize_heading_path(
                        heading_path
                    )
                )
            }

            if (
                normalized_concept
                in normalized_heading_components
            ):
                structural_counter[
                    "BOOK_HEADING_EXACT"
                ] += 1

        # --------------------------
        # Existing Topic co-occurrence
        # --------------------------

        mapping_profile = (
            document_topic_mapping_v1_candidate[
                document_id
            ]
        )

        for candidate in mapping_profile[
            "candidates"
        ]:

            cooccurring_topic_counter[
                candidate["topic_id"]
            ] += 1

            if (
                candidate["decision"]
                == "STRONG_CANDIDATE"
            ):
                strong_cooccurring_topic_counter[
                    candidate["topic_id"]
                ] += 1

    review_concept_profiles_v0[
        concept
    ] = {
        "concept": concept,
        "normalized_concept": (
            normalized_concept
        ),
        "document_ids": (
            matching_document_ids
        ),
        "document_count": len(
            matching_document_ids
        ),
        "source_counts": dict(
            source_counter
        ),
        "structural_evidence": dict(
            structural_counter
        ),
        "topic_cooccurrence": dict(
            cooccurring_topic_counter
        ),
        "strong_topic_cooccurrence": dict(
            strong_cooccurring_topic_counter
        ),
        "existing_review_reason": (
            taxonomy_gap_registry_v0[
                "review_required"
            ][concept]
        ),
        "expert_disposition": (
            "PENDING"
        ),
    }


# --------------------------------------------------
# 4. Deterministic review samples
#    up to 2 per Source per concept
# --------------------------------------------------

concept_review_samples_v0 = defaultdict(
    list
)


for concept, profile in (
    review_concept_profiles_v0.items()
):

    for source_type in (
        "website",
        "book",
        "transcript",
    ):

        candidates = [
            document_id
            for document_id in profile[
                "document_ids"
            ]
            if (
                record_lookup[
                    document_id
                ]["source_type"]
                == source_type
            )
        ]

        ranked = sorted(
            candidates,
            key=lambda document_id: (
                deterministic_review_rank(
                    document_id,
                    (
                        f"NB10-expert-concept:"
                        f"{concept}:"
                        f"{source_type}:v0"
                    ),
                )
            ),
        )

        for document_id in ranked[:2]:

            mapping_profile = (
                document_topic_mapping_v1_candidate[
                    document_id
                ]
            )

            concept_review_samples_v0[
                concept
            ].append(
                {
                    "document_id": document_id,
                    "source_type": source_type,
                    "topic_candidates": [
                        {
                            "topic_id": candidate[
                                "topic_id"
                            ],
                            "decision": candidate[
                                "decision"
                            ],
                            "confidence": candidate[
                                "confidence"
                            ],
                        }
                        for candidate in (
                            mapping_profile[
                                "candidates"
                            ]
                        )
                    ],
                }
            )


# --------------------------------------------------
# 5. Print concept evidence profiles
# --------------------------------------------------

print("\nReview Required Concept Evidence:")


for concept in review_concepts:

    profile = (
        review_concept_profiles_v0[
            concept
        ]
    )

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "Current reason:",
        profile[
            "existing_review_reason"
        ],
    )

    print(
        "Matching documents:",
        profile["document_count"],
    )

    print(
        "Source counts:",
        profile["source_counts"],
    )

    print(
        "Structural evidence:",
        profile[
            "structural_evidence"
        ],
    )

    print(
        "Expert disposition:",
        profile[
            "expert_disposition"
        ],
    )

    print(
        "Topic co-occurrence:"
    )

    topic_counts = Counter(
        profile[
            "topic_cooccurrence"
        ]
    )

    if topic_counts:

        for topic_id, count in (
            topic_counts.most_common()
        ):
            print(
                f"- {topic_seed_v0[topic_id]['label_fa']}: "
                f"{count}"
            )

    else:
        print("- None")


    print(
        "Strong Topic co-occurrence:"
    )

    strong_counts = Counter(
        profile[
            "strong_topic_cooccurrence"
        ]
    )

    if strong_counts:

        for topic_id, count in (
            strong_counts.most_common()
        ):
            print(
                f"- {topic_seed_v0[topic_id]['label_fa']}: "
                f"{count}"
            )

    else:
        print("- None")


    print(
        "Representative evidence:"
    )

    samples = (
        concept_review_samples_v0[
            concept
        ]
    )

    if not samples:
        print("- No representative sample")
        continue

    for sample in samples:

        document_id = sample[
            "document_id"
        ]

        record = record_lookup[
            document_id
        ]

        print(
            "\n  ------------------------------------"
        )

        print(
            "  Source:",
            sample["source_type"],
        )

        print(
            "  Document:",
            document_id,
        )

        print(
            "  Title:",
            record.get("title"),
        )

        if (
            sample["source_type"]
            == "website"
        ):
            print(
                "  Website tags:",
                website_tag_names_for_record(
                    record
                ),
            )

        elif (
            sample["source_type"]
            == "book"
        ):
            heading_path = (
                record.get(
                    "locator",
                    {},
                ).get(
                    "heading_path"
                )
                or record.get(
                    "source_metadata",
                    {},
                ).get(
                    "heading_path"
                )
            )

            print(
                "  Book heading path:",
                heading_path,
            )

        elif (
            sample["source_type"]
            == "transcript"
        ):
            metadata = record.get(
                "source_metadata",
                {},
            )

            print(
                "  Program:",
                metadata.get(
                    "program"
                ),
            )

        print(
            "  Existing Topic candidates:"
        )

        if sample[
            "topic_candidates"
        ]:

            for candidate in sample[
                "topic_candidates"
            ]:
                print(
                    "  -",
                    topic_seed_v0[
                        candidate[
                            "topic_id"
                        ]
                    ]["label_fa"],
                    "|",
                    candidate[
                        "decision"
                    ],
                    "| confidence=",
                    candidate[
                        "confidence"
                    ],
                )

        else:
            print(
                "  - None"
            )

        print(
            "  Excerpt:"
        )

        print(
            "  ",
            compact_excerpt(
                record
            )
            or "<EMPTY>",
        )


# --------------------------------------------------
# 6. Multi-Strong Topic conflict profile
# --------------------------------------------------

strong_topic_pair_counter = Counter()


for document_id, topic_ids in (
    documents_with_multiple_strong_topics.items()
):

    topic_ids = sorted(
        topic_ids
    )

    for index, topic_a in enumerate(
        topic_ids
    ):
        for topic_b in topic_ids[
            index + 1:
        ]:
            strong_topic_pair_counter[
                (
                    topic_a,
                    topic_b,
                )
            ] += 1


print(
    "\n\nMulti-Strong Topic Pair Evidence:"
)

print(
    "- Distinct Strong Topic pairs:",
    len(
        strong_topic_pair_counter
    ),
)


for (
    topic_pair,
    count,
) in strong_topic_pair_counter.most_common():

    print(
        "-",
        topic_seed_v0[
            topic_pair[0]
        ]["label_fa"],
        "+",
        topic_seed_v0[
            topic_pair[1]
        ]["label_fa"],
        ":",
        count,
    )


# --------------------------------------------------
# 7. One deterministic example per Strong pair
# --------------------------------------------------

strong_pair_review_samples_v0 = []


for topic_pair, pair_count in (
    strong_topic_pair_counter.items()
):

    candidate_document_ids = []

    for document_id, topic_ids in (
        documents_with_multiple_strong_topics.items()
    ):

        if (
            topic_pair[0] in topic_ids
            and topic_pair[1] in topic_ids
        ):
            candidate_document_ids.append(
                document_id
            )

    candidate_document_ids = sorted(
        candidate_document_ids,
        key=lambda document_id: (
            deterministic_review_rank(
                document_id,
                (
                    f"NB10-strong-pair:"
                    f"{topic_pair[0]}:"
                    f"{topic_pair[1]}:v0"
                ),
            )
        ),
    )

    if not candidate_document_ids:
        continue

    strong_pair_review_samples_v0.append(
        {
            "topic_pair": topic_pair,
            "pair_count": pair_count,
            "document_id": (
                candidate_document_ids[0]
            ),
            "expert_disposition": "PENDING",
        }
    )


print(
    "\nRepresentative Multi-Strong Review samples:"
)


for index, sample in enumerate(
    strong_pair_review_samples_v0,
    start=1,
):

    document_id = sample[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    print(
        "\n========================================"
    )

    print(
        f"Strong Pair Review {index}"
    )

    print(
        "Pair:",
        topic_seed_v0[
            sample[
                "topic_pair"
            ][0]
        ]["label_fa"],
        "+",
        topic_seed_v0[
            sample[
                "topic_pair"
            ][1]
        ]["label_fa"],
    )

    print(
        "Pair population:",
        sample["pair_count"],
    )

    print(
        "Document:",
        document_id,
    )

    print(
        "Source:",
        record[
            "source_type"
        ],
    )

    print(
        "Title:",
        record.get(
            "title"
        ),
    )

    print(
        "Strong candidates:"
    )

    for candidate in profile[
        "candidates"
    ]:

        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        ):
            print(
                "-",
                candidate[
                    "topic_label_fa"
                ],
                "| confidence=",
                candidate[
                    "confidence"
                ],
                "| reason=",
                candidate[
                    "reason"
                ],
            )

    print(
        "Expert disposition:",
        sample[
            "expert_disposition"
        ],
    )

    print(
        "Excerpt:"
    )

    print(
        compact_excerpt(
            record,
            limit=800,
        )
        or "<EMPTY>"
    )


# --------------------------------------------------
# 8. Review packet status
# --------------------------------------------------

concepts_without_profile = [
    concept
    for concept in review_concepts
    if concept not in (
        review_concept_profiles_v0
    )
]


concepts_auto_resolved = [
    concept
    for concept, profile in (
        review_concept_profiles_v0.items()
    )
    if profile[
        "expert_disposition"
    ] != "PENDING"
]


strong_pairs_auto_resolved = [
    row
    for row in (
        strong_pair_review_samples_v0
    )
    if row[
        "expert_disposition"
    ] != "PENDING"
]


expert_review_packet_status_v0 = {
    "packet_generated": True,
    "concept_review_items": len(
        review_concept_profiles_v0
    ),
    "strong_pair_review_items": len(
        strong_pair_review_samples_v0
    ),
    "expert_review_completed": False,
    "freeze_allowed": False,
}


print("\nExpert Review Packet status:")

for key, value in (
    expert_review_packet_status_v0.items()
):
    print(
        f"- {key}: {value}"
    )


print("\nValidation:")

print(
    "- Review concepts expected:",
    10,
)

print(
    "- Review concepts profiled:",
    len(
        review_concept_profiles_v0
    ),
)

print(
    "- Concepts without profile:",
    len(
        concepts_without_profile
    ),
)

print(
    "- Concepts auto-resolved:",
    len(
        concepts_auto_resolved
    ),
)

print(
    "- Strong pairs profiled:",
    len(
        strong_pair_review_samples_v0
    ),
)

print(
    "- Strong pairs auto-resolved:",
    len(
        strong_pairs_auto_resolved
    ),
)

print(
    "- Expert review completed:",
    expert_review_packet_status_v0[
        "expert_review_completed"
    ],
)

print(
    "- Freeze executed:",
    False,
)


assert len(
    review_concept_profiles_v0
) == 10

assert not concepts_without_profile

assert not concepts_auto_resolved

assert len(
    strong_pair_review_samples_v0
) == len(
    strong_topic_pair_counter
)

assert not (
    strong_pairs_auto_resolved
)

assert (
    expert_review_packet_status_v0[
        "expert_review_completed"
    ]
    is False
)

assert (
    expert_review_packet_status_v0[
        "freeze_allowed"
    ]
    is False
)


print("\nCODE CELL 21 GATE: PASS")
print(
    "Expert Review Packet generated successfully; "
    "all unresolved Taxonomy and Multi-Strong items "
    "remain pending explicit specialist review."
)

Expert Review Packet:
- Review Required concepts: 10
- Multi-Strong Topic documents: 96

Review Required Concept Evidence:

Concept: بخشش
Current reason: Financial/social relevance exists, but Parent Topic is not yet sufficiently established.
Matching documents: 84
Source counts: {'website': 33, 'book': 17, 'transcript': 34}
Structural evidence: {'WEBSITE_TAG_EXACT': 14}
Expert disposition: PENDING
Topic co-occurrence:
- خرج، پس‌انداز و مدیریت پول: 39
- مبانی و توانمندی سواد مالی: 14
- سرمایه‌گذاری: 14
- اعتبار، قرض، بدهی و بانک: 13
- کار، شغل و درآمد: 12
- تصمیم‌گیری مالی: 6
- ریسک و بیمه: 4
Strong Topic co-occurrence:
- خرج، پس‌انداز و مدیریت پول: 15
- مبانی و توانمندی سواد مالی: 7
- تصمیم‌گیری مالی: 2
- ریسک و بیمه: 1
Representative evidence:

  ------------------------------------
  Source: website
  Document: website:5E0005A9
  Title: آموزه‌های سواد مالی در گلستان-4
  Website tags: ['سواد مالی در ادبیات']
  Existing Topic candidates:
  - None
  Excerpt:
   عطایش را به لقایش بخشیدم

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Specialist Review Decision Draft — بدون اعمال تصمیم نهایی</h3>

<p>
Evidence Packet نشان داد موارد Review Required ماهیت یکسانی ندارند.
بنابراین پیش از Freeze، برای هر مورد یک Recommendation اولیه ساخته می‌شود تا متخصص بتواند آن را تأیید، اصلاح یا رد کند.
</p>

<h4>انواع Recommendation</h4>

<ul>
<li><strong>PROPOSE_SUBTOPIC:</strong> شواهد کافی برای قرار گرفتن مفهوم زیر یکی از Topicهای موجود دیده می‌شود.</li>
<li><strong>PROPOSE_TAXONOMY_EXTENSION:</strong> مفهوم معتبر و مهم است ولی Parent مناسب آن هنوز در Taxonomy فعلی روشن نیست.</li>
<li><strong>PROPOSE_CROSS_CUTTING_DIMENSION:</strong> مفهوم روی چند Topic اثر می‌گذارد و نباید به یک Topic محدود شود.</li>
<li><strong>KEEP_CONTEXT_DEPENDENT:</strong> واژه چندمعنایی یا بسیار عمومی است و فقط با Context باید Mapping شود.</li>
</ul>

<h4>اصل مهم</h4>

<p>
Recommendationهای این Cell تصمیم Expert محسوب نمی‌شوند.
تمام آن‌ها با وضعیت <code>PENDING_SPECIALIST_APPROVAL</code> باقی می‌مانند.
</p>

<h4>Multi-Strong Topic</h4>

<p>
وجود بیش از یک Strong Topic الزاماً Conflict نیست.
در Website چند Tag معتبر می‌تواند واقعاً نشان‌دهنده محتوای چندموضوعی باشد.
در Book نیز ممکن است Heading Path شامل Parent و Child باشد و Strong Mapping چندگانه صرفاً ناشی از Hierarchical Context باشد.
</p>

<p>
این دو حالت باید قبل از Freeze از یکدیگر تفکیک شوند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>برای هر ۱۰ مفهوم یک Recommendation Draft وجود داشته باشد.</li>
<li>هیچ Recommendation به‌عنوان تصمیم Expert ثبت نشود.</li>
<li>Topic IDهای پیشنهادی معتبر باشند.</li>
<li>واژه‌های چندمعنایی به Global Alias تبدیل نشوند.</li>
<li>Multi-Strongهای Website و Book از نظر Evidence Origin تحلیل شوند.</li>
<li>Taxonomy هنوز Freeze نشود.</li>
</ul>

</div>

In [65]:
# Code Cell 22

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Draft specialist recommendations
#
# IMPORTANT:
# These are evidence-based recommendations,
# NOT approved specialist decisions.
# --------------------------------------------------

specialist_decision_draft_v0 = {
    "بخشش": {
        "recommendation": "PROPOSE_SUBTOPIC",
        "proposed_parent_topic_id": (
            "spending_saving_money_management"
        ),
        "proposed_subtopic_id": (
            "giving_and_financial_help"
        ),
        "confidence": 0.88,
        "reason": (
            "Evidence spans all three sources. "
            "Trainer material explicitly presents giving alongside "
            "saving, investing and spending as an allocation/use "
            "of money, while Website evidence also co-occurs most "
            "strongly with spending/saving/money management."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "بخشش و  هدیه": {
        "recommendation": "PROPOSE_SUBTOPIC",
        "proposed_parent_topic_id": (
            "spending_saving_money_management"
        ),
        "proposed_subtopic_id": (
            "giving_and_gifting"
        ),
        "confidence": 0.92,
        "reason": (
            "Explicit Book instructional section and reviewed "
            "content treat giving/gifting as a planned use of "
            "financial resources."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "مالیات": {
        "recommendation": (
            "PROPOSE_TAXONOMY_EXTENSION"
        ),
        "proposed_parent_topic_id": None,
        "proposed_subtopic_id": (
            "taxation_and_financial_obligations"
        ),
        "confidence": 0.93,
        "reason": (
            "Taxation has substantial standalone evidence "
            "across Website and Transcript, including dedicated "
            "teaching on types of taxation. The adult course also "
            "names taxation as a financial-literacy topic suitable "
            "for a separate level-two unit. Existing seven Topics "
            "do not provide an unambiguous Parent."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "اقتصاد رفتاری": {
        "recommendation": (
            "PROPOSE_CROSS_CUTTING_DIMENSION"
        ),
        "proposed_parent_topic_id": None,
        "proposed_subtopic_id": (
            "behavioral_economics_lens"
        ),
        "confidence": 0.96,
        "reason": (
            "Evidence spans spending, decision-making, "
            "investment and other Topics. It functions as a "
            "cross-cutting explanatory lens rather than a single "
            "financial-literacy Topic."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "انتخاب": {
        "recommendation": (
            "KEEP_CONTEXT_DEPENDENT"
        ),
        "proposed_parent_topic_id": None,
        "proposed_subtopic_id": None,
        "confidence": 0.99,
        "reason": (
            "The term is highly generic and occurs across all "
            "Topics. Global Topic aliasing would create substantial "
            "false-positive mapping."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "قیمت": {
        "recommendation": (
            "KEEP_CONTEXT_DEPENDENT"
        ),
        "proposed_parent_topic_id": None,
        "proposed_subtopic_id": None,
        "confidence": 0.96,
        "reason": (
            "Price appears both as a foundational economic concept "
            "and inside entrepreneurship/business instruction. "
            "Its Parent must be resolved from local instructional "
            "context."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "سود": {
        "recommendation": (
            "KEEP_CONTEXT_DEPENDENT"
        ),
        "proposed_parent_topic_id": None,
        "proposed_subtopic_id": None,
        "confidence": 0.99,
        "reason": (
            "The term is polysemous: investment return, bank-related "
            "profit and business profit appear in different contexts. "
            "A global alias would collapse distinct financial concepts."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "منابع تولید": {
        "recommendation": "PROPOSE_SUBTOPIC",
        "proposed_parent_topic_id": (
            "work_income"
        ),
        "proposed_subtopic_id": (
            "entrepreneurship_production_resources"
        ),
        "confidence": 0.98,
        "reason": (
            "Explicit Book structure places the concept under "
            "children's business instruction, and strong Topic "
            "co-occurrence is overwhelmingly work/income."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "تفاوت کالا و خدمت": {
        "recommendation": "PROPOSE_SUBTOPIC",
        "proposed_parent_topic_id": (
            "work_income"
        ),
        "proposed_subtopic_id": (
            "goods_and_services_basics"
        ),
        "confidence": 0.99,
        "reason": (
            "Eight structurally evidenced Book records place this "
            "concept directly under children's business instruction "
            "and all strong co-occurrence is work/income."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },

    "معرفی محصول و تبلیغ": {
        "recommendation": "PROPOSE_SUBTOPIC",
        "proposed_parent_topic_id": (
            "work_income"
        ),
        "proposed_subtopic_id": (
            "product_presentation_and_promotion"
        ),
        "confidence": 0.98,
        "reason": (
            "Explicit Book structure places product presentation "
            "and advertising under children's business instruction."
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    },
}


# --------------------------------------------------
# 2. Validate draft registry
# --------------------------------------------------

missing_review_concepts = (
    set(review_concepts)
    - set(specialist_decision_draft_v0)
)

unexpected_review_concepts = (
    set(specialist_decision_draft_v0)
    - set(review_concepts)
)


invalid_proposed_parent_topics = []

for concept, decision in (
    specialist_decision_draft_v0.items()
):
    parent_topic_id = decision[
        "proposed_parent_topic_id"
    ]

    if (
        parent_topic_id is not None
        and parent_topic_id not in topic_seed_v0
    ):
        invalid_proposed_parent_topics.append(
            (
                concept,
                parent_topic_id,
            )
        )


print("Specialist Decision Draft:")
print(
    "- Draft decisions:",
    len(specialist_decision_draft_v0),
)


for concept in review_concepts:

    decision = (
        specialist_decision_draft_v0[
            concept
        ]
    )

    parent_topic_id = decision[
        "proposed_parent_topic_id"
    ]

    parent_label = (
        topic_seed_v0[
            parent_topic_id
        ]["label_fa"]
        if parent_topic_id
        else "-"
    )

    print(
        "\n----------------------------------------"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "Recommendation:",
        decision[
            "recommendation"
        ],
    )

    print(
        "Proposed Parent:",
        parent_label,
    )

    print(
        "Proposed Subtopic:",
        decision[
            "proposed_subtopic_id"
        ],
    )

    print(
        "Confidence:",
        decision[
            "confidence"
        ],
    )

    print(
        "Reason:",
        decision[
            "reason"
        ],
    )

    print(
        "Specialist status:",
        decision[
            "specialist_status"
        ],
    )


recommendation_counter = Counter(
    decision["recommendation"]
    for decision in (
        specialist_decision_draft_v0.values()
    )
)


print("\nRecommendation summary:")

for recommendation, count in (
    recommendation_counter.items()
):
    print(
        f"- {recommendation}: {count}"
    )


# --------------------------------------------------
# 3. Analyze all Multi-Strong pairs by
#    source and evidence origin
# --------------------------------------------------

strong_pair_evidence_profiles_v0 = {}


for topic_pair, pair_count in (
    strong_topic_pair_counter.items()
):

    pair_document_ids = []

    source_counter = Counter()

    both_website_tag_exact = 0
    book_hierarchical_path_risk = 0
    direct_content_support = 0

    for document_id, topic_ids in (
        documents_with_multiple_strong_topics.items()
    ):

        if not (
            topic_pair[0] in topic_ids
            and topic_pair[1] in topic_ids
        ):
            continue

        pair_document_ids.append(
            document_id
        )

        record = record_lookup[
            document_id
        ]

        source_type = record[
            "source_type"
        ]

        source_counter[
            source_type
        ] += 1

        profile = (
            document_topic_mapping_v1_candidate[
                document_id
            ]
        )

        pair_candidates = {
            candidate["topic_id"]: candidate
            for candidate in profile[
                "candidates"
            ]
            if (
                candidate["topic_id"]
                in topic_pair
                and candidate[
                    "decision"
                ]
                == "STRONG_CANDIDATE"
            )
        }

        if len(pair_candidates) != 2:
            continue

        evidence_types_a = {
            evidence["type"]
            for evidence in (
                pair_candidates[
                    topic_pair[0]
                ]["evidence"]
            )
        }

        evidence_types_b = {
            evidence["type"]
            for evidence in (
                pair_candidates[
                    topic_pair[1]
                ]["evidence"]
            )
        }

        if (
            source_type == "website"
            and "WEBSITE_TAG_EXACT"
            in evidence_types_a
            and "WEBSITE_TAG_EXACT"
            in evidence_types_b
        ):
            both_website_tag_exact += 1

        if source_type == "book":

            path_based_a = (
                "BOOK_HEADING_PATH"
                in evidence_types_a
            )

            path_based_b = (
                "BOOK_HEADING_PATH"
                in evidence_types_b
            )

            stronger_local_a = bool(
                evidence_types_a
                & {
                    "BOOK_ACTIVITY_TOPIC_CONTEXT",
                    "BOOK_HEADING_RECORD",
                    "BODY_LEXICAL",
                }
            )

            stronger_local_b = bool(
                evidence_types_b
                & {
                    "BOOK_ACTIVITY_TOPIC_CONTEXT",
                    "BOOK_HEADING_RECORD",
                    "BODY_LEXICAL",
                }
            )

            if (
                path_based_a
                and path_based_b
                and not (
                    stronger_local_a
                    and stronger_local_b
                )
            ):
                book_hierarchical_path_risk += 1

        if (
            (
                "BODY_LEXICAL"
                in evidence_types_a
                or "BOOK_ACTIVITY_TOPIC_CONTEXT"
                in evidence_types_a
                or "BOOK_HEADING_RECORD"
                in evidence_types_a
            )
            and
            (
                "BODY_LEXICAL"
                in evidence_types_b
                or "BOOK_ACTIVITY_TOPIC_CONTEXT"
                in evidence_types_b
                or "BOOK_HEADING_RECORD"
                in evidence_types_b
            )
        ):
            direct_content_support += 1


    # ----------------------------------------------
    # Evidence-based pair recommendation.
    # Still not an expert decision.
    # ----------------------------------------------

    if (
        pair_count > 0
        and both_website_tag_exact
        == pair_count
    ):
        recommendation = (
            "LIKELY_VALID_MULTI_TOPIC"
        )

    elif (
        book_hierarchical_path_risk > 0
        and direct_content_support
        < pair_count
    ):
        recommendation = (
            "HIERARCHICAL_CONTEXT_REVIEW"
        )

    else:
        recommendation = (
            "MULTI_TOPIC_REVIEW_REQUIRED"
        )


    strong_pair_evidence_profiles_v0[
        topic_pair
    ] = {
        "pair_count": pair_count,
        "document_ids": pair_document_ids,
        "source_counts": dict(
            source_counter
        ),
        "both_website_tag_exact": (
            both_website_tag_exact
        ),
        "book_hierarchical_path_risk": (
            book_hierarchical_path_risk
        ),
        "direct_content_support": (
            direct_content_support
        ),
        "draft_recommendation": (
            recommendation
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    }


print(
    "\n\nMulti-Strong Pair Review Draft:"
)


pair_recommendation_counter = Counter()


for topic_pair, profile in sorted(
    strong_pair_evidence_profiles_v0.items(),
    key=lambda item: (
        -item[1]["pair_count"],
        item[0],
    ),
):

    pair_recommendation_counter[
        profile[
            "draft_recommendation"
        ]
    ] += 1

    print(
        "\n----------------------------------------"
    )

    print(
        "Pair:",
        topic_seed_v0[
            topic_pair[0]
        ]["label_fa"],
        "+",
        topic_seed_v0[
            topic_pair[1]
        ]["label_fa"],
    )

    print(
        "Documents:",
        profile[
            "pair_count"
        ],
    )

    print(
        "Source counts:",
        profile[
            "source_counts"
        ],
    )

    print(
        "Both Website Tag Exact:",
        profile[
            "both_website_tag_exact"
        ],
    )

    print(
        "Book hierarchical-path risk:",
        profile[
            "book_hierarchical_path_risk"
        ],
    )

    print(
        "Direct content support:",
        profile[
            "direct_content_support"
        ],
    )

    print(
        "Draft recommendation:",
        profile[
            "draft_recommendation"
        ],
    )

    print(
        "Specialist status:",
        profile[
            "specialist_status"
        ],
    )


print(
    "\nPair recommendation summary:"
)

for recommendation, count in (
    pair_recommendation_counter.items()
):
    print(
        f"- {recommendation}: {count}"
    )


# --------------------------------------------------
# 4. Confirm nothing was auto-approved
# --------------------------------------------------

auto_approved_concepts = [
    concept
    for concept, decision in (
        specialist_decision_draft_v0.items()
    )
    if (
        decision[
            "specialist_status"
        ]
        != "PENDING_SPECIALIST_APPROVAL"
    )
]


auto_approved_pairs = [
    topic_pair
    for topic_pair, profile in (
        strong_pair_evidence_profiles_v0.items()
    )
    if (
        profile[
            "specialist_status"
        ]
        != "PENDING_SPECIALIST_APPROVAL"
    )
]


taxonomy_mutation_performed = False
freeze_executed = False


print("\nReview Draft status:")

print(
    "- Concept recommendations:",
    len(
        specialist_decision_draft_v0
    ),
)

print(
    "- Strong-pair recommendations:",
    len(
        strong_pair_evidence_profiles_v0
    ),
)

print(
    "- Auto-approved concepts:",
    len(
        auto_approved_concepts
    ),
)

print(
    "- Auto-approved pairs:",
    len(
        auto_approved_pairs
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Freeze executed:",
    freeze_executed,
)


print("\nValidation:")

print(
    "- Missing Review concepts:",
    len(
        missing_review_concepts
    ),
)

print(
    "- Unexpected Review concepts:",
    len(
        unexpected_review_concepts
    ),
)

print(
    "- Invalid proposed Parent Topics:",
    len(
        invalid_proposed_parent_topics
    ),
)

print(
    "- Pair profiles expected:",
    len(
        strong_topic_pair_counter
    ),
)

print(
    "- Pair profiles produced:",
    len(
        strong_pair_evidence_profiles_v0
    ),
)


assert not missing_review_concepts
assert not unexpected_review_concepts
assert not invalid_proposed_parent_topics

assert len(
    specialist_decision_draft_v0
) == 10

assert len(
    strong_pair_evidence_profiles_v0
) == len(
    strong_topic_pair_counter
)

assert not auto_approved_concepts
assert not auto_approved_pairs

assert taxonomy_mutation_performed is False
assert freeze_executed is False


print("\nCODE CELL 22 GATE: PASS")
print(
    "Evidence-based Specialist Decision Draft "
    "generated successfully; no recommendation "
    "has been treated as an approved expert decision."
)

Specialist Decision Draft:
- Draft decisions: 10

----------------------------------------
Concept: بخشش
Recommendation: PROPOSE_SUBTOPIC
Proposed Parent: خرج، پس‌انداز و مدیریت پول
Proposed Subtopic: giving_and_financial_help
Confidence: 0.88
Reason: Evidence spans all three sources. Trainer material explicitly presents giving alongside saving, investing and spending as an allocation/use of money, while Website evidence also co-occurs most strongly with spending/saving/money management.
Specialist status: PENDING_SPECIALIST_APPROVAL

----------------------------------------
Concept: بخشش و  هدیه
Recommendation: PROPOSE_SUBTOPIC
Proposed Parent: خرج، پس‌انداز و مدیریت پول
Proposed Subtopic: giving_and_gifting
Confidence: 0.92
Reason: Explicit Book instructional section and reviewed content treat giving/gifting as a planned use of financial resources.
Specialist status: PENDING_SPECIALIST_APPROVAL

----------------------------------------
Concept: مالیات
Recommendation: PROPOSE_TAXONOMY

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Hierarchical Multi-Strong Audit برای Book</h3>

<p>
در Specialist Decision Draft، از ۱۸ جفت Topic چندگانه، ۱۴ مورد عمدتاً چندموضوعی معتبر به نظر می‌رسند و ۴ مورد به دلیل ساختار Heading کتاب نیازمند بررسی دقیق‌تر هستند.
</p>

<p>
هدف این مرحله تشخیص این است که آیا دو Strong Topic واقعاً در محتوای همان Document حضور دارند، یا یکی از آن‌ها فقط از Heading Path سطح بالاتر به ارث رسیده است.
</p>

<h4>اصل Evidence</h4>

<ul>
<li><strong>LOCAL_STRONG:</strong> Topic دارای Activity Context، Heading Record یا Body Evidence است.</li>
<li><strong>PATH_ONLY_STRONG:</strong> Strong بودن فقط از Heading Path آمده است.</li>
<li><strong>MIXED:</strong> Heading Path همراه با Evidence محلی وجود دارد.</li>
</ul>

<p>
اگر یک Document برای Topic اول Evidence محلی داشته باشد ولی Topic دوم فقط از Parent Heading آمده باشد، این وضعیت به‌عنوان Candidate برای Demotion ثبت می‌شود؛ اما هنوز Mapping تغییر نمی‌کند.
</p>

<h4>هدف</h4>

<p>
کاهش False Multi-topic ناشی از ساختار سلسله‌مراتبی کتاب، بدون حذف اطلاعات Heading و بدون تغییر خودکار Taxonomy.
</p>

<h4>معیار PASS</h4>

<ul>
<li>فقط چهار Pair با وضعیت <code>HIERARCHICAL_CONTEXT_REVIEW</code> بررسی شوند.</li>
<li>تمام Documentهای مربوط به آن‌ها Audit شوند.</li>
<li>Evidence محلی و Path-only از هم تفکیک شود.</li>
<li>هیچ Strong Candidate به‌طور خودکار Demote نشود.</li>
<li>هیچ Mapping روی Disk نوشته نشود.</li>
</ul>

</div>

In [66]:
# Code Cell 23

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Select only hierarchical-review pairs
# --------------------------------------------------

hierarchical_review_pairs = {
    topic_pair: profile
    for topic_pair, profile in (
        strong_pair_evidence_profiles_v0.items()
    )
    if (
        profile["draft_recommendation"]
        == "HIERARCHICAL_CONTEXT_REVIEW"
    )
}


print("Hierarchical Multi-Strong Audit:")
print(
    "- Pairs requiring hierarchical review:",
    len(hierarchical_review_pairs),
)


for topic_pair, profile in (
    hierarchical_review_pairs.items()
):
    print(
        "-",
        topic_seed_v0[
            topic_pair[0]
        ]["label_fa"],
        "+",
        topic_seed_v0[
            topic_pair[1]
        ]["label_fa"],
        ":",
        profile["pair_count"],
    )


# --------------------------------------------------
# 2. Evidence classification helper
# --------------------------------------------------

local_topic_evidence_types = {
    "BOOK_ACTIVITY_TOPIC_CONTEXT",
    "BOOK_HEADING_RECORD",
    "BODY_LEXICAL",
    "TITLE_LEXICAL",
}


def classify_strong_candidate_evidence(
    candidate,
):
    evidence_types = {
        evidence["type"]
        for evidence in candidate[
            "evidence"
        ]
    }

    has_path = (
        "BOOK_HEADING_PATH"
        in evidence_types
    )

    has_local = bool(
        evidence_types
        & local_topic_evidence_types
    )

    if has_path and has_local:
        evidence_class = "MIXED"

    elif has_local:
        evidence_class = "LOCAL_STRONG"

    elif has_path:
        evidence_class = "PATH_ONLY_STRONG"

    else:
        evidence_class = "OTHER_STRONG"

    return {
        "evidence_class": evidence_class,
        "evidence_types": sorted(
            evidence_types
        ),
        "has_path": has_path,
        "has_local": has_local,
    }


# --------------------------------------------------
# 3. Audit every document in the four pairs
# --------------------------------------------------

hierarchical_document_audit_v0 = []


for topic_pair, pair_profile in (
    hierarchical_review_pairs.items()
):

    for document_id in pair_profile[
        "document_ids"
    ]:

        record = record_lookup[
            document_id
        ]

        if record[
            "source_type"
        ] != "book":
            continue

        mapping_profile = (
            document_topic_mapping_v1_candidate[
                document_id
            ]
        )

        strong_candidate_lookup = {
            candidate["topic_id"]: candidate
            for candidate in mapping_profile[
                "candidates"
            ]
            if (
                candidate["decision"]
                == "STRONG_CANDIDATE"
            )
        }

        if not (
            topic_pair[0]
            in strong_candidate_lookup
            and topic_pair[1]
            in strong_candidate_lookup
        ):
            continue

        candidate_a = (
            strong_candidate_lookup[
                topic_pair[0]
            ]
        )

        candidate_b = (
            strong_candidate_lookup[
                topic_pair[1]
            ]
        )

        evidence_a = (
            classify_strong_candidate_evidence(
                candidate_a
            )
        )

        evidence_b = (
            classify_strong_candidate_evidence(
                candidate_b
            )
        )

        if (
            evidence_a["has_local"]
            and not evidence_b[
                "has_local"
            ]
        ):
            draft_disposition = (
                "REVIEW_DEMOTE_TOPIC_B_PATH_ONLY"
            )

        elif (
            evidence_b["has_local"]
            and not evidence_a[
                "has_local"
            ]
        ):
            draft_disposition = (
                "REVIEW_DEMOTE_TOPIC_A_PATH_ONLY"
            )

        elif (
            evidence_a["has_local"]
            and evidence_b[
                "has_local"
            ]
        ):
            draft_disposition = (
                "LIKELY_VALID_MULTI_TOPIC"
            )

        elif (
            evidence_a[
                "evidence_class"
            ]
            == "PATH_ONLY_STRONG"
            and evidence_b[
                "evidence_class"
            ]
            == "PATH_ONLY_STRONG"
        ):
            draft_disposition = (
                "BOTH_PATH_ONLY_REVIEW_REQUIRED"
            )

        else:
            draft_disposition = (
                "MANUAL_REVIEW_REQUIRED"
            )

        heading_path = (
            record.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            or record.get(
                "source_metadata",
                {},
            ).get(
                "heading_path"
            )
        )

        hierarchical_document_audit_v0.append(
            {
                "document_id": document_id,
                "topic_pair": topic_pair,
                "heading_path": heading_path,
                "topic_a": {
                    "topic_id": topic_pair[0],
                    "confidence": (
                        candidate_a[
                            "confidence"
                        ]
                    ),
                    **evidence_a,
                },
                "topic_b": {
                    "topic_id": topic_pair[1],
                    "confidence": (
                        candidate_b[
                            "confidence"
                        ]
                    ),
                    **evidence_b,
                },
                "draft_disposition": (
                    draft_disposition
                ),
                "specialist_status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            }
        )


print(
    "\n- Book documents audited:",
    len(
        hierarchical_document_audit_v0
    ),
)


# --------------------------------------------------
# 4. Disposition statistics
# --------------------------------------------------

hierarchical_disposition_counter = Counter(
    row["draft_disposition"]
    for row in hierarchical_document_audit_v0
)


print("\nDraft document dispositions:")

for disposition, count in (
    hierarchical_disposition_counter.most_common()
):
    print(
        f"- {disposition}: {count}"
    )


# --------------------------------------------------
# 5. Per-pair evidence summary
# --------------------------------------------------

hierarchical_pair_audit_summary_v0 = {}


for topic_pair in hierarchical_review_pairs:

    rows = [
        row
        for row in (
            hierarchical_document_audit_v0
        )
        if row[
            "topic_pair"
        ] == topic_pair
    ]

    disposition_counter = Counter(
        row[
            "draft_disposition"
        ]
        for row in rows
    )

    topic_a_class_counter = Counter(
        row["topic_a"][
            "evidence_class"
        ]
        for row in rows
    )

    topic_b_class_counter = Counter(
        row["topic_b"][
            "evidence_class"
        ]
        for row in rows
    )

    hierarchical_pair_audit_summary_v0[
        topic_pair
    ] = {
        "document_count": len(rows),
        "dispositions": dict(
            disposition_counter
        ),
        "topic_a_evidence_classes": dict(
            topic_a_class_counter
        ),
        "topic_b_evidence_classes": dict(
            topic_b_class_counter
        ),
    }


print("\nPer-pair Audit summary:")


for topic_pair, summary in (
    hierarchical_pair_audit_summary_v0.items()
):

    print(
        "\n----------------------------------------"
    )

    print(
        "Pair:",
        topic_seed_v0[
            topic_pair[0]
        ]["label_fa"],
        "+",
        topic_seed_v0[
            topic_pair[1]
        ]["label_fa"],
    )

    print(
        "Documents:",
        summary[
            "document_count"
        ],
    )

    print(
        "Dispositions:",
        summary[
            "dispositions"
        ],
    )

    print(
        "Topic A evidence classes:",
        summary[
            "topic_a_evidence_classes"
        ],
    )

    print(
        "Topic B evidence classes:",
        summary[
            "topic_b_evidence_classes"
        ],
    )


# --------------------------------------------------
# 6. Representative examples by disposition
# --------------------------------------------------

representative_hierarchical_examples_v0 = []


for disposition in sorted(
    hierarchical_disposition_counter
):

    candidates = [
        row
        for row in (
            hierarchical_document_audit_v0
        )
        if row[
            "draft_disposition"
        ] == disposition
    ]

    if not candidates:
        continue

    candidates = sorted(
        candidates,
        key=lambda row: (
            deterministic_review_rank(
                row["document_id"],
                (
                    f"NB10-hierarchical:"
                    f"{disposition}:v0"
                ),
            )
        ),
    )

    representative_hierarchical_examples_v0.append(
        candidates[0]
    )


print(
    "\nRepresentative Hierarchical Audit examples:"
)


for index, row in enumerate(
    representative_hierarchical_examples_v0,
    start=1,
):

    document_id = row[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    print(
        "\n========================================"
    )

    print(
        f"Hierarchical Example {index}"
    )

    print(
        "Document:",
        document_id,
    )

    print(
        "Title:",
        record.get("title"),
    )

    print(
        "Heading path:",
        row["heading_path"],
    )

    print(
        "Draft disposition:",
        row[
            "draft_disposition"
        ],
    )

    print(
        "Topic A:",
        topic_seed_v0[
            row["topic_a"][
                "topic_id"
            ]
        ]["label_fa"],
    )

    print(
        "- Evidence class:",
        row["topic_a"][
            "evidence_class"
        ],
    )

    print(
        "- Evidence types:",
        row["topic_a"][
            "evidence_types"
        ],
    )

    print(
        "Topic B:",
        topic_seed_v0[
            row["topic_b"][
                "topic_id"
            ]
        ]["label_fa"],
    )

    print(
        "- Evidence class:",
        row["topic_b"][
            "evidence_class"
        ],
    )

    print(
        "- Evidence types:",
        row["topic_b"][
            "evidence_types"
        ],
    )

    print(
        "Excerpt:"
    )

    print(
        compact_excerpt(
            record,
            limit=750,
        )
        or "<EMPTY>"
    )


# --------------------------------------------------
# 7. Potential path-only demotion inventory
#
# Still recommendation only.
# --------------------------------------------------

path_only_demotion_candidates_v0 = []


for row in hierarchical_document_audit_v0:

    disposition = row[
        "draft_disposition"
    ]

    if (
        disposition
        == "REVIEW_DEMOTE_TOPIC_A_PATH_ONLY"
    ):
        path_only_topic_id = (
            row["topic_a"][
                "topic_id"
            ]
        )

    elif (
        disposition
        == "REVIEW_DEMOTE_TOPIC_B_PATH_ONLY"
    ):
        path_only_topic_id = (
            row["topic_b"][
                "topic_id"
            ]
        )

    else:
        continue

    path_only_demotion_candidates_v0.append(
        {
            "document_id": row[
                "document_id"
            ],
            "topic_id": (
                path_only_topic_id
            ),
            "proposed_action": (
                "STRONG_TO_HIERARCHICAL_CONTEXT_ONLY"
            ),
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        }
    )


print(
    "\nPotential Path-only Demotion inventory:"
)

print(
    "- Candidate rows:",
    len(
        path_only_demotion_candidates_v0
    ),
)


demotion_topic_counter = Counter(
    row["topic_id"]
    for row in (
        path_only_demotion_candidates_v0
    )
)


for topic_id, count in (
    demotion_topic_counter.most_common()
):
    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"{count}"
    )


# --------------------------------------------------
# 8. Validation
# --------------------------------------------------

invalid_pair_rows = []

for row in hierarchical_document_audit_v0:

    if (
        row["topic_pair"]
        not in hierarchical_review_pairs
    ):
        invalid_pair_rows.append(
            row["document_id"]
        )


auto_demoted_rows = [
    row
    for row in (
        path_only_demotion_candidates_v0
    )
    if (
        row["specialist_status"]
        != "PENDING_SPECIALIST_APPROVAL"
    )
]


mapping_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Hierarchical pairs expected:",
    4,
)

print(
    "- Hierarchical pairs audited:",
    len(
        hierarchical_pair_audit_summary_v0
    ),
)

print(
    "- Invalid pair rows:",
    len(
        invalid_pair_rows
    ),
)

print(
    "- Path-only Demotion candidates:",
    len(
        path_only_demotion_candidates_v0
    ),
)

print(
    "- Auto-demoted rows:",
    len(
        auto_demoted_rows
    ),
)

print(
    "- Mapping mutation performed:",
    mapping_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    hierarchical_review_pairs
) == 4

assert len(
    hierarchical_pair_audit_summary_v0
) == 4

assert not invalid_pair_rows
assert not auto_demoted_rows

assert (
    mapping_mutation_performed
    is False
)

assert (
    disk_write_performed
    is False
)


print("\nCODE CELL 23 GATE: PASS")
print(
    "Hierarchical Multi-Strong Book audit "
    "completed successfully; path-only Strong "
    "candidates were identified without mutation."
)

Hierarchical Multi-Strong Audit:
- Pairs requiring hierarchical review: 4
- تصمیم‌گیری مالی + خرج، پس‌انداز و مدیریت پول : 30
- اعتبار، قرض، بدهی و بانک + خرج، پس‌انداز و مدیریت پول : 9
- اعتبار، قرض، بدهی و بانک + سرمایه‌گذاری : 15
- خرج، پس‌انداز و مدیریت پول + کار، شغل و درآمد : 2

- Book documents audited: 25

Draft document dispositions:
- BOTH_PATH_ONLY_REVIEW_REQUIRED: 10
- REVIEW_DEMOTE_TOPIC_B_PATH_ONLY: 7
- REVIEW_DEMOTE_TOPIC_A_PATH_ONLY: 6
- LIKELY_VALID_MULTI_TOPIC: 2

Per-pair Audit summary:

----------------------------------------
Pair: تصمیم‌گیری مالی + خرج، پس‌انداز و مدیریت پول
Documents: 6
Dispositions: {'REVIEW_DEMOTE_TOPIC_B_PATH_ONLY': 4, 'LIKELY_VALID_MULTI_TOPIC': 1, 'BOTH_PATH_ONLY_REVIEW_REQUIRED': 1}
Topic A evidence classes: {'MIXED': 5, 'PATH_ONLY_STRONG': 1}
Topic B evidence classes: {'PATH_ONLY_STRONG': 5, 'MIXED': 1}

----------------------------------------
Pair: اعتبار، قرض، بدهی و بانک + خرج، پس‌انداز و مدیریت پول
Documents: 2
Dispositions: {'REVIEW_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Global Book Hierarchical Context Audit</h3>

<p>
Audit چهار Multi-Strong Pair نشان داد که بخشی از Strong Topicهای Book فقط از Heading Path ارث گرفته شده‌اند.
این مسئله باید در کل Corpus کتاب بررسی شود، نه فقط در Documentهای چندموضوعی.
</p>

<p>
از نظر مدل دانش، دو مفهوم باید از هم جدا باشند:
</p>

<ul>
<li><strong>Document Topic:</strong> خود Document دارای Evidence محلی برای Topic است.</li>
<li><strong>Book Hierarchical Topic Scope:</strong> Document داخل یک فصل یا Section موضوعی قرار دارد، ولی خود Segment الزاماً Evidence مستقل آن Topic را ندارد.</li>
</ul>

<p>
این تفکیک مشابه تفکیک <code>Document Topic</code> و <code>Conversation Topic Scope</code> در Transcript است.
</p>

<h4>Evidence Classes</h4>

<ul>
<li><strong>LOCAL_DIRECT_STRONG:</strong> Evidence مستقیم از Body، Title یا Heading خود Record.</li>
<li><strong>ACTIVITY_CONTEXT_STRONG:</strong> Document داخل Activity ساختاری و Topic-دار قرار دارد.</li>
<li><strong>MIXED_LOCAL_HIERARCHICAL:</strong> Evidence محلی همراه با Heading Context.</li>
<li><strong>HIERARCHICAL_ONLY:</strong> Strong بودن فقط از Heading Path یا Alias ارث‌رسیده حاصل شده است.</li>
</ul>

<p>
در این مرحله هیچ Strong Candidate تغییر نمی‌کند.
فقط مشخص می‌شود اگر Hierarchical-only Evidence از Document Topic جدا شود، چه اثری روی Coverage خواهد داشت.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Strong Candidateهای Book بررسی شوند.</li>
<li>Heading Path و Heading Alias ارث‌رسیده از Evidence محلی جدا شوند.</li>
<li>Candidateهای Hierarchical-only شناسایی شوند.</li>
<li>Book Hierarchical Topic Scope پیشنهادی ساخته شود.</li>
<li>اثر احتمالی Demotion محاسبه شود.</li>
<li>هیچ Mapping واقعاً تغییر نکند.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [67]:
# Code Cell 24

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Evidence classes for ALL Book Strong candidates
# --------------------------------------------------

direct_local_book_evidence_types = {
    "BODY_LEXICAL",
    "TITLE_LEXICAL",
    "BOOK_HEADING_RECORD",
}

activity_context_book_evidence_types = {
    "BOOK_ACTIVITY_TOPIC_CONTEXT",
}

hierarchical_book_evidence_types = {
    "BOOK_HEADING_PATH",
    "BOOK_HEADING_ALIAS",
}


def classify_book_strong_evidence(
    record,
    candidate,
):
    evidence_types = {
        evidence["type"]
        for evidence in candidate[
            "evidence"
        ]
    }

    has_direct_local = bool(
        evidence_types
        & direct_local_book_evidence_types
    )

    has_activity_context = bool(
        evidence_types
        & activity_context_book_evidence_types
    )

    has_hierarchical = bool(
        evidence_types
        & hierarchical_book_evidence_types
    )

    # A BOOK_HEADING_ALIAS can be genuinely local
    # when the record itself is a heading.
    metadata = record.get(
        "source_metadata",
        {},
    )

    if (
        "BOOK_HEADING_ALIAS"
        in evidence_types
        and metadata.get("is_heading") is True
    ):
        has_direct_local = True

    if (
        has_direct_local
        and has_hierarchical
    ):
        evidence_class = (
            "MIXED_LOCAL_HIERARCHICAL"
        )

    elif (
        has_activity_context
        and has_hierarchical
    ):
        evidence_class = (
            "MIXED_ACTIVITY_HIERARCHICAL"
        )

    elif has_direct_local:
        evidence_class = (
            "LOCAL_DIRECT_STRONG"
        )

    elif has_activity_context:
        evidence_class = (
            "ACTIVITY_CONTEXT_STRONG"
        )

    elif has_hierarchical:
        evidence_class = (
            "HIERARCHICAL_ONLY"
        )

    else:
        evidence_class = (
            "OTHER_STRONG"
        )

    return {
        "evidence_class": evidence_class,
        "evidence_types": sorted(
            evidence_types
        ),
        "has_direct_local": (
            has_direct_local
        ),
        "has_activity_context": (
            has_activity_context
        ),
        "has_hierarchical": (
            has_hierarchical
        ),
    }


book_strong_candidate_audit_v0 = []


for record in book_records:

    document_id = record[
        "document_id"
    ]

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    for candidate in profile[
        "candidates"
    ]:

        if (
            candidate["decision"]
            != "STRONG_CANDIDATE"
        ):
            continue

        evidence_profile = (
            classify_book_strong_evidence(
                record,
                candidate,
            )
        )

        book_strong_candidate_audit_v0.append(
            {
                "document_id": document_id,
                "topic_id": candidate[
                    "topic_id"
                ],
                "confidence": candidate[
                    "confidence"
                ],
                **evidence_profile,
            }
        )


print("Global Book Strong Candidate Audit:")
print(
    "- Book documents:",
    len(book_records),
)
print(
    "- Strong Candidate rows:",
    len(
        book_strong_candidate_audit_v0
    ),
)


# --------------------------------------------------
# 2. Evidence-class distributions
# --------------------------------------------------

book_strong_evidence_class_counter = Counter(
    row["evidence_class"]
    for row in (
        book_strong_candidate_audit_v0
    )
)


print("\nStrong Candidate evidence classes:")

for evidence_class, count in (
    book_strong_evidence_class_counter.most_common()
):
    print(
        f"- {evidence_class}: {count}"
    )


book_strong_class_by_topic = defaultdict(
    Counter
)


for row in (
    book_strong_candidate_audit_v0
):
    book_strong_class_by_topic[
        row["topic_id"]
    ][
        row["evidence_class"]
    ] += 1


print("\nEvidence classes by Topic:")

for topic_id in topic_seed_v0:

    counter = (
        book_strong_class_by_topic[
            topic_id
        ]
    )

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    for evidence_class in (
        "LOCAL_DIRECT_STRONG",
        "ACTIVITY_CONTEXT_STRONG",
        "MIXED_LOCAL_HIERARCHICAL",
        "MIXED_ACTIVITY_HIERARCHICAL",
        "HIERARCHICAL_ONLY",
        "OTHER_STRONG",
    ):
        print(
            f"- {evidence_class}: "
            f"{counter[evidence_class]}"
        )


# --------------------------------------------------
# 3. Build proposed Book Hierarchical Topic Scope
#    from HIERARCHICAL_ONLY candidates
#
# Still no mutation.
# --------------------------------------------------

book_hierarchical_topic_scope_v0 = defaultdict(
    list
)


for row in (
    book_strong_candidate_audit_v0
):

    if (
        row["evidence_class"]
        != "HIERARCHICAL_ONLY"
    ):
        continue

    book_hierarchical_topic_scope_v0[
        row["document_id"]
    ].append(
        {
            "topic_id": row[
                "topic_id"
            ],
            "topic_label_fa": (
                topic_seed_v0[
                    row["topic_id"]
                ]["label_fa"]
            ),
            "decision": (
                "BOOK_HIERARCHICAL_TOPIC_SCOPE_CANDIDATE"
            ),
            "current_confidence": (
                row["confidence"]
            ),
            "reason": (
                "INHERITED_BOOK_HEADING_CONTEXT"
            ),
            "original_evidence_types": (
                row["evidence_types"]
            ),
        }
    )


hierarchical_only_rows = [
    row
    for row in (
        book_strong_candidate_audit_v0
    )
    if (
        row["evidence_class"]
        == "HIERARCHICAL_ONLY"
    )
]


hierarchical_only_document_ids = {
    row["document_id"]
    for row in hierarchical_only_rows
}


print("\nProposed Book Hierarchical Topic Scope:")

print(
    "- Hierarchical-only Candidate rows:",
    len(hierarchical_only_rows),
)

print(
    "- Unique affected Book documents:",
    len(
        hierarchical_only_document_ids
    ),
)


hierarchical_scope_topic_counter = Counter(
    row["topic_id"]
    for row in hierarchical_only_rows
)


for topic_id, count in (
    hierarchical_scope_topic_counter.most_common()
):
    print(
        f"- {topic_seed_v0[topic_id]['label_fa']}: "
        f"{count}"
    )


# --------------------------------------------------
# 4. Prospective impact if HIERARCHICAL_ONLY
#    is separated from Document Topic
#
# NO mapping change is performed.
# --------------------------------------------------

prospective_book_status_counter = Counter()

book_documents_losing_all_strong = []
book_documents_retaining_strong = []
book_documents_path_only_strong_only = []


for record in book_records:

    document_id = record[
        "document_id"
    ]

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    strong_candidates = [
        candidate
        for candidate in profile[
            "candidates"
        ]
        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        )
    ]

    lexical_candidates = [
        candidate
        for candidate in profile[
            "candidates"
        ]
        if (
            candidate["decision"]
            == "LEXICAL_CANDIDATE"
        )
    ]

    retained_strong = []

    path_only_strong = []

    for candidate in strong_candidates:

        audit_match = [
            row
            for row in (
                book_strong_candidate_audit_v0
            )
            if (
                row["document_id"]
                == document_id
                and row["topic_id"]
                == candidate["topic_id"]
            )
        ]

        assert len(
            audit_match
        ) == 1

        evidence_class = (
            audit_match[0][
                "evidence_class"
            ]
        )

        if (
            evidence_class
            == "HIERARCHICAL_ONLY"
        ):
            path_only_strong.append(
                candidate
            )
        else:
            retained_strong.append(
                candidate
            )

    if retained_strong:

        prospective_status = (
            "HAS_STRONG_DOCUMENT_TOPIC"
        )

        book_documents_retaining_strong.append(
            document_id
        )

    elif lexical_candidates:

        prospective_status = (
            "LEXICAL_ONLY_AFTER_SCOPE_SEPARATION"
        )

        if path_only_strong:
            book_documents_losing_all_strong.append(
                document_id
            )

    else:

        prospective_status = (
            "NO_DOCUMENT_TOPIC_AFTER_SCOPE_SEPARATION"
        )

        if path_only_strong:
            book_documents_losing_all_strong.append(
                document_id
            )

    if (
        strong_candidates
        and len(path_only_strong)
        == len(strong_candidates)
    ):
        book_documents_path_only_strong_only.append(
            document_id
        )

    prospective_book_status_counter[
        prospective_status
    ] += 1


print(
    "\nProspective Book status after "
    "Hierarchical Scope separation:"
)

for status, count in (
    prospective_book_status_counter.items()
):
    print(
        f"- {status}: "
        f"{count}/{len(book_records)} "
        f"({count / len(book_records):.1%})"
    )


print(
    "\nImpact:"
)

print(
    "- Current Book documents with Strong Topic:",
    v1_source_mapping_counter[
        "book"
    ]["HAS_STRONG_CANDIDATE"],
)

print(
    "- Book documents retaining at least "
    "one Strong Document Topic:",
    len(
        book_documents_retaining_strong
    ),
)

print(
    "- Book documents losing all Strong "
    "Document Topics:",
    len(
        set(
            book_documents_losing_all_strong
        )
    ),
)

print(
    "- Book documents whose Strong evidence "
    "is entirely Hierarchical-only:",
    len(
        set(
            book_documents_path_only_strong_only
        )
    ),
)


# --------------------------------------------------
# 5. What happens to documents losing Strong status?
# --------------------------------------------------

lost_strong_new_status_counter = Counter()


for document_id in set(
    book_documents_losing_all_strong
):

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    lexical_count = sum(
        candidate[
            "decision"
        ]
        == "LEXICAL_CANDIDATE"
        for candidate in profile[
            "candidates"
        ]
    )

    if lexical_count:
        lost_strong_new_status_counter[
            "BECOMES_LEXICAL_ONLY"
        ] += 1

    else:
        lost_strong_new_status_counter[
            "BECOMES_DOCUMENT_TOPIC_UNMAPPED"
        ] += 1


print(
    "\nDocuments losing Strong status:"
)

for status, count in (
    lost_strong_new_status_counter.items()
):
    print(
        f"- {status}: {count}"
    )


# --------------------------------------------------
# 6. Representative Hierarchical-only samples
#    one per Topic when available
# --------------------------------------------------

book_hierarchical_review_samples_v0 = []


for topic_id in topic_seed_v0:

    candidates = [
        row
        for row in hierarchical_only_rows
        if (
            row["topic_id"]
            == topic_id
        )
    ]

    if not candidates:
        continue

    ranked = sorted(
        candidates,
        key=lambda row: (
            deterministic_review_rank(
                row["document_id"],
                (
                    f"NB10-global-book-hierarchy:"
                    f"{topic_id}:v0"
                ),
            )
        ),
    )

    book_hierarchical_review_samples_v0.append(
        ranked[0]
    )


print(
    "\nRepresentative Hierarchical-only samples:"
)


for index, sample in enumerate(
    book_hierarchical_review_samples_v0,
    start=1,
):

    document_id = sample[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    heading_path = (
        record.get(
            "locator",
            {},
        ).get(
            "heading_path"
        )
        or record.get(
            "source_metadata",
            {},
        ).get(
            "heading_path"
        )
    )

    print(
        "\n========================================"
    )

    print(
        f"Book Hierarchy Sample {index}"
    )

    print(
        "Document:",
        document_id,
    )

    print(
        "Topic:",
        topic_seed_v0[
            sample["topic_id"]
        ]["label_fa"],
    )

    print(
        "Heading path:",
        heading_path,
    )

    print(
        "Evidence types:",
        sample[
            "evidence_types"
        ],
    )

    print(
        "Excerpt:"
    )

    print(
        compact_excerpt(
            record,
            limit=650,
        )
        or "<EMPTY>"
    )


# --------------------------------------------------
# 7. Validation
# --------------------------------------------------

invalid_hierarchical_scope_topic_ids = []

for document_id, scopes in (
    book_hierarchical_topic_scope_v0.items()
):

    for scope in scopes:

        if (
            scope["topic_id"]
            not in topic_seed_v0
        ):
            invalid_hierarchical_scope_topic_ids.append(
                (
                    document_id,
                    scope[
                        "topic_id"
                    ],
                )
            )


audited_candidate_key_set = {
    (
        row["document_id"],
        row["topic_id"],
    )
    for row in (
        book_strong_candidate_audit_v0
    )
}


expected_candidate_key_set = set()


for record in book_records:

    document_id = record[
        "document_id"
    ]

    profile = (
        document_topic_mapping_v1_candidate[
            document_id
        ]
    )

    for candidate in profile[
        "candidates"
    ]:

        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        ):
            expected_candidate_key_set.add(
                (
                    document_id,
                    candidate[
                        "topic_id"
                    ],
                )
            )


missing_book_strong_audit_rows = (
    expected_candidate_key_set
    - audited_candidate_key_set
)


unexpected_book_strong_audit_rows = (
    audited_candidate_key_set
    - expected_candidate_key_set
)


mapping_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Expected Book Strong Candidate rows:",
    len(
        expected_candidate_key_set
    ),
)

print(
    "- Audited Book Strong Candidate rows:",
    len(
        audited_candidate_key_set
    ),
)

print(
    "- Missing audit rows:",
    len(
        missing_book_strong_audit_rows
    ),
)

print(
    "- Unexpected audit rows:",
    len(
        unexpected_book_strong_audit_rows
    ),
)

print(
    "- Invalid Hierarchical Scope Topic IDs:",
    len(
        invalid_hierarchical_scope_topic_ids
    ),
)

print(
    "- Mapping mutation performed:",
    mapping_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert expected_candidate_key_set

assert (
    audited_candidate_key_set
    == expected_candidate_key_set
)

assert not (
    invalid_hierarchical_scope_topic_ids
)

assert (
    mapping_mutation_performed
    is False
)

assert (
    disk_write_performed
    is False
)


print("\nCODE CELL 24 GATE: PASS")
print(
    "Global Book hierarchical-context audit "
    "completed successfully; inherited Topic scope "
    "has been separated analytically from direct "
    "Document Topic evidence without mutation."
)

Global Book Strong Candidate Audit:
- Book documents: 2701
- Strong Candidate rows: 2218

Strong Candidate evidence classes:
- HIERARCHICAL_ONLY: 1100
- MIXED_ACTIVITY_HIERARCHICAL: 527
- MIXED_LOCAL_HIERARCHICAL: 521
- ACTIVITY_CONTEXT_STRONG: 59
- LOCAL_DIRECT_STRONG: 11

Evidence classes by Topic:

[مبانی و توانمندی سواد مالی]
- LOCAL_DIRECT_STRONG: 0
- ACTIVITY_CONTEXT_STRONG: 13
- MIXED_LOCAL_HIERARCHICAL: 106
- MIXED_ACTIVITY_HIERARCHICAL: 0
- HIERARCHICAL_ONLY: 22
- OTHER_STRONG: 0

[خرج، پس‌انداز و مدیریت پول]
- LOCAL_DIRECT_STRONG: 3
- ACTIVITY_CONTEXT_STRONG: 7
- MIXED_LOCAL_HIERARCHICAL: 121
- MIXED_ACTIVITY_HIERARCHICAL: 65
- HIERARCHICAL_ONLY: 163
- OTHER_STRONG: 0

[اعتبار، قرض، بدهی و بانک]
- LOCAL_DIRECT_STRONG: 0
- ACTIVITY_CONTEXT_STRONG: 0
- MIXED_LOCAL_HIERARCHICAL: 86
- MIXED_ACTIVITY_HIERARCHICAL: 50
- HIERARCHICAL_ONLY: 79
- OTHER_STRONG: 0

[کار، شغل و درآمد]
- LOCAL_DIRECT_STRONG: 8
- ACTIVITY_CONTEXT_STRONG: 36
- MIXED_LOCAL_HIERARCHICAL: 29
- MIXED_ACTIVITY_H

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Candidate Mapping v2 — تفکیک Document Topic از Book Hierarchical Topic Scope</h3>

<p>
Global Book Audit نشان داد که بخشی از Topic Evidence کتاب از Heading Path معتبر به ارث می‌رسد.
این Evidence نباید حذف شود، اما نباید با Evidence مستقیم خود Document نیز یکسان تلقی شود.
</p>

<p>
بنابراین Mapping v2 سه Dimension مستقل خواهد داشت:
</p>

<ul>
<li><strong>Document Topic Candidate:</strong> Topic با Evidence مستقیم، Activity Context یا Evidence محلی معتبر.</li>
<li><strong>Book Hierarchical Topic Scope:</strong> Topic حاصل از قرار گرفتن Document در Section موضوعی کتاب.</li>
<li><strong>Conversation Topic Scope:</strong> Context موضوعی Transcript که در مراحل قبل ساخته شد.</li>
</ul>

<h4>سیاست Book</h4>

<p>
Candidateهایی که در Audit سلول 24 با کلاس <code>HIERARCHICAL_ONLY</code> شناسایی شدند، از Document Topic Strong جدا شده و در <code>book_hierarchical_topic_scope</code> نگهداری می‌شوند.
</p>

<p>
این تغییر به معنی حذف Topic نیست؛ بلکه نوع Evidence را دقیق‌تر می‌کند.
</p>

<h4>دو نوع Coverage</h4>

<ul>
<li><strong>Direct/Local Document Coverage:</strong> فقط Evidence خود Document.</li>
<li><strong>Context-aware Coverage:</strong> Document Topic به‌اضافه Book Hierarchical Scope یا Conversation Scope.</li>
</ul>

<p>
برای Retrieval آینده هر دو View ارزشمند هستند.
برای تحلیل کیفیت دانش و Claim-level Audit باید بدانیم کدام Evidence مستقل است و کدام وابسته به Context.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 19,456 Document حفظ شوند.</li>
<li>تمام 1,100 Hierarchical-only Candidate به Scope منتقل شوند.</li>
<li>هیچ Topic Evidence حذف نشود.</li>
<li>Conversation Scope قبلی بدون تغییر حفظ شود.</li>
<li>Strong Document Topic فقط برای Evidence غیر-Hierarchical-only باقی بماند.</li>
<li>Multi-Strong ambiguity دوباره محاسبه شود.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [68]:
# Code Cell 25

from collections import Counter, defaultdict
import copy


# --------------------------------------------------
# 1. Start Mapping v2 from Candidate Mapping v1
# --------------------------------------------------

document_topic_mapping_v2_candidate = copy.deepcopy(
    document_topic_mapping_v1_candidate
)


for profile in (
    document_topic_mapping_v2_candidate.values()
):
    profile["mapping_revision"] = (
        "NB10_CANDIDATE_V2"
    )

    profile[
        "book_hierarchical_topic_scope"
    ] = []


# --------------------------------------------------
# 2. Exact set of Book HIERARCHICAL_ONLY rows
# --------------------------------------------------

hierarchical_only_candidate_keys = {
    (
        row["document_id"],
        row["topic_id"],
    )
    for row in hierarchical_only_rows
}


print("Candidate Mapping v2 refinement:")
print(
    "- Hierarchical-only Book Candidate rows:",
    len(
        hierarchical_only_candidate_keys
    ),
)


# --------------------------------------------------
# 3. Move Hierarchical-only Book evidence
#    from Document Topic to Book Scope
#
# No evidence is discarded.
# --------------------------------------------------

moved_hierarchical_rows = []


for record in book_records:

    document_id = record[
        "document_id"
    ]

    profile = (
        document_topic_mapping_v2_candidate[
            document_id
        ]
    )

    retained_candidates = []

    for candidate in profile[
        "candidates"
    ]:

        candidate_key = (
            document_id,
            candidate["topic_id"],
        )

        if (
            candidate_key
            not in hierarchical_only_candidate_keys
        ):
            retained_candidates.append(
                candidate
            )
            continue

        scope_entry = {
            "topic_id": candidate[
                "topic_id"
            ],
            "topic_label_fa": candidate[
                "topic_label_fa"
            ],
            "decision": (
                "BOOK_HIERARCHICAL_TOPIC_SCOPE_CANDIDATE"
            ),
            "confidence": candidate[
                "confidence"
            ],
            "reason": (
                "INHERITED_BOOK_HEADING_CONTEXT"
            ),
            "original_decision": (
                candidate[
                    "decision"
                ]
            ),
            "original_reason": copy.deepcopy(
                candidate[
                    "reason"
                ]
            ),
            "original_evidence": copy.deepcopy(
                candidate[
                    "evidence"
                ]
            ),
        }

        profile[
            "book_hierarchical_topic_scope"
        ].append(
            scope_entry
        )

        moved_hierarchical_rows.append(
            {
                "document_id": document_id,
                "topic_id": candidate[
                    "topic_id"
                ],
            }
        )

    profile[
        "candidates"
    ] = retained_candidates


# --------------------------------------------------
# 4. Validate exact movement
# --------------------------------------------------

moved_hierarchical_key_set = {
    (
        row["document_id"],
        row["topic_id"],
    )
    for row in moved_hierarchical_rows
}


missing_hierarchical_moves = (
    hierarchical_only_candidate_keys
    - moved_hierarchical_key_set
)

unexpected_hierarchical_moves = (
    moved_hierarchical_key_set
    - hierarchical_only_candidate_keys
)


print(
    "- Hierarchical rows moved to Book Scope:",
    len(
        moved_hierarchical_key_set
    ),
)


# --------------------------------------------------
# 5. Document Topic status v2
# --------------------------------------------------

v2_document_status_counter = Counter()

v2_source_status_counter = defaultdict(
    Counter
)


def v2_document_topic_status(
    profile,
):
    candidates = profile[
        "candidates"
    ]

    if not candidates:
        return "UNMAPPED"

    if any(
        candidate["decision"]
        == "STRONG_CANDIDATE"
        for candidate in candidates
    ):
        return "HAS_STRONG_CANDIDATE"

    return "LEXICAL_ONLY"


for profile in (
    document_topic_mapping_v2_candidate.values()
):

    status = (
        v2_document_topic_status(
            profile
        )
    )

    v2_document_status_counter[
        status
    ] += 1

    v2_source_status_counter[
        profile["source_type"]
    ][status] += 1


print("\nDocument Topic Mapping status v2:")

for status in (
    "HAS_STRONG_CANDIDATE",
    "LEXICAL_ONLY",
    "UNMAPPED",
):
    count = (
        v2_document_status_counter[
            status
        ]
    )

    print(
        f"- {status}: "
        f"{count}/{len(records)} "
        f"({count / len(records):.1%})"
    )


print(
    "\nDocument Topic Mapping status by Source v2:"
)

for source_type in (
    "website",
    "book",
    "transcript",
):

    population = len(
        source_records[
            source_type
        ]
    )

    print(
        f"\n[{source_type}]"
    )

    for status in (
        "HAS_STRONG_CANDIDATE",
        "LEXICAL_ONLY",
        "UNMAPPED",
    ):

        count = (
            v2_source_status_counter[
                source_type
            ][status]
        )

        print(
            f"- {status}: "
            f"{count}/{population} "
            f"({count / population:.1%})"
        )


# --------------------------------------------------
# 6. Scope populations
# --------------------------------------------------

book_scope_document_ids = {
    document_id
    for document_id, profile in (
        document_topic_mapping_v2_candidate.items()
    )
    if profile[
        "book_hierarchical_topic_scope"
    ]
}


conversation_scope_document_ids = {
    document_id
    for document_id, profile in (
        document_topic_mapping_v2_candidate.items()
    )
    if profile[
        "conversation_topic_scope"
    ] is not None
}


print("\nContext Scope populations:")

print(
    "- Book documents with Hierarchical Topic Scope:",
    len(
        book_scope_document_ids
    ),
)

print(
    "- Transcript documents with Conversation Topic Scope:",
    len(
        conversation_scope_document_ids
    ),
)


# --------------------------------------------------
# 7. Context-aware observability
#
# A document is context-observable when it has:
# - Document Topic Candidate, OR
# - Book Hierarchical Topic Scope, OR
# - Conversation Topic Scope
# --------------------------------------------------

context_observability_counter = defaultdict(
    Counter
)


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    source_type = profile[
        "source_type"
    ]

    has_document_topic = bool(
        profile["candidates"]
    )

    has_book_scope = bool(
        profile[
            "book_hierarchical_topic_scope"
        ]
    )

    has_conversation_scope = (
        profile[
            "conversation_topic_scope"
        ]
        is not None
    )

    if has_document_topic:
        context_observability_counter[
            source_type
        ][
            "HAS_DOCUMENT_TOPIC"
        ] += 1

    elif (
        has_book_scope
        or has_conversation_scope
    ):
        context_observability_counter[
            source_type
        ][
            "SCOPE_ONLY"
        ] += 1

    else:
        context_observability_counter[
            source_type
        ][
            "NO_TOPIC_OR_SCOPE"
        ] += 1


print(
    "\nContext-aware Topic observability:"
)


for source_type in (
    "website",
    "book",
    "transcript",
):

    population = len(
        source_records[
            source_type
        ]
    )

    print(
        f"\n[{source_type}]"
    )

    for status in (
        "HAS_DOCUMENT_TOPIC",
        "SCOPE_ONLY",
        "NO_TOPIC_OR_SCOPE",
    ):

        count = (
            context_observability_counter[
                source_type
            ][status]
        )

        print(
            f"- {status}: "
            f"{count}/{population} "
            f"({count / population:.1%})"
        )


# --------------------------------------------------
# 8. Context-aware Topic coverage by Topic
# --------------------------------------------------

v2_direct_topic_counter = defaultdict(
    Counter
)

v2_book_scope_topic_counter = Counter()

v2_conversation_scope_topic_counter = Counter()


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    source_type = profile[
        "source_type"
    ]

    for candidate in profile[
        "candidates"
    ]:

        v2_direct_topic_counter[
            candidate["topic_id"]
        ][
            source_type
        ] += 1

    for scope in profile[
        "book_hierarchical_topic_scope"
    ]:

        v2_book_scope_topic_counter[
            scope["topic_id"]
        ] += 1

    conversation_scope = profile[
        "conversation_topic_scope"
    ]

    if conversation_scope is not None:

        v2_conversation_scope_topic_counter[
            conversation_scope[
                "topic_id"
            ]
        ] += 1


print(
    "\nTopic observability v2:"
)


for topic_id in topic_seed_v0:

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    print(
        "- Direct/Local Website:",
        v2_direct_topic_counter[
            topic_id
        ]["website"],
    )

    print(
        "- Direct/Local Book:",
        v2_direct_topic_counter[
            topic_id
        ]["book"],
    )

    print(
        "- Book Hierarchical Scope:",
        v2_book_scope_topic_counter[
            topic_id
        ],
    )

    print(
        "- Direct/Local Transcript:",
        v2_direct_topic_counter[
            topic_id
        ]["transcript"],
    )

    print(
        "- Transcript Conversation Scope:",
        v2_conversation_scope_topic_counter[
            topic_id
        ],
    )


# --------------------------------------------------
# 9. Recalculate Strong Multi-topic ambiguity
# --------------------------------------------------

v2_strong_topic_ids_by_document = defaultdict(
    set
)


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    for candidate in profile[
        "candidates"
    ]:

        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        ):
            v2_strong_topic_ids_by_document[
                document_id
            ].add(
                candidate[
                    "topic_id"
                ]
            )


v2_multiple_strong_documents = {
    document_id: sorted(
        topic_ids
    )
    for document_id, topic_ids in (
        v2_strong_topic_ids_by_document.items()
    )
    if len(topic_ids) > 1
}


v2_multi_strong_by_source = Counter(
    document_topic_mapping_v2_candidate[
        document_id
    ]["source_type"]
    for document_id in (
        v2_multiple_strong_documents
    )
)


print(
    "\nMulti-Strong ambiguity after hierarchy separation:"
)

print(
    "- v1 Documents with >1 Strong Topic:",
    len(
        documents_with_multiple_strong_topics
    ),
)

print(
    "- v2 Documents with >1 Strong Topic:",
    len(
        v2_multiple_strong_documents
    ),
)

for source_type in (
    "website",
    "book",
    "transcript",
):
    print(
        f"- {source_type}:",
        v2_multi_strong_by_source[
            source_type
        ],
    )


# --------------------------------------------------
# 10. Evidence preservation validation
# --------------------------------------------------

preserved_topic_evidence_keys = set()


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    for candidate in profile[
        "candidates"
    ]:

        preserved_topic_evidence_keys.add(
            (
                document_id,
                candidate[
                    "topic_id"
                ],
            )
        )

    for scope in profile[
        "book_hierarchical_topic_scope"
    ]:

        preserved_topic_evidence_keys.add(
            (
                document_id,
                scope[
                    "topic_id"
                ],
            )
        )


v1_document_topic_key_set = set()


for document_id, profile in (
    document_topic_mapping_v1_candidate.items()
):

    for candidate in profile[
        "candidates"
    ]:

        v1_document_topic_key_set.add(
            (
                document_id,
                candidate[
                    "topic_id"
                ],
            )
        )


lost_topic_evidence_keys = (
    v1_document_topic_key_set
    - preserved_topic_evidence_keys
)


unexpected_topic_evidence_keys = (
    preserved_topic_evidence_keys
    - v1_document_topic_key_set
)


# --------------------------------------------------
# 11. Independent validation
# --------------------------------------------------

authority_mismatches_v2 = []


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    original_authority = (
        record_lookup[
            document_id
        ].get(
            "attribution",
            {},
        ).get(
            "content_authority_type"
        )
    )

    if (
        profile[
            "content_authority_type"
        ]
        != original_authority
    ):
        authority_mismatches_v2.append(
            document_id
        )


conversation_scope_mismatches = []


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    original_scope = (
        document_topic_mapping_v1_candidate[
            document_id
        ][
            "conversation_topic_scope"
        ]
    )

    if (
        profile[
            "conversation_topic_scope"
        ]
        != original_scope
    ):
        conversation_scope_mismatches.append(
            document_id
        )


disk_write_performed = False


print("\nValidation:")

print(
    "- Total documents:",
    len(
        document_topic_mapping_v2_candidate
    ),
)

print(
    "- Expected Hierarchical moves:",
    len(
        hierarchical_only_candidate_keys
    ),
)

print(
    "- Applied Hierarchical moves:",
    len(
        moved_hierarchical_key_set
    ),
)

print(
    "- Missing Hierarchical moves:",
    len(
        missing_hierarchical_moves
    ),
)

print(
    "- Unexpected Hierarchical moves:",
    len(
        unexpected_hierarchical_moves
    ),
)

print(
    "- Lost Topic evidence keys:",
    len(
        lost_topic_evidence_keys
    ),
)

print(
    "- Unexpected Topic evidence keys:",
    len(
        unexpected_topic_evidence_keys
    ),
)

print(
    "- Authority mismatches:",
    len(
        authority_mismatches_v2
    ),
)

print(
    "- Conversation Scope mismatches:",
    len(
        conversation_scope_mismatches
    ),
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    document_topic_mapping_v2_candidate
) == 19456

assert (
    moved_hierarchical_key_set
    == hierarchical_only_candidate_keys
)

assert not missing_hierarchical_moves
assert not unexpected_hierarchical_moves

assert not lost_topic_evidence_keys
assert not unexpected_topic_evidence_keys

assert not authority_mismatches_v2
assert not conversation_scope_mismatches

assert disk_write_performed is False


print("\nCODE CELL 25 GATE: PASS")
print(
    "Candidate Mapping v2 validated successfully: "
    "direct/local Document Topics, Book Hierarchical "
    "Topic Scope, and Transcript Conversation Topic "
    "Scope are now preserved as separate evidence dimensions."
)

Candidate Mapping v2 refinement:
- Hierarchical-only Book Candidate rows: 1100
- Hierarchical rows moved to Book Scope: 1100

Document Topic Mapping status v2:
- HAS_STRONG_CANDIDATE: 1459/19456 (7.5%)
- LEXICAL_ONLY: 3557/19456 (18.3%)
- UNMAPPED: 14440/19456 (74.2%)

Document Topic Mapping status by Source v2:

[website]
- HAS_STRONG_CANDIDATE: 343/529 (64.8%)
- LEXICAL_ONLY: 124/529 (23.4%)
- UNMAPPED: 62/529 (11.7%)

[book]
- HAS_STRONG_CANDIDATE: 1116/2701 (41.3%)
- LEXICAL_ONLY: 607/2701 (22.5%)
- UNMAPPED: 978/2701 (36.2%)

[transcript]
- HAS_STRONG_CANDIDATE: 0/16226 (0.0%)
- LEXICAL_ONLY: 2826/16226 (17.4%)
- UNMAPPED: 13400/16226 (82.6%)

Context Scope populations:
- Book documents with Hierarchical Topic Scope: 1090
- Transcript documents with Conversation Topic Scope: 1755

Context-aware Topic observability:

[website]
- HAS_DOCUMENT_TOPIC: 467/529 (88.3%)
- SCOPE_ONLY: 0/529 (0.0%)
- NO_TOPIC_OR_SCOPE: 62/529 (11.7%)

[book]
- HAS_DOCUMENT_TOPIC: 1723/2701 (63.8%)
- SCOPE_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Taxonomy Revision Candidate v1 — تجمیع شواهد پیش از Specialist Approval</h3>

<p>
Mapping v2 نشان داد Topic Evidence اکنون به سه Dimension مستقل تقسیم شده است:
</p>

<ul>
<li>Document Topic</li>
<li>Book Hierarchical Topic Scope</li>
<li>Transcript Conversation Topic Scope</li>
</ul>

<p>
پس از این اصلاح، فقط دو Book Document هنوز بیش از یک Strong Document Topic دارند و بیشتر Multi-topicهای باقی‌مانده متعلق به Website و مبتنی بر Tagهای مستقل هستند.
</p>

<p>
در این مرحله Draft بازنگری Taxonomy ساخته می‌شود.
این Draft نتیجه Evidenceهای Notebook تا این نقطه است، اما هنوز Specialist-approved یا Frozen نیست.
</p>

<h4>اصلاحات پیشنهادی</h4>

<ul>
<li>هفت Top-level Topic فعلی بدون تغییر حفظ می‌شوند.</li>
<li>«بخشش» و «بخشش و هدیه» به‌جای دو شاخه موازی، به‌عنوان یک Subtopic Candidate مشترک در مدیریت پول تجمیع می‌شوند.</li>
<li>سه مفهوم کسب‌وکار به Subtopicهای Candidate در کار و درآمد تبدیل می‌شوند.</li>
<li>اقتصاد رفتاری به‌عنوان Cross-cutting Lens نگهداری می‌شود.</li>
<li>انتخاب، قیمت و سود Context-dependent باقی می‌مانند.</li>
<li>مالیات به‌عنوان Taxonomy Extension Candidate نگهداری می‌شود و تا Expert Review به هیچ Parent Topic تحمیل نمی‌شود.</li>
</ul>

<h4>اصل مهم</h4>

<p>
این مرحله Taxonomy را Freeze نمی‌کند و هیچ Recommendation را جایگزین تصمیم متخصص نمی‌کند.
هدف فقط تبدیل Evidenceهای پراکنده به یک Draft منسجم و قابل Review است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>۷ Top-level Topic بدون حذف یا تغییر ID حفظ شوند.</li>
<li>تمام ۱۰ Review Concept در Revision Draft حساب شوند.</li>
<li>هیچ مفهوم Context-dependent به Global Alias تبدیل نشود.</li>
<li>مالیات بدون Parent اجباری باقی بماند.</li>
<li>دو Multi-Strong باقی‌مانده Book مستقل بررسی شوند.</li>
<li>Expert Approval همچنان Pending باشد.</li>
<li>Freeze و Disk Write انجام نشود.</li>
</ul>

</div>

In [69]:
# Code Cell 26

from collections import Counter
import copy


# --------------------------------------------------
# 1. Build coherent Taxonomy Revision Candidate
# --------------------------------------------------

taxonomy_revision_candidate_v1 = {
    "version": "v1-candidate",
    "status": "PENDING_SPECIALIST_APPROVAL",

    "top_level_topics": copy.deepcopy(
        topic_seed_v0
    ),

    "subtopic_candidates": {
        "spending_saving_money_management": [
            {
                "subtopic_id": "giving_and_gifting",
                "label_fa": "بخشش و هدیه",
                "source_review_concepts": [
                    "بخشش",
                    "بخشش و  هدیه",
                ],
                "evidence_status": (
                    "EVIDENCE_SUPPORTED_CANDIDATE"
                ),
                "specialist_status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            },
        ],

        "work_income": [
            {
                "subtopic_id": (
                    "entrepreneurship_production_resources"
                ),
                "label_fa": "منابع تولید",
                "source_review_concepts": [
                    "منابع تولید",
                ],
                "evidence_status": (
                    "EVIDENCE_SUPPORTED_CANDIDATE"
                ),
                "specialist_status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            },

            {
                "subtopic_id": (
                    "goods_and_services_basics"
                ),
                "label_fa": "تفاوت کالا و خدمت",
                "source_review_concepts": [
                    "تفاوت کالا و خدمت",
                ],
                "evidence_status": (
                    "EVIDENCE_SUPPORTED_CANDIDATE"
                ),
                "specialist_status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            },

            {
                "subtopic_id": (
                    "product_presentation_and_promotion"
                ),
                "label_fa": "معرفی محصول و تبلیغ",
                "source_review_concepts": [
                    "معرفی محصول و تبلیغ",
                ],
                "evidence_status": (
                    "EVIDENCE_SUPPORTED_CANDIDATE"
                ),
                "specialist_status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            },
        ],
    },

    "cross_cutting_candidates": [
        {
            "dimension_id": (
                "behavioral_economics_lens"
            ),
            "label_fa": "اقتصاد رفتاری",
            "source_review_concepts": [
                "اقتصاد رفتاری",
            ],
            "evidence_status": (
                "EVIDENCE_SUPPORTED_CANDIDATE"
            ),
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        },
    ],

    "context_dependent_terms": [
        {
            "term": "انتخاب",
            "global_topic_alias_allowed": False,
            "reason": "GENERIC_CROSS_TOPIC_TERM",
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        },
        {
            "term": "قیمت",
            "global_topic_alias_allowed": False,
            "reason": "MULTI_CONTEXT_TERM",
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        },
        {
            "term": "سود",
            "global_topic_alias_allowed": False,
            "reason": "POLYSEMOUS_FINANCIAL_TERM",
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        },
    ],

    "taxonomy_extension_candidates": [
        {
            "extension_id": (
                "taxation_and_financial_obligations"
            ),
            "label_fa": "مالیات و تعهدات مالی",
            "source_review_concepts": [
                "مالیات",
            ],
            "proposed_parent_topic_id": None,
            "reason": (
                "Standalone financial-literacy evidence "
                "exists, but no existing Top-level Topic "
                "provides an unambiguous Parent."
            ),
            "evidence_status": (
                "EVIDENCE_SUPPORTED_EXTENSION_CANDIDATE"
            ),
            "specialist_status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        },
    ],
}


# --------------------------------------------------
# 2. Validate Top-level Topic preservation
# --------------------------------------------------

original_top_level_topic_ids = set(
    topic_seed_v0
)

revision_top_level_topic_ids = set(
    taxonomy_revision_candidate_v1[
        "top_level_topics"
    ]
)


removed_top_level_topics = (
    original_top_level_topic_ids
    - revision_top_level_topic_ids
)

unexpected_top_level_topics = (
    revision_top_level_topic_ids
    - original_top_level_topic_ids
)


print("Taxonomy Revision Candidate v1:")
print(
    "- Status:",
    taxonomy_revision_candidate_v1[
        "status"
    ],
)

print(
    "- Top-level Topics:",
    len(
        revision_top_level_topic_ids
    ),
)

print(
    "- Removed Top-level Topics:",
    len(
        removed_top_level_topics
    ),
)

print(
    "- Unexpected Top-level Topics:",
    len(
        unexpected_top_level_topics
    ),
)


# --------------------------------------------------
# 3. Print proposed Subtopic structure
# --------------------------------------------------

print("\nSubtopic Candidates:")


subtopic_ids = []
review_concepts_accounted_for = set()


for topic_id, subtopics in (
    taxonomy_revision_candidate_v1[
        "subtopic_candidates"
    ].items()
):

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    for subtopic in subtopics:

        subtopic_ids.append(
            subtopic[
                "subtopic_id"
            ]
        )

        review_concepts_accounted_for.update(
            subtopic[
                "source_review_concepts"
            ]
        )

        print(
            "-",
            subtopic[
                "label_fa"
            ],
            "|",
            subtopic[
                "subtopic_id"
            ],
            "|",
            subtopic[
                "specialist_status"
            ],
        )


# --------------------------------------------------
# 4. Cross-cutting dimensions
# --------------------------------------------------

print("\nCross-cutting Candidates:")


for dimension in (
    taxonomy_revision_candidate_v1[
        "cross_cutting_candidates"
    ]
):

    review_concepts_accounted_for.update(
        dimension[
            "source_review_concepts"
        ]
    )

    print(
        "-",
        dimension[
            "label_fa"
        ],
        "|",
        dimension[
            "dimension_id"
        ],
        "|",
        dimension[
            "specialist_status"
        ],
    )


# --------------------------------------------------
# 5. Context-dependent terms
# --------------------------------------------------

print("\nContext-dependent Terms:")


for row in (
    taxonomy_revision_candidate_v1[
        "context_dependent_terms"
    ]
):

    review_concepts_accounted_for.add(
        row["term"]
    )

    print(
        "-",
        row["term"],
        "| global_alias_allowed=",
        row[
            "global_topic_alias_allowed"
        ],
        "|",
        row[
            "specialist_status"
        ],
    )


# --------------------------------------------------
# 6. Taxonomy extension candidates
# --------------------------------------------------

print("\nTaxonomy Extension Candidates:")


for extension in (
    taxonomy_revision_candidate_v1[
        "taxonomy_extension_candidates"
    ]
):

    review_concepts_accounted_for.update(
        extension[
            "source_review_concepts"
        ]
    )

    print(
        "-",
        extension[
            "label_fa"
        ],
        "| parent=",
        extension[
            "proposed_parent_topic_id"
        ],
        "|",
        extension[
            "specialist_status"
        ],
    )


# --------------------------------------------------
# 7. Ensure all 10 Review Concepts are represented
# --------------------------------------------------

expected_review_concepts = set(
    review_concepts
)


missing_revision_review_concepts = (
    expected_review_concepts
    - review_concepts_accounted_for
)

unexpected_revision_review_concepts = (
    review_concepts_accounted_for
    - expected_review_concepts
)


print("\nReview Concept accounting:")

print(
    "- Expected Review Concepts:",
    len(
        expected_review_concepts
    ),
)

print(
    "- Accounted-for Review Concepts:",
    len(
        review_concepts_accounted_for
    ),
)

print(
    "- Missing:",
    len(
        missing_revision_review_concepts
    ),
)

print(
    "- Unexpected:",
    len(
        unexpected_revision_review_concepts
    ),
)


# --------------------------------------------------
# 8. Inspect remaining Book Multi-Strong documents
#    after Mapping v2
# --------------------------------------------------

remaining_book_multi_strong_ids = [
    document_id
    for document_id in (
        v2_multiple_strong_documents
    )
    if (
        document_topic_mapping_v2_candidate[
            document_id
        ]["source_type"]
        == "book"
    )
]


print(
    "\nRemaining Book Multi-Strong Review:"
)

print(
    "- Documents:",
    len(
        remaining_book_multi_strong_ids
    ),
)


remaining_book_multi_strong_review_v0 = []


for document_id in (
    remaining_book_multi_strong_ids
):

    record = record_lookup[
        document_id
    ]

    profile = (
        document_topic_mapping_v2_candidate[
            document_id
        ]
    )

    strong_candidates = [
        candidate
        for candidate in profile[
            "candidates"
        ]
        if (
            candidate["decision"]
            == "STRONG_CANDIDATE"
        )
    ]

    candidate_reviews = []

    all_have_non_hierarchical_evidence = True

    for candidate in strong_candidates:

        evidence_types = {
            evidence["type"]
            for evidence in candidate[
                "evidence"
            ]
        }

        non_hierarchical_evidence = (
            evidence_types
            - hierarchical_book_evidence_types
        )

        if not non_hierarchical_evidence:
            all_have_non_hierarchical_evidence = False

        candidate_reviews.append(
            {
                "topic_id": candidate[
                    "topic_id"
                ],
                "topic_label_fa": candidate[
                    "topic_label_fa"
                ],
                "confidence": candidate[
                    "confidence"
                ],
                "evidence_types": sorted(
                    evidence_types
                ),
                "non_hierarchical_evidence": sorted(
                    non_hierarchical_evidence
                ),
            }
        )

    if all_have_non_hierarchical_evidence:
        technical_recommendation = (
            "LIKELY_VALID_MULTI_TOPIC"
        )
    else:
        technical_recommendation = (
            "SPECIALIST_REVIEW_REQUIRED"
        )

    heading_path = (
        record.get(
            "locator",
            {},
        ).get(
            "heading_path"
        )
        or record.get(
            "source_metadata",
            {},
        ).get(
            "heading_path"
        )
    )

    review_row = {
        "document_id": document_id,
        "heading_path": heading_path,
        "strong_candidates": (
            candidate_reviews
        ),
        "technical_recommendation": (
            technical_recommendation
        ),
        "specialist_status": (
            "PENDING_SPECIALIST_APPROVAL"
        ),
    }

    remaining_book_multi_strong_review_v0.append(
        review_row
    )

    print(
        "\n----------------------------------------"
    )

    print(
        "Document:",
        document_id,
    )

    print(
        "Heading path:",
        heading_path,
    )

    for candidate in candidate_reviews:

        print(
            "-",
            candidate[
                "topic_label_fa"
            ],
            "| confidence=",
            candidate[
                "confidence"
            ],
            "| evidence=",
            candidate[
                "evidence_types"
            ],
            "| non-hierarchical=",
            candidate[
                "non_hierarchical_evidence"
            ],
        )

    print(
        "Technical recommendation:",
        technical_recommendation,
    )

    print(
        "Specialist status:",
        review_row[
            "specialist_status"
        ],
    )

    print(
        "Excerpt:"
    )

    print(
        compact_excerpt(
            record,
            limit=700,
        )
        or "<EMPTY>"
    )


# --------------------------------------------------
# 9. Specialist Approval matrix
# --------------------------------------------------

specialist_approval_items_v1 = []


for topic_id, subtopics in (
    taxonomy_revision_candidate_v1[
        "subtopic_candidates"
    ].items()
):

    for subtopic in subtopics:

        specialist_approval_items_v1.append(
            {
                "item_type": "SUBTOPIC",
                "item_id": subtopic[
                    "subtopic_id"
                ],
                "parent_topic_id": topic_id,
                "status": (
                    "PENDING_SPECIALIST_APPROVAL"
                ),
            }
        )


for dimension in (
    taxonomy_revision_candidate_v1[
        "cross_cutting_candidates"
    ]
):

    specialist_approval_items_v1.append(
        {
            "item_type": (
                "CROSS_CUTTING_DIMENSION"
            ),
            "item_id": dimension[
                "dimension_id"
            ],
            "parent_topic_id": None,
            "status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        }
    )


for extension in (
    taxonomy_revision_candidate_v1[
        "taxonomy_extension_candidates"
    ]
):

    specialist_approval_items_v1.append(
        {
            "item_type": (
                "TAXONOMY_EXTENSION"
            ),
            "item_id": extension[
                "extension_id"
            ],
            "parent_topic_id": extension[
                "proposed_parent_topic_id"
            ],
            "status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        }
    )


for row in (
    taxonomy_revision_candidate_v1[
        "context_dependent_terms"
    ]
):

    specialist_approval_items_v1.append(
        {
            "item_type": (
                "CONTEXT_DEPENDENT_POLICY"
            ),
            "item_id": row["term"],
            "parent_topic_id": None,
            "status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        }
    )


for row in (
    remaining_book_multi_strong_review_v0
):

    specialist_approval_items_v1.append(
        {
            "item_type": (
                "BOOK_MULTI_TOPIC_DOCUMENT"
            ),
            "item_id": row[
                "document_id"
            ],
            "parent_topic_id": None,
            "status": (
                "PENDING_SPECIALIST_APPROVAL"
            ),
        }
    )


approval_status_counter = Counter(
    row["status"]
    for row in (
        specialist_approval_items_v1
    )
)


print("\nSpecialist Approval Matrix:")

print(
    "- Total items:",
    len(
        specialist_approval_items_v1
    ),
)

for status, count in (
    approval_status_counter.items()
):
    print(
        f"- {status}: {count}"
    )


# --------------------------------------------------
# 10. Freeze readiness remains blocked
# --------------------------------------------------

all_specialist_items_approved = all(
    row["status"] == "APPROVED"
    for row in (
        specialist_approval_items_v1
    )
)


if all_specialist_items_approved:
    taxonomy_revision_freeze_readiness = (
        "READY_FOR_FREEZE"
    )
else:
    taxonomy_revision_freeze_readiness = (
        "BLOCKED_PENDING_SPECIALIST_APPROVAL"
    )


print("\nFreeze readiness:")

print(
    "- All Specialist items approved:",
    all_specialist_items_approved,
)

print(
    "- Status:",
    taxonomy_revision_freeze_readiness,
)

print(
    "- Freeze executed:",
    False,
)


# --------------------------------------------------
# 11. Validation
# --------------------------------------------------

duplicate_subtopic_ids = [
    subtopic_id
    for subtopic_id, count in (
        Counter(
            subtopic_ids
        ).items()
    )
    if count > 1
]


context_terms_with_global_alias = [
    row["term"]
    for row in (
        taxonomy_revision_candidate_v1[
            "context_dependent_terms"
        ]
    )
    if row[
        "global_topic_alias_allowed"
    ]
]


extension_with_forced_parent = [
    extension[
        "extension_id"
    ]
    for extension in (
        taxonomy_revision_candidate_v1[
            "taxonomy_extension_candidates"
        ]
    )
    if (
        extension[
            "extension_id"
        ]
        == "taxation_and_financial_obligations"
        and extension[
            "proposed_parent_topic_id"
        ]
        is not None
    )
]


auto_approved_specialist_items = [
    row
    for row in (
        specialist_approval_items_v1
    )
    if row["status"] != (
        "PENDING_SPECIALIST_APPROVAL"
    )
]


taxonomy_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Duplicate Subtopic IDs:",
    len(
        duplicate_subtopic_ids
    ),
)

print(
    "- Context-dependent terms incorrectly "
    "allowed as Global Alias:",
    len(
        context_terms_with_global_alias
    ),
)

print(
    "- Taxation Extension with forced Parent:",
    len(
        extension_with_forced_parent
    ),
)

print(
    "- Auto-approved Specialist items:",
    len(
        auto_approved_specialist_items
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert (
    revision_top_level_topic_ids
    == original_top_level_topic_ids
)

assert not removed_top_level_topics
assert not unexpected_top_level_topics

assert not (
    missing_revision_review_concepts
)

assert not (
    unexpected_revision_review_concepts
)

assert not duplicate_subtopic_ids

assert not (
    context_terms_with_global_alias
)

assert not (
    extension_with_forced_parent
)

assert len(
    remaining_book_multi_strong_ids
) == 2

assert not (
    auto_approved_specialist_items
)

assert (
    taxonomy_revision_freeze_readiness
    == "BLOCKED_PENDING_SPECIALIST_APPROVAL"
)

assert (
    taxonomy_mutation_performed
    is False
)

assert (
    disk_write_performed
    is False
)


print("\nCODE CELL 26 GATE: PASS")
print(
    "Taxonomy Revision Candidate v1 assembled "
    "successfully; all proposed revisions remain "
    "explicitly pending Specialist approval."
)

Taxonomy Revision Candidate v1:
- Status: PENDING_SPECIALIST_APPROVAL
- Top-level Topics: 7
- Removed Top-level Topics: 0
- Unexpected Top-level Topics: 0

Subtopic Candidates:

[خرج، پس‌انداز و مدیریت پول]
- بخشش و هدیه | giving_and_gifting | PENDING_SPECIALIST_APPROVAL

[کار، شغل و درآمد]
- منابع تولید | entrepreneurship_production_resources | PENDING_SPECIALIST_APPROVAL
- تفاوت کالا و خدمت | goods_and_services_basics | PENDING_SPECIALIST_APPROVAL
- معرفی محصول و تبلیغ | product_presentation_and_promotion | PENDING_SPECIALIST_APPROVAL

Cross-cutting Candidates:
- اقتصاد رفتاری | behavioral_economics_lens | PENDING_SPECIALIST_APPROVAL

Context-dependent Terms:
- انتخاب | global_alias_allowed= False | PENDING_SPECIALIST_APPROVAL
- قیمت | global_alias_allowed= False | PENDING_SPECIALIST_APPROVAL
- سود | global_alias_allowed= False | PENDING_SPECIALIST_APPROVAL

Taxonomy Extension Candidates:
- مالیات و تعهدات مالی | parent= None | PENDING_SPECIALIST_APPROVAL

Review Concept accounting:


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Specialist Review Sheet — تصمیم‌های لازم پیش از Freeze Taxonomy v1</h3>

<p>
تمام تحلیل‌های فنی و Evidence-based موردنیاز برای Candidate Taxonomy انجام شده است.
اکنون فقط تصمیم‌هایی باقی مانده‌اند که باید به‌صورت صریح توسط متخصص تأیید، اصلاح، رد یا به Review بیشتر موکول شوند.
</p>

<p>
این مرحله هیچ تصمیمی را خودکار Approved نمی‌کند.
هدف آن تبدیل تمام موارد باز به یک Review Sheet استاندارد و قابل ثبت است.
</p>

<h4>گزینه‌های تصمیم متخصص</h4>

<ul>
<li><strong>APPROVE_AS_PROPOSED:</strong> پیشنهاد فعلی تأیید می‌شود.</li>
<li><strong>APPROVE_WITH_REVISION:</strong> اصل پیشنهاد پذیرفته می‌شود ولی Parent، Label، ID یا Policy نیاز به اصلاح دارد.</li>
<li><strong>REJECT:</strong> پیشنهاد فعلی رد می‌شود.</li>
<li><strong>DEFER_FOR_MORE_EVIDENCE:</strong> Evidence فعلی برای تصمیم کافی نیست.</li>
</ul>

<h4>۱۱ تصمیم باقی‌مانده</h4>

<ul>
<li>۴ Subtopic Candidate</li>
<li>۱ Cross-cutting Dimension</li>
<li>۳ Context-dependent Term Policy</li>
<li>۱ Taxonomy Extension Candidate</li>
<li>۲ Book Multi-topic Document</li>
</ul>

<p>
برای هر تصمیم باید Recommendation فنی، Evidence Summary و تصمیم نهایی متخصص جداگانه نگهداری شود.
</p>

<p>
تا زمانی که تمام تصمیم‌های اجباری Resolve نشده باشند، Taxonomy v1 نباید Freeze شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۱ مورد Review در Sheet حضور داشته باشد.</li>
<li>Recommendation فنی از Expert Decision جدا باشد.</li>
<li>Allowed Decisions برای هر Item مشخص باشد.</li>
<li>هیچ Item خودکار Approved نشود.</li>
<li>Freeze همچنان Blocked باقی بماند.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [70]:
# Code Cell 27

from collections import Counter


allowed_specialist_decisions = {
    "APPROVE_AS_PROPOSED",
    "APPROVE_WITH_REVISION",
    "REJECT",
    "DEFER_FOR_MORE_EVIDENCE",
}


# --------------------------------------------------
# 1. Build formal Specialist Review Sheet
# --------------------------------------------------

specialist_review_sheet_v1 = []


# --------------------------------------------------
# 1A. Subtopic candidates
# --------------------------------------------------

for parent_topic_id, subtopics in (
    taxonomy_revision_candidate_v1[
        "subtopic_candidates"
    ].items()
):

    for subtopic in subtopics:

        review_concepts_for_item = (
            subtopic[
                "source_review_concepts"
            ]
        )

        evidence_document_count = sum(
            review_concept_profiles_v0[
                concept
            ]["document_count"]
            for concept in (
                review_concepts_for_item
            )
        )

        source_counter = Counter()

        for concept in (
            review_concepts_for_item
        ):
            source_counter.update(
                review_concept_profiles_v0[
                    concept
                ]["source_counts"]
            )

        specialist_review_sheet_v1.append(
            {
                "item_type": "SUBTOPIC",
                "item_id": subtopic[
                    "subtopic_id"
                ],
                "label_fa": subtopic[
                    "label_fa"
                ],
                "technical_recommendation": (
                    "APPROVE_SUBTOPIC_CANDIDATE"
                ),
                "proposed_parent_topic_id": (
                    parent_topic_id
                ),
                "proposed_parent_label_fa": (
                    topic_seed_v0[
                        parent_topic_id
                    ]["label_fa"]
                ),
                "source_review_concepts": (
                    review_concepts_for_item
                ),
                "evidence_document_count": (
                    evidence_document_count
                ),
                "evidence_source_counts": dict(
                    source_counter
                ),
                "allowed_decisions": sorted(
                    allowed_specialist_decisions
                ),
                "specialist_decision": None,
                "specialist_revision": None,
                "specialist_note": None,
                "status": "PENDING",
            }
        )


# --------------------------------------------------
# 1B. Cross-cutting candidates
# --------------------------------------------------

for dimension in (
    taxonomy_revision_candidate_v1[
        "cross_cutting_candidates"
    ]
):

    concept = dimension[
        "source_review_concepts"
    ][0]

    concept_profile = (
        review_concept_profiles_v0[
            concept
        ]
    )

    specialist_review_sheet_v1.append(
        {
            "item_type": (
                "CROSS_CUTTING_DIMENSION"
            ),
            "item_id": dimension[
                "dimension_id"
            ],
            "label_fa": dimension[
                "label_fa"
            ],
            "technical_recommendation": (
                "APPROVE_CROSS_CUTTING_DIMENSION"
            ),
            "proposed_parent_topic_id": None,
            "source_review_concepts": (
                dimension[
                    "source_review_concepts"
                ]
            ),
            "evidence_document_count": (
                concept_profile[
                    "document_count"
                ]
            ),
            "evidence_source_counts": (
                concept_profile[
                    "source_counts"
                ]
            ),
            "strong_topic_cooccurrence": (
                concept_profile[
                    "strong_topic_cooccurrence"
                ]
            ),
            "allowed_decisions": sorted(
                allowed_specialist_decisions
            ),
            "specialist_decision": None,
            "specialist_revision": None,
            "specialist_note": None,
            "status": "PENDING",
        }
    )


# --------------------------------------------------
# 1C. Context-dependent policy items
# --------------------------------------------------

for row in (
    taxonomy_revision_candidate_v1[
        "context_dependent_terms"
    ]
):

    concept_profile = (
        review_concept_profiles_v0[
            row["term"]
        ]
    )

    specialist_review_sheet_v1.append(
        {
            "item_type": (
                "CONTEXT_DEPENDENT_POLICY"
            ),
            "item_id": row["term"],
            "label_fa": row["term"],
            "technical_recommendation": (
                "KEEP_CONTEXT_DEPENDENT"
            ),
            "global_topic_alias_allowed": (
                False
            ),
            "reason": row[
                "reason"
            ],
            "evidence_document_count": (
                concept_profile[
                    "document_count"
                ]
            ),
            "evidence_source_counts": (
                concept_profile[
                    "source_counts"
                ]
            ),
            "strong_topic_cooccurrence": (
                concept_profile[
                    "strong_topic_cooccurrence"
                ]
            ),
            "allowed_decisions": sorted(
                allowed_specialist_decisions
            ),
            "specialist_decision": None,
            "specialist_revision": None,
            "specialist_note": None,
            "status": "PENDING",
        }
    )


# --------------------------------------------------
# 1D. Taxonomy extension candidates
# --------------------------------------------------

for extension in (
    taxonomy_revision_candidate_v1[
        "taxonomy_extension_candidates"
    ]
):

    concept = extension[
        "source_review_concepts"
    ][0]

    concept_profile = (
        review_concept_profiles_v0[
            concept
        ]
    )

    specialist_review_sheet_v1.append(
        {
            "item_type": (
                "TAXONOMY_EXTENSION"
            ),
            "item_id": extension[
                "extension_id"
            ],
            "label_fa": extension[
                "label_fa"
            ],
            "technical_recommendation": (
                "REVIEW_EXTENSION_WITHOUT_FORCED_PARENT"
            ),
            "proposed_parent_topic_id": (
                extension[
                    "proposed_parent_topic_id"
                ]
            ),
            "source_review_concepts": (
                extension[
                    "source_review_concepts"
                ]
            ),
            "evidence_document_count": (
                concept_profile[
                    "document_count"
                ]
            ),
            "evidence_source_counts": (
                concept_profile[
                    "source_counts"
                ]
            ),
            "strong_topic_cooccurrence": (
                concept_profile[
                    "strong_topic_cooccurrence"
                ]
            ),
            "reason": extension[
                "reason"
            ],
            "allowed_decisions": sorted(
                allowed_specialist_decisions
            ),
            "specialist_decision": None,
            "specialist_revision": None,
            "specialist_note": None,
            "status": "PENDING",
        }
    )


# --------------------------------------------------
# 1E. Remaining Book Multi-topic documents
# --------------------------------------------------

for row in (
    remaining_book_multi_strong_review_v0
):

    document_id = row[
        "document_id"
    ]

    record = record_lookup[
        document_id
    ]

    specialist_review_sheet_v1.append(
        {
            "item_type": (
                "BOOK_MULTI_TOPIC_DOCUMENT"
            ),
            "item_id": document_id,
            "label_fa": record.get(
                "title"
            ),
            "technical_recommendation": (
                row[
                    "technical_recommendation"
                ]
            ),
            "heading_path": (
                row[
                    "heading_path"
                ]
            ),
            "strong_candidates": (
                row[
                    "strong_candidates"
                ]
            ),
            "excerpt": compact_excerpt(
                record,
                limit=500,
            ),
            "allowed_decisions": sorted(
                allowed_specialist_decisions
            ),
            "specialist_decision": None,
            "specialist_revision": None,
            "specialist_note": None,
            "status": "PENDING",
        }
    )


# --------------------------------------------------
# 2. Review Sheet summary
# --------------------------------------------------

review_item_type_counter = Counter(
    row["item_type"]
    for row in (
        specialist_review_sheet_v1
    )
)


print("Specialist Review Sheet v1:")
print(
    "- Total Review items:",
    len(
        specialist_review_sheet_v1
    ),
)


for item_type, count in (
    review_item_type_counter.items()
):
    print(
        f"- {item_type}: {count}"
    )


# --------------------------------------------------
# 3. Print concise human-facing Review Sheet
# --------------------------------------------------

print("\nDecision Sheet:")


for index, item in enumerate(
    specialist_review_sheet_v1,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        f"REVIEW ITEM {index:02d}"
    )

    print(
        "Type:",
        item[
            "item_type"
        ],
    )

    print(
        "ID:",
        item[
            "item_id"
        ],
    )

    print(
        "Label:",
        item[
            "label_fa"
        ],
    )

    print(
        "Technical recommendation:",
        item[
            "technical_recommendation"
        ],
    )

    if (
        item[
            "item_type"
        ]
        == "SUBTOPIC"
    ):
        print(
            "Proposed Parent:",
            item[
                "proposed_parent_label_fa"
            ],
        )

        print(
            "Evidence documents:",
            item[
                "evidence_document_count"
            ],
        )

        print(
            "Sources:",
            item[
                "evidence_source_counts"
            ],
        )

    elif (
        item[
            "item_type"
        ]
        == "CROSS_CUTTING_DIMENSION"
    ):
        print(
            "Evidence documents:",
            item[
                "evidence_document_count"
            ],
        )

        print(
            "Sources:",
            item[
                "evidence_source_counts"
            ],
        )

        print(
            "Strong Topic co-occurrence:",
            item[
                "strong_topic_cooccurrence"
            ],
        )

    elif (
        item[
            "item_type"
        ]
        == "CONTEXT_DEPENDENT_POLICY"
    ):
        print(
            "Global Topic Alias allowed:",
            item[
                "global_topic_alias_allowed"
            ],
        )

        print(
            "Evidence documents:",
            item[
                "evidence_document_count"
            ],
        )

        print(
            "Strong Topic co-occurrence:",
            item[
                "strong_topic_cooccurrence"
            ],
        )

    elif (
        item[
            "item_type"
        ]
        == "TAXONOMY_EXTENSION"
    ):
        print(
            "Proposed Parent:",
            item[
                "proposed_parent_topic_id"
            ],
        )

        print(
            "Evidence documents:",
            item[
                "evidence_document_count"
            ],
        )

        print(
            "Sources:",
            item[
                "evidence_source_counts"
            ],
        )

        print(
            "Strong Topic co-occurrence:",
            item[
                "strong_topic_cooccurrence"
            ],
        )

    elif (
        item[
            "item_type"
        ]
        == "BOOK_MULTI_TOPIC_DOCUMENT"
    ):
        print(
            "Heading path:",
            item[
                "heading_path"
            ],
        )

        print(
            "Strong Topic candidates:"
        )

        for candidate in item[
            "strong_candidates"
        ]:
            print(
                "-",
                candidate[
                    "topic_label_fa"
                ],
                "| confidence=",
                candidate[
                    "confidence"
                ],
                "| non-hierarchical evidence=",
                candidate[
                    "non_hierarchical_evidence"
                ],
            )

        print(
            "Excerpt:",
            item[
                "excerpt"
            ],
        )

    print(
        "Allowed decisions:",
        item[
            "allowed_decisions"
        ],
    )

    print(
        "Specialist decision:",
        item[
            "specialist_decision"
        ],
    )

    print(
        "Status:",
        item[
            "status"
        ],
    )


# --------------------------------------------------
# 4. Prepare explicit decision-entry template
#
# This is intentionally empty.
# A specialist decision must be supplied explicitly.
# --------------------------------------------------

specialist_decision_entry_template_v1 = {
    item[
        "item_id"
    ]: {
        "decision": None,
        "revision": None,
        "note": None,
    }
    for item in (
        specialist_review_sheet_v1
    )
}


print(
    "\nSpecialist Decision Entry Template:"
)

for item_id, entry in (
    specialist_decision_entry_template_v1.items()
):
    print(
        f"- {item_id}: "
        f"decision={entry['decision']} "
        f"| revision={entry['revision']} "
        f"| note={entry['note']}"
    )


# --------------------------------------------------
# 5. Freeze readiness
# --------------------------------------------------

pending_review_items = [
    row
    for row in (
        specialist_review_sheet_v1
    )
    if row[
        "specialist_decision"
    ] is None
]


resolved_review_items = [
    row
    for row in (
        specialist_review_sheet_v1
    )
    if row[
        "specialist_decision"
    ] is not None
]


if pending_review_items:
    specialist_review_freeze_status_v1 = (
        "BLOCKED_PENDING_SPECIALIST_DECISIONS"
    )
else:
    specialist_review_freeze_status_v1 = (
        "READY_FOR_DECISION_VALIDATION"
    )


print("\nReview completion status:")

print(
    "- Resolved items:",
    len(
        resolved_review_items
    ),
)

print(
    "- Pending items:",
    len(
        pending_review_items
    ),
)

print(
    "- Freeze status:",
    specialist_review_freeze_status_v1,
)

print(
    "- Freeze executed:",
    False,
)


# --------------------------------------------------
# 6. Independent validation
# --------------------------------------------------

duplicate_review_item_ids = [
    item_id
    for item_id, count in (
        Counter(
            row["item_id"]
            for row in (
                specialist_review_sheet_v1
            )
        ).items()
    )
    if count > 1
]


invalid_allowed_decision_sets = [
    row["item_id"]
    for row in (
        specialist_review_sheet_v1
    )
    if set(
        row[
            "allowed_decisions"
        ]
    ) != allowed_specialist_decisions
]


auto_decided_items = [
    row["item_id"]
    for row in (
        specialist_review_sheet_v1
    )
    if row[
        "specialist_decision"
    ] is not None
]


disk_write_performed = False
freeze_executed = False


print("\nValidation:")

print(
    "- Review items expected:",
    11,
)

print(
    "- Review items generated:",
    len(
        specialist_review_sheet_v1
    ),
)

print(
    "- Duplicate Review Item IDs:",
    len(
        duplicate_review_item_ids
    ),
)

print(
    "- Invalid Allowed Decision sets:",
    len(
        invalid_allowed_decision_sets
    ),
)

print(
    "- Auto-decided items:",
    len(
        auto_decided_items
    ),
)

print(
    "- Disk write performed:",
    disk_write_performed,
)

print(
    "- Freeze executed:",
    freeze_executed,
)


assert len(
    specialist_review_sheet_v1
) == 11

assert (
    review_item_type_counter[
        "SUBTOPIC"
    ]
    == 4
)

assert (
    review_item_type_counter[
        "CROSS_CUTTING_DIMENSION"
    ]
    == 1
)

assert (
    review_item_type_counter[
        "CONTEXT_DEPENDENT_POLICY"
    ]
    == 3
)

assert (
    review_item_type_counter[
        "TAXONOMY_EXTENSION"
    ]
    == 1
)

assert (
    review_item_type_counter[
        "BOOK_MULTI_TOPIC_DOCUMENT"
    ]
    == 2
)

assert not duplicate_review_item_ids
assert not invalid_allowed_decision_sets
assert not auto_decided_items

assert len(
    pending_review_items
) == 11

assert (
    specialist_review_freeze_status_v1
    == "BLOCKED_PENDING_SPECIALIST_DECISIONS"
)

assert disk_write_performed is False
assert freeze_executed is False


print("\nCODE CELL 27 GATE: PASS")
print(
    "Formal Specialist Review Sheet generated "
    "successfully; all 11 required decisions remain "
    "explicitly pending and Taxonomy freeze is blocked."
)

Specialist Review Sheet v1:
- Total Review items: 11
- SUBTOPIC: 4
- CROSS_CUTTING_DIMENSION: 1
- CONTEXT_DEPENDENT_POLICY: 3
- TAXONOMY_EXTENSION: 1
- BOOK_MULTI_TOPIC_DOCUMENT: 2

Decision Sheet:

REVIEW ITEM 01
Type: SUBTOPIC
ID: giving_and_gifting
Label: بخشش و هدیه
Technical recommendation: APPROVE_SUBTOPIC_CANDIDATE
Proposed Parent: خرج، پس‌انداز و مدیریت پول
Evidence documents: 98
Sources: {'website': 33, 'book': 31, 'transcript': 34}
Allowed decisions: ['APPROVE_AS_PROPOSED', 'APPROVE_WITH_REVISION', 'DEFER_FOR_MORE_EVIDENCE', 'REJECT']
Specialist decision: None
Status: PENDING

REVIEW ITEM 02
Type: SUBTOPIC
ID: entrepreneurship_production_resources
Label: منابع تولید
Technical recommendation: APPROVE_SUBTOPIC_CANDIDATE
Proposed Parent: کار، شغل و درآمد
Evidence documents: 7
Sources: {'website': 1, 'book': 5, 'transcript': 1}
Allowed decisions: ['APPROVE_AS_PROPOSED', 'APPROVE_WITH_REVISION', 'DEFER_FOR_MORE_EVIDENCE', 'REJECT']
Specialist decision: None
Status: PENDING

REVIEW

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Deep Corpus Review — بازسازی ساختار مفهومی صریح کتاب‌ها و طرح‌درس‌ها</h3>

<p>
تا این مرحله Taxonomy Candidate عمدتاً از Tagها، Headingها، Activityها، Lexical Evidence و Context استخراج شده بود.
پیش از هر Specialist Review، باید بررسی عمیق‌تری انجام شود که خود منابع آموزشی چگونه مفاهیم را سازمان‌دهی کرده‌اند.
</p>

<p>
کتاب‌ها و طرح‌درس‌های آموزشی در این مرحله قوی‌ترین Evidence برای ساختار Topic/Subtopic محسوب می‌شوند، زیرا مؤلف در آن‌ها ترتیب درس، فصل، بخش، زیرعنوان و فعالیت آموزشی را به‌صورت صریح طراحی کرده است.
</p>

<h4>هدف این مرحله</h4>

<ul>
<li>بازسازی درخت واقعی Headingهای هر Work.</li>
<li>استخراج رابطه Parent → Child از خود ساختار کتاب.</li>
<li>بررسی ۱۰ Concept باز در تمام Heading Pathهای کتاب‌ها.</li>
<li>تشخیص اینکه هر Concept زیر چه Parentهایی واقعاً ظاهر شده است.</li>
<li>بررسی Descendantهای هر Concept برای فهم عمق آموزشی آن.</li>
<li>تفکیک Exact Heading Evidence از صرفاً اشاره در متن.</li>
<li>شناسایی مفاهیمی که خود Source برای آن‌ها ساختار روشن دارد.</li>
</ul>

<h4>اصل تصمیم‌گیری</h4>

<p>
اگر یک Concept در ساختار صریح کتاب یا طرح‌درس دارای Parent پایدار و روشن باشد، این Evidence باید از Co-occurrence و Keyword Matching قوی‌تر تلقی شود.
</p>

<p>
اما هیچ Parent در این Cell به Taxonomy اعمال نمی‌شود. خروجی فقط Source-defined structural evidence است.
</p>

<h4>Transcript</h4>

<p>
Transcript در مرحله بعدی جداگانه و با حفظ Speaker، Program، Episode و Conversation Context بررسی خواهد شد.
سؤال مجری، صحبت شرکت‌کننده و حاشیه گفتگو نباید به‌عنوان ساختار آموزشی صریح مؤلف تلقی شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 2,701 رکورد Book بررسی شوند.</li>
<li>Workها بدون حدس از Metadata موجود شناسایی شوند.</li>
<li>تمام Heading Pathهای واقعی استخراج شوند.</li>
<li>Parent → Child Edgeها ساخته شوند.</li>
<li>برای هر ۱۰ Review Concept وضعیت Exact Heading Structure مشخص شود.</li>
<li>هیچ Taxonomy Mutation انجام نشود.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [71]:
# Code Cell 28

from collections import Counter, defaultdict
import re


# --------------------------------------------------
# 1. Helpers
# --------------------------------------------------

def book_work_key(record):
    """
    Use existing source identity only.
    No new Work identity is inferred.
    """
    candidates = [
        record.get("source_id"),
        record.get(
            "source_metadata",
            {}
        ).get("work_id"),
        record.get(
            "source_metadata",
            {}
        ).get("book_id"),
        record.get("title"),
    ]

    for value in candidates:
        if nonempty(value):
            return str(value)

    return "<unknown-work>"


def get_book_heading_path(record):
    path = (
        record.get(
            "locator",
            {},
        ).get("heading_path")
        or record.get(
            "source_metadata",
            {},
        ).get("heading_path")
        or []
    )

    return normalize_heading_path(path)


def normalize_heading_component(value):
    return normalize_fa_for_match(
        str(value)
    )


def compact_text(value, limit=160):
    text = re.sub(
        r"\s+",
        " ",
        str(value or "").strip(),
    )
    return text[:limit]


# --------------------------------------------------
# 2. Reconstruct Work-level heading trees
# --------------------------------------------------

book_work_records = defaultdict(list)

for record in book_records:
    book_work_records[
        book_work_key(record)
    ].append(record)


print("Book / Course Structure Discovery:")
print(
    "- Book records:",
    len(book_records),
)

print(
    "- Works discovered from existing identity:",
    len(book_work_records),
)


work_heading_paths = defaultdict(set)
work_parent_child_edges = defaultdict(Counter)
work_heading_nodes = defaultdict(Counter)
work_root_headings = defaultdict(Counter)
work_max_depth = Counter()


for work_id, work_records in (
    book_work_records.items()
):

    for record in work_records:

        heading_path = (
            get_book_heading_path(record)
        )

        if not heading_path:
            continue

        normalized_path = tuple(
            normalize_heading_component(
                component
            )
            for component in heading_path
            if nonempty(component)
        )

        if not normalized_path:
            continue

        work_heading_paths[
            work_id
        ].add(normalized_path)

        work_max_depth[
            work_id
        ] = max(
            work_max_depth[
                work_id
            ],
            len(normalized_path),
        )

        work_root_headings[
            work_id
        ][normalized_path[0]] += 1

        for depth, component in enumerate(
            normalized_path
        ):
            work_heading_nodes[
                work_id
            ][component] += 1

            if depth == 0:
                continue

            parent = normalized_path[
                depth - 1
            ]

            child = component

            work_parent_child_edges[
                work_id
            ][
                (parent, child)
            ] += 1


print("\nWork structure summary:")


for work_id in sorted(
    book_work_records
):

    print(
        "\n========================================"
    )

    print(
        "Work:",
        work_id,
    )

    print(
        "- Records:",
        len(
            book_work_records[
                work_id
            ]
        ),
    )

    print(
        "- Unique Heading Paths:",
        len(
            work_heading_paths[
                work_id
            ]
        ),
    )

    print(
        "- Unique Heading Nodes:",
        len(
            work_heading_nodes[
                work_id
            ]
        ),
    )

    print(
        "- Parent→Child Edges:",
        len(
            work_parent_child_edges[
                work_id
            ]
        ),
    )

    print(
        "- Maximum Heading Depth:",
        work_max_depth[
            work_id
        ],
    )

    print(
        "- Most frequent Root Headings:"
    )

    for heading, count in (
        work_root_headings[
            work_id
        ].most_common(12)
    ):
        print(
            f"  - {heading}: {count}"
        )


# --------------------------------------------------
# 3. Global Parent → Child structural inventory
# --------------------------------------------------

global_parent_child_edges = Counter()

for work_id, edge_counter in (
    work_parent_child_edges.items()
):
    global_parent_child_edges.update(
        edge_counter
    )


print(
    "\nGlobal Book structural graph:"
)

print(
    "- Distinct Parent→Child edges:",
    len(
        global_parent_child_edges
    ),
)


# --------------------------------------------------
# 4. Deep structural review of the 10 open concepts
# --------------------------------------------------

deep_book_concept_structure_v1 = {}


for concept in review_concepts:

    normalized_concept = (
        normalize_heading_component(
            concept
        )
    )

    exact_heading_occurrences = []
    containing_heading_occurrences = []

    exact_parent_counter = Counter()
    exact_child_counter = Counter()
    work_counter = Counter()

    body_match_counter = Counter()
    body_context_heading_counter = Counter()

    exact_paths = set()

    for record in book_records:

        document_id = record[
            "document_id"
        ]

        work_id = book_work_key(
            record
        )

        heading_path = (
            get_book_heading_path(
                record
            )
        )

        normalized_path = [
            normalize_heading_component(
                component
            )
            for component in heading_path
        ]

        # ------------------------------------------
        # Heading structure
        # ------------------------------------------

        for index, component in enumerate(
            normalized_path
        ):

            if (
                component
                == normalized_concept
            ):

                parent = (
                    normalized_path[
                        index - 1
                    ]
                    if index > 0
                    else None
                )

                child = (
                    normalized_path[
                        index + 1
                    ]
                    if (
                        index + 1
                        < len(
                            normalized_path
                        )
                    )
                    else None
                )

                exact_heading_occurrences.append(
                    {
                        "document_id": (
                            document_id
                        ),
                        "work_id": work_id,
                        "heading_path": (
                            heading_path
                        ),
                        "normalized_path": (
                            normalized_path
                        ),
                        "parent": parent,
                        "child": child,
                    }
                )

                exact_paths.add(
                    tuple(
                        normalized_path
                    )
                )

                work_counter[
                    work_id
                ] += 1

                if parent:
                    exact_parent_counter[
                        parent
                    ] += 1

                if child:
                    exact_child_counter[
                        child
                    ] += 1

            elif (
                normalized_concept
                in component
                or component
                in normalized_concept
            ):
                containing_heading_occurrences.append(
                    {
                        "document_id": (
                            document_id
                        ),
                        "work_id": work_id,
                        "heading_path": (
                            heading_path
                        ),
                        "matched_component": (
                            component
                        ),
                    }
                )

        # ------------------------------------------
        # Body evidence
        # ------------------------------------------

        normalized_body = (
            normalize_fa_for_match(
                record.get(
                    "text",
                    ""
                )
            )
        )

        if (
            normalized_concept
            and normalized_concept
            in normalized_body
        ):

            body_match_counter[
                work_id
            ] += 1

            if normalized_path:
                body_context_heading_counter[
                    normalized_path[-1]
                ] += 1


    deep_book_concept_structure_v1[
        concept
    ] = {
        "normalized_concept": (
            normalized_concept
        ),
        "exact_heading_occurrences": (
            exact_heading_occurrences
        ),
        "exact_heading_count": len(
            exact_heading_occurrences
        ),
        "unique_exact_paths": sorted(
            exact_paths
        ),
        "containing_heading_count": len(
            containing_heading_occurrences
        ),
        "work_counts": dict(
            work_counter
        ),
        "exact_parent_counts": dict(
            exact_parent_counter
        ),
        "exact_child_counts": dict(
            exact_child_counter
        ),
        "body_match_counts_by_work": dict(
            body_match_counter
        ),
        "body_context_headings": dict(
            body_context_heading_counter
        ),
    }


# --------------------------------------------------
# 5. Print Source-defined structure for each concept
# --------------------------------------------------

print(
    "\n\nSource-defined Concept Structure:"
)


for concept in review_concepts:

    profile = (
        deep_book_concept_structure_v1[
            concept
        ]
    )

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "- Exact Heading occurrences:",
        profile[
            "exact_heading_count"
        ],
    )

    print(
        "- Containing Heading occurrences:",
        profile[
            "containing_heading_count"
        ],
    )

    print(
        "- Works with Exact Heading:",
        profile[
            "work_counts"
        ],
    )

    print(
        "- Body matches by Work:",
        profile[
            "body_match_counts_by_work"
        ],
    )

    print(
        "Exact structural Parents:"
    )

    parent_counter = Counter(
        profile[
            "exact_parent_counts"
        ]
    )

    if parent_counter:
        for parent, count in (
            parent_counter.most_common(10)
        ):
            print(
                f"  - {parent}: {count}"
            )
    else:
        print(
            "  - None"
        )

    print(
        "Exact structural Children:"
    )

    child_counter = Counter(
        profile[
            "exact_child_counts"
        ]
    )

    if child_counter:
        for child, count in (
            child_counter.most_common(10)
        ):
            print(
                f"  - {child}: {count}"
            )
    else:
        print(
            "  - None"
        )

    print(
        "Most common Body-context Headings:"
    )

    body_heading_counter = Counter(
        profile[
            "body_context_headings"
        ]
    )

    if body_heading_counter:
        for heading, count in (
            body_heading_counter.most_common(10)
        ):
            print(
                f"  - {heading}: {count}"
            )
    else:
        print(
            "  - None"
        )

    print(
        "Representative Exact Heading Paths:"
    )

    unique_paths = (
        profile[
            "unique_exact_paths"
        ]
    )

    if unique_paths:
        for path in unique_paths[:12]:
            print(
                "  -",
                " > ".join(path),
            )
    else:
        print(
            "  - None"
        )


# --------------------------------------------------
# 6. Structural stability indicators
#
# These are descriptive only.
# No automatic taxonomy decision.
# --------------------------------------------------

book_structural_stability_v1 = {}


for concept, profile in (
    deep_book_concept_structure_v1.items()
):

    parent_counter = Counter(
        profile[
            "exact_parent_counts"
        ]
    )

    exact_heading_count = (
        profile[
            "exact_heading_count"
        ]
    )

    if parent_counter:

        dominant_parent, dominant_count = (
            parent_counter.most_common(1)[0]
        )

        parent_support_ratio = (
            dominant_count
            / sum(
                parent_counter.values()
            )
        )

    else:
        dominant_parent = None
        dominant_count = 0
        parent_support_ratio = None


    if (
        exact_heading_count > 0
        and parent_counter
        and parent_support_ratio >= 0.80
    ):
        structural_status = (
            "STABLE_EXPLICIT_PARENT_EVIDENCE"
        )

    elif (
        exact_heading_count > 0
        and len(parent_counter) > 1
    ):
        structural_status = (
            "MULTIPLE_EXPLICIT_PARENT_CONTEXTS"
        )

    elif exact_heading_count > 0:
        structural_status = (
            "EXPLICIT_HEADING_WITHOUT_PARENT"
        )

    else:
        structural_status = (
            "NO_EXACT_BOOK_HEADING_EVIDENCE"
        )


    book_structural_stability_v1[
        concept
    ] = {
        "structural_status": (
            structural_status
        ),
        "dominant_parent": (
            dominant_parent
        ),
        "dominant_parent_count": (
            dominant_count
        ),
        "parent_support_ratio": (
            parent_support_ratio
        ),
        "exact_heading_count": (
            exact_heading_count
        ),
    }


print(
    "\n\nBook Structural Stability Summary:"
)


for concept in review_concepts:

    row = (
        book_structural_stability_v1[
            concept
        ]
    )

    print(
        "\n-",
        concept,
    )

    print(
        "  status:",
        row[
            "structural_status"
        ],
    )

    print(
        "  dominant_parent:",
        row[
            "dominant_parent"
        ],
    )

    print(
        "  parent_support_ratio:",
        (
            round(
                row[
                    "parent_support_ratio"
                ],
                3,
            )
            if (
                row[
                    "parent_support_ratio"
                ]
                is not None
            )
            else None
        ),
    )


# --------------------------------------------------
# 7. Validation
# --------------------------------------------------

concepts_without_profile = [
    concept
    for concept in review_concepts
    if concept not in (
        deep_book_concept_structure_v1
    )
]


invalid_work_ids = [
    work_id
    for work_id in (
        book_work_records
    )
    if work_id == "<unknown-work>"
]


records_accounted_for = sum(
    len(work_records)
    for work_records in (
        book_work_records.values()
    )
)


taxonomy_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Book records expected:",
    2701,
)

print(
    "- Book records accounted for:",
    records_accounted_for,
)

print(
    "- Works discovered:",
    len(
        book_work_records
    ),
)

print(
    "- Review Concepts expected:",
    10,
)

print(
    "- Review Concepts profiled:",
    len(
        deep_book_concept_structure_v1
    ),
)

print(
    "- Concepts without profile:",
    len(
        concepts_without_profile
    ),
)

print(
    "- Unknown Work identities:",
    len(
        invalid_work_ids
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert records_accounted_for == 2701

assert len(
    deep_book_concept_structure_v1
) == 10

assert not concepts_without_profile
assert not invalid_work_ids

assert (
    taxonomy_mutation_performed
    is False
)

assert (
    disk_write_performed
    is False
)


print("\nCODE CELL 28 GATE: PASS")
print(
    "Deep Book/Course source-defined hierarchy "
    "discovery completed successfully; explicit "
    "curriculum structure has been extracted "
    "without changing the Taxonomy."
)

Book / Course Structure Discovery:
- Book records: 2701
- Works discovered from existing identity: 3

Work structure summary:

Work: book:adult-financial-literacy-course:fa-source
- Records: 1333
- Unique Heading Paths: 98
- Unique Heading Nodes: 97
- Parent→Child Edges: 59
- Maximum Heading Depth: 2
- Most frequent Root Headings:
  - درس ۲۲: صندوق های سرمایه گذاری: 199
  - درس 15: رهایی از نارضایتی شغلی: 116
  - درس 14: یافتن شغل: 70
  - درس ۲: پلکان توانمندی مالی: 69
  - درس 7: بودجه بندی: 54
  - درس ۱۷: سبک ها و رویکردهای گوناگون شغلی: 50
  - درس 21: بازار سرمایه و بورس: 46
  - درس 20: گزینه های سرمایه گذاری: 45
  - درس 12: کارت اعتباری و اعتبارسنجی: 41
  - درس 13: بانک و اعتبار: 40
  - درس 24: بیمه: 38
  - درس ۲۵: مؤلفه های مالی در تصمیم گیری مالی: 38

Work: book:financial-literacy-for-all:fa-source
- Records: 873
- Unique Heading Paths: 9
- Unique Heading Nodes: 9
- Parent→Child Edges: 6
- Maximum Heading Depth: 2
- Most frequent Root Headings:
  - زمانی برای اندیشه: 620
  - ترکیب

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Deep Transcript Context Audit — Speaker، Episode و Evidence مستقیم</h3>

<p>
ساختار کتاب نشان داد که بسیاری از روابط Topic/Subtopic مستقیماً از طراحی آموزشی مؤلف قابل استخراج‌اند.
Transcript ماهیت متفاوتی دارد و نباید با همان منطق تحلیل شود.
</p>

<p>
در گفتگوهای صوتی و تصویری ممکن است مجری، مهمان، شرکت‌کننده یا فرد دیگری سؤال یا نکته‌ای را مطرح کند و پاسخ دکتر رودی به‌صورت موقت وارد یک موضوع فرعی شود.
بنابراین وجود یک واژه در Transcript به‌تنهایی نباید به معنی Topic اصلی آن Segment یا Episode باشد.
</p>

<h4>این Audit سه نوع Evidence را جدا نگه می‌دارد</h4>

<ul>
<li><strong>Primary Direct Mention:</strong> مفهوم مستقیماً در Segment منتسب به دکتر کمیل رودی دیده می‌شود.</li>
<li><strong>Other/Participant Mention:</strong> مفهوم در Segment دارای Speaker دیگری دیده می‌شود.</li>
<li><strong>Unresolved Speaker Mention:</strong> مفهوم وجود دارد ولی Speaker Canonical مشخص نیست.</li>
</ul>

<p>
همچنین برای Mentionهای غیرPrimary بررسی می‌شود آیا در Segmentهای مجاور همان Episode، پاسخ یا Context منتسب به دکتر رودی وجود دارد یا خیر.
این همسایگی فقط <strong>Conversation Context</strong> محسوب می‌شود و به‌طور خودکار Topic خود Segment را تغییر نمی‌دهد.
</p>

<h4>هدف</h4>

<p>
برای ۱۰ Concept باز مشخص شود:
در کدام Programها و Episodeها مطرح شده‌اند، چه مقدار Evidence مستقیم از دکتر رودی وجود دارد، چه مقدار متعلق به دیگر Speakerها یا Speaker نامشخص است، و آیا برخی Mentionها فقط بخشی از سؤال یا گفتگوی حاشیه‌ای هستند.
</p>

<p>
این مرحله هیچ Transcript Topic، Speaker Attribution یا Taxonomy را تغییر نمی‌دهد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 16,226 Transcript بررسی شوند.</li>
<li>Speaker Attribution موجود بدون استنتاج Speaker جدید حفظ شود.</li>
<li>۱۰ Review Concept پروفایل Transcript داشته باشند.</li>
<li>Program و Episode concentration گزارش شود.</li>
<li>Primary Direct Mention از Conversation Context جدا بماند.</li>
<li>نمونه‌های Context به‌صورت Deterministic استخراج شوند.</li>
<li>هیچ Taxonomy Mutation، Speaker Inference یا Disk Write انجام نشود.</li>
</ul>

</div>

In [72]:
# Code Cell 29

from collections import Counter, defaultdict
import re


# --------------------------------------------------
# 1. Transcript inventory
# --------------------------------------------------

transcript_records = list(
    source_records["transcript"]
)

assert len(transcript_records) == 16226


def transcript_speaker(record):
    """
    Preserve only explicit canonical speaker data.
    No speaker inference is allowed.
    """
    attribution = record.get(
        "attribution",
        {},
    )

    return attribution.get(
        "speaker"
    )


def transcript_program(record):
    metadata = record.get(
        "source_metadata",
        {},
    )

    return (
        metadata.get("program")
        or record.get("title")
        or "<unknown-program>"
    )


def transcript_speaker_role(record):
    """
    Read an explicit role only if the current
    attribution already contains one.
    """
    attribution = record.get(
        "attribution",
        {},
    )

    for key in (
        "speaker_role",
        "role",
        "participant_role",
    ):
        value = attribution.get(key)

        if nonempty(value):
            return str(value)

    return None


# --------------------------------------------------
# 2. Discover actual attribution / metadata shape
# --------------------------------------------------

transcript_attribution_key_counter = Counter()
transcript_metadata_key_counter = Counter()
transcript_speaker_counter = Counter()
transcript_role_counter = Counter()


for record in transcript_records:

    attribution = record.get(
        "attribution",
        {},
    )

    metadata = record.get(
        "source_metadata",
        {},
    )

    transcript_attribution_key_counter.update(
        attribution.keys()
    )

    transcript_metadata_key_counter.update(
        metadata.keys()
    )

    speaker = transcript_speaker(
        record
    )

    if nonempty(speaker):
        transcript_speaker_counter[
            str(speaker)
        ] += 1
    else:
        transcript_speaker_counter[
            "<null>"
        ] += 1

    role = transcript_speaker_role(
        record
    )

    if role is not None:
        transcript_role_counter[
            role
        ] += 1


print("Transcript Deep Context Audit:")
print(
    "- Transcript records:",
    len(transcript_records),
)

print(
    "- Attribution keys:",
    sorted(
        transcript_attribution_key_counter
    ),
)

print(
    "- Source metadata keys:",
    sorted(
        transcript_metadata_key_counter
    ),
)

print(
    "\nExplicit Speaker values:"
)

for speaker, count in (
    transcript_speaker_counter.most_common(20)
):
    print(
        f"- {speaker}: {count}"
    )


print(
    "\nExplicit Speaker roles:"
)

if transcript_role_counter:
    for role, count in (
        transcript_role_counter.most_common()
    ):
        print(
            f"- {role}: {count}"
        )
else:
    print(
        "- No explicit speaker-role field available"
    )


# --------------------------------------------------
# 3. Primary Speaker policy
#
# This exact canonical value was already observed
# in the accepted corpus. No aliases are inferred.
# --------------------------------------------------

PRIMARY_CANONICAL_SPEAKER = (
    "Dr. Komeil Roudi"
)


primary_speaker_record_count = (
    transcript_speaker_counter[
        PRIMARY_CANONICAL_SPEAKER
    ]
)


print(
    "\nPrimary Speaker reference:"
)

print(
    "- Canonical Speaker value:",
    PRIMARY_CANONICAL_SPEAKER,
)

print(
    "- Explicit records:",
    primary_speaker_record_count,
)


# --------------------------------------------------
# 4. Episode / conversation unit discovery
# --------------------------------------------------

def transcript_episode_key(record):
    """
    Prefer explicit metadata/locator fields.
    Fall back only to a validated document_id
    episode prefix when needed.
    """

    metadata = record.get(
        "source_metadata",
        {},
    )

    locator = record.get(
        "locator",
        {},
    )

    program = transcript_program(
        record
    )

    for container in (
        metadata,
        locator,
    ):
        for key in (
            "episode_id",
            "episode",
            "episode_number",
        ):
            value = container.get(key)

            if nonempty(value):
                return (
                    f"{program}|{value}"
                )

    document_id = str(
        record.get(
            "document_id",
            ""
        )
    )

    match = re.match(
        r"^(.*?:episode-[^:]+):"
        r"(?:utterance|merged):",
        document_id,
    )

    if match:
        return match.group(1)

    return None


def transcript_sequence_order(record):
    """
    Ordering is derived only from explicit numeric
    utterance/merged suffix when available.
    """

    document_id = str(
        record.get(
            "document_id",
            ""
        )
    )

    match = re.search(
        r":utterance:(\d+)$",
        document_id,
    )

    if match:
        return int(
            match.group(1)
        )

    match = re.search(
        r":merged:(\d+)(?:-\d+)?$",
        document_id,
    )

    if match:
        return int(
            match.group(1)
        )

    return None


episode_records = defaultdict(list)

episode_key_failures = []
sequence_order_failures = []


for record in transcript_records:

    episode_key = (
        transcript_episode_key(
            record
        )
    )

    order = (
        transcript_sequence_order(
            record
        )
    )

    if episode_key is None:
        episode_key_failures.append(
            record["document_id"]
        )
        continue

    if order is None:
        sequence_order_failures.append(
            record["document_id"]
        )

    episode_records[
        episode_key
    ].append(
        record
    )


for episode_key in episode_records:

    episode_records[
        episode_key
    ] = sorted(
        episode_records[
            episode_key
        ],
        key=lambda record: (
            transcript_sequence_order(
                record
            )
            if (
                transcript_sequence_order(
                    record
                )
                is not None
            )
            else 10**12,
            record["document_id"],
        ),
    )


print(
    "\nConversation-unit discovery:"
)

print(
    "- Episode units:",
    len(
        episode_records
    ),
)

print(
    "- Records without Episode key:",
    len(
        episode_key_failures
    ),
)

print(
    "- Records without sequence order:",
    len(
        sequence_order_failures
    ),
)


# --------------------------------------------------
# 5. Speaker evidence classification
# --------------------------------------------------

def speaker_evidence_class(record):

    speaker = transcript_speaker(
        record
    )

    if (
        speaker
        == PRIMARY_CANONICAL_SPEAKER
    ):
        return "PRIMARY_DIRECT"

    if nonempty(speaker):
        return "EXPLICIT_OTHER_SPEAKER"

    return "UNRESOLVED_SPEAKER"


# --------------------------------------------------
# 6. Build lookup for neighbor context
# --------------------------------------------------

episode_position_lookup = {}


for episode_key, records_in_episode in (
    episode_records.items()
):

    for position, record in enumerate(
        records_in_episode
    ):
        episode_position_lookup[
            record["document_id"]
        ] = {
            "episode_key": (
                episode_key
            ),
            "position": position,
        }


def transcript_neighbor_window(
    document_id,
    radius=2,
):

    location = (
        episode_position_lookup.get(
            document_id
        )
    )

    if location is None:
        return []

    episode_key = location[
        "episode_key"
    ]

    position = location[
        "position"
    ]

    records_in_episode = (
        episode_records[
            episode_key
        ]
    )

    start = max(
        0,
        position - radius,
    )

    end = min(
        len(records_in_episode),
        position + radius + 1,
    )

    return records_in_episode[
        start:end
    ]


# --------------------------------------------------
# 7. Deep Transcript profile for Review Concepts
# --------------------------------------------------

deep_transcript_concept_profile_v1 = {}


for concept in review_concepts:

    normalized_concept = (
        normalize_fa_for_match(
            concept
        )
    )

    matching_records = []

    speaker_class_counter = Counter()
    explicit_speaker_counter = Counter()
    program_counter = Counter()
    episode_counter = Counter()

    episode_primary_mentions = Counter()
    episode_other_mentions = Counter()
    episode_unresolved_mentions = Counter()

    nonprimary_with_primary_neighbor = 0
    nonprimary_without_primary_neighbor = 0

    for record in transcript_records:

        normalized_text = (
            normalize_fa_for_match(
                record.get(
                    "text",
                    ""
                )
            )
        )

        if (
            not normalized_concept
            or normalized_concept
            not in normalized_text
        ):
            continue

        document_id = record[
            "document_id"
        ]

        matching_records.append(
            record
        )

        evidence_class = (
            speaker_evidence_class(
                record
            )
        )

        speaker_class_counter[
            evidence_class
        ] += 1

        speaker = transcript_speaker(
            record
        )

        explicit_speaker_counter[
            (
                str(speaker)
                if nonempty(speaker)
                else "<null>"
            )
        ] += 1

        program = transcript_program(
            record
        )

        program_counter[
            program
        ] += 1

        episode_key = (
            transcript_episode_key(
                record
            )
        )

        if episode_key is not None:

            episode_counter[
                episode_key
            ] += 1

            if (
                evidence_class
                == "PRIMARY_DIRECT"
            ):
                episode_primary_mentions[
                    episode_key
                ] += 1

            elif (
                evidence_class
                == "EXPLICIT_OTHER_SPEAKER"
            ):
                episode_other_mentions[
                    episode_key
                ] += 1

            else:
                episode_unresolved_mentions[
                    episode_key
                ] += 1

        # ------------------------------------------
        # For non-primary mentions only:
        # is explicit Primary Speaker present nearby?
        #
        # This is Context evidence only.
        # ------------------------------------------

        if (
            evidence_class
            != "PRIMARY_DIRECT"
        ):

            neighbors = (
                transcript_neighbor_window(
                    document_id,
                    radius=2,
                )
            )

            has_primary_neighbor = any(
                (
                    neighbor[
                        "document_id"
                    ]
                    != document_id
                )
                and (
                    speaker_evidence_class(
                        neighbor
                    )
                    == "PRIMARY_DIRECT"
                )
                for neighbor in neighbors
            )

            if has_primary_neighbor:
                nonprimary_with_primary_neighbor += 1
            else:
                nonprimary_without_primary_neighbor += 1


    deep_transcript_concept_profile_v1[
        concept
    ] = {
        "normalized_concept": (
            normalized_concept
        ),
        "matching_document_ids": [
            record["document_id"]
            for record in matching_records
        ],
        "mention_count": len(
            matching_records
        ),
        "speaker_class_counts": dict(
            speaker_class_counter
        ),
        "explicit_speaker_counts": dict(
            explicit_speaker_counter
        ),
        "program_counts": dict(
            program_counter
        ),
        "episode_counts": dict(
            episode_counter
        ),
        "episode_primary_mentions": dict(
            episode_primary_mentions
        ),
        "episode_other_mentions": dict(
            episode_other_mentions
        ),
        "episode_unresolved_mentions": dict(
            episode_unresolved_mentions
        ),
        "nonprimary_with_primary_neighbor": (
            nonprimary_with_primary_neighbor
        ),
        "nonprimary_without_primary_neighbor": (
            nonprimary_without_primary_neighbor
        ),
    }


# --------------------------------------------------
# 8. Print concept-level Transcript evidence
# --------------------------------------------------

print(
    "\n\nTranscript Concept Evidence:"
)


for concept in review_concepts:

    profile = (
        deep_transcript_concept_profile_v1[
            concept
        ]
    )

    speaker_counts = Counter(
        profile[
            "speaker_class_counts"
        ]
    )

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "- Transcript mentions:",
        profile[
            "mention_count"
        ],
    )

    print(
        "- Primary Direct:",
        speaker_counts[
            "PRIMARY_DIRECT"
        ],
    )

    print(
        "- Explicit Other Speaker:",
        speaker_counts[
            "EXPLICIT_OTHER_SPEAKER"
        ],
    )

    print(
        "- Unresolved Speaker:",
        speaker_counts[
            "UNRESOLVED_SPEAKER"
        ],
    )

    print(
        "- Non-primary mentions with "
        "Primary neighbor ±2:",
        profile[
            "nonprimary_with_primary_neighbor"
        ],
    )

    print(
        "- Non-primary mentions without "
        "Primary neighbor ±2:",
        profile[
            "nonprimary_without_primary_neighbor"
        ],
    )

    print(
        "Top Programs:"
    )

    program_counter = Counter(
        profile[
            "program_counts"
        ]
    )

    if program_counter:
        for program, count in (
            program_counter.most_common(8)
        ):
            print(
                f"  - {program}: {count}"
            )
    else:
        print(
            "  - None"
        )

    print(
        "Top Episodes by exact Concept mentions:"
    )

    episode_counter = Counter(
        profile[
            "episode_counts"
        ]
    )

    if episode_counter:

        for episode_key, count in (
            episode_counter.most_common(8)
        ):

            primary_count = (
                profile[
                    "episode_primary_mentions"
                ].get(
                    episode_key,
                    0,
                )
            )

            other_count = (
                profile[
                    "episode_other_mentions"
                ].get(
                    episode_key,
                    0,
                )
            )

            unresolved_count = (
                profile[
                    "episode_unresolved_mentions"
                ].get(
                    episode_key,
                    0,
                )
            )

            print(
                f"  - {episode_key}: "
                f"total={count} "
                f"| primary={primary_count} "
                f"| other={other_count} "
                f"| unresolved={unresolved_count}"
            )

    else:
        print(
            "  - None"
        )


# --------------------------------------------------
# 9. Deterministic conversation-context samples
#
# Prefer:
# 1) non-primary mention with Primary neighbor
# 2) direct Primary mention
# at most 2 samples per concept.
# --------------------------------------------------

transcript_context_review_samples_v1 = (
    defaultdict(list)
)


for concept in review_concepts:

    profile = (
        deep_transcript_concept_profile_v1[
            concept
        ]
    )

    matching_ids = profile[
        "matching_document_ids"
    ]

    candidate_rows = []

    for document_id in matching_ids:

        record = record_lookup[
            document_id
        ]

        evidence_class = (
            speaker_evidence_class(
                record
            )
        )

        neighbors = (
            transcript_neighbor_window(
                document_id,
                radius=2,
            )
        )

        has_primary_neighbor = any(
            (
                neighbor[
                    "document_id"
                ]
                != document_id
            )
            and (
                speaker_evidence_class(
                    neighbor
                )
                == "PRIMARY_DIRECT"
            )
            for neighbor in neighbors
        )

        if (
            evidence_class
            != "PRIMARY_DIRECT"
            and has_primary_neighbor
        ):
            sample_priority = 0

        elif (
            evidence_class
            == "PRIMARY_DIRECT"
        ):
            sample_priority = 1

        else:
            sample_priority = 2

        candidate_rows.append(
            {
                "document_id": (
                    document_id
                ),
                "sample_priority": (
                    sample_priority
                ),
                "evidence_class": (
                    evidence_class
                ),
            }
        )


    candidate_rows = sorted(
        candidate_rows,
        key=lambda row: (
            row[
                "sample_priority"
            ],
            deterministic_review_rank(
                row["document_id"],
                (
                    f"NB10-transcript-context:"
                    f"{concept}:v1"
                ),
            ),
        ),
    )


    for candidate_row in (
        candidate_rows[:2]
    ):

        document_id = (
            candidate_row[
                "document_id"
            ]
        )

        target_record = record_lookup[
            document_id
        ]

        window = (
            transcript_neighbor_window(
                document_id,
                radius=2,
            )
        )

        transcript_context_review_samples_v1[
            concept
        ].append(
            {
                "document_id": document_id,
                "target_evidence_class": (
                    candidate_row[
                        "evidence_class"
                    ]
                ),
                "episode_key": (
                    transcript_episode_key(
                        target_record
                    )
                ),
                "program": (
                    transcript_program(
                        target_record
                    )
                ),
                "window": [
                    {
                        "document_id": (
                            row[
                                "document_id"
                            ]
                        ),
                        "speaker": (
                            transcript_speaker(
                                row
                            )
                        ),
                        "speaker_role": (
                            transcript_speaker_role(
                                row
                            )
                        ),
                        "speaker_class": (
                            speaker_evidence_class(
                                row
                            )
                        ),
                        "is_target": (
                            row[
                                "document_id"
                            ]
                            == document_id
                        ),
                        "text": compact_excerpt(
                            row,
                            limit=420,
                        ),
                    }
                    for row in window
                ],
            }
        )


# --------------------------------------------------
# 10. Print conversation samples
# --------------------------------------------------

print(
    "\n\nRepresentative Conversation Context:"
)


for concept in review_concepts:

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    samples = (
        transcript_context_review_samples_v1[
            concept
        ]
    )

    if not samples:
        print(
            "- No Transcript sample available"
        )
        continue

    for sample_index, sample in enumerate(
        samples,
        start=1,
    ):

        print(
            f"\nSample {sample_index}"
        )

        print(
            "- Program:",
            sample[
                "program"
            ],
        )

        print(
            "- Episode:",
            sample[
                "episode_key"
            ],
        )

        print(
            "- Target evidence class:",
            sample[
                "target_evidence_class"
            ],
        )

        for row in sample[
            "window"
        ]:

            marker = (
                "TARGET"
                if row[
                    "is_target"
                ]
                else "context"
            )

            print(
                "\n ",
                marker,
                "|",
                row[
                    "speaker_class"
                ],
                "| speaker=",
                row[
                    "speaker"
                ],
                "| role=",
                row[
                    "speaker_role"
                ],
            )

            print(
                " ",
                row["text"],
            )


# --------------------------------------------------
# 11. Descriptive transcript evidence summary
#
# No automatic Taxonomy decision.
# --------------------------------------------------

transcript_primary_evidence_summary_v1 = {}


for concept, profile in (
    deep_transcript_concept_profile_v1.items()
):

    counts = Counter(
        profile[
            "speaker_class_counts"
        ]
    )

    total_mentions = profile[
        "mention_count"
    ]

    primary_mentions = counts[
        "PRIMARY_DIRECT"
    ]

    if total_mentions:

        primary_share = (
            primary_mentions
            / total_mentions
        )

    else:
        primary_share = None

    transcript_primary_evidence_summary_v1[
        concept
    ] = {
        "total_mentions": (
            total_mentions
        ),
        "primary_direct_mentions": (
            primary_mentions
        ),
        "primary_share": (
            primary_share
        ),
        "episodes": len(
            profile[
                "episode_counts"
            ]
        ),
        "programs": len(
            profile[
                "program_counts"
            ]
        ),
    }


print(
    "\n\nTranscript Primary Evidence Summary:"
)


for concept in review_concepts:

    row = (
        transcript_primary_evidence_summary_v1[
            concept
        ]
    )

    print(
        "-",
        concept,
        "| mentions=",
        row[
            "total_mentions"
        ],
        "| primary_direct=",
        row[
            "primary_direct_mentions"
        ],
        "| primary_share=",
        (
            round(
                row[
                    "primary_share"
                ],
                3,
            )
            if (
                row[
                    "primary_share"
                ]
                is not None
            )
            else None
        ),
        "| episodes=",
        row[
            "episodes"
        ],
        "| programs=",
        row[
            "programs"
        ],
    )


# --------------------------------------------------
# 12. Independent validation
# --------------------------------------------------

concepts_without_transcript_profile = [
    concept
    for concept in review_concepts
    if concept not in (
        deep_transcript_concept_profile_v1
    )
]


speaker_inference_performed = False
taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
disk_write_performed = False


records_in_episode_index = sum(
    len(rows)
    for rows in (
        episode_records.values()
    )
)


print("\nValidation:")

print(
    "- Transcript records expected:",
    16226,
)

print(
    "- Transcript records inspected:",
    len(
        transcript_records
    ),
)

print(
    "- Records indexed into Episode units:",
    records_in_episode_index,
)

print(
    "- Episode-key failures:",
    len(
        episode_key_failures
    ),
)

print(
    "- Review Concepts expected:",
    10,
)

print(
    "- Review Concepts profiled:",
    len(
        deep_transcript_concept_profile_v1
    ),
)

print(
    "- Concepts without Transcript profile:",
    len(
        concepts_without_transcript_profile
    ),
)

print(
    "- Speaker inference performed:",
    speaker_inference_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    transcript_records
) == 16226

assert len(
    deep_transcript_concept_profile_v1
) == 10

assert not (
    concepts_without_transcript_profile
)

assert (
    speaker_inference_performed
    is False
)

assert (
    taxonomy_mutation_performed
    is False
)

assert (
    topic_mapping_mutation_performed
    is False
)

assert (
    disk_write_performed
    is False
)


print("\nCODE CELL 29 GATE: PASS")
print(
    "Deep Transcript Context Audit completed "
    "successfully; direct Primary evidence, "
    "other-speaker evidence and conversation "
    "context remain explicitly separated."
)

Transcript Deep Context Audit:
- Transcript records: 16226
- Attribution keys: ['attribution_status', 'author', 'author_id', 'authorship_assertion_source', 'byline_raw', 'content_authority_type', 'credited_author_name', 'editorial_credit_raw', 'editorial_supervisor_name', 'knowledge_owner', 'reference_expert_name', 'speaker']
- Source metadata keys: ['merge', 'processing_state', 'program', 'program_id', 'record_type', 'record_version', 'source_identity', 'source_text', 'source_text_sha256', 'speaker', 'text_state', 'timestamps', 'transcript_unit_id', 'utterance_index']

Explicit Speaker values:
- Dr. Komeil Roudi: 11210
- <null>: 5016

Explicit Speaker roles:
- No explicit speaker-role field available

Primary Speaker reference:
- Canonical Speaker value: Dr. Komeil Roudi
- Explicit records: 11210

Conversation-unit discovery:
- Episode units: 179
- Records without Episode key: 473
- Records without sequence order: 0


Transcript Concept Evidence:

Concept: بخشش
- Transcript mentions: 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Knowledge Authority Ranking v1 — تفکیک دانش اصلی، تحت‌نظارت و Context</h3>

<p>
پیش از نهایی‌سازی Taxonomy باید مشخص شود هر Document چه نقشی در ساخت Knowledge Structure دارد.
تمام Documents ارزش یکسان برای تعریف دیدگاه و ساختار آموزشی دکتر کمیل رودی ندارند.
</p>

<h4>Authority Layers</h4>

<p><strong>Tier 1 — CORE_PRIMARY_KNOWLEDGE</strong></p>
<ul>
<li>کتاب‌ها و درسنامه‌هایی که Author/Knowledge Owner آن‌ها دکتر کمیل رودی است.</li>
<li>Website با <code>direct_author</code>.</li>
<li>Transcript با Canonical Speaker برابر دکتر کمیل رودی.</li>
</ul>

<p>
این Tier می‌تواند مرجع اصلی برای Topic، Subtopic، Competency، Claim و ساختار آموزشی باشد.
</p>

<p><strong>Tier 2 — SUPERVISED_SECONDARY_KNOWLEDGE</strong></p>
<ul>
<li>محتوای Website با <code>editorial_supervision</code>.</li>
</ul>

<p>
این محتوا می‌تواند Knowledge اصلی را پشتیبانی، گسترش یا مثال‌دار کند؛
اما در صورت پراکندگی یا تعارض، نباید بر ساختار Primary غلبه کند.
</p>

<p><strong>Tier 3 — CONTEXTUAL_NONPRIMARY</strong></p>
<ul>
<li>مجری، مهمان، شرکت‌کننده یا Speaker دیگری در Transcript.</li>
<li>سؤال‌ها و دیدگاه‌های غیرPrimary داخل گفتگو.</li>
</ul>

<p>
این محتوا برای Conversation Context و Attribution حفظ می‌شود،
ولی Evidence اصلی برای دیدگاه یا Taxonomy دکتر رودی نیست.
</p>

<p><strong>Tier 4 — UNRESOLVED_CONTEXT</strong></p>
<ul>
<li>Transcriptهایی که Speaker Canonical مشخص نیست و Evidence کافی برای Attribution ندارند.</li>
</ul>

<p>
این محتوا تا زمان حل Attribution نباید وارد Core Knowledge Structure شود.
</p>

<h4>اصل عدم حذف</h4>

<p>
هیچ Document در این مرحله حذف نمی‌شود.
Authority Ranking فقط مشخص می‌کند هر Document در چه لایه‌ای از Knowledge System استفاده شود.
</p>

<h4>اصل Taxonomy</h4>

<p>
Core Taxonomy ابتدا باید از Tier 1 بازسازی شود.
Tier 2 فقط برای Support و Gap Checking وارد می‌شود.
Tier 3 و Tier 4 نباید پراکندگی خود را به Core Taxonomy تحمیل کنند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 19,456 Document دقیقاً یک Authority Layer تحلیلی دریافت کنند.</li>
<li>Book، Website و Transcript با Evidence موجود و بدون حدس طبقه‌بندی شوند.</li>
<li>Website direct و supervised از هم جدا شوند.</li>
<li>Transcript Primary و Contextual/Unresolved از هم جدا شوند.</li>
<li>هیچ Attribution جدید استنتاج نشود.</li>
<li>هیچ Document حذف نشود.</li>
<li>هیچ Taxonomy Mutation یا Disk Write انجام نشود.</li>
</ul>

</div>

In [73]:
# Code Cell 30

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Canonical Primary authority
# --------------------------------------------------

PRIMARY_CANONICAL_NAME = "Dr. Komeil Roudi"


# --------------------------------------------------
# 2. Existing-evidence helpers
# --------------------------------------------------

def attribution_value(record, key):
    return record.get(
        "attribution",
        {},
    ).get(key)


def metadata_value(record, key):
    return record.get(
        "source_metadata",
        {},
    ).get(key)


def explicit_primary_identity_present(record):
    """
    Uses only existing explicit attribution fields.
    No identity inference.
    """
    values = {
        attribution_value(
            record,
            "author",
        ),
        attribution_value(
            record,
            "knowledge_owner",
        ),
        attribution_value(
            record,
            "speaker",
        ),
    }

    return (
        PRIMARY_CANONICAL_NAME
        in values
    )


# --------------------------------------------------
# 3. Authority classification policy
# --------------------------------------------------

def classify_knowledge_authority(record):

    source_type = record[
        "source_type"
    ]

    attribution = record.get(
        "attribution",
        {},
    )

    content_authority_type = attribution.get(
        "content_authority_type"
    )

    # ----------------------------------------------
    # BOOK
    # ----------------------------------------------

    if source_type == "book":

        if explicit_primary_identity_present(
            record
        ):
            return {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "EXPLICIT_BOOK_AUTHOR_OR_KNOWLEDGE_OWNER"
                ),
                "core_taxonomy_evidence": True,
                "may_support_core": True,
            }

        return {
            "authority_tier": 4,
            "authority_class": (
                "UNRESOLVED_CONTEXT"
            ),
            "authority_basis": (
                "BOOK_PRIMARY_IDENTITY_NOT_EXPLICIT"
            ),
            "core_taxonomy_evidence": False,
            "may_support_core": False,
        }

    # ----------------------------------------------
    # WEBSITE
    # ----------------------------------------------

    if source_type == "website":

        if (
            content_authority_type
            == "direct_author"
        ):
            return {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_DIRECT_AUTHOR"
                ),
                "core_taxonomy_evidence": True,
                "may_support_core": True,
            }

        if (
            content_authority_type
            == "editorial_supervision"
        ):
            return {
                "authority_tier": 2,
                "authority_class": (
                    "SUPERVISED_SECONDARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_EDITORIAL_SUPERVISION"
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": True,
            }

        return {
            "authority_tier": 4,
            "authority_class": (
                "UNRESOLVED_CONTEXT"
            ),
            "authority_basis": (
                "WEBSITE_AUTHORITY_NOT_RESOLVED"
            ),
            "core_taxonomy_evidence": False,
            "may_support_core": False,
        }

    # ----------------------------------------------
    # TRANSCRIPT
    # ----------------------------------------------

    if source_type == "transcript":

        canonical_speaker = (
            attribution.get(
                "speaker"
            )
        )

        raw_speaker = metadata_value(
            record,
            "speaker",
        )

        if (
            canonical_speaker
            == PRIMARY_CANONICAL_NAME
        ):
            return {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "TRANSCRIPT_CANONICAL_PRIMARY_SPEAKER"
                ),
                "core_taxonomy_evidence": True,
                "may_support_core": True,
            }

        if (
            canonical_speaker is not None
            and canonical_speaker
            != PRIMARY_CANONICAL_NAME
        ):
            return {
                "authority_tier": 3,
                "authority_class": (
                    "CONTEXTUAL_NONPRIMARY"
                ),
                "authority_basis": (
                    "TRANSCRIPT_EXPLICIT_NONPRIMARY_SPEAKER"
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

        if nonempty(raw_speaker):
            return {
                "authority_tier": 3,
                "authority_class": (
                    "CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED"
                ),
                "authority_basis": (
                    "TRANSCRIPT_RAW_SPEAKER_PRESENT_"
                    "WITHOUT_CANONICAL_PRIMARY_ATTRIBUTION"
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

        return {
            "authority_tier": 4,
            "authority_class": (
                "UNRESOLVED_CONTEXT"
            ),
            "authority_basis": (
                "TRANSCRIPT_SPEAKER_UNRESOLVED"
            ),
            "core_taxonomy_evidence": False,
            "may_support_core": False,
        }

    raise ValueError(
        f"Unexpected source_type: {source_type}"
    )


# --------------------------------------------------
# 4. Apply analytically to all records
# --------------------------------------------------

knowledge_authority_profile_v1 = {}


for record in records:

    document_id = record[
        "document_id"
    ]

    knowledge_authority_profile_v1[
        document_id
    ] = (
        classify_knowledge_authority(
            record
        )
    )


assert len(
    knowledge_authority_profile_v1
) == 19456


# --------------------------------------------------
# 5. Overall Authority distribution
# --------------------------------------------------

authority_class_counter = Counter()
authority_tier_counter = Counter()

authority_by_source = defaultdict(
    Counter
)


for document_id, profile in (
    knowledge_authority_profile_v1.items()
):

    source_type = record_lookup[
        document_id
    ]["source_type"]

    authority_class_counter[
        profile[
            "authority_class"
        ]
    ] += 1

    authority_tier_counter[
        profile[
            "authority_tier"
        ]
    ] += 1

    authority_by_source[
        source_type
    ][
        profile[
            "authority_class"
        ]
    ] += 1


print("Knowledge Authority Ranking v1:")

print(
    "- Documents:",
    len(
        knowledge_authority_profile_v1
    ),
)

print("\nAuthority Tiers:")

for tier in sorted(
    authority_tier_counter
):
    print(
        f"- Tier {tier}: "
        f"{authority_tier_counter[tier]}"
    )


print("\nAuthority Classes:")

for authority_class, count in (
    authority_class_counter.most_common()
):
    print(
        f"- {authority_class}: {count}"
    )


# --------------------------------------------------
# 6. Source-specific distributions
# --------------------------------------------------

print("\nAuthority by Source:")

for source_type in (
    "book",
    "website",
    "transcript",
):

    print(
        f"\n[{source_type}]"
    )

    for authority_class, count in (
        authority_by_source[
            source_type
        ].most_common()
    ):
        print(
            f"- {authority_class}: {count}"
        )


# --------------------------------------------------
# 7. Website authority validation
# --------------------------------------------------

website_authority_type_counter = Counter(
    record.get(
        "attribution",
        {},
    ).get(
        "content_authority_type"
    )
    for record in (
        source_records["website"]
    )
)


print(
    "\nWebsite existing content_authority_type:"
)

for authority_type, count in (
    website_authority_type_counter.items()
):
    print(
        f"- {authority_type}: {count}"
    )


# --------------------------------------------------
# 8. Transcript Primary / Context split
# --------------------------------------------------

transcript_authority_basis_counter = Counter()


for record in transcript_records:

    profile = (
        knowledge_authority_profile_v1[
            record[
                "document_id"
            ]
        ]
    )

    transcript_authority_basis_counter[
        profile[
            "authority_basis"
        ]
    ] += 1


print(
    "\nTranscript authority basis:"
)

for basis, count in (
    transcript_authority_basis_counter.most_common()
):
    print(
        f"- {basis}: {count}"
    )


# --------------------------------------------------
# 9. Core Knowledge Corpus Candidate
#
# No file is written.
# --------------------------------------------------

core_primary_document_ids_v1 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v1.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "CORE_PRIMARY_KNOWLEDGE"
    )
}


supervised_secondary_document_ids_v1 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v1.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "SUPERVISED_SECONDARY_KNOWLEDGE"
    )
}


contextual_document_ids_v1 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v1.items()
    )
    if (
        profile[
            "authority_class"
        ]
        in {
            "CONTEXTUAL_NONPRIMARY",
            "CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED",
        }
    )
}


unresolved_context_document_ids_v1 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v1.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "UNRESOLVED_CONTEXT"
    )
}


print(
    "\nKnowledge-layer populations:"
)

print(
    "- Core Primary Knowledge:",
    len(
        core_primary_document_ids_v1
    ),
)

print(
    "- Supervised Secondary Knowledge:",
    len(
        supervised_secondary_document_ids_v1
    ),
)

print(
    "- Contextual Non-primary:",
    len(
        contextual_document_ids_v1
    ),
)

print(
    "- Unresolved Context:",
    len(
        unresolved_context_document_ids_v1
    ),
)


# --------------------------------------------------
# 10. Core Taxonomy Evidence by Source
# --------------------------------------------------

core_primary_by_source = Counter(
    record_lookup[
        document_id
    ]["source_type"]
    for document_id in (
        core_primary_document_ids_v1
    )
)


print(
    "\nCore Primary Knowledge by Source:"
)

for source_type in (
    "book",
    "website",
    "transcript",
):
    print(
        f"- {source_type}: "
        f"{core_primary_by_source[source_type]}"
    )


# --------------------------------------------------
# 11. Candidate Topic evidence by Authority Layer
#
# Uses Mapping v2.
# This reveals whether Topic dispersion is
# coming from Primary or lower-authority layers.
# --------------------------------------------------

topic_authority_evidence_v1 = defaultdict(
    Counter
)


for document_id, mapping_profile in (
    document_topic_mapping_v2_candidate.items()
):

    authority_class = (
        knowledge_authority_profile_v1[
            document_id
        ][
            "authority_class"
        ]
    )

    observed_topic_ids = set()

    for candidate in mapping_profile[
        "candidates"
    ]:
        observed_topic_ids.add(
            candidate[
                "topic_id"
            ]
        )

    for scope in mapping_profile[
        "book_hierarchical_topic_scope"
    ]:
        observed_topic_ids.add(
            scope[
                "topic_id"
            ]
        )

    conversation_scope = mapping_profile[
        "conversation_topic_scope"
    ]

    if conversation_scope is not None:
        observed_topic_ids.add(
            conversation_scope[
                "topic_id"
            ]
        )

    for topic_id in observed_topic_ids:
        topic_authority_evidence_v1[
            topic_id
        ][
            authority_class
        ] += 1


print(
    "\nTopic Evidence by Knowledge Authority:"
)


for topic_id in topic_seed_v0:

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    counter = (
        topic_authority_evidence_v1[
            topic_id
        ]
    )

    print(
        "- Core Primary:",
        counter[
            "CORE_PRIMARY_KNOWLEDGE"
        ],
    )

    print(
        "- Supervised Secondary:",
        counter[
            "SUPERVISED_SECONDARY_KNOWLEDGE"
        ],
    )

    print(
        "- Contextual Non-primary:",
        (
            counter[
                "CONTEXTUAL_NONPRIMARY"
            ]
            + counter[
                "CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED"
            ]
        ),
    )

    print(
        "- Unresolved Context:",
        counter[
            "UNRESOLVED_CONTEXT"
        ],
    )


# --------------------------------------------------
# 12. Policy object for later NB10 stages
# --------------------------------------------------

knowledge_authority_policy_v1 = {
    "core_taxonomy_source": (
        "CORE_PRIMARY_KNOWLEDGE"
    ),

    "supporting_source": (
        "SUPERVISED_SECONDARY_KNOWLEDGE"
    ),

    "context_only_sources": [
        "CONTEXTUAL_NONPRIMARY",
        "CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED",
        "UNRESOLVED_CONTEXT",
    ],

    "secondary_can_override_primary": False,

    "context_can_define_core_taxonomy": False,

    "context_can_define_primary_claim": False,

    "retain_all_source_records": True,

    "speaker_inference_allowed": False,

    "primary_conflict_requires_review": True,
}


print(
    "\nKnowledge Authority Policy:"
)

for key, value in (
    knowledge_authority_policy_v1.items()
):
    print(
        f"- {key}: {value}"
    )


# --------------------------------------------------
# 13. Independent validation
# --------------------------------------------------

classified_document_ids = set(
    knowledge_authority_profile_v1
)

expected_document_ids = {
    record[
        "document_id"
    ]
    for record in records
}


missing_authority_documents = (
    expected_document_ids
    - classified_document_ids
)

unexpected_authority_documents = (
    classified_document_ids
    - expected_document_ids
)


all_layer_ids = (
    core_primary_document_ids_v1
    | supervised_secondary_document_ids_v1
    | contextual_document_ids_v1
    | unresolved_context_document_ids_v1
)


layer_overlap_count = (
    len(
        core_primary_document_ids_v1
        & supervised_secondary_document_ids_v1
    )
    + len(
        core_primary_document_ids_v1
        & contextual_document_ids_v1
    )
    + len(
        core_primary_document_ids_v1
        & unresolved_context_document_ids_v1
    )
    + len(
        supervised_secondary_document_ids_v1
        & contextual_document_ids_v1
    )
    + len(
        supervised_secondary_document_ids_v1
        & unresolved_context_document_ids_v1
    )
    + len(
        contextual_document_ids_v1
        & unresolved_context_document_ids_v1
    )
)


attribution_mutation_performed = False
taxonomy_mutation_performed = False
document_deletion_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Expected documents:",
    len(
        expected_document_ids
    ),
)

print(
    "- Classified documents:",
    len(
        classified_document_ids
    ),
)

print(
    "- Missing Authority classifications:",
    len(
        missing_authority_documents
    ),
)

print(
    "- Unexpected Authority classifications:",
    len(
        unexpected_authority_documents
    ),
)

print(
    "- Documents represented in Authority layers:",
    len(
        all_layer_ids
    ),
)

print(
    "- Cross-layer overlaps:",
    layer_overlap_count,
)

print(
    "- Attribution mutation performed:",
    attribution_mutation_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Document deletion performed:",
    document_deletion_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    expected_document_ids
) == 19456

assert (
    classified_document_ids
    == expected_document_ids
)

assert (
    all_layer_ids
    == expected_document_ids
)

assert layer_overlap_count == 0

assert (
    knowledge_authority_policy_v1[
        "secondary_can_override_primary"
    ]
    is False
)

assert (
    knowledge_authority_policy_v1[
        "context_can_define_core_taxonomy"
    ]
    is False
)

assert (
    knowledge_authority_policy_v1[
        "retain_all_source_records"
    ]
    is True
)

assert attribution_mutation_performed is False
assert taxonomy_mutation_performed is False
assert document_deletion_performed is False
assert disk_write_performed is False


print("\nCODE CELL 30 GATE: PASS")
print(
    "Knowledge Authority Ranking v1 validated: "
    "Primary, supervised-secondary, contextual and "
    "unresolved evidence are separated without "
    "discarding any source record."
)

Knowledge Authority Ranking v1:
- Documents: 19456

Authority Tiers:
- Tier 1: 13987
- Tier 2: 453
- Tier 3: 5016

Authority Classes:
- CORE_PRIMARY_KNOWLEDGE: 13987
- CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED: 5016
- SUPERVISED_SECONDARY_KNOWLEDGE: 453

Authority by Source:

[book]
- CORE_PRIMARY_KNOWLEDGE: 2701

[website]
- SUPERVISED_SECONDARY_KNOWLEDGE: 453
- CORE_PRIMARY_KNOWLEDGE: 76

[transcript]
- CORE_PRIMARY_KNOWLEDGE: 11210
- CONTEXTUAL_NONPRIMARY_OR_UNRESOLVED: 5016

Website existing content_authority_type:
- editorial_supervision: 453
- direct_author: 76

Transcript authority basis:
- TRANSCRIPT_CANONICAL_PRIMARY_SPEAKER: 11210
- TRANSCRIPT_RAW_SPEAKER_PRESENT_WITHOUT_CANONICAL_PRIMARY_ATTRIBUTION: 5016

Knowledge-layer populations:
- Core Primary Knowledge: 13987
- Supervised Secondary Knowledge: 453
- Contextual Non-primary: 5016
- Unresolved Context: 0

Core Primary Knowledge by Source:
- book: 2701
- website: 76
- transcript: 11210

Topic Evidence by Knowledge Authority:

[م

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Transcript Authority Reconciliation — اتصال Ranking به Evidence پذیرفته‌شده NB09</h3>

<p>
Knowledge Authority Ranking در Cell 30 از نظر فنی PASS شد، اما طبقه‌بندی Transcript بیش از حد به فیلد <code>speaker</code> Canonical متکی بود.
</p>

<p>
در NB09، Attribution Transcript با Evidence دقیق‌تری بررسی شده و شش وضعیت متمایز به دست آمده بود:
</p>

<ul>
<li><code>PRIMARY_CANONICAL_ATTRIBUTION</code>: 11,209</li>
<li><code>PRIMARY_EXPLICIT_SOURCE_MARKER</code>: 718</li>
<li><code>PRIMARY_HUMAN_CONFIRMED_OVERLAY</code>: 1</li>
<li><code>SECONDARY_HUMAN_CONFIRMED_OVERLAY</code>: 1</li>
<li><code>NONPRIMARY_EXPLICIT_CONTEXT</code>: 1,943</li>
<li><code>UNRESOLVED_SPEAKER_CONTEXT</code>: 2,354</li>
</ul>

<p>
بنابراین 5,016 رکورد فاقد Canonical Speaker نباید به یک Authority Class واحد تبدیل شوند.
برخی از آن‌ها Primary Evidence صریح دارند، برخی Context غیرPrimary هستند و برخی واقعاً Unresolved باقی مانده‌اند.
</p>

<h4>هدف این مرحله</h4>

<ul>
<li>بررسی دقیق همان 5,016 Transcript بدون Canonical Speaker.</li>
<li>مشاهده فیلدهای واقعی Attribution و Source Metadata که NB09 بر اساس آن‌ها تصمیم گرفته است.</li>
<li>شناسایی Markerهای صریح Primary بدون استنتاج Speaker.</li>
<li>شناسایی الگوهای Non-primary و Unresolved.</li>
<li>آماده‌سازی Authority Ranking اصلاح‌شده بدون تغییر هیچ Attribution.</li>
</ul>

<h4>اصل مهم</h4>

<p>
در این Cell هیچ رکوردی دوباره طبقه‌بندی نمی‌شود.
فقط Evidence موجود Discovery می‌شود تا Ranking بعدی بر مبنای داده واقعی و سیاست پذیرفته‌شده NB09 ساخته شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>دقیقاً 5,016 رکورد فاقد Canonical Speaker بررسی شوند.</li>
<li>فیلدهای Attribution و Metadata بدون تغییر گزارش شوند.</li>
<li>Markerهای صریح Primary فقط از داده موجود استخراج شوند.</li>
<li>هیچ Speaker جدید استنتاج نشود.</li>
<li>هیچ Authority، Taxonomy یا Mapping تغییر نکند.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [74]:
# Code Cell 31

from collections import Counter
import json


# --------------------------------------------------
# 1. Exact population requiring reconciliation
# --------------------------------------------------

transcript_without_canonical_speaker = [
    record
    for record in transcript_records
    if (
        record.get(
            "attribution",
            {},
        ).get("speaker")
        is None
    )
]


print("Transcript Authority Reconciliation Discovery:")
print(
    "- Transcript records:",
    len(transcript_records),
)

print(
    "- Without Canonical Speaker:",
    len(
        transcript_without_canonical_speaker
    ),
)


# --------------------------------------------------
# 2. Fields that may carry existing attribution
#    evidence
# --------------------------------------------------

attribution_fields_to_inspect = [
    "attribution_status",
    "author",
    "author_id",
    "authorship_assertion_source",
    "byline_raw",
    "content_authority_type",
    "credited_author_name",
    "editorial_credit_raw",
    "editorial_supervisor_name",
    "knowledge_owner",
    "reference_expert_name",
    "speaker",
]

metadata_fields_to_inspect = [
    "speaker",
    "program",
    "program_id",
    "record_type",
    "processing_state",
    "source_identity",
    "text_state",
]


# --------------------------------------------------
# 3. Field-value distributions
# --------------------------------------------------

print("\nAttribution field distributions:")


attribution_field_counters = {}


for field in attribution_fields_to_inspect:

    counter = Counter()

    for record in (
        transcript_without_canonical_speaker
    ):

        value = record.get(
            "attribution",
            {},
        ).get(field)

        if nonempty(value):
            counter[
                str(value)
            ] += 1
        else:
            counter[
                "<null>"
            ] += 1

    attribution_field_counters[
        field
    ] = counter

    print(
        f"\n[{field}]"
    )

    for value, count in (
        counter.most_common(20)
    ):
        print(
            f"- {value}: {count}"
        )


print("\nSource metadata field distributions:")


metadata_field_counters = {}


for field in metadata_fields_to_inspect:

    counter = Counter()

    for record in (
        transcript_without_canonical_speaker
    ):

        value = record.get(
            "source_metadata",
            {},
        ).get(field)

        if isinstance(
            value,
            (dict, list),
        ):
            value = json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
            )

        if nonempty(value):
            counter[
                str(value)
            ] += 1
        else:
            counter[
                "<null>"
            ] += 1

    metadata_field_counters[
        field
    ] = counter

    print(
        f"\n[{field}]"
    )

    for value, count in (
        counter.most_common(25)
    ):
        print(
            f"- {value}: {count}"
        )


# --------------------------------------------------
# 4. Existing raw-speaker population
# --------------------------------------------------

raw_speaker_counter = Counter()


for record in (
    transcript_without_canonical_speaker
):

    raw_speaker = record.get(
        "source_metadata",
        {},
    ).get("speaker")

    if nonempty(raw_speaker):
        raw_speaker_counter[
            str(raw_speaker)
        ] += 1
    else:
        raw_speaker_counter[
            "<null>"
        ] += 1


print("\nRaw Speaker values:")

print(
    "- Distinct values:",
    len(
        raw_speaker_counter
    ),
)


for value, count in (
    raw_speaker_counter.most_common(50)
):
    print(
        f"- {value}: {count}"
    )


# --------------------------------------------------
# 5. Explicit Primary marker discovery
#
# This is NOT speaker inference.
# It only checks whether existing attribution fields
# explicitly contain the canonical Primary identity.
# --------------------------------------------------

primary_identity_marker_fields = [
    "author",
    "credited_author_name",
    "knowledge_owner",
    "reference_expert_name",
    "editorial_supervisor_name",
]


explicit_primary_marker_rows = []


for record in (
    transcript_without_canonical_speaker
):

    matched_fields = []

    attribution = record.get(
        "attribution",
        {},
    )

    for field in (
        primary_identity_marker_fields
    ):

        value = attribution.get(
            field
        )

        if (
            value
            == PRIMARY_CANONICAL_NAME
        ):
            matched_fields.append(
                field
            )

    if matched_fields:

        explicit_primary_marker_rows.append(
            {
                "document_id": (
                    record[
                        "document_id"
                    ]
                ),
                "matched_fields": (
                    matched_fields
                ),
                "attribution_status": (
                    attribution.get(
                        "attribution_status"
                    )
                ),
                "authorship_assertion_source": (
                    attribution.get(
                        "authorship_assertion_source"
                    )
                ),
                "raw_speaker": (
                    record.get(
                        "source_metadata",
                        {},
                    ).get(
                        "speaker"
                    )
                ),
                "program": (
                    transcript_program(
                        record
                    )
                ),
            }
        )


print(
    "\nExplicit Primary identity markers "
    "inside Canonical-Speaker-null records:"
)

print(
    "- Rows:",
    len(
        explicit_primary_marker_rows
    ),
)


primary_marker_field_counter = Counter(
    field
    for row in (
        explicit_primary_marker_rows
    )
    for field in row[
        "matched_fields"
    ]
)


for field, count in (
    primary_marker_field_counter.most_common()
):
    print(
        f"- {field}: {count}"
    )


# --------------------------------------------------
# 6. Attribution-status × assertion-source signatures
# --------------------------------------------------

status_assertion_signature_counter = Counter()


for record in (
    transcript_without_canonical_speaker
):

    attribution = record.get(
        "attribution",
        {},
    )

    signature = (
        str(
            attribution.get(
                "attribution_status"
            )
        ),
        str(
            attribution.get(
                "authorship_assertion_source"
            )
        ),
        str(
            attribution.get(
                "author"
            )
        ),
        str(
            attribution.get(
                "knowledge_owner"
            )
        ),
        str(
            record.get(
                "source_metadata",
                {},
            ).get(
                "speaker"
            )
        ),
    )

    status_assertion_signature_counter[
        signature
    ] += 1


print(
    "\nTop Attribution signatures "
    "(status, assertion source, author, "
    "knowledge owner, raw speaker):"
)


for signature, count in (
    status_assertion_signature_counter.most_common(
        40
    )
):

    print(
        "\n- count:",
        count,
    )

    print(
        "  attribution_status:",
        signature[0],
    )

    print(
        "  authorship_assertion_source:",
        signature[1],
    )

    print(
        "  author:",
        signature[2],
    )

    print(
        "  knowledge_owner:",
        signature[3],
    )

    print(
        "  raw_speaker:",
        signature[4],
    )


# --------------------------------------------------
# 7. Program-level composition
# --------------------------------------------------

program_null_canonical_counter = Counter(
    transcript_program(
        record
    )
    for record in (
        transcript_without_canonical_speaker
    )
)


print(
    "\nCanonical-Speaker-null records by Program:"
)


for program, count in (
    program_null_canonical_counter.most_common()
):
    print(
        f"- {program}: {count}"
    )


# --------------------------------------------------
# 8. Compare with accepted NB09 population totals
#
# These are accepted prior-stage reference counts,
# not newly inferred labels.
# --------------------------------------------------

nb09_transcript_authority_reference = {
    "PRIMARY_CANONICAL_ATTRIBUTION": 11209,
    "PRIMARY_EXPLICIT_SOURCE_MARKER": 718,
    "PRIMARY_HUMAN_CONFIRMED_OVERLAY": 1,
    "SECONDARY_HUMAN_CONFIRMED_OVERLAY": 1,
    "NONPRIMARY_EXPLICIT_CONTEXT": 1943,
    "UNRESOLVED_SPEAKER_CONTEXT": 2354,
}


print(
    "\nAccepted NB09 Transcript Authority reference:"
)


for status, count in (
    nb09_transcript_authority_reference.items()
):
    print(
        f"- {status}: {count}"
    )


print(
    "- Reference total:",
    sum(
        nb09_transcript_authority_reference.values()
    ),
)


# --------------------------------------------------
# 9. Validation
# --------------------------------------------------

speaker_inference_performed = False
authority_reclassification_performed = False
attribution_mutation_performed = False
taxonomy_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Expected Canonical-Speaker-null records:",
    5016,
)

print(
    "- Inspected Canonical-Speaker-null records:",
    len(
        transcript_without_canonical_speaker
    ),
)

print(
    "- Accepted NB09 Transcript total:",
    sum(
        nb09_transcript_authority_reference.values()
    ),
)

print(
    "- Speaker inference performed:",
    speaker_inference_performed,
)

print(
    "- Authority reclassification performed:",
    authority_reclassification_performed,
)

print(
    "- Attribution mutation performed:",
    attribution_mutation_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    transcript_without_canonical_speaker
) == 5016

assert sum(
    nb09_transcript_authority_reference.values()
) == 16226

assert speaker_inference_performed is False
assert authority_reclassification_performed is False
assert attribution_mutation_performed is False
assert taxonomy_mutation_performed is False
assert disk_write_performed is False


print("\nCODE CELL 31 GATE: PASS")
print(
    "Transcript authority evidence discovery "
    "completed successfully; the 5,016 records "
    "without Canonical Speaker remain unmodified "
    "and are ready for evidence-based reconciliation "
    "with accepted NB09 authority classes."
)

Transcript Authority Reconciliation Discovery:
- Transcript records: 16226
- Without Canonical Speaker: 5016

Attribution field distributions:

[attribution_status]
- <null>: 5016

[author]
- <null>: 5016

[author_id]
- <null>: 5016

[authorship_assertion_source]
- <null>: 5016

[byline_raw]
- <null>: 5016

[content_authority_type]
- <null>: 5016

[credited_author_name]
- <null>: 5016

[editorial_credit_raw]
- <null>: 5016

[editorial_supervisor_name]
- <null>: 5016

[knowledge_owner]
- <null>: 5016

[reference_expert_name]
- <null>: 5016

[speaker]
- <null>: 5016

Source metadata field distributions:

[speaker]
- {"base": {"boundary_type": "CHEHELSTON_UNLABELED_PARAGRAPH_UNRESOLVED", "canonical": null, "raw": null, "source_canonical": null, "status": "UNRESOLVED"}, "effective": {"basis": "BASE_PARSER_ATTRIBUTION", "canonical": null, "label": null, "primary_dr_roudi_corpus": null}, "human_attribution_overlay": null}: 1599
- {"base": {"boundary_type": "BOLD_ONLY_SPEAKER_BOUNDARY", "cano

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Knowledge Authority Ranking v2 — بازسازی دقیق Authority بر اساس NB09</h3>

<p>
Discovery در Cell 31 نشان داد که 5,016 Transcript فاقد Canonical Speaker یک گروه واحد نیستند.
Metadata داخلی Speaker شواهد کافی برای بازسازی همان Authority Classes پذیرفته‌شده NB09 دارد.
</p>

<p>
بنابراین Authority Ranking قبلی Cell 30 به‌عنوان نسخه اولیه و ساده‌شده در نظر گرفته می‌شود و در این مرحله با نسخه دقیق‌تر جایگزین خواهد شد.
</p>

<h4>Transcript Authority Classes پذیرفته‌شده</h4>

<ul>
<li><code>PRIMARY_CANONICAL_ATTRIBUTION</code></li>
<li><code>PRIMARY_EXPLICIT_SOURCE_MARKER</code></li>
<li><code>PRIMARY_HUMAN_CONFIRMED_OVERLAY</code></li>
<li><code>SECONDARY_HUMAN_CONFIRMED_OVERLAY</code></li>
<li><code>NONPRIMARY_EXPLICIT_CONTEXT</code></li>
<li><code>UNRESOLVED_SPEAKER_CONTEXT</code></li>
</ul>

<h4>Knowledge Authority Layers</h4>

<ul>
<li><strong>Tier 1 — CORE_PRIMARY_KNOWLEDGE:</strong> کتاب‌ها، 76 مقالۀ direct-author، و Transcriptهای Primary.</li>
<li><strong>Tier 2 — SUPERVISED_SECONDARY_KNOWLEDGE:</strong> 453 مقالۀ تحت نظارت سردبیری.</li>
<li><strong>Tier 3 — CONTEXTUAL_NONPRIMARY:</strong> مجری، مهمان، شرکت‌کننده و سایر Speakerهای غیرPrimary.</li>
<li><strong>Tier 4 — UNRESOLVED_CONTEXT:</strong> Speakerهایی که واقعاً حل نشده‌اند.</li>
</ul>

<p>
Tier 3 و Tier 4 در Corpus حفظ می‌شوند ولی حق تعریف Core Taxonomy یا Primary Claim دکتر رودی را ندارند.
</p>

<h4>اصل مهم</h4>

<p>
هیچ Speaker از Context حدس زده نمی‌شود.
طبقه‌بندی فقط از Canonical Attribution، Source Marker، Human Overlay و وضعیت Parser موجود استفاده می‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>شش کلاس Transcript دقیقاً با Countهای پذیرفته‌شده NB09 برابر باشند.</li>
<li>تمام 19,456 Document دقیقاً در یک Knowledge Layer قرار گیرند.</li>
<li>Primary Transcriptهای واقعی وارد Core شوند.</li>
<li>Non-primary و Unresolved از Core جدا بمانند.</li>
<li>هیچ Attribution یا Document تغییر نکند.</li>
<li>هیچ Disk Write انجام نشود.</li>
</ul>

</div>

In [75]:
# Code Cell 32

from collections import Counter, defaultdict
import json


# --------------------------------------------------
# 1. Exact explicit Primary source marker observed
#    in Cell 31
# --------------------------------------------------

PRIMARY_EXPLICIT_SOURCE_LABEL = (
    "دکتر کمیل رودی"
)


def transcript_speaker_metadata(record):
    value = record.get(
        "source_metadata",
        {},
    ).get("speaker")

    if isinstance(value, dict):
        return value

    return {}


# --------------------------------------------------
# 2. Discover Human Attribution Overlays
# --------------------------------------------------

human_overlay_records = []


for record in transcript_records:

    speaker_metadata = (
        transcript_speaker_metadata(
            record
        )
    )

    overlay = speaker_metadata.get(
        "human_attribution_overlay"
    )

    if overlay is not None:

        human_overlay_records.append(
            {
                "document_id": record[
                    "document_id"
                ],
                "canonical_speaker": (
                    record.get(
                        "attribution",
                        {},
                    ).get(
                        "speaker"
                    )
                ),
                "program": (
                    transcript_program(
                        record
                    )
                ),
                "overlay": overlay,
                "base": (
                    speaker_metadata.get(
                        "base"
                    )
                ),
                "effective": (
                    speaker_metadata.get(
                        "effective"
                    )
                ),
            }
        )


print(
    "Human Attribution Overlay discovery:"
)

print(
    "- Overlay records:",
    len(
        human_overlay_records
    ),
)


for row in human_overlay_records:

    print(
        "\n----------------------------------------"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Program:",
        row[
            "program"
        ],
    )

    print(
        "Canonical Speaker:",
        row[
            "canonical_speaker"
        ],
    )

    print(
        "Base:",
        json.dumps(
            row["base"],
            ensure_ascii=False,
            sort_keys=True,
        ),
    )

    print(
        "Effective:",
        json.dumps(
            row["effective"],
            ensure_ascii=False,
            sort_keys=True,
        ),
    )

    print(
        "Human Overlay:",
        json.dumps(
            row["overlay"],
            ensure_ascii=False,
            sort_keys=True,
        ),
    )


# --------------------------------------------------
# 3. Evidence-based Transcript Authority classifier
#
# No new speaker identity is inferred.
# --------------------------------------------------

def classify_transcript_authority_v2(
    record
):

    attribution = record.get(
        "attribution",
        {},
    )

    canonical_speaker = (
        attribution.get(
            "speaker"
        )
    )

    speaker_metadata = (
        transcript_speaker_metadata(
            record
        )
    )

    base = (
        speaker_metadata.get(
            "base"
        )
        or {}
    )

    effective = (
        speaker_metadata.get(
            "effective"
        )
        or {}
    )

    overlay = speaker_metadata.get(
        "human_attribution_overlay"
    )

    base_status = base.get(
        "status"
    )

    base_raw = base.get(
        "raw"
    )

    effective_label = effective.get(
        "label"
    )

    # ----------------------------------------------
    # Human-confirmed overlay has highest priority.
    # The resulting Primary/Secondary direction is
    # read from the already-materialized effective
    # canonical speaker state, not inferred.
    # ----------------------------------------------

    if overlay is not None:

        if (
            canonical_speaker
            == PRIMARY_CANONICAL_NAME
        ):
            return (
                "PRIMARY_HUMAN_CONFIRMED_OVERLAY"
            )

        return (
            "SECONDARY_HUMAN_CONFIRMED_OVERLAY"
        )

    # ----------------------------------------------
    # Existing canonical Primary attribution
    # ----------------------------------------------

    if (
        canonical_speaker
        == PRIMARY_CANONICAL_NAME
    ):
        return (
            "PRIMARY_CANONICAL_ATTRIBUTION"
        )

    # ----------------------------------------------
    # Any explicit canonical non-primary speaker
    # would be contextual.
    # ----------------------------------------------

    if nonempty(
        canonical_speaker
    ):
        return (
            "NONPRIMARY_EXPLICIT_CONTEXT"
        )

    # ----------------------------------------------
    # Explicit source marker for Dr. Roudi
    # without canonical mapping
    # ----------------------------------------------

    if (
        base_status
        == "EXPLICIT_SOURCE_MARKER"
        and (
            base_raw
            == PRIMARY_EXPLICIT_SOURCE_LABEL
            or effective_label
            == PRIMARY_EXPLICIT_SOURCE_LABEL
        )
    ):
        return (
            "PRIMARY_EXPLICIT_SOURCE_MARKER"
        )

    # ----------------------------------------------
    # Explicit non-primary source labels or
    # registered narrative cues
    # ----------------------------------------------

    if (
        base_status
        in {
            "EXPLICIT_SOURCE_MARKER",
            "NARRATIVE_CUE",
        }
    ):
        return (
            "NONPRIMARY_EXPLICIT_CONTEXT"
        )

    # ----------------------------------------------
    # Parser explicitly preserved unresolved state
    # ----------------------------------------------

    if (
        base_status
        == "UNRESOLVED"
    ):
        return (
            "UNRESOLVED_SPEAKER_CONTEXT"
        )

    # ----------------------------------------------
    # Anything outside the accepted evidence model
    # remains visible as an invalid/unclassified row.
    # ----------------------------------------------

    return "UNCLASSIFIED_TRANSCRIPT_AUTHORITY"


# --------------------------------------------------
# 4. Apply to all Transcript records
# --------------------------------------------------

transcript_authority_class_v2 = {}


for record in transcript_records:

    transcript_authority_class_v2[
        record[
            "document_id"
        ]
    ] = (
        classify_transcript_authority_v2(
            record
        )
    )


transcript_authority_counter_v2 = Counter(
    transcript_authority_class_v2.values()
)


print(
    "\nTranscript Authority Classes v2:"
)


for authority_class, count in (
    transcript_authority_counter_v2.items()
):
    print(
        f"- {authority_class}: {count}"
    )


# --------------------------------------------------
# 5. Compare EXACTLY against accepted NB09 counts
# --------------------------------------------------

print(
    "\nNB09 reconciliation:"
)


nb09_reconciliation_rows = []


for authority_class, expected_count in (
    nb09_transcript_authority_reference.items()
):

    actual_count = (
        transcript_authority_counter_v2[
            authority_class
        ]
    )

    delta = (
        actual_count
        - expected_count
    )

    nb09_reconciliation_rows.append(
        {
            "authority_class": (
                authority_class
            ),
            "expected": (
                expected_count
            ),
            "actual": (
                actual_count
            ),
            "delta": delta,
        }
    )

    print(
        f"- {authority_class}: "
        f"expected={expected_count} "
        f"| actual={actual_count} "
        f"| delta={delta}"
    )


unclassified_transcript_ids = [
    document_id
    for document_id, authority_class in (
        transcript_authority_class_v2.items()
    )
    if (
        authority_class
        == "UNCLASSIFIED_TRANSCRIPT_AUTHORITY"
    )
]


print(
    "- Unclassified Transcript rows:",
    len(
        unclassified_transcript_ids
    ),
)


# --------------------------------------------------
# 6. Build corrected Knowledge Authority Profile v2
# --------------------------------------------------

PRIMARY_TRANSCRIPT_CLASSES = {
    "PRIMARY_CANONICAL_ATTRIBUTION",
    "PRIMARY_EXPLICIT_SOURCE_MARKER",
    "PRIMARY_HUMAN_CONFIRMED_OVERLAY",
}

CONTEXTUAL_TRANSCRIPT_CLASSES = {
    "SECONDARY_HUMAN_CONFIRMED_OVERLAY",
    "NONPRIMARY_EXPLICIT_CONTEXT",
}

UNRESOLVED_TRANSCRIPT_CLASSES = {
    "UNRESOLVED_SPEAKER_CONTEXT",
}


knowledge_authority_profile_v2 = {}


for record in records:

    document_id = record[
        "document_id"
    ]

    source_type = record[
        "source_type"
    ]

    # ----------------------------------------------
    # Book
    # ----------------------------------------------

    if source_type == "book":

        profile = {
            "authority_tier": 1,
            "authority_class": (
                "CORE_PRIMARY_KNOWLEDGE"
            ),
            "authority_basis": (
                "PRIMARY_BOOK_AUTHORSHIP"
            ),
            "core_taxonomy_evidence": True,
            "may_support_core": True,
        }

    # ----------------------------------------------
    # Website
    # ----------------------------------------------

    elif source_type == "website":

        content_authority_type = (
            record.get(
                "attribution",
                {},
            ).get(
                "content_authority_type"
            )
        )

        if (
            content_authority_type
            == "direct_author"
        ):

            profile = {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_DIRECT_AUTHOR"
                ),
                "core_taxonomy_evidence": True,
                "may_support_core": True,
            }

        elif (
            content_authority_type
            == "editorial_supervision"
        ):

            profile = {
                "authority_tier": 2,
                "authority_class": (
                    "SUPERVISED_SECONDARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_EDITORIAL_SUPERVISION"
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": True,
            }

        else:

            profile = {
                "authority_tier": 4,
                "authority_class": (
                    "UNRESOLVED_CONTEXT"
                ),
                "authority_basis": (
                    "WEBSITE_AUTHORITY_UNRESOLVED"
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

    # ----------------------------------------------
    # Transcript
    # ----------------------------------------------

    elif source_type == "transcript":

        transcript_class = (
            transcript_authority_class_v2[
                document_id
            ]
        )

        if (
            transcript_class
            in PRIMARY_TRANSCRIPT_CLASSES
        ):

            profile = {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    transcript_class
                ),
                "core_taxonomy_evidence": True,
                "may_support_core": True,
            }

        elif (
            transcript_class
            in CONTEXTUAL_TRANSCRIPT_CLASSES
        ):

            profile = {
                "authority_tier": 3,
                "authority_class": (
                    "CONTEXTUAL_NONPRIMARY"
                ),
                "authority_basis": (
                    transcript_class
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

        elif (
            transcript_class
            in UNRESOLVED_TRANSCRIPT_CLASSES
        ):

            profile = {
                "authority_tier": 4,
                "authority_class": (
                    "UNRESOLVED_CONTEXT"
                ),
                "authority_basis": (
                    transcript_class
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

        else:

            profile = {
                "authority_tier": 4,
                "authority_class": (
                    "UNRESOLVED_CONTEXT"
                ),
                "authority_basis": (
                    transcript_class
                ),
                "core_taxonomy_evidence": False,
                "may_support_core": False,
            }

    else:

        raise ValueError(
            f"Unexpected source_type: "
            f"{source_type}"
        )

    knowledge_authority_profile_v2[
        document_id
    ] = profile


# --------------------------------------------------
# 7. Corrected Authority distribution
# --------------------------------------------------

authority_class_counter_v2 = Counter()
authority_tier_counter_v2 = Counter()

authority_by_source_v2 = defaultdict(
    Counter
)


for document_id, profile in (
    knowledge_authority_profile_v2.items()
):

    source_type = (
        record_lookup[
            document_id
        ]["source_type"]
    )

    authority_class_counter_v2[
        profile[
            "authority_class"
        ]
    ] += 1

    authority_tier_counter_v2[
        profile[
            "authority_tier"
        ]
    ] += 1

    authority_by_source_v2[
        source_type
    ][
        profile[
            "authority_class"
        ]
    ] += 1


print(
    "\nCorrected Knowledge Authority Ranking v2:"
)

print(
    "- Documents:",
    len(
        knowledge_authority_profile_v2
    ),
)


print("\nAuthority Tiers:")

for tier in sorted(
    authority_tier_counter_v2
):
    print(
        f"- Tier {tier}: "
        f"{authority_tier_counter_v2[tier]}"
    )


print("\nAuthority Classes:")

for authority_class, count in (
    authority_class_counter_v2.most_common()
):
    print(
        f"- {authority_class}: {count}"
    )


print("\nAuthority by Source:")

for source_type in (
    "book",
    "website",
    "transcript",
):

    print(
        f"\n[{source_type}]"
    )

    for authority_class, count in (
        authority_by_source_v2[
            source_type
        ].most_common()
    ):
        print(
            f"- {authority_class}: {count}"
        )


# --------------------------------------------------
# 8. Correct Core Primary population
# --------------------------------------------------

core_primary_document_ids_v2 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v2.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "CORE_PRIMARY_KNOWLEDGE"
    )
}


supervised_secondary_document_ids_v2 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v2.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "SUPERVISED_SECONDARY_KNOWLEDGE"
    )
}


contextual_nonprimary_document_ids_v2 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v2.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "CONTEXTUAL_NONPRIMARY"
    )
}


unresolved_context_document_ids_v2 = {
    document_id
    for document_id, profile in (
        knowledge_authority_profile_v2.items()
    )
    if (
        profile[
            "authority_class"
        ]
        == "UNRESOLVED_CONTEXT"
    )
}


print(
    "\nCorrected Knowledge-layer populations:"
)

print(
    "- Core Primary Knowledge:",
    len(
        core_primary_document_ids_v2
    ),
)

print(
    "- Supervised Secondary Knowledge:",
    len(
        supervised_secondary_document_ids_v2
    ),
)

print(
    "- Contextual Non-primary:",
    len(
        contextual_nonprimary_document_ids_v2
    ),
)

print(
    "- Unresolved Context:",
    len(
        unresolved_context_document_ids_v2
    ),
)


core_primary_by_source_v2 = Counter(
    record_lookup[
        document_id
    ]["source_type"]
    for document_id in (
        core_primary_document_ids_v2
    )
)


print(
    "\nCore Primary Knowledge by Source:"
)

for source_type in (
    "book",
    "website",
    "transcript",
):
    print(
        f"- {source_type}: "
        f"{core_primary_by_source_v2[source_type]}"
    )


# --------------------------------------------------
# 9. Recompute Topic Evidence by corrected Authority
# --------------------------------------------------

topic_authority_evidence_v2 = defaultdict(
    Counter
)


for document_id, mapping_profile in (
    document_topic_mapping_v2_candidate.items()
):

    authority_class = (
        knowledge_authority_profile_v2[
            document_id
        ][
            "authority_class"
        ]
    )

    observed_topic_ids = set()

    for candidate in mapping_profile[
        "candidates"
    ]:
        observed_topic_ids.add(
            candidate[
                "topic_id"
            ]
        )

    for scope in mapping_profile[
        "book_hierarchical_topic_scope"
    ]:
        observed_topic_ids.add(
            scope[
                "topic_id"
            ]
        )

    conversation_scope = mapping_profile[
        "conversation_topic_scope"
    ]

    if conversation_scope is not None:
        observed_topic_ids.add(
            conversation_scope[
                "topic_id"
            ]
        )

    for topic_id in observed_topic_ids:

        topic_authority_evidence_v2[
            topic_id
        ][
            authority_class
        ] += 1


print(
    "\nTopic Evidence by corrected Knowledge Authority:"
)


for topic_id in topic_seed_v0:

    counter = (
        topic_authority_evidence_v2[
            topic_id
        ]
    )

    print(
        f"\n[{topic_seed_v0[topic_id]['label_fa']}]"
    )

    print(
        "- Core Primary:",
        counter[
            "CORE_PRIMARY_KNOWLEDGE"
        ],
    )

    print(
        "- Supervised Secondary:",
        counter[
            "SUPERVISED_SECONDARY_KNOWLEDGE"
        ],
    )

    print(
        "- Contextual Non-primary:",
        counter[
            "CONTEXTUAL_NONPRIMARY"
        ],
    )

    print(
        "- Unresolved Context:",
        counter[
            "UNRESOLVED_CONTEXT"
        ],
    )


# --------------------------------------------------
# 10. Final Authority Policy v2
# --------------------------------------------------

knowledge_authority_policy_v2 = {
    "core_taxonomy_source": (
        "CORE_PRIMARY_KNOWLEDGE"
    ),

    "supporting_source": (
        "SUPERVISED_SECONDARY_KNOWLEDGE"
    ),

    "contextual_source": (
        "CONTEXTUAL_NONPRIMARY"
    ),

    "unresolved_source": (
        "UNRESOLVED_CONTEXT"
    ),

    "secondary_can_override_primary": False,

    "context_can_define_core_taxonomy": False,

    "unresolved_can_define_core_taxonomy": False,

    "context_can_define_primary_claim": False,

    "unresolved_can_define_primary_claim": False,

    "retain_all_source_records": True,

    "speaker_inference_allowed": False,

    "primary_conflict_requires_review": True,
}


# --------------------------------------------------
# 11. Independent validation
# --------------------------------------------------

nb09_count_mismatches = [
    row
    for row in (
        nb09_reconciliation_rows
    )
    if row["delta"] != 0
]


all_v2_layer_ids = (
    core_primary_document_ids_v2
    | supervised_secondary_document_ids_v2
    | contextual_nonprimary_document_ids_v2
    | unresolved_context_document_ids_v2
)


v2_layer_overlap_count = (
    len(
        core_primary_document_ids_v2
        & supervised_secondary_document_ids_v2
    )
    + len(
        core_primary_document_ids_v2
        & contextual_nonprimary_document_ids_v2
    )
    + len(
        core_primary_document_ids_v2
        & unresolved_context_document_ids_v2
    )
    + len(
        supervised_secondary_document_ids_v2
        & contextual_nonprimary_document_ids_v2
    )
    + len(
        supervised_secondary_document_ids_v2
        & unresolved_context_document_ids_v2
    )
    + len(
        contextual_nonprimary_document_ids_v2
        & unresolved_context_document_ids_v2
    )
)


expected_document_ids = {
    record[
        "document_id"
    ]
    for record in records
}


authority_mutation_performed = False
attribution_mutation_performed = False
taxonomy_mutation_performed = False
document_deletion_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- NB09 Authority count mismatches:",
    len(
        nb09_count_mismatches
    ),
)

print(
    "- Unclassified Transcript rows:",
    len(
        unclassified_transcript_ids
    ),
)

print(
    "- Expected documents:",
    len(
        expected_document_ids
    ),
)

print(
    "- Authority v2 documents:",
    len(
        knowledge_authority_profile_v2
    ),
)

print(
    "- Documents represented in v2 layers:",
    len(
        all_v2_layer_ids
    ),
)

print(
    "- Cross-layer overlaps:",
    v2_layer_overlap_count,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Attribution mutation performed:",
    attribution_mutation_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Document deletion performed:",
    document_deletion_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert not nb09_count_mismatches

assert not unclassified_transcript_ids

assert len(
    knowledge_authority_profile_v2
) == 19456

assert (
    all_v2_layer_ids
    == expected_document_ids
)

assert v2_layer_overlap_count == 0

assert (
    knowledge_authority_policy_v2[
        "secondary_can_override_primary"
    ]
    is False
)

assert (
    knowledge_authority_policy_v2[
        "context_can_define_core_taxonomy"
    ]
    is False
)

assert (
    knowledge_authority_policy_v2[
        "unresolved_can_define_core_taxonomy"
    ]
    is False
)

assert authority_mutation_performed is False
assert attribution_mutation_performed is False
assert taxonomy_mutation_performed is False
assert document_deletion_performed is False
assert disk_write_performed is False


print("\nCODE CELL 32 GATE: PASS")
print(
    "Knowledge Authority Ranking v2 reconciled "
    "exactly with accepted NB09 Transcript authority "
    "classes; Core Primary, supervised-secondary, "
    "contextual and unresolved layers are now "
    "separated for downstream Taxonomy analysis."
)

Human Attribution Overlay discovery:
- Overlay records: 2

----------------------------------------
Document: program:chehelston:episode-006:utterance:00021
Program: چهلستون
Canonical Speaker: Dr. Komeil Roudi
Base: {"boundary_type": "CHEHELSTON_UNLABELED_PARAGRAPH_UNRESOLVED", "canonical": null, "raw": null, "source_canonical": null, "status": "UNRESOLVED"}
Effective: {"basis": "HUMAN_ATTRIBUTION_OVERLAY", "canonical": "Dr. Komeil Roudi", "label": "دکتر کمیل رودی", "primary_dr_roudi_corpus": true}
Human Overlay: {"base_speaker_status": "UNRESOLVED", "decision": "HUMAN_CONFIRMED", "human_attribution_id": "human-attribution:0002", "human_confirmation_basis": "DIRECT_HUMAN_REVIEW_OF_QUESTION_AND_RESPONSE_CONTEXT", "human_confirmed_speaker": "دکتر کمیل رودی", "human_confirmed_speaker_canonical": "Dr. Komeil Roudi", "primary_dr_roudi_corpus": true, "relative_path": "data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 6.md", "source_line_end": 61, "source_line_start": 61, "

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Core Primary Concept Audit — بازسازی ساختار مفاهیم فقط از دانش اصلی دکتر رودی</h3>

<p>
پس از Authority Ranking v2، اکنون می‌توان ۱۰ Concept باز را بدون آلودگی ناشی از محتوای Secondary، مهمان، مجری، شرکت‌کننده یا Speaker نامشخص بررسی کرد.
</p>

<p>
منابع Core Primary در این مرحله عبارت‌اند از:
</p>

<ul>
<li>2,701 رکورد Book</li>
<li>76 مقالۀ Website با <code>direct_author</code></li>
<li>11,928 رکورد Transcript که در NB09 به‌صورت Primary تأیید شده‌اند</li>
</ul>

<p>
محتوای Website تحت نظارت، Transcriptهای Non-primary و Transcriptهای Unresolved حذف نمی‌شوند؛
اما فقط برای مقایسه گزارش می‌شوند و حق تعریف Core Taxonomy را ندارند.
</p>

<h4>هدف</h4>

<ul>
<li>اندازه‌گیری Evidence هر Concept در Core Primary.</li>
<li>مقایسه آن با Secondary و Contextual Evidence.</li>
<li>استفاده از ساختار صریح Book به‌عنوان قوی‌ترین Structural Evidence.</li>
<li>بررسی 76 مقالۀ مستقیم دکتر به‌صورت جدا از 453 مقالۀ تحت نظارت.</li>
<li>بررسی Transcript فقط در Segmentهای Primary واقعی.</li>
<li>تشخیص اینکه پراکندگی هر Concept واقعاً Primary است یا عمدتاً از لایه‌های پایین‌تر آمده است.</li>
</ul>

<h4>اصل مهم</h4>

<p>
زیاد بودن Mention در کل Corpus به‌تنهایی ارزش Taxonomy ندارد.
ساختار صریح کتاب و Evidence مستقیم Primary از تعداد Mention خام مهم‌تر است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 14,705 رکورد Core Primary در Audit قابل استفاده باشند.</li>
<li>76 مقالۀ مستقیم و 453 مقالۀ تحت نظارت جدا بمانند.</li>
<li>Transcript Primary، Non-primary و Unresolved جدا گزارش شوند.</li>
<li>ساختار Book استخراج‌شده در Cell 28 حفظ شود.</li>
<li>برای هر ۱۰ Concept یک Primary Evidence Profile ساخته شود.</li>
<li>هیچ Taxonomy Mutation یا Disk Write انجام نشود.</li>
</ul>

</div>

In [76]:
# Code Cell 33

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Validate corrected authority populations
# --------------------------------------------------

assert len(
    core_primary_document_ids_v2
) == 14705

assert len(
    supervised_secondary_document_ids_v2
) == 453

assert len(
    contextual_nonprimary_document_ids_v2
) == 1944

assert len(
    unresolved_context_document_ids_v2
) == 2354


website_direct_records_v1 = [
    record
    for record in source_records[
        "website"
    ]
    if (
        record.get(
            "attribution",
            {},
        ).get(
            "content_authority_type"
        )
        == "direct_author"
    )
]


website_supervised_records_v1 = [
    record
    for record in source_records[
        "website"
    ]
    if (
        record.get(
            "attribution",
            {},
        ).get(
            "content_authority_type"
        )
        == "editorial_supervision"
    )
]


primary_transcript_records_v2 = [
    record
    for record in transcript_records
    if (
        record[
            "document_id"
        ]
        in core_primary_document_ids_v2
    )
]


contextual_transcript_records_v2 = [
    record
    for record in transcript_records
    if (
        record[
            "document_id"
        ]
        in contextual_nonprimary_document_ids_v2
    )
]


unresolved_transcript_records_v2 = [
    record
    for record in transcript_records
    if (
        record[
            "document_id"
        ]
        in unresolved_context_document_ids_v2
    )
]


print("Core Primary Concept Audit:")
print(
    "- Core Primary documents:",
    len(
        core_primary_document_ids_v2
    ),
)

print(
    "- Primary Book:",
    len(book_records),
)

print(
    "- Primary Website direct-author:",
    len(
        website_direct_records_v1
    ),
)

print(
    "- Supervised Website:",
    len(
        website_supervised_records_v1
    ),
)

print(
    "- Primary Transcript:",
    len(
        primary_transcript_records_v2
    ),
)

print(
    "- Contextual Non-primary Transcript:",
    len(
        contextual_transcript_records_v2
    ),
)

print(
    "- Unresolved Transcript:",
    len(
        unresolved_transcript_records_v2
    ),
)


# --------------------------------------------------
# 2. Normalize text once
# --------------------------------------------------

normalized_text_by_document_v1 = {
    record[
        "document_id"
    ]: normalize_fa_for_match(
        record.get(
            "text",
            ""
        )
    )
    for record in records
}


normalized_title_by_document_v1 = {
    record[
        "document_id"
    ]: normalize_fa_for_match(
        record.get(
            "title",
            ""
        )
    )
    for record in records
}


# --------------------------------------------------
# 3. Website tag-like metadata extraction
#
# Discovery only.
# We read tag/category-like metadata fields
# without assuming a fixed schema.
# --------------------------------------------------

def flatten_metadata_values(value):

    values = []

    if value is None:
        return values

    if isinstance(
        value,
        str,
    ):
        if nonempty(value):
            values.append(value)

    elif isinstance(
        value,
        (list, tuple, set),
    ):
        for item in value:
            values.extend(
                flatten_metadata_values(
                    item
                )
            )

    elif isinstance(
        value,
        dict,
    ):
        for item in value.values():
            values.extend(
                flatten_metadata_values(
                    item
                )
            )

    return values


def website_tag_like_values(record):

    metadata = record.get(
        "source_metadata",
        {},
    )

    values = []

    for key, value in metadata.items():

        normalized_key = (
            str(key)
            .strip()
            .lower()
        )

        if (
            "tag" in normalized_key
            or "categor" in normalized_key
        ):
            values.extend(
                flatten_metadata_values(
                    value
                )
            )

    return sorted(
        {
            str(value).strip()
            for value in values
            if nonempty(value)
        }
    )


website_direct_tag_inventory_v1 = Counter()


for record in website_direct_records_v1:

    for value in website_tag_like_values(
        record
    ):
        website_direct_tag_inventory_v1[
            normalize_fa_for_match(
                value
            )
        ] += 1


print(
    "\nDirect-author Website tag-like inventory:"
)

print(
    "- Distinct normalized values:",
    len(
        website_direct_tag_inventory_v1
    ),
)


# --------------------------------------------------
# 4. Authority-layer Concept evidence
# --------------------------------------------------

core_primary_concept_audit_v1 = {}


for concept in review_concepts:

    normalized_concept = (
        normalize_fa_for_match(
            concept
        )
    )

    # ----------------------------------------------
    # Generic mention populations
    # ----------------------------------------------

    mention_ids_by_authority = defaultdict(
        list
    )

    mention_ids_by_source = defaultdict(
        list
    )

    for record in records:

        document_id = record[
            "document_id"
        ]

        normalized_text = (
            normalized_text_by_document_v1[
                document_id
            ]
        )

        normalized_title = (
            normalized_title_by_document_v1[
                document_id
            ]
        )

        has_mention = (
            normalized_concept
            in normalized_text
            or normalized_concept
            in normalized_title
        )

        if not has_mention:
            continue

        authority_class = (
            knowledge_authority_profile_v2[
                document_id
            ][
                "authority_class"
            ]
        )

        mention_ids_by_authority[
            authority_class
        ].append(
            document_id
        )

        mention_ids_by_source[
            record[
                "source_type"
            ]
        ].append(
            document_id
        )

    # ----------------------------------------------
    # Core Primary by source
    # ----------------------------------------------

    core_book_ids = [
        document_id
        for document_id in (
            mention_ids_by_source[
                "book"
            ]
        )
        if (
            document_id
            in core_primary_document_ids_v2
        )
    ]

    core_website_ids = [
        document_id
        for document_id in (
            mention_ids_by_source[
                "website"
            ]
        )
        if (
            document_id
            in core_primary_document_ids_v2
        )
    ]

    core_transcript_ids = [
        document_id
        for document_id in (
            mention_ids_by_source[
                "transcript"
            ]
        )
        if (
            document_id
            in core_primary_document_ids_v2
        )
    ]

    # ----------------------------------------------
    # Direct-author Website detail
    # ----------------------------------------------

    website_exact_tag_ids = []
    website_title_ids = []

    for record in website_direct_records_v1:

        document_id = record[
            "document_id"
        ]

        if (
            normalized_concept
            in normalized_title_by_document_v1[
                document_id
            ]
        ):
            website_title_ids.append(
                document_id
            )

        normalized_tags = {
            normalize_fa_for_match(
                tag
            )
            for tag in (
                website_tag_like_values(
                    record
                )
            )
        }

        if (
            normalized_concept
            in normalized_tags
        ):
            website_exact_tag_ids.append(
                document_id
            )

    # ----------------------------------------------
    # Primary Transcript detail
    # ----------------------------------------------

    primary_transcript_class_counter = Counter()
    primary_transcript_program_counter = Counter()

    for document_id in core_transcript_ids:

        transcript_class = (
            transcript_authority_class_v2[
                document_id
            ]
        )

        primary_transcript_class_counter[
            transcript_class
        ] += 1

        primary_transcript_program_counter[
            transcript_program(
                record_lookup[
                    document_id
                ]
            )
        ] += 1

    # ----------------------------------------------
    # Book structural evidence from Cell 28
    # ----------------------------------------------

    book_structure = (
        deep_book_concept_structure_v1[
            concept
        ]
    )

    structural_stability = (
        book_structural_stability_v1[
            concept
        ]
    )

    # ----------------------------------------------
    # Secondary / Context comparison
    # ----------------------------------------------

    secondary_ids = list(
        mention_ids_by_authority[
            "SUPERVISED_SECONDARY_KNOWLEDGE"
        ]
    )

    contextual_ids = list(
        mention_ids_by_authority[
            "CONTEXTUAL_NONPRIMARY"
        ]
    )

    unresolved_ids = list(
        mention_ids_by_authority[
            "UNRESOLVED_CONTEXT"
        ]
    )

    core_ids = list(
        mention_ids_by_authority[
            "CORE_PRIMARY_KNOWLEDGE"
        ]
    )

    total_mentions = (
        len(core_ids)
        + len(secondary_ids)
        + len(contextual_ids)
        + len(unresolved_ids)
    )

    if total_mentions:
        core_share = (
            len(core_ids)
            / total_mentions
        )
    else:
        core_share = None

    # ----------------------------------------------
    # Descriptive evidence condition
    #
    # No Taxonomy decision is made here.
    # ----------------------------------------------

    if (
        structural_stability[
            "structural_status"
        ]
        == "STABLE_EXPLICIT_PARENT_EVIDENCE"
    ):
        evidence_condition = (
            "PRIMARY_BOOK_STRUCTURAL_ANCHOR"
        )

    elif (
        book_structure[
            "exact_heading_count"
        ] > 0
    ):
        evidence_condition = (
            "PRIMARY_BOOK_EXPLICIT_HEADING"
        )

    elif (
        len(core_ids) > 0
    ):
        evidence_condition = (
            "PRIMARY_EVIDENCE_PRESENT_"
            "WITHOUT_EXPLICIT_BOOK_PARENT"
        )

    else:
        evidence_condition = (
            "NO_CORE_PRIMARY_EVIDENCE"
        )

    core_primary_concept_audit_v1[
        concept
    ] = {
        "normalized_concept": (
            normalized_concept
        ),

        "core_primary_mentions": (
            len(core_ids)
        ),

        "core_primary_book_mentions": (
            len(core_book_ids)
        ),

        "core_primary_website_mentions": (
            len(core_website_ids)
        ),

        "core_primary_transcript_mentions": (
            len(core_transcript_ids)
        ),

        "supervised_secondary_mentions": (
            len(secondary_ids)
        ),

        "contextual_nonprimary_mentions": (
            len(contextual_ids)
        ),

        "unresolved_mentions": (
            len(unresolved_ids)
        ),

        "total_mentions": (
            total_mentions
        ),

        "core_share": (
            core_share
        ),

        "book_exact_heading_count": (
            book_structure[
                "exact_heading_count"
            ]
        ),

        "book_structural_status": (
            structural_stability[
                "structural_status"
            ]
        ),

        "book_dominant_parent": (
            structural_stability[
                "dominant_parent"
            ]
        ),

        "book_parent_support_ratio": (
            structural_stability[
                "parent_support_ratio"
            ]
        ),

        "website_direct_exact_tag_documents": (
            website_exact_tag_ids
        ),

        "website_direct_title_documents": (
            website_title_ids
        ),

        "primary_transcript_class_counts": dict(
            primary_transcript_class_counter
        ),

        "primary_transcript_program_counts": dict(
            primary_transcript_program_counter
        ),

        "evidence_condition": (
            evidence_condition
        ),

        "core_primary_document_ids": (
            core_ids
        ),
    }


# --------------------------------------------------
# 5. Print Primary-vs-lower-authority matrix
# --------------------------------------------------

print(
    "\n\nCore Primary Concept Evidence Matrix:"
)


for concept in review_concepts:

    row = (
        core_primary_concept_audit_v1[
            concept
        ]
    )

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "- Evidence condition:",
        row[
            "evidence_condition"
        ],
    )

    print(
        "- Core Primary mentions:",
        row[
            "core_primary_mentions"
        ],
    )

    print(
        "  - Book:",
        row[
            "core_primary_book_mentions"
        ],
    )

    print(
        "  - Website direct-author:",
        row[
            "core_primary_website_mentions"
        ],
    )

    print(
        "  - Transcript Primary:",
        row[
            "core_primary_transcript_mentions"
        ],
    )

    print(
        "- Supervised Secondary mentions:",
        row[
            "supervised_secondary_mentions"
        ],
    )

    print(
        "- Contextual Non-primary mentions:",
        row[
            "contextual_nonprimary_mentions"
        ],
    )

    print(
        "- Unresolved mentions:",
        row[
            "unresolved_mentions"
        ],
    )

    print(
        "- Core share of all matched documents:",
        (
            round(
                row[
                    "core_share"
                ],
                3,
            )
            if (
                row[
                    "core_share"
                ]
                is not None
            )
            else None
        ),
    )

    print(
        "- Book Exact Heading count:",
        row[
            "book_exact_heading_count"
        ],
    )

    print(
        "- Book Structural status:",
        row[
            "book_structural_status"
        ],
    )

    print(
        "- Book Dominant Parent:",
        row[
            "book_dominant_parent"
        ],
    )

    print(
        "- Book Parent support ratio:",
        (
            round(
                row[
                    "book_parent_support_ratio"
                ],
                3,
            )
            if (
                row[
                    "book_parent_support_ratio"
                ]
                is not None
            )
            else None
        ),
    )

    print(
        "- Direct Website Exact-tag docs:",
        len(
            row[
                "website_direct_exact_tag_documents"
            ]
        ),
    )

    print(
        "- Direct Website Title docs:",
        len(
            row[
                "website_direct_title_documents"
            ]
        ),
    )

    print(
        "- Primary Transcript attribution classes:",
        row[
            "primary_transcript_class_counts"
        ],
    )

    print(
        "Top Primary Transcript Programs:"
    )

    program_counter = Counter(
        row[
            "primary_transcript_program_counts"
        ]
    )

    if program_counter:
        for program, count in (
            program_counter.most_common(6)
        ):
            print(
                f"  - {program}: {count}"
            )
    else:
        print(
            "  - None"
        )


# --------------------------------------------------
# 6. Deterministic representative Primary samples
#    by source
# --------------------------------------------------

core_primary_concept_samples_v1 = {}


for concept in review_concepts:

    row = (
        core_primary_concept_audit_v1[
            concept
        ]
    )

    samples_by_source = {}

    for source_type in (
        "book",
        "website",
        "transcript",
    ):

        candidate_ids = [
            document_id
            for document_id in (
                row[
                    "core_primary_document_ids"
                ]
            )
            if (
                record_lookup[
                    document_id
                ][
                    "source_type"
                ]
                == source_type
            )
        ]

        ranked_ids = sorted(
            candidate_ids,
            key=lambda document_id: (
                deterministic_review_rank(
                    document_id,
                    (
                        "NB10-core-primary:"
                        f"{concept}:"
                        f"{source_type}:v1"
                    ),
                )
            ),
        )

        selected_ids = (
            ranked_ids[:2]
        )

        samples_by_source[
            source_type
        ] = [
            {
                "document_id": (
                    document_id
                ),
                "title": (
                    record_lookup[
                        document_id
                    ].get(
                        "title"
                    )
                ),
                "program": (
                    transcript_program(
                        record_lookup[
                            document_id
                        ]
                    )
                    if source_type
                    == "transcript"
                    else None
                ),
                "heading_path": (
                    get_book_heading_path(
                        record_lookup[
                            document_id
                        ]
                    )
                    if source_type
                    == "book"
                    else None
                ),
                "excerpt": compact_excerpt(
                    record_lookup[
                        document_id
                    ],
                    limit=520,
                ),
            }
            for document_id in (
                selected_ids
            )
        ]

    core_primary_concept_samples_v1[
        concept
    ] = samples_by_source


print(
    "\n\nRepresentative Core Primary Samples:"
)


for concept in review_concepts:

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    samples_by_source = (
        core_primary_concept_samples_v1[
            concept
        ]
    )

    for source_type in (
        "book",
        "website",
        "transcript",
    ):

        print(
            f"\n[{source_type}]"
        )

        samples = (
            samples_by_source[
                source_type
            ]
        )

        if not samples:
            print(
                "- No Core Primary sample"
            )
            continue

        for sample in samples:

            print(
                "\n- Document:",
                sample[
                    "document_id"
                ],
            )

            if sample[
                "heading_path"
            ]:
                print(
                    "  Heading path:",
                    sample[
                        "heading_path"
                    ],
                )

            if sample[
                "program"
            ]:
                print(
                    "  Program:",
                    sample[
                        "program"
                    ],
                )

            print(
                "  Excerpt:",
                sample[
                    "excerpt"
                ],
            )


# --------------------------------------------------
# 7. Identify where apparent dispersion is mostly
#    outside Core Primary
#
# Descriptive flag only.
# --------------------------------------------------

primary_vs_external_dispersion_v1 = {}


for concept, row in (
    core_primary_concept_audit_v1.items()
):

    lower_authority_mentions = (
        row[
            "supervised_secondary_mentions"
        ]
        + row[
            "contextual_nonprimary_mentions"
        ]
        + row[
            "unresolved_mentions"
        ]
    )

    core_mentions = (
        row[
            "core_primary_mentions"
        ]
    )

    if (
        lower_authority_mentions
        > core_mentions
    ):
        dispersion_status = (
            "LOWER_AUTHORITY_DOMINANT"
        )

    elif (
        lower_authority_mentions > 0
        and core_mentions > 0
    ):
        dispersion_status = (
            "PRIMARY_DOMINANT_WITH_EXTERNAL_DISPERSION"
        )

    elif core_mentions > 0:
        dispersion_status = (
            "PRIMARY_ONLY_OR_NEAR_PRIMARY_ONLY"
        )

    else:
        dispersion_status = (
            "NO_PRIMARY_EVIDENCE"
        )

    primary_vs_external_dispersion_v1[
        concept
    ] = {
        "core_mentions": (
            core_mentions
        ),
        "lower_authority_mentions": (
            lower_authority_mentions
        ),
        "dispersion_status": (
            dispersion_status
        ),
    }


print(
    "\n\nPrimary vs External Dispersion:"
)


for concept in review_concepts:

    row = (
        primary_vs_external_dispersion_v1[
            concept
        ]
    )

    print(
        "-",
        concept,
        "| core=",
        row[
            "core_mentions"
        ],
        "| lower_authority=",
        row[
            "lower_authority_mentions"
        ],
        "|",
        row[
            "dispersion_status"
        ],
    )


# --------------------------------------------------
# 8. Validation
# --------------------------------------------------

concepts_without_core_audit = [
    concept
    for concept in review_concepts
    if concept not in (
        core_primary_concept_audit_v1
    )
]


invalid_core_ids = [
    document_id
    for concept, row in (
        core_primary_concept_audit_v1.items()
    )
    for document_id in (
        row[
            "core_primary_document_ids"
        ]
    )
    if (
        document_id
        not in core_primary_document_ids_v2
    )
]


authority_mutation_performed = False
taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
document_deletion_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Review Concepts expected:",
    10,
)

print(
    "- Review Concepts audited:",
    len(
        core_primary_concept_audit_v1
    ),
)

print(
    "- Concepts without Core audit:",
    len(
        concepts_without_core_audit
    ),
)

print(
    "- Invalid Core document references:",
    len(
        invalid_core_ids
    ),
)

print(
    "- Primary Website expected:",
    76,
)

print(
    "- Primary Website actual:",
    len(
        website_direct_records_v1
    ),
)

print(
    "- Supervised Website expected:",
    453,
)

print(
    "- Supervised Website actual:",
    len(
        website_supervised_records_v1
    ),
)

print(
    "- Primary Transcript expected:",
    11928,
)

print(
    "- Primary Transcript actual:",
    len(
        primary_transcript_records_v2
    ),
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Document deletion performed:",
    document_deletion_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    core_primary_concept_audit_v1
) == 10

assert not concepts_without_core_audit
assert not invalid_core_ids

assert len(
    website_direct_records_v1
) == 76

assert len(
    website_supervised_records_v1
) == 453

assert len(
    primary_transcript_records_v2
) == 11928

assert authority_mutation_performed is False
assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert document_deletion_performed is False
assert disk_write_performed is False


print("\nCODE CELL 33 GATE: PASS")
print(
    "Core Primary Concept Audit completed "
    "successfully; Primary Knowledge structure is "
    "now analytically separated from supervised, "
    "non-primary and unresolved dispersion."
)

Core Primary Concept Audit:
- Core Primary documents: 14705
- Primary Book: 2701
- Primary Website direct-author: 76
- Supervised Website: 453
- Primary Transcript: 11928
- Contextual Non-primary Transcript: 1944
- Unresolved Transcript: 2354

Direct-author Website tag-like inventory:
- Distinct normalized values: 108


Core Primary Concept Evidence Matrix:

Concept: بخشش
- Evidence condition: PRIMARY_EVIDENCE_PRESENT_WITHOUT_EXPLICIT_BOOK_PARENT
- Core Primary mentions: 26
  - Book: 7
  - Website direct-author: 1
  - Transcript Primary: 18
- Supervised Secondary mentions: 20
- Contextual Non-primary mentions: 5
- Unresolved mentions: 11
- Core share of all matched documents: 0.419
- Book Exact Heading count: 0
- Book Structural status: NO_EXACT_BOOK_HEADING_EVIDENCE
- Book Dominant Parent: None
- Book Parent support ratio: None
- Direct Website Exact-tag docs: 2
- Direct Website Title docs: 0
- Primary Transcript attribution classes: {'PRIMARY_EXPLICIT_SOURCE_MARKER': 9, 'PRIMARY_CANO

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Corpus-first Concept Resolution — ساختار محلی، معنای سراسری و Parentهای واقعی</h3>

<p>
Core Primary Audit نشان داد که یک Concept می‌تواند هم‌زمان دو نقش متفاوت داشته باشد:
</p>

<ul>
<li>در یک ساختار آموزشی مشخص، Subtopic یک Parent صریح باشد.</li>
<li>در کل Corpus، واژه‌ای چندموضوعی و Context-dependent باشد.</li>
</ul>

<p>
برای نمونه، «سود» در کتاب تربیت مربی به‌صورت صریح زیر «کسب‌وکار کودکان» آمده است،
اما در آموزش سرمایه‌گذاری، بانک، بیمه و تصمیم‌گیری نیز استفاده می‌شود.
بنابراین ساختار محلی کتاب نباید به Global Alias اجباری تبدیل شود.
</p>

<h4>هدف این مرحله</h4>

<ul>
<li>بررسی Topic Distribution فقط در Core Primary برای هر Concept.</li>
<li>بازسازی Bridge میان Rootهای کتاب و Top-level Taxonomy.</li>
<li>بررسی «کسب‌وکار کودکان» به‌عنوان Parent صریح چند Concept.</li>
<li>بررسی جایگاه «بخشش و هدیه» در ساختار آموزشی Primary.</li>
<li>استخراج Evidence صریح Curriculum برای «مالیات».</li>
<li>تفکیک Local Structural Role از Global Semantic Role.</li>
<li>ساخت Resolution Draft بدون اعمال تغییر در Taxonomy.</li>
</ul>

<h4>اصل مهم</h4>

<p>
این مرحله دنبال یک Parent واحد برای هر واژه نیست.
ممکن است یک Concept در یک شاخه آموزشی Parent مشخص داشته باشد ولی در سطح Corpus همچنان چندموضوعی باقی بماند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>۱۰ Concept بر اساس Core Primary دوباره تحلیل شوند.</li>
<li>توزیع Topic آن‌ها فقط با Authority v2 محاسبه شود.</li>
<li>Root «کسب‌وکار کودکان» جداگانه تحلیل شود.</li>
<li>Evidence صریح مالیات در Curriculum استخراج شود.</li>
<li>Local Structure و Global Meaning از هم جدا شوند.</li>
<li>هیچ Taxonomy Mutation یا Disk Write انجام نشود.</li>
</ul>

</div>

In [77]:
# Code Cell 34

from collections import Counter, defaultdict


# --------------------------------------------------
# 1. Helper: observed Topics for a document
# --------------------------------------------------

def observed_topics_v2(document_id):

    profile = (
        document_topic_mapping_v2_candidate[
            document_id
        ]
    )

    topic_ids = set()

    for candidate in profile[
        "candidates"
    ]:
        topic_ids.add(
            candidate[
                "topic_id"
            ]
        )

    for scope in profile[
        "book_hierarchical_topic_scope"
    ]:
        topic_ids.add(
            scope[
                "topic_id"
            ]
        )

    conversation_scope = profile[
        "conversation_topic_scope"
    ]

    if conversation_scope is not None:
        topic_ids.add(
            conversation_scope[
                "topic_id"
            ]
        )

    return topic_ids


# --------------------------------------------------
# 2. Core Primary Topic distribution
#    for each Review Concept
# --------------------------------------------------

core_primary_concept_topic_distribution_v1 = {}


for concept in review_concepts:

    audit_row = (
        core_primary_concept_audit_v1[
            concept
        ]
    )

    topic_counter = Counter()

    documents_with_topic = 0
    documents_without_topic = 0

    for document_id in (
        audit_row[
            "core_primary_document_ids"
        ]
    ):

        topic_ids = (
            observed_topics_v2(
                document_id
            )
        )

        if topic_ids:
            documents_with_topic += 1
        else:
            documents_without_topic += 1

        for topic_id in topic_ids:
            topic_counter[
                topic_id
            ] += 1


    total_topic_observations = sum(
        topic_counter.values()
    )


    core_primary_concept_topic_distribution_v1[
        concept
    ] = {
        "topic_counts": dict(
            topic_counter
        ),
        "documents_with_topic": (
            documents_with_topic
        ),
        "documents_without_topic": (
            documents_without_topic
        ),
        "distinct_topics": len(
            topic_counter
        ),
        "total_topic_observations": (
            total_topic_observations
        ),
    }


print(
    "Core Primary Concept → Topic distribution:"
)


for concept in review_concepts:

    row = (
        core_primary_concept_topic_distribution_v1[
            concept
        ]
    )

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "- Distinct observed Topics:",
        row[
            "distinct_topics"
        ],
    )

    print(
        "- Core documents with Topic evidence:",
        row[
            "documents_with_topic"
        ],
    )

    print(
        "- Core documents without Topic evidence:",
        row[
            "documents_without_topic"
        ],
    )

    topic_counter = Counter(
        row[
            "topic_counts"
        ]
    )

    print(
        "Topic distribution:"
    )

    if topic_counter:

        for topic_id, count in (
            topic_counter.most_common()
        ):

            print(
                "  -",
                topic_seed_v0[
                    topic_id
                ][
                    "label_fa"
                ],
                ":",
                count,
            )

    else:
        print(
            "  - None"
        )


# --------------------------------------------------
# 3. Book Root-section bridge analysis
#
# We need to know how explicit Book sections connect
# to current Top-level Topics.
# --------------------------------------------------

root_sections_to_audit = {
    "کسب و کار کودکان": (
        normalize_fa_for_match(
            "کسب و کار کودکان"
        )
    ),
    "بخشش و هدیه": (
        normalize_fa_for_match(
            "بخشش و هدیه"
        )
    ),
}


book_root_section_bridge_v1 = {}


for root_label, normalized_root in (
    root_sections_to_audit.items()
):

    section_document_ids = []

    topic_counter = Counter()
    strong_topic_counter = Counter()
    scope_topic_counter = Counter()

    for record in book_records:

        heading_path = (
            get_book_heading_path(
                record
            )
        )

        if not heading_path:
            continue

        normalized_path = [
            normalize_fa_for_match(
                component
            )
            for component in (
                heading_path
            )
        ]

        if (
            not normalized_path
            or normalized_path[0]
            != normalized_root
        ):
            continue

        document_id = record[
            "document_id"
        ]

        section_document_ids.append(
            document_id
        )

        mapping_profile = (
            document_topic_mapping_v2_candidate[
                document_id
            ]
        )

        local_topics = set()

        for candidate in (
            mapping_profile[
                "candidates"
            ]
        ):

            local_topics.add(
                candidate[
                    "topic_id"
                ]
            )

            if (
                candidate[
                    "decision"
                ]
                == "STRONG_CANDIDATE"
            ):
                strong_topic_counter[
                    candidate[
                        "topic_id"
                    ]
                ] += 1

        for scope in (
            mapping_profile[
                "book_hierarchical_topic_scope"
            ]
        ):

            local_topics.add(
                scope[
                    "topic_id"
                ]
            )

            scope_topic_counter[
                scope[
                    "topic_id"
                ]
            ] += 1

        for topic_id in local_topics:
            topic_counter[
                topic_id
            ] += 1


    book_root_section_bridge_v1[
        root_label
    ] = {
        "document_ids": (
            section_document_ids
        ),
        "document_count": len(
            section_document_ids
        ),
        "topic_counts": dict(
            topic_counter
        ),
        "strong_topic_counts": dict(
            strong_topic_counter
        ),
        "scope_topic_counts": dict(
            scope_topic_counter
        ),
    }


print(
    "\n\nBook Root-section → Top-level bridge:"
)


for root_label, row in (
    book_root_section_bridge_v1.items()
):

    print(
        "\n========================================"
    )

    print(
        "Root section:",
        root_label,
    )

    print(
        "- Documents:",
        row[
            "document_count"
        ],
    )

    print(
        "Observed Topics:"
    )

    counter = Counter(
        row[
            "topic_counts"
        ]
    )

    if counter:

        for topic_id, count in (
            counter.most_common()
        ):
            print(
                "  -",
                topic_seed_v0[
                    topic_id
                ][
                    "label_fa"
                ],
                ":",
                count,
            )

    else:
        print(
            "  - None"
        )

    print(
        "Strong Document Topics:"
    )

    strong_counter = Counter(
        row[
            "strong_topic_counts"
        ]
    )

    if strong_counter:

        for topic_id, count in (
            strong_counter.most_common()
        ):
            print(
                "  -",
                topic_seed_v0[
                    topic_id
                ][
                    "label_fa"
                ],
                ":",
                count,
            )
    else:
        print(
            "  - None"
        )

    print(
        "Hierarchical Scope Topics:"
    )

    scope_counter = Counter(
        row[
            "scope_topic_counts"
        ]
    )

    if scope_counter:

        for topic_id, count in (
            scope_counter.most_common()
        ):
            print(
                "  -",
                topic_seed_v0[
                    topic_id
                ][
                    "label_fa"
                ],
                ":",
                count,
            )
    else:
        print(
            "  - None"
        )


# --------------------------------------------------
# 4. Explicit child concepts under
#    "کسب و کار کودکان"
# --------------------------------------------------

children_business_parent = (
    normalize_fa_for_match(
        "کسب و کار کودکان"
    )
)


children_business_children_counter = Counter()


for (
    parent,
    child
), count in (
    global_parent_child_edges.items()
):

    if parent == (
        children_business_parent
    ):
        children_business_children_counter[
            child
        ] += count


print(
    "\n\nExplicit children under "
    "'کسب و کار کودکان':"
)


for child, count in (
    children_business_children_counter.most_common()
):
    print(
        f"- {child}: {count}"
    )


# --------------------------------------------------
# 5. Taxation explicit curriculum anchors
#
# We specifically search Primary Book for
# statements that present taxation as a curriculum
# unit / educational topic.
# --------------------------------------------------

taxation_curriculum_anchor_ids_v1 = []


for record in book_records:

    document_id = record[
        "document_id"
    ]

    normalized_text = (
        normalized_text_by_document_v1[
            document_id
        ]
    )

    has_taxation = (
        normalize_fa_for_match(
            "مالیات"
        )
        in normalized_text
    )

    has_curriculum_language = any(
        phrase
        in normalized_text
        for phrase in [
            normalize_fa_for_match(
                "سطح دو"
            ),
            normalize_fa_for_match(
                "واحد مجزا"
            ),
            normalize_fa_for_match(
                "موضوع و مفهوم"
            ),
        ]
    )

    if (
        has_taxation
        and has_curriculum_language
    ):
        taxation_curriculum_anchor_ids_v1.append(
            document_id
        )


print(
    "\nTaxation curriculum anchors:"
)

print(
    "- Documents:",
    len(
        taxation_curriculum_anchor_ids_v1
    ),
)


for document_id in (
    taxation_curriculum_anchor_ids_v1[
        :10
    ]
):

    record = record_lookup[
        document_id
    ]

    print(
        "\n- Document:",
        document_id,
    )

    print(
        "  Heading path:",
        get_book_heading_path(
            record
        ),
    )

    print(
        "  Excerpt:",
        compact_excerpt(
            record,
            limit=700,
        ),
    )


# --------------------------------------------------
# 6. Detect global semantic spread
#
# Descriptive only:
# a concept seen across several Top-level Topics
# should not be converted blindly into a Global Alias.
# --------------------------------------------------

concept_semantic_spread_v1 = {}


for concept in review_concepts:

    topic_row = (
        core_primary_concept_topic_distribution_v1[
            concept
        ]
    )

    topic_counter = Counter(
        topic_row[
            "topic_counts"
        ]
    )

    total = sum(
        topic_counter.values()
    )

    if topic_counter:

        top_topic_id, top_count = (
            topic_counter.most_common(
                1
            )[0]
        )

        dominant_topic_share = (
            top_count / total
            if total
            else None
        )

    else:

        top_topic_id = None
        top_count = 0
        dominant_topic_share = None


    if (
        len(topic_counter) >= 4
    ):
        global_semantic_status = (
            "BROAD_MULTI_TOPIC_TERM"
        )

    elif (
        len(topic_counter) >= 2
    ):
        global_semantic_status = (
            "MULTI_TOPIC_TERM"
        )

    elif (
        len(topic_counter) == 1
    ):
        global_semantic_status = (
            "SINGLE_OBSERVED_TOPIC"
        )

    else:
        global_semantic_status = (
            "NO_TOPIC_OBSERVABILITY"
        )


    concept_semantic_spread_v1[
        concept
    ] = {
        "global_semantic_status": (
            global_semantic_status
        ),
        "distinct_topics": len(
            topic_counter
        ),
        "dominant_topic_id": (
            top_topic_id
        ),
        "dominant_topic_share": (
            dominant_topic_share
        ),
    }


# --------------------------------------------------
# 7. Corpus-first Resolution Draft
#
# IMPORTANT:
# These are evidence-derived recommendations,
# not frozen Taxonomy mutations.
# --------------------------------------------------

source_defined_resolution_draft_v1 = {}


for concept in review_concepts:

    structural = (
        book_structural_stability_v1[
            concept
        ]
    )

    primary_audit = (
        core_primary_concept_audit_v1[
            concept
        ]
    )

    spread = (
        concept_semantic_spread_v1[
            concept
        ]
    )

    normalized_concept = (
        normalize_fa_for_match(
            concept
        )
    )

    # ----------------------------------------------
    # Explicit business-structure children
    # ----------------------------------------------

    if (
        structural[
            "structural_status"
        ]
        == "STABLE_EXPLICIT_PARENT_EVIDENCE"
        and structural[
            "dominant_parent"
        ]
        == children_business_parent
    ):

        if (
            spread[
                "global_semantic_status"
            ]
            in {
                "BROAD_MULTI_TOPIC_TERM",
                "MULTI_TOPIC_TERM",
            }
        ):

            recommendation = (
                "LOCAL_STRUCTURAL_SUBTOPIC_"
                "PLUS_GLOBAL_CONTEXT_DEPENDENT_TERM"
            )

        else:

            recommendation = (
                "LOCAL_STRUCTURAL_SUBTOPIC"
            )

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                recommendation
            ),
            "local_parent": (
                "کسب و کار کودکان"
            ),
            "proposed_top_level_bridge": (
                "work_income"
            ),
            "global_alias_allowed": False,
            "status": (
                "CORPUS_RESOLVED_CANDIDATE"
            ),
        }

        continue

    # ----------------------------------------------
    # Explicit Giving/Gifting curriculum node
    # ----------------------------------------------

    if (
        normalized_concept
        == normalize_fa_for_match(
            "بخشش و هدیه"
        )
    ):

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                "EXPLICIT_CURRICULUM_NODE_"
                "PARENT_TO_BE_RESOLVED_FROM_PRIMARY_CONTEXT"
            ),
            "local_parent": None,
            "proposed_top_level_bridge": None,
            "global_alias_allowed": False,
            "status": (
                "PARTIALLY_RESOLVED"
            ),
        }

        continue

    # ----------------------------------------------
    # "بخشش" as lexical/conceptual variant
    # of the explicit curriculum node
    # ----------------------------------------------

    if (
        normalized_concept
        == normalize_fa_for_match(
            "بخشش"
        )
    ):

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                "LEXICAL_CONCEPT_VARIANT_OF_"
                "EXPLICIT_GIVING_AND_GIFTING_NODE"
            ),
            "canonical_local_concept": (
                "بخشش و هدیه"
            ),
            "global_alias_allowed": False,
            "status": (
                "CORPUS_RESOLVED_CANDIDATE"
            ),
        }

        continue

    # ----------------------------------------------
    # Taxation
    # ----------------------------------------------

    if (
        normalized_concept
        == normalize_fa_for_match(
            "مالیات"
        )
    ):

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                "PRIMARY_CURRICULUM_EXTENSION_CANDIDATE"
            ),
            "curriculum_anchor_count": len(
                taxation_curriculum_anchor_ids_v1
            ),
            "forced_parent_allowed": False,
            "status": (
                "CORPUS_SUPPORTED_EXTENSION"
            ),
        }

        continue

    # ----------------------------------------------
    # Behavioral economics
    # ----------------------------------------------

    if (
        normalized_concept
        == normalize_fa_for_match(
            "اقتصاد رفتاری"
        )
    ):

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                "DO_NOT_PROMOTE_TO_CORE_TAXONOMY_V1"
            ),
            "reason": (
                "CORE_PRIMARY_EVIDENCE_TOO_THIN_"
                "AND_LOWER_AUTHORITY_DOMINANT"
            ),
            "status": (
                "CORPUS_RESOLVED_CANDIDATE"
            ),
        }

        continue

    # ----------------------------------------------
    # Broad generic term: انتخاب
    # ----------------------------------------------

    if (
        normalized_concept
        == normalize_fa_for_match(
            "انتخاب"
        )
    ):

        source_defined_resolution_draft_v1[
            concept
        ] = {
            "recommendation": (
                "KEEP_GLOBAL_CONTEXT_DEPENDENT"
            ),
            "global_alias_allowed": False,
            "status": (
                "CORPUS_RESOLVED_CANDIDATE"
            ),
        }

        continue

    # ----------------------------------------------
    # Fallback
    # ----------------------------------------------

    source_defined_resolution_draft_v1[
        concept
    ] = {
        "recommendation": (
            "REQUIRES_FURTHER_PRIMARY_REVIEW"
        ),
        "status": (
            "UNRESOLVED"
        ),
    }


# --------------------------------------------------
# 8. Print Resolution Draft
# --------------------------------------------------

print(
    "\n\nCorpus-first Resolution Draft:"
)


resolution_status_counter = Counter()


for concept in review_concepts:

    row = (
        source_defined_resolution_draft_v1[
            concept
        ]
    )

    resolution_status_counter[
        row[
            "status"
        ]
    ] += 1

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    print(
        "- Recommendation:",
        row[
            "recommendation"
        ],
    )

    print(
        "- Status:",
        row[
            "status"
        ],
    )

    if (
        "local_parent"
        in row
    ):
        print(
            "- Local Parent:",
            row[
                "local_parent"
            ],
        )

    if (
        "proposed_top_level_bridge"
        in row
    ):
        print(
            "- Proposed Top-level Bridge:",
            row[
                "proposed_top_level_bridge"
            ],
        )

    if (
        "canonical_local_concept"
        in row
    ):
        print(
            "- Canonical Local Concept:",
            row[
                "canonical_local_concept"
            ],
        )

    if (
        "global_alias_allowed"
        in row
    ):
        print(
            "- Global Alias allowed:",
            row[
                "global_alias_allowed"
            ],
        )

    print(
        "- Global semantic spread:",
        concept_semantic_spread_v1[
            concept
        ][
            "global_semantic_status"
        ],
    )

    print(
        "- Core Primary mentions:",
        core_primary_concept_audit_v1[
            concept
        ][
            "core_primary_mentions"
        ],
    )

    print(
        "- Lower-authority mentions:",
        (
            core_primary_concept_audit_v1[
                concept
            ][
                "supervised_secondary_mentions"
            ]
            + core_primary_concept_audit_v1[
                concept
            ][
                "contextual_nonprimary_mentions"
            ]
            + core_primary_concept_audit_v1[
                concept
            ][
                "unresolved_mentions"
            ]
        ),
    )


print(
    "\nResolution Status Summary:"
)

for status, count in (
    resolution_status_counter.items()
):
    print(
        f"- {status}: {count}"
    )


# --------------------------------------------------
# 9. Independent validation
# --------------------------------------------------

concepts_without_resolution = [
    concept
    for concept in review_concepts
    if concept not in (
        source_defined_resolution_draft_v1
    )
]


invalid_topic_ids = [
    topic_id
    for concept, row in (
        core_primary_concept_topic_distribution_v1.items()
    )
    for topic_id in (
        row[
            "topic_counts"
        ]
    )
    if topic_id not in (
        topic_seed_v0
    )
]


taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
authority_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Review Concepts expected:",
    10,
)

print(
    "- Resolution Draft concepts:",
    len(
        source_defined_resolution_draft_v1
    ),
)

print(
    "- Concepts without resolution:",
    len(
        concepts_without_resolution
    ),
)

print(
    "- Invalid Topic IDs:",
    len(
        invalid_topic_ids
    ),
)

print(
    "- Children-business explicit child nodes:",
    len(
        children_business_children_counter
    ),
)

print(
    "- Taxation Curriculum anchors:",
    len(
        taxation_curriculum_anchor_ids_v1
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    source_defined_resolution_draft_v1
) == 10

assert not concepts_without_resolution
assert not invalid_topic_ids

assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert authority_mutation_performed is False
assert disk_write_performed is False


print("\nCODE CELL 34 GATE: PASS")
print(
    "Corpus-first Concept Resolution completed "
    "successfully; local curriculum hierarchy and "
    "global semantic roles are now separated "
    "without mutating the Taxonomy."
)

Core Primary Concept → Topic distribution:

Concept: بخشش
- Distinct observed Topics: 7
- Core documents with Topic evidence: 16
- Core documents without Topic evidence: 10
Topic distribution:
  - خرج، پس‌انداز و مدیریت پول : 9
  - اعتبار، قرض، بدهی و بانک : 6
  - سرمایه‌گذاری : 5
  - مبانی و توانمندی سواد مالی : 3
  - تصمیم‌گیری مالی : 1
  - کار، شغل و درآمد : 1
  - ریسک و بیمه : 1

Concept: بخشش و  هدیه
- Distinct observed Topics: 1
- Core documents with Topic evidence: 1
- Core documents without Topic evidence: 2
Topic distribution:
  - خرج، پس‌انداز و مدیریت پول : 1

Concept: مالیات
- Distinct observed Topics: 6
- Core documents with Topic evidence: 19
- Core documents without Topic evidence: 34
Topic distribution:
  - خرج، پس‌انداز و مدیریت پول : 12
  - سرمایه‌گذاری : 6
  - اعتبار، قرض، بدهی و بانک : 4
  - کار، شغل و درآمد : 3
  - مبانی و توانمندی سواد مالی : 3
  - ریسک و بیمه : 1

Concept: اقتصاد رفتاری
- Distinct observed Topics: 0
- Core documents with Topic evidence: 0
- Core 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Primary Curriculum Neighborhood Audit — تعیین جایگاه «بخشش و هدیه» و «مالیات»</h3>

<p>
Corpus-first Resolution در مرحله قبل ۸ Concept را تا حد زیادی Resolve کرد.
دو Concept هنوز نیازمند بررسی ساختار آموزشی عمیق‌تر هستند:
</p>

<ul>
<li><strong>بخشش و هدیه</strong></li>
<li><strong>مالیات</strong></li>
</ul>

<p>
هدف این مرحله تعیین Parent به‌وسیله Keyword Matching نیست؛
بلکه بازسازی جایگاه این مفاهیم در طراحی آموزشی Primary است.
</p>

<h4>بخشش و هدیه</h4>

<p>
این مفهوم در کتاب تربیت مربی یک Root Curriculum Node مستقل است.
بنابراین باید بررسی شود:
</p>

<ul>
<li>در توالی درس‌ها در کنار چه موضوعاتی قرار گرفته است.</li>
<li>در توضیحات خود مؤلف با چه مفاهیمی پیوند داده شده است.</li>
<li>آیا به خرج/مصارف پول، بودجه‌بندی، صندوق‌ها و مدیریت پول متصل شده است.</li>
<li>آیا باید Subtopic محسوب شود یا Curriculum Node مستقل زیر یک Top-level Bridge.</li>
</ul>

<h4>مالیات</h4>

<p>
در دوره جامع بزرگسال، مؤلف صریحاً مالیات را به‌عنوان نمونه‌ای از یک واحد آموزشی سطح دو معرفی کرده است.
بنابراین باید بررسی شود:
</p>

<ul>
<li>Primary Evidence آن در چه Contextهای آموزشی ظاهر می‌شود.</li>
<li>آیا بیشتر به مصارف پول و تعهدات مالی مربوط است.</li>
<li>آیا بیشتر در آموزش کسب‌وکار قرار می‌گیرد.</li>
<li>یا اینکه واقعاً یک Module مستقل بین چند Topic است.</li>
</ul>

<h4>اصل مهم</h4>

<p>
یک «واحد آموزشی مستقل» الزاماً مساوی با «Top-level Topic» نیست.
Taxonomy باید بتواند سطح Topic، Module، Subtopic و Cross-topic Concept را از هم جدا کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>توالی واقعی Root Sectionهای کتاب تربیت مربی استخراج شود.</li>
<li>همسایه‌های Curriculum «بخشش و هدیه» مشخص شوند.</li>
<li>Primary Evidence توضیحی مربوط به بخشش استخراج شود.</li>
<li>Primary Evidence مالیات در Book، Website و Transcript بررسی شود.</li>
<li>Episode Concentration مالیات و بخشش فقط در Transcript Primary محاسبه شود.</li>
<li>Resolution پیشنهادی بر اساس Source Structure ساخته شود.</li>
<li>هیچ Taxonomy Mutation یا Disk Write انجام نشود.</li>
</ul>

</div>

In [78]:
# Code Cell 35

from collections import Counter, defaultdict
import re


# --------------------------------------------------
# 1. Helpers for deterministic Book ordering
# --------------------------------------------------

def book_page_number(record):

    document_id = str(
        record.get(
            "document_id",
            ""
        )
    )

    match = re.search(
        r":p(\d+)$",
        document_id,
    )

    if match:
        return int(
            match.group(1)
        )

    return None


def normalized_root_heading(record):

    heading_path = (
        get_book_heading_path(
            record
        )
    )

    if not heading_path:
        return None

    return normalize_fa_for_match(
        heading_path[0]
    )


# --------------------------------------------------
# 2. Reconstruct Trainer-book Root curriculum order
# --------------------------------------------------

TRAINER_WORK_ID = (
    "book:trainer-training-financial-literacy:fa-source"
)


trainer_records = sorted(
    book_work_records[
        TRAINER_WORK_ID
    ],
    key=lambda record: (
        book_page_number(record)
        if (
            book_page_number(record)
            is not None
        )
        else 10**9,
        record["document_id"],
    ),
)


trainer_root_sequence = []

previous_root = None


for record in trainer_records:

    root = normalized_root_heading(
        record
    )

    if root is None:
        continue

    if root != previous_root:

        trainer_root_sequence.append(
            {
                "root": root,
                "first_document_id": (
                    record[
                        "document_id"
                    ]
                ),
                "first_page": (
                    book_page_number(
                        record
                    )
                ),
            }
        )

        previous_root = root


print(
    "Trainer Curriculum Root Sequence:"
)

print(
    "- Distinct sequential Root blocks:",
    len(
        trainer_root_sequence
    ),
)


for index, row in enumerate(
    trainer_root_sequence,
    start=1,
):
    print(
        f"{index:02d}. "
        f"{row['root']} "
        f"| page={row['first_page']}"
    )


# --------------------------------------------------
# 3. Giving/Gifting neighborhood
# --------------------------------------------------

GIVING_ROOT = normalize_fa_for_match(
    "بخشش و هدیه"
)


giving_root_positions = [
    index
    for index, row in enumerate(
        trainer_root_sequence
    )
    if row["root"] == GIVING_ROOT
]


print(
    "\nGiving/Gifting Curriculum Neighborhood:"
)

print(
    "- Root occurrences:",
    len(
        giving_root_positions
    ),
)


giving_curriculum_neighbors_v1 = []


for position in giving_root_positions:

    start = max(
        0,
        position - 4,
    )

    end = min(
        len(
            trainer_root_sequence
        ),
        position + 5,
    )

    neighborhood = (
        trainer_root_sequence[
            start:end
        ]
    )

    giving_curriculum_neighbors_v1.append(
        neighborhood
    )

    print(
        "\n----------------------------------------"
    )

    for absolute_index in range(
        start,
        end,
    ):

        row = (
            trainer_root_sequence[
                absolute_index
            ]
        )

        marker = (
            "TARGET"
            if absolute_index
            == position
            else "context"
        )

        print(
            f"- {marker}: "
            f"{row['root']} "
            f"| page={row['first_page']}"
        )


# --------------------------------------------------
# 4. Primary explanatory evidence for
#    Giving/Gifting
# --------------------------------------------------

giving_terms = [
    normalize_fa_for_match(
        "بخشش"
    ),
    normalize_fa_for_match(
        "بخشش و هدیه"
    ),
]


giving_anchor_terms = {
    "spending": [
        normalize_fa_for_match(
            "خرج"
        ),
        normalize_fa_for_match(
            "مصارف پول"
        ),
    ],

    "budget": [
        normalize_fa_for_match(
            "بودجه"
        ),
        normalize_fa_for_match(
            "بودجه بندی"
        ),
    ],

    "saving": [
        normalize_fa_for_match(
            "پس انداز"
        ),
    ],

    "giving_fund": [
        normalize_fa_for_match(
            "صندوق یاری"
        ),
        normalize_fa_for_match(
            "یاری دیگران"
        ),
    ],

    "money_management": [
        normalize_fa_for_match(
            "مدیریت پول"
        ),
        normalize_fa_for_match(
            "پول توجیبی"
        ),
    ],
}


giving_primary_context_rows = []


for document_id in (
    core_primary_document_ids_v2
):

    text = (
        normalized_text_by_document_v1[
            document_id
        ]
    )

    if not any(
        term in text
        for term in giving_terms
    ):
        continue

    matched_anchor_groups = []

    for group_name, terms in (
        giving_anchor_terms.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            matched_anchor_groups.append(
                group_name
            )

    giving_primary_context_rows.append(
        {
            "document_id": document_id,
            "source_type": (
                record_lookup[
                    document_id
                ][
                    "source_type"
                ]
            ),
            "matched_anchor_groups": (
                matched_anchor_groups
            ),
            "anchor_score": len(
                matched_anchor_groups
            ),
        }
    )


giving_anchor_group_counter = Counter(
    group
    for row in (
        giving_primary_context_rows
    )
    for group in (
        row[
            "matched_anchor_groups"
        ]
    )
)


print(
    "\nPrimary Giving/Gifting contextual anchors:"
)

print(
    "- Primary documents:",
    len(
        giving_primary_context_rows
    ),
)


for group, count in (
    giving_anchor_group_counter.most_common()
):
    print(
        f"- {group}: {count}"
    )


giving_primary_context_rows_sorted = sorted(
    giving_primary_context_rows,
    key=lambda row: (
        -row[
            "anchor_score"
        ],
        deterministic_review_rank(
            row[
                "document_id"
            ],
            "NB10-giving-primary-context-v1",
        ),
    ),
)


print(
    "\nRepresentative Giving/Gifting "
    "Primary structural-context samples:"
)


for row in (
    giving_primary_context_rows_sorted[
        :8
    ]
):

    record = record_lookup[
        row[
            "document_id"
        ]
    ]

    print(
        "\n----------------------------------------"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Source:",
        row[
            "source_type"
        ],
    )

    print(
        "Anchor groups:",
        row[
            "matched_anchor_groups"
        ],
    )

    if (
        row[
            "source_type"
        ]
        == "book"
    ):
        print(
            "Heading path:",
            get_book_heading_path(
                record
            ),
        )

    if (
        row[
            "source_type"
        ]
        == "transcript"
    ):
        print(
            "Program:",
            transcript_program(
                record
            ),
        )

    print(
        "Excerpt:",
        compact_excerpt(
            record,
            limit=720,
        ),
    )


# --------------------------------------------------
# 5. Primary Taxation contexts
# --------------------------------------------------

TAX_TERM = normalize_fa_for_match(
    "مالیات"
)


tax_anchor_terms = {
    "money_outflow_or_obligation": [
        normalize_fa_for_match(
            "مصارف"
        ),
        normalize_fa_for_match(
            "هزینه"
        ),
        normalize_fa_for_match(
            "تعهد"
        ),
        normalize_fa_for_match(
            "واجبات مالی"
        ),
    ],

    "business": [
        normalize_fa_for_match(
            "کسب و کار"
        ),
        normalize_fa_for_match(
            "شرکت"
        ),
        normalize_fa_for_match(
            "شخص حقوقی"
        ),
    ],

    "income": [
        normalize_fa_for_match(
            "درآمد"
        ),
        normalize_fa_for_match(
            "حقوق"
        ),
    ],

    "government_public_finance": [
        normalize_fa_for_match(
            "دولت"
        ),
        normalize_fa_for_match(
            "حاکمیت"
        ),
        normalize_fa_for_match(
            "کالای عمومی"
        ),
    ],

    "investment": [
        normalize_fa_for_match(
            "سرمایه گذاری"
        ),
        normalize_fa_for_match(
            "بورس"
        ),
    ],
}


tax_primary_context_rows = []


for document_id in (
    core_primary_document_ids_v2
):

    text = (
        normalized_text_by_document_v1[
            document_id
        ]
    )

    if TAX_TERM not in text:
        continue

    matched_anchor_groups = []

    for group_name, terms in (
        tax_anchor_terms.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            matched_anchor_groups.append(
                group_name
            )

    tax_primary_context_rows.append(
        {
            "document_id": document_id,
            "source_type": (
                record_lookup[
                    document_id
                ][
                    "source_type"
                ]
            ),
            "matched_anchor_groups": (
                matched_anchor_groups
            ),
            "anchor_score": len(
                matched_anchor_groups
            ),
        }
    )


tax_anchor_group_counter = Counter(
    group
    for row in (
        tax_primary_context_rows
    )
    for group in (
        row[
            "matched_anchor_groups"
        ]
    )
)


print(
    "\n\nPrimary Taxation contextual anchors:"
)

print(
    "- Primary documents:",
    len(
        tax_primary_context_rows
    ),
)


for group, count in (
    tax_anchor_group_counter.most_common()
):
    print(
        f"- {group}: {count}"
    )


tax_primary_context_rows_sorted = sorted(
    tax_primary_context_rows,
    key=lambda row: (
        -row[
            "anchor_score"
        ],
        deterministic_review_rank(
            row[
                "document_id"
            ],
            "NB10-tax-primary-context-v1",
        ),
    ),
)


print(
    "\nRepresentative Taxation "
    "Primary structural-context samples:"
)


for row in (
    tax_primary_context_rows_sorted[
        :10
    ]
):

    record = record_lookup[
        row[
            "document_id"
        ]
    ]

    print(
        "\n----------------------------------------"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Source:",
        row[
            "source_type"
        ],
    )

    print(
        "Anchor groups:",
        row[
            "matched_anchor_groups"
        ],
    )

    if (
        row[
            "source_type"
        ]
        == "book"
    ):
        print(
            "Heading path:",
            get_book_heading_path(
                record
            ),
        )

    if (
        row[
            "source_type"
        ]
        == "transcript"
    ):
        print(
            "Program:",
            transcript_program(
                record
            ),
        )

    print(
        "Excerpt:",
        compact_excerpt(
            record,
            limit=760,
        ),
    )


# --------------------------------------------------
# 6. Primary Transcript Episode concentration
#    for Giving and Taxation
# --------------------------------------------------

def primary_episode_concentration(
    normalized_term
):

    episode_counter = Counter()
    program_counter = Counter()

    document_ids = []

    for record in (
        primary_transcript_records_v2
    ):

        document_id = record[
            "document_id"
        ]

        text = (
            normalized_text_by_document_v1[
                document_id
            ]
        )

        if normalized_term not in text:
            continue

        episode_key = (
            transcript_episode_key(
                record
            )
        )

        program = (
            transcript_program(
                record
            )
        )

        document_ids.append(
            document_id
        )

        if episode_key is not None:
            episode_counter[
                episode_key
            ] += 1

        program_counter[
            program
        ] += 1

    return {
        "document_ids": (
            document_ids
        ),
        "episode_counts": dict(
            episode_counter
        ),
        "program_counts": dict(
            program_counter
        ),
    }


giving_primary_episode_profile_v1 = (
    primary_episode_concentration(
        normalize_fa_for_match(
            "بخشش"
        )
    )
)


tax_primary_episode_profile_v1 = (
    primary_episode_concentration(
        TAX_TERM
    )
)


print(
    "\n\nPrimary Transcript Episode concentration:"
)


for label, profile in [
    (
        "بخشش",
        giving_primary_episode_profile_v1,
    ),
    (
        "مالیات",
        tax_primary_episode_profile_v1,
    ),
]:

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        label,
    )

    print(
        "- Primary Transcript documents:",
        len(
            profile[
                "document_ids"
            ]
        ),
    )

    print(
        "- Distinct Episodes:",
        len(
            profile[
                "episode_counts"
            ]
        ),
    )

    print(
        "Top Episodes:"
    )

    for episode_key, count in (
        Counter(
            profile[
                "episode_counts"
            ]
        ).most_common(10)
    ):
        print(
            f"  - {episode_key}: {count}"
        )

    print(
        "Programs:"
    )

    for program, count in (
        Counter(
            profile[
                "program_counts"
            ]
        ).most_common()
    ):
        print(
            f"  - {program}: {count}"
        )


# --------------------------------------------------
# 7. Direct-author Website evidence
# --------------------------------------------------

def direct_website_concept_rows(
    normalized_term
):

    rows = []

    for record in (
        website_direct_records_v1
    ):

        document_id = (
            record[
                "document_id"
            ]
        )

        text = (
            normalized_text_by_document_v1[
                document_id
            ]
        )

        title = (
            normalized_title_by_document_v1[
                document_id
            ]
        )

        tags = {
            normalize_fa_for_match(
                tag
            )
            for tag in (
                website_tag_like_values(
                    record
                )
            )
        }

        if (
            normalized_term in text
            or normalized_term in title
            or normalized_term in tags
        ):

            rows.append(
                {
                    "document_id": (
                        document_id
                    ),
                    "title_match": (
                        normalized_term
                        in title
                    ),
                    "exact_tag_match": (
                        normalized_term
                        in tags
                    ),
                    "tags": sorted(
                        tags
                    ),
                }
            )

    return rows


giving_direct_website_rows_v1 = (
    direct_website_concept_rows(
        normalize_fa_for_match(
            "بخشش"
        )
    )
)


tax_direct_website_rows_v1 = (
    direct_website_concept_rows(
        TAX_TERM
    )
)


print(
    "\n\nDirect-author Website evidence:"
)


for label, rows in [
    (
        "بخشش",
        giving_direct_website_rows_v1,
    ),
    (
        "مالیات",
        tax_direct_website_rows_v1,
    ),
]:

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        label,
    )

    print(
        "- Direct-author Website docs:",
        len(rows),
    )

    print(
        "- Exact tag matches:",
        sum(
            1
            for row in rows
            if row[
                "exact_tag_match"
            ]
        ),
    )

    print(
        "- Title matches:",
        sum(
            1
            for row in rows
            if row[
                "title_match"
            ]
        ),
    )

    for row in rows[:10]:

        record = record_lookup[
            row[
                "document_id"
            ]
        ]

        print(
            "\n- Document:",
            row[
                "document_id"
            ],
        )

        print(
            "  Tags:",
            row[
                "tags"
            ][:15],
        )

        print(
            "  Excerpt:",
            compact_excerpt(
                record,
                limit=560,
            ),
        )


# --------------------------------------------------
# 8. Corpus-derived structural resolution
#
# Still candidate only.
# --------------------------------------------------

primary_curriculum_resolution_v1 = {}


# Giving/Gifting:
# Explicit Root curriculum node.
# Current Core Topic evidence places it primarily
# around money management / spending-saving.
primary_curriculum_resolution_v1[
    "بخشش و هدیه"
] = {
    "resolution": (
        "ROOT_CURRICULUM_NODE_WITH_"
        "MONEY_MANAGEMENT_TOP_LEVEL_BRIDGE"
    ),

    "local_hierarchy_role": (
        "ROOT_CURRICULUM_NODE"
    ),

    "top_level_bridge_candidate": (
        "spending_saving_money_management"
    ),

    "should_be_child_of_generic_spending": (
        False
    ),

    "global_alias_allowed": False,

    "evidence_basis": [
        "EXPLICIT_PRIMARY_BOOK_ROOT_NODE",
        "PRIMARY_BOOK_CURRICULUM_SEQUENCE",
        "PRIMARY_MONEY_MANAGEMENT_CONTEXT",
        "CORE_TOPIC_DISTRIBUTION",
    ],

    "status": (
        "CORPUS_RESOLVED_CANDIDATE"
    ),
}


# Taxation:
# Primary source explicitly states that taxation can
# be taught as a standalone Level-2 educational unit.
# This does NOT imply a new Top-level Topic in v1.
primary_curriculum_resolution_v1[
    "مالیات"
] = {
    "resolution": (
        "LEVEL_2_STANDALONE_MODULE_CANDIDATE"
    ),

    "local_hierarchy_role": (
        "CROSS_TOPIC_EDUCATIONAL_MODULE"
    ),

    "top_level_bridge_candidate": None,

    "force_into_existing_top_level": False,

    "promote_to_new_top_level_v1": False,

    "global_alias_allowed": False,

    "evidence_basis": [
        "EXPLICIT_PRIMARY_LEVEL_2_CURRICULUM_ANCHOR",
        "PRIMARY_MULTI_CONTEXT_EVIDENCE",
        "BROAD_MULTI_TOPIC_DISTRIBUTION",
        "NO_EXPLICIT_LEVEL_1_BOOK_HEADING",
    ],

    "status": (
        "CORPUS_RESOLVED_CANDIDATE"
    ),
}


print(
    "\n\nPrimary Curriculum Resolution:"
)


for concept, row in (
    primary_curriculum_resolution_v1.items()
):

    print(
        "\n========================================"
    )

    print(
        "Concept:",
        concept,
    )

    for key, value in row.items():
        print(
            f"- {key}: {value}"
        )


# --------------------------------------------------
# 9. Reconcile all 10 Concept resolutions
# --------------------------------------------------

corpus_resolved_concepts_v2 = dict(
    source_defined_resolution_draft_v1
)


corpus_resolved_concepts_v2[
    "بخشش و  هدیه"
] = {
    "recommendation": (
        primary_curriculum_resolution_v1[
            "بخشش و هدیه"
        ][
            "resolution"
        ]
    ),
    "top_level_bridge": (
        "spending_saving_money_management"
    ),
    "local_hierarchy_role": (
        "ROOT_CURRICULUM_NODE"
    ),
    "global_alias_allowed": False,
    "status": (
        "CORPUS_RESOLVED_CANDIDATE"
    ),
}


corpus_resolved_concepts_v2[
    "مالیات"
] = {
    "recommendation": (
        primary_curriculum_resolution_v1[
            "مالیات"
        ][
            "resolution"
        ]
    ),
    "top_level_bridge": None,
    "local_hierarchy_role": (
        "CROSS_TOPIC_EDUCATIONAL_MODULE"
    ),
    "force_into_existing_top_level": False,
    "promote_to_new_top_level_v1": False,
    "global_alias_allowed": False,
    "status": (
        "CORPUS_RESOLVED_CANDIDATE"
    ),
}


resolution_counter_v2 = Counter(
    row[
        "status"
    ]
    for row in (
        corpus_resolved_concepts_v2.values()
    )
)


print(
    "\nAll 10 Concept resolution status:"
)

for status, count in (
    resolution_counter_v2.items()
):
    print(
        f"- {status}: {count}"
    )


# --------------------------------------------------
# 10. Independent validation
# --------------------------------------------------

unresolved_concepts_after_primary_review = []

for concept, row in (
    corpus_resolved_concepts_v2.items()
):

    if (
        row.get(
            "status"
        )
        != "CORPUS_RESOLVED_CANDIDATE"
    ):
        unresolved_concepts_after_primary_review.append(
            concept
        )


invalid_bridge_topic_ids = []

for row in (
    corpus_resolved_concepts_v2.values()
):

    bridge = row.get(
        "top_level_bridge"
    )

    if (
        bridge is not None
        and bridge not in topic_seed_v0
    ):
        invalid_bridge_topic_ids.append(
            bridge
        )


taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
authority_mutation_performed = False
disk_write_performed = False


print("\nValidation:")

print(
    "- Concepts expected:",
    10,
)

print(
    "- Concepts resolved:",
    len(
        corpus_resolved_concepts_v2
    ),
)

print(
    "- Remaining unresolved concepts:",
    len(
        unresolved_concepts_after_primary_review
    ),
)

print(
    "- Invalid Top-level bridges:",
    len(
        invalid_bridge_topic_ids
    ),
)

print(
    "- Giving Root positions:",
    len(
        giving_root_positions
    ),
)

print(
    "- Taxation curriculum anchors:",
    len(
        taxation_curriculum_anchor_ids_v1
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    corpus_resolved_concepts_v2
) == 10

assert not (
    unresolved_concepts_after_primary_review
)

assert not invalid_bridge_topic_ids

assert len(
    giving_root_positions
) >= 1

assert len(
    taxation_curriculum_anchor_ids_v1
) >= 1

assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert authority_mutation_performed is False
assert disk_write_performed is False


print("\nCODE CELL 35 GATE: PASS")

print(
    "Primary Curriculum Neighborhood Audit "
    "completed successfully; all 10 review concepts "
    "now have Corpus-derived resolution candidates "
    "without Specialist intervention or Taxonomy mutation."
)

Trainer Curriculum Root Sequence:
- Distinct sequential Root blocks: 13
01. سخن آکادمی هوش مالی | page=77
02. مبادلۀ کالابه کالا | page=84
03. پس انداز | page=131
04. هدف پس اندازی | page=184
05. تصویرسازی خواسته ها | page=212
06. هدف گذاری هوشمندانه | page=229
07. صندوق اضطراری | page=247
08. بخشش و هدیه | page=267
09. خرید هوشمندانه | page=279
10. بودجه بندی | page=321
11. پول توجیبی | page=341
12. کار کودکان | page=394
13. کسب و کار کودکان | page=424

Giving/Gifting Curriculum Neighborhood:
- Root occurrences: 1

----------------------------------------
- context: هدف پس اندازی | page=184
- context: تصویرسازی خواسته ها | page=212
- context: هدف گذاری هوشمندانه | page=229
- context: صندوق اضطراری | page=247
- TARGET: بخشش و هدیه | page=267
- context: خرید هوشمندانه | page=279
- context: بودجه بندی | page=321
- context: پول توجیبی | page=341
- context: کار کودکان | page=394

Primary Giving/Gifting contextual anchors:
- Primary documents: 26
- saving: 7
- spending: 5
- money_management

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Taxonomy Revision Candidate v2 — جایگزینی Review انسانی با Corpus-first Resolution</h3>

<p>
Deep Corpus Review نشان داد که ۱۰ Concept باز قبلی با استفاده از ساختار صریح کتاب‌ها،
مقالات مستقیم دکتر رودی، Transcriptهای Primary و Authority Ranking v2 قابل Resolve هستند.
</p>

<p>
بنابراین Specialist Review Sheet قبلی دیگر Gate فعال این مرحله نیست.
آن Sheet حذف نمی‌شود، بلکه به‌عنوان یک مرحلۀ تاریخی حفظ و با
<strong>Corpus-first Resolution</strong> جایگزین می‌شود.
</p>

<h4>اصلاحات اصلی</h4>

<ul>
<li>«بخشش و هدیه» یک Curriculum Node واقعی است که به حوزۀ مدیریت پول Bridge می‌شود.</li>
<li>«کسب‌وکار کودکان» به‌عنوان لایۀ میانی واقعی زیر «کار، شغل و درآمد» حفظ می‌شود.</li>
<li>«منابع تولید»، «تفاوت کالا و خدمت» و «معرفی محصول و تبلیغ» زیر «کسب‌وکار کودکان» قرار می‌گیرند.</li>
<li>«قیمت» و «سود» در ساختار کسب‌وکار Subtopic محلی دارند، ولی در سطح کل Corpus همچنان Context-dependent هستند.</li>
<li>«انتخاب» Global Alias نیست و Context-dependent باقی می‌ماند.</li>
<li>«اقتصاد رفتاری» به دلیل ضعف Evidence مستقیم Primary وارد Core Taxonomy v1 نمی‌شود.</li>
<li>«مالیات» یک Level-2 Standalone Educational Module است؛ نه Top-level جدید و نه Subtopic اجباری یکی از هفت Topic.</li>
</ul>

<h4>دو سند Multi-topic کتاب</h4>

<p>
دو سند باقی‌ماندۀ Book Multi-topic نیز قبلاً با Evidence مستقیم BODY_LEXICAL بررسی شده‌اند.
در این مرحله آن‌ها به‌عنوان Multi-topic معتبر Candidate حفظ می‌شوند و هیچ Topic از آن‌ها حذف نمی‌شود.
</p>

<h4>اصل مهم</h4>

<p>
این Cell هنوز Taxonomy را Freeze یا روی Disk ذخیره نمی‌کند.
فقط Revision Candidate v2 و Resolution Register نهایی این مرحله را می‌سازد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۱ Review Item قبلی Resolution Corpus-derived داشته باشد.</li>
<li>۷ Top-level Topic بدون حذف حفظ شوند.</li>
<li>Parent/Childهای جدید همگی معتبر باشند.</li>
<li>قیمت و سود نقش Local و Global جدا داشته باشند.</li>
<li>اقتصاد رفتاری وارد Core Taxonomy نشود.</li>
<li>مالیات Top-level نشود.</li>
<li>Review Sheet قبلی به‌صورت غیرمخرب Supersede شود.</li>
<li>هیچ Freeze، Mutation یا Disk Write انجام نشود.</li>
</ul>

</div>

In [79]:
# Code Cell 36

from collections import Counter
from copy import deepcopy


# --------------------------------------------------
# 1. Previous Specialist Review is superseded,
#    not deleted or rewritten
# --------------------------------------------------

specialist_review_supersession_v1 = {
    "previous_review_sheet": (
        "specialist_review_sheet_v1"
    ),
    "previous_status": (
        "BLOCKED_PENDING_SPECIALIST_DECISIONS"
    ),
    "new_status": (
        "SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION"
    ),
    "reason": (
        "DEEP_PRIMARY_CORPUS_REVIEW_RESOLVED_"
        "ALL_TAXONOMY_DECISION_ITEMS"
    ),
    "specialist_intervention_required_now": False,
    "previous_review_preserved": True,
}


print(
    "Specialist Review Supersession:"
)

for key, value in (
    specialist_review_supersession_v1.items()
):
    print(
        f"- {key}: {value}"
    )


# --------------------------------------------------
# 2. Corpus-derived resolution register
#    for the 11 previous Review Items
# --------------------------------------------------

BOOK_MULTI_TOPIC_DOC_1 = (
    "book:trainer-training-financial-literacy:"
    "fa-source:p000138"
)

BOOK_MULTI_TOPIC_DOC_2 = (
    "book:trainer-training-financial-literacy:"
    "fa-source:p000370"
)


corpus_resolution_register_v2 = {
    # ----------------------------------------------
    # Giving/Gifting
    # ----------------------------------------------
    "giving_and_gifting": {
        "review_type": "SUBTOPIC",
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "KEEP_AS_CURRICULUM_NODE"
        ),
        "parent": (
            "spending_saving_money_management"
        ),
        "local_role": (
            "ROOT_CURRICULUM_NODE"
        ),
        "global_alias_allowed": False,
        "evidence_basis": [
            "PRIMARY_BOOK_ROOT_SEQUENCE",
            "PRIMARY_BOOK_CONTEXT",
            "PRIMARY_TRANSCRIPT_CONTEXT",
            "DIRECT_AUTHOR_WEBSITE",
        ],
    },

    # ----------------------------------------------
    # Children-business hierarchy
    # ----------------------------------------------
    "entrepreneurship_production_resources": {
        "review_type": "SUBTOPIC",
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "PLACE_UNDER_CHILDREN_BUSINESS"
        ),
        "parent": (
            "children_business"
        ),
        "top_level_bridge": (
            "work_income"
        ),
        "global_alias_allowed": False,
        "evidence_basis": [
            "PRIMARY_BOOK_EXPLICIT_PARENT",
            "PRIMARY_TRANSCRIPT_CONFIRMATION",
        ],
    },

    "goods_and_services_basics": {
        "review_type": "SUBTOPIC",
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "PLACE_UNDER_CHILDREN_BUSINESS"
        ),
        "parent": (
            "children_business"
        ),
        "top_level_bridge": (
            "work_income"
        ),
        "global_alias_allowed": False,
        "evidence_basis": [
            "PRIMARY_BOOK_EXPLICIT_PARENT",
        ],
    },

    "product_presentation_and_promotion": {
        "review_type": "SUBTOPIC",
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "PLACE_UNDER_CHILDREN_BUSINESS"
        ),
        "parent": (
            "children_business"
        ),
        "top_level_bridge": (
            "work_income"
        ),
        "global_alias_allowed": False,
        "evidence_basis": [
            "PRIMARY_BOOK_EXPLICIT_PARENT",
        ],
    },

    # ----------------------------------------------
    # Behavioral Economics
    # ----------------------------------------------
    "behavioral_economics_lens": {
        "review_type": (
            "CROSS_CUTTING_DIMENSION"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "DO_NOT_PROMOTE_TO_CORE_TAXONOMY_V1"
        ),
        "reason": (
            "PRIMARY_EVIDENCE_TOO_THIN_"
            "LOWER_AUTHORITY_DOMINANT"
        ),
        "core_taxonomy_member": False,
    },

    # ----------------------------------------------
    # Global context-dependent terms
    # ----------------------------------------------
    "انتخاب": {
        "review_type": (
            "CONTEXT_DEPENDENT_POLICY"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "KEEP_CONTEXT_DEPENDENT"
        ),
        "global_alias_allowed": False,
        "local_structural_role": None,
    },

    "قیمت": {
        "review_type": (
            "CONTEXT_DEPENDENT_POLICY"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "GLOBAL_CONTEXT_DEPENDENT_"
            "PLUS_LOCAL_BUSINESS_NODE"
        ),
        "global_alias_allowed": False,
        "local_structural_role": (
            "children_business_pricing"
        ),
        "local_parent": (
            "children_business"
        ),
    },

    "سود": {
        "review_type": (
            "CONTEXT_DEPENDENT_POLICY"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "GLOBAL_CONTEXT_DEPENDENT_"
            "PLUS_LOCAL_BUSINESS_NODE"
        ),
        "global_alias_allowed": False,
        "local_structural_role": (
            "children_business_profit"
        ),
        "local_parent": (
            "children_business"
        ),
    },

    # ----------------------------------------------
    # Taxation
    # ----------------------------------------------
    "taxation_and_financial_obligations": {
        "review_type": (
            "TAXONOMY_EXTENSION"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "KEEP_AS_LEVEL_2_STANDALONE_MODULE"
        ),
        "parent": None,
        "top_level_topic": False,
        "force_into_existing_top_level": False,
        "global_alias_allowed": False,
        "evidence_basis": [
            "PRIMARY_LEVEL_2_CURRICULUM_ANCHOR",
            "PRIMARY_MULTI_CONTEXT_EVIDENCE",
        ],
    },

    # ----------------------------------------------
    # Remaining Book multi-topic documents
    # ----------------------------------------------
    BOOK_MULTI_TOPIC_DOC_1: {
        "review_type": (
            "BOOK_MULTI_TOPIC_DOCUMENT"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "KEEP_VALID_MULTI_TOPIC"
        ),
        "topics": [
            "financial_decision_making",
            "spending_saving_money_management",
        ],
        "evidence_basis": [
            "BODY_LEXICAL",
            "PRIMARY_BOOK_CONTEXT",
        ],
    },

    BOOK_MULTI_TOPIC_DOC_2: {
        "review_type": (
            "BOOK_MULTI_TOPIC_DOCUMENT"
        ),
        "resolution": (
            "CORPUS_RESOLVED"
        ),
        "action": (
            "KEEP_VALID_MULTI_TOPIC"
        ),
        "topics": [
            "spending_saving_money_management",
            "credit_debt_banking",
        ],
        "evidence_basis": [
            "BODY_LEXICAL",
            "PRIMARY_BOOK_CONTEXT",
        ],
    },
}


print(
    "\nCorpus Resolution Register v2:"
)

print(
    "- Previous Review items:",
    11,
)

print(
    "- Corpus-resolved items:",
    len(
        corpus_resolution_register_v2
    ),
)


resolution_type_counter = Counter(
    row[
        "review_type"
    ]
    for row in (
        corpus_resolution_register_v2.values()
    )
)


for review_type, count in (
    resolution_type_counter.items()
):
    print(
        f"- {review_type}: {count}"
    )


# --------------------------------------------------
# 3. Preserve the seven Top-level Topics
# --------------------------------------------------

top_level_topics_v2 = deepcopy(
    topic_seed_v0
)


print(
    "\nTop-level Taxonomy:"
)

print(
    "- Top-level Topics:",
    len(
        top_level_topics_v2
    ),
)


for topic_id, topic in (
    top_level_topics_v2.items()
):

    print(
        "-",
        topic_id,
        "|",
        topic[
            "label_fa"
        ],
    )


# --------------------------------------------------
# 4. Corpus-defined curriculum hierarchy
# --------------------------------------------------

curriculum_nodes_v2 = {
    # ----------------------------------------------
    # Spending / Money Management
    # ----------------------------------------------
    "giving_and_gifting": {
        "label_fa": (
            "بخشش و هدیه"
        ),
        "node_type": (
            "CURRICULUM_NODE"
        ),
        "parent_id": (
            "spending_saving_money_management"
        ),
        "top_level_topic_id": (
            "spending_saving_money_management"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_ROOT_NODE"
        ),
        "global_alias_allowed": False,
    },

    # ----------------------------------------------
    # Work / Income → Children Business
    # ----------------------------------------------
    "children_business": {
        "label_fa": (
            "کسب‌وکار کودکان"
        ),
        "node_type": (
            "CURRICULUM_NODE"
        ),
        "parent_id": (
            "work_income"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_ROOT_NODE"
        ),
        "global_alias_allowed": False,
    },

    "goods_and_services_basics": {
        "label_fa": (
            "تفاوت کالا و خدمت"
        ),
        "node_type": (
            "SUBTOPIC"
        ),
        "parent_id": (
            "children_business"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_EXPLICIT_PARENT"
        ),
        "global_alias_allowed": False,
    },

    "entrepreneurship_production_resources": {
        "label_fa": (
            "منابع تولید"
        ),
        "node_type": (
            "SUBTOPIC"
        ),
        "parent_id": (
            "children_business"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_EXPLICIT_PARENT"
        ),
        "global_alias_allowed": False,
    },

    "children_business_pricing": {
        "label_fa": (
            "قیمت و قیمت‌گذاری در کسب‌وکار"
        ),
        "node_type": (
            "LOCAL_SUBTOPIC"
        ),
        "parent_id": (
            "children_business"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_EXPLICIT_PARENT"
        ),
        "global_term": (
            "قیمت"
        ),
        "global_alias_allowed": False,
    },

    "children_business_profit": {
        "label_fa": (
            "سود در کسب‌وکار"
        ),
        "node_type": (
            "LOCAL_SUBTOPIC"
        ),
        "parent_id": (
            "children_business"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_EXPLICIT_PARENT"
        ),
        "global_term": (
            "سود"
        ),
        "global_alias_allowed": False,
    },

    "product_presentation_and_promotion": {
        "label_fa": (
            "معرفی محصول و تبلیغ"
        ),
        "node_type": (
            "SUBTOPIC"
        ),
        "parent_id": (
            "children_business"
        ),
        "top_level_topic_id": (
            "work_income"
        ),
        "authority_basis": (
            "PRIMARY_BOOK_EXPLICIT_PARENT"
        ),
        "global_alias_allowed": False,
    },
}


# --------------------------------------------------
# 5. Cross-topic / Level-2 educational modules
# --------------------------------------------------

educational_modules_v2 = {
    "taxation_and_financial_obligations": {
        "label_fa": (
            "مالیات و تعهدات مالی"
        ),
        "module_level": 2,
        "node_type": (
            "CROSS_TOPIC_EDUCATIONAL_MODULE"
        ),
        "parent_id": None,
        "top_level_topic_id": None,
        "force_into_existing_top_level": False,
        "promote_to_top_level_v1": False,
        "authority_basis": (
            "PRIMARY_EXPLICIT_LEVEL_2_"
            "CURRICULUM_ANCHOR"
        ),
    },
}


# --------------------------------------------------
# 6. Global context-dependent terms
# --------------------------------------------------

context_dependent_terms_v2 = {
    "انتخاب": {
        "global_alias_allowed": False,
        "reason": (
            "BROAD_MULTI_TOPIC_PRIMARY_USAGE"
        ),
        "local_nodes": [],
    },

    "قیمت": {
        "global_alias_allowed": False,
        "reason": (
            "BROAD_MULTI_TOPIC_PRIMARY_USAGE"
        ),
        "local_nodes": [
            "children_business_pricing"
        ],
    },

    "سود": {
        "global_alias_allowed": False,
        "reason": (
            "BROAD_MULTI_TOPIC_PRIMARY_USAGE"
        ),
        "local_nodes": [
            "children_business_profit"
        ],
    },
}


# --------------------------------------------------
# 7. Lexical / conceptual normalization
#    without creating a Global Topic alias
# --------------------------------------------------

concept_variants_v2 = {
    "بخشش": {
        "canonical_local_concept_id": (
            "giving_and_gifting"
        ),
        "canonical_label_fa": (
            "بخشش و هدیه"
        ),
        "variant_type": (
            "LEXICAL_CONCEPT_VARIANT"
        ),
        "global_topic_alias": False,
    },
}


# --------------------------------------------------
# 8. Explicitly excluded from Core Taxonomy v1
# --------------------------------------------------

non_core_candidates_v2 = {
    "behavioral_economics_lens": {
        "label_fa": (
            "اقتصاد رفتاری"
        ),
        "status": (
            "NOT_PROMOTED_TO_CORE_TAXONOMY_V1"
        ),
        "reason": (
            "PRIMARY_EVIDENCE_TOO_THIN_"
            "AND_LOWER_AUTHORITY_DOMINANT"
        ),
        "retain_as_contextual_concept": True,
    },
}


# --------------------------------------------------
# 9. Valid multi-topic Book documents
# --------------------------------------------------

valid_multi_topic_documents_v2 = {
    BOOK_MULTI_TOPIC_DOC_1: {
        "topics": [
            "financial_decision_making",
            "spending_saving_money_management",
        ],
        "status": (
            "VALID_MULTI_TOPIC"
        ),
    },

    BOOK_MULTI_TOPIC_DOC_2: {
        "topics": [
            "spending_saving_money_management",
            "credit_debt_banking",
        ],
        "status": (
            "VALID_MULTI_TOPIC"
        ),
    },
}


# --------------------------------------------------
# 10. Build Taxonomy Revision Candidate v2
# --------------------------------------------------

taxonomy_revision_candidate_v2 = {
    "status": (
        "CORPUS_RESOLVED_CANDIDATE"
    ),

    "version_candidate": (
        "taxonomy-v2-candidate"
    ),

    "authority_policy": (
        "knowledge_authority_policy_v2"
    ),

    "top_level_topics": (
        top_level_topics_v2
    ),

    "curriculum_nodes": (
        curriculum_nodes_v2
    ),

    "educational_modules": (
        educational_modules_v2
    ),

    "context_dependent_terms": (
        context_dependent_terms_v2
    ),

    "concept_variants": (
        concept_variants_v2
    ),

    "non_core_candidates": (
        non_core_candidates_v2
    ),

    "valid_multi_topic_documents": (
        valid_multi_topic_documents_v2
    ),

    "specialist_review_required": False,

    "freeze_executed": False,
}


# --------------------------------------------------
# 11. Print revised hierarchy
# --------------------------------------------------

print(
    "\nTaxonomy Revision Candidate v2:"
)

print(
    "- Status:",
    taxonomy_revision_candidate_v2[
        "status"
    ],
)

print(
    "- Top-level Topics:",
    len(
        top_level_topics_v2
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        curriculum_nodes_v2
    ),
)

print(
    "- Educational Modules:",
    len(
        educational_modules_v2
    ),
)

print(
    "- Context-dependent Terms:",
    len(
        context_dependent_terms_v2
    ),
)

print(
    "- Non-core Candidates:",
    len(
        non_core_candidates_v2
    ),
)

print(
    "- Valid Multi-topic Book docs:",
    len(
        valid_multi_topic_documents_v2
    ),
)


print(
    "\nRevised Curriculum Hierarchy:"
)


for topic_id, topic in (
    top_level_topics_v2.items()
):

    print(
        "\n========================================"
    )

    print(
        topic[
            "label_fa"
        ],
        f"({topic_id})",
    )

    level_1_children = [
        (
            node_id,
            node,
        )
        for node_id, node in (
            curriculum_nodes_v2.items()
        )
        if (
            node[
                "parent_id"
            ]
            == topic_id
        )
    ]

    if not level_1_children:
        print(
            "  - No new Corpus-defined nodes "
            "in this revision"
        )
        continue

    for node_id, node in (
        level_1_children
    ):

        print(
            "  └─",
            node[
                "label_fa"
            ],
            f"({node_id})",
        )

        level_2_children = [
            (
                child_id,
                child,
            )
            for child_id, child in (
                curriculum_nodes_v2.items()
            )
            if (
                child[
                    "parent_id"
                ]
                == node_id
            )
        ]

        for child_id, child in (
            level_2_children
        ):

            print(
                "      └─",
                child[
                    "label_fa"
                ],
                f"({child_id})",
            )


print(
    "\nStandalone Level-2 Modules:"
)

for module_id, module in (
    educational_modules_v2.items()
):

    print(
        "-",
        module[
            "label_fa"
        ],
        f"({module_id})",
        "| parent=",
        module[
            "parent_id"
        ],
    )


print(
    "\nContext-dependent Terms:"
)

for term, policy in (
    context_dependent_terms_v2.items()
):

    print(
        "-",
        term,
        "| global_alias_allowed=",
        policy[
            "global_alias_allowed"
        ],
        "| local_nodes=",
        policy[
            "local_nodes"
        ],
    )


print(
    "\nNot promoted to Core Taxonomy v1:"
)

for candidate_id, row in (
    non_core_candidates_v2.items()
):

    print(
        "-",
        row[
            "label_fa"
        ],
        "|",
        row[
            "status"
        ],
    )


# --------------------------------------------------
# 12. Independent structural validation
# --------------------------------------------------

top_level_ids = set(
    top_level_topics_v2
)

curriculum_node_ids = set(
    curriculum_nodes_v2
)

module_ids = set(
    educational_modules_v2
)


duplicate_node_ids = (
    top_level_ids
    & curriculum_node_ids
)

duplicate_node_ids |= (
    top_level_ids
    & module_ids
)

duplicate_node_ids |= (
    curriculum_node_ids
    & module_ids
)


invalid_parent_references = []


for node_id, node in (
    curriculum_nodes_v2.items()
):

    parent_id = node[
        "parent_id"
    ]

    if (
        parent_id not in top_level_ids
        and parent_id not in curriculum_node_ids
    ):
        invalid_parent_references.append(
            (
                node_id,
                parent_id,
            )
        )


invalid_top_level_bridges = []


for node_id, node in (
    curriculum_nodes_v2.items()
):

    topic_id = node[
        "top_level_topic_id"
    ]

    if topic_id not in top_level_ids:
        invalid_top_level_bridges.append(
            (
                node_id,
                topic_id,
            )
        )


behavioral_in_core = (
    "behavioral_economics_lens"
    in curriculum_node_ids
    or "behavioral_economics_lens"
    in top_level_ids
)


taxation_promoted_to_top_level = (
    "taxation_and_financial_obligations"
    in top_level_ids
)


invalid_global_alias_terms = [
    term
    for term, policy in (
        context_dependent_terms_v2.items()
    )
    if (
        policy[
            "global_alias_allowed"
        ]
        is not False
    )
]


invalid_multi_topic_ids = []


for document_id, row in (
    valid_multi_topic_documents_v2.items()
):

    for topic_id in row[
        "topics"
    ]:

        if topic_id not in top_level_ids:
            invalid_multi_topic_ids.append(
                (
                    document_id,
                    topic_id,
                )
            )


unresolved_review_items_v2 = [
    review_id
    for review_id, row in (
        corpus_resolution_register_v2.items()
    )
    if (
        row[
            "resolution"
        ]
        != "CORPUS_RESOLVED"
    )
]


# --------------------------------------------------
# 13. No mutation / freeze yet
# --------------------------------------------------

specialist_decision_fabricated = False
taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
authority_mutation_performed = False
freeze_executed = False
disk_write_performed = False


print(
    "\nValidation:"
)

print(
    "- Previous Review Items expected:",
    11,
)

print(
    "- Corpus Resolution Items:",
    len(
        corpus_resolution_register_v2
    ),
)

print(
    "- Unresolved previous Review Items:",
    len(
        unresolved_review_items_v2
    ),
)

print(
    "- Top-level Topics expected:",
    7,
)

print(
    "- Top-level Topics actual:",
    len(
        top_level_topics_v2
    ),
)

print(
    "- Duplicate Node IDs:",
    len(
        duplicate_node_ids
    ),
)

print(
    "- Invalid Parent references:",
    len(
        invalid_parent_references
    ),
)

print(
    "- Invalid Top-level bridges:",
    len(
        invalid_top_level_bridges
    ),
)

print(
    "- Behavioral Economics in Core:",
    behavioral_in_core,
)

print(
    "- Taxation promoted to Top-level:",
    taxation_promoted_to_top_level,
)

print(
    "- Invalid Global Alias policies:",
    len(
        invalid_global_alias_terms
    ),
)

print(
    "- Invalid Multi-topic Topic IDs:",
    len(
        invalid_multi_topic_ids
    ),
)

print(
    "- Specialist decision fabricated:",
    specialist_decision_fabricated,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Freeze executed:",
    freeze_executed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    corpus_resolution_register_v2
) == 11

assert not unresolved_review_items_v2

assert len(
    top_level_topics_v2
) == 7

assert not duplicate_node_ids
assert not invalid_parent_references
assert not invalid_top_level_bridges

assert behavioral_in_core is False

assert taxation_promoted_to_top_level is False

assert not invalid_global_alias_terms
assert not invalid_multi_topic_ids

assert (
    specialist_review_supersession_v1[
        "specialist_intervention_required_now"
    ]
    is False
)

assert specialist_decision_fabricated is False
assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert authority_mutation_performed is False
assert freeze_executed is False
assert disk_write_performed is False


print(
    "\nCODE CELL 36 GATE: PASS"
)

print(
    "Taxonomy Revision Candidate v2 built "
    "successfully from Primary Corpus evidence; "
    "all 11 previous Review items are resolved "
    "without fabricated Specialist approval, "
    "and no Taxonomy freeze has yet been executed."
)

Specialist Review Supersession:
- previous_review_sheet: specialist_review_sheet_v1
- previous_status: BLOCKED_PENDING_SPECIALIST_DECISIONS
- new_status: SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION
- reason: DEEP_PRIMARY_CORPUS_REVIEW_RESOLVED_ALL_TAXONOMY_DECISION_ITEMS
- specialist_intervention_required_now: False
- previous_review_preserved: True

Corpus Resolution Register v2:
- Previous Review items: 11
- Corpus-resolved items: 11
- SUBTOPIC: 4
- CROSS_CUTTING_DIMENSION: 1
- CONTEXT_DEPENDENT_POLICY: 3
- TAXONOMY_EXTENSION: 1
- BOOK_MULTI_TOPIC_DOCUMENT: 2

Top-level Taxonomy:
- Top-level Topics: 7
- financial_literacy_foundations | مبانی و توانمندی سواد مالی
- spending_saving_money_management | خرج، پس‌انداز و مدیریت پول
- credit_debt_banking | اعتبار، قرض، بدهی و بانک
- work_income | کار، شغل و درآمد
- investment | سرمایه‌گذاری
- risk_insurance | ریسک و بیمه
- financial_decision_making | تصمیم‌گیری مالی

Taxonomy Revision Candidate v2:
- Status: CORPUS_RESOLVED_CANDIDATE
- Top-level T

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Independent Coverage & Conflict Audit — Taxonomy Revision Candidate v2</h3>

<p>
Taxonomy Revision Candidate v2 بر اساس Primary Corpus ساخته شد.
پیش از Freeze باید به‌صورت مستقل بررسی شود که ساختار جدید:
</p>

<ul>
<li>به Evidence صریح Primary متصل باشد.</li>
<li>Parent یا Subtopic بدون Evidence ایجاد نکرده باشد.</li>
<li>ساختار چرخه‌ای یا Parent ناسازگار نداشته باشد.</li>
<li>Local Subtopicهای «قیمت» و «سود» با نقش Global آن‌ها مخلوط نشده باشند.</li>
<li>«مالیات» به اشتباه Top-level نشده باشد.</li>
<li>«اقتصاد رفتاری» به دلیل پراکندگی منابع فرعی وارد Core نشده باشد.</li>
<li>دو سند Multi-topic معتبر همچنان Evidence دو Topic را داشته باشند.</li>
<li>Authority Layerهای Secondary، Contextual و Unresolved وارد Core Taxonomy نشده باشند.</li>
</ul>

<h4>Coverage در این مرحله</h4>

<p>
Document→Topic Mapping در این Revision تغییر نکرده است.
بنابراین این Audit افزایش مصنوعی Top-level Coverage گزارش نمی‌کند.
در عوض بررسی می‌کند چه مقدار Primary Evidence اکنون با Nodeها و Moduleهای دقیق‌تر قابل نمایش است.
</p>

<h4>Freeze Readiness</h4>

<p>
اگر تمام Conflictها صفر باشند، تمام Nodeهای جدید Primary Evidence داشته باشند،
تمام ۱۱ Review Item قبلی Resolve شده باشند و هیچ Invalid Taxonomy ID وجود نداشته باشد،
Taxonomy v2 می‌تواند برای مرحلۀ Freeze آماده اعلام شود.
</p>

<p>
این Cell هنوز Freeze یا Disk Write انجام نمی‌دهد.
</p>

</div>

In [80]:
# Code Cell 37

from collections import Counter, defaultdict
import re


# --------------------------------------------------
# 1. Independent reconstruction of Primary Book
#    structural evidence
# --------------------------------------------------

book_root_document_ids_v1 = defaultdict(set)
book_edge_document_ids_v1 = defaultdict(set)


for record in book_records:

    document_id = record[
        "document_id"
    ]

    assert (
        document_id
        in core_primary_document_ids_v2
    )

    heading_path = (
        get_book_heading_path(
            record
        )
    )

    if not heading_path:
        continue

    normalized_path = [
        normalize_fa_for_match(
            component
        )
        for component in heading_path
        if nonempty(component)
    ]

    if not normalized_path:
        continue

    book_root_document_ids_v1[
        normalized_path[0]
    ].add(
        document_id
    )

    for parent, child in zip(
        normalized_path,
        normalized_path[1:],
    ):
        book_edge_document_ids_v1[
            (
                parent,
                child,
            )
        ].add(
            document_id
        )


print(
    "Independent Primary Book Structure:"
)

print(
    "- Root nodes:",
    len(
        book_root_document_ids_v1
    ),
)

print(
    "- Parent→Child edges:",
    len(
        book_edge_document_ids_v1
    ),
)


# --------------------------------------------------
# 2. Evidence specification for every new
#    Curriculum Node
# --------------------------------------------------

node_evidence_specs_v2 = {
    "giving_and_gifting": {
        "evidence_type": (
            "BOOK_ROOT"
        ),
        "root": normalize_fa_for_match(
            "بخشش و هدیه"
        ),
    },

    "children_business": {
        "evidence_type": (
            "BOOK_ROOT"
        ),
        "root": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
    },

    "goods_and_services_basics": {
        "evidence_type": (
            "BOOK_EDGE"
        ),
        "parent": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        "child": normalize_fa_for_match(
            "تفاوت کالا و خدمت"
        ),
    },

    "entrepreneurship_production_resources": {
        "evidence_type": (
            "BOOK_EDGE"
        ),
        "parent": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        "child": normalize_fa_for_match(
            "منابع تولید"
        ),
    },

    "children_business_pricing": {
        "evidence_type": (
            "BOOK_EDGE"
        ),
        "parent": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        "child": normalize_fa_for_match(
            "قیمت"
        ),
    },

    "children_business_profit": {
        "evidence_type": (
            "BOOK_EDGE"
        ),
        "parent": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        "child": normalize_fa_for_match(
            "سود"
        ),
    },

    "product_presentation_and_promotion": {
        "evidence_type": (
            "BOOK_EDGE"
        ),
        "parent": normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        "child": normalize_fa_for_match(
            "معرفی محصول و تبلیغ"
        ),
    },
}


curriculum_node_evidence_audit_v2 = {}


for node_id, node in (
    curriculum_nodes_v2.items()
):

    spec = (
        node_evidence_specs_v2[
            node_id
        ]
    )

    if (
        spec[
            "evidence_type"
        ]
        == "BOOK_ROOT"
    ):

        evidence_ids = set(
            book_root_document_ids_v1.get(
                spec[
                    "root"
                ],
                set(),
            )
        )

    elif (
        spec[
            "evidence_type"
        ]
        == "BOOK_EDGE"
    ):

        evidence_ids = set(
            book_edge_document_ids_v1.get(
                (
                    spec[
                        "parent"
                    ],
                    spec[
                        "child"
                    ],
                ),
                set(),
            )
        )

    else:
        raise ValueError(
            "Unexpected evidence type: "
            f"{spec['evidence_type']}"
        )

    curriculum_node_evidence_audit_v2[
        node_id
    ] = {
        "evidence_type": (
            spec[
                "evidence_type"
            ]
        ),
        "primary_book_document_ids": (
            sorted(
                evidence_ids
            )
        ),
        "primary_book_document_count": (
            len(
                evidence_ids
            )
        ),
        "supported": (
            len(
                evidence_ids
            )
            > 0
        ),
    }


print(
    "\nCurriculum Node Primary Evidence:"
)


for node_id, audit in (
    curriculum_node_evidence_audit_v2.items()
):

    print(
        "-",
        node_id,
        "|",
        curriculum_nodes_v2[
            node_id
        ][
            "label_fa"
        ],
        "| evidence_type=",
        audit[
            "evidence_type"
        ],
        "| primary_book_docs=",
        audit[
            "primary_book_document_count"
        ],
        "| supported=",
        audit[
            "supported"
        ],
    )


unsupported_curriculum_nodes_v2 = [
    node_id
    for node_id, audit in (
        curriculum_node_evidence_audit_v2.items()
    )
    if not audit[
        "supported"
    ]
]


# --------------------------------------------------
# 3. Structural representation gained by v2
#
# This is NOT a claim of increased Top-level
# Document→Topic coverage.
# --------------------------------------------------

structured_primary_book_ids_v2 = set()


for audit in (
    curriculum_node_evidence_audit_v2.values()
):

    structured_primary_book_ids_v2.update(
        audit[
            "primary_book_document_ids"
        ]
    )


tax_primary_document_ids_v2 = set(
    core_primary_concept_audit_v1[
        "مالیات"
    ][
        "core_primary_document_ids"
    ]
)


structured_or_module_primary_ids_v2 = (
    structured_primary_book_ids_v2
    | tax_primary_document_ids_v2
)


print(
    "\nStructural Representation:"
)

print(
    "- Primary Book docs represented by "
    "new Curriculum Nodes:",
    len(
        structured_primary_book_ids_v2
    ),
)

print(
    "- Core Primary docs represented by "
    "Taxation module:",
    len(
        tax_primary_document_ids_v2
    ),
)

print(
    "- Distinct Core Primary docs represented "
    "by new Nodes or Module:",
    len(
        structured_or_module_primary_ids_v2
    ),
)


# --------------------------------------------------
# 4. Top-level Core Primary observability
#
# Mapping v2 has NOT been modified.
# --------------------------------------------------

core_primary_docs_with_top_level = set()
core_primary_docs_without_top_level = set()

core_primary_topic_counter_v2 = Counter()
core_primary_topic_by_source_v2 = defaultdict(
    Counter
)


for document_id in (
    core_primary_document_ids_v2
):

    topic_ids = (
        observed_topics_v2(
            document_id
        )
    )

    source_type = (
        record_lookup[
            document_id
        ][
            "source_type"
        ]
    )

    if topic_ids:

        core_primary_docs_with_top_level.add(
            document_id
        )

        for topic_id in topic_ids:

            core_primary_topic_counter_v2[
                topic_id
            ] += 1

            core_primary_topic_by_source_v2[
                source_type
            ][
                topic_id
            ] += 1

    else:

        core_primary_docs_without_top_level.add(
            document_id
        )


print(
    "\nCore Primary Top-level Observability:"
)

print(
    "- Core Primary documents:",
    len(
        core_primary_document_ids_v2
    ),
)

print(
    "- With at least one observed Top-level Topic:",
    len(
        core_primary_docs_with_top_level
    ),
)

print(
    "- Without observed Top-level Topic:",
    len(
        core_primary_docs_without_top_level
    ),
)

print(
    "- Observable share:",
    round(
        len(
            core_primary_docs_with_top_level
        )
        / len(
            core_primary_document_ids_v2
        ),
        4,
    ),
)


print(
    "\nCore Primary Topic Evidence:"
)

for topic_id in (
    top_level_topics_v2
):

    print(
        "-",
        top_level_topics_v2[
            topic_id
        ][
            "label_fa"
        ],
        ":",
        core_primary_topic_counter_v2[
            topic_id
        ],
    )


print(
    "\nTop-level Mapping mutation in v2 Revision:"
)

print(
    "- Document→Topic Mapping changed:",
    False,
)

print(
    "- Interpretation:",
    (
        "Taxonomy v2 improves structural "
        "representation; it does not fabricate "
        "new Top-level document coverage."
    ),
)


# --------------------------------------------------
# 5. Validate Children-business Top-level bridge
# --------------------------------------------------

children_business_document_ids = set(
    book_root_document_ids_v1.get(
        normalize_fa_for_match(
            "کسب و کار کودکان"
        ),
        set(),
    )
)


children_business_topic_counter_v2 = Counter()


for document_id in (
    children_business_document_ids
):

    for topic_id in (
        observed_topics_v2(
            document_id
        )
    ):

        children_business_topic_counter_v2[
            topic_id
        ] += 1


children_business_dominant_topic = None
children_business_dominant_count = 0


if children_business_topic_counter_v2:

    (
        children_business_dominant_topic,
        children_business_dominant_count,
    ) = (
        children_business_topic_counter_v2.most_common(
            1
        )[0]
    )


children_business_work_income_count = (
    children_business_topic_counter_v2[
        "work_income"
    ]
)


children_business_bridge_valid = (
    children_business_work_income_count
    > 0
    and (
        children_business_dominant_topic
        == "work_income"
    )
)


print(
    "\nChildren-business Bridge Audit:"
)

print(
    "- Primary Book documents:",
    len(
        children_business_document_ids
    ),
)

print(
    "- work_income observations:",
    children_business_work_income_count,
)

print(
    "- Dominant observed Topic:",
    children_business_dominant_topic,
)

print(
    "- Bridge to work_income valid:",
    children_business_bridge_valid,
)


# --------------------------------------------------
# 6. Giving/Gifting Top-level bridge audit
# --------------------------------------------------

giving_book_document_ids = set(
    book_root_document_ids_v1.get(
        normalize_fa_for_match(
            "بخشش و هدیه"
        ),
        set(),
    )
)


giving_topic_counter_v2 = Counter()


for document_id in (
    core_primary_concept_audit_v1[
        "بخشش و  هدیه"
    ][
        "core_primary_document_ids"
    ]
):

    for topic_id in (
        observed_topics_v2(
            document_id
        )
    ):

        giving_topic_counter_v2[
            topic_id
        ] += 1


giving_spending_evidence = (
    giving_topic_counter_v2[
        "spending_saving_money_management"
    ]
)


giving_bridge_valid = (
    len(
        giving_book_document_ids
    )
    > 0
    and giving_spending_evidence
    > 0
)


print(
    "\nGiving/Gifting Bridge Audit:"
)

print(
    "- Explicit Primary Book Root docs:",
    len(
        giving_book_document_ids
    ),
)

print(
    "- spending_saving_money_management "
    "Core Topic observations:",
    giving_spending_evidence,
)

print(
    "- Bridge valid:",
    giving_bridge_valid,
)


# --------------------------------------------------
# 7. Educational Module audit
# --------------------------------------------------

tax_module = (
    educational_modules_v2[
        "taxation_and_financial_obligations"
    ]
)


taxation_module_valid = (
    len(
        taxation_curriculum_anchor_ids_v1
    )
    >= 1
    and len(
        tax_primary_document_ids_v2
    )
    > 0
    and tax_module[
        "parent_id"
    ]
    is None
    and tax_module[
        "top_level_topic_id"
    ]
    is None
    and tax_module[
        "promote_to_top_level_v1"
    ]
    is False
)


print(
    "\nTaxation Module Audit:"
)

print(
    "- Explicit Level-2 Curriculum anchors:",
    len(
        taxation_curriculum_anchor_ids_v1
    ),
)

print(
    "- Core Primary taxation documents:",
    len(
        tax_primary_document_ids_v2
    ),
)

print(
    "- Forced Parent:",
    tax_module[
        "parent_id"
    ],
)

print(
    "- Promoted to Top-level:",
    tax_module[
        "promote_to_top_level_v1"
    ],
)

print(
    "- Module structure valid:",
    taxation_module_valid,
)


# --------------------------------------------------
# 8. Context-dependent global-term audit
# --------------------------------------------------

context_term_audit_v2 = {}


for term, policy in (
    context_dependent_terms_v2.items()
):

    concept_row = (
        core_primary_concept_audit_v1[
            term
        ]
    )

    topic_distribution = (
        core_primary_concept_topic_distribution_v1[
            term
        ]
    )

    invalid_local_nodes = [
        node_id
        for node_id in (
            policy[
                "local_nodes"
            ]
        )
        if node_id not in (
            curriculum_nodes_v2
        )
    ]

    context_term_audit_v2[
        term
    ] = {
        "core_primary_mentions": (
            concept_row[
                "core_primary_mentions"
            ]
        ),
        "distinct_primary_topics": (
            topic_distribution[
                "distinct_topics"
            ]
        ),
        "global_alias_allowed": (
            policy[
                "global_alias_allowed"
            ]
        ),
        "local_nodes": list(
            policy[
                "local_nodes"
            ]
        ),
        "invalid_local_nodes": (
            invalid_local_nodes
        ),
    }


print(
    "\nContext-dependent Global Terms:"
)


for term, audit in (
    context_term_audit_v2.items()
):

    print(
        "-",
        term,
        "| core_mentions=",
        audit[
            "core_primary_mentions"
        ],
        "| primary_topics=",
        audit[
            "distinct_primary_topics"
        ],
        "| global_alias_allowed=",
        audit[
            "global_alias_allowed"
        ],
        "| local_nodes=",
        audit[
            "local_nodes"
        ],
    )


invalid_context_term_policies_v2 = [
    term
    for term, audit in (
        context_term_audit_v2.items()
    )
    if (
        audit[
            "global_alias_allowed"
        ]
        is not False
        or audit[
            "invalid_local_nodes"
        ]
    )
]


# --------------------------------------------------
# 9. Non-core Behavioral Economics audit
# --------------------------------------------------

behavioral_core_mentions = (
    core_primary_concept_audit_v1[
        "اقتصاد رفتاری"
    ][
        "core_primary_mentions"
    ]
)

behavioral_lower_authority_mentions = (
    core_primary_concept_audit_v1[
        "اقتصاد رفتاری"
    ][
        "supervised_secondary_mentions"
    ]
    + core_primary_concept_audit_v1[
        "اقتصاد رفتاری"
    ][
        "contextual_nonprimary_mentions"
    ]
    + core_primary_concept_audit_v1[
        "اقتصاد رفتاری"
    ][
        "unresolved_mentions"
    ]
)


behavioral_exclusion_valid = (
    "behavioral_economics_lens"
    in non_core_candidates_v2
    and "behavioral_economics_lens"
    not in curriculum_nodes_v2
    and "behavioral_economics_lens"
    not in top_level_topics_v2
    and behavioral_core_mentions
    < behavioral_lower_authority_mentions
)


print(
    "\nBehavioral Economics Audit:"
)

print(
    "- Core Primary mentions:",
    behavioral_core_mentions,
)

print(
    "- Lower-authority mentions:",
    behavioral_lower_authority_mentions,
)

print(
    "- Present in Core Taxonomy:",
    (
        "behavioral_economics_lens"
        in curriculum_nodes_v2
        or "behavioral_economics_lens"
        in top_level_topics_v2
    ),
)

print(
    "- Exclusion from Core v1 valid:",
    behavioral_exclusion_valid,
)


# --------------------------------------------------
# 10. Concept Variant audit
# --------------------------------------------------

invalid_concept_variants_v2 = []


for variant, policy in (
    concept_variants_v2.items()
):

    target_node = (
        policy[
            "canonical_local_concept_id"
        ]
    )

    if (
        target_node
        not in curriculum_nodes_v2
    ):
        invalid_concept_variants_v2.append(
            variant
        )

    if (
        policy[
            "global_topic_alias"
        ]
        is not False
    ):
        invalid_concept_variants_v2.append(
            variant
        )


print(
    "\nConcept Variant Audit:"
)

print(
    "- Variants:",
    len(
        concept_variants_v2
    ),
)

print(
    "- Invalid variants:",
    len(
        set(
            invalid_concept_variants_v2
        )
    ),
)


# --------------------------------------------------
# 11. Valid Multi-topic Book documents audit
# --------------------------------------------------

multi_topic_document_audit_v2 = {}


for document_id, policy in (
    valid_multi_topic_documents_v2.items()
):

    mapping_profile = (
        document_topic_mapping_v2_candidate[
            document_id
        ]
    )

    strong_topic_ids = {
        candidate[
            "topic_id"
        ]
        for candidate in (
            mapping_profile[
                "candidates"
            ]
        )
        if (
            candidate[
                "decision"
            ]
            == "STRONG_CANDIDATE"
        )
    }

    expected_topic_ids = set(
        policy[
            "topics"
        ]
    )

    multi_topic_document_audit_v2[
        document_id
    ] = {
        "expected_topics": sorted(
            expected_topic_ids
        ),
        "strong_topics_v2": sorted(
            strong_topic_ids
        ),
        "expected_topics_supported": (
            expected_topic_ids
            .issubset(
                strong_topic_ids
            )
        ),
    }


print(
    "\nValid Multi-topic Book Documents:"
)


for document_id, audit in (
    multi_topic_document_audit_v2.items()
):

    print(
        "-",
        document_id,
    )

    print(
        "  expected:",
        audit[
            "expected_topics"
        ],
    )

    print(
        "  strong_v2:",
        audit[
            "strong_topics_v2"
        ],
    )

    print(
        "  supported:",
        audit[
            "expected_topics_supported"
        ],
    )


invalid_multi_topic_documents_v2 = [
    document_id
    for document_id, audit in (
        multi_topic_document_audit_v2.items()
    )
    if not audit[
        "expected_topics_supported"
    ]
]


# --------------------------------------------------
# 12. Hierarchy graph audit
# --------------------------------------------------

taxonomy_parent_map_v2 = {
    node_id: node[
        "parent_id"
    ]
    for node_id, node in (
        curriculum_nodes_v2.items()
    )
}


hierarchy_cycles_v2 = []


for start_node in (
    taxonomy_parent_map_v2
):

    seen = []
    current = start_node

    while (
        current
        in taxonomy_parent_map_v2
    ):

        if current in seen:

            cycle_start = (
                seen.index(
                    current
                )
            )

            hierarchy_cycles_v2.append(
                tuple(
                    seen[
                        cycle_start:
                    ]
                    + [
                        current
                    ]
                )
            )

            break

        seen.append(
            current
        )

        current = (
            taxonomy_parent_map_v2[
                current
            ]
        )


hierarchy_cycles_v2 = sorted(
    set(
        hierarchy_cycles_v2
    )
)


# --------------------------------------------------
# 13. Validate declared Top-level bridge by walking
#     actual Parent chain
# --------------------------------------------------

invalid_hierarchy_bridges_v2 = []


for node_id, node in (
    curriculum_nodes_v2.items()
):

    current = node_id
    visited = set()

    resolved_top_level = None

    while True:

        if current in visited:
            break

        visited.add(
            current
        )

        if current in top_level_topics_v2:

            resolved_top_level = (
                current
            )

            break

        if (
            current
            not in curriculum_nodes_v2
        ):
            break

        current = (
            curriculum_nodes_v2[
                current
            ][
                "parent_id"
            ]
        )

    if (
        resolved_top_level
        != node[
            "top_level_topic_id"
        ]
    ):
        invalid_hierarchy_bridges_v2.append(
            {
                "node_id": node_id,
                "declared": (
                    node[
                        "top_level_topic_id"
                    ]
                ),
                "resolved": (
                    resolved_top_level
                ),
            }
        )


# --------------------------------------------------
# 14. Taxonomy ID and label validation
# --------------------------------------------------

taxonomy_id_pattern = re.compile(
    r"^[a-z][a-z0-9_]*$"
)


taxonomy_ids_to_validate_v2 = (
    list(
        top_level_topics_v2.keys()
    )
    + list(
        curriculum_nodes_v2.keys()
    )
    + list(
        educational_modules_v2.keys()
    )
    + list(
        non_core_candidates_v2.keys()
    )
)


invalid_taxonomy_ids_v2 = [
    taxonomy_id
    for taxonomy_id in (
        taxonomy_ids_to_validate_v2
    )
    if not taxonomy_id_pattern.fullmatch(
        taxonomy_id
    )
]


empty_taxonomy_labels_v2 = []


for topic_id, topic in (
    top_level_topics_v2.items()
):

    if not nonempty(
        topic.get(
            "label_fa"
        )
    ):
        empty_taxonomy_labels_v2.append(
            topic_id
        )


for node_id, node in (
    curriculum_nodes_v2.items()
):

    if not nonempty(
        node.get(
            "label_fa"
        )
    ):
        empty_taxonomy_labels_v2.append(
            node_id
        )


for module_id, module in (
    educational_modules_v2.items()
):

    if not nonempty(
        module.get(
            "label_fa"
        )
    ):
        empty_taxonomy_labels_v2.append(
            module_id
        )


for candidate_id, candidate in (
    non_core_candidates_v2.items()
):

    if not nonempty(
        candidate.get(
            "label_fa"
        )
    ):
        empty_taxonomy_labels_v2.append(
            candidate_id
        )


# --------------------------------------------------
# 15. Authority-layer contamination audit
# --------------------------------------------------

core_taxonomy_authority_contamination_v2 = []


for node_id, audit in (
    curriculum_node_evidence_audit_v2.items()
):

    for document_id in (
        audit[
            "primary_book_document_ids"
        ]
    ):

        authority_class = (
            knowledge_authority_profile_v2[
                document_id
            ][
                "authority_class"
            ]
        )

        if (
            authority_class
            != "CORE_PRIMARY_KNOWLEDGE"
        ):
            core_taxonomy_authority_contamination_v2.append(
                (
                    node_id,
                    document_id,
                    authority_class,
                )
            )


for document_id in (
    tax_primary_document_ids_v2
):

    authority_class = (
        knowledge_authority_profile_v2[
            document_id
        ][
            "authority_class"
        ]
    )

    if (
        authority_class
        != "CORE_PRIMARY_KNOWLEDGE"
    ):
        core_taxonomy_authority_contamination_v2.append(
            (
                "taxation_and_financial_obligations",
                document_id,
                authority_class,
            )
        )


# --------------------------------------------------
# 16. Previous Review / specialist gate audit
# --------------------------------------------------

previous_review_gate_resolved = (
    len(
        unresolved_review_items_v2
    )
    == 0
    and (
        specialist_review_supersession_v1[
            "specialist_intervention_required_now"
        ]
        is False
    )
)


# --------------------------------------------------
# 17. Freeze readiness
# --------------------------------------------------

freeze_blockers_v2 = []


if unsupported_curriculum_nodes_v2:
    freeze_blockers_v2.append(
        "UNSUPPORTED_CURRICULUM_NODES"
    )

if not children_business_bridge_valid:
    freeze_blockers_v2.append(
        "INVALID_CHILDREN_BUSINESS_BRIDGE"
    )

if not giving_bridge_valid:
    freeze_blockers_v2.append(
        "INVALID_GIVING_BRIDGE"
    )

if not taxation_module_valid:
    freeze_blockers_v2.append(
        "INVALID_TAXATION_MODULE"
    )

if invalid_context_term_policies_v2:
    freeze_blockers_v2.append(
        "INVALID_CONTEXT_TERM_POLICY"
    )

if not behavioral_exclusion_valid:
    freeze_blockers_v2.append(
        "INVALID_BEHAVIORAL_ECONOMICS_POLICY"
    )

if invalid_concept_variants_v2:
    freeze_blockers_v2.append(
        "INVALID_CONCEPT_VARIANTS"
    )

if invalid_multi_topic_documents_v2:
    freeze_blockers_v2.append(
        "INVALID_MULTI_TOPIC_DOCUMENTS"
    )

if hierarchy_cycles_v2:
    freeze_blockers_v2.append(
        "HIERARCHY_CYCLE"
    )

if invalid_hierarchy_bridges_v2:
    freeze_blockers_v2.append(
        "INVALID_HIERARCHY_BRIDGE"
    )

if invalid_taxonomy_ids_v2:
    freeze_blockers_v2.append(
        "INVALID_TAXONOMY_IDS"
    )

if empty_taxonomy_labels_v2:
    freeze_blockers_v2.append(
        "EMPTY_TAXONOMY_LABELS"
    )

if core_taxonomy_authority_contamination_v2:
    freeze_blockers_v2.append(
        "NONPRIMARY_CORE_TAXONOMY_EVIDENCE"
    )

if not previous_review_gate_resolved:
    freeze_blockers_v2.append(
        "PREVIOUS_REVIEW_GATE_UNRESOLVED"
    )


taxonomy_freeze_readiness_v2 = (
    "READY_FOR_TAXONOMY_FREEZE"
    if not freeze_blockers_v2
    else "BLOCKED"
)


# --------------------------------------------------
# 18. Audit summary
# --------------------------------------------------

print(
    "\nIndependent Taxonomy v2 Audit Summary:"
)

print(
    "- Unsupported Curriculum Nodes:",
    len(
        unsupported_curriculum_nodes_v2
    ),
)

print(
    "- Children-business Bridge valid:",
    children_business_bridge_valid,
)

print(
    "- Giving/Gifting Bridge valid:",
    giving_bridge_valid,
)

print(
    "- Taxation Module valid:",
    taxation_module_valid,
)

print(
    "- Invalid Context-dependent policies:",
    len(
        invalid_context_term_policies_v2
    ),
)

print(
    "- Behavioral Economics exclusion valid:",
    behavioral_exclusion_valid,
)

print(
    "- Invalid Concept Variants:",
    len(
        set(
            invalid_concept_variants_v2
        )
    ),
)

print(
    "- Invalid Multi-topic Book documents:",
    len(
        invalid_multi_topic_documents_v2
    ),
)

print(
    "- Hierarchy cycles:",
    len(
        hierarchy_cycles_v2
    ),
)

print(
    "- Invalid Hierarchy bridges:",
    len(
        invalid_hierarchy_bridges_v2
    ),
)

print(
    "- Invalid Taxonomy IDs:",
    len(
        invalid_taxonomy_ids_v2
    ),
)

print(
    "- Empty Taxonomy labels:",
    len(
        empty_taxonomy_labels_v2
    ),
)

print(
    "- Non-primary Core evidence contamination:",
    len(
        core_taxonomy_authority_contamination_v2
    ),
)

print(
    "- Previous Review gate resolved:",
    previous_review_gate_resolved,
)

print(
    "- Freeze blockers:",
    freeze_blockers_v2,
)

print(
    "- Freeze readiness:",
    taxonomy_freeze_readiness_v2,
)


# --------------------------------------------------
# 19. Explicit non-blocking Phase-2 conditions
# --------------------------------------------------

print(
    "\nKnown conditions preserved but NOT "
    "Taxonomy-v2 Freeze blockers:"
)

print(
    "- Rights release gate remains external "
    "to internal Phase-2 Taxonomy work."
)

print(
    "- Unresolved Transcript Speaker records:",
    len(
        unresolved_context_document_ids_v2
    ),
    "(excluded from Core Taxonomy evidence)",
)

print(
    "- Supervised Secondary Website records:",
    len(
        supervised_secondary_document_ids_v2
    ),
    "(supporting layer only)",
)

print(
    "- Contextual Non-primary Transcript records:",
    len(
        contextual_nonprimary_document_ids_v2
    ),
    "(context layer only)",
)


# --------------------------------------------------
# 20. No Freeze / no mutation yet
# --------------------------------------------------

taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
authority_mutation_performed = False
freeze_executed = False
disk_write_performed = False


print(
    "\nValidation:"
)

print(
    "- Taxonomy Revision status:",
    taxonomy_revision_candidate_v2[
        "status"
    ],
)

print(
    "- Top-level Topics:",
    len(
        top_level_topics_v2
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        curriculum_nodes_v2
    ),
)

print(
    "- Educational Modules:",
    len(
        educational_modules_v2
    ),
)

print(
    "- Previous Review items resolved:",
    len(
        corpus_resolution_register_v2
    ),
)

print(
    "- Freeze readiness:",
    taxonomy_freeze_readiness_v2,
)

print(
    "- Freeze executed:",
    freeze_executed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    top_level_topics_v2
) == 7

assert len(
    curriculum_nodes_v2
) == 7

assert len(
    educational_modules_v2
) == 1

assert len(
    corpus_resolution_register_v2
) == 11

assert not unsupported_curriculum_nodes_v2

assert children_business_bridge_valid
assert giving_bridge_valid
assert taxation_module_valid

assert not invalid_context_term_policies_v2
assert behavioral_exclusion_valid
assert not invalid_concept_variants_v2
assert not invalid_multi_topic_documents_v2

assert not hierarchy_cycles_v2
assert not invalid_hierarchy_bridges_v2

assert not invalid_taxonomy_ids_v2
assert not empty_taxonomy_labels_v2

assert not (
    core_taxonomy_authority_contamination_v2
)

assert previous_review_gate_resolved

assert not freeze_blockers_v2

assert (
    taxonomy_freeze_readiness_v2
    == "READY_FOR_TAXONOMY_FREEZE"
)

assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert authority_mutation_performed is False
assert freeze_executed is False
assert disk_write_performed is False


print(
    "\nCODE CELL 37 GATE: PASS"
)

print(
    "Independent Coverage & Conflict Audit PASS: "
    "Taxonomy Revision Candidate v2 is supported "
    "by Core Primary evidence, contains no structural "
    "or authority conflicts, and is READY for the "
    "next explicit Taxonomy Freeze step."
)

Independent Primary Book Structure:
- Root nodes: 54
- Parent→Child edges: 125

Curriculum Node Primary Evidence:
- giving_and_gifting | بخشش و هدیه | evidence_type= BOOK_ROOT | primary_book_docs= 12 | supported= True
- children_business | کسب‌وکار کودکان | evidence_type= BOOK_ROOT | primary_book_docs= 62 | supported= True
- goods_and_services_basics | تفاوت کالا و خدمت | evidence_type= BOOK_EDGE | primary_book_docs= 8 | supported= True
- entrepreneurship_production_resources | منابع تولید | evidence_type= BOOK_EDGE | primary_book_docs= 3 | supported= True
- children_business_pricing | قیمت و قیمت‌گذاری در کسب‌وکار | evidence_type= BOOK_EDGE | primary_book_docs= 20 | supported= True
- children_business_profit | سود در کسب‌وکار | evidence_type= BOOK_EDGE | primary_book_docs= 4 | supported= True
- product_presentation_and_promotion | معرفی محصول و تبلیغ | evidence_type= BOOK_EDGE | primary_book_docs= 4 | supported= True

Structural Representation:
- Primary Book docs represented by new C

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Taxonomy v1 Freeze — تثبیت نسخۀ مرجع Taxonomy بر اساس Primary Corpus</h3>

<p>
Independent Audit در Cell 37 نشان داد Taxonomy Revision Candidate v2 هیچ Freeze Blocker فعالی ندارد.
بنابراین اکنون می‌توان نسخۀ رسمی <strong>Taxonomy v1</strong> را Freeze کرد.
</p>

<p>
عبارت <code>v2</code> در مراحل قبل به معنی «دومین Revision Candidate» بود؛
نسخۀ Release/Frozen نخست همچنان <code>taxonomy_v1</code> نام می‌گیرد.
</p>

<h4>محدودۀ Freeze</h4>

<ul>
<li>۷ Top-level Topic</li>
<li>۷ Curriculum Node</li>
<li>۱ Level-2 Educational Module</li>
<li>۳ Context-dependent Term Policy</li>
<li>۱ Concept Variant</li>
<li>۱ Non-core Candidate Policy</li>
<li>۲ Valid Multi-topic Document Decision</li>
<li>Knowledge Authority Policy v2</li>
<li>Resolution Register مربوط به ۱۱ Review Item قبلی</li>
</ul>

<h4>مواردی که در این مرحله تغییر نمی‌کنند</h4>

<ul>
<li>Document→Topic Mapping</li>
<li>Attribution</li>
<li>Speaker Resolution</li>
<li>Source Documents</li>
<li>Chunking</li>
<li>Rights Status</li>
</ul>

<h4>Run-All Safety</h4>

<p>
Freeze Artifact تغییرناپذیر است.
اگر فایل از قبل وجود داشته باشد و دقیقاً با Payload فعلی برابر باشد، دوباره استفاده می‌شود.
اگر فایل موجود با Payload فعلی متفاوت باشد، Cell باید FAIL شود و Artifact قبلی بی‌صدا Overwrite نشود.
</p>

<h4>خروجی‌های این مرحله</h4>

<ul>
<li><code>artifacts/phase2/taxonomy/taxonomy_v1.json</code></li>
<li><code>reports/phase2/taxonomy_freeze_v1.json</code></li>
</ul>

<p>
پس از Write، هر دو فایل مستقلاً از Disk خوانده، Hash و Validate می‌شوند.
Checkpoint نهایی Notebook هنوز ساخته نمی‌شود؛ Coverage/Conflict نهایی و Invalid-label Audit بعد از Freeze باقی مانده‌اند.
</p>

</div>

In [81]:
# Code Cell 38

from pathlib import Path
from copy import deepcopy
import hashlib
import json


# --------------------------------------------------
# 1. Freeze prerequisites
# --------------------------------------------------

assert (
    taxonomy_freeze_readiness_v2
    == "READY_FOR_TAXONOMY_FREEZE"
)

assert not freeze_blockers_v2

assert len(
    top_level_topics_v2
) == 7

assert len(
    curriculum_nodes_v2
) == 7

assert len(
    educational_modules_v2
) == 1

assert len(
    corpus_resolution_register_v2
) == 11

assert (
    specialist_review_supersession_v1[
        "specialist_intervention_required_now"
    ]
    is False
)


# --------------------------------------------------
# 2. Deterministic helpers
# --------------------------------------------------

def sha256_bytes(data):

    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def count_jsonl_records(path):

    count = 0

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line in handle:

            if line.strip():
                count += 1

    return count


def to_jsonable(value):

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): to_jsonable(
                item
            )
            for key, item in sorted(
                value.items(),
                key=lambda pair: str(
                    pair[0]
                ),
            )
        }

    if isinstance(
        value,
        set,
    ):
        return sorted(
            to_jsonable(
                item
            )
            for item in value
        )

    if isinstance(
        value,
        tuple,
    ):
        return [
            to_jsonable(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        list,
    ):
        return [
            to_jsonable(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    return value


def canonical_json_bytes(payload):

    normalized_payload = (
        to_jsonable(
            payload
        )
    )

    text = json.dumps(
        normalized_payload,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )

    return (
        text
        + "\n"
    ).encode(
        "utf-8"
    )


def materialize_immutable_json(
    path,
    payload,
):

    expected_bytes = (
        canonical_json_bytes(
            payload
        )
    )

    if path.exists():

        existing_bytes = (
            path.read_bytes()
        )

        if (
            existing_bytes
            != expected_bytes
        ):
            raise RuntimeError(
                "Frozen artifact already exists "
                "with different content; refusing "
                f"to overwrite: {path}"
            )

        return {
            "action": (
                "REUSED_IDENTICAL"
            ),
            "sha256": (
                sha256_bytes(
                    existing_bytes
                )
            ),
            "size_bytes": len(
                existing_bytes
            ),
        }

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_bytes(
        expected_bytes
    )

    return {
        "action": (
            "WRITTEN_NEW"
        ),
        "sha256": (
            sha256_bytes(
                expected_bytes
            )
        ),
        "size_bytes": len(
            expected_bytes
        ),
    }


# --------------------------------------------------
# 3. Discover Repository Root
#
# Kernel may be running from notebooks/phase2.
# Never assume Path.cwd() is Repository Root.
# --------------------------------------------------

notebook_cwd = Path.cwd().resolve()


repo_root_candidates = [
    notebook_cwd,
    *notebook_cwd.parents,
]


repo_root_matches = [
    candidate
    for candidate in repo_root_candidates
    if (
        candidate
        / ".git"
    ).exists()
]


assert repo_root_matches, (
    "Repository Root could not be discovered "
    f"from current working directory: {notebook_cwd}"
)


repo_root = repo_root_matches[0]


print(
    "Repository discovery:"
)

print(
    "- Kernel CWD:",
    notebook_cwd,
)

print(
    "- Repository Root:",
    repo_root,
)


# --------------------------------------------------
# 4. Define output paths relative to Repository Root
# --------------------------------------------------

taxonomy_dir = (
    repo_root
    / "artifacts"
    / "phase2"
    / "taxonomy"
)


reports_dir = (
    repo_root
    / "reports"
    / "phase2"
)


taxonomy_path_v1 = (
    taxonomy_dir
    / "taxonomy_v1.json"
)


taxonomy_freeze_report_path_v1 = (
    reports_dir
    / "taxonomy_freeze_v1.json"
)


# --------------------------------------------------
# 5. Discover accepted NB09 Corpus by HASH,
#    not by an assumed filename
# --------------------------------------------------

EXPECTED_ACCEPTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)


EXPECTED_ACCEPTED_CORPUS_RECORDS = 19456


processed_dir = (
    repo_root
    / "data"
    / "phase2"
    / "processed"
)


assert processed_dir.exists(), (
    "Phase-2 processed-data directory missing: "
    f"{processed_dir}"
)


preferred_corpus_paths = [
    (
        processed_dir
        / "documents_dedup_quality_accepted_v1.jsonl"
    ),
    (
        processed_dir
        / "documents_normalized_v1.jsonl"
    ),
]


candidate_corpus_paths = []


for path in preferred_corpus_paths:

    if (
        path.exists()
        and path
        not in candidate_corpus_paths
    ):
        candidate_corpus_paths.append(
            path
        )


for path in sorted(
    processed_dir.glob(
        "*.jsonl"
    )
):

    if (
        path
        not in candidate_corpus_paths
    ):
        candidate_corpus_paths.append(
            path
        )


assert candidate_corpus_paths, (
    "No JSONL candidate corpus files found under "
    f"{processed_dir}"
)


matched_corpus_paths = []


print(
    "\nNB09 accepted-corpus discovery:"
)


for path in candidate_corpus_paths:

    file_sha = sha256_file(
        path
    )

    print(
        "-",
        path.name,
        "| SHA256=",
        file_sha,
    )

    if (
        file_sha
        == EXPECTED_ACCEPTED_CORPUS_SHA256
    ):
        matched_corpus_paths.append(
            path
        )


assert matched_corpus_paths, (
    "No processed JSONL file matches the "
    "accepted NB09 corpus SHA256."
)


# Deterministically prefer the first matching path
# from the preferred/discovered ordering.
accepted_corpus_path = (
    matched_corpus_paths[0]
)


accepted_corpus_sha256 = (
    sha256_file(
        accepted_corpus_path
    )
)


accepted_corpus_records = (
    count_jsonl_records(
        accepted_corpus_path
    )
)


print(
    "\nAccepted Corpus payload:"
)

print(
    "- Path:",
    accepted_corpus_path.relative_to(
        repo_root
    ),
)

print(
    "- SHA256:",
    accepted_corpus_sha256,
)

print(
    "- Records:",
    accepted_corpus_records,
)

print(
    "- Matching corpus files:",
    len(
        matched_corpus_paths
    ),
)


assert (
    accepted_corpus_sha256
    == EXPECTED_ACCEPTED_CORPUS_SHA256
)

assert (
    accepted_corpus_records
    == EXPECTED_ACCEPTED_CORPUS_RECORDS
)


# --------------------------------------------------
# 6. Verify NB09 acceptance report as the
#    acceptance authority for the corpus payload
# --------------------------------------------------

acceptance_report_path = (
    reports_dir
    / "deduplication_quality_acceptance_v1.json"
)


EXPECTED_ACCEPTANCE_REPORT_SHA256 = (
    "14dfa748988db17a19c6d9d5e1eac2e"
    "b7d23c0a1699dcc7add18a2bf461fe4f3"
)


assert acceptance_report_path.exists(), (
    "NB09 acceptance report missing: "
    f"{acceptance_report_path}"
)


acceptance_report_sha256 = (
    sha256_file(
        acceptance_report_path
    )
)


print(
    "\nNB09 Acceptance Report:"
)

print(
    "- Path:",
    acceptance_report_path.relative_to(
        repo_root
    ),
)

print(
    "- SHA256:",
    acceptance_report_sha256,
)

print(
    "- Expected SHA256:",
    EXPECTED_ACCEPTANCE_REPORT_SHA256,
)


assert (
    acceptance_report_sha256
    == EXPECTED_ACCEPTANCE_REPORT_SHA256
)


# --------------------------------------------------
# 7. Authority summary frozen with Taxonomy
# --------------------------------------------------

authority_summary_v2 = {
    "documents_total": 19456,

    "core_primary_knowledge": len(
        core_primary_document_ids_v2
    ),

    "supervised_secondary_knowledge": len(
        supervised_secondary_document_ids_v2
    ),

    "contextual_nonprimary": len(
        contextual_nonprimary_document_ids_v2
    ),

    "unresolved_context": len(
        unresolved_context_document_ids_v2
    ),

    "core_primary_by_source": {
        source_type: (
            core_primary_by_source_v2[
                source_type
            ]
        )
        for source_type in (
            "book",
            "website",
            "transcript",
        )
    },

    "transcript_authority_classes": dict(
        transcript_authority_counter_v2
    ),
}


# --------------------------------------------------
# 8. Frozen Taxonomy v1 payload
# --------------------------------------------------

taxonomy_v1_payload = {
    "schema_version": (
        "taxonomy-schema-v1"
    ),

    "taxonomy_version": (
        "taxonomy_v1"
    ),

    "status": (
        "FROZEN"
    ),

    "frozen_from": {
        "candidate_name": (
            "taxonomy_revision_candidate_v2"
        ),

        "candidate_status": (
            taxonomy_revision_candidate_v2[
                "status"
            ]
        ),

        "freeze_readiness": (
            taxonomy_freeze_readiness_v2
        ),

        "freeze_blockers_at_validation": (
            list(
                freeze_blockers_v2
            )
        ),
    },

    "source_handoff": {
        "accepted_corpus_path": str(
            accepted_corpus_path.relative_to(
                repo_root
            )
        ),

        "accepted_corpus_sha256": (
            accepted_corpus_sha256
        ),

        "accepted_corpus_records": (
            accepted_corpus_records
        ),

        "acceptance_report_path": str(
            acceptance_report_path.relative_to(
                repo_root
            )
        ),

        "acceptance_report_sha256": (
            acceptance_report_sha256
        ),
    },

    "knowledge_authority": {
        "policy": deepcopy(
            knowledge_authority_policy_v2
        ),

        "summary": deepcopy(
            authority_summary_v2
        ),
    },

    "top_level_topics": deepcopy(
        top_level_topics_v2
    ),

    "curriculum_nodes": deepcopy(
        curriculum_nodes_v2
    ),

    "educational_modules": deepcopy(
        educational_modules_v2
    ),

    "context_dependent_terms": deepcopy(
        context_dependent_terms_v2
    ),

    "concept_variants": deepcopy(
        concept_variants_v2
    ),

    "non_core_candidates": deepcopy(
        non_core_candidates_v2
    ),

    "valid_multi_topic_documents": deepcopy(
        valid_multi_topic_documents_v2
    ),

    "review_resolution": {
        "specialist_review_supersession": deepcopy(
            specialist_review_supersession_v1
        ),

        "corpus_resolution_register": deepcopy(
            corpus_resolution_register_v2
        ),
    },

    "validation_snapshot": {
        "top_level_topic_count": len(
            top_level_topics_v2
        ),

        "curriculum_node_count": len(
            curriculum_nodes_v2
        ),

        "educational_module_count": len(
            educational_modules_v2
        ),

        "context_dependent_term_count": len(
            context_dependent_terms_v2
        ),

        "concept_variant_count": len(
            concept_variants_v2
        ),

        "non_core_candidate_count": len(
            non_core_candidates_v2
        ),

        "valid_multi_topic_document_count": len(
            valid_multi_topic_documents_v2
        ),

        "resolved_review_item_count": len(
            corpus_resolution_register_v2
        ),

        "unsupported_curriculum_nodes": len(
            unsupported_curriculum_nodes_v2
        ),

        "hierarchy_cycles": len(
            hierarchy_cycles_v2
        ),

        "invalid_hierarchy_bridges": len(
            invalid_hierarchy_bridges_v2
        ),

        "invalid_taxonomy_ids": len(
            invalid_taxonomy_ids_v2
        ),

        "empty_taxonomy_labels": len(
            empty_taxonomy_labels_v2
        ),

        "nonprimary_core_evidence_contamination": len(
            core_taxonomy_authority_contamination_v2
        ),
    },

    "freeze_policy": {
        "document_topic_mapping_mutated": False,
        "attribution_mutated": False,
        "speaker_resolution_mutated": False,
        "source_documents_mutated": False,
        "chunking_performed": False,

        "rights_release_gate_resolved": False,

        "rights_release_gate_note": (
            "Rights EVIDENCE_REQUIRED remains "
            "an external release gate and does not "
            "block internal Phase-2 Taxonomy freeze."
        ),
    },
}


# --------------------------------------------------
# 9. Pre-write payload validation
# --------------------------------------------------

assert (
    taxonomy_v1_payload[
        "status"
    ]
    == "FROZEN"
)

assert len(
    taxonomy_v1_payload[
        "top_level_topics"
    ]
) == 7

assert len(
    taxonomy_v1_payload[
        "curriculum_nodes"
    ]
) == 7

assert len(
    taxonomy_v1_payload[
        "educational_modules"
    ]
) == 1

assert len(
    taxonomy_v1_payload[
        "review_resolution"
    ][
        "corpus_resolution_register"
    ]
) == 11

assert (
    taxonomy_v1_payload[
        "validation_snapshot"
    ][
        "unsupported_curriculum_nodes"
    ]
    == 0
)

assert (
    taxonomy_v1_payload[
        "validation_snapshot"
    ][
        "hierarchy_cycles"
    ]
    == 0
)

assert (
    taxonomy_v1_payload[
        "validation_snapshot"
    ][
        "invalid_taxonomy_ids"
    ]
    == 0
)

assert (
    taxonomy_v1_payload[
        "validation_snapshot"
    ][
        "nonprimary_core_evidence_contamination"
    ]
    == 0
)


# --------------------------------------------------
# 10. Materialize immutable Taxonomy artifact
# --------------------------------------------------

taxonomy_materialization = (
    materialize_immutable_json(
        taxonomy_path_v1,
        taxonomy_v1_payload,
    )
)


taxonomy_v1_sha256 = (
    taxonomy_materialization[
        "sha256"
    ]
)


print(
    "\nTaxonomy v1 materialization:"
)

print(
    "- Path:",
    taxonomy_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Action:",
    taxonomy_materialization[
        "action"
    ],
)

print(
    "- SHA256:",
    taxonomy_v1_sha256,
)

print(
    "- Size bytes:",
    taxonomy_materialization[
        "size_bytes"
    ],
)


# --------------------------------------------------
# 11. Deterministic Freeze report
# --------------------------------------------------

taxonomy_freeze_report_v1 = {
    "schema_version": (
        "taxonomy-freeze-report-v1"
    ),

    "taxonomy_version": (
        "taxonomy_v1"
    ),

    "freeze_status": (
        "PASS"
    ),

    "taxonomy_artifact": {
        "path": str(
            taxonomy_path_v1.relative_to(
                repo_root
            )
        ),

        "sha256": (
            taxonomy_v1_sha256
        ),

        "records_in_source_corpus": (
            accepted_corpus_records
        ),
    },

    "source_handoff": {
        "accepted_corpus_path": str(
            accepted_corpus_path.relative_to(
                repo_root
            )
        ),

        "accepted_corpus_sha256": (
            accepted_corpus_sha256
        ),

        "acceptance_report_path": str(
            acceptance_report_path.relative_to(
                repo_root
            )
        ),

        "acceptance_report_sha256": (
            acceptance_report_sha256
        ),
    },

    "frozen_structure": {
        "top_level_topics": len(
            top_level_topics_v2
        ),

        "curriculum_nodes": len(
            curriculum_nodes_v2
        ),

        "educational_modules": len(
            educational_modules_v2
        ),

        "context_dependent_terms": len(
            context_dependent_terms_v2
        ),

        "concept_variants": len(
            concept_variants_v2
        ),

        "non_core_candidates": len(
            non_core_candidates_v2
        ),

        "valid_multi_topic_documents": len(
            valid_multi_topic_documents_v2
        ),
    },

    "authority_layers": deepcopy(
        authority_summary_v2
    ),

    "review_resolution": {
        "previous_review_items": 11,

        "corpus_resolved_items": len(
            corpus_resolution_register_v2
        ),

        "specialist_intervention_required": False,

        "previous_review_preserved": True,
    },

    "independent_audit": {
        "unsupported_curriculum_nodes": len(
            unsupported_curriculum_nodes_v2
        ),

        "children_business_bridge_valid": (
            children_business_bridge_valid
        ),

        "giving_bridge_valid": (
            giving_bridge_valid
        ),

        "taxation_module_valid": (
            taxation_module_valid
        ),

        "invalid_context_term_policies": len(
            invalid_context_term_policies_v2
        ),

        "behavioral_exclusion_valid": (
            behavioral_exclusion_valid
        ),

        "invalid_multi_topic_documents": len(
            invalid_multi_topic_documents_v2
        ),

        "hierarchy_cycles": len(
            hierarchy_cycles_v2
        ),

        "invalid_hierarchy_bridges": len(
            invalid_hierarchy_bridges_v2
        ),

        "invalid_taxonomy_ids": len(
            invalid_taxonomy_ids_v2
        ),

        "empty_taxonomy_labels": len(
            empty_taxonomy_labels_v2
        ),

        "nonprimary_core_evidence_contamination": len(
            core_taxonomy_authority_contamination_v2
        ),

        "freeze_blockers": list(
            freeze_blockers_v2
        ),
    },

    "known_nonblocking_conditions": {
        "rights_release_gate_open": True,

        "unresolved_transcript_records": len(
            unresolved_context_document_ids_v2
        ),

        "supervised_secondary_website_records": len(
            supervised_secondary_document_ids_v2
        ),

        "contextual_nonprimary_transcript_records": len(
            contextual_nonprimary_document_ids_v2
        ),
    },

    "next_required_nb10_stage": (
        "POST_FREEZE_COVERAGE_CONFLICT_"
        "AND_INVALID_LABEL_AUDIT"
    ),
}


freeze_report_materialization = (
    materialize_immutable_json(
        taxonomy_freeze_report_path_v1,
        taxonomy_freeze_report_v1,
    )
)


taxonomy_freeze_report_sha256 = (
    freeze_report_materialization[
        "sha256"
    ]
)


print(
    "\nTaxonomy Freeze Report materialization:"
)

print(
    "- Path:",
    taxonomy_freeze_report_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Action:",
    freeze_report_materialization[
        "action"
    ],
)

print(
    "- SHA256:",
    taxonomy_freeze_report_sha256,
)

print(
    "- Size bytes:",
    freeze_report_materialization[
        "size_bytes"
    ],
)


# --------------------------------------------------
# 12. Independent re-read from Disk
# --------------------------------------------------

taxonomy_v1_disk = json.loads(
    taxonomy_path_v1.read_text(
        encoding="utf-8"
    )
)


freeze_report_v1_disk = json.loads(
    taxonomy_freeze_report_path_v1.read_text(
        encoding="utf-8"
    )
)


taxonomy_disk_sha256 = (
    sha256_file(
        taxonomy_path_v1
    )
)


freeze_report_disk_sha256 = (
    sha256_file(
        taxonomy_freeze_report_path_v1
    )
)


print(
    "\nIndependent Disk re-validation:"
)

print(
    "- Taxonomy disk SHA256:",
    taxonomy_disk_sha256,
)

print(
    "- Taxonomy expected SHA256:",
    taxonomy_v1_sha256,
)

print(
    "- Freeze Report disk SHA256:",
    freeze_report_disk_sha256,
)

print(
    "- Freeze Report expected SHA256:",
    taxonomy_freeze_report_sha256,
)


# --------------------------------------------------
# 13. Independent structural validation from
#     re-read frozen artifact
# --------------------------------------------------

frozen_top_level_ids = set(
    taxonomy_v1_disk[
        "top_level_topics"
    ]
)


frozen_curriculum_nodes = (
    taxonomy_v1_disk[
        "curriculum_nodes"
    ]
)


frozen_curriculum_node_ids = set(
    frozen_curriculum_nodes
)


frozen_module_ids = set(
    taxonomy_v1_disk[
        "educational_modules"
    ]
)


frozen_duplicate_ids = (
    frozen_top_level_ids
    & frozen_curriculum_node_ids
)


frozen_duplicate_ids |= (
    frozen_top_level_ids
    & frozen_module_ids
)


frozen_duplicate_ids |= (
    frozen_curriculum_node_ids
    & frozen_module_ids
)


frozen_invalid_parents = []


for node_id, node in (
    frozen_curriculum_nodes.items()
):

    parent_id = node[
        "parent_id"
    ]

    if (
        parent_id
        not in frozen_top_level_ids
        and parent_id
        not in frozen_curriculum_node_ids
    ):
        frozen_invalid_parents.append(
            (
                node_id,
                parent_id,
            )
        )


frozen_invalid_topic_bridges = []


for node_id, node in (
    frozen_curriculum_nodes.items()
):

    if (
        node[
            "top_level_topic_id"
        ]
        not in frozen_top_level_ids
    ):
        frozen_invalid_topic_bridges.append(
            (
                node_id,
                node[
                    "top_level_topic_id"
                ],
            )
        )


# --------------------------------------------------
# 14. Freeze-state variables for downstream NB10
# --------------------------------------------------

taxonomy_v1_frozen = (
    taxonomy_v1_disk
)


taxonomy_v1_frozen_path = (
    taxonomy_path_v1
)


taxonomy_v1_frozen_sha256 = (
    taxonomy_disk_sha256
)


taxonomy_v1_freeze_report = (
    freeze_report_v1_disk
)


taxonomy_v1_freeze_report_sha256 = (
    freeze_report_disk_sha256
)


taxonomy_freeze_executed_v1 = True


# --------------------------------------------------
# 15. Final validation
# --------------------------------------------------

source_document_mutation_performed = False
topic_mapping_mutation_performed = False
attribution_mutation_performed = False
speaker_resolution_mutation_performed = False
chunking_performed = False


print(
    "\nValidation:"
)

print(
    "- Frozen Taxonomy exists:",
    taxonomy_path_v1.exists(),
)

print(
    "- Freeze Report exists:",
    taxonomy_freeze_report_path_v1.exists(),
)

print(
    "- Frozen Taxonomy status:",
    taxonomy_v1_disk[
        "status"
    ],
)

print(
    "- Top-level Topics:",
    len(
        taxonomy_v1_disk[
            "top_level_topics"
        ]
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        taxonomy_v1_disk[
            "curriculum_nodes"
        ]
    ),
)

print(
    "- Educational Modules:",
    len(
        taxonomy_v1_disk[
            "educational_modules"
        ]
    ),
)

print(
    "- Resolved Review Items:",
    len(
        taxonomy_v1_disk[
            "review_resolution"
        ][
            "corpus_resolution_register"
        ]
    ),
)

print(
    "- Frozen Duplicate IDs:",
    len(
        frozen_duplicate_ids
    ),
)

print(
    "- Frozen Invalid Parent references:",
    len(
        frozen_invalid_parents
    ),
)

print(
    "- Frozen Invalid Top-level bridges:",
    len(
        frozen_invalid_topic_bridges
    ),
)

print(
    "- Taxonomy SHA verified:",
    (
        taxonomy_disk_sha256
        == taxonomy_v1_sha256
    ),
)

print(
    "- Freeze Report SHA verified:",
    (
        freeze_report_disk_sha256
        == taxonomy_freeze_report_sha256
    ),
)

print(
    "- Source Document mutation performed:",
    source_document_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Attribution mutation performed:",
    attribution_mutation_performed,
)

print(
    "- Speaker Resolution mutation performed:",
    speaker_resolution_mutation_performed,
)

print(
    "- Chunking performed:",
    chunking_performed,
)

print(
    "- Taxonomy Freeze executed:",
    taxonomy_freeze_executed_v1,
)


assert taxonomy_path_v1.exists()

assert (
    taxonomy_freeze_report_path_v1.exists()
)

assert (
    taxonomy_disk_sha256
    == taxonomy_v1_sha256
)

assert (
    freeze_report_disk_sha256
    == taxonomy_freeze_report_sha256
)

assert (
    taxonomy_v1_disk[
        "status"
    ]
    == "FROZEN"
)

assert (
    taxonomy_v1_disk[
        "taxonomy_version"
    ]
    == "taxonomy_v1"
)

assert len(
    taxonomy_v1_disk[
        "top_level_topics"
    ]
) == 7

assert len(
    taxonomy_v1_disk[
        "curriculum_nodes"
    ]
) == 7

assert len(
    taxonomy_v1_disk[
        "educational_modules"
    ]
) == 1

assert len(
    taxonomy_v1_disk[
        "review_resolution"
    ][
        "corpus_resolution_register"
    ]
) == 11

assert not frozen_duplicate_ids
assert not frozen_invalid_parents
assert not frozen_invalid_topic_bridges

assert (
    taxonomy_v1_disk[
        "validation_snapshot"
    ][
        "unsupported_curriculum_nodes"
    ]
    == 0
)

assert (
    taxonomy_v1_disk[
        "validation_snapshot"
    ][
        "hierarchy_cycles"
    ]
    == 0
)

assert (
    taxonomy_v1_disk[
        "validation_snapshot"
    ][
        "nonprimary_core_evidence_contamination"
    ]
    == 0
)

assert (
    freeze_report_v1_disk[
        "freeze_status"
    ]
    == "PASS"
)

assert (
    freeze_report_v1_disk[
        "taxonomy_artifact"
    ][
        "sha256"
    ]
    == taxonomy_v1_sha256
)

assert source_document_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert attribution_mutation_performed is False
assert speaker_resolution_mutation_performed is False
assert chunking_performed is False

assert taxonomy_freeze_executed_v1 is True


print(
    "\nCODE CELL 38 GATE: PASS"
)

print(
    "Taxonomy v1 Freeze PASS: the Corpus-derived "
    "Taxonomy has been immutably materialized, "
    "re-read from Disk, hash-verified and "
    "structurally validated without modifying "
    "documents, mappings, attribution or chunking."
)

Repository discovery:
- Kernel CWD: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2
- Repository Root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance

NB09 accepted-corpus discovery:
- documents_dedup_quality_accepted_v1.jsonl | SHA256= 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
- documents_normalized_v1.jsonl | SHA256= 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
- dedup_cross_source_candidates_v1.jsonl | SHA256= 37f6c60093c8d64a45a5f3d148caea701dd7db4bbe126e5c044319f12c505880
- dedup_exact_relations_v1.jsonl | SHA256= 6dc003fd5eaca8661b36b87094e50a1fa03a048f7d3043e87605c456e00a041c
- dedup_near_candidates_v1.jsonl | SHA256= 4b83fcebd59b34d8b8d792ba83fb5a284b40c897ad7edd7c01685f099a9f90ad
- dedup_quality_evidence_records_v1.jsonl | SHA256= 50617f1915d600a67f738178e10655f3afcb251a1742f5bf7bf80c428436aa5a
- dedup_quality_review_queue_v1.jsonl | SHA256= 525bb513e332bb2b9ff94f65

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Post-Freeze Coverage, Conflict & Invalid-label Audit</h3>

<p>
Taxonomy v1 اکنون Freeze شده است. این مرحله بررسی می‌کند که Mappingهای موجود و Evidenceهای Phase 2
با نسخۀ Frozen سازگار باشند و هیچ Topic ID نامعتبر، Conflict تأییدنشده یا آلودگی ساختاری ایجاد نشده باشد.
</p>

<h4>سه مفهوم متفاوت</h4>

<ul>
<li><strong>Top-level Mapping Coverage:</strong> آیا Document به یکی از ۷ Topic اصلی قابل مشاهده است؟</li>
<li><strong>Structural Representation:</strong> آیا Document توسط Node یا Module دقیق‌تر Taxonomy پوشش ساختاری دارد؟</li>
<li><strong>Knowledge Gap:</strong> آیا برای یک Competency موردنیاز، دانش کافی واقعاً در Corpus وجود ندارد؟</li>
</ul>

<p>
این سه مورد نباید با هم اشتباه شوند.
یک Document می‌تواند Top-level Mapping نداشته باشد ولی همچنان دانش معتبر Primary باشد.
بنابراین Unmapped بودن به‌تنهایی Knowledge Gap محسوب نمی‌شود.
</p>

<h4>Conflict Policy</h4>

<p>
داشتن چند Topic lexical یا structural به‌تنهایی Conflict نیست.
در این Audit فقط Documentهایی که بیش از یک <code>STRONG_CANDIDATE</code> دارند بررسی می‌شوند.
دو Book Document که قبلاً Corpus-first به‌عنوان Multi-topic معتبر Resolve شده‌اند مجاز هستند؛
هر Multi-strong جدید باید به‌عنوان Conflict ظاهر شود.
</p>

<h4>Invalid-label Policy</h4>

<p>
تمام Topic IDهایی که در Candidate Mapping، Book Hierarchical Scope یا Conversation Scope استفاده شده‌اند
باید در ۷ Top-level Topic نسخۀ Frozen وجود داشته باشند.
هدف نهایی این Audit رسیدن به <strong>Invalid Labels = 0</strong> است.
</p>

<h4>این Cell انجام نمی‌دهد</h4>

<ul>
<li>Taxonomy را تغییر نمی‌دهد.</li>
<li>Mapping را تغییر نمی‌دهد.</li>
<li>Unmapped Document را به زور Map نمی‌کند.</li>
<li>Knowledge Gap اعلام نمی‌کند.</li>
<li>فایل جدیدی روی Disk نمی‌نویسد.</li>
</ul>

</div>

In [82]:
# Code Cell 39

from collections import Counter, defaultdict
import json


# --------------------------------------------------
# 1. Frozen Taxonomy integrity
# --------------------------------------------------

assert taxonomy_freeze_executed_v1 is True
assert taxonomy_v1_frozen_path.exists()


frozen_taxonomy_sha_before_audit = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


assert (
    frozen_taxonomy_sha_before_audit
    == taxonomy_v1_frozen_sha256
)


frozen_taxonomy_postfreeze_v1 = json.loads(
    taxonomy_v1_frozen_path.read_text(
        encoding="utf-8"
    )
)


assert (
    frozen_taxonomy_postfreeze_v1[
        "taxonomy_version"
    ]
    == "taxonomy_v1"
)

assert (
    frozen_taxonomy_postfreeze_v1[
        "status"
    ]
    == "FROZEN"
)


frozen_top_level_topics_v1 = (
    frozen_taxonomy_postfreeze_v1[
        "top_level_topics"
    ]
)


frozen_curriculum_nodes_v1 = (
    frozen_taxonomy_postfreeze_v1[
        "curriculum_nodes"
    ]
)


frozen_educational_modules_v1 = (
    frozen_taxonomy_postfreeze_v1[
        "educational_modules"
    ]
)


frozen_top_level_ids_v1 = set(
    frozen_top_level_topics_v1
)


print(
    "Frozen Taxonomy Integrity:"
)

print(
    "- Path:",
    taxonomy_v1_frozen_path.relative_to(
        repo_root
    ),
)

print(
    "- SHA256:",
    frozen_taxonomy_sha_before_audit,
)

print(
    "- Status:",
    frozen_taxonomy_postfreeze_v1[
        "status"
    ],
)

print(
    "- Top-level Topics:",
    len(
        frozen_top_level_ids_v1
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        frozen_curriculum_nodes_v1
    ),
)

print(
    "- Educational Modules:",
    len(
        frozen_educational_modules_v1
    ),
)


# --------------------------------------------------
# 2. Independent Mapping-reference extraction
# --------------------------------------------------

mapping_reference_audit_v1 = {}

all_mapping_topic_ids_v1 = set()

candidate_topic_ids_v1 = set()
book_scope_topic_ids_v1 = set()
conversation_scope_topic_ids_v1 = set()

candidate_decision_counter_v1 = Counter()


for document_id, profile in (
    document_topic_mapping_v2_candidate.items()
):

    candidate_topics = set()
    strong_topics = set()
    book_scope_topics = set()
    conversation_topics = set()

    for candidate in profile.get(
        "candidates",
        [],
    ):

        topic_id = candidate.get(
            "topic_id"
        )

        decision = candidate.get(
            "decision"
        )

        candidate_decision_counter_v1[
            str(
                decision
            )
        ] += 1

        if topic_id is None:
            continue

        candidate_topics.add(
            topic_id
        )

        candidate_topic_ids_v1.add(
            topic_id
        )

        all_mapping_topic_ids_v1.add(
            topic_id
        )

        if (
            decision
            == "STRONG_CANDIDATE"
        ):
            strong_topics.add(
                topic_id
            )

    for scope in profile.get(
        "book_hierarchical_topic_scope",
        [],
    ):

        topic_id = scope.get(
            "topic_id"
        )

        if topic_id is None:
            continue

        book_scope_topics.add(
            topic_id
        )

        book_scope_topic_ids_v1.add(
            topic_id
        )

        all_mapping_topic_ids_v1.add(
            topic_id
        )

    conversation_scope = profile.get(
        "conversation_topic_scope"
    )

    if conversation_scope is not None:

        topic_id = (
            conversation_scope.get(
                "topic_id"
            )
        )

        if topic_id is not None:

            conversation_topics.add(
                topic_id
            )

            conversation_scope_topic_ids_v1.add(
                topic_id
            )

            all_mapping_topic_ids_v1.add(
                topic_id
            )

    mapping_reference_audit_v1[
        document_id
    ] = {
        "candidate_topics": sorted(
            candidate_topics
        ),

        "strong_topics": sorted(
            strong_topics
        ),

        "book_scope_topics": sorted(
            book_scope_topics
        ),

        "conversation_scope_topics": sorted(
            conversation_topics
        ),
    }


print(
    "\nMapping Reference Inventory:"
)

print(
    "- Candidate Topic IDs:",
    len(
        candidate_topic_ids_v1
    ),
)

print(
    "- Book-scope Topic IDs:",
    len(
        book_scope_topic_ids_v1
    ),
)

print(
    "- Conversation-scope Topic IDs:",
    len(
        conversation_scope_topic_ids_v1
    ),
)

print(
    "- Distinct Topic IDs anywhere in Mapping:",
    len(
        all_mapping_topic_ids_v1
    ),
)


print(
    "\nCandidate decision inventory:"
)

for decision, count in (
    candidate_decision_counter_v1.most_common()
):

    print(
        f"- {decision}: {count}"
    )


# --------------------------------------------------
# 3. Invalid Top-level label audit
# --------------------------------------------------

invalid_candidate_topic_ids_v1 = sorted(
    candidate_topic_ids_v1
    - frozen_top_level_ids_v1
)


invalid_book_scope_topic_ids_v1 = sorted(
    book_scope_topic_ids_v1
    - frozen_top_level_ids_v1
)


invalid_conversation_topic_ids_v1 = sorted(
    conversation_scope_topic_ids_v1
    - frozen_top_level_ids_v1
)


invalid_mapping_topic_ids_v1 = sorted(
    all_mapping_topic_ids_v1
    - frozen_top_level_ids_v1
)


print(
    "\nInvalid-label Audit:"
)

print(
    "- Invalid Candidate Topic IDs:",
    len(
        invalid_candidate_topic_ids_v1
    ),
)

print(
    "- Invalid Book-scope Topic IDs:",
    len(
        invalid_book_scope_topic_ids_v1
    ),
)

print(
    "- Invalid Conversation-scope Topic IDs:",
    len(
        invalid_conversation_topic_ids_v1
    ),
)

print(
    "- Invalid Mapping Topic IDs total:",
    len(
        invalid_mapping_topic_ids_v1
    ),
)


if invalid_mapping_topic_ids_v1:

    print(
        "- Invalid IDs:",
        invalid_mapping_topic_ids_v1,
    )


# --------------------------------------------------
# 4. Source-aware Strong Multi-topic audit
#
# IMPORTANT:
# >1 STRONG_CANDIDATE is NOT automatically conflict.
#
# Website:
#   Multi-topic documents are allowed.
#
# Book:
#   Because explicit curriculum hierarchy exists,
#   multi-strong documents must match the
#   Corpus Resolution Register.
#
# Transcript:
#   A future multi-strong result is review-required,
#   because Transcript context propagation must not
#   silently become a structural multi-label claim.
# --------------------------------------------------

multi_topic_policy_v1 = {
    "website": (
        "SOURCE_MULTI_TOPIC_ALLOWED"
    ),

    "book": (
        "REQUIRES_CORPUS_RESOLUTION_REGISTER"
    ),

    "transcript": (
        "REQUIRES_REVIEW_IF_MULTI_STRONG"
    ),
}


multi_strong_documents_v1 = {}


for document_id, audit in (
    mapping_reference_audit_v1.items()
):

    strong_topics = set(
        audit[
            "strong_topics"
        ]
    )

    if len(
        strong_topics
    ) <= 1:
        continue

    source_type = (
        record_lookup[
            document_id
        ][
            "source_type"
        ]
    )

    authority_class = (
        knowledge_authority_profile_v2[
            document_id
        ][
            "authority_class"
        ]
    )

    multi_strong_documents_v1[
        document_id
    ] = {
        "source_type": (
            source_type
        ),

        "authority_class": (
            authority_class
        ),

        "strong_topics": sorted(
            strong_topics
        ),

        "topic_count": len(
            strong_topics
        ),
    }


multi_strong_by_source_v1 = Counter(
    row[
        "source_type"
    ]
    for row in (
        multi_strong_documents_v1.values()
    )
)


print(
    "\nStrong Multi-topic Source Inventory:"
)

print(
    "- Documents with >1 Strong Topic:",
    len(
        multi_strong_documents_v1
    ),
)

for source_type in [
    "book",
    "website",
    "transcript",
]:

    print(
        f"- {source_type}:",
        multi_strong_by_source_v1[
            source_type
        ],
    )


# --------------------------------------------------
# 5. Book multi-strong validation
# --------------------------------------------------

approved_book_multi_strong_v1 = {}

invalid_book_multi_strong_v1 = {}


for document_id, row in (
    multi_strong_documents_v1.items()
):

    if (
        row[
            "source_type"
        ]
        != "book"
    ):
        continue

    actual_topics = set(
        row[
            "strong_topics"
        ]
    )

    if (
        document_id
        not in valid_multi_topic_documents_v2
    ):

        invalid_book_multi_strong_v1[
            document_id
        ] = {
            "reason": (
                "BOOK_MULTI_STRONG_NOT_IN_"
                "CORPUS_RESOLUTION_REGISTER"
            ),

            "actual": sorted(
                actual_topics
            ),
        }

        continue

    expected_topics = set(
        valid_multi_topic_documents_v2[
            document_id
        ][
            "topics"
        ]
    )

    if (
        actual_topics
        != expected_topics
    ):

        invalid_book_multi_strong_v1[
            document_id
        ] = {
            "reason": (
                "BOOK_MULTI_STRONG_TOPIC_SET_"
                "DOES_NOT_MATCH_RESOLUTION"
            ),

            "expected": sorted(
                expected_topics
            ),

            "actual": sorted(
                actual_topics
            ),
        }

        continue

    approved_book_multi_strong_v1[
        document_id
    ] = {
        "expected": sorted(
            expected_topics
        ),

        "actual": sorted(
            actual_topics
        ),

        "resolution_status": (
            "VALID_MULTI_TOPIC"
        ),
    }


print(
    "\nBook Strong Multi-topic Audit:"
)

print(
    "- Book Multi-strong documents:",
    multi_strong_by_source_v1[
        "book"
    ],
)

print(
    "- Approved Book Multi-topic:",
    len(
        approved_book_multi_strong_v1
    ),
)

print(
    "- Invalid Book Multi-topic:",
    len(
        invalid_book_multi_strong_v1
    ),
)


for document_id, row in (
    approved_book_multi_strong_v1.items()
):

    print(
        "\n- APPROVED BOOK:",
        document_id,
    )

    print(
        "  expected:",
        row[
            "expected"
        ],
    )

    print(
        "  actual:",
        row[
            "actual"
        ],
    )


# --------------------------------------------------
# 6. Website multi-strong validation
#
# Website is a naturally multi-topic document source.
# Strong evidence can legitimately support multiple
# Top-level Topics in one article.
# --------------------------------------------------

valid_website_multi_strong_v1 = {}


for document_id, row in (
    multi_strong_documents_v1.items()
):

    if (
        row[
            "source_type"
        ]
        != "website"
    ):
        continue

    strong_topics = set(
        row[
            "strong_topics"
        ]
    )

    invalid_topics = (
        strong_topics
        - frozen_top_level_ids_v1
    )

    if invalid_topics:

        continue

    valid_website_multi_strong_v1[
        document_id
    ] = {
        "strong_topics": sorted(
            strong_topics
        ),

        "topic_count": len(
            strong_topics
        ),

        "authority_class": (
            row[
                "authority_class"
            ]
        ),

        "status": (
            "VALID_SOURCE_MULTI_TOPIC"
        ),
    }


website_multi_strong_authority_v1 = Counter(
    row[
        "authority_class"
    ]
    for row in (
        valid_website_multi_strong_v1.values()
    )
)


website_multi_strong_cardinality_v1 = Counter(
    row[
        "topic_count"
    ]
    for row in (
        valid_website_multi_strong_v1.values()
    )
)


print(
    "\nWebsite Strong Multi-topic Audit:"
)

print(
    "- Website Multi-strong documents:",
    multi_strong_by_source_v1[
        "website"
    ],
)

print(
    "- Valid Website Multi-topic:",
    len(
        valid_website_multi_strong_v1
    ),
)


print(
    "Website Multi-topic by Authority:"
)

for authority_class, count in (
    website_multi_strong_authority_v1.most_common()
):

    print(
        f"- {authority_class}: {count}"
    )


print(
    "Website Strong-topic cardinality:"
)

for topic_count, count in sorted(
    website_multi_strong_cardinality_v1.items()
):

    print(
        f"- {topic_count} Topics: {count} documents"
    )


# --------------------------------------------------
# 7. Transcript / unknown-source multi-strong
#    conflict blockers
# --------------------------------------------------

review_required_multi_strong_v1 = {}


for document_id, row in (
    multi_strong_documents_v1.items()
):

    source_type = row[
        "source_type"
    ]

    if (
        source_type
        == "transcript"
    ):

        review_required_multi_strong_v1[
            document_id
        ] = {
            "reason": (
                "TRANSCRIPT_MULTI_STRONG_"
                "REQUIRES_CONTEXT_REVIEW"
            ),

            "actual": row[
                "strong_topics"
            ],
        }

    elif (
        source_type
        not in {
            "book",
            "website",
            "transcript",
        }
    ):

        review_required_multi_strong_v1[
            document_id
        ] = {
            "reason": (
                "UNKNOWN_SOURCE_MULTI_STRONG"
            ),

            "source_type": (
                source_type
            ),

            "actual": row[
                "strong_topics"
            ],
        }


# --------------------------------------------------
# 8. Source-aware Conflict result
# --------------------------------------------------

unapproved_multi_strong_documents_v1 = {}


unapproved_multi_strong_documents_v1.update(
    invalid_book_multi_strong_v1
)


unapproved_multi_strong_documents_v1.update(
    review_required_multi_strong_v1
)


accounted_multi_strong_count_v1 = (
    len(
        approved_book_multi_strong_v1
    )
    + len(
        valid_website_multi_strong_v1
    )
    + len(
        unapproved_multi_strong_documents_v1
    )
)


print(
    "\nSource-aware Strong Multi-topic Result:"
)

print(
    "- Total Multi-strong:",
    len(
        multi_strong_documents_v1
    ),
)

print(
    "- Approved Book Multi-topic:",
    len(
        approved_book_multi_strong_v1
    ),
)

print(
    "- Valid Website Multi-topic:",
    len(
        valid_website_multi_strong_v1
    ),
)

print(
    "- Review-required / invalid:",
    len(
        unapproved_multi_strong_documents_v1
    ),
)

print(
    "- Accounted Multi-strong:",
    accounted_multi_strong_count_v1,
)


if unapproved_multi_strong_documents_v1:

    print(
        "\nRemaining Multi-strong blockers:"
    )

    for document_id, row in (
        unapproved_multi_strong_documents_v1.items()
    ):

        print(
            "-",
            document_id,
            row,
        )


# --------------------------------------------------
# 9. Independent Top-level observability
#    across all Knowledge Authority layers
# --------------------------------------------------

authority_observability_v1 = defaultdict(
    Counter
)


core_source_observability_v1 = defaultdict(
    Counter
)


all_documents_with_observed_topic_v1 = set()
all_documents_without_observed_topic_v1 = set()


for record in records:

    document_id = record[
        "document_id"
    ]

    source_type = record[
        "source_type"
    ]

    authority_class = (
        knowledge_authority_profile_v2[
            document_id
        ][
            "authority_class"
        ]
    )

    observed_topic_ids = (
        observed_topics_v2(
            document_id
        )
    )

    authority_observability_v1[
        authority_class
    ][
        "total"
    ] += 1

    if observed_topic_ids:

        all_documents_with_observed_topic_v1.add(
            document_id
        )

        authority_observability_v1[
            authority_class
        ][
            "observed"
        ] += 1

    else:

        all_documents_without_observed_topic_v1.add(
            document_id
        )

        authority_observability_v1[
            authority_class
        ][
            "unobserved"
        ] += 1

    if (
        authority_class
        == "CORE_PRIMARY_KNOWLEDGE"
    ):

        core_source_observability_v1[
            source_type
        ][
            "total"
        ] += 1

        if observed_topic_ids:

            core_source_observability_v1[
                source_type
            ][
                "observed"
            ] += 1

        else:

            core_source_observability_v1[
                source_type
            ][
                "unobserved"
            ] += 1


print(
    "\nTop-level Observability by Knowledge Authority:"
)


for authority_class in [
    "CORE_PRIMARY_KNOWLEDGE",
    "SUPERVISED_SECONDARY_KNOWLEDGE",
    "CONTEXTUAL_NONPRIMARY",
    "UNRESOLVED_CONTEXT",
]:

    row = (
        authority_observability_v1[
            authority_class
        ]
    )

    total = row[
        "total"
    ]

    observed = row[
        "observed"
    ]

    share = (
        observed / total
        if total
        else 0
    )

    print(
        f"\n[{authority_class}]"
    )

    print(
        "- Total:",
        total,
    )

    print(
        "- Observed:",
        observed,
    )

    print(
        "- Unobserved:",
        row[
            "unobserved"
        ],
    )

    print(
        "- Observable share:",
        round(
            share,
            4,
        ),
    )


print(
    "\nCore Primary observability by Source:"
)


for source_type in [
    "book",
    "website",
    "transcript",
]:

    row = (
        core_source_observability_v1[
            source_type
        ]
    )

    total = row[
        "total"
    ]

    observed = row[
        "observed"
    ]

    share = (
        observed / total
        if total
        else 0
    )

    print(
        f"\n[{source_type}]"
    )

    print(
        "- Total Core Primary:",
        total,
    )

    print(
        "- Observed:",
        observed,
    )

    print(
        "- Unobserved:",
        row[
            "unobserved"
        ],
    )

    print(
        "- Observable share:",
        round(
            share,
            4,
        ),
    )


# --------------------------------------------------
# 10. Frozen Top-level Topic evidence
# --------------------------------------------------

frozen_topic_evidence_core_v1 = Counter()
frozen_topic_evidence_all_v1 = Counter()


for record in records:

    document_id = record[
        "document_id"
    ]

    observed_topic_ids = (
        observed_topics_v2(
            document_id
        )
    )

    for topic_id in (
        observed_topic_ids
    ):

        frozen_topic_evidence_all_v1[
            topic_id
        ] += 1

        if (
            document_id
            in core_primary_document_ids_v2
        ):

            frozen_topic_evidence_core_v1[
                topic_id
            ] += 1


print(
    "\nFrozen Top-level Topic Evidence:"
)


frozen_topics_without_core_evidence_v1 = []


for topic_id, topic in (
    frozen_top_level_topics_v1.items()
):

    core_count = (
        frozen_topic_evidence_core_v1[
            topic_id
        ]
    )

    all_count = (
        frozen_topic_evidence_all_v1[
            topic_id
        ]
    )

    print(
        "-",
        topic[
            "label_fa"
        ],
        "| core_primary=",
        core_count,
        "| all_layers=",
        all_count,
    )

    if core_count == 0:

        frozen_topics_without_core_evidence_v1.append(
            topic_id
        )


# --------------------------------------------------
# 11. Structural representation among
#     Core Primary unobserved docs
# --------------------------------------------------

core_primary_unobserved_ids_v1 = (
    set(
        core_primary_document_ids_v2
    )
    - set(
        all_documents_with_observed_topic_v1
    )
)


core_primary_observed_ids_v1 = (
    set(
        core_primary_document_ids_v2
    )
    & set(
        all_documents_with_observed_topic_v1
    )
)


structurally_represented_but_top_level_unobserved_v1 = (
    core_primary_unobserved_ids_v1
    & structured_or_module_primary_ids_v2
)


structurally_represented_and_top_level_observed_v1 = (
    core_primary_observed_ids_v1
    & structured_or_module_primary_ids_v2
)


print(
    "\nCore Primary Representation Diagnostics:"
)

print(
    "- Core Primary documents:",
    len(
        core_primary_document_ids_v2
    ),
)

print(
    "- Top-level observed:",
    len(
        core_primary_observed_ids_v1
    ),
)

print(
    "- Top-level unobserved:",
    len(
        core_primary_unobserved_ids_v1
    ),
)

print(
    "- New Node/Module represented AND "
    "Top-level observed:",
    len(
        structurally_represented_and_top_level_observed_v1
    ),
)

print(
    "- New Node/Module represented BUT "
    "Top-level unobserved:",
    len(
        structurally_represented_but_top_level_unobserved_v1
    ),
)


core_unobserved_by_source_v1 = Counter(
    record_lookup[
        document_id
    ][
        "source_type"
    ]
    for document_id in (
        core_primary_unobserved_ids_v1
    )
)


print(
    "\nTop-level-unobserved Core Primary by Source:"
)


for source_type in [
    "book",
    "website",
    "transcript",
]:

    print(
        "-",
        source_type,
        ":",
        core_unobserved_by_source_v1[
            source_type
        ],
    )


# --------------------------------------------------
# 12. Frozen active-label uniqueness
# --------------------------------------------------

active_taxonomy_labels_v1 = []


for taxonomy_id, row in (
    frozen_top_level_topics_v1.items()
):

    active_taxonomy_labels_v1.append(
        (
            taxonomy_id,
            normalize_fa_for_match(
                row[
                    "label_fa"
                ]
            ),
        )
    )


for taxonomy_id, row in (
    frozen_curriculum_nodes_v1.items()
):

    active_taxonomy_labels_v1.append(
        (
            taxonomy_id,
            normalize_fa_for_match(
                row[
                    "label_fa"
                ]
            ),
        )
    )


for taxonomy_id, row in (
    frozen_educational_modules_v1.items()
):

    active_taxonomy_labels_v1.append(
        (
            taxonomy_id,
            normalize_fa_for_match(
                row[
                    "label_fa"
                ]
            ),
        )
    )


taxonomy_label_to_ids_v1 = defaultdict(
    list
)


for taxonomy_id, normalized_label in (
    active_taxonomy_labels_v1
):

    taxonomy_label_to_ids_v1[
        normalized_label
    ].append(
        taxonomy_id
    )


duplicate_active_taxonomy_labels_v1 = {
    label: ids
    for label, ids in (
        taxonomy_label_to_ids_v1.items()
    )
    if len(ids) > 1
}


print(
    "\nFrozen Active-label Audit:"
)

print(
    "- Active Topic/Node/Module labels:",
    len(
        active_taxonomy_labels_v1
    ),
)

print(
    "- Duplicate normalized active labels:",
    len(
        duplicate_active_taxonomy_labels_v1
    ),
)


# --------------------------------------------------
# 13. Frozen hierarchy references
# --------------------------------------------------

invalid_frozen_node_parents_v1 = []
invalid_frozen_node_bridges_v1 = []


frozen_curriculum_ids_v1 = set(
    frozen_curriculum_nodes_v1
)


for node_id, node in (
    frozen_curriculum_nodes_v1.items()
):

    parent_id = node.get(
        "parent_id"
    )

    top_level_topic_id = node.get(
        "top_level_topic_id"
    )

    if (
        parent_id
        not in frozen_top_level_ids_v1
        and parent_id
        not in frozen_curriculum_ids_v1
    ):

        invalid_frozen_node_parents_v1.append(
            (
                node_id,
                parent_id,
            )
        )

    if (
        top_level_topic_id
        not in frozen_top_level_ids_v1
    ):

        invalid_frozen_node_bridges_v1.append(
            (
                node_id,
                top_level_topic_id,
            )
        )


invalid_frozen_module_references_v1 = []


for module_id, module in (
    frozen_educational_modules_v1.items()
):

    parent_id = module.get(
        "parent_id"
    )

    top_level_topic_id = module.get(
        "top_level_topic_id"
    )

    if (
        parent_id is not None
        and parent_id
        not in frozen_top_level_ids_v1
        and parent_id
        not in frozen_curriculum_ids_v1
    ):

        invalid_frozen_module_references_v1.append(
            (
                module_id,
                "parent_id",
                parent_id,
            )
        )

    if (
        top_level_topic_id is not None
        and top_level_topic_id
        not in frozen_top_level_ids_v1
    ):

        invalid_frozen_module_references_v1.append(
            (
                module_id,
                "top_level_topic_id",
                top_level_topic_id,
            )
        )


# --------------------------------------------------
# 14. Knowledge-gap interpretation guardrail
# --------------------------------------------------

knowledge_gap_policy_v1 = {
    "top_level_unobserved_is_knowledge_gap": False,

    "mapping_failure_is_knowledge_gap": False,

    "lower_authority_only_evidence_proves_gap": False,

    "knowledge_gap_requires": [
        "TARGET_COMPETENCY_REQUIREMENT",
        "INSUFFICIENT_PRIMARY_OR_ACCEPTABLE_SUPPORT_EVIDENCE",
        "RETRIEVAL_FAILURE_RULED_OUT",
        "ANSWERABILITY_OR_EDUCATIONAL_COVERAGE_EVALUATION",
    ],

    "current_nb10_gap_status": (
        "KNOWLEDGE_GAPS_NOT_YET_ESTABLISHED"
    ),
}


print(
    "\nKnowledge-gap Guardrail:"
)


for key, value in (
    knowledge_gap_policy_v1.items()
):

    print(
        f"- {key}: {value}"
    )


# --------------------------------------------------
# 15. Post-Freeze audit object
# --------------------------------------------------

post_freeze_taxonomy_audit_v1 = {
    "taxonomy_version": (
        "taxonomy_v1"
    ),

    "taxonomy_sha256": (
        frozen_taxonomy_sha_before_audit
    ),

    "documents_total": len(
        records
    ),

    "invalid_labels": {
        "candidate_topic_ids": (
            invalid_candidate_topic_ids_v1
        ),

        "book_scope_topic_ids": (
            invalid_book_scope_topic_ids_v1
        ),

        "conversation_scope_topic_ids": (
            invalid_conversation_topic_ids_v1
        ),

        "all_invalid_mapping_topic_ids": (
            invalid_mapping_topic_ids_v1
        ),
    },

    "multi_topic_policy": (
        multi_topic_policy_v1
    ),

    "multi_topic_audit": {
        "total_multi_strong": (
            len(
                multi_strong_documents_v1
            )
        ),

        "by_source": dict(
            multi_strong_by_source_v1
        ),

        "approved_book_multi_topic": (
            approved_book_multi_strong_v1
        ),

        "valid_website_multi_topic": (
            valid_website_multi_strong_v1
        ),

        "unapproved_or_review_required": (
            unapproved_multi_strong_documents_v1
        ),
    },

    "coverage": {
        "core_primary_total": len(
            core_primary_document_ids_v2
        ),

        "core_primary_top_level_observed": len(
            core_primary_observed_ids_v1
        ),

        "core_primary_top_level_unobserved": len(
            core_primary_unobserved_ids_v1
        ),

        "structurally_represented_but_top_level_unobserved": len(
            structurally_represented_but_top_level_unobserved_v1
        ),
    },

    "frozen_topics_without_core_evidence": (
        frozen_topics_without_core_evidence_v1
    ),

    "duplicate_active_taxonomy_labels": (
        duplicate_active_taxonomy_labels_v1
    ),

    "invalid_frozen_node_parents": (
        invalid_frozen_node_parents_v1
    ),

    "invalid_frozen_node_bridges": (
        invalid_frozen_node_bridges_v1
    ),

    "invalid_frozen_module_references": (
        invalid_frozen_module_references_v1
    ),

    "knowledge_gap_policy": (
        knowledge_gap_policy_v1
    ),
}


# --------------------------------------------------
# 16. Freeze immutability re-check
# --------------------------------------------------

frozen_taxonomy_sha_after_audit = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


freeze_artifact_unchanged_v1 = (
    frozen_taxonomy_sha_after_audit
    == frozen_taxonomy_sha_before_audit
    == taxonomy_v1_frozen_sha256
)


# --------------------------------------------------
# 17. Mutation / write declarations
# --------------------------------------------------

taxonomy_mutation_performed = False
topic_mapping_mutation_performed = False
authority_mutation_performed = False
document_mutation_performed = False
disk_write_performed = False


# --------------------------------------------------
# 18. Final validation
# --------------------------------------------------

print(
    "\nPost-Freeze Validation:"
)

print(
    "- Frozen Taxonomy SHA unchanged:",
    freeze_artifact_unchanged_v1,
)

print(
    "- Invalid Mapping Topic IDs:",
    len(
        invalid_mapping_topic_ids_v1
    ),
)

print(
    "- Frozen Topics without Core evidence:",
    len(
        frozen_topics_without_core_evidence_v1
    ),
)

print(
    "- Multi-strong documents:",
    len(
        multi_strong_documents_v1
    ),
)

print(
    "- Valid Website Multi-topic:",
    len(
        valid_website_multi_strong_v1
    ),
)

print(
    "- Approved Book Multi-topic:",
    len(
        approved_book_multi_strong_v1
    ),
)

print(
    "- Unapproved / Review-required "
    "Multi-strong:",
    len(
        unapproved_multi_strong_documents_v1
    ),
)

print(
    "- Duplicate Active Taxonomy labels:",
    len(
        duplicate_active_taxonomy_labels_v1
    ),
)

print(
    "- Invalid Frozen Node parents:",
    len(
        invalid_frozen_node_parents_v1
    ),
)

print(
    "- Invalid Frozen Node bridges:",
    len(
        invalid_frozen_node_bridges_v1
    ),
)

print(
    "- Invalid Frozen Module references:",
    len(
        invalid_frozen_module_references_v1
    ),
)

print(
    "- Core Primary Top-level observed:",
    len(
        core_primary_observed_ids_v1
    ),
)

print(
    "- Core Primary Top-level unobserved:",
    len(
        core_primary_unobserved_ids_v1
    ),
)

print(
    "- Knowledge Gap established by this audit:",
    False,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Authority mutation performed:",
    authority_mutation_performed,
)

print(
    "- Document mutation performed:",
    document_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


# --------------------------------------------------
# 19. Assertions
# --------------------------------------------------

assert len(
    records
) == 19456


assert len(
    frozen_top_level_ids_v1
) == 7


assert not invalid_mapping_topic_ids_v1


assert not frozen_topics_without_core_evidence_v1


# Every multi-strong document must now be accounted
# for by source-aware policy.
assert (
    accounted_multi_strong_count_v1
    == len(
        multi_strong_documents_v1
    )
)


# Book is structurally constrained and must match
# the explicit Corpus Resolution Register.
assert not invalid_book_multi_strong_v1


assert (
    len(
        approved_book_multi_strong_v1
    )
    == len(
        valid_multi_topic_documents_v2
    )
)


# Every Website multi-strong document is allowed as
# source-level multi-topic only if all IDs are valid.
assert (
    len(
        valid_website_multi_strong_v1
    )
    == multi_strong_by_source_v1[
        "website"
    ]
)


# Transcript multi-strong remains review-required.
# Current accepted mapping should have none.
assert not review_required_multi_strong_v1


assert not unapproved_multi_strong_documents_v1


assert not duplicate_active_taxonomy_labels_v1


assert not invalid_frozen_node_parents_v1


assert not invalid_frozen_node_bridges_v1


assert not invalid_frozen_module_references_v1


assert freeze_artifact_unchanged_v1 is True


assert (
    knowledge_gap_policy_v1[
        "top_level_unobserved_is_knowledge_gap"
    ]
    is False
)


assert (
    knowledge_gap_policy_v1[
        "mapping_failure_is_knowledge_gap"
    ]
    is False
)


assert taxonomy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert authority_mutation_performed is False
assert document_mutation_performed is False
assert disk_write_performed is False


print(
    "\nCODE CELL 39 GATE: PASS"
)

print(
    "Post-Freeze Coverage, Conflict & Invalid-label "
    "Audit PASS: all Mapping Topic IDs are valid "
    "against frozen taxonomy_v1; Website multi-topic "
    "documents are correctly treated as valid "
    "source-level multi-label evidence, the two "
    "Book multi-topic documents match the Corpus "
    "Resolution Register, no Transcript multi-strong "
    "conflicts exist, and no Top-level-unobserved "
    "document was misclassified as a Knowledge Gap."
)

Frozen Taxonomy Integrity:
- Path: artifacts/phase2/taxonomy/taxonomy_v1.json
- SHA256: 335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977
- Status: FROZEN
- Top-level Topics: 7
- Curriculum Nodes: 7
- Educational Modules: 1

Mapping Reference Inventory:
- Candidate Topic IDs: 7
- Book-scope Topic IDs: 7
- Conversation-scope Topic IDs: 7
- Distinct Topic IDs anywhere in Mapping: 7

Candidate decision inventory:
- LEXICAL_CANDIDATE: 5691
- STRONG_CANDIDATE: 1549

Invalid-label Audit:
- Invalid Candidate Topic IDs: 0
- Invalid Book-scope Topic IDs: 0
- Invalid Conversation-scope Topic IDs: 0
- Invalid Mapping Topic IDs total: 0

Strong Multi-topic Source Inventory:
- Documents with >1 Strong Topic: 73
- book: 2
- website: 71
- transcript: 0

Book Strong Multi-topic Audit:
- Book Multi-strong documents: 2
- Approved Book Multi-topic: 2
- Invalid Book Multi-topic: 0

- APPROVED BOOK: book:trainer-training-financial-literacy:fa-source:p000138
  expected: ['financial_decision_m

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>جمع‌بندی حل مسائل اصلی سازمان‌دهی دانش و Taxonomy</h2>

<p>
در مراحل ابتدایی Notebook 10 چند مسئلۀ اساسی در سازمان‌دهی Corpus وجود داشت که در صورت حل‌نشدن،
می‌توانست Taxonomy، Retrieval و در نهایت پاسخ‌های RAG را دچار خطا کند.
پس از Deep Corpus Review، Authority Reconciliation و Independent Validation،
بخش اصلی این مسائل اکنون حل شده است.
</p>

<h3>1. تفکیک دانش اصلی دکتر رودی از سایر محتواها — حل شد</h3>

<p>
در Corpus، تمام Transcriptها و Websiteها از یک سطح Authority برخوردار نبودند.
اگر تمام این محتواها به‌صورت یکسان وارد Taxonomy می‌شدند،
حرف مهمان، مجری، شرکت‌کننده یا نویسندۀ ثانویه می‌توانست ساختار دانش اصلی دکتر رودی را تغییر دهد.
</p>

<p>
با بازسازی Authority بر اساس NB09، چهار لایۀ مستقل ایجاد و Validate شد:
</p>

<ul>
<li><strong>CORE_PRIMARY_KNOWLEDGE:</strong> 14,705 سند</li>
<li><strong>SUPERVISED_SECONDARY_KNOWLEDGE:</strong> 453 سند</li>
<li><strong>CONTEXTUAL_NONPRIMARY:</strong> 1,944 سند</li>
<li><strong>UNRESOLVED_CONTEXT:</strong> 2,354 سند</li>
</ul>

<p>
از این مرحله به بعد فقط Core Primary Knowledge حق تعریف یا تغییر ساختار اصلی Taxonomy را دارد.
لایه‌های دیگر حذف نشده‌اند، ولی نقش آن‌ها Supporting یا Contextual است.
</p>

<h3>2. مشکل Speaker Attribution در Transcript — برای Taxonomy حل شد</h3>

<p>
در ابتدا 5,016 رکورد Transcript فاقد Speaker canonical در Attribution سطح بالا بودند.
بررسی عمیق نشان داد اطلاعات واقعی Speaker داخل
<code>source_metadata.speaker</code>
قرار دارد.
</p>

<p>
Authority Transcript دقیقاً با NB09 Reconcile شد:
</p>

<ul>
<li>PRIMARY_CANONICAL_ATTRIBUTION: 11,209</li>
<li>PRIMARY_EXPLICIT_SOURCE_MARKER: 718</li>
<li>PRIMARY_HUMAN_CONFIRMED_OVERLAY: 1</li>
<li>SECONDARY_HUMAN_CONFIRMED_OVERLAY: 1</li>
<li>NONPRIMARY_EXPLICIT_CONTEXT: 1,943</li>
<li>UNRESOLVED_SPEAKER_CONTEXT: 2,354</li>
</ul>

<p>
بنابراین Transcriptهای نامشخص یا Non-primary دیگر نمی‌توانند ساختار دانش اصلی را تعریف کنند.
2,354 رکورد Unresolved همچنان برای Context حفظ شده‌اند و حذف نشده‌اند.
</p>

<h3>3. ساختار Top-level Taxonomy — حل و Freeze شد</h3>

<p>
پس از بررسی Book، Website و Transcript Primary،
هفت Topic اصلی حفظ و به‌عنوان ساختار Top-level پذیرفته شدند:
</p>

<ul>
<li>مبانی و توانمندی سواد مالی</li>
<li>خرج، پس‌انداز و مدیریت پول</li>
<li>اعتبار، قرض، بدهی و بانک</li>
<li>کار، شغل و درآمد</li>
<li>سرمایه‌گذاری</li>
<li>ریسک و بیمه</li>
<li>تصمیم‌گیری مالی</li>
</ul>

<p>
هیچ Evidence معتبری برای افزودن یا حذف Top-level Topic در Taxonomy v1 پیدا نشد.
</p>

<h3>4. مشکل Parent/Subtopicهای گم‌شده — حل شد</h3>

<p>
Deep Book Hierarchy Audit نشان داد بعضی مفاهیمی که در Mapping اولیه مستقیماً
به Top-level Topic متصل شده بودند، در ساختار واقعی کتاب یک Parent میانی دارند.
</p>

<p>
مهم‌ترین ساختار بازیابی‌شده:
</p>

<p>
<strong>
کار، شغل و درآمد
→ کسب‌وکار کودکان
→ تفاوت کالا و خدمت / منابع تولید / قیمت و قیمت‌گذاری / سود / معرفی محصول و تبلیغ
</strong>
</p>

<p>
بنابراین «کسب‌وکار کودکان» به‌عنوان Curriculum Node واقعی بازیابی شد
و Subtopicهای آن بر اساس Parent صریح کتاب ثبت شدند.
</p>

<h3>5. مشکل «قیمت» و «سود» — حل شد</h3>

<p>
«قیمت» و «سود» در کتاب تربیت مربی Parent صریح «کسب‌وکار کودکان» دارند،
اما در کل Primary Corpus در چندین Topic مختلف استفاده شده‌اند.
</p>

<p>
بنابراین این دو مفهوم دارای دو نقش متفاوت شناخته شدند:
</p>

<ul>
<li>در شاخه کسب‌وکار کودکان: Local Structural Subtopic</li>
<li>در کل Corpus: Context-dependent multi-topic term</li>
</ul>

<p>
در نتیجه هیچ Global Alias اجباری برای آن‌ها ساخته نشد.
</p>

<h3>6. جایگاه «بخشش و هدیه» — حل شد</h3>

<p>
بررسی کتاب تربیت مربی نشان داد «بخشش و هدیه» یک Root Curriculum Node واقعی است
و در توالی آموزشی میان صندوق اضطراری، خرید هوشمندانه، بودجه‌بندی و پول‌توجیبی قرار دارد.
</p>

<p>
Evidenceهای Book، Website مستقیم و Transcript Primary نیز ارتباط آن را با
پس‌انداز، بودجه‌بندی، خرج و مدیریت پول تأیید کردند.
</p>

<p>
بنابراین:
</p>

<ul>
<li>«بخشش و هدیه» به‌عنوان Curriculum Node حفظ شد.</li>
<li>Top-level Bridge آن «خرج، پس‌انداز و مدیریت پول» تعیین شد.</li>
<li>واژۀ عمومی «بخشش» به‌عنوان Variant مفهومی آن شناخته شد، نه Global Topic Alias.</li>
</ul>

<h3>7. جایگاه «مالیات» — حل شد</h3>

<p>
Primary Book Evidence نشان داد مؤلف صریحاً مالیات را به‌عنوان نمونه‌ای از
یک واحد آموزشی مستقل در سطح دو معرفی کرده است.
</p>

<p>
در عین حال Evidence Primary مالیات در Contextهای مختلفی مانند درآمد،
تعهدات مالی، کسب‌وکار، دولت و سرمایه‌گذاری پراکنده است.
</p>

<p>
بنابراین:
</p>

<ul>
<li>مالیات به Top-level Topic جدید تبدیل نشد.</li>
<li>به زور زیر یکی از هفت Topic قرار نگرفت.</li>
<li>به‌عنوان <strong>Level-2 Cross-topic Educational Module</strong> ثبت شد.</li>
</ul>

<h3>8. «اقتصاد رفتاری» — تصمیم Corpus-first گرفته شد</h3>

<p>
برای اقتصاد رفتاری فقط 1 Evidence مستقیم Core Primary در برابر 23 Evidence
از لایه‌های Secondary، Contextual یا Unresolved مشاهده شد.
</p>

<p>
در نتیجه این مفهوم در Taxonomy v1 به Core Taxonomy ارتقا داده نشد
و به‌عنوان مفهوم Contextual حفظ شد.
</p>

<p>
این تصمیم مانع از آن می‌شود که پراکندگی گفت‌وگوهای مهمان‌ها ساختار اصلی دانش دکتر رودی را تغییر دهد.
</p>

<h3>9. مشکل مفاهیم عمومی مانند «انتخاب» — حل شد</h3>

<p>
«انتخاب» در تمام هفت Topic اصلی Primary Evidence دارد،
اما Heading مستقل یا Parent آموزشی ثابت ندارد.
</p>

<p>
بنابراین این واژه به‌عنوان Context-dependent term باقی ماند
و Global Alias برای آن ساخته نشد.
</p>

<h3>10. Multi-topic Documentها — سیاست صحیح تعیین شد</h3>

<p>
وجود بیش از یک Strong Topic به‌تنهایی Conflict محسوب نمی‌شود.
Audit نهایی نشان داد:
</p>

<ul>
<li>2 Book Document چندموضوعی واقعی وجود دارد که هر دو با Corpus Resolution تأیید شده‌اند.</li>
<li>71 Website Document چند Strong Topic دارند و همگی multi-topic معتبر هستند.</li>
<li>Transcript Multi-strong Conflict: صفر</li>
<li>Unapproved Multi-strong Conflict: صفر</li>
</ul>

<p>
بنابراین Taxonomy به‌صورت Multi-label طراحی شده و یک سند می‌تواند واقعاً چند Topic معتبر داشته باشد.
</p>

<h3>11. Invalid Label و Structural Conflict — حل شد</h3>

<p>
Post-Freeze Audit نتایج زیر را تأیید کرد:
</p>

<ul>
<li>Invalid Mapping Topic IDs: 0</li>
<li>Invalid Parent References: 0</li>
<li>Invalid Top-level Bridges: 0</li>
<li>Hierarchy Cycles: 0</li>
<li>Duplicate Active Labels: 0</li>
<li>Non-primary Core Evidence Contamination: 0</li>
<li>Unapproved Multi-topic Conflicts: 0</li>
</ul>

<h3>12. Specialist Review قبلی — دیگر Gate فعال نیست</h3>

<p>
Review Sheet اولیه شامل 11 موردی بود که تصور می‌شد ممکن است نیازمند تصمیم متخصص باشد.
با Deep Primary Corpus Review، هر 11 مورد با Evidence موجود Resolve شدند.
</p>

<p>
بنابراین Review Sheet قبلی حذف نشد،
اما با وضعیت زیر حفظ شد:
</p>

<p>
<code>SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION</code>
</p>

<p>
در حال حاضر برای این 11 تصمیم Taxonomy نیازی به مراجعه به دکتر رودی وجود ندارد.
اگر در مراحل Retrieval/Evaluation شکاف واقعی دانش کشف شود،
آن موضوع می‌تواند مجدداً برای Expert Review مطرح شود.
</p>

<h3>13. Taxonomy v1 — Freeze انجام شد</h3>

<p>
Taxonomy نهایی این مرحله به‌صورت Immutable Artifact ذخیره و دوباره از Disk Validate شد:
</p>

<ul>
<li><code>artifacts/phase2/taxonomy/taxonomy_v1.json</code></li>
<li>SHA256: <code>335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977</code></li>
</ul>

<p>
Freeze Report نیز در:
</p>

<p>
<code>reports/phase2/taxonomy_freeze_v1.json</code>
</p>

<p>
ذخیره شده است.
</p>

<h3>14. چه چیزی هنوز حل نشده است؟</h3>

<p>
حل Taxonomy به معنی پایان کامل Phase 2 یا آماده بودن RAG نیست.
موارد زیر هنوز مراحل بعدی کار هستند:
</p>

<ul>
<li>Reconciliation مدل Competency و Prerequisite با Taxonomy Frozen</li>
<li>بررسی Mapping و Retrievability مخصوص Transcriptها</li>
<li>Chunking در Notebook بعدی</li>
<li>Retrieval Evaluation</li>
<li>Answerability و Question-based Evaluation</li>
<li>تشخیص Knowledge Gap واقعی</li>
<li>Rights Release Gate</li>
</ul>

<p>
به‌خصوص 9,305 سند Core Primary در Mapping فعلی Top-level Topic قابل مشاهده ندارند
که 9,117 مورد آن Transcript است.
این عدد به‌تنهایی Knowledge Gap یا Taxonomy Failure محسوب نمی‌شود
و در مراحل Retrieval/Context Mapping باید بررسی شود.
</p>

<h3>نتیجۀ این مرحله</h3>

<p>
<strong>
مشکل اصلی سازمان‌دهی Authority، Topic، Subtopic، Parent/Child، مفاهیم چندموضوعی،
و Conflictهای Taxonomy در این مرحله حل شده است.
Taxonomy v1 اکنون یک ساختار Corpus-derived، Primary-first، قابل Audit و Frozen است.
</strong>
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Competency / Prerequisite Discovery — بازخوانی مدل موجود پیش از Reconciliation</h3>

<p>
Taxonomy v1 اکنون Freeze شده و Post-Freeze Audit نیز PASS است.
پیش از Reconcile کردن Competency Model باید ساختار واقعی اشیایی را که در Cellهای 10–11 ساخته شده‌اند از حافظۀ Notebook کشف کنیم.
</p>

<p>
در این مرحله هیچ نام متغیر یا Schemaای حدس زده نمی‌شود.
Cell فقط Objectهای مرتبط با Competency، Prerequisite و Learning Stage را پیدا و ساختار آن‌ها را خلاصه می‌کند.
</p>

<h4>هدف</h4>

<ul>
<li>کشف نام واقعی متغیرهای Competency Model.</li>
<li>کشف ساختار 24 Competency موجود.</li>
<li>کشف Edgeهای Prerequisite/Recommended موجود.</li>
<li>کشف Stageهای آموزشی موجود.</li>
<li>شناسایی اینکه Competencyها به Topic ID، Label یا ساختار دیگری متصل‌اند.</li>
<li>آماده‌سازی Reconciliation با Frozen Taxonomy بدون Mutation.</li>
</ul>

<h4>این Cell انجام نمی‌دهد</h4>

<ul>
<li>Competency جدید ایجاد نمی‌کند.</li>
<li>Prerequisite را تغییر نمی‌دهد.</li>
<li>Taxonomy را تغییر نمی‌دهد.</li>
<li>هیچ فایل جدیدی نمی‌نویسد.</li>
</ul>

</div>

In [83]:
# Code Cell 40

from collections import Counter
import json


# --------------------------------------------------
# 1. Frozen Taxonomy prerequisite
# --------------------------------------------------

assert taxonomy_freeze_executed_v1 is True

assert (
    taxonomy_v1_frozen[
        "taxonomy_version"
    ]
    == "taxonomy_v1"
)

assert (
    taxonomy_v1_frozen[
        "status"
    ]
    == "FROZEN"
)


frozen_taxonomy_sha_before_competency_discovery_v1 = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


assert (
    frozen_taxonomy_sha_before_competency_discovery_v1
    == taxonomy_v1_frozen_sha256
)


# --------------------------------------------------
# 2. Use the actual model objects discovered
#    from the previous partial Cell-40 run
#
# We now know their real names.
# Do NOT recursively include Cell-40 discovery
# artifacts themselves.
# --------------------------------------------------

expected_competency_model_object_names_v1 = [
    "competency_model_v0",
    "competency_lookup",
    "competency_inventory_v1",
    "competency_ids",
    "competency_evidence_rows",
    "candidate_competency_relations_v0",
    "hard_prerequisite_relations_v0",
    "prerequisite_policy_v0",
    "learning_stage_rank",
    "competencies_without_model_evidence",
    "topics_without_competency",
    "duplicate_competency_ids",
    "total_competencies",
]


missing_competency_model_objects_v1 = [
    variable_name
    for variable_name in (
        expected_competency_model_object_names_v1
    )
    if variable_name not in globals()
]


print(
    "Competency Model object discovery:"
)

print(
    "- Expected model objects:",
    len(
        expected_competency_model_object_names_v1
    ),
)

print(
    "- Missing model objects:",
    len(
        missing_competency_model_objects_v1
    ),
)


if missing_competency_model_objects_v1:

    print(
        "- Missing:",
        missing_competency_model_objects_v1,
    )


assert not (
    missing_competency_model_objects_v1
)


competency_related_globals_v2 = {
    variable_name: globals()[
        variable_name
    ]
    for variable_name in (
        expected_competency_model_object_names_v1
    )
}


print(
    "\nActual Competency-related model objects:"
)


for variable_name, value in (
    competency_related_globals_v2.items()
):

    object_type = type(
        value
    ).__name__

    try:

        object_length = len(
            value
        )

    except TypeError:

        object_length = None

    print(
        "-",
        variable_name,
        "| type=",
        object_type,
        "| len=",
        object_length,
    )


# --------------------------------------------------
# 3. Compact safe preview helper
# --------------------------------------------------

def safe_json_preview_v2(
    value,
    max_items=5,
    max_chars=1600,
):

    if isinstance(
        value,
        dict,
    ):

        preview_value = {}

        for index, (
            key,
            item,
        ) in enumerate(
            value.items()
        ):

            if index >= max_items:
                break

            preview_value[
                str(
                    key
                )
            ] = item

    elif isinstance(
        value,
        (list, tuple),
    ):

        preview_value = list(
            value[
                :max_items
            ]
        )

    elif isinstance(
        value,
        set,
    ):

        preview_value = sorted(
            value,
            key=lambda item: repr(
                item
            ),
        )[
            :max_items
        ]

    else:

        preview_value = value

    try:

        preview_text = json.dumps(
            to_jsonable(
                preview_value
            ),
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )

    except Exception:

        preview_text = repr(
            preview_value
        )

    if len(
        preview_text
    ) > max_chars:

        preview_text = (
            preview_text[
                :max_chars
            ]
            + "\n...<truncated>"
        )

    return preview_text


print(
    "\nCompetency Model previews:"
)


for variable_name in [
    "competency_model_v0",
    "competency_lookup",
    "candidate_competency_relations_v0",
    "hard_prerequisite_relations_v0",
    "prerequisite_policy_v0",
    "learning_stage_rank",
]:

    print(
        "\n========================================"
    )

    print(
        "Object:",
        variable_name,
    )

    print(
        safe_json_preview_v2(
            competency_related_globals_v2[
                variable_name
            ],
            max_items=5,
            max_chars=1800,
        )
    )


# --------------------------------------------------
# 4. Cycle-safe recursive traversal helpers
#
# IMPORTANT:
# Every recursive traversal tracks object identity
# so self-references / repeated references cannot
# cause RecursionError.
# --------------------------------------------------

def is_recursive_container_v2(
    value,
):

    return isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
        ),
    )


def collect_mapping_keys_safe_v2(
    value,
    output_counter,
    visited=None,
    depth=0,
    max_depth=8,
):

    if visited is None:
        visited = set()

    if depth > max_depth:
        return

    if is_recursive_container_v2(
        value
    ):

        object_id = id(
            value
        )

        if object_id in visited:
            return

        visited.add(
            object_id
        )

    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in (
            value.items()
        ):

            output_counter[
                str(
                    key
                )
            ] += 1

            collect_mapping_keys_safe_v2(
                nested_value,
                output_counter,
                visited=visited,
                depth=depth + 1,
                max_depth=max_depth,
            )

    elif isinstance(
        value,
        (list, tuple, set),
    ):

        for nested_value in value:

            collect_mapping_keys_safe_v2(
                nested_value,
                output_counter,
                visited=visited,
                depth=depth + 1,
                max_depth=max_depth,
            )


def collect_exact_string_matches_safe_v2(
    value,
    allowed_strings,
    output_counter,
    visited=None,
    depth=0,
    max_depth=8,
):

    if visited is None:
        visited = set()

    if depth > max_depth:
        return

    if is_recursive_container_v2(
        value
    ):

        object_id = id(
            value
        )

        if object_id in visited:
            return

        visited.add(
            object_id
        )

    if isinstance(
        value,
        str,
    ):

        if value in allowed_strings:

            output_counter[
                value
            ] += 1

        return

    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in (
            value.items()
        ):

            if (
                isinstance(
                    key,
                    str,
                )
                and key in allowed_strings
            ):

                output_counter[
                    key
                ] += 1

            collect_exact_string_matches_safe_v2(
                nested_value,
                allowed_strings,
                output_counter,
                visited=visited,
                depth=depth + 1,
                max_depth=max_depth,
            )

    elif isinstance(
        value,
        (list, tuple, set),
    ):

        for nested_value in value:

            collect_exact_string_matches_safe_v2(
                nested_value,
                allowed_strings,
                output_counter,
                visited=visited,
                depth=depth + 1,
                max_depth=max_depth,
            )


# --------------------------------------------------
# 5. Actual Competency Model shape
# --------------------------------------------------

print(
    "\nCompetency Model shape:"
)

print(
    "- Top-level Topic groups:",
    len(
        competency_model_v0
    ),
)

print(
    "- Competency IDs:",
    len(
        competency_ids
    ),
)

print(
    "- Competency lookup entries:",
    len(
        competency_lookup
    ),
)

print(
    "- Competency inventory entries:",
    len(
        competency_inventory_v1
    ),
)

print(
    "- Declared total competencies:",
    total_competencies,
)

print(
    "- Candidate progression relations:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Hard prerequisite relations:",
    len(
        hard_prerequisite_relations_v0
    ),
)

print(
    "- Learning stages:",
    len(
        learning_stage_rank
    ),
)


# --------------------------------------------------
# 6. Competency count by Top-level Topic
# --------------------------------------------------

competency_count_by_topic_v1 = {}


for topic_id, topic_model in (
    competency_model_v0.items()
):

    competencies_for_topic = (
        topic_model.get(
            "competencies",
            [],
        )
    )

    competency_count_by_topic_v1[
        topic_id
    ] = len(
        competencies_for_topic
    )


print(
    "\nCompetencies by Top-level Topic:"
)


for topic_id in (
    taxonomy_v1_frozen[
        "top_level_topics"
    ]
):

    print(
        "-",
        topic_id,
        "|",
        taxonomy_v1_frozen[
            "top_level_topics"
        ][
            topic_id
        ][
            "label_fa"
        ],
        ":",
        competency_count_by_topic_v1.get(
            topic_id,
            0,
        ),
    )


# --------------------------------------------------
# 7. Independent competency flattening from
#    competency_model_v0
# --------------------------------------------------

competency_flattened_from_model_v1 = []


for topic_id, topic_model in (
    competency_model_v0.items()
):

    topic_label_fa = (
        topic_model.get(
            "label_fa"
        )
    )

    for competency_row in (
        topic_model.get(
            "competencies",
            [],
        )
    ):

        flattened_row = {
            "id": (
                competency_row.get(
                    "id"
                )
            ),

            "topic_id": (
                topic_id
            ),

            "topic_label_fa": (
                topic_label_fa
            ),

            "stage": (
                competency_row.get(
                    "stage"
                )
            ),

            "statement_fa": (
                competency_row.get(
                    "statement_fa"
                )
            ),

            "evidence": list(
                competency_row.get(
                    "evidence",
                    [],
                )
            ),
        }

        competency_flattened_from_model_v1.append(
            flattened_row
        )


flattened_competency_ids_v1 = [
    row[
        "id"
    ]
    for row in (
        competency_flattened_from_model_v1
    )
]


flattened_competency_id_set_v1 = set(
    flattened_competency_ids_v1
)


print(
    "\nIndependent model flattening:"
)

print(
    "- Flattened competencies:",
    len(
        competency_flattened_from_model_v1
    ),
)

print(
    "- Distinct flattened IDs:",
    len(
        flattened_competency_id_set_v1
    ),
)


# --------------------------------------------------
# 8. Competency ID consistency
# --------------------------------------------------

competency_lookup_ids_v1 = set(
    competency_lookup
)


competency_ids_set_v1 = set(
    competency_ids
)


duplicate_flattened_competency_ids_v1 = [
    competency_id
    for competency_id, count in Counter(
        flattened_competency_ids_v1
    ).items()
    if count > 1
]


missing_from_lookup_v1 = sorted(
    flattened_competency_id_set_v1
    - competency_lookup_ids_v1
)


extra_in_lookup_v1 = sorted(
    competency_lookup_ids_v1
    - flattened_competency_id_set_v1
)


missing_from_competency_ids_v1 = sorted(
    flattened_competency_id_set_v1
    - competency_ids_set_v1
)


extra_in_competency_ids_v1 = sorted(
    competency_ids_set_v1
    - flattened_competency_id_set_v1
)


print(
    "\nCompetency ID consistency:"
)

print(
    "- Duplicate flattened IDs:",
    len(
        duplicate_flattened_competency_ids_v1
    ),
)

print(
    "- Missing from competency_lookup:",
    len(
        missing_from_lookup_v1
    ),
)

print(
    "- Extra in competency_lookup:",
    len(
        extra_in_lookup_v1
    ),
)

print(
    "- Missing from competency_ids:",
    len(
        missing_from_competency_ids_v1
    ),
)

print(
    "- Extra in competency_ids:",
    len(
        extra_in_competency_ids_v1
    ),
)


# --------------------------------------------------
# 9. Frozen Top-level Topic consistency
# --------------------------------------------------

frozen_topic_ids_for_competency_v1 = set(
    taxonomy_v1_frozen[
        "top_level_topics"
    ]
)


competency_model_topic_ids_v1 = set(
    competency_model_v0
)


invalid_competency_model_topic_ids_v1 = sorted(
    competency_model_topic_ids_v1
    - frozen_topic_ids_for_competency_v1
)


frozen_topics_missing_competency_model_v1 = sorted(
    frozen_topic_ids_for_competency_v1
    - competency_model_topic_ids_v1
)


invalid_lookup_topic_ids_v1 = []


for competency_id, row in (
    competency_lookup.items()
):

    topic_id = row.get(
        "topic_id"
    )

    if (
        topic_id
        not in frozen_topic_ids_for_competency_v1
    ):

        invalid_lookup_topic_ids_v1.append(
            (
                competency_id,
                topic_id,
            )
        )


print(
    "\nFrozen Taxonomy Topic reconciliation:"
)

print(
    "- Frozen Top-level Topics:",
    len(
        frozen_topic_ids_for_competency_v1
    ),
)

print(
    "- Competency Model Topics:",
    len(
        competency_model_topic_ids_v1
    ),
)

print(
    "- Invalid Competency Model Topic IDs:",
    len(
        invalid_competency_model_topic_ids_v1
    ),
)

print(
    "- Frozen Topics missing Competency Model:",
    len(
        frozen_topics_missing_competency_model_v1
    ),
)

print(
    "- Invalid competency_lookup Topic IDs:",
    len(
        invalid_lookup_topic_ids_v1
    ),
)


# --------------------------------------------------
# 10. Learning Stage discovery and validation
# --------------------------------------------------

expected_stage_labels_v1 = {
    "understand",
    "self_assess",
    "analyze",
    "plan",
    "act",
    "review_and_improve",
}


actual_stage_labels_v1 = set(
    learning_stage_rank
)


competency_stage_counter_v1 = Counter(
    row[
        "stage"
    ]
    for row in (
        competency_flattened_from_model_v1
    )
)


invalid_competency_stages_v1 = sorted(
    set(
        competency_stage_counter_v1
    )
    - expected_stage_labels_v1
)


missing_declared_stage_labels_v1 = sorted(
    expected_stage_labels_v1
    - actual_stage_labels_v1
)


unexpected_declared_stage_labels_v1 = sorted(
    actual_stage_labels_v1
    - expected_stage_labels_v1
)


print(
    "\nLearning Stage model:"
)

print(
    "- Declared Stages:",
    sorted(
        actual_stage_labels_v1,
        key=lambda stage: (
            learning_stage_rank[
                stage
            ]
        ),
    ),
)

print(
    "- Invalid Competency stages:",
    len(
        invalid_competency_stages_v1
    ),
)

print(
    "- Missing expected declared Stages:",
    len(
        missing_declared_stage_labels_v1
    ),
)

print(
    "- Unexpected declared Stages:",
    len(
        unexpected_declared_stage_labels_v1
    ),
)


print(
    "\nCompetencies by Stage:"
)


for stage in sorted(
    actual_stage_labels_v1,
    key=lambda stage: (
        learning_stage_rank[
            stage
        ]
    ),
):

    print(
        "-",
        stage,
        ":",
        competency_stage_counter_v1[
            stage
        ],
    )


# --------------------------------------------------
# 11. Cycle-safe field inventory
# --------------------------------------------------

competency_field_inventory_v2 = Counter()


for value in (
    competency_related_globals_v2.values()
):

    collect_mapping_keys_safe_v2(
        value,
        competency_field_inventory_v2,
        visited=set(),
        max_depth=8,
    )


print(
    "\nObserved model field names:"
)


for field_name, count in (
    competency_field_inventory_v2.most_common(
        50
    )
):

    print(
        f"- {field_name}: {count}"
    )


# --------------------------------------------------
# 12. Cycle-safe Frozen Topic reference discovery
# --------------------------------------------------

topic_reference_occurrences_v2 = Counter()


for value in (
    competency_related_globals_v2.values()
):

    collect_exact_string_matches_safe_v2(
        value,
        frozen_topic_ids_for_competency_v1,
        topic_reference_occurrences_v2,
        visited=set(),
        max_depth=8,
    )


print(
    "\nFrozen Top-level Topic references "
    "inside Competency Model objects:"
)


for topic_id in (
    taxonomy_v1_frozen[
        "top_level_topics"
    ]
):

    print(
        "-",
        topic_id,
        "|",
        taxonomy_v1_frozen[
            "top_level_topics"
        ][
            topic_id
        ][
            "label_fa"
        ],
        ":",
        topic_reference_occurrences_v2[
            topic_id
        ],
    )


# --------------------------------------------------
# 13. Progression / Prerequisite relation structure
# --------------------------------------------------

valid_relation_types_v1 = {
    "HARD_PREREQUISITE",
    "RECOMMENDED_LEARNING_PROGRESSION",
    "CURRICULAR_ORDER_EVIDENCE",
}


candidate_relation_type_counter_v1 = Counter()


invalid_relation_endpoints_v1 = []
invalid_relation_types_v1 = []


for relation in (
    candidate_competency_relations_v0
):

    relation_type = relation.get(
        "type"
    )

    from_id = relation.get(
        "from"
    )

    to_id = relation.get(
        "to"
    )

    candidate_relation_type_counter_v1[
        relation_type
    ] += 1

    if (
        relation_type
        not in valid_relation_types_v1
    ):

        invalid_relation_types_v1.append(
            relation
        )

    if (
        from_id
        not in competency_lookup_ids_v1
        or to_id
        not in competency_lookup_ids_v1
    ):

        invalid_relation_endpoints_v1.append(
            relation
        )


print(
    "\nCompetency relation structure:"
)

print(
    "- Candidate relations:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Hard prerequisite relations:",
    len(
        hard_prerequisite_relations_v0
    ),
)

print(
    "- Invalid relation types:",
    len(
        invalid_relation_types_v1
    ),
)

print(
    "- Invalid relation endpoints:",
    len(
        invalid_relation_endpoints_v1
    ),
)


print(
    "Relation types:"
)


for relation_type, count in (
    candidate_relation_type_counter_v1.items()
):

    print(
        "-",
        relation_type,
        ":",
        count,
    )


# --------------------------------------------------
# 14. Detect actual cross-topic relations
# --------------------------------------------------

cross_topic_candidate_relations_v1 = []


for relation in (
    candidate_competency_relations_v0
):

    from_id = relation[
        "from"
    ]

    to_id = relation[
        "to"
    ]

    from_topic = (
        competency_lookup[
            from_id
        ][
            "topic_id"
        ]
    )

    to_topic = (
        competency_lookup[
            to_id
        ][
            "topic_id"
        ]
    )

    if (
        from_topic
        != to_topic
    ):

        cross_topic_candidate_relations_v1.append(
            {
                "from": (
                    from_id
                ),

                "to": (
                    to_id
                ),

                "from_topic": (
                    from_topic
                ),

                "to_topic": (
                    to_topic
                ),

                "type": (
                    relation[
                        "type"
                    ]
                ),
            }
        )


print(
    "\nCross-topic Candidate relations:"
)

print(
    "- Count:",
    len(
        cross_topic_candidate_relations_v1
    ),
)


for relation in (
    cross_topic_candidate_relations_v1
):

    print(
        "-",
        relation,
    )


# --------------------------------------------------
# 15. Independent cycle detection in candidate
#     learning-progression graph
# --------------------------------------------------

relation_graph_v1 = {
    competency_id: []
    for competency_id in (
        competency_lookup_ids_v1
    )
}


for relation in (
    candidate_competency_relations_v0
):

    from_id = relation[
        "from"
    ]

    to_id = relation[
        "to"
    ]

    relation_graph_v1[
        from_id
    ].append(
        to_id
    )


progression_cycles_v1 = []


def detect_progression_cycles_v1():

    visiting = set()
    visited = set()
    path = []

    def visit(
        node,
    ):

        if node in visiting:

            cycle_start = (
                path.index(
                    node
                )
            )

            progression_cycles_v1.append(
                tuple(
                    path[
                        cycle_start:
                    ]
                    + [
                        node
                    ]
                )
            )

            return

        if node in visited:
            return

        visiting.add(
            node
        )

        path.append(
            node
        )

        for neighbor in (
            relation_graph_v1[
                node
            ]
        ):

            visit(
                neighbor
            )

        path.pop()

        visiting.remove(
            node
        )

        visited.add(
            node
        )

    for node in (
        relation_graph_v1
    ):

        visit(
            node
        )


detect_progression_cycles_v1()


progression_cycles_v1 = sorted(
    set(
        progression_cycles_v1
    )
)


print(
    "\nLearning-progression graph:"
)

print(
    "- Nodes:",
    len(
        relation_graph_v1
    ),
)

print(
    "- Edges:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Cycles:",
    len(
        progression_cycles_v1
    ),
)


# --------------------------------------------------
# 16. Existing Prerequisite policy discovery
#
# IMPORTANT:
# We only inspect it here.
# Cell 41 will reconcile the outdated
# expert-review requirement with the now-frozen
# Corpus-first Taxonomy.
# --------------------------------------------------

print(
    "\nExisting Prerequisite Policy:"
)


for key, value in (
    prerequisite_policy_v0.items()
):

    print(
        "-",
        key,
        ":",
        value,
    )


outdated_expert_review_flag_v1 = (
    prerequisite_policy_v0.get(
        "expert_review_required_before_freeze"
    )
    is True
)


print(
    "- Outdated expert-review flag detected:",
    outdated_expert_review_flag_v1,
)


# --------------------------------------------------
# 17. Competency model evidence status
# --------------------------------------------------

print(
    "\nCompetency Model evidence status:"
)

print(
    "- Competencies without model evidence:",
    len(
        competencies_without_model_evidence
    ),
)

print(
    "- Topics without competency:",
    len(
        topics_without_competency
    ),
)

print(
    "- Duplicate competency IDs:",
    len(
        duplicate_competency_ids
    ),
)

print(
    "- Competency evidence rows:",
    len(
        competency_evidence_rows
    ),
)


# --------------------------------------------------
# 18. Discovery result
# --------------------------------------------------

competency_discovery_v2 = {
    "model_object_names": list(
        expected_competency_model_object_names_v1
    ),

    "competency_count": (
        len(
            competency_flattened_from_model_v1
        )
    ),

    "topic_count": (
        len(
            competency_model_v0
        )
    ),

    "stage_count": (
        len(
            learning_stage_rank
        )
    ),

    "candidate_relation_count": (
        len(
            candidate_competency_relations_v0
        )
    ),

    "hard_prerequisite_count": (
        len(
            hard_prerequisite_relations_v0
        )
    ),

    "cross_topic_candidate_relation_count": (
        len(
            cross_topic_candidate_relations_v1
        )
    ),

    "progression_cycle_count": (
        len(
            progression_cycles_v1
        )
    ),

    "invalid_model_topic_ids": (
        invalid_competency_model_topic_ids_v1
    ),

    "frozen_topics_missing_model": (
        frozen_topics_missing_competency_model_v1
    ),

    "invalid_competency_stages": (
        invalid_competency_stages_v1
    ),

    "invalid_relation_endpoints": (
        len(
            invalid_relation_endpoints_v1
        )
    ),

    "outdated_expert_review_flag": (
        outdated_expert_review_flag_v1
    ),

    "field_inventory": dict(
        competency_field_inventory_v2
    ),

    "frozen_topic_reference_counts": dict(
        topic_reference_occurrences_v2
    ),
}


# --------------------------------------------------
# 19. No mutation / write
# --------------------------------------------------

taxonomy_mutation_performed = False
competency_mutation_performed = False
prerequisite_mutation_performed = False
topic_mapping_mutation_performed = False
disk_write_performed = False


# --------------------------------------------------
# 20. Final validation
# --------------------------------------------------

frozen_taxonomy_sha_after_competency_discovery_v1 = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


print(
    "\nValidation:"
)

print(
    "- Competencies expected:",
    24,
)

print(
    "- Competencies discovered:",
    len(
        competency_flattened_from_model_v1
    ),
)

print(
    "- Competency Topics expected:",
    7,
)

print(
    "- Competency Topics discovered:",
    len(
        competency_model_v0
    ),
)

print(
    "- Learning Stages expected:",
    6,
)

print(
    "- Learning Stages discovered:",
    len(
        learning_stage_rank
    ),
)

print(
    "- Candidate Progression relations:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Hard Prerequisites:",
    len(
        hard_prerequisite_relations_v0
    ),
)

print(
    "- Cross-topic candidate relations:",
    len(
        cross_topic_candidate_relations_v1
    ),
)

print(
    "- Progression cycles:",
    len(
        progression_cycles_v1
    ),
)

print(
    "- Invalid model Topic IDs:",
    len(
        invalid_competency_model_topic_ids_v1
    ),
)

print(
    "- Frozen Topics missing Competency Model:",
    len(
        frozen_topics_missing_competency_model_v1
    ),
)

print(
    "- Invalid Competency stages:",
    len(
        invalid_competency_stages_v1
    ),
)

print(
    "- Invalid relation endpoints:",
    len(
        invalid_relation_endpoints_v1
    ),
)

print(
    "- Competencies without model evidence:",
    len(
        competencies_without_model_evidence
    ),
)

print(
    "- Topics without competency:",
    len(
        topics_without_competency
    ),
)

print(
    "- Outdated expert-review flag detected:",
    outdated_expert_review_flag_v1,
)

print(
    "- Frozen Taxonomy SHA unchanged:",
    (
        frozen_taxonomy_sha_after_competency_discovery_v1
        == frozen_taxonomy_sha_before_competency_discovery_v1
        == taxonomy_v1_frozen_sha256
    ),
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Competency mutation performed:",
    competency_mutation_performed,
)

print(
    "- Prerequisite mutation performed:",
    prerequisite_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert (
    total_competencies
    == 24
)

assert len(
    competency_flattened_from_model_v1
) == 24

assert len(
    competency_lookup
) == 24

assert len(
    competency_ids
) == 24

assert len(
    competency_inventory_v1
) == 24

assert len(
    competency_model_v0
) == 7

assert len(
    learning_stage_rank
) == 6

assert (
    actual_stage_labels_v1
    == expected_stage_labels_v1
)

assert len(
    candidate_competency_relations_v0
) == 17

assert len(
    hard_prerequisite_relations_v0
) == 0

assert not duplicate_flattened_competency_ids_v1

assert not missing_from_lookup_v1
assert not extra_in_lookup_v1

assert not missing_from_competency_ids_v1
assert not extra_in_competency_ids_v1

assert not invalid_competency_model_topic_ids_v1

assert not frozen_topics_missing_competency_model_v1

assert not invalid_lookup_topic_ids_v1

assert not invalid_competency_stages_v1

assert not missing_declared_stage_labels_v1
assert not unexpected_declared_stage_labels_v1

assert not invalid_relation_types_v1
assert not invalid_relation_endpoints_v1

assert not progression_cycles_v1

assert not competencies_without_model_evidence
assert not topics_without_competency
assert not duplicate_competency_ids

assert (
    frozen_taxonomy_sha_after_competency_discovery_v1
    == taxonomy_v1_frozen_sha256
)

assert taxonomy_mutation_performed is False
assert competency_mutation_performed is False
assert prerequisite_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert disk_write_performed is False


print(
    "\nCODE CELL 40 GATE: PASS"
)

print(
    "Competency / Prerequisite Discovery PASS: "
    "the actual 24-competency, 7-topic, 6-stage "
    "model and its 17 recommended progression "
    "relations were recovered and independently "
    "validated without recursion, reconstruction, "
    "mutation or disk write. The existing "
    "prerequisite policy is now ready for "
    "Frozen-Taxonomy reconciliation."
)

Competency Model object discovery:
- Expected model objects: 13
- Missing model objects: 0

Actual Competency-related model objects:
- competency_model_v0 | type= dict | len= 7
- competency_lookup | type= dict | len= 24
- competency_inventory_v1 | type= list | len= 24
- competency_ids | type= list | len= 24
- competency_evidence_rows | type= list | len= 48
- candidate_competency_relations_v0 | type= list | len= 17
- hard_prerequisite_relations_v0 | type= list | len= 0
- prerequisite_policy_v0 | type= dict | len= 4
- learning_stage_rank | type= dict | len= 6
- competencies_without_model_evidence | type= list | len= 0
- topics_without_competency | type= list | len= 0
- duplicate_competency_ids | type= list | len= 0
- total_competencies | type= int | len= None

Competency Model previews:

Object: competency_model_v0
{
  "credit_debt_banking": {
    "competencies": [
      {
        "evidence": [
          "قواعد قرض",
          "کارت اعتباری",
          "اعتبارسنجی",
          "بانک و اعت

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h3>Frozen-Taxonomy Competency / Prerequisite Reconciliation</h3>

<p>
Competency Discovery نشان داد مدل موجود شامل 24 Competency معتبر در هر 7 Top-level Topic است.
تمام Topic IDها با Taxonomy v1 Frozen سازگارند و 17 رابطۀ موجود همگی
<code>RECOMMENDED_LEARNING_PROGRESSION</code>
هستند.
</p>

<p>
هیچ Hard Prerequisite، Cross-topic Prerequisite یا Cycle در مدل وجود ندارد.
بنابراین تغییرات Taxonomy در سطح Curriculum Node لزوماً نیازمند بازطراحی Competencyهای Top-level نیست.
</p>

<h4>اصل تفکیک Taxonomy و Competency</h4>

<p>
Taxonomy مشخص می‌کند دانش درباره چه موضوعاتی سازمان‌دهی شده است؛
Competency مشخص می‌کند یادگیرنده پس از آموزش باید چه توانایی‌ای داشته باشد.
</p>

<p>
در نتیجه:
</p>

<ul>
<li>هر Taxonomy Node الزاماً Competency مستقل ندارد.</li>
<li>هر Subtopic الزاماً Learning Outcome مستقل ایجاد نمی‌کند.</li>
<li>Competency جدید فقط زمانی ساخته می‌شود که Primary Corpus یک Outcome آموزشی قابل دفاع را پشتیبانی کند.</li>
<li>Level-2 Module می‌تواند در نسخۀ فعلی Taxonomy ثبت شود ولی Competency Expansion آن به نسخۀ بعدی موکول شود.</li>
</ul>

<h4>موارد جدید Taxonomy که باید با Competency Model مقایسه شوند</h4>

<ul>
<li>بخشش و هدیه</li>
<li>کسب‌وکار کودکان</li>
<li>تفاوت کالا و خدمت</li>
<li>منابع تولید</li>
<li>قیمت و قیمت‌گذاری در کسب‌وکار</li>
<li>سود در کسب‌وکار</li>
<li>معرفی محصول و تبلیغ</li>
<li>Module سطح دو مالیات و تعهدات مالی</li>
</ul>

<h4>Prerequisite Policy</h4>

<p>
فلگ قدیمی
<code>expert_review_required_before_freeze=True</code>
مربوط به زمانی بود که 11 Review Item هنوز حل نشده بودند.
پس از Deep Primary Corpus Review، هر 11 مورد Corpus-first Resolve شدند و Taxonomy v1 نیز Freeze شد.
بنابراین این فلگ در Policy جدید باید Supersede شود، بدون بازنویسی یا حذف Policy تاریخی v0.
</p>

<h4>Stage review_and_improve</h4>

<p>
Stage Model شش مرحلۀ آموزشی دارد اما در مدل فعلی هیچ Competency با Stage
<code>review_and_improve</code>
وجود ندارد.
این وضعیت به‌تنهایی خطا محسوب نمی‌شود.
در این مرحله Evidence موجود بررسی می‌شود تا مشخص شود آیا Outcome مستقلی برای بازبینی و بهبود وجود دارد یا خیر.
هیچ Competency مصنوعی صرفاً برای پرکردن Stage ساخته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>24 Competency فعلی بدون تغییر حفظ شوند.</li>
<li>هر 7 Top-level Topic همچنان Competency داشته باشند.</li>
<li>17 رابطۀ Recommended معتبر باقی بمانند.</li>
<li>Hard Prerequisite برابر صفر باقی بماند.</li>
<li>Cycle و Cross-topic Hard Prerequisite صفر بماند.</li>
<li>Policy تاریخی Expert Review به‌صورت مستند Supersede شود.</li>
<li>Nodeها و Module جدید برای Evidence مرتبط با Competency بررسی شوند.</li>
<li>Stage بدون Competency به Knowledge Gap ساختگی تبدیل نشود.</li>
<li>هیچ فایل یا Artifact در این Cell نوشته نشود.</li>
</ul>

</div>

In [84]:
# Code Cell 41

from collections import Counter, defaultdict
from copy import deepcopy


# --------------------------------------------------
# 1. Frozen Taxonomy + Competency prerequisites
# --------------------------------------------------

assert taxonomy_freeze_executed_v1 is True

assert (
    taxonomy_v1_frozen[
        "status"
    ]
    == "FROZEN"
)

assert (
    taxonomy_v1_frozen[
        "taxonomy_version"
    ]
    == "taxonomy_v1"
)

assert len(
    competency_lookup
) == 24

assert len(
    competency_model_v0
) == 7

assert len(
    candidate_competency_relations_v0
) == 17

assert len(
    hard_prerequisite_relations_v0
) == 0

assert not progression_cycles_v1


frozen_taxonomy_sha_before_reconciliation_v1 = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


assert (
    frozen_taxonomy_sha_before_reconciliation_v1
    == taxonomy_v1_frozen_sha256
)


# --------------------------------------------------
# 2. Validate Topic labels as well as Topic IDs
# --------------------------------------------------

competency_topic_label_mismatches_v1 = []


for topic_id, topic_model in (
    competency_model_v0.items()
):

    frozen_topic = (
        taxonomy_v1_frozen[
            "top_level_topics"
        ][
            topic_id
        ]
    )

    model_label = (
        normalize_fa_for_match(
            topic_model.get(
                "label_fa",
                "",
            )
        )
    )

    frozen_label = (
        normalize_fa_for_match(
            frozen_topic.get(
                "label_fa",
                "",
            )
        )
    )

    if (
        model_label
        != frozen_label
    ):

        competency_topic_label_mismatches_v1.append(
            {
                "topic_id": (
                    topic_id
                ),

                "competency_label": (
                    topic_model.get(
                        "label_fa"
                    )
                ),

                "frozen_label": (
                    frozen_topic.get(
                        "label_fa"
                    )
                ),
            }
        )


print(
    "Frozen Topic / Competency label reconciliation:"
)

print(
    "- Topic groups:",
    len(
        competency_model_v0
    ),
)

print(
    "- Label mismatches:",
    len(
        competency_topic_label_mismatches_v1
    ),
)


# --------------------------------------------------
# 3. Validate progression monotonicity
#
# Recommended progression may remain in the same
# stage, but it must not move backwards.
# --------------------------------------------------

non_monotonic_progression_relations_v1 = []


for relation in (
    candidate_competency_relations_v0
):

    from_id = relation[
        "from"
    ]

    to_id = relation[
        "to"
    ]

    from_stage = (
        competency_lookup[
            from_id
        ][
            "stage"
        ]
    )

    to_stage = (
        competency_lookup[
            to_id
        ][
            "stage"
        ]
    )

    from_rank = (
        learning_stage_rank[
            from_stage
        ]
    )

    to_rank = (
        learning_stage_rank[
            to_stage
        ]
    )

    if (
        from_rank
        > to_rank
    ):

        non_monotonic_progression_relations_v1.append(
            {
                "from": (
                    from_id
                ),

                "to": (
                    to_id
                ),

                "from_stage": (
                    from_stage
                ),

                "to_stage": (
                    to_stage
                ),

                "from_rank": (
                    from_rank
                ),

                "to_rank": (
                    to_rank
                ),
            }
        )


print(
    "\nLearning Progression monotonicity:"
)

print(
    "- Recommended relations:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Backward-stage relations:",
    len(
        non_monotonic_progression_relations_v1
    ),
)


# --------------------------------------------------
# 4. Explicit scope policy
#
# Taxonomy Node != mandatory dedicated Competency.
# --------------------------------------------------

competency_scope_policy_v1_candidate = {
    "model_scope": (
        "TOP_LEVEL_COMPETENCY_FRAMEWORK"
    ),

    "taxonomy_version": (
        "taxonomy_v1"
    ),

    "top_level_topic_requires_competency": True,

    "curriculum_node_requires_dedicated_competency": (
        False
    ),

    "educational_module_requires_dedicated_competency": (
        False
    ),

    "competency_creation_requires": [
        "PRIMARY_CURRICULUM_EVIDENCE",
        "DISTINCT_LEARNING_OUTCOME",
        "NON_DUPLICATIVE_WITH_EXISTING_COMPETENCY",
    ],

    "absence_of_dedicated_node_competency_is_gap": (
        False
    ),

    "absence_of_stage_population_is_gap": (
        False
    ),

    "knowledge_gap_requires_separate_evaluation": (
        True
    ),
}


print(
    "\nCompetency Scope Policy candidate:"
)


for key, value in (
    competency_scope_policy_v1_candidate.items()
):

    print(
        "-",
        key,
        ":",
        value,
    )


# --------------------------------------------------
# 5. Build searchable Competency text
# --------------------------------------------------

competency_search_text_v1 = {}


for competency_id, row in (
    competency_lookup.items()
):

    parts = [
        str(
            row.get(
                "statement_fa",
                "",
            )
        )
    ]

    parts.extend(
        str(
            item
        )
        for item in row.get(
            "evidence",
            []
        )
    )

    competency_search_text_v1[
        competency_id
    ] = (
        normalize_fa_for_match(
            " ".join(
                parts
            )
        )
    )


# --------------------------------------------------
# 6. Evidence terms for new Frozen Taxonomy
#    Curriculum Nodes
#
# Matching here is descriptive only.
# It does NOT create a Competency.
# --------------------------------------------------

curriculum_node_competency_terms_v1 = {
    "giving_and_gifting": [
        "بخشش",
        "هدیه",
    ],

    "children_business": [
        "کسب و کار کودکان",
        "کسب و کار",
    ],

    "goods_and_services_basics": [
        "تفاوت کالا و خدمت",
        "کالا و خدمت",
    ],

    "entrepreneurship_production_resources": [
        "منابع تولید",
    ],

    "children_business_pricing": [
        "قیمت گذاری",
        "قیمت",
    ],

    "children_business_profit": [
        "سود",
    ],

    "product_presentation_and_promotion": [
        "معرفی محصول",
        "تبلیغ",
    ],
}


curriculum_node_competency_match_v1 = {}


for node_id, terms in (
    curriculum_node_competency_terms_v1.items()
):

    normalized_terms = [
        normalize_fa_for_match(
            term
        )
        for term in terms
    ]

    direct_matches = []

    for competency_id, text in (
        competency_search_text_v1.items()
    ):

        matched_terms = [
            term
            for term in normalized_terms
            if term in text
        ]

        if not matched_terms:
            continue

        direct_matches.append(
            {
                "competency_id": (
                    competency_id
                ),

                "topic_id": (
                    competency_lookup[
                        competency_id
                    ][
                        "topic_id"
                    ]
                ),

                "stage": (
                    competency_lookup[
                        competency_id
                    ][
                        "stage"
                    ]
                ),

                "statement_fa": (
                    competency_lookup[
                        competency_id
                    ][
                        "statement_fa"
                    ]
                ),

                "matched_terms": (
                    matched_terms
                ),
            }
        )

    frozen_node = (
        taxonomy_v1_frozen[
            "curriculum_nodes"
        ][
            node_id
        ]
    )

    parent_topic_id = (
        frozen_node[
            "top_level_topic_id"
        ]
    )

    parent_topic_competencies = sorted(
        competency_id
        for competency_id, competency in (
            competency_lookup.items()
        )
        if (
            competency[
                "topic_id"
            ]
            == parent_topic_id
        )
    )

    curriculum_node_competency_match_v1[
        node_id
    ] = {
        "label_fa": (
            frozen_node[
                "label_fa"
            ]
        ),

        "top_level_topic_id": (
            parent_topic_id
        ),

        "parent_topic_competencies": (
            parent_topic_competencies
        ),

        "direct_competency_matches": (
            direct_matches
        ),

        "direct_match_count": len(
            direct_matches
        ),

        "dedicated_competency_required": (
            False
        ),
    }


print(
    "\nNew Curriculum Nodes → existing Competency evidence:"
)


for node_id, row in (
    curriculum_node_competency_match_v1.items()
):

    print(
        "\n========================================"
    )

    print(
        "Node:",
        row[
            "label_fa"
        ],
        f"({node_id})",
    )

    print(
        "- Top-level Topic:",
        row[
            "top_level_topic_id"
        ],
    )

    print(
        "- Parent-topic Competencies:",
        row[
            "parent_topic_competencies"
        ],
    )

    print(
        "- Direct existing Competency matches:",
        row[
            "direct_match_count"
        ],
    )

    for match in (
        row[
            "direct_competency_matches"
        ]
    ):

        print(
            "  -",
            match[
                "competency_id"
            ],
            "| stage=",
            match[
                "stage"
            ],
            "| matched=",
            match[
                "matched_terms"
            ],
        )


# --------------------------------------------------
# 7. Level-2 Taxation Module → existing
#    Competency evidence
# --------------------------------------------------

taxation_competency_terms_v1 = [
    normalize_fa_for_match(
        "مالیات"
    ),
    normalize_fa_for_match(
        "تعهدات مالی"
    ),
]


taxation_existing_competency_matches_v1 = []


for competency_id, text in (
    competency_search_text_v1.items()
):

    matched_terms = [
        term
        for term in (
            taxation_competency_terms_v1
        )
        if term in text
    ]

    if not matched_terms:
        continue

    taxation_existing_competency_matches_v1.append(
        {
            "competency_id": (
                competency_id
            ),

            "topic_id": (
                competency_lookup[
                    competency_id
                ][
                    "topic_id"
                ]
            ),

            "stage": (
                competency_lookup[
                    competency_id
                ][
                    "stage"
                ]
            ),

            "statement_fa": (
                competency_lookup[
                    competency_id
                ][
                    "statement_fa"
                ]
            ),

            "matched_terms": (
                matched_terms
            ),
        }
    )


taxation_competency_extension_v1 = {
    "module_id": (
        "taxation_and_financial_obligations"
    ),

    "module_level": 2,

    "existing_competency_match_count": len(
        taxation_existing_competency_matches_v1
    ),

    "existing_competency_matches": (
        taxation_existing_competency_matches_v1
    ),

    "dedicated_competency_required_for_v1": (
        False
    ),

    "future_level_2_competency_expansion": (
        "EVIDENCE_DRIVEN_IF_PRODUCT_SCOPE_REQUIRES"
    ),

    "absence_is_knowledge_gap": False,
}


print(
    "\nTaxation Level-2 Module → existing Competency evidence:"
)

print(
    "- Existing direct Competency matches:",
    taxation_competency_extension_v1[
        "existing_competency_match_count"
    ],
)

print(
    "- Dedicated Competency required for v1:",
    taxation_competency_extension_v1[
        "dedicated_competency_required_for_v1"
    ],
)

print(
    "- Future Level-2 expansion:",
    taxation_competency_extension_v1[
        "future_level_2_competency_expansion"
    ],
)


for match in (
    taxation_existing_competency_matches_v1
):

    print(
        "  -",
        match[
            "competency_id"
        ],
        "|",
        match[
            "topic_id"
        ],
        "| stage=",
        match[
            "stage"
        ],
    )


# --------------------------------------------------
# 8. review_and_improve Stage Evidence Audit
#
# Do not fabricate a Competency to fill the stage.
# --------------------------------------------------

REVIEW_STAGE = (
    "review_and_improve"
)


current_review_stage_competencies_v1 = [
    competency_id
    for competency_id, row in (
        competency_lookup.items()
    )
    if (
        row[
            "stage"
        ]
        == REVIEW_STAGE
    )
]


review_improve_terms_v1 = [
    "بازبینی",
    "بازنگری",
    "مرور",
    "پیگیری",
    "بهبود",
    "اصلاح",
    "ارزیابی مجدد",
    "بازارزیابی",
]


normalized_review_improve_terms_v1 = [
    normalize_fa_for_match(
        term
    )
    for term in (
        review_improve_terms_v1
    )
]


review_signal_competencies_v1 = []


for competency_id, text in (
    competency_search_text_v1.items()
):

    matched_terms = [
        term
        for term in (
            normalized_review_improve_terms_v1
        )
        if term in text
    ]

    if not matched_terms:
        continue

    review_signal_competencies_v1.append(
        {
            "competency_id": (
                competency_id
            ),

            "topic_id": (
                competency_lookup[
                    competency_id
                ][
                    "topic_id"
                ]
            ),

            "current_stage": (
                competency_lookup[
                    competency_id
                ][
                    "stage"
                ]
            ),

            "statement_fa": (
                competency_lookup[
                    competency_id
                ][
                    "statement_fa"
                ]
            ),

            "matched_terms": (
                matched_terms
            ),
        }
    )


# --------------------------------------------------
# 9. Review signals in the 48 Primary Book
#    activity-evidence rows
# --------------------------------------------------

review_signal_activity_rows_v1 = []


for row in (
    competency_evidence_rows
):

    search_text = (
        normalize_fa_for_match(
            " ".join(
                [
                    str(
                        row.get(
                            "heading",
                            "",
                        )
                    ),
                    str(
                        row.get(
                            "nearest_parent",
                            "",
                        )
                    ),
                ]
            )
        )
    )

    matched_terms = [
        term
        for term in (
            normalized_review_improve_terms_v1
        )
        if term in search_text
    ]

    if not matched_terms:
        continue

    review_signal_activity_rows_v1.append(
        {
            "work_id": (
                row.get(
                    "work_id"
                )
            ),

            "heading": (
                row.get(
                    "heading"
                )
            ),

            "nearest_parent": (
                row.get(
                    "nearest_parent"
                )
            ),

            "activity_type": (
                row.get(
                    "activity_type"
                )
            ),

            "topic_ids": (
                row.get(
                    "topic_ids"
                )
            ),

            "matched_terms": (
                matched_terms
            ),
        }
    )


if (
    current_review_stage_competencies_v1
):

    review_stage_status_v1 = (
        "POPULATED"
    )

elif (
    review_signal_competencies_v1
    or review_signal_activity_rows_v1
):

    review_stage_status_v1 = (
        "UNPOPULATED_WITH_REVIEW_SIGNALS_"
        "REQUIRES_FUTURE_EVIDENCE_REVIEW"
    )

else:

    review_stage_status_v1 = (
        "UNPOPULATED_WITHOUT_DIRECT_MODEL_SIGNAL"
    )


review_and_improve_reconciliation_v1 = {
    "declared_stage": (
        REVIEW_STAGE
    ),

    "current_competency_count": len(
        current_review_stage_competencies_v1
    ),

    "current_competencies": (
        current_review_stage_competencies_v1
    ),

    "competency_text_signal_count": len(
        review_signal_competencies_v1
    ),

    "activity_signal_count": len(
        review_signal_activity_rows_v1
    ),

    "status": (
        review_stage_status_v1
    ),

    "create_competency_automatically": False,

    "knowledge_gap_declared": False,
}


print(
    "\nreview_and_improve Stage Audit:"
)

print(
    "- Current Stage Competencies:",
    len(
        current_review_stage_competencies_v1
    ),
)

print(
    "- Competency-text review signals:",
    len(
        review_signal_competencies_v1
    ),
)

print(
    "- Activity review signals:",
    len(
        review_signal_activity_rows_v1
    ),
)

print(
    "- Status:",
    review_stage_status_v1,
)

print(
    "- Automatic Competency creation:",
    False,
)

print(
    "- Knowledge Gap declared:",
    False,
)


if review_signal_competencies_v1:

    print(
        "\nCompetency review/improve signals:"
    )

    for row in (
        review_signal_competencies_v1
    ):

        print(
            "-",
            row[
                "competency_id"
            ],
            "| stage=",
            row[
                "current_stage"
            ],
            "| terms=",
            row[
                "matched_terms"
            ],
        )


if review_signal_activity_rows_v1:

    print(
        "\nActivity review/improve signals:"
    )

    for row in (
        review_signal_activity_rows_v1[
            :15
        ]
    ):

        print(
            "-",
            row[
                "heading"
            ],
            "| activity=",
            row[
                "activity_type"
            ],
            "| terms=",
            row[
                "matched_terms"
            ],
        )


# --------------------------------------------------
# 10. Reconcile Prerequisite Policy
#
# Preserve v0 exactly.
# Build a new candidate policy.
# --------------------------------------------------

prerequisite_policy_v0_snapshot = (
    deepcopy(
        prerequisite_policy_v0
    )
)


prerequisite_policy_v1_candidate = (
    deepcopy(
        prerequisite_policy_v0
    )
)


prerequisite_policy_v1_candidate[
    "expert_review_required_before_freeze"
] = False


prerequisite_policy_v1_candidate[
    "expert_review_resolution"
] = (
    "SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION"
)


prerequisite_policy_v1_candidate[
    "expert_review_resolution_basis"
] = [
    "ALL_11_PREVIOUS_REVIEW_ITEMS_CORPUS_RESOLVED",
    "INDEPENDENT_TAXONOMY_AUDIT_PASS",
    "TAXONOMY_V1_FROZEN",
]


prerequisite_policy_v1_candidate[
    "hard_prerequisite_count"
] = len(
    hard_prerequisite_relations_v0
)


prerequisite_policy_v1_candidate[
    "recommended_learning_progression_count"
] = len(
    candidate_competency_relations_v0
)


prerequisite_policy_v1_candidate[
    "cross_topic_relation_count"
] = len(
    cross_topic_candidate_relations_v1
)


prerequisite_policy_v1_candidate[
    "curriculum_node_implies_prerequisite"
] = False


prerequisite_policy_v1_candidate[
    "taxonomy_parent_child_implies_prerequisite"
] = False


print(
    "\nPrerequisite Policy reconciliation:"
)

print(
    "- Historical v0 expert-review flag:",
    prerequisite_policy_v0[
        "expert_review_required_before_freeze"
    ],
)

print(
    "- Candidate v1 expert-review flag:",
    prerequisite_policy_v1_candidate[
        "expert_review_required_before_freeze"
    ],
)

print(
    "- Resolution:",
    prerequisite_policy_v1_candidate[
        "expert_review_resolution"
    ],
)

print(
    "- Hard Prerequisites:",
    prerequisite_policy_v1_candidate[
        "hard_prerequisite_count"
    ],
)

print(
    "- Recommended Progressions:",
    prerequisite_policy_v1_candidate[
        "recommended_learning_progression_count"
    ],
)

print(
    "- Cross-topic relations:",
    prerequisite_policy_v1_candidate[
        "cross_topic_relation_count"
    ],
)


# --------------------------------------------------
# 11. Confirm historical Policy was NOT mutated
# --------------------------------------------------

historical_prerequisite_policy_unchanged_v1 = (
    prerequisite_policy_v0
    == prerequisite_policy_v0_snapshot
)


# --------------------------------------------------
# 12. Build Competency reconciliation candidate
# --------------------------------------------------

competency_reconciliation_v1_candidate = {
    "taxonomy_version": (
        "taxonomy_v1"
    ),

    "taxonomy_sha256": (
        taxonomy_v1_frozen_sha256
    ),

    "competency_model_source": (
        "competency_model_v0"
    ),

    "competency_count": (
        24
    ),

    "top_level_topic_count": (
        7
    ),

    "learning_stage_count": (
        6
    ),

    "recommended_progression_count": (
        17
    ),

    "hard_prerequisite_count": (
        0
    ),

    "cross_topic_relation_count": (
        0
    ),

    "progression_cycle_count": (
        0
    ),

    "scope_policy": (
        competency_scope_policy_v1_candidate
    ),

    "curriculum_node_reconciliation": (
        curriculum_node_competency_match_v1
    ),

    "educational_module_reconciliation": {
        "taxation_and_financial_obligations": (
            taxation_competency_extension_v1
        ),
    },

    "stage_reconciliation": {
        "review_and_improve": (
            review_and_improve_reconciliation_v1
        ),
    },

    "prerequisite_policy_candidate": (
        prerequisite_policy_v1_candidate
    ),

    "competencies_added": 0,

    "competencies_removed": 0,

    "competencies_reassigned": 0,

    "relations_added": 0,

    "relations_removed": 0,

    "knowledge_gap_declared": False,

    "status": (
        "RECONCILED_CANDIDATE"
    ),
}


# --------------------------------------------------
# 13. Freeze-readiness blockers
#
# review_and_improve = 0 is intentionally NOT
# an automatic blocker.
# --------------------------------------------------

competency_freeze_blockers_v1 = []


if competency_topic_label_mismatches_v1:

    competency_freeze_blockers_v1.append(
        "TOPIC_LABEL_MISMATCH"
    )


if non_monotonic_progression_relations_v1:

    competency_freeze_blockers_v1.append(
        "BACKWARD_LEARNING_PROGRESSION"
    )


if progression_cycles_v1:

    competency_freeze_blockers_v1.append(
        "LEARNING_PROGRESSION_CYCLE"
    )


if invalid_relation_endpoints_v1:

    competency_freeze_blockers_v1.append(
        "INVALID_RELATION_ENDPOINT"
    )


if invalid_relation_types_v1:

    competency_freeze_blockers_v1.append(
        "INVALID_RELATION_TYPE"
    )


if cross_topic_candidate_relations_v1:

    competency_freeze_blockers_v1.append(
        "UNRESOLVED_CROSS_TOPIC_RELATION"
    )


if hard_prerequisite_relations_v0:

    competency_freeze_blockers_v1.append(
        "UNVALIDATED_HARD_PREREQUISITE"
    )


if not historical_prerequisite_policy_unchanged_v1:

    competency_freeze_blockers_v1.append(
        "HISTORICAL_POLICY_MUTATED"
    )


competency_freeze_readiness_v1 = (
    "READY_FOR_COMPETENCY_FREEZE"
    if not competency_freeze_blockers_v1
    else "BLOCKED"
)


# --------------------------------------------------
# 14. Reconciliation summary
# --------------------------------------------------

print(
    "\nCompetency / Prerequisite Reconciliation Summary:"
)

print(
    "- Competencies preserved:",
    len(
        competency_lookup
    ),
)

print(
    "- Competencies added:",
    0,
)

print(
    "- Competencies removed:",
    0,
)

print(
    "- Competencies reassigned:",
    0,
)

print(
    "- Recommended progressions preserved:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Hard Prerequisites:",
    len(
        hard_prerequisite_relations_v0
    ),
)

print(
    "- Cross-topic relations:",
    len(
        cross_topic_candidate_relations_v1
    ),
)

print(
    "- Progression cycles:",
    len(
        progression_cycles_v1
    ),
)

print(
    "- review_and_improve status:",
    review_stage_status_v1,
)

print(
    "- Knowledge Gap declared:",
    False,
)

print(
    "- Historical Prerequisite Policy unchanged:",
    historical_prerequisite_policy_unchanged_v1,
)

print(
    "- Freeze blockers:",
    competency_freeze_blockers_v1,
)

print(
    "- Freeze readiness:",
    competency_freeze_readiness_v1,
)


# --------------------------------------------------
# 15. No mutation / no Disk write
# --------------------------------------------------

taxonomy_mutation_performed = False
competency_mutation_performed = False
historical_prerequisite_policy_mutation_performed = False
topic_mapping_mutation_performed = False
disk_write_performed = False


frozen_taxonomy_sha_after_reconciliation_v1 = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


# --------------------------------------------------
# 16. Validation
# --------------------------------------------------

print(
    "\nValidation:"
)

print(
    "- Frozen Taxonomy SHA unchanged:",
    (
        frozen_taxonomy_sha_after_reconciliation_v1
        == frozen_taxonomy_sha_before_reconciliation_v1
        == taxonomy_v1_frozen_sha256
    ),
)

print(
    "- Topic label mismatches:",
    len(
        competency_topic_label_mismatches_v1
    ),
)

print(
    "- Backward progression relations:",
    len(
        non_monotonic_progression_relations_v1
    ),
)

print(
    "- Existing Competencies:",
    len(
        competency_lookup
    ),
)

print(
    "- Existing Recommended relations:",
    len(
        candidate_competency_relations_v0
    ),
)

print(
    "- Existing Hard Prerequisites:",
    len(
        hard_prerequisite_relations_v0
    ),
)

print(
    "- Review Stage Competencies:",
    len(
        current_review_stage_competencies_v1
    ),
)

print(
    "- Competency Freeze readiness:",
    competency_freeze_readiness_v1,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Competency mutation performed:",
    competency_mutation_performed,
)

print(
    "- Historical Prerequisite Policy mutation:",
    historical_prerequisite_policy_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


assert len(
    competency_lookup
) == 24

assert len(
    competency_model_v0
) == 7

assert len(
    candidate_competency_relations_v0
) == 17

assert len(
    hard_prerequisite_relations_v0
) == 0

assert not competency_topic_label_mismatches_v1

assert not non_monotonic_progression_relations_v1

assert not progression_cycles_v1

assert not invalid_relation_endpoints_v1
assert not invalid_relation_types_v1

assert not cross_topic_candidate_relations_v1

assert (
    prerequisite_policy_v0[
        "expert_review_required_before_freeze"
    ]
    is True
)

assert (
    prerequisite_policy_v1_candidate[
        "expert_review_required_before_freeze"
    ]
    is False
)

assert (
    prerequisite_policy_v1_candidate[
        "expert_review_resolution"
    ]
    == "SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION"
)

assert historical_prerequisite_policy_unchanged_v1

assert (
    competency_scope_policy_v1_candidate[
        "curriculum_node_requires_dedicated_competency"
    ]
    is False
)

assert (
    competency_scope_policy_v1_candidate[
        "absence_of_dedicated_node_competency_is_gap"
    ]
    is False
)

assert (
    review_and_improve_reconciliation_v1[
        "create_competency_automatically"
    ]
    is False
)

assert (
    review_and_improve_reconciliation_v1[
        "knowledge_gap_declared"
    ]
    is False
)

assert not competency_freeze_blockers_v1

assert (
    competency_freeze_readiness_v1
    == "READY_FOR_COMPETENCY_FREEZE"
)

assert (
    frozen_taxonomy_sha_after_reconciliation_v1
    == taxonomy_v1_frozen_sha256
)

assert taxonomy_mutation_performed is False
assert competency_mutation_performed is False
assert historical_prerequisite_policy_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert disk_write_performed is False


print(
    "\nCODE CELL 41 GATE: PASS"
)

print(
    "Frozen-Taxonomy Competency / Prerequisite "
    "Reconciliation PASS: all 24 existing "
    "Competencies and 17 recommended learning "
    "progressions remain valid against taxonomy_v1; "
    "the obsolete expert-review freeze flag is "
    "superseded in a new candidate policy without "
    "mutating historical v0, no new Competency was "
    "fabricated for Taxonomy Nodes or the empty "
    "review_and_improve stage, and the model is "
    "READY for an explicit Competency Freeze."
)

Frozen Topic / Competency label reconciliation:
- Topic groups: 7
- Label mismatches: 0

Learning Progression monotonicity:
- Recommended relations: 17
- Backward-stage relations: 0

Competency Scope Policy candidate:
- model_scope : TOP_LEVEL_COMPETENCY_FRAMEWORK
- taxonomy_version : taxonomy_v1
- top_level_topic_requires_competency : True
- curriculum_node_requires_dedicated_competency : False
- educational_module_requires_dedicated_competency : False
- competency_creation_requires : ['PRIMARY_CURRICULUM_EVIDENCE', 'DISTINCT_LEARNING_OUTCOME', 'NON_DUPLICATIVE_WITH_EXISTING_COMPETENCY']
- absence_of_dedicated_node_competency_is_gap : False
- absence_of_stage_population_is_gap : False
- knowledge_gap_requires_separate_evaluation : True

New Curriculum Nodes → existing Competency evidence:

Node: بخشش و هدیه (giving_and_gifting)
- Top-level Topic: spending_saving_money_management
- Parent-topic Competencies: ['SSM-C01', 'SSM-C02', 'SSM-C03', 'SSM-C04']
- Direct existing Competency matc

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h3>Competency / Prerequisite v1 Freeze — تثبیت مدل توانمندی و مسیر یادگیری</h3>

<p>
پس از Reconciliation با Taxonomy v1 Frozen، مدل Competency موجود بدون نیاز به بازطراحی معتبر باقی ماند.
این مرحله نسخۀ رسمی Competency / Prerequisite v1 را به‌صورت Artifact تغییرناپذیر Freeze می‌کند.
</p>

<h4>ساختار تثبیت‌شده</h4>

<ul>
<li>24 Competency</li>
<li>7 Top-level Topic دارای Competency</li>
<li>6 Learning Stage تعریف‌شده</li>
<li>17 Recommended Learning Progression</li>
<li>0 Hard Prerequisite</li>
<li>0 Cross-topic Relation</li>
<li>0 Progression Cycle</li>
</ul>

<h4>تصمیم مهم درباره Taxonomy Nodeها</h4>

<p>
افزوده‌شدن Curriculum Node یا Level-2 Module به Taxonomy الزاماً به معنی ایجاد Competency مستقل نیست.
Competency جدید فقط در صورت وجود Learning Outcome مستقل، Evidence مستقیم Primary و نبود هم‌پوشانی با Competencyهای موجود ایجاد می‌شود.
</p>

<p>
در نسخۀ فعلی هیچ Competency جدیدی برای «بخشش و هدیه»، «کسب‌وکار کودکان»، زیرموضوع‌های آن یا Module «مالیات» ساخته نشده است.
این یک تصمیم آگاهانه است، نه کمبود خودکار مدل.
</p>

<h4>Stage review_and_improve</h4>

<p>
Stage <code>review_and_improve</code> در مدل آموزشی وجود دارد ولی فعلاً Competency مستقلی در آن قرار نگرفته است.
سه Competency فعلی نشانه‌هایی از بازبینی یا اصلاح دارند، اما Evidence فعلی برای ایجاد یک Competency مستقل کافی تشخیص داده نشده است.
بنابراین این Stage حفظ می‌شود و برای بررسی آتی علامت‌گذاری می‌شود؛ Knowledge Gap نیز اعلام نمی‌شود.
</p>

<h4>Prerequisite Policy</h4>

<p>
Policy تاریخی v0 بدون تغییر حفظ می‌شود.
فلگ قدیمی <code>expert_review_required_before_freeze=True</code> در Policy جدید با وضعیت
<code>SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION</code>
جایگزین می‌شود، زیرا هر 11 Review Item قبلی با خود Corpus Resolve شده‌اند.
</p>

<h4>Run-All Safety</h4>

<p>
Artifact Freeze تغییرناپذیر است:
اگر فایل موجود دقیقاً برابر Payload فعلی باشد Reuse می‌شود؛
اگر متفاوت باشد Cell به‌جای Overwrite کردن فایل قبلی FAIL می‌شود.
</p>

<h4>خروجی‌ها</h4>

<ul>
<li><code>artifacts/phase2/competency/competency_prerequisite_v1.json</code></li>
<li><code>reports/phase2/competency_freeze_v1.json</code></li>
</ul>

<p>
پس از Write، هر دو فایل مجدداً از Disk خوانده، Hash و مستقل Validate می‌شوند.
</p>

</div>

In [85]:
# Code Cell 42

from pathlib import Path
from copy import deepcopy
from collections import Counter
import json


# --------------------------------------------------
# 1. Freeze prerequisites
# --------------------------------------------------

assert (
    competency_freeze_readiness_v1
    == "READY_FOR_COMPETENCY_FREEZE"
)

assert not competency_freeze_blockers_v1

assert taxonomy_freeze_executed_v1 is True

assert (
    taxonomy_v1_frozen[
        "status"
    ]
    == "FROZEN"
)

assert (
    taxonomy_v1_frozen_sha256
    == "335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977"
)

assert len(
    competency_lookup
) == 24

assert len(
    competency_model_v0
) == 7

assert len(
    learning_stage_rank
) == 6

assert len(
    candidate_competency_relations_v0
) == 17

assert len(
    hard_prerequisite_relations_v0
) == 0

assert len(
    cross_topic_candidate_relations_v1
) == 0

assert len(
    progression_cycles_v1
) == 0


# --------------------------------------------------
# 2. Output paths
# --------------------------------------------------

competency_artifact_dir = (
    repo_root
    / "artifacts"
    / "phase2"
    / "competency"
)


competency_artifact_path_v1 = (
    competency_artifact_dir
    / "competency_prerequisite_v1.json"
)


competency_freeze_report_path_v1 = (
    repo_root
    / "reports"
    / "phase2"
    / "competency_freeze_v1.json"
)


print(
    "Competency Freeze paths:"
)

print(
    "- Artifact:",
    competency_artifact_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Report:",
    competency_freeze_report_path_v1.relative_to(
        repo_root
    ),
)


# --------------------------------------------------
# 3. Independent canonical Competency list
# --------------------------------------------------

frozen_competency_list_v1 = []


for topic_id in (
    taxonomy_v1_frozen[
        "top_level_topics"
    ]
):

    topic_model = (
        competency_model_v0[
            topic_id
        ]
    )

    for competency in (
        topic_model[
            "competencies"
        ]
    ):

        competency_id = (
            competency[
                "id"
            ]
        )

        lookup_row = (
            competency_lookup[
                competency_id
            ]
        )

        frozen_competency_list_v1.append(
            {
                "competency_id": (
                    competency_id
                ),

                "topic_id": (
                    topic_id
                ),

                "topic_label_fa": (
                    taxonomy_v1_frozen[
                        "top_level_topics"
                    ][
                        topic_id
                    ][
                        "label_fa"
                    ]
                ),

                "stage": (
                    competency[
                        "stage"
                    ]
                ),

                "statement_fa": (
                    competency[
                        "statement_fa"
                    ]
                ),

                "evidence": list(
                    competency.get(
                        "evidence",
                        [],
                    )
                ),

                "lookup_consistent": (
                    lookup_row[
                        "topic_id"
                    ]
                    == topic_id
                    and lookup_row[
                        "stage"
                    ]
                    == competency[
                        "stage"
                    ]
                    and lookup_row[
                        "statement_fa"
                    ]
                    == competency[
                        "statement_fa"
                    ]
                ),
            }
        )


frozen_competency_ids_v1 = [
    row[
        "competency_id"
    ]
    for row in (
        frozen_competency_list_v1
    )
]


frozen_competency_stage_counter_v1 = Counter(
    row[
        "stage"
    ]
    for row in (
        frozen_competency_list_v1
    )
)


frozen_competency_topic_counter_v1 = Counter(
    row[
        "topic_id"
    ]
    for row in (
        frozen_competency_list_v1
    )
)


print(
    "\nCanonical Competency inventory:"
)

print(
    "- Competencies:",
    len(
        frozen_competency_list_v1
    ),
)

print(
    "- Distinct IDs:",
    len(
        set(
            frozen_competency_ids_v1
        )
    ),
)


print(
    "\nCompetencies by Topic:"
)

for topic_id in (
    taxonomy_v1_frozen[
        "top_level_topics"
    ]
):

    print(
        "-",
        topic_id,
        ":",
        frozen_competency_topic_counter_v1[
            topic_id
        ],
    )


print(
    "\nCompetencies by Stage:"
)

for stage in sorted(
    learning_stage_rank,
    key=lambda stage_name: (
        learning_stage_rank[
            stage_name
        ]
    ),
):

    print(
        "-",
        stage,
        ":",
        frozen_competency_stage_counter_v1[
            stage
        ],
    )


# --------------------------------------------------
# 4. Canonical Learning Progression relations
# --------------------------------------------------

frozen_recommended_progressions_v1 = []


for relation in sorted(
    candidate_competency_relations_v0,
    key=lambda row: (
        row[
            "from"
        ],
        row[
            "to"
        ],
        row[
            "type"
        ],
    ),
):

    frozen_recommended_progressions_v1.append(
        {
            "from": (
                relation[
                    "from"
                ]
            ),

            "to": (
                relation[
                    "to"
                ]
            ),

            "type": (
                relation[
                    "type"
                ]
            ),

            "reason": (
                relation[
                    "reason"
                ]
            ),

            "evidence_basis": (
                relation[
                    "evidence_basis"
                ]
            ),
        }
    )


assert all(
    relation[
        "type"
    ]
    == "RECOMMENDED_LEARNING_PROGRESSION"
    for relation in (
        frozen_recommended_progressions_v1
    )
)


# --------------------------------------------------
# 5. Freeze payload
# --------------------------------------------------

competency_prerequisite_v1_payload = {
    "schema_version": (
        "competency-prerequisite-schema-v1"
    ),

    "competency_model_version": (
        "competency_prerequisite_v1"
    ),

    "status": (
        "FROZEN"
    ),

    "taxonomy_reference": {
        "taxonomy_version": (
            "taxonomy_v1"
        ),

        "taxonomy_path": str(
            taxonomy_v1_frozen_path.relative_to(
                repo_root
            )
        ),

        "taxonomy_sha256": (
            taxonomy_v1_frozen_sha256
        ),
    },

    "model_scope": deepcopy(
        competency_scope_policy_v1_candidate
    ),

    "learning_stages": {
        "rank": deepcopy(
            learning_stage_rank
        ),

        "declared_stage_count": len(
            learning_stage_rank
        ),

        "competency_population": {
            stage: (
                frozen_competency_stage_counter_v1[
                    stage
                ]
            )
            for stage in (
                learning_stage_rank
            )
        },

        "review_and_improve_reconciliation": deepcopy(
            review_and_improve_reconciliation_v1
        ),
    },

    "competencies": deepcopy(
        frozen_competency_list_v1
    ),

    "competency_summary": {
        "total_competencies": (
            len(
                frozen_competency_list_v1
            )
        ),

        "distinct_competency_ids": (
            len(
                set(
                    frozen_competency_ids_v1
                )
            )
        ),

        "topic_count": (
            len(
                competency_model_v0
            )
        ),

        "competencies_by_topic": {
            topic_id: (
                frozen_competency_topic_counter_v1[
                    topic_id
                ]
            )
            for topic_id in (
                taxonomy_v1_frozen[
                    "top_level_topics"
                ]
            )
        },

        "competencies_by_stage": {
            stage: (
                frozen_competency_stage_counter_v1[
                    stage
                ]
            )
            for stage in (
                learning_stage_rank
            )
        },
    },

    "learning_progression": {
        "relationship_type": (
            "RECOMMENDED_LEARNING_PROGRESSION"
        ),

        "relations": deepcopy(
            frozen_recommended_progressions_v1
        ),

        "relation_count": len(
            frozen_recommended_progressions_v1
        ),

        "cross_topic_relation_count": (
            len(
                cross_topic_candidate_relations_v1
            )
        ),

        "cycle_count": (
            len(
                progression_cycles_v1
            )
        ),
    },

    "hard_prerequisites": {
        "relations": deepcopy(
            hard_prerequisite_relations_v0
        ),

        "relation_count": (
            len(
                hard_prerequisite_relations_v0
            )
        ),
    },

    "prerequisite_policy": {
        "historical_v0": deepcopy(
            prerequisite_policy_v0
        ),

        "frozen_v1": deepcopy(
            prerequisite_policy_v1_candidate
        ),

        "historical_v0_preserved": (
            historical_prerequisite_policy_unchanged_v1
        ),
    },

    "taxonomy_reconciliation": {
        "curriculum_nodes": deepcopy(
            curriculum_node_competency_match_v1
        ),

        "educational_modules": {
            "taxation_and_financial_obligations": deepcopy(
                taxation_competency_extension_v1
            )
        },

        "competencies_added_for_new_taxonomy_nodes": (
            0
        ),

        "competencies_removed": (
            0
        ),

        "competencies_reassigned": (
            0
        ),
    },

    "knowledge_gap_policy": {
        "absence_of_dedicated_node_competency_is_gap": (
            False
        ),

        "absence_of_stage_population_is_gap": (
            False
        ),

        "knowledge_gap_declared_in_nb10": (
            False
        ),

        "requires_separate_evaluation": (
            True
        ),
    },

    "validation_snapshot": {
        "competency_count": (
            24
        ),

        "topic_count": (
            7
        ),

        "learning_stage_count": (
            6
        ),

        "recommended_progression_count": (
            17
        ),

        "hard_prerequisite_count": (
            0
        ),

        "cross_topic_relation_count": (
            0
        ),

        "progression_cycle_count": (
            0
        ),

        "topic_label_mismatch_count": len(
            competency_topic_label_mismatches_v1
        ),

        "backward_progression_count": len(
            non_monotonic_progression_relations_v1
        ),

        "invalid_relation_endpoint_count": len(
            invalid_relation_endpoints_v1
        ),

        "invalid_relation_type_count": len(
            invalid_relation_types_v1
        ),

        "competencies_without_model_evidence": len(
            competencies_without_model_evidence
        ),

        "topics_without_competency": len(
            topics_without_competency
        ),

        "duplicate_competency_ids": len(
            duplicate_competency_ids
        ),

        "freeze_blockers": list(
            competency_freeze_blockers_v1
        ),
    },
}


# --------------------------------------------------
# 6. Pre-write independent validation
# --------------------------------------------------

assert (
    competency_prerequisite_v1_payload[
        "status"
    ]
    == "FROZEN"
)

assert (
    competency_prerequisite_v1_payload[
        "taxonomy_reference"
    ][
        "taxonomy_sha256"
    ]
    == taxonomy_v1_frozen_sha256
)

assert len(
    competency_prerequisite_v1_payload[
        "competencies"
    ]
) == 24

assert (
    competency_prerequisite_v1_payload[
        "competency_summary"
    ][
        "distinct_competency_ids"
    ]
    == 24
)

assert (
    competency_prerequisite_v1_payload[
        "learning_progression"
    ][
        "relation_count"
    ]
    == 17
)

assert (
    competency_prerequisite_v1_payload[
        "hard_prerequisites"
    ][
        "relation_count"
    ]
    == 0
)

assert (
    competency_prerequisite_v1_payload[
        "learning_progression"
    ][
        "cross_topic_relation_count"
    ]
    == 0
)

assert (
    competency_prerequisite_v1_payload[
        "learning_progression"
    ][
        "cycle_count"
    ]
    == 0
)

assert (
    competency_prerequisite_v1_payload[
        "prerequisite_policy"
    ][
        "frozen_v1"
    ][
        "expert_review_required_before_freeze"
    ]
    is False
)

assert (
    competency_prerequisite_v1_payload[
        "prerequisite_policy"
    ][
        "historical_v0"
    ][
        "expert_review_required_before_freeze"
    ]
    is True
)

assert all(
    row[
        "lookup_consistent"
    ]
    is True
    for row in (
        competency_prerequisite_v1_payload[
            "competencies"
        ]
    )
)

assert not (
    competency_prerequisite_v1_payload[
        "validation_snapshot"
    ][
        "freeze_blockers"
    ]
)


# --------------------------------------------------
# 7. Immutable Artifact materialization
# --------------------------------------------------

competency_materialization_v1 = (
    materialize_immutable_json(
        competency_artifact_path_v1,
        competency_prerequisite_v1_payload,
    )
)


competency_prerequisite_v1_sha256 = (
    competency_materialization_v1[
        "sha256"
    ]
)


print(
    "\nCompetency / Prerequisite v1 materialization:"
)

print(
    "- Path:",
    competency_artifact_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Action:",
    competency_materialization_v1[
        "action"
    ],
)

print(
    "- SHA256:",
    competency_prerequisite_v1_sha256,
)

print(
    "- Size bytes:",
    competency_materialization_v1[
        "size_bytes"
    ],
)


# --------------------------------------------------
# 8. Freeze report
# --------------------------------------------------

competency_freeze_report_v1 = {
    "schema_version": (
        "competency-freeze-report-v1"
    ),

    "competency_model_version": (
        "competency_prerequisite_v1"
    ),

    "freeze_status": (
        "PASS"
    ),

    "artifact": {
        "path": str(
            competency_artifact_path_v1.relative_to(
                repo_root
            )
        ),

        "sha256": (
            competency_prerequisite_v1_sha256
        ),
    },

    "taxonomy_reference": {
        "path": str(
            taxonomy_v1_frozen_path.relative_to(
                repo_root
            )
        ),

        "sha256": (
            taxonomy_v1_frozen_sha256
        ),
    },

    "model_summary": {
        "competencies": (
            24
        ),

        "top_level_topics": (
            7
        ),

        "learning_stages": (
            6
        ),

        "recommended_progressions": (
            17
        ),

        "hard_prerequisites": (
            0
        ),

        "cross_topic_relations": (
            0
        ),

        "progression_cycles": (
            0
        ),
    },

    "taxonomy_reconciliation": {
        "competencies_added": (
            0
        ),

        "competencies_removed": (
            0
        ),

        "competencies_reassigned": (
            0
        ),

        "curriculum_node_requires_dedicated_competency": (
            False
        ),

        "educational_module_requires_dedicated_competency": (
            False
        ),
    },

    "review_and_improve": deepcopy(
        review_and_improve_reconciliation_v1
    ),

    "prerequisite_policy": {
        "historical_expert_review_flag": (
            prerequisite_policy_v0[
                "expert_review_required_before_freeze"
            ]
        ),

        "frozen_expert_review_flag": (
            prerequisite_policy_v1_candidate[
                "expert_review_required_before_freeze"
            ]
        ),

        "resolution": (
            prerequisite_policy_v1_candidate[
                "expert_review_resolution"
            ]
        ),
    },

    "knowledge_gap_declared": (
        False
    ),

    "freeze_blockers": list(
        competency_freeze_blockers_v1
    ),

    "next_required_nb10_stage": (
        "FINAL_NB10_CHECKPOINT_AND_NOTEBOOK_HEALTH"
    ),
}


competency_report_materialization_v1 = (
    materialize_immutable_json(
        competency_freeze_report_path_v1,
        competency_freeze_report_v1,
    )
)


competency_freeze_report_sha256_v1 = (
    competency_report_materialization_v1[
        "sha256"
    ]
)


print(
    "\nCompetency Freeze Report materialization:"
)

print(
    "- Path:",
    competency_freeze_report_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Action:",
    competency_report_materialization_v1[
        "action"
    ],
)

print(
    "- SHA256:",
    competency_freeze_report_sha256_v1,
)

print(
    "- Size bytes:",
    competency_report_materialization_v1[
        "size_bytes"
    ],
)


# --------------------------------------------------
# 9. Independent disk re-read
# --------------------------------------------------

competency_prerequisite_v1_disk = json.loads(
    competency_artifact_path_v1.read_text(
        encoding="utf-8"
    )
)


competency_freeze_report_v1_disk = json.loads(
    competency_freeze_report_path_v1.read_text(
        encoding="utf-8"
    )
)


competency_disk_sha256_v1 = (
    sha256_file(
        competency_artifact_path_v1
    )
)


competency_report_disk_sha256_v1 = (
    sha256_file(
        competency_freeze_report_path_v1
    )
)


print(
    "\nIndependent Disk re-validation:"
)

print(
    "- Competency disk SHA256:",
    competency_disk_sha256_v1,
)

print(
    "- Competency expected SHA256:",
    competency_prerequisite_v1_sha256,
)

print(
    "- Report disk SHA256:",
    competency_report_disk_sha256_v1,
)

print(
    "- Report expected SHA256:",
    competency_freeze_report_sha256_v1,
)


# --------------------------------------------------
# 10. Independent graph validation from
#     re-read Frozen Artifact
# --------------------------------------------------

disk_competencies_v1 = (
    competency_prerequisite_v1_disk[
        "competencies"
    ]
)


disk_competency_ids_v1 = {
    row[
        "competency_id"
    ]
    for row in (
        disk_competencies_v1
    )
}


disk_progressions_v1 = (
    competency_prerequisite_v1_disk[
        "learning_progression"
    ][
        "relations"
    ]
)


disk_invalid_relation_endpoints_v1 = []


for relation in (
    disk_progressions_v1
):

    if (
        relation[
            "from"
        ]
        not in disk_competency_ids_v1
        or relation[
            "to"
        ]
        not in disk_competency_ids_v1
    ):

        disk_invalid_relation_endpoints_v1.append(
            relation
        )


disk_graph_v1 = {
    competency_id: []
    for competency_id in (
        disk_competency_ids_v1
    )
}


for relation in (
    disk_progressions_v1
):

    disk_graph_v1[
        relation[
            "from"
        ]
    ].append(
        relation[
            "to"
        ]
    )


disk_progression_cycles_v1 = []


def detect_disk_cycles_v1():

    visiting = set()
    visited = set()
    path = []

    def visit(
        node,
    ):

        if node in visiting:

            cycle_start = (
                path.index(
                    node
                )
            )

            disk_progression_cycles_v1.append(
                tuple(
                    path[
                        cycle_start:
                    ]
                    + [
                        node
                    ]
                )
            )

            return

        if node in visited:
            return

        visiting.add(
            node
        )

        path.append(
            node
        )

        for neighbor in (
            disk_graph_v1[
                node
            ]
        ):

            visit(
                neighbor
            )

        path.pop()

        visiting.remove(
            node
        )

        visited.add(
            node
        )

    for node in (
        disk_graph_v1
    ):

        visit(
            node
        )


detect_disk_cycles_v1()


disk_progression_cycles_v1 = sorted(
    set(
        disk_progression_cycles_v1
    )
)


# --------------------------------------------------
# 11. Frozen state variables for downstream NB10
# --------------------------------------------------

competency_prerequisite_v1_frozen = (
    competency_prerequisite_v1_disk
)


competency_prerequisite_v1_frozen_path = (
    competency_artifact_path_v1
)


competency_prerequisite_v1_frozen_sha256 = (
    competency_disk_sha256_v1
)


competency_freeze_report_v1_frozen = (
    competency_freeze_report_v1_disk
)


competency_freeze_report_v1_frozen_sha256 = (
    competency_report_disk_sha256_v1
)


competency_freeze_executed_v1 = True


# --------------------------------------------------
# 12. Final validation
# --------------------------------------------------

taxonomy_sha_after_competency_freeze_v1 = (
    sha256_file(
        taxonomy_v1_frozen_path
    )
)


historical_policy_still_unchanged_v1 = (
    prerequisite_policy_v0
    == prerequisite_policy_v0_snapshot
)


source_document_mutation_performed = False
topic_mapping_mutation_performed = False
taxonomy_mutation_performed = False
historical_policy_mutation_performed = False


print(
    "\nValidation:"
)

print(
    "- Competency Artifact exists:",
    competency_artifact_path_v1.exists(),
)

print(
    "- Competency Freeze Report exists:",
    competency_freeze_report_path_v1.exists(),
)

print(
    "- Frozen status:",
    competency_prerequisite_v1_disk[
        "status"
    ],
)

print(
    "- Frozen Competencies:",
    len(
        disk_competencies_v1
    ),
)

print(
    "- Frozen Progressions:",
    len(
        disk_progressions_v1
    ),
)

print(
    "- Frozen Hard Prerequisites:",
    competency_prerequisite_v1_disk[
        "hard_prerequisites"
    ][
        "relation_count"
    ],
)

print(
    "- Disk invalid relation endpoints:",
    len(
        disk_invalid_relation_endpoints_v1
    ),
)

print(
    "- Disk progression cycles:",
    len(
        disk_progression_cycles_v1
    ),
)

print(
    "- Competency SHA verified:",
    (
        competency_disk_sha256_v1
        == competency_prerequisite_v1_sha256
    ),
)

print(
    "- Report SHA verified:",
    (
        competency_report_disk_sha256_v1
        == competency_freeze_report_sha256_v1
    ),
)

print(
    "- Taxonomy SHA unchanged:",
    (
        taxonomy_sha_after_competency_freeze_v1
        == taxonomy_v1_frozen_sha256
    ),
)

print(
    "- Historical Policy unchanged:",
    historical_policy_still_unchanged_v1,
)

print(
    "- Source Document mutation performed:",
    source_document_mutation_performed,
)

print(
    "- Topic Mapping mutation performed:",
    topic_mapping_mutation_performed,
)

print(
    "- Taxonomy mutation performed:",
    taxonomy_mutation_performed,
)

print(
    "- Historical Policy mutation performed:",
    historical_policy_mutation_performed,
)

print(
    "- Competency Freeze executed:",
    competency_freeze_executed_v1,
)


# --------------------------------------------------
# 13. Final assertions
# --------------------------------------------------

assert competency_artifact_path_v1.exists()

assert (
    competency_freeze_report_path_v1.exists()
)

assert (
    competency_disk_sha256_v1
    == competency_prerequisite_v1_sha256
)

assert (
    competency_report_disk_sha256_v1
    == competency_freeze_report_sha256_v1
)

assert (
    competency_prerequisite_v1_disk[
        "status"
    ]
    == "FROZEN"
)

assert (
    competency_prerequisite_v1_disk[
        "competency_model_version"
    ]
    == "competency_prerequisite_v1"
)

# IMPORTANT:
# taxonomy_reference schema uses taxonomy_sha256,
# not sha256.
assert (
    competency_prerequisite_v1_disk[
        "taxonomy_reference"
    ][
        "taxonomy_sha256"
    ]
    == taxonomy_v1_frozen_sha256
)

assert len(
    disk_competencies_v1
) == 24

assert len(
    disk_competency_ids_v1
) == 24

assert len(
    disk_progressions_v1
) == 17

assert (
    competency_prerequisite_v1_disk[
        "hard_prerequisites"
    ][
        "relation_count"
    ]
    == 0
)

assert not disk_invalid_relation_endpoints_v1

assert not disk_progression_cycles_v1

assert (
    competency_prerequisite_v1_disk[
        "learning_progression"
    ][
        "cross_topic_relation_count"
    ]
    == 0
)

assert (
    competency_prerequisite_v1_disk[
        "prerequisite_policy"
    ][
        "historical_v0"
    ][
        "expert_review_required_before_freeze"
    ]
    is True
)

assert (
    competency_prerequisite_v1_disk[
        "prerequisite_policy"
    ][
        "frozen_v1"
    ][
        "expert_review_required_before_freeze"
    ]
    is False
)

assert (
    competency_prerequisite_v1_disk[
        "knowledge_gap_policy"
    ][
        "knowledge_gap_declared_in_nb10"
    ]
    is False
)

assert (
    taxonomy_sha_after_competency_freeze_v1
    == taxonomy_v1_frozen_sha256
)

assert historical_policy_still_unchanged_v1

assert source_document_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert taxonomy_mutation_performed is False
assert historical_policy_mutation_performed is False

assert competency_freeze_executed_v1 is True


print(
    "\nCODE CELL 42 GATE: PASS"
)

print(
    "Competency / Prerequisite v1 Freeze PASS: "
    "the validated 24-competency model, six-stage "
    "framework, 17 recommended learning progressions "
    "and reconciled prerequisite policy have been "
    "immutably materialized and independently "
    "re-validated against frozen taxonomy_v1, "
    "without fabricating new competencies or "
    "hard prerequisites."
)

Competency Freeze paths:
- Artifact: artifacts/phase2/competency/competency_prerequisite_v1.json
- Report: reports/phase2/competency_freeze_v1.json

Canonical Competency inventory:
- Competencies: 24
- Distinct IDs: 24

Competencies by Topic:
- credit_debt_banking : 3
- financial_decision_making : 4
- financial_literacy_foundations : 3
- investment : 4
- risk_insurance : 3
- spending_saving_money_management : 4
- work_income : 3

Competencies by Stage:
- understand : 5
- self_assess : 7
- analyze : 2
- plan : 8
- act : 2
- review_and_improve : 0

Competency / Prerequisite v1 materialization:
- Path: artifacts/phase2/competency/competency_prerequisite_v1.json
- Action: REUSED_IDENTICAL
- SHA256: f12f6c1631d77f29042b2009ca9dc8dee406d23ccdccdbb103c0e8cade781e87
- Size bytes: 26332

Competency Freeze Report materialization:
- Path: reports/phase2/competency_freeze_v1.json
- Action: REUSED_IDENTICAL
- SHA256: 19bb27288f49e057859da3bc668f54fac440a89a2636ff5d2e49033665f0dd11
- Size bytes: 165

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h3>Final NB10 Preflight — Artifact Integrity & Notebook Structural Health</h3>

<p>
Taxonomy v1 و Competency / Prerequisite v1 اکنون Freeze شده‌اند.
پیش از Restart Kernel و Run All، این مرحله یک بررسی مستقل روی Artifactهای نهایی و خود فایل Notebook انجام می‌دهد.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>SHA256 Corpus پذیرفته‌شده NB09</li>
<li>SHA256 Acceptance Report</li>
<li>SHA256 Taxonomy v1</li>
<li>SHA256 Taxonomy Freeze Report</li>
<li>SHA256 Competency / Prerequisite v1</li>
<li>SHA256 Competency Freeze Report</li>
<li>Cross-reference بین Artifactها و Reportها</li>
<li>کشف فایل واقعی Notebook 10</li>
<li>Parse واقعی فایل <code>.ipynb</code></li>
<li>بررسی توالی Code Cellهای 01 تا 42</li>
<li>بررسی Duplicate یا Missing Cell Number</li>
<li>بررسی Syntax تمام Code Cellها</li>
<li>بررسی وجود Markdown مربوط به جمع‌بندی مسائل حل‌شده</li>
</ul>

<p>
Execution Count در این مرحله معیار PASS نیست، زیرا Notebook هنوز Restart/Run All نهایی نشده است.
همچنین Error Output نهایی در مرحلۀ Post-Run-All Health بررسی خواهد شد.
</p>

<h4>این Cell هیچ تغییری ایجاد نمی‌کند</h4>

<ul>
<li>Artifact جدید نمی‌سازد.</li>
<li>Taxonomy را تغییر نمی‌دهد.</li>
<li>Competency را تغییر نمی‌دهد.</li>
<li>Mapping را تغییر نمی‌دهد.</li>
<li>Notebook را بازنویسی نمی‌کند.</li>
</ul>

</div>

In [86]:
# Code Cell 43

from pathlib import Path
from collections import Counter, defaultdict
import ast
import difflib
import hashlib
import json
import re


# ==================================================
# 1. Repository / Frozen-state prerequisites
# ==================================================

assert repo_root.exists()

assert taxonomy_freeze_executed_v1 is True
assert competency_freeze_executed_v1 is True


EXPECTED_HASHES_NB10_V1 = {
    "accepted_corpus": (
        "9d72c709f81b1218c5aac064052c7ed1"
        "da34869f671e6e32762864e0ad70ea40"
    ),

    "nb09_acceptance_report": (
        "14dfa748988db17a19c6d9d5e1eac2e"
        "b7d23c0a1699dcc7add18a2bf461fe4f3"
    ),

    "taxonomy_v1": (
        "335690044b2e4eff684f7f347809f2c5"
        "cb8c39624a939dc01408d9ca898fe977"
    ),

    "taxonomy_freeze_report_v1": (
        "fe182b0c7c6d5ad7d5dfc2aa794fa096"
        "e85001b517040a8396d5fb4c0b17e530"
    ),

    "competency_prerequisite_v1": (
        "f12f6c1631d77f29042b2009ca9dc8de"
        "e406d23ccdccdbb103c0e8cade781e87"
    ),

    "competency_freeze_report_v1": (
        "19bb27288f49e057859da3bc668f54fac"
        "440a89a2636ff5d2e49033665f0dd11"
    ),
}


# ==================================================
# 2. Final Artifact paths
# ==================================================

nb10_final_paths_v1 = {
    "accepted_corpus": (
        repo_root
        / "data"
        / "phase2"
        / "processed"
        / "documents_dedup_quality_accepted_v1.jsonl"
    ),

    "nb09_acceptance_report": (
        repo_root
        / "reports"
        / "phase2"
        / "deduplication_quality_acceptance_v1.json"
    ),

    "taxonomy_v1": (
        repo_root
        / "artifacts"
        / "phase2"
        / "taxonomy"
        / "taxonomy_v1.json"
    ),

    "taxonomy_freeze_report_v1": (
        repo_root
        / "reports"
        / "phase2"
        / "taxonomy_freeze_v1.json"
    ),

    "competency_prerequisite_v1": (
        repo_root
        / "artifacts"
        / "phase2"
        / "competency"
        / "competency_prerequisite_v1.json"
    ),

    "competency_freeze_report_v1": (
        repo_root
        / "reports"
        / "phase2"
        / "competency_freeze_v1.json"
    ),
}


missing_final_artifacts_v1 = [
    name
    for name, path in nb10_final_paths_v1.items()
    if not path.exists()
]


print("Final NB10 Artifact Inventory:")

for name, path in nb10_final_paths_v1.items():

    print(
        "-",
        name,
        ":",
        path.relative_to(repo_root),
        "| exists=",
        path.exists(),
    )


assert not missing_final_artifacts_v1


# ==================================================
# 3. Independent SHA validation
# ==================================================

actual_hashes_nb10_v1 = {}


print("\nFinal Artifact SHA256 validation:")


for name, path in nb10_final_paths_v1.items():

    actual_sha = sha256_file(path)

    expected_sha = EXPECTED_HASHES_NB10_V1[
        name
    ]

    actual_hashes_nb10_v1[
        name
    ] = actual_sha

    print(
        "-",
        name,
    )

    print(
        "  actual  :",
        actual_sha,
    )

    print(
        "  expected:",
        expected_sha,
    )

    print(
        "  match   :",
        actual_sha == expected_sha,
    )


hash_mismatches_nb10_v1 = {
    name: {
        "actual": actual_hashes_nb10_v1[
            name
        ],

        "expected": EXPECTED_HASHES_NB10_V1[
            name
        ],
    }

    for name in EXPECTED_HASHES_NB10_V1

    if (
        actual_hashes_nb10_v1[name]
        != EXPECTED_HASHES_NB10_V1[name]
    )
}


assert not hash_mismatches_nb10_v1


# ==================================================
# 4. Independent Artifact re-read
# ==================================================

taxonomy_disk_final_v1 = json.loads(
    nb10_final_paths_v1[
        "taxonomy_v1"
    ].read_text(
        encoding="utf-8"
    )
)


taxonomy_report_disk_final_v1 = json.loads(
    nb10_final_paths_v1[
        "taxonomy_freeze_report_v1"
    ].read_text(
        encoding="utf-8"
    )
)


competency_disk_final_v1 = json.loads(
    nb10_final_paths_v1[
        "competency_prerequisite_v1"
    ].read_text(
        encoding="utf-8"
    )
)


competency_report_disk_final_v1 = json.loads(
    nb10_final_paths_v1[
        "competency_freeze_report_v1"
    ].read_text(
        encoding="utf-8"
    )
)


# ==================================================
# 5. Cross-artifact integrity
# ==================================================

cross_artifact_failures_nb10_v1 = []


if (
    taxonomy_disk_final_v1[
        "status"
    ]
    != "FROZEN"
):

    cross_artifact_failures_nb10_v1.append(
        "TAXONOMY_NOT_FROZEN"
    )


if (
    taxonomy_report_disk_final_v1[
        "freeze_status"
    ]
    != "PASS"
):

    cross_artifact_failures_nb10_v1.append(
        "TAXONOMY_REPORT_NOT_PASS"
    )


if (
    taxonomy_report_disk_final_v1[
        "taxonomy_artifact"
    ][
        "sha256"
    ]
    != actual_hashes_nb10_v1[
        "taxonomy_v1"
    ]
):

    cross_artifact_failures_nb10_v1.append(
        "TAXONOMY_REPORT_HASH_REFERENCE_MISMATCH"
    )


if (
    competency_disk_final_v1[
        "status"
    ]
    != "FROZEN"
):

    cross_artifact_failures_nb10_v1.append(
        "COMPETENCY_NOT_FROZEN"
    )


if (
    competency_report_disk_final_v1[
        "freeze_status"
    ]
    != "PASS"
):

    cross_artifact_failures_nb10_v1.append(
        "COMPETENCY_REPORT_NOT_PASS"
    )


if (
    competency_report_disk_final_v1[
        "artifact"
    ][
        "sha256"
    ]
    != actual_hashes_nb10_v1[
        "competency_prerequisite_v1"
    ]
):

    cross_artifact_failures_nb10_v1.append(
        "COMPETENCY_REPORT_HASH_REFERENCE_MISMATCH"
    )


if (
    competency_disk_final_v1[
        "taxonomy_reference"
    ][
        "taxonomy_sha256"
    ]
    != actual_hashes_nb10_v1[
        "taxonomy_v1"
    ]
):

    cross_artifact_failures_nb10_v1.append(
        "COMPETENCY_TO_TAXONOMY_HASH_MISMATCH"
    )


if (
    competency_report_disk_final_v1[
        "taxonomy_reference"
    ][
        "sha256"
    ]
    != actual_hashes_nb10_v1[
        "taxonomy_v1"
    ]
):

    cross_artifact_failures_nb10_v1.append(
        "COMPETENCY_REPORT_TO_TAXONOMY_HASH_MISMATCH"
    )


print(
    "\nCross-artifact integrity:"
)

print(
    "- Failures:",
    cross_artifact_failures_nb10_v1,
)


assert not cross_artifact_failures_nb10_v1


# ==================================================
# 6. Frozen Semantic Snapshot
# ==================================================

print(
    "\nFrozen Semantic Snapshot:"
)

print(
    "- Top-level Topics:",
    len(
        taxonomy_disk_final_v1[
            "top_level_topics"
        ]
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        taxonomy_disk_final_v1[
            "curriculum_nodes"
        ]
    ),
)

print(
    "- Educational Modules:",
    len(
        taxonomy_disk_final_v1[
            "educational_modules"
        ]
    ),
)

print(
    "- Resolved Review Items:",
    len(
        taxonomy_disk_final_v1[
            "review_resolution"
        ][
            "corpus_resolution_register"
        ]
    ),
)

print(
    "- Competencies:",
    len(
        competency_disk_final_v1[
            "competencies"
        ]
    ),
)

print(
    "- Learning Stages:",
    competency_disk_final_v1[
        "learning_stages"
    ][
        "declared_stage_count"
    ],
)

print(
    "- Recommended Progressions:",
    competency_disk_final_v1[
        "learning_progression"
    ][
        "relation_count"
    ],
)

print(
    "- Hard Prerequisites:",
    competency_disk_final_v1[
        "hard_prerequisites"
    ][
        "relation_count"
    ],
)

print(
    "- Knowledge Gap declared:",
    competency_disk_final_v1[
        "knowledge_gap_policy"
    ][
        "knowledge_gap_declared_in_nb10"
    ],
)


# ==================================================
# 7. Explicit canonical Notebook-10 identity
# ==================================================

notebooks_phase2_dir = (
    repo_root
    / "notebooks"
    / "phase2"
)


EXPECTED_NB10_FILENAME_V1 = (
    "10_taxonomy_and_knowledge_engineering.ipynb"
)


notebook10_path_v1 = (
    notebooks_phase2_dir
    / EXPECTED_NB10_FILENAME_V1
)


print(
    "\nNotebook 10 identity discovery:"
)

print(
    "- Expected filename:",
    EXPECTED_NB10_FILENAME_V1,
)

print(
    "- Expected path:",
    notebook10_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Exists:",
    notebook10_path_v1.exists(),
)


assert notebook10_path_v1.exists()


print(
    "- Selected Notebook:",
    notebook10_path_v1.relative_to(
        repo_root
    ),
)

print(
    "- Selection method:",
    "EXACT_CANONICAL_FILENAME",
)


# ==================================================
# 8. Parse Notebook-10
# ==================================================

notebook10_raw_text_v1 = (
    notebook10_path_v1.read_text(
        encoding="utf-8"
    )
)


notebook10_payload_v1 = json.loads(
    notebook10_raw_text_v1
)


assert isinstance(
    notebook10_payload_v1,
    dict,
)

assert (
    "cells"
    in notebook10_payload_v1
)


notebook10_cells_v1 = (
    notebook10_payload_v1[
        "cells"
    ]
)


notebook10_code_cells_v1 = [
    cell
    for cell in notebook10_cells_v1
    if (
        cell.get(
            "cell_type"
        )
        == "code"
    )
]


notebook10_markdown_cells_v1 = [
    cell
    for cell in notebook10_cells_v1
    if (
        cell.get(
            "cell_type"
        )
        == "markdown"
    )
]


print(
    "\nNotebook 10 structure:"
)

print(
    "- Total cells:",
    len(
        notebook10_cells_v1
    ),
)

print(
    "- Code cells:",
    len(
        notebook10_code_cells_v1
    ),
)

print(
    "- Markdown cells:",
    len(
        notebook10_markdown_cells_v1
    ),
)


# ==================================================
# 9. Code Cell numbering extraction
# ==================================================

CODE_CELL_PATTERN_V1 = re.compile(
    r"(?m)^\s*#\s*Code Cell\s+(\d+)\s*$",
    re.IGNORECASE,
)


numbered_code_cells_v1 = []

unnumbered_code_cells_v1 = []


for physical_index, cell in enumerate(
    notebook10_cells_v1
):

    if (
        cell.get(
            "cell_type"
        )
        != "code"
    ):
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    match = CODE_CELL_PATTERN_V1.search(
        source
    )

    if match:

        numbered_code_cells_v1.append(
            {
                "physical_index": physical_index,

                "number": int(
                    match.group(1)
                ),

                "execution_count": cell.get(
                    "execution_count"
                ),

                "source": source,
            }
        )

    else:

        unnumbered_code_cells_v1.append(
            {
                "physical_index": physical_index,

                "execution_count": cell.get(
                    "execution_count"
                ),

                "source": source,
            }
        )


number_sequence_v1 = [
    row[
        "number"
    ]
    for row in numbered_code_cells_v1
]


number_counter_v1 = Counter(
    number_sequence_v1
)


duplicate_code_cell_numbers_v1 = sorted(
    number
    for number, count in number_counter_v1.items()
    if count > 1
)


EXPECTED_CODE_CELL_SEQUENCE_V1 = list(
    range(
        1,
        44,
    )
)


missing_code_cell_numbers_v1 = sorted(
    set(
        EXPECTED_CODE_CELL_SEQUENCE_V1
    )
    - set(
        number_sequence_v1
    )
)


unexpected_code_cell_numbers_v1 = sorted(
    set(
        number_sequence_v1
    )
    - set(
        EXPECTED_CODE_CELL_SEQUENCE_V1
    )
)


number_order_valid_v1 = (
    number_sequence_v1
    == EXPECTED_CODE_CELL_SEQUENCE_V1
)


print(
    "\nCode Cell numbering audit:"
)

print(
    "- Numbered Code Cells:",
    len(
        number_sequence_v1
    ),
)

print(
    "- Expected Numbered Code Cells:",
    len(
        EXPECTED_CODE_CELL_SEQUENCE_V1
    ),
)

print(
    "- First:",
    (
        number_sequence_v1[0]
        if number_sequence_v1
        else None
    ),
)

print(
    "- Last:",
    (
        number_sequence_v1[-1]
        if number_sequence_v1
        else None
    ),
)

print(
    "- Missing numbers:",
    missing_code_cell_numbers_v1,
)

print(
    "- Duplicate numbers:",
    duplicate_code_cell_numbers_v1,
)

print(
    "- Unexpected numbers:",
    unexpected_code_cell_numbers_v1,
)

print(
    "- Unnumbered Code Cells:",
    len(
        unnumbered_code_cells_v1
    ),
)

print(
    "- Exact sequential order:",
    number_order_valid_v1,
)


# ==================================================
# 10. Duplicate Code Cell diagnostics
# ==================================================

duplicate_code_cell_details_v1 = defaultdict(
    list
)


for row in numbered_code_cells_v1:

    if (
        row[
            "number"
        ]
        not in duplicate_code_cell_numbers_v1
    ):
        continue

    source_lines = [
        line.strip()
        for line in (
            row[
                "source"
            ].splitlines()
        )
        if line.strip()
    ]

    preview_lines = [
        line
        for line in source_lines
        if not CODE_CELL_PATTERN_V1.fullmatch(
            line
        )
    ]

    source_preview = (
        preview_lines[0][:160]
        if preview_lines
        else "<empty after marker>"
    )

    source_sha = hashlib.sha256(
        row[
            "source"
        ].encode(
            "utf-8"
        )
    ).hexdigest()

    duplicate_code_cell_details_v1[
        row[
            "number"
        ]
    ].append(
        {
            "physical_index_zero_based": (
                row[
                    "physical_index"
                ]
            ),

            "notebook_cell_position_one_based": (
                row[
                    "physical_index"
                ]
                + 1
            ),

            "execution_count": (
                row[
                    "execution_count"
                ]
            ),

            "source_preview": (
                source_preview
            ),

            "source_sha256": (
                source_sha
            ),

            "source_chars": len(
                row[
                    "source"
                ]
            ),

            "source_lines": len(
                row[
                    "source"
                ].splitlines()
            ),

            "source": (
                row[
                    "source"
                ]
            ),
        }
    )


if duplicate_code_cell_details_v1:

    print(
        "\nDuplicate Code Cell details:"
    )

    for code_cell_number in sorted(
        duplicate_code_cell_details_v1
    ):

        print(
            f"\n- Code Cell {code_cell_number}:"
        )

        for occurrence_index, details in enumerate(
            duplicate_code_cell_details_v1[
                code_cell_number
            ],
            start=1,
        ):

            print(
                "  Occurrence",
                occurrence_index,
            )

            print(
                "    notebook position:",
                details[
                    "notebook_cell_position_one_based"
                ],
            )

            print(
                "    physical index:",
                details[
                    "physical_index_zero_based"
                ],
            )

            print(
                "    execution_count:",
                details[
                    "execution_count"
                ],
            )

            print(
                "    source SHA256:",
                details[
                    "source_sha256"
                ],
            )

            print(
                "    chars:",
                details[
                    "source_chars"
                ],
            )

            print(
                "    lines:",
                details[
                    "source_lines"
                ],
            )

            print(
                "    preview:",
                details[
                    "source_preview"
                ],
            )


# ==================================================
# 11. Deep comparison of duplicate Cell 18
# ==================================================

duplicate_18_comparison_v1 = None


duplicate_18_rows_v1 = [
    row
    for row in numbered_code_cells_v1
    if row[
        "number"
    ] == 18
]


if len(
    duplicate_18_rows_v1
) == 2:

    source_a = (
        duplicate_18_rows_v1[
            0
        ][
            "source"
        ]
    )

    source_b = (
        duplicate_18_rows_v1[
            1
        ][
            "source"
        ]
    )

    sha_a = hashlib.sha256(
        source_a.encode(
            "utf-8"
        )
    ).hexdigest()

    sha_b = hashlib.sha256(
        source_b.encode(
            "utf-8"
        )
    ).hexdigest()

    exactly_identical = (
        source_a
        == source_b
    )

    unified_diff_lines = list(
        difflib.unified_diff(
            source_a.splitlines(),
            source_b.splitlines(),
            fromfile="Cell18_occurrence_1",
            tofile="Cell18_occurrence_2",
            lineterm="",
        )
    )

    duplicate_18_comparison_v1 = {
        "occurrence_1": {
            "notebook_position": (
                duplicate_18_rows_v1[
                    0
                ][
                    "physical_index"
                ]
                + 1
            ),

            "execution_count": (
                duplicate_18_rows_v1[
                    0
                ][
                    "execution_count"
                ]
            ),

            "sha256": (
                sha_a
            ),

            "chars": len(
                source_a
            ),

            "lines": len(
                source_a.splitlines()
            ),
        },

        "occurrence_2": {
            "notebook_position": (
                duplicate_18_rows_v1[
                    1
                ][
                    "physical_index"
                ]
                + 1
            ),

            "execution_count": (
                duplicate_18_rows_v1[
                    1
                ][
                    "execution_count"
                ]
            ),

            "sha256": (
                sha_b
            ),

            "chars": len(
                source_b
            ),

            "lines": len(
                source_b.splitlines()
            ),
        },

        "exactly_identical": (
            exactly_identical
        ),

        "diff_line_count": len(
            unified_diff_lines
        ),

        "diff": (
            unified_diff_lines
        ),
    }


    print(
        "\n========================================"
    )

    print(
        "Deep Comparison — Duplicate Code Cell 18"
    )

    print(
        "========================================"
    )

    print(
        "\nOccurrence 1:"
    )

    print(
        "- Notebook position:",
        duplicate_18_comparison_v1[
            "occurrence_1"
        ][
            "notebook_position"
        ],
    )

    print(
        "- Execution count:",
        duplicate_18_comparison_v1[
            "occurrence_1"
        ][
            "execution_count"
        ],
    )

    print(
        "- Source SHA256:",
        duplicate_18_comparison_v1[
            "occurrence_1"
        ][
            "sha256"
        ],
    )

    print(
        "- Chars:",
        duplicate_18_comparison_v1[
            "occurrence_1"
        ][
            "chars"
        ],
    )

    print(
        "- Lines:",
        duplicate_18_comparison_v1[
            "occurrence_1"
        ][
            "lines"
        ],
    )


    print(
        "\nOccurrence 2:"
    )

    print(
        "- Notebook position:",
        duplicate_18_comparison_v1[
            "occurrence_2"
        ][
            "notebook_position"
        ],
    )

    print(
        "- Execution count:",
        duplicate_18_comparison_v1[
            "occurrence_2"
        ][
            "execution_count"
        ],
    )

    print(
        "- Source SHA256:",
        duplicate_18_comparison_v1[
            "occurrence_2"
        ][
            "sha256"
        ],
    )

    print(
        "- Chars:",
        duplicate_18_comparison_v1[
            "occurrence_2"
        ][
            "chars"
        ],
    )

    print(
        "- Lines:",
        duplicate_18_comparison_v1[
            "occurrence_2"
        ][
            "lines"
        ],
    )


    print(
        "\n- Exactly identical:",
        duplicate_18_comparison_v1[
            "exactly_identical"
        ],
    )

    print(
        "- Diff lines:",
        duplicate_18_comparison_v1[
            "diff_line_count"
        ],
    )


    if exactly_identical:

        print(
            "\nRESULT:"
        )

        print(
            "The two Code Cell 18 copies are "
            "byte-for-byte identical."
        )

        print(
            "One copy can be removed without "
            "losing any logic."
        )

    else:

        print(
            "\nCell 18 unified diff:"
        )

        for line in unified_diff_lines[
            :300
        ]:

            print(
                line
            )

        if len(
            unified_diff_lines
        ) > 300:

            print(
                "\n... diff truncated after "
                "300 lines ..."
            )


# ==================================================
# 12. Static Python syntax audit
# ==================================================

try:

    from IPython.core.inputtransformer2 import (
        TransformerManager,
    )

    input_transformer_v1 = (
        TransformerManager()
    )

except Exception:

    input_transformer_v1 = None


syntax_failures_v1 = []


for row in numbered_code_cells_v1:

    source = row[
        "source"
    ]

    try:

        if (
            input_transformer_v1
            is not None
        ):

            source_for_ast = (
                input_transformer_v1.transform_cell(
                    source
                )
            )

        else:

            source_for_ast = (
                source
            )

        ast.parse(
            source_for_ast
        )

    except Exception as exc:

        syntax_failures_v1.append(
            {
                "code_cell": (
                    row[
                        "number"
                    ]
                ),

                "notebook_position": (
                    row[
                        "physical_index"
                    ]
                    + 1
                ),

                "error_type": (
                    type(
                        exc
                    ).__name__
                ),

                "error": str(
                    exc
                ),
            }
        )


print(
    "\nStatic Code Syntax Audit:"
)

print(
    "- Numbered Cells checked:",
    len(
        numbered_code_cells_v1
    ),
)

print(
    "- Syntax failures:",
    len(
        syntax_failures_v1
    ),
)


for failure in syntax_failures_v1:

    print(
        "- Code Cell",
        failure[
            "code_cell"
        ],
        "| notebook position=",
        failure[
            "notebook_position"
        ],
        "|",
        failure[
            "error_type"
        ],
        "|",
        failure[
            "error"
        ],
    )


# ==================================================
# 13. Important Markdown documentation audit
# ==================================================

all_markdown_text_v1 = "\n".join(
    "".join(
        cell.get(
            "source",
            []
        )
    )
    for cell in notebook10_markdown_cells_v1
)


required_markdown_signals_v1 = {
    "problem_resolution_summary": (
        "جمع‌بندی حل مسائل اصلی سازمان‌دهی دانش و Taxonomy"
    ),

    "taxonomy_freeze": (
        "Taxonomy v1 Freeze"
    ),

    "competency_reconciliation": (
        "Frozen-Taxonomy Competency / Prerequisite Reconciliation"
    ),

    "competency_freeze": (
        "Competency / Prerequisite v1 Freeze"
    ),

    "final_preflight": (
        "Final NB10 Preflight"
    ),
}


markdown_signal_status_v1 = {
    signal_name: (
        signal_text
        in all_markdown_text_v1
    )

    for signal_name, signal_text
    in required_markdown_signals_v1.items()
}


print(
    "\nImportant Markdown documentation:"
)


for signal_name, present in (
    markdown_signal_status_v1.items()
):

    print(
        "-",
        signal_name,
        ":",
        present,
    )


missing_required_markdown_v1 = [
    signal_name
    for signal_name, present
    in markdown_signal_status_v1.items()
    if not present
]


# ==================================================
# 14. Saved Error Output inventory
# ==================================================

saved_error_outputs_pre_runall_v1 = []


for physical_index, cell in enumerate(
    notebook10_cells_v1
):

    if (
        cell.get(
            "cell_type"
        )
        != "code"
    ):
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    marker = CODE_CELL_PATTERN_V1.search(
        source
    )

    code_cell_number = (
        int(
            marker.group(1)
        )
        if marker
        else None
    )

    for output in cell.get(
        "outputs",
        []
    ):

        if (
            output.get(
                "output_type"
            )
            == "error"
        ):

            saved_error_outputs_pre_runall_v1.append(
                {
                    "physical_index": (
                        physical_index
                    ),

                    "notebook_position": (
                        physical_index
                        + 1
                    ),

                    "code_cell_number": (
                        code_cell_number
                    ),

                    "ename": (
                        output.get(
                            "ename"
                        )
                    ),

                    "evalue": (
                        output.get(
                            "evalue"
                        )
                    ),
                }
            )


print(
    "\nSaved Error Outputs before Final Run-All:"
)

print(
    "- Count:",
    len(
        saved_error_outputs_pre_runall_v1
    ),
)


for row in saved_error_outputs_pre_runall_v1:

    print(
        "- Code Cell",
        row[
            "code_cell_number"
        ],
        "| notebook position=",
        row[
            "notebook_position"
        ],
        "|",
        row[
            "ename"
        ],
        "|",
        row[
            "evalue"
        ],
    )


# ==================================================
# 15. Execution-count diagnostics
# ==================================================

execution_counts_pre_runall_v1 = [
    cell.get(
        "execution_count"
    )
    for cell in notebook10_code_cells_v1
]


none_execution_count_v1 = sum(
    value is None
    for value in execution_counts_pre_runall_v1
)


numeric_execution_counts_v1 = [
    value
    for value in execution_counts_pre_runall_v1
    if isinstance(
        value,
        int,
    )
]


duplicate_execution_counts_pre_runall_v1 = sorted(
    count
    for count, occurrences
    in Counter(
        numeric_execution_counts_v1
    ).items()
    if occurrences > 1
)


print(
    "\nExecution-count diagnostics "
    "(descriptive before final Run-All):"
)

print(
    "- Code Cells with execution_count=None:",
    none_execution_count_v1,
)

print(
    "- Duplicate numeric execution counts:",
    duplicate_execution_counts_pre_runall_v1,
)


# ==================================================
# 16. Final Pre-Run-All blockers
# ==================================================

nb10_pre_runall_blockers_v1 = []


if hash_mismatches_nb10_v1:

    nb10_pre_runall_blockers_v1.append(
        "FROZEN_ARTIFACT_HASH_MISMATCH"
    )


if cross_artifact_failures_nb10_v1:

    nb10_pre_runall_blockers_v1.append(
        "CROSS_ARTIFACT_REFERENCE_FAILURE"
    )


if missing_code_cell_numbers_v1:

    nb10_pre_runall_blockers_v1.append(
        "MISSING_CODE_CELL_NUMBER"
    )


if duplicate_code_cell_numbers_v1:

    nb10_pre_runall_blockers_v1.append(
        "DUPLICATE_CODE_CELL_NUMBER"
    )


if unexpected_code_cell_numbers_v1:

    nb10_pre_runall_blockers_v1.append(
        "UNEXPECTED_CODE_CELL_NUMBER"
    )


if unnumbered_code_cells_v1:

    nb10_pre_runall_blockers_v1.append(
        "UNNUMBERED_CODE_CELL"
    )


if not number_order_valid_v1:

    nb10_pre_runall_blockers_v1.append(
        "CODE_CELL_SEQUENCE_NOT_STRICTLY_ORDERED"
    )


if syntax_failures_v1:

    nb10_pre_runall_blockers_v1.append(
        "STATIC_CODE_SYNTAX_FAILURE"
    )


if missing_required_markdown_v1:

    nb10_pre_runall_blockers_v1.append(
        "REQUIRED_DECISION_MARKDOWN_MISSING"
    )


nb10_pre_runall_readiness_v1 = (
    "READY_FOR_RESTART_RUN_ALL"
    if not nb10_pre_runall_blockers_v1
    else "BLOCKED"
)


# ==================================================
# 17. No mutation / write
# ==================================================

artifact_mutation_performed = False
taxonomy_mutation_performed = False
competency_mutation_performed = False
topic_mapping_mutation_performed = False
notebook_mutation_performed = False
disk_write_performed = False


# ==================================================
# 18. Final validation summary
# ==================================================

print(
    "\n========================================"
)

print(
    "Final NB10 Pre-Run-All Validation"
)

print(
    "========================================"
)


print(
    "- Selected canonical Notebook:",
    notebook10_path_v1.name,
)

print(
    "- Frozen Artifact hash mismatches:",
    len(
        hash_mismatches_nb10_v1
    ),
)

print(
    "- Cross-artifact failures:",
    len(
        cross_artifact_failures_nb10_v1
    ),
)

print(
    "- Missing Code Cell numbers:",
    len(
        missing_code_cell_numbers_v1
    ),
)

print(
    "- Duplicate Code Cell numbers:",
    len(
        duplicate_code_cell_numbers_v1
    ),
)

print(
    "- Unexpected Code Cell numbers:",
    len(
        unexpected_code_cell_numbers_v1
    ),
)

print(
    "- Unnumbered Code Cells:",
    len(
        unnumbered_code_cells_v1
    ),
)

print(
    "- Static syntax failures:",
    len(
        syntax_failures_v1
    ),
)

print(
    "- Missing required Markdown sections:",
    len(
        missing_required_markdown_v1
    ),
)

print(
    "- Saved historical Error outputs:",
    len(
        saved_error_outputs_pre_runall_v1
    ),
)

print(
    "- Pre-Run-All blockers:",
    nb10_pre_runall_blockers_v1,
)

print(
    "- Readiness:",
    nb10_pre_runall_readiness_v1,
)

print(
    "- Disk write performed:",
    disk_write_performed,
)


# ==================================================
# 19. Gate logic
#
# Artifact / semantic failures remain assertions.
# Duplicate Cell 18 is intentionally reported
# clearly so we can decide which occurrence to remove.
# ==================================================

assert not hash_mismatches_nb10_v1

assert not cross_artifact_failures_nb10_v1

assert (
    notebook10_path_v1.name
    == EXPECTED_NB10_FILENAME_V1
)

assert len(
    taxonomy_disk_final_v1[
        "top_level_topics"
    ]
) == 7

assert len(
    taxonomy_disk_final_v1[
        "curriculum_nodes"
    ]
) == 7

assert len(
    taxonomy_disk_final_v1[
        "educational_modules"
    ]
) == 1

assert len(
    taxonomy_disk_final_v1[
        "review_resolution"
    ][
        "corpus_resolution_register"
    ]
) == 11

assert len(
    competency_disk_final_v1[
        "competencies"
    ]
) == 24

assert (
    competency_disk_final_v1[
        "learning_progression"
    ][
        "relation_count"
    ]
    == 17
)

assert (
    competency_disk_final_v1[
        "hard_prerequisites"
    ][
        "relation_count"
    ]
    == 0
)

assert not syntax_failures_v1

assert not missing_required_markdown_v1

assert artifact_mutation_performed is False
assert taxonomy_mutation_performed is False
assert competency_mutation_performed is False
assert topic_mapping_mutation_performed is False
assert notebook_mutation_performed is False
assert disk_write_performed is False


# ==================================================
# 20. PASS or controlled BLOCKED result
# ==================================================

if duplicate_code_cell_numbers_v1:

    print(
        "\nCODE CELL 43 GATE: BLOCKED"
    )

    print(
        "Reason: duplicate numbered Code Cell(s) "
        "exist in the saved NB10 notebook."
    )

    print(
        "Duplicates:",
        duplicate_code_cell_numbers_v1,
    )

    if (
        duplicate_18_comparison_v1
        is not None
    ):

        print(
            "Cell 18 exactly identical:",
            duplicate_18_comparison_v1[
                "exactly_identical"
            ],
        )

        print(
            "Cell 18 diff lines:",
            duplicate_18_comparison_v1[
                "diff_line_count"
            ],
        )

    raise AssertionError(
        "NB10 structural health is blocked by "
        "duplicate Code Cell numbering. "
        "Review the Deep Comparison output above "
        "before deleting either Cell 18 occurrence."
    )


assert not missing_code_cell_numbers_v1
assert not unexpected_code_cell_numbers_v1
assert not unnumbered_code_cells_v1

assert number_order_valid_v1

assert not nb10_pre_runall_blockers_v1

assert (
    nb10_pre_runall_readiness_v1
    == "READY_FOR_RESTART_RUN_ALL"
)


print(
    "\nCODE CELL 43 GATE: PASS"
)

print(
    "Final NB10 Preflight PASS: the canonical "
    "10_taxonomy_and_knowledge_engineering.ipynb "
    "was explicitly selected; all frozen artifacts "
    "and cross-references are hash-valid; Code Cells "
    "01–43 occur exactly once and in strict order; "
    "static syntax is valid; required decision "
    "documentation is present; and Notebook 10 is "
    "READY for Restart Kernel → Run All → Save → "
    "final Zero-Error health audit."
)

Final NB10 Artifact Inventory:
- accepted_corpus : data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl | exists= True
- nb09_acceptance_report : reports/phase2/deduplication_quality_acceptance_v1.json | exists= True
- taxonomy_v1 : artifacts/phase2/taxonomy/taxonomy_v1.json | exists= True
- taxonomy_freeze_report_v1 : reports/phase2/taxonomy_freeze_v1.json | exists= True
- competency_prerequisite_v1 : artifacts/phase2/competency/competency_prerequisite_v1.json | exists= True
- competency_freeze_report_v1 : reports/phase2/competency_freeze_v1.json | exists= True

Final Artifact SHA256 validation:
- accepted_corpus
  actual  : 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
  expected: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
  match   : True
- nb09_acceptance_report
  actual  : 14dfa748988db17a19c6d9d5e1eac2eb7d23c0a1699dcc7add18a2bf461fe4f3
  expected: 14dfa748988db17a19c6d9d5e1eac2eb7d23c0a1699dcc7add18a2bf461fe4f3
  match   : True


AssertionError: 

In [ ]:
# Code Cell 44

from pathlib import Path
from collections import Counter
import ast
import hashlib
import json
import re


# ==================================================
# 1. Independent repository discovery
# ==================================================

cwd_final_v1 = Path.cwd().resolve()

repo_root_final_v1 = None


for candidate in [
    cwd_final_v1,
    *cwd_final_v1.parents,
]:

    expected_notebook = (
        candidate
        / "notebooks"
        / "phase2"
        / "10_taxonomy_and_knowledge_engineering.ipynb"
    )

    if expected_notebook.exists():

        repo_root_final_v1 = candidate
        break


assert (
    repo_root_final_v1
    is not None
), (
    "Could not discover repository root "
    "containing canonical Notebook 10."
)


notebook10_final_path_v1 = (
    repo_root_final_v1
    / "notebooks"
    / "phase2"
    / "10_taxonomy_and_knowledge_engineering.ipynb"
)


assert notebook10_final_path_v1.exists()


print(
    "=" * 72
)

print(
    "NB10 PERMANENT FINAL HEALTH AUDIT"
)

print(
    "=" * 72
)

print(
    "\nNotebook:",
    notebook10_final_path_v1.relative_to(
        repo_root_final_v1
    ),
)


# ==================================================
# 2. Independent local SHA256 helper
# ==================================================

def nb10_final_sha256_file_v1(
    path,
):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ==================================================
# 3. Frozen Artifact contract
# ==================================================

EXPECTED_FINAL_HASHES_V1 = {
    "accepted_corpus": (
        "9d72c709f81b1218c5aac064052c7ed1"
        "da34869f671e6e32762864e0ad70ea40"
    ),

    "nb09_acceptance_report": (
        "14dfa748988db17a19c6d9d5e1eac2e"
        "b7d23c0a1699dcc7add18a2bf461fe4f3"
    ),

    "taxonomy_v1": (
        "335690044b2e4eff684f7f347809f2c5"
        "cb8c39624a939dc01408d9ca898fe977"
    ),

    "taxonomy_freeze_report_v1": (
        "fe182b0c7c6d5ad7d5dfc2aa794fa096"
        "e85001b517040a8396d5fb4c0b17e530"
    ),

    "competency_prerequisite_v1": (
        "f12f6c1631d77f29042b2009ca9dc8de"
        "e406d23ccdccdbb103c0e8cade781e87"
    ),

    "competency_freeze_report_v1": (
        "19bb27288f49e057859da3bc668f54fac"
        "440a89a2636ff5d2e49033665f0dd11"
    ),
}


FINAL_ARTIFACT_PATHS_V1 = {
    "accepted_corpus": (
        repo_root_final_v1
        / "data"
        / "phase2"
        / "processed"
        / "documents_dedup_quality_accepted_v1.jsonl"
    ),

    "nb09_acceptance_report": (
        repo_root_final_v1
        / "reports"
        / "phase2"
        / "deduplication_quality_acceptance_v1.json"
    ),

    "taxonomy_v1": (
        repo_root_final_v1
        / "artifacts"
        / "phase2"
        / "taxonomy"
        / "taxonomy_v1.json"
    ),

    "taxonomy_freeze_report_v1": (
        repo_root_final_v1
        / "reports"
        / "phase2"
        / "taxonomy_freeze_v1.json"
    ),

    "competency_prerequisite_v1": (
        repo_root_final_v1
        / "artifacts"
        / "phase2"
        / "competency"
        / "competency_prerequisite_v1.json"
    ),

    "competency_freeze_report_v1": (
        repo_root_final_v1
        / "reports"
        / "phase2"
        / "competency_freeze_v1.json"
    ),
}


# ==================================================
# 4. Frozen Artifact integrity
# ==================================================

missing_final_artifacts_v1 = []

final_hash_mismatches_v1 = []


print(
    "\nFROZEN ARTIFACT INTEGRITY"
)


for name, path in (
    FINAL_ARTIFACT_PATHS_V1.items()
):

    if not path.exists():

        missing_final_artifacts_v1.append(
            name
        )

        print(
            "-",
            name,
            ": MISSING",
        )

        continue

    actual_sha = (
        nb10_final_sha256_file_v1(
            path
        )
    )

    expected_sha = (
        EXPECTED_FINAL_HASHES_V1[
            name
        ]
    )

    matched = (
        actual_sha
        == expected_sha
    )

    print(
        "-",
        name,
        ":",
        (
            "PASS"
            if matched
            else "FAIL"
        ),
    )

    print(
        "  SHA256:",
        actual_sha,
    )

    if not matched:

        final_hash_mismatches_v1.append(
            {
                "name": name,
                "actual": actual_sha,
                "expected": expected_sha,
            }
        )


assert not missing_final_artifacts_v1
assert not final_hash_mismatches_v1


# ==================================================
# 5. Independent Frozen Artifact re-read
# ==================================================

taxonomy_final_disk_v1 = json.loads(
    FINAL_ARTIFACT_PATHS_V1[
        "taxonomy_v1"
    ].read_text(
        encoding="utf-8"
    )
)


taxonomy_report_final_disk_v1 = json.loads(
    FINAL_ARTIFACT_PATHS_V1[
        "taxonomy_freeze_report_v1"
    ].read_text(
        encoding="utf-8"
    )
)


competency_final_disk_v1 = json.loads(
    FINAL_ARTIFACT_PATHS_V1[
        "competency_prerequisite_v1"
    ].read_text(
        encoding="utf-8"
    )
)


competency_report_final_disk_v1 = json.loads(
    FINAL_ARTIFACT_PATHS_V1[
        "competency_freeze_report_v1"
    ].read_text(
        encoding="utf-8"
    )
)


# ==================================================
# 6. Frozen semantic / cross-reference integrity
# ==================================================

semantic_failures_final_v1 = []


if (
    taxonomy_final_disk_v1[
        "status"
    ]
    != "FROZEN"
):

    semantic_failures_final_v1.append(
        "TAXONOMY_NOT_FROZEN"
    )


if (
    taxonomy_report_final_disk_v1[
        "freeze_status"
    ]
    != "PASS"
):

    semantic_failures_final_v1.append(
        "TAXONOMY_FREEZE_REPORT_NOT_PASS"
    )


if (
    taxonomy_report_final_disk_v1[
        "taxonomy_artifact"
    ][
        "sha256"
    ]
    != EXPECTED_FINAL_HASHES_V1[
        "taxonomy_v1"
    ]
):

    semantic_failures_final_v1.append(
        "TAXONOMY_REPORT_HASH_REFERENCE_MISMATCH"
    )


if (
    competency_final_disk_v1[
        "status"
    ]
    != "FROZEN"
):

    semantic_failures_final_v1.append(
        "COMPETENCY_NOT_FROZEN"
    )


if (
    competency_report_final_disk_v1[
        "freeze_status"
    ]
    != "PASS"
):

    semantic_failures_final_v1.append(
        "COMPETENCY_FREEZE_REPORT_NOT_PASS"
    )


if (
    competency_final_disk_v1[
        "taxonomy_reference"
    ][
        "taxonomy_sha256"
    ]
    != EXPECTED_FINAL_HASHES_V1[
        "taxonomy_v1"
    ]
):

    semantic_failures_final_v1.append(
        "COMPETENCY_TO_TAXONOMY_HASH_MISMATCH"
    )


if (
    competency_report_final_disk_v1[
        "taxonomy_reference"
    ][
        "sha256"
    ]
    != EXPECTED_FINAL_HASHES_V1[
        "taxonomy_v1"
    ]
):

    semantic_failures_final_v1.append(
        "COMPETENCY_REPORT_TO_TAXONOMY_HASH_MISMATCH"
    )


print(
    "\nFROZEN SEMANTIC INTEGRITY"
)

print(
    "- Failures:",
    semantic_failures_final_v1,
)

print(
    "- Top-level Topics:",
    len(
        taxonomy_final_disk_v1[
            "top_level_topics"
        ]
    ),
)

print(
    "- Curriculum Nodes:",
    len(
        taxonomy_final_disk_v1[
            "curriculum_nodes"
        ]
    ),
)

print(
    "- Educational Modules:",
    len(
        taxonomy_final_disk_v1[
            "educational_modules"
        ]
    ),
)

print(
    "- Resolved Review Items:",
    len(
        taxonomy_final_disk_v1[
            "review_resolution"
        ][
            "corpus_resolution_register"
        ]
    ),
)

print(
    "- Competencies:",
    len(
        competency_final_disk_v1[
            "competencies"
        ]
    ),
)

print(
    "- Learning Stages:",
    competency_final_disk_v1[
        "learning_stages"
    ][
        "declared_stage_count"
    ],
)

print(
    "- Recommended Progressions:",
    competency_final_disk_v1[
        "learning_progression"
    ][
        "relation_count"
    ],
)

print(
    "- Hard Prerequisites:",
    competency_final_disk_v1[
        "hard_prerequisites"
    ][
        "relation_count"
    ],
)


assert not semantic_failures_final_v1

assert len(
    taxonomy_final_disk_v1[
        "top_level_topics"
    ]
) == 7

assert len(
    taxonomy_final_disk_v1[
        "curriculum_nodes"
    ]
) == 7

assert len(
    taxonomy_final_disk_v1[
        "educational_modules"
    ]
) == 1

assert len(
    taxonomy_final_disk_v1[
        "review_resolution"
    ][
        "corpus_resolution_register"
    ]
) == 11

assert len(
    competency_final_disk_v1[
        "competencies"
    ]
) == 24

assert (
    competency_final_disk_v1[
        "learning_stages"
    ][
        "declared_stage_count"
    ]
    == 6
)

assert (
    competency_final_disk_v1[
        "learning_progression"
    ][
        "relation_count"
    ]
    == 17
)

assert (
    competency_final_disk_v1[
        "hard_prerequisites"
    ][
        "relation_count"
    ]
    == 0
)


# ==================================================
# 7. Runtime state from current Kernel
# ==================================================

runtime_checks_final_v1 = {
    "taxonomy_freeze_executed_v1": (
        globals().get(
            "taxonomy_freeze_executed_v1"
        )
        is True
    ),

    "competency_freeze_executed_v1": (
        globals().get(
            "competency_freeze_executed_v1"
        )
        is True
    ),

    "cell43_ready": (
        globals().get(
            "nb10_pre_runall_readiness_v1"
        )
        == "READY_FOR_RESTART_RUN_ALL"
    ),
}


print(
    "\nCURRENT KERNEL RUNTIME HEALTH"
)


for name, passed in (
    runtime_checks_final_v1.items()
):

    print(
        "-",
        name,
        ":",
        (
            "PASS"
            if passed
            else "FAIL"
        ),
    )


failed_runtime_checks_final_v1 = [
    name
    for name, passed in (
        runtime_checks_final_v1.items()
    )
    if not passed
]


assert not failed_runtime_checks_final_v1


# ==================================================
# 8. Read saved Notebook
# ==================================================

notebook_final_payload_v1 = json.loads(
    notebook10_final_path_v1.read_text(
        encoding="utf-8"
    )
)


notebook_final_cells_v1 = (
    notebook_final_payload_v1[
        "cells"
    ]
)


CODE_CELL_PATTERN_FINAL_V1 = re.compile(
    r"(?m)^\s*#\s*Code Cell\s+(\d+)\s*$",
    re.IGNORECASE,
)


numbered_final_cells_v1 = []

unnumbered_final_code_cells_v1 = []


for physical_index, cell in enumerate(
    notebook_final_cells_v1
):

    if (
        cell.get(
            "cell_type"
        )
        != "code"
    ):

        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    match = (
        CODE_CELL_PATTERN_FINAL_V1.search(
            source
        )
    )

    if match:

        numbered_final_cells_v1.append(
            {
                "number": int(
                    match.group(
                        1
                    )
                ),

                "physical_index": (
                    physical_index
                ),

                "execution_count": (
                    cell.get(
                        "execution_count"
                    )
                ),

                "outputs": (
                    cell.get(
                        "outputs",
                        []
                    )
                ),

                "source": source,
            }
        )

    else:

        unnumbered_final_code_cells_v1.append(
            {
                "physical_index": (
                    physical_index
                ),

                "execution_count": (
                    cell.get(
                        "execution_count"
                    )
                ),

                "source": source,
            }
        )


# ==================================================
# 9. Numbering integrity — 01 through 44
# ==================================================

number_sequence_final_v1 = [
    row[
        "number"
    ]
    for row in (
        numbered_final_cells_v1
    )
]


number_counter_final_v1 = Counter(
    number_sequence_final_v1
)


duplicate_numbers_final_v1 = sorted(
    number
    for number, count in (
        number_counter_final_v1.items()
    )
    if count > 1
)


EXPECTED_FINAL_SEQUENCE_V1 = list(
    range(
        1,
        45,
    )
)


missing_numbers_final_v1 = sorted(
    set(
        EXPECTED_FINAL_SEQUENCE_V1
    )
    - set(
        number_sequence_final_v1
    )
)


unexpected_numbers_final_v1 = sorted(
    set(
        number_sequence_final_v1
    )
    - set(
        EXPECTED_FINAL_SEQUENCE_V1
    )
)


strict_number_order_final_v1 = (
    number_sequence_final_v1
    == EXPECTED_FINAL_SEQUENCE_V1
)


print(
    "\nNOTEBOOK STRUCTURAL HEALTH"
)

print(
    "- Numbered Code Cells:",
    len(
        numbered_final_cells_v1
    ),
)

print(
    "- Expected:",
    44,
)

print(
    "- Missing:",
    missing_numbers_final_v1,
)

print(
    "- Duplicates:",
    duplicate_numbers_final_v1,
)

print(
    "- Unexpected:",
    unexpected_numbers_final_v1,
)

print(
    "- Unnumbered Code Cells:",
    len(
        unnumbered_final_code_cells_v1
    ),
)

print(
    "- Exact order 01-44:",
    strict_number_order_final_v1,
)


assert not missing_numbers_final_v1
assert not duplicate_numbers_final_v1
assert not unexpected_numbers_final_v1
assert not unnumbered_final_code_cells_v1

assert strict_number_order_final_v1


# ==================================================
# 10. Static syntax audit — all 44 Cells
# ==================================================

try:

    from IPython.core.inputtransformer2 import (
        TransformerManager,
    )

    transformer_final_v1 = (
        TransformerManager()
    )

except Exception:

    transformer_final_v1 = None


syntax_failures_final_v1 = []


for row in (
    numbered_final_cells_v1
):

    try:

        source_for_ast = (
            row[
                "source"
            ]
        )

        if (
            transformer_final_v1
            is not None
        ):

            source_for_ast = (
                transformer_final_v1.transform_cell(
                    source_for_ast
                )
            )

        ast.parse(
            source_for_ast
        )

    except Exception as exc:

        syntax_failures_final_v1.append(
            {
                "code_cell": (
                    row[
                        "number"
                    ]
                ),

                "error_type": (
                    type(
                        exc
                    ).__name__
                ),

                "error": str(
                    exc
                ),
            }
        )


print(
    "\nSTATIC SYNTAX HEALTH"
)

print(
    "- Cells checked:",
    len(
        numbered_final_cells_v1
    ),
)

print(
    "- Syntax failures:",
    len(
        syntax_failures_final_v1
    ),
)


for failure in (
    syntax_failures_final_v1
):

    print(
        "- Cell",
        failure[
            "code_cell"
        ],
        "|",
        failure[
            "error_type"
        ],
        "|",
        failure[
            "error"
        ],
    )


assert not syntax_failures_final_v1


# ==================================================
# 11. Saved Run-All execution health
#
# Cells 01-43 MUST be executed in saved notebook.
# Cell 44 itself may still have execution_count=None
# because it is currently executing.
# ==================================================

prior_cells_final_v1 = [
    row
    for row in (
        numbered_final_cells_v1
    )
    if (
        1
        <= row[
            "number"
        ]
        <= 43
    )
]


not_executed_prior_cells_v1 = [
    row[
        "number"
    ]
    for row in (
        prior_cells_final_v1
    )
    if (
        row[
            "execution_count"
        ]
        is None
    )
]


print(
    "\nSAVED RUN-ALL EXECUTION HEALTH"
)

print(
    "- Prior Cells expected:",
    43,
)

print(
    "- Prior Cells found:",
    len(
        prior_cells_final_v1
    ),
)

print(
    "- Prior Cells not executed:",
    not_executed_prior_cells_v1,
)


assert len(
    prior_cells_final_v1
) == 43

assert not not_executed_prior_cells_v1


# ==================================================
# 12. Saved Error-output audit — Cells 01-43
# ==================================================

saved_errors_final_v1 = []


for row in (
    prior_cells_final_v1
):

    for output in (
        row[
            "outputs"
        ]
    ):

        if (
            output.get(
                "output_type"
            )
            != "error"
        ):

            continue

        saved_errors_final_v1.append(
            {
                "code_cell": (
                    row[
                        "number"
                    ]
                ),

                "execution_count": (
                    row[
                        "execution_count"
                    ]
                ),

                "ename": (
                    output.get(
                        "ename",
                        ""
                    )
                ),

                "evalue": (
                    output.get(
                        "evalue",
                        ""
                    )
                ),
            }
        )


print(
    "\nSAVED ERROR-OUTPUT HEALTH"
)

print(
    "- Error outputs in Cells 01-43:",
    len(
        saved_errors_final_v1
    ),
)


for error in (
    saved_errors_final_v1
):

    print(
        "- Cell",
        error[
            "code_cell"
        ],
        "|",
        error[
            "ename"
        ],
        "|",
        error[
            "evalue"
        ],
    )


assert not saved_errors_final_v1


# ==================================================
# 13. Critical Gate PASS-marker audit
# ==================================================

CRITICAL_FINAL_GATES_V1 = [
    38,
    39,
    40,
    41,
    42,
    43,
]


critical_gate_status_final_v1 = {}


for gate_number in (
    CRITICAL_FINAL_GATES_V1
):

    gate_row = next(
        row
        for row in (
            prior_cells_final_v1
        )
        if (
            row[
                "number"
            ]
            == gate_number
        )
    )

    output_text_parts = []


    for output in (
        gate_row[
            "outputs"
        ]
    ):

        if (
            output.get(
                "output_type"
            )
            == "stream"
        ):

            text_value = output.get(
                "text",
                ""
            )

            if isinstance(
                text_value,
                list,
            ):

                output_text_parts.extend(
                    text_value
                )

            else:

                output_text_parts.append(
                    str(
                        text_value
                    )
                )

        elif (
            output.get(
                "output_type"
            )
            in {
                "execute_result",
                "display_data",
            }
        ):

            text_value = (
                output.get(
                    "data",
                    {}
                ).get(
                    "text/plain",
                    ""
                )
            )

            if isinstance(
                text_value,
                list,
            ):

                output_text_parts.extend(
                    text_value
                )

            else:

                output_text_parts.append(
                    str(
                        text_value
                    )
                )


    output_text = "\n".join(
        output_text_parts
    )


    expected_marker = (
        f"CODE CELL {gate_number} GATE: PASS"
    )


    critical_gate_status_final_v1[
        gate_number
    ] = (
        expected_marker
        in output_text
    )


print(
    "\nCRITICAL GATE HEALTH"
)


for gate_number in (
    CRITICAL_FINAL_GATES_V1
):

    print(
        f"- Cell {gate_number}:",
        (
            "PASS"
            if critical_gate_status_final_v1[
                gate_number
            ]
            else "FAIL"
        ),
    )


failed_critical_gates_final_v1 = [
    gate_number
    for gate_number, passed in (
        critical_gate_status_final_v1.items()
    )
    if not passed
]


assert not failed_critical_gates_final_v1


# ==================================================
# 14. Final Markdown documentation
# ==================================================

all_final_markdown_text_v1 = "\n".join(
    "".join(
        cell.get(
            "source",
            []
        )
    )
    for cell in (
        notebook_final_cells_v1
    )
    if (
        cell.get(
            "cell_type"
        )
        == "markdown"
    )
)


FINAL_MARKDOWN_SIGNALS_V1 = {
    "problem_resolution": (
        "جمع‌بندی حل مسائل اصلی سازمان‌دهی دانش و Taxonomy"
    ),

    "taxonomy_freeze": (
        "Taxonomy v1 Freeze"
    ),

    "competency_reconciliation": (
        "Frozen-Taxonomy Competency / Prerequisite Reconciliation"
    ),

    "competency_freeze": (
        "Competency / Prerequisite v1 Freeze"
    ),

    "final_nb10_summary": (
        "جمع‌بندی نهایی Notebook 10"
    ),

    "final_health_policy": (
        "Final Notebook Health Policy"
    ),
}


missing_final_markdown_signals_v1 = []


print(
    "\nFINAL DOCUMENTATION HEALTH"
)


for signal_name, signal_text in (
    FINAL_MARKDOWN_SIGNALS_V1.items()
):

    present = (
        signal_text
        in all_final_markdown_text_v1
    )

    print(
        "-",
        signal_name,
        ":",
        present,
    )

    if not present:

        missing_final_markdown_signals_v1.append(
            signal_name
        )


assert not missing_final_markdown_signals_v1


# ==================================================
# 15. Knowledge-gap / future-work guardrails
# ==================================================

assert (
    competency_final_disk_v1[
        "knowledge_gap_policy"
    ][
        "knowledge_gap_declared_in_nb10"
    ]
    is False
)


nb10_open_future_items_v1 = [
    "RIGHTS_RELEASE_GATE",
    "UNRESOLVED_TRANSCRIPT_SPEAKER_CONTEXT_2354",
    "TRANSCRIPT_RETRIEVAL_CONTEXT_EVALUATION",
    "CHUNKING_NB11",
    "RETRIEVAL_EVALUATION",
    "ANSWERABILITY_EVALUATION",
    "QUESTION_BASED_EVALUATION",
    "KNOWLEDGE_GAP_EVALUATION",
    "REVIEW_AND_IMPROVE_STAGE_FUTURE_REVIEW",
]


print(
    "\nDOWNSTREAM / OPEN ITEMS"
)


for item in (
    nb10_open_future_items_v1
):

    print(
        "-",
        item,
    )


# ==================================================
# 16. No mutation / no write
# ==================================================

source_document_mutation_final_v1 = False
topic_mapping_mutation_final_v1 = False
taxonomy_mutation_final_v1 = False
competency_mutation_final_v1 = False
notebook_mutation_final_v1 = False
disk_write_final_v1 = False


# ==================================================
# 17. Permanent Final Health result
# ==================================================

NB10_FINAL_HEALTH_PASS = True


nb10_final_health_v1 = {
    "notebook": (
        "10_taxonomy_and_knowledge_engineering.ipynb"
    ),

    "status": (
        "PASS"
    ),

    "numbered_code_cells": (
        44
    ),

    "saved_run_all_cells_validated": (
        43
    ),

    "saved_error_outputs_01_43": (
        len(
            saved_errors_final_v1
        )
    ),

    "unexecuted_cells_01_43": (
        len(
            not_executed_prior_cells_v1
        )
    ),

    "critical_gates_passed": (
        sum(
            critical_gate_status_final_v1.values()
        )
    ),

    "critical_gates_expected": (
        len(
            CRITICAL_FINAL_GATES_V1
        )
    ),

    "artifact_hash_mismatches": (
        len(
            final_hash_mismatches_v1
        )
    ),

    "taxonomy_status": (
        taxonomy_final_disk_v1[
            "status"
        ]
    ),

    "competency_status": (
        competency_final_disk_v1[
            "status"
        ]
    ),

    "knowledge_gap_declared": (
        False
    ),

    "ready_for_nb11": (
        True
    ),
}


print(
    "\n"
    + "=" * 72
)

print(
    "FINAL NB10 HEALTH SUMMARY"
)

print(
    "=" * 72
)


for key, value in (
    nb10_final_health_v1.items()
):

    print(
        "-",
        key,
        ":",
        value,
    )


print(
    "\nCLEAN_RUN_ALL =",
    (
        len(
            saved_errors_final_v1
        )
        == 0
        and len(
            not_executed_prior_cells_v1
        )
        == 0
        and not duplicate_numbers_final_v1
        and not missing_numbers_final_v1
        and not unexpected_numbers_final_v1
        and not failed_critical_gates_final_v1
    ),
)


assert (
    nb10_final_health_v1[
        "status"
    ]
    == "PASS"
)

assert (
    nb10_final_health_v1[
        "ready_for_nb11"
    ]
    is True
)

assert (
    NB10_FINAL_HEALTH_PASS
    is True
)

assert source_document_mutation_final_v1 is False
assert topic_mapping_mutation_final_v1 is False
assert taxonomy_mutation_final_v1 is False
assert competency_mutation_final_v1 is False
assert notebook_mutation_final_v1 is False
assert disk_write_final_v1 is False


print(
    "\nCODE CELL 44 GATE: PASS"
)

print(
    "Notebook 10 PERMANENT FINAL HEALTH PASS: "
    "the frozen input chain is hash-valid; "
    "Taxonomy v1 and Competency/Prerequisite v1 "
    "remain FROZEN; the saved Run-All state for "
    "Cells 01-43 contains zero Error outputs and "
    "zero unexecuted cells; Code Cells 01-44 are "
    "unique and strictly ordered; critical gates "
    "are PASS; required documentation is present; "
    "and Notebook 10 is READY FOR NB11."
)

NB10 PERMANENT FINAL HEALTH AUDIT

Notebook: notebooks/phase2/10_taxonomy_and_knowledge_engineering.ipynb

FROZEN ARTIFACT INTEGRITY
- accepted_corpus : PASS
  SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
- nb09_acceptance_report : PASS
  SHA256: 14dfa748988db17a19c6d9d5e1eac2eb7d23c0a1699dcc7add18a2bf461fe4f3
- taxonomy_v1 : PASS
  SHA256: 335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977
- taxonomy_freeze_report_v1 : PASS
  SHA256: fe182b0c7c6d5ad7d5dfc2aa794fa096e85001b517040a8396d5fb4c0b17e530
- competency_prerequisite_v1 : PASS
  SHA256: f12f6c1631d77f29042b2009ca9dc8dee406d23ccdccdbb103c0e8cade781e87
- competency_freeze_report_v1 : PASS
  SHA256: 19bb27288f49e057859da3bc668f54fac440a89a2636ff5d2e49033665f0dd11

FROZEN SEMANTIC INTEGRITY
- Failures: []
- Top-level Topics: 7
- Curriculum Nodes: 7
- Educational Modules: 1
- Resolved Review Items: 11
- Competencies: 24
- Learning Stages: 6
- Recommended Progressions: 17
- Hard Prerequisites:

AssertionError: 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>جمع‌بندی نهایی Notebook 10 — Taxonomy & Knowledge Engineering</h2>

<p>
Notebook 10 مسئول استخراج، اعتبارسنجی، سازمان‌دهی و Freeze ساختار دانش Phase 2 است.
ورودی این Notebook همان Corpus پذیرفته‌شدۀ NB09 است و در این مرحله هیچ Rechunking،
حذف سند یا بازنویسی محتوای Source انجام نشده است.
</p>

<h3>Authority Model</h3>

<p>
مرجع اصلی دانش این پروژه دکتر کمیل رودی است.
برای جلوگیری از تأثیر محتوای مهمان، مجری، شرکت‌کننده یا محتوای Secondary بر ساختار اصلی دانش،
Corpus به چهار لایۀ Authority تفکیک شد:
</p>

<ul>
<li><strong>CORE_PRIMARY_KNOWLEDGE:</strong> 14,705 سند</li>
<li><strong>SUPERVISED_SECONDARY_KNOWLEDGE:</strong> 453 سند</li>
<li><strong>CONTEXTUAL_NONPRIMARY:</strong> 1,944 سند</li>
<li><strong>UNRESOLVED_CONTEXT:</strong> 2,354 سند</li>
</ul>

<p>
فقط Core Primary Knowledge مجاز است ساختار اصلی Taxonomy را تعریف یا تغییر دهد.
سایر لایه‌ها حذف نشده‌اند و برای Supporting Evidence، Context و Evaluationهای بعدی حفظ شده‌اند.
</p>

<h3>Taxonomy v1</h3>

<p>
هفت Top-level Topic اصلی با شواهد Primary Corpus تثبیت شدند:
</p>

<ul>
<li>مبانی و توانمندی سواد مالی</li>
<li>خرج، پس‌انداز و مدیریت پول</li>
<li>اعتبار، قرض، بدهی و بانک</li>
<li>کار، شغل و درآمد</li>
<li>سرمایه‌گذاری</li>
<li>ریسک و بیمه</li>
<li>تصمیم‌گیری مالی</li>
</ul>

<p>
Deep Book/Course Hierarchy Review نشان داد بعضی مفاهیم باید به‌عنوان Curriculum Node یا Subtopic
نمایش داده شوند و نباید مستقیماً به Top-level Topic تبدیل شوند.
</p>

<p>
از جمله شاخۀ آموزشی زیر بازیابی شد:
</p>

<p>
<strong>
کار، شغل و درآمد
→ کسب‌وکار کودکان
→ تفاوت کالا و خدمت / منابع تولید / قیمت و قیمت‌گذاری /
سود / معرفی محصول و تبلیغ
</strong>
</p>

<p>
«بخشش و هدیه» نیز به‌عنوان Curriculum Node واقعی با Bridge به
«خرج، پس‌انداز و مدیریت پول» ثبت شد.
</p>

<p>
«مالیات و تعهدات مالی» به‌عنوان Cross-topic Level-2 Educational Module ثبت شد
و به زور به یک Top-level Topic خاص متصل نشد.
</p>

<p>
مفاهیمی مانند «قیمت»، «سود» و «انتخاب» در سطح کل Corpus Context-dependent باقی ماندند.
اقتصاد رفتاری نیز به دلیل نبود Primary Evidence کافی به Core Taxonomy v1 ارتقا داده نشد.
</p>

<h3>Structural Validation</h3>

<ul>
<li>Invalid Mapping Topic IDs: 0</li>
<li>Invalid Parent References: 0</li>
<li>Invalid Top-level Bridges: 0</li>
<li>Hierarchy Cycles: 0</li>
<li>Duplicate Active Labels: 0</li>
<li>Non-primary Core Evidence Contamination: 0</li>
<li>Unapproved Multi-topic Conflicts: 0</li>
</ul>

<p>
هر 11 مورد Review قبلی با Deep Primary Corpus Review حل شدند و Expert Review تاریخی با وضعیت
<code>SUPERSEDED_BY_CORPUS_FIRST_RESOLUTION</code>
حفظ شد.
</p>

<h3>Frozen Taxonomy Artifact</h3>

<p>
<code>artifacts/phase2/taxonomy/taxonomy_v1.json</code>
</p>

<p>
SHA256:
<code>335690044b2e4eff684f7f347809f2c5cb8c39624a939dc01408d9ca898fe977</code>
</p>

<h3>Competency / Prerequisite v1</h3>

<ul>
<li>Competencies: 24</li>
<li>Top-level Topics represented: 7</li>
<li>Learning Stages: 6</li>
<li>Recommended Learning Progressions: 17</li>
<li>Hard Prerequisites: 0</li>
<li>Cross-topic Relations: 0</li>
<li>Progression Cycles: 0</li>
</ul>

<p>
Taxonomy Node الزاماً Competency مستقل ایجاد نمی‌کند.
Competency جدید تنها در صورت وجود Primary Curriculum Evidence،
Learning Outcome مستقل و نبود هم‌پوشانی با Competencyهای موجود قابل ایجاد است.
</p>

<p>
Stage
<code>review_and_improve</code>
در چارچوب آموزشی تعریف شده ولی در v1 Competency مستقلی ندارد.
این وضعیت Knowledge Gap اعلام نشده و برای Evaluation بعدی حفظ می‌شود.
</p>

<h3>Frozen Competency Artifact</h3>

<p>
<code>artifacts/phase2/competency/competency_prerequisite_v1.json</code>
</p>

<p>
SHA256:
<code>f12f6c1631d77f29042b2009ca9dc8dee406d23ccdccdbb103c0e8cade781e87</code>
</p>

<h3>Knowledge Gap Policy</h3>

<p>
Unmapped بودن سند به‌تنهایی Knowledge Gap نیست.
Knowledge Gap واقعی فقط پس از بررسی مستقل Competency Coverage،
Retrievability و Answerability قابل اعلام است.
</p>

<h3>موارد باز برای مراحل بعد</h3>

<ul>
<li>2,354 Transcript Segment دارای Unresolved Speaker Context</li>
<li>Rights Release Gate</li>
<li>Chunking و Chunk Quality</li>
<li>Transcript Retrieval Context Evaluation</li>
<li>Retrieval Evaluation</li>
<li>Answerability Evaluation</li>
<li>Question-based Evaluation</li>
<li>Knowledge Gap Evaluation</li>
<li>بررسی آتی Stage review_and_improve</li>
</ul>

<h3>Final Notebook Health Policy</h3>

<p>
از این نقطه به بعد Code Cell 44 به‌عنوان Health Check دائمی Notebook حفظ می‌شود.
این Cell در انتهای هر اجرای Notebook بررسی می‌کند که:
</p>

<ul>
<li>Code Cellهای 01 تا 44 یکتا و مرتب باشند.</li>
<li>Cellهای 01 تا 43 واقعاً اجرا شده باشند.</li>
<li>Saved Error Output در Cellهای 01 تا 43 صفر باشد.</li>
<li>Gateهای اصلی Notebook PASS باشند.</li>
<li>Artifactهای Frozen تغییر نکرده باشند.</li>
<li>Cross-reference میان Taxonomy و Competency سالم باشد.</li>
<li>Notebook برای Handoff به NB11 آماده باشد.</li>
</ul>

<p>
<strong>
Code Cell 44 بخشی دائمی از Notebook است و نباید پس از PASS حذف شود.
همین الگوی Final Health Audit در Notebookهای بعدی Phase 2 نیز باید حفظ شود.
</strong>
</p>

</div>